NATURE STEP1, NEW

In [ ]:
# ============================================================
# PepCVAE / AngioPep-VAE — Step 1
# Corpus audit, sequence-quality control, source heterogeneity,
# filtering summary, and publication-ready figure generation
#
# Improved full version
# Target standard: Nature Computational Science
#
# Main Figure:
#   Fig1_corpus_heterogeneity_main
#     a) source counts
#     b) length
#     c) net charge
#     d) hydrophobicity
#     e) entropy
#     f) dominant amino acid fraction
#
# Supplementary Figures:
#   FigS1_filtering_flow
#   FigS2_pairwise_source_shift_heatmap
#   FigS3_distribution_diagnostics
#
# Key upgrades:
# - notebook-friendly + CLI-friendly execution
# - white-background publication style
# - pairwise source-shift analysis
# - explicit filtering-flow export
# - retained dataset export
# - sequence hash + cross-source duplicate structure
# - cleaner labels and stronger figure hierarchy
# ============================================================

from __future__ import annotations

import os
import json
import hashlib
import platform
import argparse
from dataclasses import dataclass, asdict, field
from datetime import datetime
from itertools import combinations
from typing import Dict, Tuple, Any, List, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import ks_2samp
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Step1Config:
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/OncoPep_merged_clean_labeled4.csv"
    out_dir_name: str = "nature_computational/step1"

    required_columns: Tuple[str, ...] = (
        "sequence",
        "source_db4",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
    )

    source_aliases: Tuple[str, ...] = (
        "source_db4",
        "source",
        "source_db",
        "dataset",
        "db",
        "dataset_source",
    )

    optional_label_candidates: Tuple[str, ...] = (
        "label_acp",
        "acp_label",
        "is_acp",
        "label",
        "class_label",
    )

    source_order: Tuple[str, ...] = (
        "dbAMPseq",
        "APD3",
        "CancerPPD2",
        "DCP",
    )

    canonical_aa: str = "ACDEFGHIKLMNPQRSTVWY"

    min_len: int = 5
    max_len: int = 60

    cliffs_max_pairs: int = 200_000
    cliffs_subsample_n: int = 700
    rng_seed: int = 42

    png_dpi: int = 300
    tif_dpi: int = 600

    font_family: str = "sans-serif"
    font_list: Tuple[str, ...] = ("Arial", "Helvetica", "DejaVu Sans")
    font_size: int = 8
    title_size: int = 9
    axis_label_size: int = 8
    tick_label_size: int = 8
    axis_linewidth: float = 0.9
    grid_alpha: float = 0.7
    grid_lw: float = 0.35
    box_alpha: float = 0.90

    count_label_factor: float = 1.02
    count_label_font: int = 10
    main_count_top_expand: float = 1.28

    heatmap_fontsize: int = 8
    heatmap_label_rot: int = 22

    colors: Dict[str, str] = field(default_factory=lambda: {
        "dbAMPseq": "#B8C4DC",
        "APD3": "#27476E",
        "CancerPPD2": "#F2B674",
        "DCP": "#D6606D",
        "gray": "#707070",
        "black": "#111111",
        "green": "#4E9F74",
        "orange": "#D78A33",
        "grid": "#E6E6E6",
    })

    @property
    def base_dir(self) -> str:
        return os.path.dirname(os.path.abspath(self.input_csv))

    @property
    def out_dir(self) -> str:
        return os.path.join(self.base_dir, self.out_dir_name)

    @property
    def tables_dir(self) -> str:
        return os.path.join(self.out_dir, "tables")

    @property
    def reports_dir(self) -> str:
        return os.path.join(self.out_dir, "reports")

    @property
    def fig_main_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_main")

    @property
    def fig_supp_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_supplementary")


# ============================================================
# PLOT STYLE
# ============================================================

def set_plot_style(cfg: Step1Config) -> None:
    plt.rcParams.update({
        "font.family": cfg.font_family,
        "font.sans-serif": list(cfg.font_list),
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "axes.linewidth": cfg.axis_linewidth,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
    })


# ============================================================
# IO HELPERS
# ============================================================

def ensure_dir(path: str) -> str:
    path = os.path.abspath(path)
    os.makedirs(path, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step1Config) -> None:
    for p in [cfg.out_dir, cfg.tables_dir, cfg.reports_dir, cfg.fig_main_dir, cfg.fig_supp_dir]:
        ensure_dir(p)


def sha256_file(path: str, chunk_size: int = 2**20) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


def sequence_sha256(seq: str) -> str:
    return hashlib.sha256(seq.encode("utf-8")).hexdigest()


def json_default(obj: Any):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def save_csv(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)


def save_text(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def save_fig(fig, basepath_no_ext: str, cfg: Step1Config) -> None:
    fig.savefig(basepath_no_ext + ".pdf", bbox_inches="tight", pad_inches=0.08)
    fig.savefig(basepath_no_ext + ".png", dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.08)
    fig.savefig(basepath_no_ext + ".tif", dpi=cfg.tif_dpi, bbox_inches="tight", pad_inches=0.08)


# ============================================================
# BASIC HELPERS
# ============================================================

def beautify_axis(ax, cfg: Step1Config, grid_axis: str = "y") -> None:
    ax.grid(True, axis=grid_axis, linewidth=cfg.grid_lw, alpha=cfg.grid_alpha, color=cfg.colors["grid"])
    for spine in ax.spines.values():
        spine.set_linewidth(cfg.axis_linewidth)


def panel_letter(ax, letter: str, cfg: Step1Config, dx: float = -0.11, dy: float = 1.06) -> None:
    ax.text(
        dx, dy, letter,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
        ha="left",
        color=cfg.colors["black"],
    )


def finite_array(x: Sequence[float]) -> np.ndarray:
    arr = np.asarray(x, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_sequence_column(series: pd.Series) -> pd.Series:
    out = series.where(series.notna(), "")
    out = out.astype(str).str.strip().str.upper()
    return out


def ecdf(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    x = finite_array(x)
    if len(x) == 0:
        return np.array([]), np.array([])
    xs = np.sort(x)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys


def summary_stats(x: np.ndarray) -> Dict[str, Any]:
    x = finite_array(x)
    if len(x) == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "min": np.nan,
            "max": np.nan,
        }
    return {
        "n": int(len(x)),
        "mean": float(np.mean(x)),
        "sd": float(np.std(x, ddof=1)) if len(x) > 1 else np.nan,
        "median": float(np.median(x)),
        "q25": float(np.percentile(x, 25)),
        "q75": float(np.percentile(x, 75)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }


# ============================================================
# NORMALIZATION / VALIDATION
# ============================================================

def harmonize_column_names(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    out = df.copy()
    rename_map = {}

    if "source_db4" not in out.columns:
        for cand in cfg.source_aliases:
            if cand != "source_db4" and cand in out.columns:
                rename_map[cand] = "source_db4"
                break

    if rename_map:
        out = out.rename(columns=rename_map)

    return out


def infer_optional_label_column(df: pd.DataFrame, cfg: Step1Config) -> Optional[str]:
    for col in cfg.optional_label_candidates:
        if col in df.columns:
            return col
    return None


def validate_required_columns(df: pd.DataFrame, cfg: Step1Config) -> None:
    missing = [c for c in cfg.required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def validate_sources(df: pd.DataFrame, cfg: Step1Config) -> Dict[str, Any]:
    found = sorted(df["source_db4"].dropna().astype(str).unique().tolist())
    allowed = list(cfg.source_order)
    unexpected = sorted(set(found) - set(allowed))
    missing_expected = sorted(set(allowed) - set(found))
    if unexpected:
        raise ValueError(f"Unexpected source_db4 values detected: {unexpected}")
    return {
        "allowed_sources": allowed,
        "found_sources": found,
        "missing_expected_sources": missing_expected,
        "unexpected_sources": unexpected,
    }


def prepare_dataframe(raw_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = harmonize_column_names(df, cfg)
    validate_required_columns(df, cfg)

    label_col = infer_optional_label_column(df, cfg)
    if label_col is not None and label_col != "label_acp":
        df = df.rename(columns={label_col: "label_acp"})

    df["sequence_raw"] = df["sequence"]
    df["sequence"] = normalize_sequence_column(df["sequence"])
    df["source_db4"] = df["source_db4"].astype(str).str.strip()

    for col in ["length", "net_charge_pH7", "hydrophobicity_KD"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    validate_sources(df, cfg)
    return df


# ============================================================
# QC / FEATURE ENGINEERING
# ============================================================

AA20_SET = set("ACDEFGHIKLMNPQRSTVWY")


def canonical_only(seq: str) -> bool:
    return isinstance(seq, str) and len(seq) > 0 and set(seq).issubset(AA20_SET)


def max_run_length(seq: str) -> int:
    if not seq:
        return 0
    best = 1
    cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best


def shannon_entropy(seq: str) -> float:
    if not seq:
        return np.nan
    counts = {}
    for aa in seq:
        counts[aa] = counts.get(aa, 0) + 1
    n = len(seq)
    ent = 0.0
    for c in counts.values():
        p = c / n
        ent -= p * np.log2(p)
    return float(ent)


def dominant_aa_fraction(seq: str) -> float:
    if not seq:
        return np.nan
    counts = {}
    for aa in seq:
        counts[aa] = counts.get(aa, 0) + 1
    return float(max(counts.values()) / len(seq))


def add_qc_columns(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    out = df.copy()

    out["row_id_step1"] = np.arange(len(out)) + 1
    out["sequence_sha256"] = out["sequence"].apply(sequence_sha256)

    out["len_from_seq"] = out["sequence"].str.len()
    out["sequence_missing"] = out["sequence"].eq("")
    out["length_missing"] = out["length"].isna()
    out["charge_missing"] = out["net_charge_pH7"].isna()
    out["hydro_missing"] = out["hydrophobicity_KD"].isna()

    out["length_noninteger_flag"] = np.where(
        out["length"].notna(),
        np.abs(out["length"] - np.round(out["length"])) > 1e-8,
        False,
    )

    out["length_rounded_int"] = np.where(out["length"].notna(), np.round(out["length"]), np.nan)
    out["length_rounded_int"] = pd.Series(out["length_rounded_int"]).astype("Int64")

    out["len_mismatch"] = np.where(
        out["length"].notna(),
        out["length_rounded_int"] != out["len_from_seq"].astype("Int64"),
        False,
    )

    out["is_canonical"] = out["sequence"].apply(canonical_only)
    out["entropy"] = out["sequence"].apply(shannon_entropy)
    out["max_runlen"] = out["sequence"].apply(max_run_length)
    out["dominant_aa_frac"] = out["sequence"].apply(dominant_aa_fraction)

    out["length_out_of_range"] = np.where(
        out["sequence_missing"],
        False,
        (out["len_from_seq"] < cfg.min_len) | (out["len_from_seq"] > cfg.max_len),
    )

    out["is_exact_duplicate_global"] = out["sequence"].duplicated(keep=False)
    out["is_exact_duplicate_within_source"] = out.duplicated(subset=["source_db4", "sequence"], keep=False)

    source_counts = (
        out.groupby("sequence")["source_db4"]
        .nunique()
        .rename("n_sources_per_sequence")
        .reset_index()
    )
    out = out.merge(source_counts, on="sequence", how="left")
    out["n_sources_per_sequence"] = out["n_sources_per_sequence"].fillna(1).astype(int)
    out["is_cross_source_duplicate"] = out["n_sources_per_sequence"] > 1

    source_membership = (
        out.groupby("sequence")["source_db4"]
        .apply(lambda x: "|".join(sorted(set(map(str, x)))))
        .rename("source_membership_pattern")
        .reset_index()
    )
    out = out.merge(source_membership, on="sequence", how="left")

    # explicit filtering
    out["exclude_sequence_missing"] = out["sequence_missing"]
    out["exclude_noncanonical"] = ~out["is_canonical"]
    out["exclude_length_missing"] = out["length_missing"]
    out["exclude_length_mismatch"] = out["len_mismatch"]
    out["exclude_charge_missing"] = out["charge_missing"]
    out["exclude_hydro_missing"] = out["hydro_missing"]
    out["exclude_length_out_of_range"] = out["length_out_of_range"]

    exclusion_cols = [
        "exclude_sequence_missing",
        "exclude_noncanonical",
        "exclude_length_missing",
        "exclude_length_mismatch",
        "exclude_charge_missing",
        "exclude_hydro_missing",
        "exclude_length_out_of_range",
    ]
    out["exclude_any_step1"] = out[exclusion_cols].any(axis=1)
    out["include_final_step1"] = ~out["exclude_any_step1"]

    return out


# ============================================================
# STATISTICS
# ============================================================

def cliffs_delta(x: np.ndarray, y: np.ndarray, cfg: Step1Config) -> Tuple[float, Dict[str, Any]]:
    x = finite_array(x)
    y = finite_array(y)
    meta = {
        "x_n_input": int(len(x)),
        "y_n_input": int(len(y)),
        "subsampled": False,
        "x_n_used": int(len(x)),
        "y_n_used": int(len(y)),
    }
    if len(x) == 0 or len(y) == 0:
        return np.nan, meta

    rng = np.random.default_rng(cfg.rng_seed)
    if len(x) * len(y) > cfg.cliffs_max_pairs:
        meta["subsampled"] = True
        if len(x) > cfg.cliffs_subsample_n:
            x = rng.choice(x, size=cfg.cliffs_subsample_n, replace=False)
        if len(y) > cfg.cliffs_subsample_n:
            y = rng.choice(y, size=cfg.cliffs_subsample_n, replace=False)
        meta["x_n_used"] = int(len(x))
        meta["y_n_used"] = int(len(y))

    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    n_pairs = len(x) * len(y)
    delta = (gt - lt) / max(1, n_pairs)
    return float(delta), meta


def ks_test(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    if not HAS_SCIPY:
        return np.nan, np.nan
    x = finite_array(x)
    y = finite_array(y)
    if len(x) < 10 or len(y) < 10:
        return np.nan, np.nan
    stat, p = ks_2samp(x, y)
    return float(stat), float(p)


# ============================================================
# TABLE BUILDERS
# ============================================================

def build_source_summary_table(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    rows = []
    for src in cfg.source_order:
        sub = df.loc[df["source_db4"] == src].copy()

        length_stats = summary_stats(sub["length"].to_numpy())
        charge_stats = summary_stats(sub["net_charge_pH7"].to_numpy())
        hydro_stats = summary_stats(sub["hydrophobicity_KD"].to_numpy())
        entropy_stats = summary_stats(sub["entropy"].to_numpy())
        run_stats = summary_stats(sub["max_runlen"].to_numpy())
        dom_stats = summary_stats(sub["dominant_aa_frac"].to_numpy())

        row = {
            "source_db4": src,
            "n_total_rows": int(len(sub)),
            "n_unique_sequences": int(sub["sequence"].nunique()),
            "n_retained_final": int(sub["include_final_step1"].sum()),
            "canonical_fraction": float(sub["is_canonical"].mean()) if len(sub) else np.nan,
            "length_missing_fraction": float(sub["length_missing"].mean()) if len(sub) else np.nan,
            "charge_missing_fraction": float(sub["charge_missing"].mean()) if len(sub) else np.nan,
            "hydro_missing_fraction": float(sub["hydro_missing"].mean()) if len(sub) else np.nan,
            "length_mismatch_fraction": float(sub["len_mismatch"].mean()) if len(sub) else np.nan,
            "duplicate_fraction_global": float(sub["is_exact_duplicate_global"].mean()) if len(sub) else np.nan,
            "duplicate_fraction_within_source": float(sub["is_exact_duplicate_within_source"].mean()) if len(sub) else np.nan,
            "cross_source_duplicate_fraction": float(sub["is_cross_source_duplicate"].mean()) if len(sub) else np.nan,

            "length_median": length_stats["median"],
            "length_q25": length_stats["q25"],
            "length_q75": length_stats["q75"],

            "charge_median": charge_stats["median"],
            "charge_q25": charge_stats["q25"],
            "charge_q75": charge_stats["q75"],

            "hydrophobicity_median": hydro_stats["median"],
            "hydrophobicity_q25": hydro_stats["q25"],
            "hydrophobicity_q75": hydro_stats["q75"],

            "entropy_median": entropy_stats["median"],
            "entropy_q25": entropy_stats["q25"],
            "entropy_q75": entropy_stats["q75"],

            "max_runlen_median": run_stats["median"],
            "max_runlen_q25": run_stats["q25"],
            "max_runlen_q75": run_stats["q75"],

            "dominant_aa_frac_median": dom_stats["median"],
            "dominant_aa_frac_q25": dom_stats["q25"],
            "dominant_aa_frac_q75": dom_stats["q75"],
        }

        if "label_acp" in sub.columns:
            row["label_acp_nonmissing_fraction"] = float(sub["label_acp"].notna().mean()) if len(sub) else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


def build_missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        missing_fraction = float(df["sequence"].eq("").mean()) if col == "sequence" else float(df[col].isna().mean())
        rows.append({
            "column": col,
            "missing_fraction": missing_fraction,
            "dtype": str(df[col].dtype),
        })
    return pd.DataFrame(rows)


def build_missingness_by_source_table(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    rows = []
    cols = ["sequence", "length", "net_charge_pH7", "hydrophobicity_KD"]
    for src in cfg.source_order:
        sub = df.loc[df["source_db4"] == src].copy()
        for col in cols:
            if col == "sequence":
                frac = float(sub["sequence"].eq("").mean()) if len(sub) else np.nan
            else:
                frac = float(sub[col].isna().mean()) if len(sub) else np.nan
            rows.append({
                "source_db4": src,
                "column": col,
                "missing_fraction": frac,
            })
    return pd.DataFrame(rows)


def build_filtering_flow_table(df: pd.DataFrame) -> pd.DataFrame:
    total = len(df)
    rules = [
        ("exclude_sequence_missing", "Remove empty sequence"),
        ("exclude_noncanonical", "Remove non-canonical residues"),
        ("exclude_length_missing", "Remove missing length"),
        ("exclude_length_mismatch", "Remove length mismatch"),
        ("exclude_charge_missing", "Remove missing charge"),
        ("exclude_hydro_missing", "Remove missing hydrophobicity"),
        ("exclude_length_out_of_range", "Apply length window"),
    ]

    rows = []
    rows.append({
        "step_order": 0,
        "rule_column": "input_total",
        "rule_label": "Initial input",
        "n_flagged": total,
        "fraction_flagged": 1.0,
        "n_remaining_after_cumulative_filter": total,
    })

    cumulative_keep = pd.Series(True, index=df.index)
    for i, (col, label) in enumerate(rules, start=1):
        flagged = int(df[col].sum())
        cumulative_keep = cumulative_keep & (~df[col])
        rows.append({
            "step_order": i,
            "rule_column": col,
            "rule_label": label,
            "n_flagged": flagged,
            "fraction_flagged": flagged / total if total else np.nan,
            "n_remaining_after_cumulative_filter": int(cumulative_keep.sum()),
        })

    rows.append({
        "step_order": len(rules) + 1,
        "rule_column": "include_final_step1",
        "rule_label": "Final retained corpus",
        "n_flagged": int(df["include_final_step1"].sum()),
        "fraction_flagged": float(df["include_final_step1"].mean()) if total else np.nan,
        "n_remaining_after_cumulative_filter": int(df["include_final_step1"].sum()),
    })
    return pd.DataFrame(rows)


def build_domain_shift_table_pairwise(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    features = [
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "entropy",
        "max_runlen",
        "dominant_aa_frac",
    ]
    rows = []
    for src_a, src_b in combinations(cfg.source_order, 2):
        sub_a = df.loc[df["source_db4"] == src_a].copy()
        sub_b = df.loc[df["source_db4"] == src_b].copy()
        for feat in features:
            x = sub_a[feat].to_numpy()
            y = sub_b[feat].to_numpy()
            ks_stat, ks_p = ks_test(x, y)
            cd, meta = cliffs_delta(x, y, cfg)
            rows.append({
                "source_a": src_a,
                "source_b": src_b,
                "feature": feat,
                "ks_stat": ks_stat,
                "ks_pvalue": ks_p,
                "cliffs_delta": cd,
                "cliffs_subsampled": bool(meta["subsampled"]),
                "x_n_input": meta["x_n_input"],
                "y_n_input": meta["y_n_input"],
                "x_n_used": meta["x_n_used"],
                "y_n_used": meta["y_n_used"],
            })
    return pd.DataFrame(rows)


# ============================================================
# PLOTTING HELPERS
# ============================================================

def plot_box_or_violin(
    ax,
    df: pd.DataFrame,
    col: str,
    ylabel: str,
    cfg: Step1Config,
    y_limits: Optional[Tuple[float, float]] = None,
) -> None:
    data = []
    for src in cfg.source_order:
        x = finite_array(df.loc[df["source_db4"] == src, col].to_numpy())
        data.append(x)

    positions = np.arange(1, len(cfg.source_order) + 1)

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=0.70,
        showmeans=False,
        showmedians=True,
        showextrema=False,
    )
    for body, src in zip(vp["bodies"], cfg.source_order):
        body.set_facecolor(cfg.colors[src])
        body.set_edgecolor(cfg.colors["gray"])
        body.set_alpha(0.58)
        body.set_linewidth(0.8)

    if "cmedians" in vp:
        vp["cmedians"].set_color(cfg.colors["black"])
        vp["cmedians"].set_linewidth(1.2)

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color=cfg.colors["black"], linewidth=1.2),
        whiskerprops=dict(color=cfg.colors["gray"], linewidth=0.8),
        capprops=dict(color=cfg.colors["gray"], linewidth=0.8),
        boxprops=dict(color=cfg.colors["gray"], linewidth=0.8),
    )
    for patch, src in zip(bp["boxes"], cfg.source_order):
        patch.set_facecolor(cfg.colors[src])
        patch.set_alpha(0.90)
        patch.set_edgecolor(cfg.colors["gray"])

    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(cfg.source_order, rotation=12, ha="right")
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    beautify_axis(ax, cfg, "y")


def plot_ecdf_overlay(ax, df: pd.DataFrame, col: str, xlabel: str, cfg: Step1Config) -> None:
    for src in cfg.source_order:
        x = df.loc[df["source_db4"] == src, col].to_numpy()
        xs, ys = ecdf(x)
        if len(xs):
            ax.plot(xs, ys, linewidth=1.6, color=cfg.colors[src], label=src)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("ECDF")
    beautify_axis(ax, cfg, "both")


# ============================================================
# FIGURES
# ============================================================

def fig_main_corpus_heterogeneity(df: pd.DataFrame, cfg: Step1Config) -> None:
    df_main = df.loc[df["include_final_step1"]].copy()
    counts = df_main["source_db4"].value_counts().reindex(cfg.source_order).fillna(0).astype(int)

    fig, axes = plt.subplots(2, 3, figsize=(12.0, 6.8))
    fig.subplots_adjust(left=0.07, right=0.985, top=0.95, bottom=0.12, wspace=0.32, hspace=0.36)

    # a) source composition
    ax = axes[0, 0]
    x = np.arange(len(cfg.source_order))
    bars = ax.bar(
        x,
        counts.values,
        width=0.78,
        color=[cfg.colors[s] for s in cfg.source_order],
        edgecolor=cfg.colors["gray"],
        linewidth=0.8,
    )
    ax.set_yscale("log")
    ax.set_ylabel("Number of sequences (log scale)")
    ax.set_xticks(x)
    ax.set_xticklabels(cfg.source_order, rotation=12, ha="right")
    beautify_axis(ax, cfg, "y")

    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * cfg.main_count_top_expand)

    for b, v in zip(bars, counts.values):
        ax.text(
            b.get_x() + b.get_width() / 2,
            v * cfg.count_label_factor,
            f"{v:,}",
            ha="center",
            va="bottom",
            fontsize=cfg.count_label_font,
            color=cfg.colors["black"],
            fontweight="medium",
        )
    panel_letter(ax, "a", cfg)

    # b) length
    plot_box_or_violin(axes[0, 1], df_main, "length", "Length (aa)", cfg)
    panel_letter(axes[0, 1], "b", cfg)

    # c) charge
    plot_box_or_violin(axes[0, 2], df_main, "net_charge_pH7", "Net charge (pH 7)", cfg)
    panel_letter(axes[0, 2], "c", cfg)

    # d) hydrophobicity
    plot_box_or_violin(axes[1, 0], df_main, "hydrophobicity_KD", "Hydrophobicity (Kyte–Doolittle)", cfg)
    panel_letter(axes[1, 0], "d", cfg)

    # e) entropy
    plot_box_or_violin(axes[1, 1], df_main, "entropy", "Shannon entropy (bits)", cfg)
    panel_letter(axes[1, 1], "e", cfg)

    # f) dominant amino acid fraction
    plot_box_or_violin(axes[1, 2], df_main, "dominant_aa_frac", "Dominant amino acid fraction", cfg, y_limits=(0.05, 1.0))
    panel_letter(axes[1, 2], "f", cfg)

    save_fig(fig, os.path.join(cfg.fig_main_dir, "Fig1_corpus_heterogeneity_main"), cfg)
    plt.close(fig)


def fig_supp_filtering_flow(flow_df: pd.DataFrame, cfg: Step1Config) -> None:
    fig, ax = plt.subplots(1, 1, figsize=(8.4, 3.9))
    fig.subplots_adjust(left=0.10, right=0.985, top=0.92, bottom=0.28)

    plot_df = flow_df.copy()
    xx = np.arange(len(plot_df))
    colors = [cfg.colors["green"]] * (len(plot_df) - 1) + [cfg.colors["orange"]]

    bars = ax.bar(
        xx,
        plot_df["n_remaining_after_cumulative_filter"].values,
        color=colors,
        edgecolor=cfg.colors["gray"],
        linewidth=0.8,
    )
    ax.set_ylabel("Remaining sequences")
    ax.set_xticks(xx)
    ax.set_xticklabels(plot_df["rule_label"], rotation=28, ha="right")
    beautify_axis(ax, cfg, "y")

    ypad = max(plot_df["n_remaining_after_cumulative_filter"].max() * 0.008, 1)
    for bar, val in zip(bars, plot_df["n_remaining_after_cumulative_filter"].values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + ypad,
            f"{int(val):,}",
            ha="center",
            va="bottom",
            fontsize=7,
            color=cfg.colors["black"],
        )

    panel_letter(ax, "a", cfg, dx=-0.08, dy=1.06)
    save_fig(fig, os.path.join(cfg.fig_supp_dir, "FigS1_filtering_flow"), cfg)
    plt.close(fig)


def fig_supp_source_shift_summary(shift_df: pd.DataFrame, cfg: Step1Config) -> None:
    features = [
        ("length", "Length"),
        ("net_charge_pH7", "Charge"),
        ("hydrophobicity_KD", "Hydrophobicity"),
        ("entropy", "Entropy"),
        ("max_runlen", "Max run length"),
        ("dominant_aa_frac", "Dominant amino acid fraction"),
    ]
    pair_labels = [f"{a} vs {b}" for a, b in combinations(cfg.source_order, 2)]

    mat = np.full((len(pair_labels), len(features)), np.nan)
    for i, (src_a, src_b) in enumerate(combinations(cfg.source_order, 2)):
        for j, (feat_raw, _) in enumerate(features):
            vals = shift_df.loc[
                (shift_df["source_a"] == src_a) &
                (shift_df["source_b"] == src_b) &
                (shift_df["feature"] == feat_raw),
                "cliffs_delta"
            ].to_numpy()
            if len(vals):
                mat[i, j] = vals[0]

    fig, ax = plt.subplots(1, 1, figsize=(8.7, 4.1))
    fig.subplots_adjust(left=0.22, right=0.93, top=0.92, bottom=0.24)

    im = ax.imshow(mat, aspect="auto", vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(np.arange(len(features)))
    ax.set_xticklabels([x[1] for x in features], rotation=cfg.heatmap_label_rot, ha="right")
    ax.set_yticks(np.arange(len(pair_labels)))
    ax.set_yticklabels(pair_labels)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if np.isfinite(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=cfg.heatmap_fontsize)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("Cliff's δ", fontsize=8)
    panel_letter(ax, "a", cfg, dx=-0.10, dy=1.08)

    save_fig(fig, os.path.join(cfg.fig_supp_dir, "FigS2_pairwise_source_shift_summary"), cfg)
    plt.close(fig)


def fig_supp_integrity_and_distribution_diagnostics(df: pd.DataFrame, cfg: Step1Config) -> None:
    df_main = df.loc[df["include_final_step1"]].copy()

    fig, axes = plt.subplots(2, 2, figsize=(8.8, 6.0))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.94, bottom=0.12, wspace=0.30, hspace=0.34)

    plot_ecdf_overlay(axes[0, 0], df_main, "length", "Length (aa)", cfg)
    axes[0, 0].legend(frameon=False, fontsize=7, loc="lower right")
    panel_letter(axes[0, 0], "a", cfg)

    plot_ecdf_overlay(axes[0, 1], df_main, "net_charge_pH7", "Net charge (pH 7)", cfg)
    panel_letter(axes[0, 1], "b", cfg)

    plot_ecdf_overlay(axes[1, 0], df_main, "hydrophobicity_KD", "Hydrophobicity (Kyte–Doolittle)", cfg)
    panel_letter(axes[1, 0], "c", cfg)

    plot_ecdf_overlay(axes[1, 1], df_main, "max_runlen", "Max run length", cfg)
    panel_letter(axes[1, 1], "d", cfg)

    save_fig(fig, os.path.join(cfg.fig_supp_dir, "FigS3_integrity_and_distribution_diagnostics"), cfg)
    plt.close(fig)


# ============================================================
# REPORTING
# ============================================================

def build_integrity_summary(df: pd.DataFrame, cfg: Step1Config) -> Dict[str, Any]:
    return {
        "n_rows_total": int(len(df)),
        "n_rows_retained_final": int(df["include_final_step1"].sum()),
        "counts_by_source": {src: int((df["source_db4"] == src).sum()) for src in cfg.source_order},
        "retained_by_source": {src: int(((df["source_db4"] == src) & df["include_final_step1"]).sum()) for src in cfg.source_order},
        "sequence_missing_fraction": float(df["sequence_missing"].mean()),
        "length_missing_fraction": float(df["length_missing"].mean()),
        "charge_missing_fraction": float(df["charge_missing"].mean()),
        "hydro_missing_fraction": float(df["hydro_missing"].mean()),
        "canonical_fraction": float(df["is_canonical"].mean()),
        "noncanonical_fraction": float((~df["is_canonical"]).mean()),
        "length_mismatch_fraction": float(df["len_mismatch"].mean()),
        "length_noninteger_fraction": float(df["length_noninteger_flag"].mean()),
        "duplicate_fraction_global": float(df["is_exact_duplicate_global"].mean()),
        "duplicate_fraction_within_source": float(df["is_exact_duplicate_within_source"].mean()),
        "cross_source_duplicate_fraction": float(df["is_cross_source_duplicate"].mean()),
    }


def build_run_report(cfg: Step1Config, source_report: Dict[str, Any], integrity: Dict[str, Any]) -> str:
    lines = []
    lines.append("PepCVAE — Step 1 run report")
    lines.append("=" * 72)
    lines.append(f"Timestamp: {datetime.now().isoformat()}")
    lines.append(f"Input CSV: {cfg.input_csv}")
    lines.append("")
    lines.append("Purpose:")
    lines.append("  Establish a reproducible, source-traceable peptide corpus audit,")
    lines.append("  quantify source heterogeneity, export publication-grade Step 1")
    lines.append("  figures, and define a retained corpus for downstream PepCVAE modeling.")
    lines.append("")
    lines.append("Source validation:")
    lines.append(f"  Allowed sources: {source_report['allowed_sources']}")
    lines.append(f"  Found sources:   {source_report['found_sources']}")
    if source_report["missing_expected_sources"]:
        lines.append(f"  Missing expected sources: {source_report['missing_expected_sources']}")
    if source_report["unexpected_sources"]:
        lines.append(f"  Unexpected sources: {source_report['unexpected_sources']}")
    lines.append("")
    lines.append("Integrity summary:")
    for k, v in integrity.items():
        lines.append(f"  - {k}: {v}")
    lines.append("")
    lines.append("Outputs:")
    lines.append(f"  tables: {cfg.tables_dir}")
    lines.append(f"  reports: {cfg.reports_dir}")
    lines.append(f"  figures_main: {cfg.fig_main_dir}")
    lines.append(f"  figures_supplementary: {cfg.fig_supp_dir}")
    lines.append("")
    lines.append("Interpretive note:")
    lines.append("  Step 1 defines dataset composition, sequence-quality structure,")
    lines.append("  and retained-corpus properties that motivate downstream leakage-aware")
    lines.append("  splitting, conditional modeling, and novelty auditing.")
    return "\n".join(lines)


# ============================================================
# MAIN EXECUTION
# ============================================================

def run_step1(cfg: Step1Config) -> None:
    ensure_output_tree(cfg)
    set_plot_style(cfg)

    start_manifest = {
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python_env_note": "Executed with publication-oriented Step 1 audit script",
        "input_csv": cfg.input_csv,
        "input_sha256": sha256_file(cfg.input_csv),
        "config": asdict(cfg),
        "scipy_available": HAS_SCIPY,
    }
    save_json(start_manifest, os.path.join(cfg.reports_dir, "step1_manifest.json"))

    raw_df = pd.read_csv(cfg.input_csv, low_memory=False)

    df = prepare_dataframe(raw_df, cfg)
    source_report = validate_sources(df, cfg)
    df = add_qc_columns(df, cfg)

    source_summary_df = build_source_summary_table(df, cfg)
    missingness_df = build_missingness_table(df)
    missingness_by_source_df = build_missingness_by_source_table(df, cfg)
    filtering_flow_df = build_filtering_flow_table(df)
    shift_df = build_domain_shift_table_pairwise(df, cfg)

    qc_cols = [
        "row_id_step1",
        "sequence_raw",
        "sequence",
        "sequence_sha256",
        "source_db4",
        "length",
        "length_rounded_int",
        "len_from_seq",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "sequence_missing",
        "length_missing",
        "charge_missing",
        "hydro_missing",
        "length_noninteger_flag",
        "len_mismatch",
        "is_canonical",
        "entropy",
        "max_runlen",
        "dominant_aa_frac",
        "is_exact_duplicate_global",
        "is_exact_duplicate_within_source",
        "n_sources_per_sequence",
        "is_cross_source_duplicate",
        "source_membership_pattern",
        "exclude_sequence_missing",
        "exclude_noncanonical",
        "exclude_length_missing",
        "exclude_length_mismatch",
        "exclude_charge_missing",
        "exclude_hydro_missing",
        "exclude_length_out_of_range",
        "exclude_any_step1",
        "include_final_step1",
    ]
    if "label_acp" in df.columns:
        qc_cols.insert(5, "label_acp")

    save_csv(source_summary_df, os.path.join(cfg.tables_dir, "step1_source_summary.csv"))
    save_csv(missingness_df, os.path.join(cfg.tables_dir, "step1_missingness_table.csv"))
    save_csv(missingness_by_source_df, os.path.join(cfg.tables_dir, "step1_missingness_by_source.csv"))
    save_csv(filtering_flow_df, os.path.join(cfg.tables_dir, "step1_filtering_flow.csv"))
    save_csv(shift_df, os.path.join(cfg.tables_dir, "step1_domain_shift_pairwise.csv"))
    save_csv(df[qc_cols], os.path.join(cfg.tables_dir, "step1_qc_table.csv"))
    save_csv(df, os.path.join(cfg.tables_dir, "step1_audited_dataset.csv"))
    save_csv(df.loc[df["include_final_step1"]].copy(), os.path.join(cfg.tables_dir, "step1_retained_dataset.csv"))

    fig_main_corpus_heterogeneity(df, cfg)
    fig_supp_filtering_flow(filtering_flow_df, cfg)
    fig_supp_source_shift_summary(shift_df, cfg)
    fig_supp_integrity_and_distribution_diagnostics(df, cfg)

    integrity = build_integrity_summary(df, cfg)
    audit_summary = {
        "timestamp": datetime.now().isoformat(),
        "source_validation": source_report,
        "integrity_summary": integrity,
        "notes": [
            "Main figure emphasizes retained-corpus heterogeneity across key peptide descriptors.",
            "Filtering summary is reported in Supplementary to keep the main figure focused on biological and physicochemical diversity.",
            "Pairwise effect sizes are emphasized over a single arbitrary reference source.",
            "Sequence-complexity fields exported here are also useful for later low-complexity and failure-mode analyses.",
        ],
    }
    save_json(audit_summary, os.path.join(cfg.reports_dir, "step1_audit_summary.json"))

    run_report = build_run_report(cfg, source_report, integrity)
    save_text(run_report, os.path.join(cfg.reports_dir, "step1_run_report.txt"))

    print("✅ Step 1 completed.")
    print("Outputs:")
    print("  tables:", cfg.tables_dir)
    print("  reports:", cfg.reports_dir)
    print("  figures_main:", cfg.fig_main_dir)
    print("  figures_supplementary:", cfg.fig_supp_dir)
    print("  retained dataset:", os.path.join(cfg.tables_dir, "step1_retained_dataset.csv"))
    print("")
    print("Ready to write the Methodology and Results for Step 1 and proceed to split design.")


# ============================================================
# NOTEBOOK / CLI ENTRY
# ============================================================

def main_notebook(
    input_csv: str,
    out_dir_name: str = "nature_computational/step1",
    min_len: int = 5,
    max_len: int = 60,
) -> None:
    cfg = Step1Config(
        input_csv=input_csv,
        out_dir_name=out_dir_name,
        min_len=min_len,
        max_len=max_len,
    )
    run_step1(cfg)


def parse_args():
    parser = argparse.ArgumentParser(description="PepCVAE Step 1 corpus audit")
    parser.add_argument("--input_csv", type=str, required=True, help="Path to merged peptide CSV")
    parser.add_argument("--out_dir_name", type=str, default="nature_computational/step1")
    parser.add_argument("--min_len", type=int, default=5)
    parser.add_argument("--max_len", type=int, default=60)
    return parser.parse_args()



In [ ]:
main_notebook(
    input_csv="/home/data3/mohamed/peponco/clean dataset/OncoPep_merged_clean_labeled4.csv",
    out_dir_name="nature_computational2/step1",
    min_len=5,
    max_len=60,
)

step2

In [ ]:
# ============================================================
# PepCVAE / AngioPep-VAE — Step 2
# Leakage-aware split design and novelty-reference construction
#
# Full corrected Jupyter-ready version
# ============================================================

from __future__ import annotations

import os
import json
import math
import time
import hashlib
import platform
from dataclasses import dataclass, asdict, field
from datetime import datetime
from collections import defaultdict, Counter
from itertools import combinations
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Step2Config:
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step1/tables/step1_retained_dataset.csv"
    out_dir_name: str = "step2"
    project_root_override: Optional[str] = None

    train_frac: float = 0.80
    val_frac: float = 0.10
    test_frac: float = 0.10

    kmer_k: int = 3
    minhash_perm: int = 64
    lsh_bands: int = 16
    main_jaccard_threshold: float = 0.55
    threshold_sensitivity: Tuple[float, ...] = (0.45, 0.55, 0.65)

    min_shared_kmers_to_compare: int = 2
    max_bucket_size: int = 250
    max_pairs_per_bucket: int = 4000

    hash_seed: int = 42
    split_seed: int = 42
    sampled_cross_split_pairs: int = 3000
    nn_candidate_cap: int = 3000
    nn_random_fallback: int = 256

    sequence_col: str = "sequence"
    hash_col: str = "sequence_sha256"
    source_col: str = "source_db4"

    descriptor_cols: Tuple[str, ...] = (
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "entropy",
        "dominant_aa_frac",
    )

    include_source_distribution_in_assignment: bool = True
    include_descriptor_balance_in_assignment: bool = True
    descriptor_balance_weight: float = 0.10
    source_balance_weight: float = 0.20
    overshoot_penalty_weight: float = 4.0

    min_allowed_split_fraction: float = 0.05
    max_allowed_split_fraction: float = 0.90

    save_cluster_members: bool = True
    save_similarity_edges: bool = True

    png_dpi: int = 300
    tif_dpi: int = 600
    font_family: str = "sans-serif"
    font_list: Tuple[str, ...] = ("Arial", "Helvetica", "DejaVu Sans")
    font_size: int = 8
    title_size: int = 9
    axis_label_size: int = 8
    tick_label_size: int = 8
    axis_linewidth: float = 0.9

    colors: Dict[str, str] = field(default_factory=lambda: {
        "train": "#4E79A7",
        "val": "#F28E2B",
        "test": "#E15759",
        "gray": "#707070",
        "black": "#111111",
        "grid": "#E6E6E6",
        "green": "#59A14F",
        "purple": "#B07AA1",
        "blue2": "#76B7B2",
    })

    @property
    def project_root(self) -> str:
        if self.project_root_override is not None:
            return os.path.abspath(self.project_root_override)
        return os.path.dirname(
            os.path.dirname(
                os.path.dirname(os.path.abspath(self.input_csv))
            )
        )

    @property
    def out_dir(self) -> str:
        return os.path.join(self.project_root, self.out_dir_name)

    @property
    def splits_dir(self) -> str:
        return os.path.join(self.out_dir, "splits")

    @property
    def tables_dir(self) -> str:
        return os.path.join(self.out_dir, "tables")

    @property
    def reports_dir(self) -> str:
        return os.path.join(self.out_dir, "reports")

    @property
    def fig_main_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_main")

    @property
    def fig_supp_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_supplementary")


# ============================================================
# GENERAL HELPERS
# ============================================================

def ensure_dir(path: str) -> str:
    path = os.path.abspath(path)
    os.makedirs(path, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step2Config) -> None:
    for path in [
        cfg.out_dir,
        cfg.splits_dir,
        cfg.tables_dir,
        cfg.reports_dir,
        cfg.fig_main_dir,
        cfg.fig_supp_dir,
    ]:
        ensure_dir(path)


def save_csv(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)


def save_text(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def json_default(obj: Any):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def sha256_string(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path: str, chunk_size: int = 2**20) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


def finite_array(values) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    seq = "".join(seq.split())
    return seq


def median_or_nan(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().any():
        return float(x.median())
    return np.nan


def first_non_null(x: pd.Series):
    x = x.dropna()
    return x.iloc[0] if len(x) else np.nan


def safe_frac(numer: float, denom: float) -> float:
    if denom == 0:
        return 0.0
    return float(numer) / float(denom)


def complete_ordered_table(
    df: pd.DataFrame,
    key_col: str,
    ordered_keys: Sequence[str],
    agg_dict: Optional[Dict[str, Any]] = None,
    fill_value: Any = 0,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame({key_col: list(ordered_keys)})

    work = df.copy()

    if agg_dict is None:
        agg_dict = {}
        for col in work.columns:
            if col == key_col:
                continue
            if pd.api.types.is_numeric_dtype(work[col]):
                agg_dict[col] = "sum"
            else:
                agg_dict[col] = "first"

    grouped = work.groupby(key_col, as_index=False).agg(agg_dict)
    order_df = pd.DataFrame({key_col: list(ordered_keys)})
    out = order_df.merge(grouped, on=key_col, how="left")

    for col in out.columns:
        if col == key_col:
            continue
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(fill_value)

    return out


# ============================================================
# PLOT HELPERS
# ============================================================

def set_plot_style(cfg: Step2Config) -> None:
    plt.rcParams.update({
        "font.family": cfg.font_family,
        "font.sans-serif": list(cfg.font_list),
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "axes.linewidth": cfg.axis_linewidth,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
    })


def beautify_axis(ax, cfg: Step2Config, grid_axis: str = "y") -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis=grid_axis, color=cfg.colors["grid"], linewidth=0.8)
    ax.set_axisbelow(True)


def panel_letter(ax, cfg: Step2Config, letter: str) -> None:
    ax.text(
        -0.14, 1.06, letter,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        va="top",
        ha="left",
    )


def save_fig(fig, stem_path: str, cfg: Step2Config) -> None:
    fig.savefig(f"{stem_path}.png", dpi=cfg.png_dpi, bbox_inches="tight")
    fig.savefig(f"{stem_path}.pdf", bbox_inches="tight")
    fig.savefig(f"{stem_path}.tiff", dpi=cfg.tif_dpi, bbox_inches="tight")


def _safe_boxplot(ax, data_list, positions, colors):
    bp = ax.boxplot(
        data_list,
        positions=positions,
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        whis=(5, 95),
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor("white")
        patch.set_edgecolor(color)
        patch.set_linewidth(1.0)
    for med in bp["medians"]:
        med.set_color("#111111")
        med.set_linewidth(1.1)
    for whisker in bp["whiskers"]:
        whisker.set_color("#666666")
        whisker.set_linewidth(0.9)
    for cap in bp["caps"]:
        cap.set_color("#666666")
        cap.set_linewidth(0.9)


def plot_violin_box(
    ax,
    df: pd.DataFrame,
    value_col: str,
    ylabel: str,
    cfg: Step2Config,
    split_order=("train", "val", "test"),
) -> None:
    positions = np.arange(len(split_order))
    raw_data = []
    colors = []

    for split in split_order:
        vals = finite_array(df.loc[df["assigned_split"] == split, value_col].values)
        raw_data.append(vals if len(vals) else np.array([]))
        colors.append(cfg.colors[split])

    has_enough = sum(len(v) > 1 for v in raw_data) >= 1

    if has_enough:
        vp = ax.violinplot(
            raw_data,
            positions=positions,
            widths=0.75,
            showmeans=False,
            showmedians=True,
            showextrema=False,
        )
        for i, body in enumerate(vp["bodies"]):
            body.set_facecolor(colors[i])
            body.set_alpha(0.35)
            body.set_edgecolor(colors[i])
            body.set_linewidth(0.8)
        if "cmedians" in vp:
            vp["cmedians"].set_color(cfg.colors["black"])
            vp["cmedians"].set_linewidth(1.2)
    else:
        rng = np.random.default_rng(123)
        for pos, vals, split in zip(positions, raw_data, split_order):
            vals = finite_array(vals)
            if len(vals):
                jitter = rng.normal(0, 0.03, size=len(vals))
                ax.scatter(
                    np.full(len(vals), pos) + jitter,
                    vals,
                    s=10,
                    alpha=0.6,
                    color=cfg.colors[split],
                    edgecolors="none",
                    zorder=2,
                )

    _safe_boxplot(ax, raw_data, positions, colors)
    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(split_order)
    beautify_axis(ax, cfg, "y")


# ============================================================
# DATA LOADING / VALIDATION
# ============================================================

def validate_config(cfg: Step2Config) -> None:
    fracs = [cfg.train_frac, cfg.val_frac, cfg.test_frac]
    if any(f <= 0 for f in fracs):
        raise ValueError("All split fractions must be > 0.")
    if not np.isclose(sum(fracs), 1.0, atol=1e-8):
        raise ValueError(f"Split fractions must sum to 1.0, got {sum(fracs):.6f}.")
    if cfg.kmer_k < 1:
        raise ValueError("kmer_k must be >= 1.")
    if cfg.minhash_perm < 8:
        raise ValueError("minhash_perm must be >= 8.")
    if cfg.lsh_bands < 1:
        raise ValueError("lsh_bands must be >= 1.")
    if cfg.minhash_perm % cfg.lsh_bands != 0:
        raise ValueError("minhash_perm must be divisible by lsh_bands.")
    if not (0 < cfg.main_jaccard_threshold <= 1):
        raise ValueError("main_jaccard_threshold must be in (0, 1].")
    for t in cfg.threshold_sensitivity:
        if not (0 < t <= 1):
            raise ValueError("Each threshold_sensitivity value must be in (0, 1].")
    if cfg.min_allowed_split_fraction <= 0 or cfg.max_allowed_split_fraction >= 1:
        raise ValueError("Split fraction guardrails must lie inside (0, 1).")
    if cfg.min_allowed_split_fraction >= cfg.max_allowed_split_fraction:
        raise ValueError("min_allowed_split_fraction must be < max_allowed_split_fraction.")


def load_step1_retained_dataset(cfg: Step2Config) -> pd.DataFrame:
    if not os.path.exists(cfg.input_csv):
        raise FileNotFoundError(f"Input CSV not found: {cfg.input_csv}")

    df = pd.read_csv(cfg.input_csv)
    if len(df) == 0:
        raise ValueError("Input CSV is empty.")

    required = [cfg.sequence_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}")

    df = df.copy()
    df[cfg.sequence_col] = df[cfg.sequence_col].map(normalize_sequence)
    df = df.loc[df[cfg.sequence_col] != ""].copy()

    if cfg.hash_col not in df.columns:
        df[cfg.hash_col] = df[cfg.sequence_col].map(sha256_string)
    else:
        df[cfg.hash_col] = df[cfg.hash_col].astype(str)
        needs_fill = df[cfg.hash_col].isna() | (df[cfg.hash_col].str.strip() == "")
        if needs_fill.any():
            df.loc[needs_fill, cfg.hash_col] = df.loc[needs_fill, cfg.sequence_col].map(sha256_string)

    if cfg.source_col not in df.columns:
        df[cfg.source_col] = "unknown"

    for col in cfg.descriptor_cols:
        if col not in df.columns:
            df[col] = np.nan

    return df.reset_index(drop=True)


# ============================================================
# UNIQUE-SEQUENCE COLLAPSE
# ============================================================

def collapse_to_unique_sequences(df: pd.DataFrame, cfg: Step2Config) -> pd.DataFrame:
    work = df.copy()

    grouped = work.groupby(cfg.sequence_col, sort=False)

    out = grouped.agg(
        n_rows=(cfg.sequence_col, "size"),
        sequence_sha256=(cfg.hash_col, first_non_null),
        source_primary=(cfg.source_col, first_non_null),
    ).reset_index()

    source_lists = (
        grouped[cfg.source_col]
        .apply(lambda x: sorted(set(str(v) for v in x.dropna())))
        .reset_index(name="source_list")
    )
    out = out.merge(source_lists, on=cfg.sequence_col, how="left")
    out["n_sources"] = out["source_list"].map(len)

    for col in cfg.descriptor_cols:
        tmp = grouped[col].apply(median_or_nan).reset_index(name=col)
        out = out.merge(tmp, on=cfg.sequence_col, how="left")

    if "length" not in out.columns or out["length"].isna().all():
        out["length"] = out[cfg.sequence_col].str.len()
    else:
        miss_len = out["length"].isna()
        out.loc[miss_len, "length"] = out.loc[miss_len, cfg.sequence_col].str.len()

    miss_hash = out["sequence_sha256"].isna() | (out["sequence_sha256"].astype(str).str.strip() == "")
    out.loc[miss_hash, "sequence_sha256"] = out.loc[miss_hash, cfg.sequence_col].map(sha256_string)

    if out[cfg.sequence_col].duplicated().any():
        dup = out.loc[out[cfg.sequence_col].duplicated(), cfg.sequence_col].head(5).tolist()
        raise ValueError(f"Unique-sequence collapse failed; duplicate sequences remain. Examples: {dup}")
    if out["sequence_sha256"].duplicated().any():
        dup = out.loc[out["sequence_sha256"].duplicated(), "sequence_sha256"].head(5).tolist()
        raise ValueError(f"Hash uniqueness failed after collapse. Example duplicate hashes: {dup}")

    return out.reset_index(drop=True)


# ============================================================
# K-MERS / MINHASH / LSH
# ============================================================

MERSENNE_PRIME_61 = (1 << 61) - 1


def make_kmers(seq: str, k: int) -> frozenset:
    seq = normalize_sequence(seq)
    if len(seq) == 0:
        return frozenset()
    if len(seq) < k:
        return frozenset([seq])
    return frozenset(seq[i:i + k] for i in range(len(seq) - k + 1))


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(h[:8], "big", signed=False)


def get_minhash_params(num_perm: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    a = rng.integers(1, MERSENNE_PRIME_61 - 1, size=num_perm, dtype=np.int64)
    b = rng.integers(0, MERSENNE_PRIME_61 - 1, size=num_perm, dtype=np.int64)
    return a, b


def minhash_signature(kmers: frozenset, a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(kmers) == 0:
        return np.full(len(a), MERSENNE_PRIME_61 - 1, dtype=np.uint64)

    token_hashes = np.array([stable_int_hash(tok) % MERSENNE_PRIME_61 for tok in kmers], dtype=np.uint64)
    vals = (token_hashes[:, None] * a.astype(np.uint64)[None, :] + b.astype(np.uint64)[None, :]) % np.uint64(MERSENNE_PRIME_61)
    return vals.min(axis=0).astype(np.uint64)

def build_sequence_features(seq_df: pd.DataFrame, cfg: Step2Config) -> pd.DataFrame:
    a, b = get_minhash_params(cfg.minhash_perm, cfg.hash_seed)

    work = seq_df.copy()
    kmers_list = []
    sigs = []

    for seq in work[cfg.sequence_col].tolist():
        km = make_kmers(seq, cfg.kmer_k)
        kmers_list.append(km)
        sigs.append(minhash_signature(km, a, b))

    work["kmers"] = kmers_list
    work["minhash_sig"] = sigs
    return work


def lsh_candidate_pairs(seq_df: pd.DataFrame, cfg: Step2Config) -> pd.DataFrame:
    rows_per_band = cfg.minhash_perm // cfg.lsh_bands
    buckets = defaultdict(list)

    for idx, sig in enumerate(seq_df["minhash_sig"].values):
        for band in range(cfg.lsh_bands):
            start = band * rows_per_band
            end = (band + 1) * rows_per_band
            band_tuple = tuple(int(x) for x in sig[start:end])
            buckets[(band, band_tuple)].append(idx)

    candidate_pairs = set()

    for members in buckets.values():
        if len(members) < 2:
            continue
        if len(members) > cfg.max_bucket_size:
            continue

        n_pairs = len(members) * (len(members) - 1) // 2
        if n_pairs > cfg.max_pairs_per_bucket:
            continue

        for i, j in combinations(members, 2):
            if i > j:
                i, j = j, i
            candidate_pairs.add((i, j))

    records = []
    seqs = seq_df[cfg.sequence_col].tolist()
    hashes = seq_df["sequence_sha256"].tolist()
    kmers = seq_df["kmers"].tolist()

    for i, j in sorted(candidate_pairs):
        shared = len(kmers[i].intersection(kmers[j]))
        if shared < cfg.min_shared_kmers_to_compare:
            continue
        records.append({
            "idx_a": i,
            "idx_b": j,
            "sequence_a": seqs[i],
            "sequence_b": seqs[j],
            "hash_a": hashes[i],
            "hash_b": hashes[j],
            "shared_kmers": shared,
        })

    return pd.DataFrame(records)


def jaccard_similarity(a: frozenset, b: frozenset) -> float:
    if len(a) == 0 and len(b) == 0:
        return 1.0
    union_n = len(a.union(b))
    if union_n == 0:
        return 0.0
    return len(a.intersection(b)) / union_n


def compute_similarity_edges(
    seq_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    threshold: float,
) -> pd.DataFrame:
    if len(candidate_df) == 0:
        return pd.DataFrame(columns=[
            "idx_a", "idx_b",
            "sequence_a", "sequence_b",
            "hash_a", "hash_b",
            "shared_kmers", "jaccard"
        ])

    if "jaccard" in candidate_df.columns:
        out = candidate_df.loc[candidate_df["jaccard"] >= threshold].copy()
        return out.reset_index(drop=True)

    kmers = seq_df["kmers"].tolist()
    records = []

    for row in candidate_df.itertuples(index=False):
        jac = jaccard_similarity(kmers[row.idx_a], kmers[row.idx_b])
        if jac >= threshold:
            records.append({
                "idx_a": row.idx_a,
                "idx_b": row.idx_b,
                "sequence_a": row.sequence_a,
                "sequence_b": row.sequence_b,
                "hash_a": row.hash_a,
                "hash_b": row.hash_b,
                "shared_kmers": row.shared_kmers,
                "jaccard": float(jac),
            })

    return pd.DataFrame(records)

def build_similarity_clusters(
    seq_df: pd.DataFrame,
    edge_df: pd.DataFrame,
    cfg: Step2Config,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    uf = UnionFind(len(seq_df))

    for row in edge_df.itertuples(index=False):
        uf.union(int(row.idx_a), int(row.idx_b))

    roots = [uf.find(i) for i in range(len(seq_df))]
    root_to_cluster = {}
    cluster_ids = []
    next_id = 0
    for r in roots:
        if r not in root_to_cluster:
            root_to_cluster[r] = next_id
            next_id += 1
        cluster_ids.append(root_to_cluster[r])

    seq_cluster_df = seq_df.copy()
    seq_cluster_df["cluster_id"] = cluster_ids

    cluster_summary = (
        seq_cluster_df.groupby("cluster_id", as_index=False)
        .agg(
            cluster_size_sequences=(cfg.sequence_col, "size"),
            cluster_size_rows=("n_rows", "sum"),
            median_length=("length", median_or_nan),
            median_charge=("net_charge_pH7", median_or_nan),
            median_hydrophobicity=("hydrophobicity_KD", median_or_nan),
            median_entropy=("entropy", median_or_nan),
            median_dominant_aa_frac=("dominant_aa_frac", median_or_nan),
        )
    )

    cluster_source_records = []
    for cid, sub in seq_cluster_df.groupby("cluster_id"):
        flat_sources = []
        for lst in sub["source_list"]:
            if isinstance(lst, list):
                flat_sources.extend(lst)
        counts = Counter(flat_sources)
        total = sum(counts.values())
        source_profile = {k: safe_frac(v, total) for k, v in counts.items()} if total > 0 else {}
        cluster_source_records.append({
            "cluster_id": cid,
            "source_profile_json": json.dumps(source_profile, sort_keys=True),
        })

    cluster_source_df = pd.DataFrame(cluster_source_records)
    cluster_summary = cluster_summary.merge(cluster_source_df, on="cluster_id", how="left")

    if cluster_summary["cluster_id"].duplicated().any():
        raise ValueError("cluster_summary has duplicate cluster_id values; this should never happen.")

    return seq_cluster_df, cluster_summary



# ============================================================
# UNION-FIND / CONNECTED COMPONENTS
# ============================================================

class UnionFind:
    """
    Disjoint-set union with path compression and union by rank.
    Used to convert retained similarity edges into connected components
    so that similarity-linked peptides are assigned as a unit.
    """
    def __init__(self, n: int):
        if n < 0:
            raise ValueError("UnionFind size must be non-negative.")
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        if x < 0 or x >= len(self.parent):
            raise IndexError(f"UnionFind index out of range: {x}")
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1

# ============================================================
# SPLIT ASSIGNMENT
# ============================================================

def target_sizes(n_total: int, cfg: Step2Config) -> Dict[str, int]:
    raw = {
        "train": n_total * cfg.train_frac,
        "val": n_total * cfg.val_frac,
        "test": n_total * cfg.test_frac,
    }

    floor_vals = {k: int(math.floor(v)) for k, v in raw.items()}
    remainder = n_total - sum(floor_vals.values())

    order = sorted(raw.keys(), key=lambda k: raw[k] - floor_vals[k], reverse=True)
    for k in order[:remainder]:
        floor_vals[k] += 1

    return floor_vals


def source_profile_from_json(s: str) -> Dict[str, float]:
    if pd.isna(s) or str(s).strip() == "":
        return {}
    try:
        return dict(json.loads(s))
    except Exception:
        return {}


def descriptor_vector(row: pd.Series) -> np.ndarray:
    cols = [
        "median_length",
        "median_charge",
        "median_hydrophobicity",
        "median_entropy",
        "median_dominant_aa_frac",
    ]
    vals = []
    for c in cols:
        v = row.get(c, np.nan)
        vals.append(float(v) if pd.notna(v) else np.nan)
    return np.asarray(vals, dtype=float)


def weighted_l1_source_distance(profile_a: Dict[str, float], profile_b: Dict[str, float]) -> float:
    keys = set(profile_a.keys()).union(profile_b.keys())
    return float(sum(abs(profile_a.get(k, 0.0) - profile_b.get(k, 0.0)) for k in keys))


def descriptor_distance(a: np.ndarray, b: np.ndarray) -> float:
    mask = np.isfinite(a) & np.isfinite(b)
    if not mask.any():
        return 0.0
    return float(np.mean(np.abs(a[mask] - b[mask])))


def assign_clusters_to_splits(cluster_df: pd.DataFrame, cfg: Step2Config) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.split_seed)

    work = cluster_df.copy()
    work["_tie"] = rng.random(len(work))
    work = work.sort_values(
        ["cluster_size_sequences", "cluster_size_rows", "_tie"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    total_sequences = int(work["cluster_size_sequences"].sum())
    split_targets = target_sizes(total_sequences, cfg)

    global_source_counter = Counter()
    for s in work["source_profile_json"]:
        prof = source_profile_from_json(s)
        for k, v in prof.items():
            global_source_counter[k] += v
    total_source_weight = sum(global_source_counter.values())
    global_source_profile = {
        k: safe_frac(v, total_source_weight) for k, v in global_source_counter.items()
    } if total_source_weight > 0 else {}

    global_desc = np.nanmedian(
        np.vstack([descriptor_vector(r) for _, r in work.iterrows()]),
        axis=0
    )

    assigned = []
    split_sizes = {"train": 0, "val": 0, "test": 0}
    split_source_sums = {"train": Counter(), "val": Counter(), "test": Counter()}
    split_desc_rows = {"train": [], "val": [], "test": []}

    for row in work.itertuples(index=False):
        row_size = int(row.cluster_size_sequences)
        row_source = source_profile_from_json(row.source_profile_json)
        row_desc = descriptor_vector(pd.Series(row._asdict()))

        scores = {}

        for split in ["train", "val", "test"]:
            current_size = split_sizes[split]
            proposed_size = current_size + row_size
            target = split_targets[split]

            base_error = abs(proposed_size - target)
            overshoot = max(0, proposed_size - target)
            overshoot_penalty = cfg.overshoot_penalty_weight * overshoot
            score = base_error + overshoot_penalty

            if cfg.include_source_distribution_in_assignment:
                tmp_counter = split_source_sums[split].copy()
                for k, v in row_source.items():
                    tmp_counter[k] += v
                tmp_total = sum(tmp_counter.values())
                tmp_profile = {k: safe_frac(v, tmp_total) for k, v in tmp_counter.items()} if tmp_total > 0 else {}
                score += cfg.source_balance_weight * weighted_l1_source_distance(tmp_profile, global_source_profile)

            if cfg.include_descriptor_balance_in_assignment:
                existing = split_desc_rows[split]
                if len(existing) == 0:
                    tmp_desc = row_desc.copy()
                else:
                    tmp_desc = np.nanmedian(np.vstack(existing + [row_desc]), axis=0)
                score += cfg.descriptor_balance_weight * descriptor_distance(tmp_desc, global_desc)

            scores[split] = score

        best_split = min(scores, key=scores.get)
        assigned.append(best_split)
        split_sizes[best_split] += row_size

        if cfg.include_source_distribution_in_assignment:
            for k, v in row_source.items():
                split_source_sums[best_split][k] += v
        if cfg.include_descriptor_balance_in_assignment:
            split_desc_rows[best_split].append(row_desc)

    out = work.copy()
    out["assigned_split"] = assigned
    out = out.drop(columns=["_tie"])

    realized = (
        out.groupby("assigned_split", as_index=False)["cluster_size_sequences"]
        .sum()
        .rename(columns={"cluster_size_sequences": "n_sequences"})
    )
    realized["fraction"] = realized["n_sequences"] / total_sequences

    frac_map = dict(zip(realized["assigned_split"], realized["fraction"]))
    for split in ["train", "val", "test"]:
        frac = frac_map.get(split, 0.0)
        if frac < cfg.min_allowed_split_fraction:
            raise ValueError(
                f"Invalid split: '{split}' fraction is {frac:.4f}, below "
                f"min_allowed_split_fraction={cfg.min_allowed_split_fraction:.4f}."
            )
        if frac > cfg.max_allowed_split_fraction:
            raise ValueError(
                f"Invalid split: '{split}' fraction is {frac:.4f}, above "
                f"max_allowed_split_fraction={cfg.max_allowed_split_fraction:.4f}."
            )

    return out


def propagate_split_to_sequences(
    seq_cluster_df: pd.DataFrame,
    assigned_cluster_df: pd.DataFrame,
    cfg: Step2Config,
) -> pd.DataFrame:
    key_df = assigned_cluster_df[["cluster_id", "assigned_split"]].copy()
    if key_df["cluster_id"].duplicated().any():
        raise ValueError("assigned_cluster_df contains duplicate cluster_id values.")

    seq_split_df = seq_cluster_df.merge(key_df, on="cluster_id", how="left")

    if seq_split_df["assigned_split"].isna().any():
        n_missing = int(seq_split_df["assigned_split"].isna().sum())
        raise ValueError(f"{n_missing} sequences could not be assigned to a split.")

    return seq_split_df


def propagate_split_to_rows(
    row_df: pd.DataFrame,
    seq_split_df: pd.DataFrame,
    cfg: Step2Config,
) -> pd.DataFrame:
    key_cols = [cfg.sequence_col, "sequence_sha256", "cluster_id", "assigned_split"]
    key_df = seq_split_df[key_cols].copy()

    if key_df[cfg.sequence_col].duplicated().any():
        raise ValueError("seq_split_df must contain unique sequences before row propagation.")

    out = row_df.merge(key_df, on=cfg.sequence_col, how="left")

    if out["assigned_split"].isna().any():
        n_missing = int(out["assigned_split"].isna().sum())
        raise ValueError(f"{n_missing} row-level records could not be assigned to a split.")

    return out


# ============================================================
# AUDITS
# ============================================================

def exact_overlap_audit(seq_split_df: pd.DataFrame, cfg: Step2Config) -> pd.DataFrame:
    split_sets = {
        split: set(seq_split_df.loc[seq_split_df["assigned_split"] == split, cfg.sequence_col].tolist())
        for split in ["train", "val", "test"]
    }

    records = []
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = split_sets[a].intersection(split_sets[b])
        records.append({
            "audit_type": "exact_sequence_overlap",
            "split_a": a,
            "split_b": b,
            "n_overlap": int(len(overlap)),
        })
    return pd.DataFrame(records)


def cluster_overlap_audit(seq_split_df: pd.DataFrame) -> pd.DataFrame:
    split_cluster_sets = {
        split: set(seq_split_df.loc[seq_split_df["assigned_split"] == split, "cluster_id"].tolist())
        for split in ["train", "val", "test"]
    }

    records = []
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = split_cluster_sets[a].intersection(split_cluster_sets[b])
        records.append({
            "audit_type": "cluster_overlap",
            "split_a": a,
            "split_b": b,
            "n_overlap": int(len(overlap)),
        })
    return pd.DataFrame(records)


def sampled_cross_split_jaccard_audit(
    seq_split_df: pd.DataFrame,
    cfg: Step2Config,
) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.split_seed + 1)
    records = []

    split_order = [("train", "val"), ("train", "test"), ("val", "test")]
    split_groups = {
        split: seq_split_df.loc[seq_split_df["assigned_split"] == split, [cfg.sequence_col, "kmers"]].reset_index(drop=True)
        for split in ["train", "val", "test"]
    }

    for a, b in split_order:
        aa = split_groups[a]
        bb = split_groups[b]

        if len(aa) == 0 or len(bb) == 0:
            records.append({
                "audit_type": "sampled_cross_split_jaccard",
                "split_a": a,
                "split_b": b,
                "n_sampled_pairs": 0,
                "max_jaccard": np.nan,
                "mean_jaccard": np.nan,
                "threshold": cfg.main_jaccard_threshold,
            })
            continue

        n_possible = len(aa) * len(bb)
        n_samples = min(cfg.sampled_cross_split_pairs, n_possible)

        sampled = []
        if n_samples == n_possible and n_possible <= 200000:
            for i in range(len(aa)):
                for j in range(len(bb)):
                    sampled.append((i, j))
        else:
            seen = set()
            while len(sampled) < n_samples:
                i = int(rng.integers(0, len(aa)))
                j = int(rng.integers(0, len(bb)))
                if (i, j) not in seen:
                    seen.add((i, j))
                    sampled.append((i, j))

        jacs = []
        for i, j in sampled:
            jacs.append(jaccard_similarity(aa.loc[i, "kmers"], bb.loc[j, "kmers"]))

        records.append({
            "audit_type": "sampled_cross_split_jaccard",
            "split_a": a,
            "split_b": b,
            "n_sampled_pairs": int(len(jacs)),
            "max_jaccard": float(np.max(jacs)) if len(jacs) else np.nan,
            "mean_jaccard": float(np.mean(jacs)) if len(jacs) else np.nan,
            "threshold": cfg.main_jaccard_threshold,
        })

    return pd.DataFrame(records)


def nearest_neighbor_audit(
    seq_split_df: pd.DataFrame,
    cfg: Step2Config,
) -> pd.DataFrame:
    required_cols = [cfg.sequence_col, "kmers", "assigned_split", "minhash_sig"]
    missing = [c for c in required_cols if c not in seq_split_df.columns]
    if missing:
        raise ValueError(f"nearest_neighbor_audit missing required columns: {missing}")

    train_df = (
        seq_split_df.loc[
            seq_split_df["assigned_split"] == "train",
            [cfg.sequence_col, "kmers", "minhash_sig"]
        ]
        .reset_index(drop=True)
        .copy()
    )

    query_df = (
        seq_split_df.loc[
            seq_split_df["assigned_split"].isin(["val", "test"]),
            [cfg.sequence_col, "kmers", "assigned_split", "minhash_sig"]
        ]
        .reset_index(drop=True)
        .copy()
    )

    if len(train_df) == 0 or len(query_df) == 0:
        return pd.DataFrame(columns=[
            "query_sequence",
            "query_split",
            "nearest_train_sequence",
            "nearest_train_jaccard",
            "n_candidates_scanned",
        ])

    rows_per_band = cfg.minhash_perm // cfg.lsh_bands
    train_buckets = defaultdict(list)
    for idx, sig in enumerate(train_df["minhash_sig"].values):
        for band in range(cfg.lsh_bands):
            start = band * rows_per_band
            end = (band + 1) * rows_per_band
            band_tuple = tuple(int(x) for x in sig[start:end])
            train_buckets[(band, band_tuple)].append(idx)

    rng = np.random.default_rng(cfg.hash_seed + 17)
    train_seqs = train_df[cfg.sequence_col].tolist()
    train_kmers = train_df["kmers"].tolist()
    train_all_idx = np.arange(len(train_df))

    records = []

    for _, row in query_df.iterrows():
        query_seq = row[cfg.sequence_col]
        query_split = row["assigned_split"]
        qk = row["kmers"]
        qs = row["minhash_sig"]

        cand = set()
        for band in range(cfg.lsh_bands):
            start = band * rows_per_band
            end = (band + 1) * rows_per_band
            band_tuple = tuple(int(x) for x in qs[start:end])
            cand.update(train_buckets.get((band, band_tuple), []))

        if len(cand) < cfg.nn_random_fallback and len(train_all_idx) > 0:
            extra_n = min(cfg.nn_random_fallback, len(train_all_idx))
            cand.update(int(x) for x in rng.choice(train_all_idx, size=extra_n, replace=False).tolist())

        cand = list(cand)
        if len(cand) > cfg.nn_candidate_cap:
            cand = rng.choice(np.asarray(cand, dtype=np.int32), size=cfg.nn_candidate_cap, replace=False).tolist()

        best_j = -1.0
        best_seq = None
        for j in cand:
            jac = jaccard_similarity(qk, train_kmers[j])
            if jac > best_j:
                best_j = jac
                best_seq = train_seqs[j]

        records.append({
            "query_sequence": query_seq,
            "query_split": query_split,
            "nearest_train_sequence": best_seq,
            "nearest_train_jaccard": float(best_j) if best_j >= 0 else np.nan,
            "n_candidates_scanned": int(len(cand)),
        })

    return pd.DataFrame(records)

def threshold_sensitivity_analysis(
    seq_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    cfg: Step2Config,
) -> pd.DataFrame:
    records = []

    work_candidates = candidate_df.copy()
    if len(work_candidates) and "jaccard" not in work_candidates.columns:
        kmers = seq_df["kmers"].tolist()
        jaccs = []
        for row in work_candidates.itertuples(index=False):
            jaccs.append(jaccard_similarity(kmers[row.idx_a], kmers[row.idx_b]))
        work_candidates["jaccard"] = np.asarray(jaccs, dtype=float)

    for thr in cfg.threshold_sensitivity:
        edge_df = compute_similarity_edges(seq_df, work_candidates, threshold=thr)
        seq_cluster_df_thr, cluster_summary_thr = build_similarity_clusters(seq_df, edge_df, cfg)
        assigned_thr = assign_clusters_to_splits(cluster_summary_thr, cfg)
        seq_split_thr = propagate_split_to_sequences(seq_cluster_df_thr, assigned_thr, cfg)

        counts = (
            seq_split_thr.groupby("assigned_split", as_index=False)
            .agg(n_unique_sequences=(cfg.sequence_col, "size"),
                 n_clusters=("cluster_id", "nunique"))
        )

        count_map = {r["assigned_split"]: r for _, r in counts.iterrows()}
        ex = exact_overlap_audit(seq_split_thr, cfg)
        samp = sampled_cross_split_jaccard_audit(seq_split_thr, cfg)

        records.append({
            "threshold": thr,
            "n_edges": int(len(edge_df)),
            "n_clusters_total": int(cluster_summary_thr["cluster_id"].nunique()),
            "train_n_sequences": int(count_map.get("train", {}).get("n_unique_sequences", 0)),
            "val_n_sequences": int(count_map.get("val", {}).get("n_unique_sequences", 0)),
            "test_n_sequences": int(count_map.get("test", {}).get("n_unique_sequences", 0)),
            "train_n_clusters": int(count_map.get("train", {}).get("n_clusters", 0)),
            "val_n_clusters": int(count_map.get("val", {}).get("n_clusters", 0)),
            "test_n_clusters": int(count_map.get("test", {}).get("n_clusters", 0)),
            "max_exact_overlap": int(ex["n_overlap"].max()) if len(ex) else 0,
            "max_sampled_cross_split_jaccard": float(samp["max_jaccard"].max()) if len(samp) and samp["max_jaccard"].notna().any() else np.nan,
        })

    return pd.DataFrame(records)

def build_split_summary(
    row_split_df: pd.DataFrame,
    seq_split_df: pd.DataFrame,
    cluster_summary_df: pd.DataFrame,
    cfg: Step2Config,
) -> pd.DataFrame:
    row_counts = (
        row_split_df.groupby("assigned_split", as_index=False)
        .size()
        .rename(columns={"size": "n_rows"})
    )

    seq_counts = (
        seq_split_df.groupby("assigned_split", as_index=False)
        .agg(n_unique_sequences=(cfg.sequence_col, "size"))
    )

    cluster_counts = (
        cluster_summary_df.groupby("assigned_split", as_index=False)
        .agg(n_clusters=("cluster_id", "nunique"))
    )

    out = row_counts.merge(seq_counts, on="assigned_split", how="outer")
    out = out.merge(cluster_counts, on="assigned_split", how="outer")
    out = out.rename(columns={"assigned_split": "split"})

    out = complete_ordered_table(
        out,
        key_col="split",
        ordered_keys=["train", "val", "test"],
        agg_dict={
            "n_rows": "sum",
            "n_unique_sequences": "sum",
            "n_clusters": "sum",
        },
        fill_value=0,
    )

    for c in ["n_rows", "n_unique_sequences", "n_clusters"]:
        out[c] = out[c].fillna(0).astype(int)

    return out


def build_novelty_reference_sets(seq_split_df: pd.DataFrame, cfg: Step2Config) -> Dict[str, pd.DataFrame]:
    seq_meta = seq_split_df[[cfg.sequence_col, "sequence_sha256", "assigned_split"]].reset_index(drop=True).copy()

    train_ref = (
        seq_meta.loc[seq_meta["assigned_split"] == "train", [cfg.sequence_col, "sequence_sha256"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    full_ref = (
        seq_meta[[cfg.sequence_col, "sequence_sha256"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return {
        "train_only_reference": train_ref,
        "full_corpus_reference": full_ref,
    }


def system_report(cfg: Step2Config) -> Dict[str, Any]:
    return {
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "input_csv": os.path.abspath(cfg.input_csv),
        "input_sha256": sha256_file(cfg.input_csv),
    }


# ============================================================
# FIGURES
# ============================================================

def fig_main_step2_split_design(
    split_summary_df: pd.DataFrame,
    leakage_audit_df: pd.DataFrame,
    cluster_summary_df: pd.DataFrame,
    cfg: Step2Config,
) -> None:
    set_plot_style(cfg)
    split_order = ["train", "val", "test"]
    pair_order = ["train vs val", "train vs test", "val vs test"]

    split_summary = complete_ordered_table(
        split_summary_df,
        key_col="split",
        ordered_keys=split_order,
        agg_dict={
            "n_rows": "sum",
            "n_unique_sequences": "sum",
            "n_clusters": "sum",
        },
        fill_value=0,
    )

    exact_df = leakage_audit_df.loc[leakage_audit_df["audit_type"] == "exact_sequence_overlap"].copy()
    if len(exact_df):
        exact_df["pair"] = exact_df["split_a"].astype(str) + " vs " + exact_df["split_b"].astype(str)
        exact_df = complete_ordered_table(
            exact_df,
            key_col="pair",
            ordered_keys=pair_order,
            agg_dict={"n_overlap": "sum"},
            fill_value=0,
        )
    else:
        exact_df = pd.DataFrame({"pair": pair_order, "n_overlap": [0, 0, 0]})

    cluster_counts = (
        cluster_summary_df.groupby("assigned_split", as_index=False)["cluster_id"]
        .nunique()
        .rename(columns={"assigned_split": "split", "cluster_id": "n_clusters"})
    )
    cluster_counts = complete_ordered_table(
        cluster_counts,
        key_col="split",
        ordered_keys=split_order,
        agg_dict={"n_clusters": "sum"},
        fill_value=0,
    )

    cluster_median = (
        cluster_summary_df.groupby("assigned_split", as_index=False)["cluster_size_sequences"]
        .median()
        .rename(columns={"assigned_split": "split"})
    )
    cluster_median = complete_ordered_table(
        cluster_median,
        key_col="split",
        ordered_keys=split_order,
        agg_dict={"cluster_size_sequences": "median"},
        fill_value=0,
    )

    fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.0))
    fig.subplots_adjust(left=0.08, right=0.985, top=0.95, bottom=0.12, wspace=0.34, hspace=0.38)

    ax = axes[0, 0]
    vals = split_summary["n_unique_sequences"].to_numpy()
    bars = ax.bar(np.arange(len(split_order)), vals, color=[cfg.colors[s] for s in split_order],
                  edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylabel("Unique sequences")
    ax.set_xticks(np.arange(len(split_order)))
    ax.set_xticklabels(split_order)
    beautify_axis(ax, cfg, "y")
    ymax = max(vals) if len(vals) else 0
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + max(ymax * 0.01, 0.1), f"{int(v):,}",
                ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "a")

    ax = axes[0, 1]
    vals = cluster_counts["n_clusters"].to_numpy()
    bars = ax.bar(np.arange(len(split_order)), vals, color=[cfg.colors[s] for s in split_order],
                  edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylabel("Similarity clusters")
    ax.set_xticks(np.arange(len(split_order)))
    ax.set_xticklabels(split_order)
    beautify_axis(ax, cfg, "y")
    ymax = max(vals) if len(vals) else 0
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + max(ymax * 0.01, 0.1), f"{int(v):,}",
                ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "b")

    ax = axes[1, 0]
    vals = exact_df["n_overlap"].to_numpy()
    bars = ax.bar(np.arange(len(pair_order)), vals, color=cfg.colors["green"],
                  edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_xticks(np.arange(len(pair_order)))
    ax.set_xticklabels(pair_order, rotation=12, ha="right")
    ymax = max(1, np.max(vals)) if len(vals) else 1
    ax.set_ylim(0, ymax * 1.25)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + max(ymax * 0.03, 0.05), f"{int(v)}",
                ha="center", va="bottom", fontsize=8)
    ax.set_ylabel("Exact sequence overlap")
    beautify_axis(ax, cfg, "y")
    panel_letter(ax, cfg, "c")

    ax = axes[1, 1]
    vals = cluster_median["cluster_size_sequences"].to_numpy()
    bars = ax.bar(np.arange(len(split_order)), vals, color=[cfg.colors[s] for s in split_order],
                  edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylabel("Median cluster size")
    ax.set_xticks(np.arange(len(split_order)))
    ax.set_xticklabels(split_order)
    beautify_axis(ax, cfg, "y")
    ymax = max(vals) if len(vals) else 0
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + max(ymax * 0.03, 0.1), f"{float(v):.1f}",
                ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "d")

    save_fig(fig, os.path.join(cfg.fig_main_dir, "Fig2_leakage_aware_split_design_main"), cfg)
    plt.close(fig)


def fig_supp_step2_balance(row_split_df: pd.DataFrame, cfg: Step2Config) -> None:
    set_plot_style(cfg)

    fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.6))
    fig.subplots_adjust(left=0.08, right=0.985, top=0.95, bottom=0.12, wspace=0.32, hspace=0.34)

    plot_violin_box(axes[0, 0], row_split_df, "length", "Length (aa)", cfg)
    panel_letter(axes[0, 0], cfg, "a")

    plot_violin_box(axes[0, 1], row_split_df, "net_charge_pH7", "Net charge (pH 7)", cfg)
    panel_letter(axes[0, 1], cfg, "b")

    plot_violin_box(axes[1, 0], row_split_df, "hydrophobicity_KD", "Hydrophobicity (Kyte–Doolittle)", cfg)
    panel_letter(axes[1, 0], cfg, "c")

    plot_violin_box(axes[1, 1], row_split_df, "entropy", "Shannon entropy (bits)", cfg)
    panel_letter(axes[1, 1], cfg, "d")

    save_fig(fig, os.path.join(cfg.fig_supp_dir, "FigS4_split_balance_diagnostics"), cfg)
    plt.close(fig)


def fig_supp_step2_leakage(
    leakage_audit_df: pd.DataFrame,
    cluster_summary_df: pd.DataFrame,
    nearest_neighbor_df: pd.DataFrame,
    cfg: Step2Config,
) -> None:
    set_plot_style(cfg)
    pair_order = ["train vs val", "train vs test", "val vs test"]

    sampled_df = leakage_audit_df.loc[leakage_audit_df["audit_type"] == "sampled_cross_split_jaccard"].copy()
    if len(sampled_df):
        sampled_df["pair"] = sampled_df["split_a"].astype(str) + " vs " + sampled_df["split_b"].astype(str)
        sampled_df = complete_ordered_table(
            sampled_df,
            key_col="pair",
            ordered_keys=pair_order,
            agg_dict={
                "n_sampled_pairs": "sum",
                "max_jaccard": "max",
                "mean_jaccard": "mean",
                "threshold": "first",
            },
            fill_value=0,
        )
    else:
        sampled_df = pd.DataFrame({
            "pair": pair_order,
            "n_sampled_pairs": [0, 0, 0],
            "max_jaccard": [0.0, 0.0, 0.0],
            "mean_jaccard": [0.0, 0.0, 0.0],
            "threshold": [cfg.main_jaccard_threshold] * 3,
        })

    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.9))
    fig.subplots_adjust(left=0.06, right=0.985, top=0.92, bottom=0.18, wspace=0.34)

    ax = axes[0]
    vals = sampled_df["max_jaccard"].fillna(0).to_numpy()
    bars = ax.bar(np.arange(len(pair_order)), vals, color=cfg.colors["purple"],
                  edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_xticks(np.arange(len(pair_order)))
    ax.set_xticklabels(pair_order, rotation=12, ha="right")
    thr = sampled_df["threshold"].dropna().iloc[0] if sampled_df["threshold"].notna().any() else cfg.main_jaccard_threshold
    ax.axhline(float(thr), color=cfg.colors["black"], linestyle="--", linewidth=1.0)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{float(v):.2f}",
                ha="center", va="bottom", fontsize=8)
    ax.set_ylabel("Max sampled cross-split Jaccard")
    beautify_axis(ax, cfg, "y")
    panel_letter(ax, cfg, "a")

    ax = axes[1]
    if len(nearest_neighbor_df):
        vals = finite_array(nearest_neighbor_df["nearest_train_jaccard"].values)
        if len(vals):
            ax.hist(vals, bins=20, edgecolor=cfg.colors["gray"], linewidth=0.8, color=cfg.colors["blue2"], alpha=0.8)
            ax.axvline(float(np.median(vals)), color=cfg.colors["black"], linestyle="--", linewidth=1.0)
        else:
            ax.text(0.5, 0.5, "No NN values", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.text(0.5, 0.5, "No nearest-neighbor audit", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("Nearest-train Jaccard")
    ax.set_ylabel("Count")
    beautify_axis(ax, cfg, "y")
    panel_letter(ax, cfg, "b")

    ax = axes[2]
    vals = finite_array(cluster_summary_df["cluster_size_sequences"].values)
    if len(vals):
        bins = np.arange(1, int(vals.max()) + 2)
        ax.hist(vals, bins=bins, edgecolor=cfg.colors["gray"], linewidth=0.8, color=cfg.colors["green"], alpha=0.8)
    else:
        ax.text(0.5, 0.5, "No clusters", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("Cluster size (sequences)")
    ax.set_ylabel("Count")
    beautify_axis(ax, cfg, "y")
    panel_letter(ax, cfg, "c")

    save_fig(fig, os.path.join(cfg.fig_supp_dir, "FigS5_leakage_diagnostics"), cfg)
    plt.close(fig)


def fig_supp_step2_threshold_sensitivity(
    sensitivity_df: pd.DataFrame,
    cfg: Step2Config,
) -> None:
    set_plot_style(cfg)

    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.8))
    fig.subplots_adjust(left=0.08, right=0.985, top=0.90, bottom=0.18, wspace=0.32)

    ax = axes[0]
    ax.plot(sensitivity_df["threshold"], sensitivity_df["n_edges"], marker="o", linewidth=1.4)
    ax.set_xlabel("Jaccard threshold")
    ax.set_ylabel("Retained similarity edges")
    beautify_axis(ax, cfg, "y")
    panel_letter(ax, cfg, "a")

    ax = axes[1]
    for split, color in [("train", cfg.colors["train"]), ("val", cfg.colors["val"]), ("test", cfg.colors["test"])]:
        ax.plot(
            sensitivity_df["threshold"],
            sensitivity_df[f"{split}_n_sequences"],
            marker="o",
            linewidth=1.4,
            label=split,
            color=color,
        )
    ax.set_xlabel("Jaccard threshold")
    ax.set_ylabel("Unique sequences in split")
    beautify_axis(ax, cfg, "y")
    ax.legend(frameon=False)
    panel_letter(ax, cfg, "b")

    save_fig(fig, os.path.join(cfg.fig_supp_dir, "FigS6_threshold_sensitivity"), cfg)
    plt.close(fig)


# ============================================================
# REPORT WRITING
# ============================================================

def build_method_report(
    cfg: Step2Config,
    row_df: pd.DataFrame,
    seq_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    edge_df: pd.DataFrame,
    cluster_summary_df: pd.DataFrame,
    split_summary_df: pd.DataFrame,
    leakage_audit_df: pd.DataFrame,
    sensitivity_df: pd.DataFrame,
) -> str:
    n_rows = len(row_df)
    n_unique = len(seq_df)
    n_candidates = len(candidate_df)
    n_edges = len(edge_df)
    n_clusters = cluster_summary_df["cluster_id"].nunique()

    split_map = split_summary_df.set_index("split").to_dict(orient="index")

    exact_df = leakage_audit_df.loc[leakage_audit_df["audit_type"] == "exact_sequence_overlap"].copy()
    sampled_df = leakage_audit_df.loc[leakage_audit_df["audit_type"] == "sampled_cross_split_jaccard"].copy()

    exact_max = int(exact_df["n_overlap"].max()) if len(exact_df) else 0
    sampled_max = float(sampled_df["max_jaccard"].max()) if len(sampled_df) and sampled_df["max_jaccard"].notna().any() else np.nan

    lines = []
    lines.append("Step 2 — Leakage-aware split design and novelty-reference construction")
    lines.append("=" * 78)
    lines.append("")
    lines.append(
        "This step used unique peptide sequence as the split unit, then constructed a "
        "similarity graph using k-mer representation, deterministic MinHash signatures, "
        "LSH-based candidate retrieval, and exact Jaccard filtering. Connected components "
        "in this graph were assigned at cluster level to train, validation, and test sets "
        "to reduce cross-split sequence-similarity leakage."
    )
    lines.append("")
    lines.append(f"Input row-level records: {n_rows:,}")
    lines.append(f"Unique sequences after collapse: {n_unique:,}")
    lines.append(f"LSH candidate pairs retained for exact comparison: {n_candidates:,}")
    lines.append(f"Similarity edges retained at threshold {cfg.main_jaccard_threshold:.2f}: {n_edges:,}")
    lines.append(f"Connected components (similarity clusters): {n_clusters:,}")
    lines.append("")

    for split in ["train", "val", "test"]:
        info = split_map.get(split, {"n_rows": 0, "n_unique_sequences": 0, "n_clusters": 0})
        lines.append(
            f"{split:>5s}: "
            f"{int(info['n_rows']):,} rows | "
            f"{int(info['n_unique_sequences']):,} unique sequences | "
            f"{int(info['n_clusters']):,} clusters"
        )

    lines.append("")
    lines.append(f"Maximum exact sequence overlap across split pairs: {exact_max}")
    if pd.notna(sampled_max):
        lines.append(f"Maximum sampled cross-split Jaccard: {sampled_max:.3f}")
    else:
        lines.append("Maximum sampled cross-split Jaccard: NA")

    lines.append("")
    lines.append(
        "Novelty-reference exports were generated as two explicit resources: "
        "(i) a train-only reference for memorization auditing during generation, and "
        "(ii) a full-corpus reference for stricter novelty reporting against the entire "
        "observed peptide universe used in this project."
    )
    lines.append("")
    lines.append("Threshold sensitivity summary:")
    for row in sensitivity_df.itertuples(index=False):
        lines.append(
            f"  threshold={row.threshold:.2f} | edges={int(row.n_edges):,} | "
            f"clusters={int(row.n_clusters_total):,} | "
            f"train/val/test={int(row.train_n_sequences):,}/"
            f"{int(row.val_n_sequences):,}/{int(row.test_n_sequences):,} | "
            f"max exact overlap={int(row.max_exact_overlap)} | "
            f"max sampled Jaccard={row.max_sampled_cross_split_jaccard if pd.notna(row.max_sampled_cross_split_jaccard) else 'NA'}"
        )

    lines.append("")
    lines.append("Interpretation:")
    lines.append(
        "This step is stronger than routine random splitting because it controls leakage at "
        "the level of similarity-connected peptide groups rather than only exact duplicates. "
        "For a generative ACP study, this makes downstream novelty claims materially more defensible."
    )

    return "\n".join(lines)


# ============================================================
# MAIN PIPELINE
# ============================================================

def run_step2_pipeline(cfg: Step2Config) -> Dict[str, Any]:
    validate_config(cfg)
    ensure_output_tree(cfg)

    t0 = time.time()

    row_df = load_step1_retained_dataset(cfg)
    seq_df = collapse_to_unique_sequences(row_df, cfg)
    seq_df = build_sequence_features(seq_df, cfg)

    candidate_df = lsh_candidate_pairs(seq_df, cfg)
    if len(candidate_df):
        kmers_all = seq_df["kmers"].tolist()
        candidate_df = candidate_df.copy()
        candidate_df["jaccard"] = [
            jaccard_similarity(kmers_all[r.idx_a], kmers_all[r.idx_b])
            for r in candidate_df.itertuples(index=False)
        ]
    edge_df = compute_similarity_edges(seq_df, candidate_df, cfg.main_jaccard_threshold)

    seq_cluster_df, cluster_summary_df = build_similarity_clusters(seq_df, edge_df, cfg)
    assigned_cluster_df = assign_clusters_to_splits(cluster_summary_df, cfg)

    cluster_summary_df = cluster_summary_df.merge(
        assigned_cluster_df[["cluster_id", "assigned_split"]],
        on="cluster_id",
        how="left",
        validate="one_to_one",
    )

    seq_split_df = propagate_split_to_sequences(seq_cluster_df, assigned_cluster_df, cfg)
    row_split_df = propagate_split_to_rows(row_df, seq_split_df, cfg)

    exact_df = exact_overlap_audit(seq_split_df, cfg)
    cluster_df = cluster_overlap_audit(seq_split_df)
    sampled_df = sampled_cross_split_jaccard_audit(seq_split_df, cfg)
    leakage_audit_df = pd.concat([exact_df, cluster_df, sampled_df], ignore_index=True)

    nn_df = nearest_neighbor_audit(seq_split_df, cfg)
    sensitivity_df = threshold_sensitivity_analysis(seq_df, candidate_df, cfg)

    split_summary_df = build_split_summary(row_split_df, seq_split_df, cluster_summary_df, cfg)
    novelty_refs = build_novelty_reference_sets(seq_split_df, cfg)

    save_csv(row_split_df, os.path.join(cfg.splits_dir, "step2_row_level_with_splits.csv"))
    save_csv(seq_split_df.drop(columns=["kmers", "minhash_sig"]), os.path.join(cfg.splits_dir, "step2_unique_sequences_with_splits.csv"))
    save_csv(cluster_summary_df, os.path.join(cfg.tables_dir, "step2_cluster_summary.csv"))
    save_csv(split_summary_df, os.path.join(cfg.tables_dir, "step2_split_summary.csv"))
    save_csv(leakage_audit_df, os.path.join(cfg.tables_dir, "step2_leakage_audits.csv"))
    save_csv(nn_df, os.path.join(cfg.tables_dir, "step2_nearest_neighbor_audit.csv"))
    save_csv(sensitivity_df, os.path.join(cfg.tables_dir, "step2_threshold_sensitivity.csv"))
    save_csv(novelty_refs["train_only_reference"], os.path.join(cfg.tables_dir, "step2_train_only_novelty_reference.csv"))
    save_csv(novelty_refs["full_corpus_reference"], os.path.join(cfg.tables_dir, "step2_full_corpus_novelty_reference.csv"))

    if cfg.save_cluster_members:
        cluster_members = seq_split_df[[cfg.sequence_col, "sequence_sha256", "cluster_id", "assigned_split"]].copy()
        save_csv(cluster_members, os.path.join(cfg.tables_dir, "step2_cluster_members.csv"))

    if cfg.save_similarity_edges:
        save_csv(candidate_df, os.path.join(cfg.tables_dir, "step2_candidate_pairs_lsh.csv"))
        save_csv(edge_df, os.path.join(cfg.tables_dir, "step2_similarity_edges_exact_jaccard.csv"))

    run_meta = {
        "config": asdict(cfg),
        "system": system_report(cfg),
        "runtime_seconds": float(time.time() - t0),
        "n_input_rows": int(len(row_df)),
        "n_unique_sequences": int(len(seq_df)),
        "n_candidate_pairs": int(len(candidate_df)),
        "n_similarity_edges": int(len(edge_df)),
        "n_clusters": int(cluster_summary_df["cluster_id"].nunique()),
    }
    save_json(run_meta, os.path.join(cfg.reports_dir, "step2_run_metadata.json"))

    method_text = build_method_report(
        cfg=cfg,
        row_df=row_df,
        seq_df=seq_df,
        candidate_df=candidate_df,
        edge_df=edge_df,
        cluster_summary_df=cluster_summary_df,
        split_summary_df=split_summary_df,
        leakage_audit_df=leakage_audit_df,
        sensitivity_df=sensitivity_df,
    )
    save_text(method_text, os.path.join(cfg.reports_dir, "step2_method_and_results_summary.txt"))

    fig_main_step2_split_design(split_summary_df, leakage_audit_df, cluster_summary_df, cfg)
    fig_supp_step2_balance(row_split_df, cfg)
    fig_supp_step2_leakage(leakage_audit_df, cluster_summary_df, nn_df, cfg)
    fig_supp_step2_threshold_sensitivity(sensitivity_df, cfg)

    return {
        "row_df": row_df,
        "seq_df": seq_df,
        "candidate_df": candidate_df,
        "edge_df": edge_df,
        "seq_cluster_df": seq_cluster_df,
        "cluster_summary_df": cluster_summary_df,
        "seq_split_df": seq_split_df,
        "row_split_df": row_split_df,
        "split_summary_df": split_summary_df,
        "leakage_audit_df": leakage_audit_df,
        "nearest_neighbor_df": nn_df,
        "sensitivity_df": sensitivity_df,
        "novelty_refs": novelty_refs,
        "run_meta": run_meta,
    }


# ============================================================
# NOTEBOOK ENTRYPOINT
# ============================================================

def main_notebook(
    input_csv: str,
    out_dir_name: str = "step2",
    project_root_override: Optional[str] = None,
    train_frac: float = 0.80,
    val_frac: float = 0.10,
    test_frac: float = 0.10,
    kmer_k: int = 3,
    minhash_perm: int = 64,
    lsh_bands: int = 16,
    main_jaccard_threshold: float = 0.55,
    threshold_sensitivity: Tuple[float, ...] = (0.45, 0.55, 0.65),
) -> Dict[str, Any]:
    cfg = Step2Config(
        input_csv=input_csv,
        out_dir_name=out_dir_name,
        project_root_override=project_root_override,
        train_frac=train_frac,
        val_frac=val_frac,
        test_frac=test_frac,
        kmer_k=kmer_k,
        minhash_perm=minhash_perm,
        lsh_bands=lsh_bands,
        main_jaccard_threshold=main_jaccard_threshold,
        threshold_sensitivity=threshold_sensitivity,
    )

    results = run_step2_pipeline(cfg)

    print("=" * 80)
    print("Step 2 completed successfully.")
    print(f"Output directory: {cfg.out_dir}")
    print("-" * 80)
    print(results["split_summary_df"].to_string(index=False))
    print("-" * 80)

    exact_df = results["leakage_audit_df"].loc[
        results["leakage_audit_df"]["audit_type"] == "exact_sequence_overlap"
    ]
    if len(exact_df):
        print("Exact overlap audit:")
        print(exact_df[["split_a", "split_b", "n_overlap"]].to_string(index=False))

    sampled_df = results["leakage_audit_df"].loc[
        results["leakage_audit_df"]["audit_type"] == "sampled_cross_split_jaccard"
    ]
    if len(sampled_df):
        print("-" * 80)
        print("Sampled cross-split Jaccard audit:")
        print(sampled_df[["split_a", "split_b", "max_jaccard", "mean_jaccard"]].to_string(index=False))

    return results


# ============================================================
# EXAMPLE RUN
# ============================================================

# results = main_notebook(
#     input_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step1/tables/step1_retained_dataset.csv",
#     out_dir_name="step2",
#     train_frac=0.80,
#     val_frac=0.10,
#     test_frac=0.10,
#     kmer_k=3,
#     minhash_perm=64,
#     lsh_bands=16,
#     main_jaccard_threshold=0.55,
#     threshold_sensitivity=(0.45, 0.55, 0.65),
# )

In [ ]:
results = main_notebook(
    input_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step1/tables/step1_retained_dataset.csv",
    out_dir_name="step2",
    train_frac=0.80,
    val_frac=0.10,
    test_frac=0.10,
    kmer_k=3,
    minhash_perm=64,
    lsh_bands=16,
    main_jaccard_threshold=0.55,
    threshold_sensitivity=(0.45, 0.55, 0.65),
)

In [ ]:
from pathlib import Path

step2_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2")

print(f"Step 2 directory exists: {step2_dir.exists()}")
print(f"Step 2 directory: {step2_dir}\n")

if step2_dir.exists():
    print("=== Top-level contents ===")
    for p in sorted(step2_dir.iterdir()):
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"{kind:4}  {p.name}")

    print("\n=== Full recursive file listing ===")
    for p in sorted(step2_dir.rglob("*")):
        rel = p.relative_to(step2_dir)
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"{kind:4}  {rel}")
else:
    print("Step 2 directory was not found.")

In [ ]:
from pathlib import Path

step2_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2")
splits_dir = step2_dir / "splits"
tables_dir = step2_dir / "tables"

print("=== SPLITS DIRECTORY ===")
if splits_dir.exists():
    split_files = sorted([p for p in splits_dir.iterdir() if p.is_file()])
    for p in split_files:
        print(p.name)
else:
    print("splits directory not found")

print("\n=== TABLES DIRECTORY ===")
if tables_dir.exists():
    table_files = sorted([p for p in tables_dir.iterdir() if p.is_file()])
    for p in table_files:
        print(p.name)
else:
    print("tables directory not found")

print("\n=== CANDIDATE PRIMARY STEP 3 INPUT ===")
candidates = []
if splits_dir.exists():
    for p in splits_dir.iterdir():
        if p.is_file() and p.suffix.lower() == ".csv":
            name = p.name.lower()
            if "unique" in name and "split" in name:
                candidates.append(p)

if candidates:
    for c in candidates:
        print(c)
else:
    print("No obvious unique-sequence split CSV found automatically.")

New Step 3: nature computational science

In [ ]:

# ============================================================
# PepCVAE / AngioPep-VAE — Step 3
# Conditioning design and controllable label schema
#
# Full Jupyter-ready version
# ============================================================

from __future__ import annotations

import os
import json
import math
import time
import hashlib
import platform
from dataclasses import dataclass, asdict, field
from datetime import datetime
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Step3Config:
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2/splits/step2_unique_sequences_with_splits.csv"
    out_dir_name: str = "step3"
    project_root_override: Optional[str] = None

    # expected columns / fallback candidates
    sequence_col: str = "sequence"
    hash_col: str = "sequence_sha256"
    split_col: str = "assigned_split"
    source_col: str = "source_db4"

    descriptor_candidates: Dict[str, Tuple[str, ...]] = field(default_factory=lambda: {
        "length": ("length", "seq_length", "peptide_length"),
        "net_charge_pH7": ("net_charge_pH7", "net_charge", "charge", "charge_pH7"),
        "hydrophobicity_KD": ("hydrophobicity_KD", "hydrophobicity", "hydrophobicity_kd", "kd_hydrophobicity"),
        "entropy": ("entropy", "shannon_entropy", "sequence_entropy"),
        "dominant_aa_frac": ("dominant_aa_frac", "dominant_aa_fraction", "dominant_residue_fraction"),
    })

    # condition design
    primary_condition_keys: Tuple[str, ...] = ("length", "net_charge_pH7", "hydrophobicity_KD")
    secondary_condition_keys: Tuple[str, ...] = ("entropy",)
    n_bins_primary: int = 3
    n_bins_secondary: int = 3
    bin_labels_primary: Tuple[str, ...] = ("low", "mid", "high")
    bin_labels_secondary: Tuple[str, ...] = ("low", "mid", "high")
    min_train_examples_per_bin: int = 50
    min_non_missing_fraction: float = 0.95
    max_allowed_missing_fraction: float = 0.05
    drop_rows_with_missing_primary_conditions: bool = False
    use_train_only_thresholds: bool = True

    random_seed: int = 42

    png_dpi: int = 300
    tif_dpi: int = 600
    font_family: str = "sans-serif"
    font_list: Tuple[str, ...] = ("Arial", "Helvetica", "DejaVu Sans")
    font_size: int = 8
    title_size: int = 9
    axis_label_size: int = 8
    tick_label_size: int = 8
    axis_linewidth: float = 0.9

    colors: Dict[str, str] = field(default_factory=lambda: {
        "train": "#4E79A7",
        "val": "#F28E2B",
        "test": "#E15759",
        "gray": "#707070",
        "black": "#111111",
        "grid": "#E6E6E6",
        "green": "#59A14F",
        "purple": "#B07AA1",
        "blue2": "#76B7B2",
    })

    @property
    def project_root(self) -> str:
        if self.project_root_override is not None:
            return os.path.abspath(self.project_root_override)
        return os.path.dirname(
            os.path.dirname(
                os.path.dirname(os.path.abspath(self.input_csv))
            )
        )

    @property
    def out_dir(self) -> str:
        return os.path.join(self.project_root, self.out_dir_name)

    @property
    def tables_dir(self) -> str:
        return os.path.join(self.out_dir, "tables")

    @property
    def reports_dir(self) -> str:
        return os.path.join(self.out_dir, "reports")

    @property
    def figures_main_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_main")

    @property
    def figures_supp_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_supplementary")

    @property
    def schemas_dir(self) -> str:
        return os.path.join(self.out_dir, "schemas")


# ============================================================
# GENERAL HELPERS
# ============================================================

def ensure_dir(path: str) -> str:
    path = os.path.abspath(path)
    os.makedirs(path, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step3Config) -> None:
    for path in [cfg.out_dir, cfg.tables_dir, cfg.reports_dir, cfg.figures_main_dir, cfg.figures_supp_dir, cfg.schemas_dir]:
        ensure_dir(path)


def save_csv(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)


def save_text(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def json_default(obj: Any):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(list(obj))
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def sha256_string(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path: str, chunk_size: int = 2**20) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


def normalize_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(seq.split())


def finite_array(values) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def complete_ordered_table(
    df: pd.DataFrame,
    key_col: str,
    ordered_keys: Sequence[str],
    agg_dict: Optional[Dict[str, Any]] = None,
    fill_value: Any = 0,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame({key_col: list(ordered_keys)})

    work = df.copy()
    if agg_dict is None:
        agg_dict = {}
        for col in work.columns:
            if col == key_col:
                continue
            agg_dict[col] = "sum" if pd.api.types.is_numeric_dtype(work[col]) else "first"

    grouped = work.groupby(key_col, as_index=False).agg(agg_dict)
    order_df = pd.DataFrame({key_col: list(ordered_keys)})
    out = order_df.merge(grouped, on=key_col, how="left")

    for col in out.columns:
        if col == key_col:
            continue
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(fill_value)

    return out


def format_interval(left: float, right: float, last_bin: bool = False) -> str:
    if pd.isna(left) or pd.isna(right):
        return "NA"
    right_bracket = "]" if last_bin else ")"
    return f"[{left:.3f}, {right:.3f}{right_bracket}"


def choose_existing_column(df: pd.DataFrame, preferred: str, alternatives: Sequence[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for c in (preferred, *alternatives):
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None


def system_report(cfg: Step3Config) -> Dict[str, Any]:
    return {
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "input_csv": os.path.abspath(cfg.input_csv),
        "input_sha256": sha256_file(cfg.input_csv),
    }


# ============================================================
# PLOT HELPERS
# ============================================================

def set_plot_style(cfg: Step3Config) -> None:
    plt.rcParams.update({
        "font.family": cfg.font_family,
        "font.sans-serif": list(cfg.font_list),
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "axes.linewidth": cfg.axis_linewidth,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
    })


def beautify_axis(ax, cfg: Step3Config, grid_axis: str = "y") -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis=grid_axis, color=cfg.colors["grid"], linewidth=0.8)
    ax.set_axisbelow(True)


def panel_letter(ax, cfg: Step3Config, letter: str) -> None:
    ax.text(-0.14, 1.06, letter, transform=ax.transAxes, fontsize=10, fontweight="bold", va="top", ha="left")


def save_fig(fig, stem_path: str, cfg: Step3Config) -> None:
    fig.savefig(f"{stem_path}.png", dpi=cfg.png_dpi, bbox_inches="tight")
    fig.savefig(f"{stem_path}.pdf", bbox_inches="tight")
    fig.savefig(f"{stem_path}.tiff", dpi=cfg.tif_dpi, bbox_inches="tight")


def _safe_boxplot(ax, data_list, positions, colors):
    bp = ax.boxplot(
        data_list,
        positions=positions,
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        whis=(5, 95),
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor("white")
        patch.set_edgecolor(color)
        patch.set_linewidth(1.0)
    for med in bp["medians"]:
        med.set_color("#111111")
        med.set_linewidth(1.1)
    for whisker in bp["whiskers"]:
        whisker.set_color("#666666")
        whisker.set_linewidth(0.9)
    for cap in bp["caps"]:
        cap.set_color("#666666")
        cap.set_linewidth(0.9)


def plot_violin_box(ax, df: pd.DataFrame, value_col: str, ylabel: str, cfg: Step3Config, split_order=("train", "val", "test")) -> None:
    positions = np.arange(len(split_order))
    raw_data = []
    colors = []

    for split in split_order:
        vals = finite_array(df.loc[df[cfg_current.split_col] == split, value_col].values)
        raw_data.append(vals if len(vals) else np.array([]))
        colors.append(cfg.colors[split])

    has_enough = sum(len(v) > 1 for v in raw_data) >= 1
    if has_enough:
        vp = ax.violinplot(
            raw_data,
            positions=positions,
            widths=0.75,
            showmeans=False,
            showmedians=True,
            showextrema=False,
        )
        for i, body in enumerate(vp["bodies"]):
            body.set_facecolor(colors[i])
            body.set_alpha(0.35)
            body.set_edgecolor(colors[i])
            body.set_linewidth(0.8)
        if "cmedians" in vp:
            vp["cmedians"].set_color(cfg.colors["black"])
            vp["cmedians"].set_linewidth(1.2)

    _safe_boxplot(ax, raw_data, positions, colors)
    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(split_order)
    beautify_axis(ax, cfg, "y")


# ============================================================
# DATA LOADING / VALIDATION
# ============================================================

def validate_config(cfg: Step3Config) -> None:
    if not os.path.exists(cfg.input_csv):
        raise FileNotFoundError(f"Step 3 input CSV was not found: {cfg.input_csv}")
    if cfg.n_bins_primary < 2:
        raise ValueError("n_bins_primary must be >= 2.")
    if cfg.n_bins_secondary < 2:
        raise ValueError("n_bins_secondary must be >= 2.")
    if len(cfg.bin_labels_primary) != cfg.n_bins_primary:
        raise ValueError("len(bin_labels_primary) must equal n_bins_primary.")
    if len(cfg.bin_labels_secondary) != cfg.n_bins_secondary:
        raise ValueError("len(bin_labels_secondary) must equal n_bins_secondary.")
    if cfg.min_train_examples_per_bin < 1:
        raise ValueError("min_train_examples_per_bin must be >= 1.")
    if cfg.max_allowed_missing_fraction < 0 or cfg.max_allowed_missing_fraction >= 1:
        raise ValueError("max_allowed_missing_fraction must be in [0, 1).")


def load_step2_unique_sequences(cfg: Step3Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.input_csv, low_memory=False)
    if len(df) == 0:
        raise ValueError("Input CSV is empty.")

    # normalize sequence column
    seq_col = choose_existing_column(df, cfg.sequence_col, ())
    if seq_col is None:
        raise ValueError(f"Required sequence column '{cfg.sequence_col}' was not found.")
    if seq_col != cfg.sequence_col:
        df = df.rename(columns={seq_col: cfg.sequence_col})

    df[cfg.sequence_col] = df[cfg.sequence_col].map(normalize_sequence)
    df = df.loc[df[cfg.sequence_col] != ""].reset_index(drop=True)

    # hash column
    hash_col = choose_existing_column(df, cfg.hash_col, ())
    if hash_col is None:
        df[cfg.hash_col] = df[cfg.sequence_col].map(sha256_string)
    elif hash_col != cfg.hash_col:
        df = df.rename(columns={hash_col: cfg.hash_col})

    # split column
    split_col = choose_existing_column(df, cfg.split_col, ("split", "data_split"))
    if split_col is None:
        raise ValueError(f"Required split column '{cfg.split_col}' was not found.")
    if split_col != cfg.split_col:
        df = df.rename(columns={split_col: cfg.split_col})

    df[cfg.split_col] = df[cfg.split_col].astype(str).str.strip().str.lower()
    allowed_splits = {"train", "val", "test"}
    bad_splits = sorted(set(df[cfg.split_col]) - allowed_splits)
    if bad_splits:
        raise ValueError(f"Unexpected split labels found: {bad_splits}")

    if df[cfg.sequence_col].duplicated().any():
        dups = int(df[cfg.sequence_col].duplicated().sum())
        raise ValueError(f"Input Step 2 unique-sequence file is not unique at the sequence level; duplicated sequences found: {dups}")

    return df


def harmonize_descriptor_columns(df: pd.DataFrame, cfg: Step3Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    resolved = {}
    out = df.copy()

    for canonical, candidates in cfg.descriptor_candidates.items():
        found = choose_existing_column(out, canonical, candidates)
        if found is None:
            resolved[canonical] = None
            continue
        resolved[canonical] = found
        if found != canonical:
            out = out.rename(columns={found: canonical})
        out[canonical] = pd.to_numeric(out[canonical], errors="coerce")

    # source column is optional but useful
    source_found = choose_existing_column(out, cfg.source_col, ("source", "source_db", "dataset_source"))
    if source_found is not None and source_found != cfg.source_col:
        out = out.rename(columns={source_found: cfg.source_col})

    return out, resolved


def validate_required_columns(df: pd.DataFrame, cfg: Step3Config) -> None:
    required = [cfg.sequence_col, cfg.hash_col, cfg.split_col, *cfg.primary_condition_keys]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Required columns were not found after harmonization: {missing}")

    # missingness check on primary condition keys
    missingness = df[list(cfg.primary_condition_keys)].isna().mean()
    too_missing = missingness[missingness > cfg.max_allowed_missing_fraction]
    if len(too_missing):
        raise ValueError(
            "Primary condition columns exceed allowed missingness: "
            + ", ".join([f"{k}={v:.3f}" for k, v in too_missing.items()])
        )


# ============================================================
# CONDITION FITTING / APPLICATION
# ============================================================

def robust_quantile_edges(x: pd.Series, n_bins: int) -> np.ndarray:
    vals = pd.to_numeric(x, errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        raise ValueError("Cannot fit bins on an empty numeric series.")
    q = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(vals, q, method="linear")
    edges = np.asarray(edges, dtype=float)

    # enforce monotonicity and uniqueness
    edges = np.maximum.accumulate(edges)
    if np.unique(edges).size < (n_bins + 1):
        uniq = np.unique(vals)
        if uniq.size < n_bins:
            raise ValueError(f"Not enough unique values to form {n_bins} bins.")
        fallback_q = np.linspace(0, uniq.size - 1, n_bins + 1)
        edges = np.interp(fallback_q, np.arange(uniq.size), uniq)

    # tiny epsilon padding to ensure inclusion of extrema
    eps = 1e-9
    edges[0] = edges[0] - eps
    edges[-1] = edges[-1] + eps
    edges = np.maximum.accumulate(edges)
    return edges


def assign_bins(series: pd.Series, edges: np.ndarray, labels: Sequence[str]) -> pd.Categorical:
    out = pd.cut(series, bins=edges, labels=list(labels), include_lowest=True, right=False, ordered=True)
    # last interval closed on the right by epsilon padding
    return out


def fit_condition_schema(df: pd.DataFrame, cfg: Step3Config) -> Dict[str, Any]:
    train_df = df.loc[df[cfg.split_col] == "train"].copy()
    if len(train_df) == 0:
        raise ValueError("No training rows were found in the Step 3 input.")
    schema = {
        "step": "step3_conditioning_schema",
        "created_at": datetime.now().isoformat(),
        "input_csv": os.path.abspath(cfg.input_csv),
        "input_sha256": sha256_file(cfg.input_csv),
        "use_train_only_thresholds": bool(cfg.use_train_only_thresholds),
        "sequence_col": cfg.sequence_col,
        "hash_col": cfg.hash_col,
        "split_col": cfg.split_col,
        "source_col": cfg.source_col if cfg.source_col in df.columns else None,
        "conditions": {},
    }

    all_keys = list(cfg.primary_condition_keys) + [k for k in cfg.secondary_condition_keys if k in df.columns]
    for key in all_keys:
        labels = cfg.bin_labels_primary if key in cfg.primary_condition_keys else cfg.bin_labels_secondary
        n_bins = cfg.n_bins_primary if key in cfg.primary_condition_keys else cfg.n_bins_secondary
        fit_source = train_df[key] if cfg.use_train_only_thresholds else df[key]
        edges = robust_quantile_edges(fit_source, n_bins=n_bins)

        train_bins = assign_bins(train_df[key], edges, labels)
        counts = pd.Series(train_bins).value_counts(dropna=False).reindex(list(labels), fill_value=0)

        small = counts[counts < cfg.min_train_examples_per_bin]
        if len(small):
            raise ValueError(
                f"Condition '{key}' produced train bins below min_train_examples_per_bin={cfg.min_train_examples_per_bin}: "
                + ", ".join([f"{k}={int(v)}" for k, v in small.items()])
            )

        intervals = []
        for i in range(len(labels)):
            intervals.append(format_interval(edges[i], edges[i + 1], last_bin=(i == len(labels) - 1)))

        schema["conditions"][key] = {
            "type": "binned_numeric",
            "is_primary": bool(key in cfg.primary_condition_keys),
            "n_bins": int(n_bins),
            "labels": list(labels),
            "edges": [float(v) for v in edges],
            "interval_labels": intervals,
            "fit_population": "train_only" if cfg.use_train_only_thresholds else "all_splits",
            "train_bin_counts": {str(k): int(v) for k, v in counts.items()},
        }

    return schema


def apply_condition_schema(df: pd.DataFrame, schema: Dict[str, Any], cfg: Step3Config) -> pd.DataFrame:
    out = df.copy()

    for key, meta in schema["conditions"].items():
        edges = np.asarray(meta["edges"], dtype=float)
        labels = meta["labels"]
        out[f"{key}_bin"] = assign_bins(pd.to_numeric(out[key], errors="coerce"), edges, labels).astype("object")
        label_to_idx = {lab: i for i, lab in enumerate(labels)}
        out[f"{key}_bin_idx"] = out[f"{key}_bin"].map(label_to_idx)

    # completeness
    primary_bin_cols = [f"{k}_bin" for k in cfg.primary_condition_keys]
    secondary_bin_cols = [f"{k}_bin" for k in schema["conditions"].keys() if k not in cfg.primary_condition_keys]
    out["is_primary_condition_complete"] = out[primary_bin_cols].notna().all(axis=1)
    out["is_all_condition_complete"] = out[primary_bin_cols + secondary_bin_cols].notna().all(axis=1) if secondary_bin_cols else out["is_primary_condition_complete"]

    # condition vectors / IDs
    def make_cond_id(row, keys):
        parts = []
        for k in keys:
            v = row.get(f"{k}_bin", np.nan)
            parts.append(f"{k}={v}" if pd.notna(v) else f"{k}=NA")
        return "|".join(parts)

    primary_keys = list(cfg.primary_condition_keys)
    all_keys = list(schema["conditions"].keys())

    out["primary_condition_id"] = out.apply(lambda r: make_cond_id(r, primary_keys), axis=1)
    out["full_condition_id"] = out.apply(lambda r: make_cond_id(r, all_keys), axis=1)

    def make_vector(row, keys):
        vect = {}
        for k in keys:
            vect[k] = {
                "raw": None if pd.isna(row.get(k, np.nan)) else float(row[k]),
                "bin": None if pd.isna(row.get(f"{k}_bin", np.nan)) else str(row[f"{k}_bin"]),
                "bin_idx": None if pd.isna(row.get(f"{k}_bin_idx", np.nan)) else int(row[f"{k}_bin_idx"]),
            }
        return json.dumps(vect, ensure_ascii=False, sort_keys=True)

    out["primary_condition_vector_json"] = out.apply(lambda r: make_vector(r, primary_keys), axis=1)
    out["full_condition_vector_json"] = out.apply(lambda r: make_vector(r, all_keys), axis=1)

    return out


def maybe_filter_missing_primary(df: pd.DataFrame, cfg: Step3Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not cfg.drop_rows_with_missing_primary_conditions:
        excluded = df.loc[~df["is_primary_condition_complete"]].copy()
        return df, excluded

    keep = df.loc[df["is_primary_condition_complete"]].copy().reset_index(drop=True)
    excluded = df.loc[~df["is_primary_condition_complete"]].copy().reset_index(drop=True)
    return keep, excluded


# ============================================================
# SUMMARIES / QC
# ============================================================

def build_step3_summary(df: pd.DataFrame, schema: Dict[str, Any], cfg: Step3Config) -> pd.DataFrame:
    split_order = ["train", "val", "test"]
    out = (
        df.groupby(cfg.split_col, as_index=False)
        .agg(
            n_sequences=(cfg.sequence_col, "size"),
            n_primary_condition_complete=("is_primary_condition_complete", "sum"),
            n_all_condition_complete=("is_all_condition_complete", "sum"),
            n_primary_condition_ids=("primary_condition_id", "nunique"),
            n_full_condition_ids=("full_condition_id", "nunique"),
        )
        .rename(columns={cfg.split_col: "split"})
    )
    out = complete_ordered_table(
        out,
        key_col="split",
        ordered_keys=split_order,
        agg_dict={
            "n_sequences": "sum",
            "n_primary_condition_complete": "sum",
            "n_all_condition_complete": "sum",
            "n_primary_condition_ids": "sum",
            "n_full_condition_ids": "sum",
        },
        fill_value=0,
    )
    out["frac_primary_condition_complete"] = out["n_primary_condition_complete"] / out["n_sequences"].clip(lower=1)
    out["frac_all_condition_complete"] = out["n_all_condition_complete"] / out["n_sequences"].clip(lower=1)
    return out


def build_condition_balance_by_split(df: pd.DataFrame, schema: Dict[str, Any], cfg: Step3Config) -> pd.DataFrame:
    records = []
    for key, meta in schema["conditions"].items():
        bin_col = f"{key}_bin"
        levels = meta["labels"]
        for split in ["train", "val", "test"]:
            sub = df.loc[df[cfg.split_col] == split].copy()
            total = max(len(sub), 1)
            counts = sub[bin_col].value_counts(dropna=False)
            for level in levels:
                n = int(counts.get(level, 0))
                records.append({
                    "condition": key,
                    "split": split,
                    "level": level,
                    "n_sequences": n,
                    "fraction": float(n / total),
                })
            n_missing = int(sub[bin_col].isna().sum())
            records.append({
                "condition": key,
                "split": split,
                "level": "missing",
                "n_sequences": n_missing,
                "fraction": float(n_missing / total),
            })
    return pd.DataFrame(records)


def build_condition_combination_summary(df: pd.DataFrame, cfg: Step3Config) -> pd.DataFrame:
    out = (
        df.groupby([cfg.split_col, "primary_condition_id"], as_index=False)
        .agg(n_sequences=(cfg.sequence_col, "size"))
        .rename(columns={cfg.split_col: "split"})
        .sort_values(["split", "n_sequences", "primary_condition_id"], ascending=[True, False, True])
        .reset_index(drop=True)
    )
    return out


def build_missingness_table(df: pd.DataFrame, cfg: Step3Config) -> pd.DataFrame:
    candidate_cols = [cfg.sequence_col, cfg.hash_col, cfg.split_col, cfg.source_col] + list(cfg.descriptor_candidates.keys())
    candidate_cols = [c for c in candidate_cols if c in df.columns]
    records = []
    n = len(df)
    for col in candidate_cols:
        n_missing = int(df[col].isna().sum())
        records.append({
            "column": col,
            "n_missing": n_missing,
            "fraction_missing": float(n_missing / max(n, 1)),
        })
    return pd.DataFrame(records).sort_values(["fraction_missing", "column"], ascending=[False, True]).reset_index(drop=True)


def build_next_step_inputs_table(cfg: Step3Config) -> pd.DataFrame:
    rows = [
        {
            "next_step": "Step 4",
            "purpose": "representation, tokenization, and model-ready preprocessing",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"),
        },
        {
            "next_step": "Step 4",
            "purpose": "frozen condition thresholds / schema reuse",
            "recommended_input": os.path.join(cfg.schemas_dir, "step3_condition_schema.json"),
        },
        {
            "next_step": "Step 4",
            "purpose": "condition balance quality control",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_condition_balance_by_split.csv"),
        },
        {
            "next_step": "Step 5",
            "purpose": "baseline benchmark design",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"),
        },
        {
            "next_step": "Step 6",
            "purpose": "conditional generative model development",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"),
        },
        {
            "next_step": "Step 7",
            "purpose": "generation-time controllability auditing",
            "recommended_input": os.path.join(cfg.schemas_dir, "step3_condition_schema.json"),
        },
        {
            "next_step": "Step 10",
            "purpose": "Streamlit condition controls and user-facing schema",
            "recommended_input": os.path.join(cfg.schemas_dir, "step3_condition_schema.json"),
        },
    ]
    return pd.DataFrame(rows)


# ============================================================
# FIGURES
# ============================================================

def fig_main_step3_condition_design(df: pd.DataFrame, balance_df: pd.DataFrame, cfg: Step3Config) -> None:
    set_plot_style(cfg)
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.5))
    fig.subplots_adjust(left=0.07, right=0.985, top=0.90, bottom=0.20, wspace=0.40)

    # a: primary condition coverage by split
    ax = axes[0]
    coverage = (
        df.groupby(cfg.split_col, as_index=False)["is_primary_condition_complete"]
        .mean()
        .rename(columns={cfg.split_col: "split", "is_primary_condition_complete": "coverage"})
    )
    coverage = complete_ordered_table(coverage, "split", ["train", "val", "test"], {"coverage": "mean"}, 0)
    vals = coverage["coverage"].to_numpy()
    bars = ax.bar(np.arange(3), vals, color=[cfg.colors[s] for s in ["train", "val", "test"]], edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Primary-condition completeness")
    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(["train", "val", "test"])
    beautify_axis(ax, cfg, "y")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "a")

    # b: number of primary condition IDs by split
    ax = axes[1]
    cond_counts = (
        df.groupby(cfg.split_col, as_index=False)["primary_condition_id"]
        .nunique()
        .rename(columns={cfg.split_col: "split", "primary_condition_id": "n_ids"})
    )
    cond_counts = complete_ordered_table(cond_counts, "split", ["train", "val", "test"], {"n_ids": "sum"}, 0)
    vals = cond_counts["n_ids"].to_numpy()
    bars = ax.bar(np.arange(3), vals, color=[cfg.colors[s] for s in ["train", "val", "test"]], edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylabel("Unique primary condition IDs")
    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(["train", "val", "test"])
    beautify_axis(ax, cfg, "y")
    ymax = max(vals) if len(vals) else 1
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + max(ymax * 0.02, 0.1), f"{int(v)}", ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "b")

    # c: train distribution for primary conditions
    ax = axes[2]
    train_balance = balance_df.loc[(balance_df["split"] == "train") & (balance_df["condition"].isin(cfg.primary_condition_keys)) & (balance_df["level"] != "missing")].copy()
    if len(train_balance):
        pivot = train_balance.pivot_table(index="condition", columns="level", values="fraction", fill_value=0.0)
        col_order = list(cfg.bin_labels_primary)
        pivot = pivot.reindex(columns=col_order, fill_value=0.0)
        y = np.arange(len(pivot.index))
        left = np.zeros(len(y))
        palette = [cfg.colors["train"], cfg.colors["blue2"], cfg.colors["purple"]]
        for level, color in zip(col_order, palette):
            vals = pivot[level].to_numpy()
            ax.barh(y, vals, left=left, color=color, edgecolor="white", linewidth=0.6, label=level)
            left += vals
        ax.set_yticks(y)
        ax.set_yticklabels(pivot.index.tolist())
        ax.set_xlabel("Train fraction")
        ax.set_xlim(0, 1.0)
        ax.legend(frameon=False, fontsize=7, loc="lower right")
    beautify_axis(ax, cfg, "x")
    panel_letter(ax, cfg, "c")

    save_fig(fig, os.path.join(cfg.figures_main_dir, "Fig3_condition_schema_main"), cfg)
    plt.close(fig)


def fig_supp_step3_raw_distributions(df: pd.DataFrame, cfg: Step3Config) -> None:
    set_plot_style(cfg)
    plot_keys = [k for k in ["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"] if k in df.columns]
    n = len(plot_keys)
    if n == 0:
        return

    fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.6))
    axes = np.asarray(axes).reshape(-1)
    fig.subplots_adjust(left=0.08, right=0.985, top=0.95, bottom=0.10, wspace=0.32, hspace=0.34)
    labels = {
        "length": "Length (aa)",
        "net_charge_pH7": "Net charge (pH 7)",
        "hydrophobicity_KD": "Hydrophobicity (Kyte–Doolittle)",
        "entropy": "Shannon entropy (bits)",
    }

    for ax, key, letter in zip(axes, plot_keys, ["a", "b", "c", "d"]):
        plot_violin_box(ax, df, key, labels.get(key, key), cfg)
        panel_letter(ax, cfg, letter)

    for ax in axes[n:]:
        ax.axis("off")

    save_fig(fig, os.path.join(cfg.figures_supp_dir, "FigS7_step3_raw_condition_distributions"), cfg)
    plt.close(fig)


def fig_supp_step3_bin_balance(balance_df: pd.DataFrame, cfg: Step3Config) -> None:
    set_plot_style(cfg)
    conds = sorted([c for c in balance_df["condition"].dropna().unique().tolist() if c in (set(cfg.primary_condition_keys) | set(cfg.secondary_condition_keys))])
    if len(conds) == 0:
        return

    fig, axes = plt.subplots(len(conds), 1, figsize=(8.2, 2.0 * len(conds)), squeeze=False)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.96, bottom=0.10, hspace=0.45)

    split_order = ["train", "val", "test"]
    palette = [cfg.colors["train"], cfg.colors["blue2"], cfg.colors["purple"], cfg.colors["gray"]]

    for i, cond in enumerate(conds):
        ax = axes[i, 0]
        sub = balance_df.loc[(balance_df["condition"] == cond) & (balance_df["level"] != "missing")].copy()
        pivot = sub.pivot_table(index="split", columns="level", values="fraction", fill_value=0.0)
        all_levels = cfg.bin_labels_primary if cond in cfg.primary_condition_keys else cfg.bin_labels_secondary
        pivot = pivot.reindex(index=split_order, columns=list(all_levels), fill_value=0.0)
        bottoms = np.zeros(len(split_order))
        for level, color in zip(pivot.columns.tolist(), palette):
            vals = pivot[level].to_numpy()
            ax.bar(np.arange(len(split_order)), vals, bottom=bottoms, color=color, edgecolor="white", linewidth=0.7, label=level)
            bottoms += vals
        ax.set_ylim(0, 1.0)
        ax.set_ylabel(cond)
        ax.set_xticks(np.arange(len(split_order)))
        ax.set_xticklabels(split_order)
        beautify_axis(ax, cfg, "y")
        if i == 0:
            ax.legend(frameon=False, fontsize=7, ncol=len(pivot.columns), loc="upper right")
        panel_letter(ax, cfg, chr(ord("a") + i))

    save_fig(fig, os.path.join(cfg.figures_supp_dir, "FigS8_step3_condition_bin_balance"), cfg)
    plt.close(fig)


# ============================================================
# REPORTING
# ============================================================

def build_method_report(
    cfg: Step3Config,
    df_in: pd.DataFrame,
    df_out: pd.DataFrame,
    excluded_df: pd.DataFrame,
    schema: Dict[str, Any],
    summary_df: pd.DataFrame,
    balance_df: pd.DataFrame,
) -> str:
    lines = []
    lines.append("Step 3 — Conditioning design and controllable label schema")
    lines.append("=" * 80)
    lines.append(f"Input CSV: {os.path.abspath(cfg.input_csv)}")
    lines.append(f"Input SHA256: {sha256_file(cfg.input_csv)}")
    lines.append(f"Total input sequences: {len(df_in):,}")
    lines.append(f"Sequences retained after optional primary-condition filtering: {len(df_out):,}")
    lines.append(f"Sequences excluded for incomplete primary conditions: {len(excluded_df):,}")
    lines.append("")
    lines.append("Condition schema")
    lines.append("-" * 80)
    for key, meta in schema["conditions"].items():
        lines.append(
            f"{key}: labels={meta['labels']}, fit_population={meta['fit_population']}, "
            f"intervals={meta['interval_labels']}"
        )
    lines.append("")
    lines.append("Split summary")
    lines.append("-" * 80)
    lines.append(summary_df.to_string(index=False))
    lines.append("")
    lines.append("Condition balance (head)")
    lines.append("-" * 80)
    lines.append(balance_df.head(24).to_string(index=False))
    lines.append("")
    lines.append("Interpretation")
    lines.append("-" * 80)
    lines.append(
        "This step froze a train-derived conditioning schema for controllable generation. "
        "Primary condition bins were fitted using the training split only and then applied unchanged "
        "to validation and test data. The resulting exports are intended for Step 4 representation "
        "building, Step 6 conditional generative model development, Step 7 controllability auditing, "
        "and Step 10 Streamlit condition controls."
    )
    return "\n".join(lines)


def print_next_step_file_locations(cfg: Step3Config) -> None:
    next_inputs = build_next_step_inputs_table(cfg)
    print("=" * 80)
    print("Recommended file locations for later steps")
    print(next_inputs.to_string(index=False))
    print("=" * 80)

    print("Existing Step 3 output tree")
    for root, dirs, files in os.walk(cfg.out_dir):
        rel = os.path.relpath(root, cfg.out_dir)
        print(f"[DIR] {rel}")
        for fn in sorted(files):
            print(f"  - {fn}")


# ============================================================
# PIPELINE
# ============================================================

def run_step3_pipeline(cfg: Step3Config) -> Dict[str, Any]:
    global cfg_current
    cfg_current = cfg  # used by plot helper for split_col

    t0 = time.time()
    np.random.seed(cfg.random_seed)

    validate_config(cfg)
    ensure_output_tree(cfg)

    print("Loading Step 2 unique-sequence split file...")
    df = load_step2_unique_sequences(cfg)
    df, resolved = harmonize_descriptor_columns(df, cfg)
    validate_required_columns(df, cfg)

    print("Fitting train-derived condition schema...")
    schema = fit_condition_schema(df, cfg)

    print("Applying condition schema to all splits...")
    conditioned_df = apply_condition_schema(df, schema, cfg)
    conditioned_df, excluded_df = maybe_filter_missing_primary(conditioned_df, cfg)

    print("Building summaries and QC tables...")
    summary_df = build_step3_summary(conditioned_df, schema, cfg)
    balance_df = build_condition_balance_by_split(conditioned_df, schema, cfg)
    combination_df = build_condition_combination_summary(conditioned_df, cfg)
    missingness_df = build_missingness_table(df, cfg)
    next_step_df = build_next_step_inputs_table(cfg)

    print("Writing exports...")
    save_csv(conditioned_df, os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"))
    save_csv(conditioned_df.loc[conditioned_df[cfg.split_col] == "train"].copy(), os.path.join(cfg.tables_dir, "step3_train_conditioned_unique_sequences.csv"))
    save_csv(conditioned_df.loc[conditioned_df[cfg.split_col] == "val"].copy(), os.path.join(cfg.tables_dir, "step3_val_conditioned_unique_sequences.csv"))
    save_csv(conditioned_df.loc[conditioned_df[cfg.split_col] == "test"].copy(), os.path.join(cfg.tables_dir, "step3_test_conditioned_unique_sequences.csv"))
    save_csv(summary_df, os.path.join(cfg.tables_dir, "step3_split_condition_summary.csv"))
    save_csv(balance_df, os.path.join(cfg.tables_dir, "step3_condition_balance_by_split.csv"))
    save_csv(combination_df, os.path.join(cfg.tables_dir, "step3_primary_condition_combinations.csv"))
    save_csv(missingness_df, os.path.join(cfg.tables_dir, "step3_missingness_table.csv"))
    save_csv(excluded_df, os.path.join(cfg.tables_dir, "step3_excluded_incomplete_primary_conditions.csv"))
    save_csv(next_step_df, os.path.join(cfg.tables_dir, "step3_next_step_input_locations.csv"))

    save_json(schema, os.path.join(cfg.schemas_dir, "step3_condition_schema.json"))

    run_meta = {
        "config": asdict(cfg),
        "system": system_report(cfg),
        "resolved_descriptor_columns": resolved,
        "runtime_seconds": float(time.time() - t0),
        "n_input_sequences": int(len(df)),
        "n_retained_sequences": int(len(conditioned_df)),
        "n_excluded_sequences": int(len(excluded_df)),
        "n_primary_condition_ids": int(conditioned_df["primary_condition_id"].nunique()),
        "n_full_condition_ids": int(conditioned_df["full_condition_id"].nunique()),
    }
    save_json(run_meta, os.path.join(cfg.reports_dir, "step3_run_metadata.json"))

    method_text = build_method_report(
        cfg=cfg,
        df_in=df,
        df_out=conditioned_df,
        excluded_df=excluded_df,
        schema=schema,
        summary_df=summary_df,
        balance_df=balance_df,
    )
    save_text(method_text, os.path.join(cfg.reports_dir, "step3_method_and_results_summary.txt"))

    print("Generating figures...")
    fig_main_step3_condition_design(conditioned_df, balance_df, cfg)
    fig_supp_step3_raw_distributions(conditioned_df, cfg)
    fig_supp_step3_bin_balance(balance_df, cfg)

    print("Step 3 completed successfully.")
    print(f"Output directory: {cfg.out_dir}")
    print("-" * 80)
    print(summary_df.to_string(index=False))
    print_next_step_file_locations(cfg)

    return {
        "input_df": df,
        "conditioned_df": conditioned_df,
        "excluded_df": excluded_df,
        "schema": schema,
        "summary_df": summary_df,
        "balance_df": balance_df,
        "combination_df": combination_df,
        "missingness_df": missingness_df,
        "next_step_df": next_step_df,
        "run_meta": run_meta,
    }


# ============================================================
# NOTEBOOK ENTRYPOINT
# ============================================================

def main_notebook(
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2/splits/step2_unique_sequences_with_splits.csv",
    out_dir_name: str = "step3",
    project_root_override: Optional[str] = None,
    random_seed: int = 42,
    n_bins_primary: int = 3,
    n_bins_secondary: int = 3,
    min_train_examples_per_bin: int = 50,
    drop_rows_with_missing_primary_conditions: bool = False,
) -> Dict[str, Any]:
    cfg = Step3Config(
        input_csv=input_csv,
        out_dir_name=out_dir_name,
        project_root_override=project_root_override,
        random_seed=random_seed,
        n_bins_primary=n_bins_primary,
        n_bins_secondary=n_bins_secondary,
        min_train_examples_per_bin=min_train_examples_per_bin,
        drop_rows_with_missing_primary_conditions=drop_rows_with_missing_primary_conditions,
    )
    return run_step3_pipeline(cfg)


In [ ]:
results = main_notebook(
    input_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2/splits/step2_unique_sequences_with_splits.csv",
    out_dir_name="step3",
)

In [ ]:

# ============================================================
# PepCVAE / AngioPep-VAE — Step 3
# Conditioning design and controllable label schema
#
# Full Jupyter-ready version
# ============================================================

from __future__ import annotations

import os
import json
import math
import time
import hashlib
import platform
from dataclasses import dataclass, asdict, field
from datetime import datetime
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Step3Config:
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2/splits/step2_unique_sequences_with_splits.csv"
    out_dir_name: str = "step3"
    project_root_override: Optional[str] = None

    # expected columns / fallback candidates
    sequence_col: str = "sequence"
    hash_col: str = "sequence_sha256"
    split_col: str = "assigned_split"
    source_col: str = "source_db4"

    descriptor_candidates: Dict[str, Tuple[str, ...]] = field(default_factory=lambda: {
        "length": ("length", "seq_length", "peptide_length"),
        "net_charge_pH7": ("net_charge_pH7", "net_charge", "charge", "charge_pH7"),
        "hydrophobicity_KD": ("hydrophobicity_KD", "hydrophobicity", "hydrophobicity_kd", "kd_hydrophobicity"),
        "entropy": ("entropy", "shannon_entropy", "sequence_entropy"),
        "dominant_aa_frac": ("dominant_aa_frac", "dominant_aa_fraction", "dominant_residue_fraction"),
    })

    # condition design
    primary_condition_keys: Tuple[str, ...] = ("length", "net_charge_pH7", "hydrophobicity_KD")
    secondary_condition_keys: Tuple[str, ...] = ("entropy",)
    n_bins_primary: int = 3
    n_bins_secondary: int = 3
    bin_labels_primary: Tuple[str, ...] = ("low", "mid", "high")
    bin_labels_secondary: Tuple[str, ...] = ("low", "mid", "high")
    min_train_examples_per_bin: int = 50
    min_non_missing_fraction: float = 0.95
    max_allowed_missing_fraction: float = 0.05
    drop_rows_with_missing_primary_conditions: bool = True
    use_train_only_thresholds: bool = True
    enforce_train_supported_primary_conditions: bool = True
    unseen_primary_condition_policy: str = "exclude"  # exclude | error | keep

    random_seed: int = 42

    png_dpi: int = 300
    tif_dpi: int = 600
    font_family: str = "sans-serif"
    font_list: Tuple[str, ...] = ("Arial", "Helvetica", "DejaVu Sans")
    font_size: int = 8
    title_size: int = 9
    axis_label_size: int = 8
    tick_label_size: int = 8
    axis_linewidth: float = 0.9

    colors: Dict[str, str] = field(default_factory=lambda: {
        "train": "#4E79A7",
        "val": "#F28E2B",
        "test": "#E15759",
        "gray": "#707070",
        "black": "#111111",
        "grid": "#E6E6E6",
        "green": "#59A14F",
        "purple": "#B07AA1",
        "blue2": "#76B7B2",
    })

    @property
    def project_root(self) -> str:
        if self.project_root_override is not None:
            return os.path.abspath(self.project_root_override)
        return os.path.dirname(
            os.path.dirname(
                os.path.dirname(os.path.abspath(self.input_csv))
            )
        )

    @property
    def out_dir(self) -> str:
        return os.path.join(self.project_root, self.out_dir_name)

    @property
    def tables_dir(self) -> str:
        return os.path.join(self.out_dir, "tables")

    @property
    def reports_dir(self) -> str:
        return os.path.join(self.out_dir, "reports")

    @property
    def figures_main_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_main")

    @property
    def figures_supp_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_supplementary")

    @property
    def schemas_dir(self) -> str:
        return os.path.join(self.out_dir, "schemas")


# ============================================================
# GENERAL HELPERS
# ============================================================

def ensure_dir(path: str) -> str:
    path = os.path.abspath(path)
    os.makedirs(path, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step3Config) -> None:
    for path in [cfg.out_dir, cfg.tables_dir, cfg.reports_dir, cfg.figures_main_dir, cfg.figures_supp_dir, cfg.schemas_dir]:
        ensure_dir(path)


def save_csv(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)


def save_text(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def json_default(obj: Any):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(list(obj))
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def sha256_string(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path: str, chunk_size: int = 2**20) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


def normalize_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(seq.split())


def finite_array(values) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def complete_ordered_table(
    df: pd.DataFrame,
    key_col: str,
    ordered_keys: Sequence[str],
    agg_dict: Optional[Dict[str, Any]] = None,
    fill_value: Any = 0,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame({key_col: list(ordered_keys)})

    work = df.copy()
    if agg_dict is None:
        agg_dict = {}
        for col in work.columns:
            if col == key_col:
                continue
            agg_dict[col] = "sum" if pd.api.types.is_numeric_dtype(work[col]) else "first"

    grouped = work.groupby(key_col, as_index=False).agg(agg_dict)
    order_df = pd.DataFrame({key_col: list(ordered_keys)})
    out = order_df.merge(grouped, on=key_col, how="left")

    for col in out.columns:
        if col == key_col:
            continue
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(fill_value)

    return out


def format_interval(left: float, right: float, last_bin: bool = False) -> str:
    if pd.isna(left) or pd.isna(right):
        return "NA"
    right_bracket = "]" if last_bin else ")"
    return f"[{left:.3f}, {right:.3f}{right_bracket}"


def choose_existing_column(df: pd.DataFrame, preferred: str, alternatives: Sequence[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for c in (preferred, *alternatives):
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None


def system_report(cfg: Step3Config) -> Dict[str, Any]:
    return {
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "input_csv": os.path.abspath(cfg.input_csv),
        "input_sha256": sha256_file(cfg.input_csv),
    }


# ============================================================
# PLOT HELPERS
# ============================================================

def set_plot_style(cfg: Step3Config) -> None:
    plt.rcParams.update({
        "font.family": cfg.font_family,
        "font.sans-serif": list(cfg.font_list),
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "axes.linewidth": cfg.axis_linewidth,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
    })


def beautify_axis(ax, cfg: Step3Config, grid_axis: str = "y") -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis=grid_axis, color=cfg.colors["grid"], linewidth=0.8)
    ax.set_axisbelow(True)


def panel_letter(ax, cfg: Step3Config, letter: str) -> None:
    ax.text(-0.14, 1.06, letter, transform=ax.transAxes, fontsize=10, fontweight="bold", va="top", ha="left")


def save_fig(fig, stem_path: str, cfg: Step3Config) -> None:
    fig.savefig(f"{stem_path}.png", dpi=cfg.png_dpi, bbox_inches="tight")
    fig.savefig(f"{stem_path}.pdf", bbox_inches="tight")
    fig.savefig(f"{stem_path}.tiff", dpi=cfg.tif_dpi, bbox_inches="tight")


def _safe_boxplot(ax, data_list, positions, colors):
    bp = ax.boxplot(
        data_list,
        positions=positions,
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        whis=(5, 95),
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor("white")
        patch.set_edgecolor(color)
        patch.set_linewidth(1.0)
    for med in bp["medians"]:
        med.set_color("#111111")
        med.set_linewidth(1.1)
    for whisker in bp["whiskers"]:
        whisker.set_color("#666666")
        whisker.set_linewidth(0.9)
    for cap in bp["caps"]:
        cap.set_color("#666666")
        cap.set_linewidth(0.9)


def plot_violin_box(ax, df: pd.DataFrame, value_col: str, ylabel: str, cfg: Step3Config, split_order=("train", "val", "test")) -> None:
    positions = np.arange(len(split_order))
    raw_data = []
    colors = []

    for split in split_order:
        vals = finite_array(df.loc[df[cfg_current.split_col] == split, value_col].values)
        raw_data.append(vals if len(vals) else np.array([]))
        colors.append(cfg.colors[split])

    has_enough = sum(len(v) > 1 for v in raw_data) >= 1
    if has_enough:
        vp = ax.violinplot(
            raw_data,
            positions=positions,
            widths=0.75,
            showmeans=False,
            showmedians=True,
            showextrema=False,
        )
        for i, body in enumerate(vp["bodies"]):
            body.set_facecolor(colors[i])
            body.set_alpha(0.35)
            body.set_edgecolor(colors[i])
            body.set_linewidth(0.8)
        if "cmedians" in vp:
            vp["cmedians"].set_color(cfg.colors["black"])
            vp["cmedians"].set_linewidth(1.2)

    _safe_boxplot(ax, raw_data, positions, colors)
    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(split_order)
    beautify_axis(ax, cfg, "y")


# ============================================================
# DATA LOADING / VALIDATION
# ============================================================

def validate_config(cfg: Step3Config) -> None:
    if not os.path.exists(cfg.input_csv):
        raise FileNotFoundError(f"Step 3 input CSV was not found: {cfg.input_csv}")
    if cfg.n_bins_primary < 2:
        raise ValueError("n_bins_primary must be >= 2.")
    if cfg.n_bins_secondary < 2:
        raise ValueError("n_bins_secondary must be >= 2.")
    if len(cfg.bin_labels_primary) != cfg.n_bins_primary:
        raise ValueError("len(bin_labels_primary) must equal n_bins_primary.")
    if len(cfg.bin_labels_secondary) != cfg.n_bins_secondary:
        raise ValueError("len(bin_labels_secondary) must equal n_bins_secondary.")
    if cfg.min_train_examples_per_bin < 1:
        raise ValueError("min_train_examples_per_bin must be >= 1.")
    if cfg.unseen_primary_condition_policy not in {"exclude", "error", "keep"}:
        raise ValueError("unseen_primary_condition_policy must be one of: exclude, error, keep.")
    if cfg.max_allowed_missing_fraction < 0 or cfg.max_allowed_missing_fraction >= 1:
        raise ValueError("max_allowed_missing_fraction must be in [0, 1).")


def load_step2_unique_sequences(cfg: Step3Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.input_csv, low_memory=False)
    if len(df) == 0:
        raise ValueError("Input CSV is empty.")

    # normalize sequence column
    seq_col = choose_existing_column(df, cfg.sequence_col, ())
    if seq_col is None:
        raise ValueError(f"Required sequence column '{cfg.sequence_col}' was not found.")
    if seq_col != cfg.sequence_col:
        df = df.rename(columns={seq_col: cfg.sequence_col})

    df[cfg.sequence_col] = df[cfg.sequence_col].map(normalize_sequence)
    df = df.loc[df[cfg.sequence_col] != ""].reset_index(drop=True)

    # hash column
    hash_col = choose_existing_column(df, cfg.hash_col, ())
    if hash_col is None:
        df[cfg.hash_col] = df[cfg.sequence_col].map(sha256_string)
    elif hash_col != cfg.hash_col:
        df = df.rename(columns={hash_col: cfg.hash_col})

    # split column
    split_col = choose_existing_column(df, cfg.split_col, ("split", "data_split"))
    if split_col is None:
        raise ValueError(f"Required split column '{cfg.split_col}' was not found.")
    if split_col != cfg.split_col:
        df = df.rename(columns={split_col: cfg.split_col})

    df[cfg.split_col] = df[cfg.split_col].astype(str).str.strip().str.lower()
    allowed_splits = {"train", "val", "test"}
    bad_splits = sorted(set(df[cfg.split_col]) - allowed_splits)
    if bad_splits:
        raise ValueError(f"Unexpected split labels found: {bad_splits}")

    if df[cfg.sequence_col].duplicated().any():
        dups = int(df[cfg.sequence_col].duplicated().sum())
        raise ValueError(f"Input Step 2 unique-sequence file is not unique at the sequence level; duplicated sequences found: {dups}")

    return df


def harmonize_descriptor_columns(df: pd.DataFrame, cfg: Step3Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    resolved = {}
    out = df.copy()

    for canonical, candidates in cfg.descriptor_candidates.items():
        found = choose_existing_column(out, canonical, candidates)
        if found is None:
            resolved[canonical] = None
            continue
        resolved[canonical] = found
        if found != canonical:
            out = out.rename(columns={found: canonical})
        out[canonical] = pd.to_numeric(out[canonical], errors="coerce")

    # source column is optional but useful
    source_found = choose_existing_column(out, cfg.source_col, ("source", "source_db", "dataset_source"))
    if source_found is not None and source_found != cfg.source_col:
        out = out.rename(columns={source_found: cfg.source_col})

    return out, resolved


def validate_required_columns(df: pd.DataFrame, cfg: Step3Config) -> None:
    required = [cfg.sequence_col, cfg.hash_col, cfg.split_col, *cfg.primary_condition_keys]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Required columns were not found after harmonization: {missing}")

    # missingness check on primary condition keys
    missingness = df[list(cfg.primary_condition_keys)].isna().mean()
    too_missing = missingness[missingness > cfg.max_allowed_missing_fraction]
    if len(too_missing):
        raise ValueError(
            "Primary condition columns exceed allowed missingness: "
            + ", ".join([f"{k}={v:.3f}" for k, v in too_missing.items()])
        )


# ============================================================
# CONDITION FITTING / APPLICATION
# ============================================================

def robust_quantile_edges(x: pd.Series, n_bins: int) -> np.ndarray:
    vals = pd.to_numeric(x, errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        raise ValueError("Cannot fit bins on an empty numeric series.")
    q = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(vals, q, method="linear")
    edges = np.asarray(edges, dtype=float)

    # enforce monotonicity and uniqueness
    edges = np.maximum.accumulate(edges)
    if np.unique(edges).size < (n_bins + 1):
        uniq = np.unique(vals)
        if uniq.size < n_bins:
            raise ValueError(f"Not enough unique values to form {n_bins} bins.")
        fallback_q = np.linspace(0, uniq.size - 1, n_bins + 1)
        edges = np.interp(fallback_q, np.arange(uniq.size), uniq)

    # tiny epsilon padding to ensure inclusion of extrema
    eps = 1e-9
    edges[0] = edges[0] - eps
    edges[-1] = edges[-1] + eps
    edges = np.maximum.accumulate(edges)
    return edges


def assign_bins(series: pd.Series, edges: np.ndarray, labels: Sequence[str]) -> pd.Categorical:
    out = pd.cut(series, bins=edges, labels=list(labels), include_lowest=True, right=False, ordered=True)
    # last interval closed on the right by epsilon padding
    return out


def fit_condition_schema(df: pd.DataFrame, cfg: Step3Config) -> Dict[str, Any]:
    train_df = df.loc[df[cfg.split_col] == "train"].copy()
    if len(train_df) == 0:
        raise ValueError("No training rows were found in the Step 3 input.")
    schema = {
        "step": "step3_conditioning_schema",
        "created_at": datetime.now().isoformat(),
        "input_csv": os.path.abspath(cfg.input_csv),
        "input_sha256": sha256_file(cfg.input_csv),
        "use_train_only_thresholds": bool(cfg.use_train_only_thresholds),
        "sequence_col": cfg.sequence_col,
        "hash_col": cfg.hash_col,
        "split_col": cfg.split_col,
        "source_col": cfg.source_col if cfg.source_col in df.columns else None,
        "conditions": {},
    }

    all_keys = list(cfg.primary_condition_keys) + [k for k in cfg.secondary_condition_keys if k in df.columns]
    for key in all_keys:
        labels = cfg.bin_labels_primary if key in cfg.primary_condition_keys else cfg.bin_labels_secondary
        n_bins = cfg.n_bins_primary if key in cfg.primary_condition_keys else cfg.n_bins_secondary
        fit_source = train_df[key] if cfg.use_train_only_thresholds else df[key]
        edges = robust_quantile_edges(fit_source, n_bins=n_bins)

        train_bins = assign_bins(train_df[key], edges, labels)
        counts = pd.Series(train_bins).value_counts(dropna=False).reindex(list(labels), fill_value=0)

        small = counts[counts < cfg.min_train_examples_per_bin]
        if len(small):
            raise ValueError(
                f"Condition '{key}' produced train bins below min_train_examples_per_bin={cfg.min_train_examples_per_bin}: "
                + ", ".join([f"{k}={int(v)}" for k, v in small.items()])
            )

        intervals = []
        for i in range(len(labels)):
            intervals.append(format_interval(edges[i], edges[i + 1], last_bin=(i == len(labels) - 1)))

        schema["conditions"][key] = {
            "type": "binned_numeric",
            "is_primary": bool(key in cfg.primary_condition_keys),
            "n_bins": int(n_bins),
            "labels": list(labels),
            "edges": [float(v) for v in edges],
            "interval_labels": intervals,
            "fit_population": "train_only" if cfg.use_train_only_thresholds else "all_splits",
            "train_bin_counts": {str(k): int(v) for k, v in counts.items()},
        }

    return schema


def apply_condition_schema(df: pd.DataFrame, schema: Dict[str, Any], cfg: Step3Config) -> pd.DataFrame:
    out = df.copy()

    for key, meta in schema["conditions"].items():
        edges = np.asarray(meta["edges"], dtype=float)
        labels = meta["labels"]
        out[f"{key}_bin"] = assign_bins(pd.to_numeric(out[key], errors="coerce"), edges, labels).astype("object")
        label_to_idx = {lab: i for i, lab in enumerate(labels)}
        out[f"{key}_bin_idx"] = out[f"{key}_bin"].map(label_to_idx)

    # completeness
    primary_bin_cols = [f"{k}_bin" for k in cfg.primary_condition_keys]
    secondary_bin_cols = [f"{k}_bin" for k in schema["conditions"].keys() if k not in cfg.primary_condition_keys]
    out["is_primary_condition_complete"] = out[primary_bin_cols].notna().all(axis=1)
    out["is_all_condition_complete"] = out[primary_bin_cols + secondary_bin_cols].notna().all(axis=1) if secondary_bin_cols else out["is_primary_condition_complete"]

    # condition vectors / IDs
    def make_cond_id(row, keys):
        parts = []
        for k in keys:
            v = row.get(f"{k}_bin", np.nan)
            parts.append(f"{k}={v}" if pd.notna(v) else f"{k}=NA")
        return "|".join(parts)

    primary_keys = list(cfg.primary_condition_keys)
    all_keys = list(schema["conditions"].keys())

    out["primary_condition_id"] = out.apply(lambda r: make_cond_id(r, primary_keys), axis=1)
    out["full_condition_id"] = out.apply(lambda r: make_cond_id(r, all_keys), axis=1)

    def make_vector(row, keys):
        vect = {}
        for k in keys:
            vect[k] = {
                "raw": None if pd.isna(row.get(k, np.nan)) else float(row[k]),
                "bin": None if pd.isna(row.get(f"{k}_bin", np.nan)) else str(row[f"{k}_bin"]),
                "bin_idx": None if pd.isna(row.get(f"{k}_bin_idx", np.nan)) else int(row[f"{k}_bin_idx"]),
            }
        return json.dumps(vect, ensure_ascii=False, sort_keys=True)

    out["primary_condition_vector_json"] = out.apply(lambda r: make_vector(r, primary_keys), axis=1)
    out["full_condition_vector_json"] = out.apply(lambda r: make_vector(r, all_keys), axis=1)

    return out


def maybe_filter_missing_primary(df: pd.DataFrame, cfg: Step3Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    excluded = df.loc[~df["is_primary_condition_complete"]].copy().reset_index(drop=True)
    if len(excluded):
        excluded["exclusion_reason"] = "incomplete_primary_condition"

    if not cfg.drop_rows_with_missing_primary_conditions:
        return df.copy().reset_index(drop=True), excluded

    keep = df.loc[df["is_primary_condition_complete"]].copy().reset_index(drop=True)
    return keep, excluded


def enforce_train_supported_primary_conditions(
    df: pd.DataFrame,
    cfg: Step3Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if "primary_condition_id" not in df.columns:
        raise ValueError("primary_condition_id column is required before train-support auditing.")

    work = df.copy().reset_index(drop=True)
    train_df = work.loc[work[cfg.split_col] == "train"].copy()
    train_supported_ids = set(train_df["primary_condition_id"].dropna().astype(str).unique().tolist())

    support_df = (
        train_df.groupby("primary_condition_id", as_index=False)
        .agg(train_n_sequences=(cfg.sequence_col, "size"))
        .sort_values(["train_n_sequences", "primary_condition_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

    audit_rows = []
    for split in ["train", "val", "test"]:
        sub = work.loc[work[cfg.split_col] == split].copy()
        ids = set(sub["primary_condition_id"].dropna().astype(str).unique().tolist())
        unseen_ids = sorted([x for x in ids if x not in train_supported_ids]) if split != "train" else []
        unseen_mask = sub["primary_condition_id"].astype(str).isin(unseen_ids) if len(sub) else pd.Series([], dtype=bool)
        audit_rows.append({
            "split": split,
            "n_sequences": int(len(sub)),
            "n_unique_primary_condition_ids": int(len(ids)),
            "n_unique_ids_supported_by_train": int(len(ids & train_supported_ids)),
            "n_unique_ids_not_in_train": int(len(unseen_ids)),
            "n_sequences_with_unseen_primary_condition_id": int(unseen_mask.sum()) if len(sub) else 0,
            "fraction_sequences_with_unseen_primary_condition_id": float(unseen_mask.mean()) if len(sub) else 0.0,
            "unseen_primary_condition_ids": " || ".join(unseen_ids) if unseen_ids else "",
        })
    audit_df = pd.DataFrame(audit_rows)

    unseen_holdout_mask = (
        work[cfg.split_col].isin(["val", "test"]) &
        ~work["primary_condition_id"].astype(str).isin(train_supported_ids)
    )
    excluded_unseen = work.loc[unseen_holdout_mask].copy().reset_index(drop=True)
    if len(excluded_unseen):
        excluded_unseen["exclusion_reason"] = "unseen_primary_condition_id_not_in_train"

    if not cfg.enforce_train_supported_primary_conditions or cfg.unseen_primary_condition_policy == "keep":
        kept = work
    elif cfg.unseen_primary_condition_policy == "error" and len(excluded_unseen):
        unseen_preview = audit_df.loc[audit_df["split"].isin(["val", "test"]), "unseen_primary_condition_ids"]
        unseen_preview = [x for x in unseen_preview.tolist() if str(x).strip()]
        raise ValueError(
            "Validation/test contain primary condition IDs absent from train. "
            f"Examples: {unseen_preview[:5]}"
        )
    else:
        kept = work.loc[~unseen_holdout_mask].copy().reset_index(drop=True)

    return kept, excluded_unseen, audit_df, support_df


# ============================================================
# SUMMARIES / QC
# ============================================================

def build_step3_summary(df: pd.DataFrame, schema: Dict[str, Any], cfg: Step3Config) -> pd.DataFrame:
    split_order = ["train", "val", "test"]
    out = (
        df.groupby(cfg.split_col, as_index=False)
        .agg(
            n_sequences=(cfg.sequence_col, "size"),
            n_primary_condition_complete=("is_primary_condition_complete", "sum"),
            n_all_condition_complete=("is_all_condition_complete", "sum"),
            n_primary_condition_ids=("primary_condition_id", "nunique"),
            n_full_condition_ids=("full_condition_id", "nunique"),
        )
        .rename(columns={cfg.split_col: "split"})
    )
    out = complete_ordered_table(
        out,
        key_col="split",
        ordered_keys=split_order,
        agg_dict={
            "n_sequences": "sum",
            "n_primary_condition_complete": "sum",
            "n_all_condition_complete": "sum",
            "n_primary_condition_ids": "sum",
            "n_full_condition_ids": "sum",
        },
        fill_value=0,
    )
    out["frac_primary_condition_complete"] = out["n_primary_condition_complete"] / out["n_sequences"].clip(lower=1)
    out["frac_all_condition_complete"] = out["n_all_condition_complete"] / out["n_sequences"].clip(lower=1)
    return out


def build_condition_balance_by_split(df: pd.DataFrame, schema: Dict[str, Any], cfg: Step3Config) -> pd.DataFrame:
    records = []
    for key, meta in schema["conditions"].items():
        bin_col = f"{key}_bin"
        levels = meta["labels"]
        for split in ["train", "val", "test"]:
            sub = df.loc[df[cfg.split_col] == split].copy()
            total = max(len(sub), 1)
            counts = sub[bin_col].value_counts(dropna=False)
            for level in levels:
                n = int(counts.get(level, 0))
                records.append({
                    "condition": key,
                    "split": split,
                    "level": level,
                    "n_sequences": n,
                    "fraction": float(n / total),
                })
            n_missing = int(sub[bin_col].isna().sum())
            records.append({
                "condition": key,
                "split": split,
                "level": "missing",
                "n_sequences": n_missing,
                "fraction": float(n_missing / total),
            })
    return pd.DataFrame(records)


def build_condition_combination_summary(df: pd.DataFrame, cfg: Step3Config) -> pd.DataFrame:
    out = (
        df.groupby([cfg.split_col, "primary_condition_id"], as_index=False)
        .agg(n_sequences=(cfg.sequence_col, "size"))
        .rename(columns={cfg.split_col: "split"})
        .sort_values(["split", "n_sequences", "primary_condition_id"], ascending=[True, False, True])
        .reset_index(drop=True)
    )
    return out


def build_missingness_table(df: pd.DataFrame, cfg: Step3Config) -> pd.DataFrame:
    candidate_cols = [cfg.sequence_col, cfg.hash_col, cfg.split_col, cfg.source_col] + list(cfg.descriptor_candidates.keys())
    candidate_cols = [c for c in candidate_cols if c in df.columns]
    records = []
    n = len(df)
    for col in candidate_cols:
        n_missing = int(df[col].isna().sum())
        records.append({
            "column": col,
            "n_missing": n_missing,
            "fraction_missing": float(n_missing / max(n, 1)),
        })
    return pd.DataFrame(records).sort_values(["fraction_missing", "column"], ascending=[False, True]).reset_index(drop=True)


def build_next_step_inputs_table(cfg: Step3Config) -> pd.DataFrame:
    rows = [
        {
            "next_step": "Step 4",
            "purpose": "representation, tokenization, and model-ready preprocessing",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"),
        },
        {
            "next_step": "Step 4",
            "purpose": "frozen condition thresholds / schema reuse",
            "recommended_input": os.path.join(cfg.schemas_dir, "step3_condition_schema.json"),
        },
        {
            "next_step": "Step 4",
            "purpose": "condition balance quality control",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_condition_balance_by_split.csv"),
        },
        {
            "next_step": "Step 5",
            "purpose": "baseline benchmark design",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"),
        },
        {
            "next_step": "Step 6",
            "purpose": "conditional generative model development",
            "recommended_input": os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"),
        },
        {
            "next_step": "Step 7",
            "purpose": "generation-time controllability auditing",
            "recommended_input": os.path.join(cfg.schemas_dir, "step3_condition_schema.json"),
        },
        {
            "next_step": "Step 10",
            "purpose": "Streamlit condition controls and user-facing schema",
            "recommended_input": os.path.join(cfg.schemas_dir, "step3_condition_schema.json"),
        },
    ]
    return pd.DataFrame(rows)


# ============================================================
# FIGURES
# ============================================================

def fig_main_step3_condition_design(df: pd.DataFrame, balance_df: pd.DataFrame, cfg: Step3Config) -> None:
    set_plot_style(cfg)
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.5))
    fig.subplots_adjust(left=0.07, right=0.985, top=0.90, bottom=0.20, wspace=0.40)

    # a: primary condition coverage by split
    ax = axes[0]
    coverage = (
        df.groupby(cfg.split_col, as_index=False)["is_primary_condition_complete"]
        .mean()
        .rename(columns={cfg.split_col: "split", "is_primary_condition_complete": "coverage"})
    )
    coverage = complete_ordered_table(coverage, "split", ["train", "val", "test"], {"coverage": "mean"}, 0)
    vals = coverage["coverage"].to_numpy()
    bars = ax.bar(np.arange(3), vals, color=[cfg.colors[s] for s in ["train", "val", "test"]], edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Primary-condition completeness")
    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(["train", "val", "test"])
    beautify_axis(ax, cfg, "y")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "a")

    # b: number of primary condition IDs by split
    ax = axes[1]
    cond_counts = (
        df.groupby(cfg.split_col, as_index=False)["primary_condition_id"]
        .nunique()
        .rename(columns={cfg.split_col: "split", "primary_condition_id": "n_ids"})
    )
    cond_counts = complete_ordered_table(cond_counts, "split", ["train", "val", "test"], {"n_ids": "sum"}, 0)
    vals = cond_counts["n_ids"].to_numpy()
    bars = ax.bar(np.arange(3), vals, color=[cfg.colors[s] for s in ["train", "val", "test"]], edgecolor=cfg.colors["gray"], linewidth=0.8)
    ax.set_ylabel("Unique primary condition IDs")
    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(["train", "val", "test"])
    beautify_axis(ax, cfg, "y")
    ymax = max(vals) if len(vals) else 1
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + max(ymax * 0.02, 0.1), f"{int(v)}", ha="center", va="bottom", fontsize=8)
    panel_letter(ax, cfg, "b")

    # c: train distribution for primary conditions
    ax = axes[2]
    train_balance = balance_df.loc[(balance_df["split"] == "train") & (balance_df["condition"].isin(cfg.primary_condition_keys)) & (balance_df["level"] != "missing")].copy()
    if len(train_balance):
        pivot = train_balance.pivot_table(index="condition", columns="level", values="fraction", fill_value=0.0)
        col_order = list(cfg.bin_labels_primary)
        pivot = pivot.reindex(columns=col_order, fill_value=0.0)
        y = np.arange(len(pivot.index))
        left = np.zeros(len(y))
        palette = [cfg.colors["train"], cfg.colors["blue2"], cfg.colors["purple"]]
        for level, color in zip(col_order, palette):
            vals = pivot[level].to_numpy()
            ax.barh(y, vals, left=left, color=color, edgecolor="white", linewidth=0.6, label=level)
            left += vals
        ax.set_yticks(y)
        ax.set_yticklabels(pivot.index.tolist())
        ax.set_xlabel("Train fraction")
        ax.set_xlim(0, 1.0)
        ax.legend(frameon=False, fontsize=7, loc="lower right")
    beautify_axis(ax, cfg, "x")
    panel_letter(ax, cfg, "c")

    save_fig(fig, os.path.join(cfg.figures_main_dir, "Fig3_condition_schema_main"), cfg)
    plt.close(fig)


def fig_supp_step3_raw_distributions(df: pd.DataFrame, cfg: Step3Config) -> None:
    set_plot_style(cfg)
    plot_keys = [k for k in ["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"] if k in df.columns]
    n = len(plot_keys)
    if n == 0:
        return

    fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.6))
    axes = np.asarray(axes).reshape(-1)
    fig.subplots_adjust(left=0.08, right=0.985, top=0.95, bottom=0.10, wspace=0.32, hspace=0.34)
    labels = {
        "length": "Length (aa)",
        "net_charge_pH7": "Net charge (pH 7)",
        "hydrophobicity_KD": "Hydrophobicity (Kyte–Doolittle)",
        "entropy": "Shannon entropy (bits)",
    }

    for ax, key, letter in zip(axes, plot_keys, ["a", "b", "c", "d"]):
        plot_violin_box(ax, df, key, labels.get(key, key), cfg)
        panel_letter(ax, cfg, letter)

    for ax in axes[n:]:
        ax.axis("off")

    save_fig(fig, os.path.join(cfg.figures_supp_dir, "FigS7_step3_raw_condition_distributions"), cfg)
    plt.close(fig)


def fig_supp_step3_bin_balance(balance_df: pd.DataFrame, cfg: Step3Config) -> None:
    set_plot_style(cfg)
    conds = sorted([c for c in balance_df["condition"].dropna().unique().tolist() if c in (set(cfg.primary_condition_keys) | set(cfg.secondary_condition_keys))])
    if len(conds) == 0:
        return

    fig, axes = plt.subplots(len(conds), 1, figsize=(8.2, 2.0 * len(conds)), squeeze=False)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.96, bottom=0.10, hspace=0.45)

    split_order = ["train", "val", "test"]
    palette = [cfg.colors["train"], cfg.colors["blue2"], cfg.colors["purple"], cfg.colors["gray"]]

    for i, cond in enumerate(conds):
        ax = axes[i, 0]
        sub = balance_df.loc[(balance_df["condition"] == cond) & (balance_df["level"] != "missing")].copy()
        pivot = sub.pivot_table(index="split", columns="level", values="fraction", fill_value=0.0)
        all_levels = cfg.bin_labels_primary if cond in cfg.primary_condition_keys else cfg.bin_labels_secondary
        pivot = pivot.reindex(index=split_order, columns=list(all_levels), fill_value=0.0)
        bottoms = np.zeros(len(split_order))
        for level, color in zip(pivot.columns.tolist(), palette):
            vals = pivot[level].to_numpy()
            ax.bar(np.arange(len(split_order)), vals, bottom=bottoms, color=color, edgecolor="white", linewidth=0.7, label=level)
            bottoms += vals
        ax.set_ylim(0, 1.0)
        ax.set_ylabel(cond)
        ax.set_xticks(np.arange(len(split_order)))
        ax.set_xticklabels(split_order)
        beautify_axis(ax, cfg, "y")
        if i == 0:
            ax.legend(frameon=False, fontsize=7, ncol=len(pivot.columns), loc="upper right")
        panel_letter(ax, cfg, chr(ord("a") + i))

    save_fig(fig, os.path.join(cfg.figures_supp_dir, "FigS8_step3_condition_bin_balance"), cfg)
    plt.close(fig)


# ============================================================
# REPORTING
# ============================================================

def build_method_report(
    cfg: Step3Config,
    df_in: pd.DataFrame,
    df_out: pd.DataFrame,
    excluded_df: pd.DataFrame,
    unseen_excluded_df: pd.DataFrame,
    schema: Dict[str, Any],
    summary_df: pd.DataFrame,
    balance_df: pd.DataFrame,
    train_support_audit_df: pd.DataFrame,
) -> str:
    lines = []
    lines.append("Step 3 — Conditioning design and controllable label schema")
    lines.append("=" * 80)
    lines.append(f"Input CSV: {os.path.abspath(cfg.input_csv)}")
    lines.append(f"Input SHA256: {sha256_file(cfg.input_csv)}")
    lines.append(f"Total input sequences: {len(df_in):,}")
    lines.append(f"Sequences retained after optional primary-condition filtering: {len(df_out):,}")
    lines.append(f"Sequences excluded for incomplete primary conditions: {len(excluded_df):,}")
    lines.append(f"Sequences excluded for held-out primary condition IDs not supported by train: {len(unseen_excluded_df):,}")
    lines.append("")
    lines.append("Condition schema")
    lines.append("-" * 80)
    for key, meta in schema["conditions"].items():
        lines.append(
            f"{key}: labels={meta['labels']}, fit_population={meta['fit_population']}, "
            f"intervals={meta['interval_labels']}"
        )
    lines.append("")
    lines.append("Split summary")
    lines.append("-" * 80)
    lines.append(summary_df.to_string(index=False))
    lines.append("")
    lines.append("Train-support audit")
    lines.append("-" * 80)
    lines.append(train_support_audit_df.to_string(index=False))
    lines.append("")
    lines.append("Condition balance (head)")
    lines.append("-" * 80)
    lines.append(balance_df.head(24).to_string(index=False))
    lines.append("")
    lines.append("Interpretation")
    lines.append("-" * 80)
    lines.append(
        "This step froze a train-derived conditioning schema for controllable generation. "
        "Primary condition bins were fitted using the training split only and then applied unchanged "
        "to validation and test data. Rows with incomplete primary conditions and, when enabled, held-out "
        "rows carrying primary condition IDs absent from train were exported separately to prevent unsupported "
        "conditioning states from entering downstream model development. The resulting exports are intended "
        "for Step 4 representation building, Step 6 conditional generative model development, Step 7 "
        "controllability auditing, and Step 10 Streamlit condition controls."
    )
    return "\n".join(lines)


def print_next_step_file_locations(cfg: Step3Config) -> None:
    next_inputs = build_next_step_inputs_table(cfg)
    print("=" * 80)
    print("Recommended file locations for later steps")
    print(next_inputs.to_string(index=False))
    print("=" * 80)

    print("Existing Step 3 output tree")
    for root, dirs, files in os.walk(cfg.out_dir):
        rel = os.path.relpath(root, cfg.out_dir)
        print(f"[DIR] {rel}")
        for fn in sorted(files):
            print(f"  - {fn}")


# ============================================================
# PIPELINE
# ============================================================

def run_step3_pipeline(cfg: Step3Config) -> Dict[str, Any]:
    global cfg_current
    cfg_current = cfg  # used by plot helper for split_col

    t0 = time.time()
    np.random.seed(cfg.random_seed)

    validate_config(cfg)
    ensure_output_tree(cfg)

    print("Loading Step 2 unique-sequence split file...")
    df = load_step2_unique_sequences(cfg)
    df, resolved = harmonize_descriptor_columns(df, cfg)
    validate_required_columns(df, cfg)

    print("Fitting train-derived condition schema...")
    schema = fit_condition_schema(df, cfg)

    print("Applying condition schema to all splits...")
    conditioned_df = apply_condition_schema(df, schema, cfg)
    conditioned_df, excluded_missing_df = maybe_filter_missing_primary(conditioned_df, cfg)

    print("Auditing train-supported primary condition IDs...")
    conditioned_df, excluded_unseen_df, train_support_audit_df, train_primary_support_df = enforce_train_supported_primary_conditions(conditioned_df, cfg)

    excluded_frames = [x for x in [excluded_missing_df, excluded_unseen_df] if len(x)]
    excluded_df = pd.concat(excluded_frames, ignore_index=True) if excluded_frames else pd.DataFrame(columns=list(conditioned_df.columns) + ["exclusion_reason"])

    print("Building summaries and QC tables...")
    summary_df = build_step3_summary(conditioned_df, schema, cfg)
    balance_df = build_condition_balance_by_split(conditioned_df, schema, cfg)
    combination_df = build_condition_combination_summary(conditioned_df, cfg)
    missingness_df = build_missingness_table(df, cfg)
    next_step_df = build_next_step_inputs_table(cfg)

    print("Writing exports...")
    save_csv(conditioned_df, os.path.join(cfg.tables_dir, "step3_conditioned_unique_sequences.csv"))
    save_csv(conditioned_df.loc[conditioned_df[cfg.split_col] == "train"].copy(), os.path.join(cfg.tables_dir, "step3_train_conditioned_unique_sequences.csv"))
    save_csv(conditioned_df.loc[conditioned_df[cfg.split_col] == "val"].copy(), os.path.join(cfg.tables_dir, "step3_val_conditioned_unique_sequences.csv"))
    save_csv(conditioned_df.loc[conditioned_df[cfg.split_col] == "test"].copy(), os.path.join(cfg.tables_dir, "step3_test_conditioned_unique_sequences.csv"))
    save_csv(summary_df, os.path.join(cfg.tables_dir, "step3_split_condition_summary.csv"))
    save_csv(balance_df, os.path.join(cfg.tables_dir, "step3_condition_balance_by_split.csv"))
    save_csv(combination_df, os.path.join(cfg.tables_dir, "step3_primary_condition_combinations.csv"))
    save_csv(train_support_audit_df, os.path.join(cfg.tables_dir, "step3_train_support_audit.csv"))
    save_csv(train_primary_support_df, os.path.join(cfg.tables_dir, "step3_train_primary_condition_support.csv"))
    save_csv(missingness_df, os.path.join(cfg.tables_dir, "step3_missingness_table.csv"))
    save_csv(excluded_df, os.path.join(cfg.tables_dir, "step3_excluded_sequences.csv"))
    save_csv(excluded_missing_df, os.path.join(cfg.tables_dir, "step3_excluded_incomplete_primary_conditions.csv"))
    save_csv(excluded_unseen_df, os.path.join(cfg.tables_dir, "step3_excluded_unseen_primary_condition_ids.csv"))
    save_csv(next_step_df, os.path.join(cfg.tables_dir, "step3_next_step_input_locations.csv"))

    save_json(schema, os.path.join(cfg.schemas_dir, "step3_condition_schema.json"))

    run_meta = {
        "config": asdict(cfg),
        "system": system_report(cfg),
        "resolved_descriptor_columns": resolved,
        "runtime_seconds": float(time.time() - t0),
        "n_input_sequences": int(len(df)),
        "n_retained_sequences": int(len(conditioned_df)),
        "n_excluded_sequences": int(len(excluded_df)),
        "n_excluded_incomplete_primary_conditions": int(len(excluded_missing_df)),
        "n_excluded_unseen_primary_condition_ids": int(len(excluded_unseen_df)),
        "n_primary_condition_ids": int(conditioned_df["primary_condition_id"].nunique()),
        "n_full_condition_ids": int(conditioned_df["full_condition_id"].nunique()),
    }
    save_json(run_meta, os.path.join(cfg.reports_dir, "step3_run_metadata.json"))

    method_text = build_method_report(
        cfg=cfg,
        df_in=df,
        df_out=conditioned_df,
        excluded_df=excluded_missing_df,
        unseen_excluded_df=excluded_unseen_df,
        schema=schema,
        summary_df=summary_df,
        balance_df=balance_df,
        train_support_audit_df=train_support_audit_df,
    )
    save_text(method_text, os.path.join(cfg.reports_dir, "step3_method_and_results_summary.txt"))

    print("Generating figures...")
    fig_main_step3_condition_design(conditioned_df, balance_df, cfg)
    fig_supp_step3_raw_distributions(conditioned_df, cfg)
    fig_supp_step3_bin_balance(balance_df, cfg)

    print("Step 3 completed successfully.")
    print(f"Output directory: {cfg.out_dir}")
    print("-" * 80)
    print(summary_df.to_string(index=False))
    print("-" * 80)
    print("Train-support audit:")
    print(train_support_audit_df.to_string(index=False))
    print_next_step_file_locations(cfg)

    return {
        "input_df": df,
        "conditioned_df": conditioned_df,
        "excluded_df": excluded_df,
        "schema": schema,
        "summary_df": summary_df,
        "balance_df": balance_df,
        "combination_df": combination_df,
        "train_support_audit_df": train_support_audit_df,
        "train_primary_support_df": train_primary_support_df,
        "missingness_df": missingness_df,
        "next_step_df": next_step_df,
        "run_meta": run_meta,
    }


# ============================================================
# NOTEBOOK ENTRYPOINT
# ============================================================

def main_notebook(
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2/splits/step2_unique_sequences_with_splits.csv",
    out_dir_name: str = "step3",
    project_root_override: Optional[str] = None,
    random_seed: int = 42,
    n_bins_primary: int = 3,
    n_bins_secondary: int = 3,
    min_train_examples_per_bin: int = 50,
    drop_rows_with_missing_primary_conditions: bool = True,
    enforce_train_supported_primary_conditions: bool = True,
    unseen_primary_condition_policy: str = "exclude",
) -> Dict[str, Any]:
    cfg = Step3Config(
        input_csv=input_csv,
        out_dir_name=out_dir_name,
        project_root_override=project_root_override,
        random_seed=random_seed,
        n_bins_primary=n_bins_primary,
        n_bins_secondary=n_bins_secondary,
        min_train_examples_per_bin=min_train_examples_per_bin,
        drop_rows_with_missing_primary_conditions=drop_rows_with_missing_primary_conditions,
        enforce_train_supported_primary_conditions=enforce_train_supported_primary_conditions,
        unseen_primary_condition_policy=unseen_primary_condition_policy,
    )
    return run_step3_pipeline(cfg)


In [ ]:
results = main_notebook(
    input_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step2/splits/step2_unique_sequences_with_splits.csv",
    out_dir_name="step3",
    drop_rows_with_missing_primary_conditions=True,
    enforce_train_supported_primary_conditions=True,
    unseen_primary_condition_policy="exclude",
)

step 4: Representation, tokenization, and model-ready preprocessing

In [ ]:
from __future__ import annotations

import json
import math
import os
import hashlib
import platform
from dataclasses import dataclass, asdict, field
from datetime import datetime
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PepCVAE / AngioPep-VAE — Step 4
# Representation, tokenization, and model-ready preprocessing
# Nature-aligned, Jupyter-ready version
# ============================================================


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Step4Config:
    input_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step3/tables/step3_conditioned_unique_sequences.csv"
    out_dir_name: str = "step4"
    project_root_override: Optional[str] = None

    # expected columns / fallback candidates
    sequence_col: str = "sequence"
    hash_col: str = "sequence_sha256"
    split_col: str = "assigned_split"
    primary_condition_id_col: str = "primary_condition_id"

    primary_condition_component_candidates: Dict[str, Tuple[str, ...]] = field(default_factory=lambda: {
        "length_bin": ("length_bin",),
        "net_charge_pH7_bin": ("net_charge_pH7_bin", "charge_bin"),
        "hydrophobicity_KD_bin": ("hydrophobicity_KD_bin", "hydrophobicity_bin"),
    })

    numeric_descriptor_candidates: Dict[str, Tuple[str, ...]] = field(default_factory=lambda: {
        "length": ("length", "seq_length", "peptide_length"),
        "net_charge_pH7": ("net_charge_pH7", "net_charge", "charge", "charge_pH7"),
        "hydrophobicity_KD": ("hydrophobicity_KD", "hydrophobicity", "hydrophobicity_kd"),
        "entropy": ("entropy", "shannon_entropy", "sequence_entropy"),
        "dominant_aa_frac": ("dominant_aa_frac", "dominant_aa_fraction", "dominant_residue_fraction"),
    })

    allowed_aas: str = "ACDEFGHIKLMNPQRSTVWY"
    pad_token: str = "<PAD>"
    bos_token: str = "<BOS>"
    eos_token: str = "<EOS>"
    unk_token: str = "<UNK>"

    # model-facing representation policy
    max_length_policy: str = "train_exact_max"  # train_exact_max | train_percentile
    train_length_percentile: float = 99.5
    min_allowed_max_model_length: int = 8
    enforce_no_truncation_in_final: bool = True

    # baseline feature branch
    include_baseline_composition: bool = True
    include_numeric_descriptors_in_baseline: bool = True
    normalize_numeric_features: bool = True

    # reproducibility / validation
    random_seed: int = 42
    fail_on_invalid_residues: bool = True
    fail_on_missing_primary_condition_id: bool = True
    fail_on_duplicate_hashes: bool = False
    low_memory: bool = False

    # plotting
    png_dpi: int = 300
    tif_dpi: int = 600
    font_family: str = "sans-serif"
    font_list: Tuple[str, ...] = ("Arial", "Helvetica", "DejaVu Sans")
    font_size: int = 8
    title_size: int = 9
    axis_label_size: int = 8
    tick_label_size: int = 8
    axis_linewidth: float = 0.9

    colors: Dict[str, str] = field(default_factory=lambda: {
        "train": "#4E79A7",
        "val": "#F28E2B",
        "test": "#E15759",
        "gray": "#707070",
        "black": "#111111",
        "grid": "#E6E6E6",
        "green": "#59A14F",
        "purple": "#B07AA1",
        "blue2": "#76B7B2",
    })

    @property
    def project_root(self) -> str:
        if self.project_root_override is not None:
            return os.path.abspath(self.project_root_override)
        # input_csv = .../step3/tables/file.csv -> project root is parent of step3
        return os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(self.input_csv))))

    @property
    def out_dir(self) -> str:
        return os.path.join(self.project_root, self.out_dir_name)

    @property
    def tables_dir(self) -> str:
        return os.path.join(self.out_dir, "tables")

    @property
    def reports_dir(self) -> str:
        return os.path.join(self.out_dir, "reports")

    @property
    def figures_main_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_main")

    @property
    def figures_supp_dir(self) -> str:
        return os.path.join(self.out_dir, "figures_supplementary")

    @property
    def artifacts_dir(self) -> str:
        return os.path.join(self.out_dir, "artifacts")

    @property
    def arrays_dir(self) -> str:
        return os.path.join(self.out_dir, "arrays")


# ============================================================
# GENERAL HELPERS
# ============================================================

def ensure_dir(path: str) -> str:
    path = os.path.abspath(path)
    os.makedirs(path, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step4Config) -> None:
    for path in [cfg.out_dir, cfg.tables_dir, cfg.reports_dir, cfg.figures_main_dir, cfg.figures_supp_dir, cfg.artifacts_dir, cfg.arrays_dir]:
        ensure_dir(path)


def save_csv(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)


def save_text(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def json_default(obj: Any):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(list(obj))
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def sha256_file(path: str, chunk_size: int = 2**20) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


def normalize_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(seq.split())


def choose_existing_column(df: pd.DataFrame, preferred: str, alternatives: Sequence[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for c in (preferred, *alternatives):
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None


def complete_ordered_table(df: pd.DataFrame, key_col: str, ordered_keys: Sequence[str]) -> pd.DataFrame:
    order_df = pd.DataFrame({key_col: list(ordered_keys)})
    if df is None or len(df) == 0:
        return order_df
    return order_df.merge(df, on=key_col, how="left")


def finite_array(values: Sequence[Any]) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def system_report(cfg: Step4Config) -> Dict[str, Any]:
    return {
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "input_csv": os.path.abspath(cfg.input_csv),
        "input_sha256": sha256_file(cfg.input_csv),
    }


# ============================================================
# PLOT HELPERS
# ============================================================

def set_plot_style(cfg: Step4Config) -> None:
    plt.rcParams.update({
        "font.family": cfg.font_family,
        "font.sans-serif": list(cfg.font_list),
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "axes.linewidth": cfg.axis_linewidth,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.bbox": "tight",
    })


def panel_label(ax, label: str) -> None:
    ax.text(-0.18, 1.06, label, transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")


def save_figure(fig: plt.Figure, base_path_without_ext: str, cfg: Step4Config) -> None:
    fig.savefig(base_path_without_ext + ".png", dpi=cfg.png_dpi)
    fig.savefig(base_path_without_ext + ".pdf")
    fig.savefig(base_path_without_ext + ".tiff", dpi=cfg.tif_dpi)
    plt.close(fig)


# ============================================================
# LOADING AND VALIDATION
# ============================================================

def load_input_table(cfg: Step4Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.input_csv, low_memory=cfg.low_memory)
    if df.empty:
        raise ValueError("Input Step 3 conditioned table is empty.")
    return df


def resolve_columns(df: pd.DataFrame, cfg: Step4Config) -> Dict[str, str]:
    sequence_col = choose_existing_column(df, cfg.sequence_col, ())
    if sequence_col is None:
        raise KeyError(f"Could not find sequence column '{cfg.sequence_col}'.")

    hash_col = choose_existing_column(df, cfg.hash_col, ())
    split_col = choose_existing_column(df, cfg.split_col, ())
    if split_col is None:
        raise KeyError(f"Could not find split column '{cfg.split_col}'.")

    primary_condition_id_col = choose_existing_column(df, cfg.primary_condition_id_col, ())
    if primary_condition_id_col is None:
        raise KeyError(f"Could not find primary condition ID column '{cfg.primary_condition_id_col}'.")

    component_cols: Dict[str, str] = {}
    for logical_name, alts in cfg.primary_condition_component_candidates.items():
        col = choose_existing_column(df, logical_name, alts)
        if col is None:
            raise KeyError(f"Could not find required primary condition component column for '{logical_name}'.")
        component_cols[logical_name] = col

    numeric_cols: Dict[str, Optional[str]] = {}
    for logical_name, alts in cfg.numeric_descriptor_candidates.items():
        numeric_cols[logical_name] = choose_existing_column(df, logical_name, alts)

    return {
        "sequence_col": sequence_col,
        "hash_col": hash_col or "",
        "split_col": split_col,
        "primary_condition_id_col": primary_condition_id_col,
        **{f"component::{k}": v for k, v in component_cols.items()},
        **{f"numeric::{k}": (v or "") for k, v in numeric_cols.items()},
    }


def validate_and_prepare_input(df: pd.DataFrame, cfg: Step4Config, cols: Dict[str, str]) -> pd.DataFrame:
    work = df.copy()
    seq_col = cols["sequence_col"]
    split_col = cols["split_col"]
    pcid_col = cols["primary_condition_id_col"]

    work[seq_col] = work[seq_col].map(normalize_sequence)
    if (work[seq_col] == "").any():
        bad_n = int((work[seq_col] == "").sum())
        raise ValueError(f"Found {bad_n} empty normalized sequences in Step 3 conditioned table.")

    if cols["hash_col"] == "":
        work["sequence_sha256"] = work[seq_col].map(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
        cols["hash_col"] = "sequence_sha256"

    if cfg.fail_on_missing_primary_condition_id and work[pcid_col].isna().any():
        bad_n = int(work[pcid_col].isna().sum())
        raise ValueError(f"Found {bad_n} rows with missing primary condition ID.")

    expected_splits = ["train", "val", "test"]
    observed = set(work[split_col].astype(str))
    missing = [s for s in expected_splits if s not in observed]
    if missing:
        raise ValueError(f"Missing expected split(s): {missing}")

    allowed = set(cfg.allowed_aas)
    invalid_records = []
    for i, seq in enumerate(work[seq_col].astype(str)):
        bad = sorted(set(seq) - allowed)
        if bad:
            invalid_records.append((i, seq, "".join(bad)))
            if len(invalid_records) >= 10:
                break
    if invalid_records and cfg.fail_on_invalid_residues:
        msg = "; ".join([f"row={i}, invalid={bad}" for i, _, bad in invalid_records])
        raise ValueError(f"Found invalid residues outside canonical alphabet: {msg}")

    dup_hashes = work[cols["hash_col"]].duplicated().sum()
    if dup_hashes and cfg.fail_on_duplicate_hashes:
        raise ValueError(f"Found {int(dup_hashes)} duplicated sequence hashes in Step 3 table.")

    work = work.sort_values([split_col, cols["hash_col"], seq_col], kind="mergesort").reset_index(drop=True)
    return work


# ============================================================
# REPRESENTATION / TOKENIZATION
# ============================================================

def determine_max_model_length(train_lengths: np.ndarray, cfg: Step4Config) -> Dict[str, Any]:
    if len(train_lengths) == 0:
        raise ValueError("Training split has no sequences.")
    train_seq_max = int(np.max(train_lengths))
    if cfg.max_length_policy == "train_exact_max":
        seq_cap = train_seq_max
    elif cfg.max_length_policy == "train_percentile":
        seq_cap = int(math.ceil(np.percentile(train_lengths, cfg.train_length_percentile)))
        seq_cap = max(seq_cap, cfg.min_allowed_max_model_length - 2)
    else:
        raise ValueError(f"Unknown max_length_policy: {cfg.max_length_policy}")

    max_model_length = int(seq_cap + 2)  # + BOS + EOS
    max_model_length = max(max_model_length, cfg.min_allowed_max_model_length)
    return {
        "policy": cfg.max_length_policy,
        "train_length_percentile": cfg.train_length_percentile,
        "train_max_sequence_length": train_seq_max,
        "chosen_max_sequence_length": int(seq_cap),
        "max_model_length_including_special_tokens": max_model_length,
    }


def build_vocabulary(cfg: Step4Config) -> Dict[str, Any]:
    tokens = [cfg.pad_token, cfg.bos_token, cfg.eos_token, cfg.unk_token] + list(cfg.allowed_aas)
    token_to_idx = {tok: i for i, tok in enumerate(tokens)}
    idx_to_token = {i: tok for tok, i in token_to_idx.items()}
    return {
        "tokens": tokens,
        "token_to_idx": token_to_idx,
        "idx_to_token": idx_to_token,
        "pad_idx": token_to_idx[cfg.pad_token],
        "bos_idx": token_to_idx[cfg.bos_token],
        "eos_idx": token_to_idx[cfg.eos_token],
        "unk_idx": token_to_idx[cfg.unk_token],
    }


def encode_sequence(seq: str, vocab: Dict[str, Any], max_model_length: int) -> Tuple[np.ndarray, np.ndarray, int, int]:
    token_to_idx = vocab["token_to_idx"]
    bos_idx = vocab["bos_idx"]
    eos_idx = vocab["eos_idx"]
    pad_idx = vocab["pad_idx"]
    unk_idx = vocab["unk_idx"]

    ids = [bos_idx]
    unk_count = 0
    for aa in seq:
        idx = token_to_idx.get(aa, unk_idx)
        if idx == unk_idx:
            unk_count += 1
        ids.append(idx)
    ids.append(eos_idx)

    truncated = 0
    if len(ids) > max_model_length:
        truncated = 1
        ids = ids[:max_model_length]
        ids[-1] = eos_idx

    attention = [1] * len(ids)
    pad_needed = max_model_length - len(ids)
    if pad_needed > 0:
        ids += [pad_idx] * pad_needed
        attention += [0] * pad_needed

    return np.asarray(ids, dtype=np.int32), np.asarray(attention, dtype=np.int8), int(len(seq)), int(unk_count + 0 * truncated)


def encode_primary_conditions(df: pd.DataFrame, cols: Dict[str, str]) -> Tuple[pd.DataFrame, Dict[str, Dict[str, int]]]:
    out = df.copy()
    mappings: Dict[str, Dict[str, int]] = {}

    pcid_col = cols["primary_condition_id_col"]
    unique_pcid = sorted(out[pcid_col].astype(str).unique().tolist())
    mappings[pcid_col] = {v: i for i, v in enumerate(unique_pcid)}
    out[pcid_col + "_idx"] = out[pcid_col].astype(str).map(mappings[pcid_col]).astype(int)

    for logical_name in ["length_bin", "net_charge_pH7_bin", "hydrophobicity_KD_bin"]:
        col = cols[f"component::{logical_name}"]
        vals = sorted(out[col].astype(str).unique().tolist())
        mappings[col] = {v: i for i, v in enumerate(vals)}
        out[col + "_idx"] = out[col].astype(str).map(mappings[col]).astype(int)

    return out, mappings


# ============================================================
# BASELINE FEATURE BRANCH
# ============================================================

def amino_acid_composition(seq: str, aas: str) -> Dict[str, float]:
    n = max(len(seq), 1)
    return {f"aa_frac_{aa}": seq.count(aa) / n for aa in aas}


def build_baseline_feature_table(df: pd.DataFrame, cfg: Step4Config, cols: Dict[str, str]) -> Tuple[pd.DataFrame, List[str], Dict[str, Dict[str, float]]]:
    seq_col = cols["sequence_col"]
    split_col = cols["split_col"]
    hash_col = cols["hash_col"]

    records: List[Dict[str, Any]] = []
    for row in df.itertuples(index=False):
        row_dict = row._asdict()
        seq = str(row_dict[seq_col])
        rec = {
            hash_col: row_dict[hash_col],
            seq_col: seq,
            split_col: row_dict[split_col],
            cols["primary_condition_id_col"]: row_dict[cols["primary_condition_id_col"]],
            cols["primary_condition_id_col"] + "_idx": row_dict[cols["primary_condition_id_col"] + "_idx"],
        }
        if cfg.include_baseline_composition:
            rec.update(amino_acid_composition(seq, cfg.allowed_aas))
        if cfg.include_numeric_descriptors_in_baseline:
            for logical_name in cfg.numeric_descriptor_candidates:
                col = cols.get(f"numeric::{logical_name}", "")
                if col:
                    rec[logical_name] = pd.to_numeric(row_dict[col], errors="coerce")
        records.append(rec)

    feat_df = pd.DataFrame(records)

    feature_cols = [c for c in feat_df.columns if c not in {hash_col, seq_col, split_col, cols["primary_condition_id_col"], cols["primary_condition_id_col"] + "_idx"}]
    normalization: Dict[str, Dict[str, float]] = {}

    if cfg.normalize_numeric_features and feature_cols:
        train_mask = feat_df[split_col].astype(str) == "train"
        for c in feature_cols:
            if not pd.api.types.is_numeric_dtype(feat_df[c]):
                continue
            train_vals = pd.to_numeric(feat_df.loc[train_mask, c], errors="coerce")
            mu = float(train_vals.mean()) if np.isfinite(train_vals.mean()) else 0.0
            sd = float(train_vals.std(ddof=0)) if np.isfinite(train_vals.std(ddof=0)) else 1.0
            if sd == 0.0:
                sd = 1.0
            normalization[c] = {"mean_train": mu, "std_train": sd}
            feat_df[c + "_z"] = (pd.to_numeric(feat_df[c], errors="coerce") - mu) / sd

    return feat_df, feature_cols, normalization


# ============================================================
# QC SUMMARIES
# ============================================================

def build_length_and_token_qc(df: pd.DataFrame, cfg: Step4Config, cols: Dict[str, str], length_policy: Dict[str, Any], encoded: Dict[str, Dict[str, Any]], vocab: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split_col = cols["split_col"]
    seq_col = cols["sequence_col"]

    seq_lengths = df[seq_col].astype(str).map(len).astype(int)
    qc_records = []
    length_records = []
    token_records = []

    for split in ["train", "val", "test"]:
        mask = df[split_col].astype(str) == split
        split_df = df.loc[mask].copy()
        lens = split_df[seq_col].astype(str).map(len).astype(int).to_numpy()
        enc = encoded[split]
        trunc_flags = enc["truncated_flag"]
        attention = enc["attention_mask"]
        token_ids = enc["token_ids"]
        n = len(split_df)

        length_records.append({
            "split": split,
            "n_sequences": n,
            "min_length": int(np.min(lens)) if n else np.nan,
            "median_length": float(np.median(lens)) if n else np.nan,
            "mean_length": float(np.mean(lens)) if n else np.nan,
            "max_length": int(np.max(lens)) if n else np.nan,
            "chosen_max_sequence_length": length_policy["chosen_max_sequence_length"],
            "max_model_length_including_special_tokens": length_policy["max_model_length_including_special_tokens"],
            "n_truncated": int(np.sum(trunc_flags)) if n else 0,
            "fraction_truncated": float(np.mean(trunc_flags)) if n else 0.0,
            "mean_padding_fraction": float(np.mean(1.0 - attention.mean(axis=1))) if n else 0.0,
        })

        total_nonpad = int(attention.sum()) if n else 0
        unk_id = vocab["unk_idx"]
        unk_count = int(((token_ids == unk_id) & (attention == 1)).sum()) if n else 0
        token_records.append({
            "split": split,
            "n_sequences": n,
            "vocab_size": len(vocab["tokens"]),
            "unknown_token_count": unk_count,
            "unknown_token_fraction_nonpad": float(unk_count / total_nonpad) if total_nonpad > 0 else 0.0,
            "nonpad_token_count": total_nonpad,
        })

        qc_records.append({
            "split": split,
            "n_sequences": n,
            "n_primary_condition_ids": int(split_df[cols["primary_condition_id_col"]].nunique()),
            "fraction_primary_condition_id_missing": float(split_df[cols["primary_condition_id_col"]].isna().mean()) if n else 0.0,
            "fraction_truncated": float(np.mean(trunc_flags)) if n else 0.0,
            "unknown_token_fraction_nonpad": float(unk_count / total_nonpad) if total_nonpad > 0 else 0.0,
            "mean_padding_fraction": float(np.mean(1.0 - attention.mean(axis=1))) if n else 0.0,
        })

    return pd.DataFrame(length_records), pd.DataFrame(token_records), pd.DataFrame(qc_records)


def build_step4_summary(df: pd.DataFrame, cols: Dict[str, str]) -> pd.DataFrame:
    split_col = cols["split_col"]
    seq_col = cols["sequence_col"]
    pcid_col = cols["primary_condition_id_col"]
    out = (
        df.groupby(split_col, as_index=False)
        .agg(
            n_sequences=(seq_col, "size"),
            n_unique_primary_condition_ids=(pcid_col, "nunique"),
            min_length=(seq_col, lambda s: int(s.astype(str).map(len).min())),
            median_length=(seq_col, lambda s: float(np.median(s.astype(str).map(len)))),
            max_length=(seq_col, lambda s: int(s.astype(str).map(len).max())),
        )
        .rename(columns={split_col: "split"})
    )
    return complete_ordered_table(out, "split", ["train", "val", "test"])


def build_next_step_locations(cfg: Step4Config) -> pd.DataFrame:
    rows = [
        {
            "next_step": "Step 5",
            "purpose": "baseline predictive benchmarks using frozen Step 4 model-ready data",
            "recommended_input": os.path.join(cfg.tables_dir, "step4_model_ready_sequences.csv"),
        },
        {
            "next_step": "Step 5",
            "purpose": "baseline feature matrix",
            "recommended_input": os.path.join(cfg.tables_dir, "step4_baseline_features.csv"),
        },
        {
            "next_step": "Step 6",
            "purpose": "PepCVAE sequence token arrays",
            "recommended_input": os.path.join(cfg.arrays_dir, "step4_tokenized_sequences.npz"),
        },
        {
            "next_step": "Step 6",
            "purpose": "PepCVAE frozen vocabulary and special-token policy",
            "recommended_input": os.path.join(cfg.artifacts_dir, "step4_vocab.json"),
        },
        {
            "next_step": "Step 6",
            "purpose": "PepCVAE condition ID mappings",
            "recommended_input": os.path.join(cfg.artifacts_dir, "step4_condition_id_mapping.json"),
        },
        {
            "next_step": "Step 7",
            "purpose": "generation-time schema reuse",
            "recommended_input": os.path.join(cfg.artifacts_dir, "step4_preprocessing_contract.json"),
        },
        {
            "next_step": "Step 8",
            "purpose": "reproducibility and deployment handoff",
            "recommended_input": os.path.join(cfg.artifacts_dir, "step4_artifact_hashes.json"),
        },
    ]
    return pd.DataFrame(rows)


def print_next_step_file_locations(cfg: Step4Config) -> None:
    df = build_next_step_locations(cfg)
    print("=" * 80)
    print("Recommended file locations for later steps")
    print(df.to_string(index=False))


# ============================================================
# FIGURES
# ============================================================

def make_main_figure(df: pd.DataFrame, cfg: Step4Config, cols: Dict[str, str], length_summary: pd.DataFrame, token_summary: pd.DataFrame, qc_summary: pd.DataFrame, length_policy: Dict[str, Any]) -> str:
    set_plot_style(cfg)
    seq_col = cols["sequence_col"]
    split_col = cols["split_col"]

    fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.0))

    # a) length distributions + chosen cap
    ax = axes[0]
    bins = np.arange(0, max(df[seq_col].astype(str).map(len).max() + 2, 4), 1)
    for split in ["train", "val", "test"]:
        vals = df.loc[df[split_col].astype(str) == split, seq_col].astype(str).map(len).to_numpy()
        ax.hist(vals, bins=bins, density=True, histtype="step", linewidth=1.3, label=split, color=cfg.colors[split])
    ax.axvline(length_policy["chosen_max_sequence_length"], linestyle="--", linewidth=1.1, color=cfg.colors["black"])
    ax.set_xlabel("Sequence length")
    ax.set_ylabel("Density")
    ax.set_title("Length distribution and cap")
    ax.legend(frameon=False)
    panel_label(ax, "a")

    # b) token coverage / unknown-token rate
    ax = axes[1]
    x = np.arange(3)
    unk = token_summary.set_index("split").reindex(["train", "val", "test"])["unknown_token_fraction_nonpad"].fillna(0.0).to_numpy()
    vocab = token_summary.set_index("split").reindex(["train", "val", "test"])["vocab_size"].fillna(0).to_numpy()
    bars = ax.bar(x, unk, color=[cfg.colors[s] for s in ["train", "val", "test"]], width=0.7)
    ax.set_xticks(x, ["train", "val", "test"])
    ax.set_ylabel("Unknown-token fraction")
    ax.set_title("Token coverage")
    ax.set_ylim(0, max(0.01, float(unk.max()) * 1.2 + 1e-6))
    for i, b in enumerate(bars):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + ax.get_ylim()[1]*0.03, f"V={int(vocab[i])}", ha="center", va="bottom", fontsize=7)
    panel_label(ax, "b")

    # c) split-safe representation summary
    ax = axes[2]
    summary = qc_summary.set_index("split").reindex(["train", "val", "test"])
    nseq = summary["n_sequences"].fillna(0).to_numpy(dtype=float)
    npcid = summary["n_primary_condition_ids"].fillna(0).to_numpy(dtype=float)
    x = np.arange(3)
    ax.bar(x - 0.18, nseq / nseq.max(), width=0.35, color=[cfg.colors[s] for s in ["train", "val", "test"]], label="Sequences (scaled)")
    ax.bar(x + 0.18, npcid / max(npcid.max(), 1), width=0.35, color=cfg.colors["gray"], label="Primary condition IDs (scaled)")
    ax.set_xticks(x, ["train", "val", "test"])
    ax.set_ylabel("Scaled fraction")
    ax.set_ylim(0, 1.1)
    ax.set_title("Representation integrity")
    ax.legend(frameon=False, fontsize=7)
    panel_label(ax, "c")

    fig.tight_layout()
    out_base = os.path.join(cfg.figures_main_dir, "Fig4_model_ready_preprocessing_main")
    save_figure(fig, out_base, cfg)
    return out_base + ".png"


def make_supp_fig_padding_truncation(length_summary: pd.DataFrame, cfg: Step4Config) -> str:
    set_plot_style(cfg)
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))
    order = ["train", "val", "test"]
    tmp = length_summary.set_index("split").reindex(order)

    ax = axes[0]
    ax.bar(order, tmp["fraction_truncated"].fillna(0.0).to_numpy(), color=[cfg.colors[s] for s in order])
    ax.set_ylabel("Fraction truncated")
    ax.set_title("Truncation diagnostics")
    panel_label(ax, "a")

    ax = axes[1]
    ax.bar(order, tmp["mean_padding_fraction"].fillna(0.0).to_numpy(), color=[cfg.colors[s] for s in order])
    ax.set_ylabel("Mean padding fraction")
    ax.set_title("Padding burden by split")
    panel_label(ax, "b")

    fig.tight_layout()
    out_base = os.path.join(cfg.figures_supp_dir, "FigS7_padding_truncation_diagnostics")
    save_figure(fig, out_base, cfg)
    return out_base + ".png"


def make_supp_fig_token_freq(df: pd.DataFrame, cfg: Step4Config, cols: Dict[str, str]) -> str:
    set_plot_style(cfg)
    seq_col = cols["sequence_col"]
    split_col = cols["split_col"]
    aas = list(cfg.allowed_aas)
    rows = []
    for split in ["train", "val", "test"]:
        seqs = df.loc[df[split_col].astype(str) == split, seq_col].astype(str).tolist()
        concat = "".join(seqs)
        n = max(len(concat), 1)
        for aa in aas:
            rows.append({"split": split, "amino_acid": aa, "fraction": concat.count(aa) / n})
    plot_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8.2, 3.0))
    width = 0.26
    x = np.arange(len(aas))
    for i, split in enumerate(["train", "val", "test"]):
        vals = plot_df.loc[plot_df["split"] == split, "fraction"].to_numpy()
        ax.bar(x + (i - 1) * width, vals, width=width, color=cfg.colors[split], label=split)
    ax.set_xticks(x, aas)
    ax.set_ylabel("Fraction")
    ax.set_title("Amino-acid composition after preprocessing")
    ax.legend(frameon=False, ncol=3)
    panel_label(ax, "a")
    fig.tight_layout()
    out_base = os.path.join(cfg.figures_supp_dir, "FigS8_token_frequency_composition")
    save_figure(fig, out_base, cfg)
    return out_base + ".png"


# ============================================================
# REPORTING
# ============================================================

def build_method_report(cfg: Step4Config, cols: Dict[str, str], length_policy: Dict[str, Any], vocab: Dict[str, Any], normalization: Dict[str, Dict[str, float]], summary_df: pd.DataFrame, qc_df: pd.DataFrame) -> str:
    lines = []
    lines.append("Step 4 — Representation, tokenization, and model-ready preprocessing")
    lines.append("=" * 72)
    lines.append(f"Generated: {datetime.now().isoformat()}")
    lines.append("")
    lines.append("Input and split contract")
    lines.append("-" * 72)
    lines.append(f"Input CSV: {os.path.abspath(cfg.input_csv)}")
    lines.append(f"Sequence column: {cols['sequence_col']}")
    lines.append(f"Split column: {cols['split_col']}")
    lines.append(f"Primary condition ID column: {cols['primary_condition_id_col']}")
    lines.append("")
    lines.append("Vocabulary and tokenization")
    lines.append("-" * 72)
    lines.append(f"Allowed amino acids: {cfg.allowed_aas}")
    lines.append(f"Vocabulary size: {len(vocab['tokens'])}")
    lines.append(f"Special tokens: PAD={cfg.pad_token}, BOS={cfg.bos_token}, EOS={cfg.eos_token}, UNK={cfg.unk_token}")
    lines.append(f"Length policy: {length_policy['policy']}")
    lines.append(f"Chosen max sequence length (pre-special tokens): {length_policy['chosen_max_sequence_length']}")
    lines.append(f"Max model length (including BOS/EOS): {length_policy['max_model_length_including_special_tokens']}")
    lines.append("")
    lines.append("Normalization")
    lines.append("-" * 72)
    lines.append(f"Number of train-frozen normalized baseline features: {len(normalization)}")
    lines.append("")
    lines.append("Split summary")
    lines.append("-" * 72)
    lines.append(summary_df.to_string(index=False))
    lines.append("")
    lines.append("QC summary")
    lines.append("-" * 72)
    lines.append(qc_df.to_string(index=False))
    return "\n".join(lines)


# ============================================================
# PIPELINE
# ============================================================

def run_step4_pipeline(cfg: Step4Config) -> Dict[str, Any]:
    np.random.seed(cfg.random_seed)
    ensure_output_tree(cfg)

    print("Loading Step 3 conditioned table...")
    df = load_input_table(cfg)
    cols = resolve_columns(df, cfg)
    df = validate_and_prepare_input(df, cfg, cols)

    print("Building train-frozen vocabulary and length policy...")
    train_mask = df[cols["split_col"]].astype(str) == "train"
    train_lengths = df.loc[train_mask, cols["sequence_col"]].astype(str).map(len).to_numpy(dtype=int)
    length_policy = determine_max_model_length(train_lengths, cfg)
    vocab = build_vocabulary(cfg)

    print("Encoding sequences and condition IDs...")
    df_enc, condition_mappings = encode_primary_conditions(df, cols)

    encoded_by_split: Dict[str, Dict[str, Any]] = {}
    all_parts = []
    max_len = length_policy["max_model_length_including_special_tokens"]
    for split in ["train", "val", "test"]:
        split_df = df_enc.loc[df_enc[cols["split_col"]].astype(str) == split].copy().reset_index(drop=True)
        token_ids = np.zeros((len(split_df), max_len), dtype=np.int32)
        attention_mask = np.zeros((len(split_df), max_len), dtype=np.int8)
        seq_lengths = np.zeros(len(split_df), dtype=np.int32)
        truncated_flag = np.zeros(len(split_df), dtype=np.int8)
        unk_counts = np.zeros(len(split_df), dtype=np.int32)

        for i, seq in enumerate(split_df[cols["sequence_col"]].astype(str)):
            ids, attn, seq_len, unk_count = encode_sequence(seq, vocab, max_len)
            token_ids[i] = ids
            attention_mask[i] = attn
            seq_lengths[i] = seq_len
            truncated_flag[i] = 1 if (seq_len + 2 > max_len) else 0
            unk_counts[i] = unk_count

        encoded_by_split[split] = {
            "token_ids": token_ids,
            "attention_mask": attention_mask,
            "sequence_length": seq_lengths,
            "truncated_flag": truncated_flag,
            "unknown_token_count": unk_counts,
            "primary_condition_id_idx": split_df[cols["primary_condition_id_col"] + "_idx"].to_numpy(dtype=np.int32),
            "length_bin_idx": split_df[cols["component::length_bin"] + "_idx"].to_numpy(dtype=np.int32),
            "net_charge_pH7_bin_idx": split_df[cols["component::net_charge_pH7_bin"] + "_idx"].to_numpy(dtype=np.int32),
            "hydrophobicity_KD_bin_idx": split_df[cols["component::hydrophobicity_KD_bin"] + "_idx"].to_numpy(dtype=np.int32),
            "sequence_sha256": split_df[cols["hash_col"]].astype(str).to_numpy(),
        }

        part = split_df.copy()
        part["sequence_length_raw"] = seq_lengths
        part["sequence_length_with_special_tokens"] = seq_lengths + 2
        part["max_model_length"] = max_len
        part["truncated_flag"] = truncated_flag
        part["unknown_token_count"] = unk_counts
        part["attention_nonpad_tokens"] = attention_mask.sum(axis=1)
        part["padding_fraction"] = 1.0 - attention_mask.mean(axis=1)
        all_parts.append(part)

    model_ready_df = pd.concat(all_parts, ignore_index=True)

    trunc_total = int(model_ready_df["truncated_flag"].sum())
    if cfg.enforce_no_truncation_in_final and trunc_total > 0:
        raise ValueError(
            f"Length policy produced {trunc_total} truncated sequences under enforce_no_truncation_in_final=True. "
            "Use train_exact_max or raise the sequence cap before finalizing Step 4."
        )

    print("Building baseline feature branch...")
    baseline_df, baseline_feature_cols, normalization = build_baseline_feature_table(df_enc, cfg, cols)

    print("Building QC summaries...")
    summary_df = build_step4_summary(model_ready_df, cols)
    length_summary_df, token_summary_df, qc_summary_df = build_length_and_token_qc(df_enc, cfg, cols, length_policy, encoded_by_split, vocab)

    # exact row-alignment audit
    source_counts = df.groupby(cols["split_col"], as_index=False).size().rename(columns={cols["split_col"]: "split", "size": "n_sequences_source"})
    ready_counts = model_ready_df.groupby(cols["split_col"], as_index=False).size().rename(columns={cols["split_col"]: "split", "size": "n_sequences_model_ready"})
    alignment_df = source_counts.merge(ready_counts, on="split", how="outer")
    alignment_df["delta"] = alignment_df["n_sequences_model_ready"].fillna(0) - alignment_df["n_sequences_source"].fillna(0)
    if (alignment_df["delta"] != 0).any():
        raise AssertionError("Source and model-ready split counts are not aligned.")

    print("Writing exports...")
    save_csv(model_ready_df, os.path.join(cfg.tables_dir, "step4_model_ready_sequences.csv"))
    for split in ["train", "val", "test"]:
        split_df = model_ready_df.loc[model_ready_df[cols["split_col"]].astype(str) == split].copy()
        save_csv(split_df, os.path.join(cfg.tables_dir, f"step4_{split}.csv"))

    save_csv(baseline_df, os.path.join(cfg.tables_dir, "step4_baseline_features.csv"))
    save_csv(summary_df, os.path.join(cfg.tables_dir, "step4_split_summary.csv"))
    save_csv(length_summary_df, os.path.join(cfg.tables_dir, "step4_sequence_length_summary.csv"))
    save_csv(token_summary_df, os.path.join(cfg.tables_dir, "step4_token_coverage_summary.csv"))
    save_csv(qc_summary_df, os.path.join(cfg.tables_dir, "step4_tokenization_qc.csv"))
    save_csv(alignment_df, os.path.join(cfg.tables_dir, "step4_row_alignment_audit.csv"))

    next_step_df = build_next_step_locations(cfg)
    save_csv(next_step_df, os.path.join(cfg.tables_dir, "step4_next_step_input_locations.csv"))

    save_json(vocab, os.path.join(cfg.artifacts_dir, "step4_vocab.json"))
    save_json({
        "pad_token": cfg.pad_token,
        "bos_token": cfg.bos_token,
        "eos_token": cfg.eos_token,
        "unk_token": cfg.unk_token,
        "pad_idx": vocab["pad_idx"],
        "bos_idx": vocab["bos_idx"],
        "eos_idx": vocab["eos_idx"],
        "unk_idx": vocab["unk_idx"],
    }, os.path.join(cfg.artifacts_dir, "step4_special_tokens.json"))
    save_json(condition_mappings, os.path.join(cfg.artifacts_dir, "step4_condition_id_mapping.json"))
    save_json(normalization, os.path.join(cfg.artifacts_dir, "step4_numeric_normalization.json"))
    save_json(length_policy, os.path.join(cfg.artifacts_dir, "step4_length_policy.json"))

    preprocessing_contract = {
        "step": 4,
        "description": "Split-safe residue-level tokenization and model-ready preprocessing",
        "input_csv": os.path.abspath(cfg.input_csv),
        "columns": cols,
        "allowed_aas": cfg.allowed_aas,
        "vocab_size": len(vocab["tokens"]),
        "length_policy": length_policy,
        "baseline_feature_columns": baseline_feature_cols,
        "normalization_keys": sorted(list(normalization.keys())),
        "config": asdict(cfg),
    }
    save_json(preprocessing_contract, os.path.join(cfg.artifacts_dir, "step4_preprocessing_contract.json"))

    # Save arrays per split and a combined NPZ
    combined_npz: Dict[str, Any] = {}
    for split, enc in encoded_by_split.items():
        split_npz_path = os.path.join(cfg.arrays_dir, f"step4_{split}_tokenized.npz")
        np.savez_compressed(split_npz_path, **enc)
        for k, v in enc.items():
            combined_npz[f"{split}__{k}"] = v
    np.savez_compressed(os.path.join(cfg.arrays_dir, "step4_tokenized_sequences.npz"), **combined_npz)

    artifact_hashes = {
        "input_csv": {"path": os.path.abspath(cfg.input_csv), "sha256": sha256_file(cfg.input_csv)},
    }
    for rel in [
        os.path.join(cfg.tables_dir, "step4_model_ready_sequences.csv"),
        os.path.join(cfg.tables_dir, "step4_baseline_features.csv"),
        os.path.join(cfg.artifacts_dir, "step4_vocab.json"),
        os.path.join(cfg.artifacts_dir, "step4_preprocessing_contract.json"),
        os.path.join(cfg.arrays_dir, "step4_tokenized_sequences.npz"),
    ]:
        artifact_hashes[os.path.basename(rel)] = {"path": os.path.abspath(rel), "sha256": sha256_file(rel)}
    save_json(artifact_hashes, os.path.join(cfg.artifacts_dir, "step4_artifact_hashes.json"))
    save_json(system_report(cfg), os.path.join(cfg.reports_dir, "step4_system_report.json"))

    method_report = build_method_report(cfg, cols, length_policy, vocab, normalization, summary_df, qc_summary_df)
    save_text(method_report, os.path.join(cfg.reports_dir, "step4_method_report.txt"))

    print("Generating figures...")
    fig_main = make_main_figure(df_enc, cfg, cols, length_summary_df, token_summary_df, qc_summary_df, length_policy)
    fig_supp_1 = make_supp_fig_padding_truncation(length_summary_df, cfg)
    fig_supp_2 = make_supp_fig_token_freq(df_enc, cfg, cols)

    print("Step 4 completed successfully.")
    print(f"Output directory: {cfg.out_dir}")
    print("-" * 80)
    print(summary_df.to_string(index=False))
    print("-" * 80)
    print("Length / token QC:")
    print(qc_summary_df.to_string(index=False))
    print_next_step_file_locations(cfg)

    return {
        "config": asdict(cfg),
        "resolved_columns": cols,
        "length_policy": length_policy,
        "summary_df": summary_df,
        "length_summary_df": length_summary_df,
        "token_summary_df": token_summary_df,
        "qc_summary_df": qc_summary_df,
        "alignment_df": alignment_df,
        "main_figure_png": fig_main,
        "supp_figure_1_png": fig_supp_1,
        "supp_figure_2_png": fig_supp_2,
        "next_step_locations": next_step_df,
        "output_dir": cfg.out_dir,
    }


# ============================================================
# NOTEBOOK ENTRYPOINT
# ============================================================

def main_notebook(
    input_csv: str,
    out_dir_name: str = "step4",
    max_length_policy: str = "train_exact_max",
    train_length_percentile: float = 99.5,
    enforce_no_truncation_in_final: bool = True,
    random_seed: int = 42,
) -> Dict[str, Any]:
    cfg = Step4Config(
        input_csv=input_csv,
        out_dir_name=out_dir_name,
        max_length_policy=max_length_policy,
        train_length_percentile=train_length_percentile,
        enforce_no_truncation_in_final=enforce_no_truncation_in_final,
        random_seed=random_seed,
    )
    return run_step4_pipeline(cfg)


if __name__ == "__main__":
    results = main_notebook(
        input_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step3/tables/step3_conditioned_unique_sequences.csv",
        out_dir_name="step4",
        max_length_policy="train_exact_max",
        train_length_percentile=99.5,
        enforce_no_truncation_in_final=True,
        random_seed=42,
    )


In [ ]:
results = main_notebook(
    input_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step3/tables/step3_conditioned_unique_sequences.csv",
    out_dir_name="step4",
    max_length_policy="train_exact_max",
    train_length_percentile=99.5,
    enforce_no_truncation_in_final=True,
    random_seed=42,
)

fig4

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# ============================================================
# CONFIG
# ============================================================
STEP4_DIR = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")
TABLES_DIR = STEP4_DIR / "tables"
ARTIFACTS_DIR = STEP4_DIR / "artifacts"
FIG_MAIN_DIR = STEP4_DIR / "figures_main"
FIG_MAIN_DIR.mkdir(parents=True, exist_ok=True)

OUT_BASENAME = "Fig4_representation_preprocessing_main_improved"

# Color-blind-safe, manuscript-friendly palette
SPLIT_COLORS = {
    "train": "#4E79A7",   # blue
    "val":   "#F28E2B",   # orange
    "test":  "#E15759",   # red
}

METRIC_COLORS = {
    "Valid condition ID": "#4E79A7",
    "Non-truncated": "#59A14F",
    "Known-token coverage": "#9C755F",
}

# ============================================================
# HELPERS
# ============================================================
def find_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(f"None of the candidate files exist:\n" + "\n".join(str(p) for p in paths))

def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find any of these columns: {candidates}\nAvailable: {list(df.columns)}")
    return None

def add_panel_label(ax, label, x=-0.16, y=1.08, fontsize=16):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=fontsize,
        fontweight="bold",
        va="top", ha="left"
    )

def save_figure(fig, out_base: Path):
    fig.savefig(out_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".tiff"), dpi=600, bbox_inches="tight")
    print(f"Saved:\n- {out_base.with_suffix('.png')}\n- {out_base.with_suffix('.pdf')}\n- {out_base.with_suffix('.tiff')}")

# ============================================================
# LOAD INPUTS
# ============================================================
model_ready_fp = find_existing([
    TABLES_DIR / "step4_model_ready_sequences.csv",
    TABLES_DIR / "step4_train.csv",  # fallback not used unless needed
])

qc_fp = find_existing([
    TABLES_DIR / "step4_tokenization_qc.csv",
    TABLES_DIR / "step4_length_token_qc.csv",
])

vocab_fp = find_existing([
    ARTIFACTS_DIR / "step4_vocab.json",
])

contract_fp = None
for p in [
    ARTIFACTS_DIR / "step4_preprocessing_contract.json",
    ARTIFACTS_DIR / "step4_preprocessing_config.json",
]:
    if p.exists():
        contract_fp = p
        break

df = pd.read_csv(model_ready_fp, low_memory=False)
qc = pd.read_csv(qc_fp, low_memory=False)

with open(vocab_fp, "r", encoding="utf-8") as f:
    vocab = json.load(f)

contract = {}
if contract_fp is not None:
    with open(contract_fp, "r", encoding="utf-8") as f:
        contract = json.load(f)

# ============================================================
# COLUMN DETECTION
# ============================================================
split_col = pick_col(df, ["assigned_split", "split"])
seq_col = pick_col(df, ["sequence", "peptide", "seq"])

# derive sequence length
if "sequence_length" in df.columns:
    df["sequence_length_plot"] = df["sequence_length"].astype(int)
elif "length" in df.columns:
    df["sequence_length_plot"] = df["length"].astype(int)
else:
    df["sequence_length_plot"] = df[seq_col].astype(str).str.len()

qc_split_col = pick_col(qc, ["assigned_split", "split"])

# Step 4 cap
length_cap = None
for key in ["max_length", "sequence_max_length", "model_max_length", "length_cap"]:
    if key in contract:
        length_cap = int(contract[key])
        break
if length_cap is None:
    length_cap = int(df["sequence_length_plot"].max())

vocab_size = len(vocab)

# standardize split order
split_order = ["train", "val", "test"]
df[split_col] = pd.Categorical(df[split_col], categories=split_order, ordered=True)
qc[qc_split_col] = pd.Categorical(qc[qc_split_col], categories=split_order, ordered=True)

# ============================================================
# PANEL C METRICS
# Replace old "scaled counts" panel with scientifically meaningful integrity metrics
# ============================================================
panel_c = qc[[qc_split_col]].copy()
panel_c["Valid condition ID"] = 1.0 - qc["fraction_primary_condition_id_missing"].astype(float)
panel_c["Non-truncated"] = 1.0 - qc["fraction_truncated"].astype(float)
panel_c["Known-token coverage"] = 1.0 - qc["unknown_token_fraction_nonpad"].astype(float)

panel_c = panel_c.rename(columns={qc_split_col: "split"})

# ============================================================
# PLOT
# ============================================================
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=False)
ax1, ax2, ax3 = axes

# ------------------------------------------------------------
# a) Length distribution and cap
# ------------------------------------------------------------
bins = np.arange(0, max(61, int(df["sequence_length_plot"].max()) + 2), 1)

for split in split_order:
    sub = df[df[split_col] == split]["sequence_length_plot"].dropna().astype(int).values
    if len(sub) == 0:
        continue
    ax1.hist(
        sub,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2.0,
        color=SPLIT_COLORS[split],
        label=split
    )

ax1.axvline(length_cap, color="black", linestyle="--", linewidth=2)
ax1.set_title("Length distribution and cap")
ax1.set_xlabel("Sequence length")
ax1.set_ylabel("Density")
ax1.legend(frameon=False, loc="upper right")
ax1.grid(axis="y", alpha=0.25)

# ------------------------------------------------------------
# b) Token coverage
# ------------------------------------------------------------
unknown_by_split = (
    qc[[qc_split_col, "unknown_token_fraction_nonpad"]]
    .rename(columns={qc_split_col: "split"})
    .sort_values("split")
)

x_b = np.arange(len(split_order))
y_b = [
    float(unknown_by_split.loc[unknown_by_split["split"] == s, "unknown_token_fraction_nonpad"].iloc[0])
    for s in split_order
]

bars_b = ax2.bar(
    x_b,
    y_b,
    color=[SPLIT_COLORS[s] for s in split_order],
    width=0.58
)
ax2.set_xticks(x_b)
ax2.set_xticklabels(split_order)
ax2.set_ylabel("Unknown-token fraction")
ax2.set_title("Token coverage")
upper_b = max(0.01, max(y_b) * 1.4 if max(y_b) > 0 else 0.01)
ax2.set_ylim(0, upper_b)
ax2.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
ax2.grid(axis="y", alpha=0.25)

for i, split in enumerate(split_order):
    ax2.text(i, upper_b * 0.05, f"V={vocab_size}", ha="center", va="bottom", fontsize=10)

# ------------------------------------------------------------
# c) Preprocessing integrity
# ------------------------------------------------------------
metrics = ["Valid condition ID", "Non-truncated", "Known-token coverage"]
x = np.arange(len(split_order))
width = 0.22

for j, metric in enumerate(metrics):
    vals = [
        float(panel_c.loc[panel_c["split"] == s, metric].iloc[0])
        for s in split_order
    ]
    bars = ax3.bar(
        x + (j - 1) * width,
        vals,
        width=width,
        label=metric,
        color=METRIC_COLORS[metric]
    )
    for b in bars:
        ax3.text(
            b.get_x() + b.get_width() / 2,
            min(1.03, b.get_height() + 0.01),
            f"{b.get_height():.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax3.set_xticks(x)
ax3.set_xticklabels(split_order)
ax3.set_ylim(0, 1.08)
ax3.set_ylabel("Fraction")
ax3.set_title("Preprocessing integrity")
ax3.grid(axis="y", alpha=0.25)

# Legend BELOW panel c
ax3.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3
)

# ------------------------------------------------------------
# Panel labels outside axes
# ------------------------------------------------------------
add_panel_label(ax1, "a")
add_panel_label(ax2, "b")
add_panel_label(ax3, "c")

# ------------------------------------------------------------
# Final spacing
# ------------------------------------------------------------
fig.subplots_adjust(
    left=0.06,
    right=0.99,
    top=0.90,
    bottom=0.22,
    wspace=0.30
)

# ============================================================
# SAVE
# ============================================================
out_base = FIG_MAIN_DIR / OUT_BASENAME
save_figure(fig, out_base)
plt.show()

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# ============================================================
# CONFIG
# ============================================================
STEP4_DIR = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")
TABLES_DIR = STEP4_DIR / "tables"
ARTIFACTS_DIR = STEP4_DIR / "artifacts"
FIG_MAIN_DIR = STEP4_DIR / "figures_main"
FIG_MAIN_DIR.mkdir(parents=True, exist_ok=True)

OUT_BASENAME = "Fig4_representation_preprocessing_main_improved_v2"

# Color-blind-safe palette
SPLIT_COLORS = {
    "train": "#4E79A7",   # blue
    "val":   "#F28E2B",   # orange
    "test":  "#E15759",   # red
}

METRIC_COLORS = {
    "Valid condition ID": "#A4A1F0DC",
    "Non-truncated": "#5CE3F2DE",
    "Known-token coverage": "#EC6E25",
}

# ============================================================
# HELPERS
# ============================================================
def find_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of the candidate files exist:\n" + "\n".join(str(p) for p in paths)
    )

def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"Could not find any of these columns: {candidates}\nAvailable: {list(df.columns)}"
        )
    return None

def add_panel_label(ax, label, x=-0.16, y=1.08, fontsize=16):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=fontsize,
        fontweight="bold",
        va="top", ha="left"
    )

def save_figure(fig, out_base: Path):
    fig.savefig(out_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".tiff"), dpi=600, bbox_inches="tight")
    print(f"Saved:\n- {out_base.with_suffix('.png')}\n- {out_base.with_suffix('.pdf')}\n- {out_base.with_suffix('.tiff')}")

# ============================================================
# LOAD INPUTS
# ============================================================
model_ready_fp = find_existing([
    TABLES_DIR / "step4_model_ready_sequences.csv",
])

qc_fp = find_existing([
    TABLES_DIR / "step4_tokenization_qc.csv",
    TABLES_DIR / "step4_length_token_qc.csv",
])

vocab_fp = find_existing([
    ARTIFACTS_DIR / "step4_vocab.json",
])

contract_fp = None
for p in [
    ARTIFACTS_DIR / "step4_preprocessing_contract.json",
    ARTIFACTS_DIR / "step4_preprocessing_config.json",
]:
    if p.exists():
        contract_fp = p
        break

df = pd.read_csv(model_ready_fp, low_memory=False)
qc = pd.read_csv(qc_fp, low_memory=False)

with open(vocab_fp, "r", encoding="utf-8") as f:
    vocab = json.load(f)

contract = {}
if contract_fp is not None:
    with open(contract_fp, "r", encoding="utf-8") as f:
        contract = json.load(f)

# ============================================================
# COLUMN DETECTION
# ============================================================
split_col = pick_col(df, ["assigned_split", "split"])
seq_col = pick_col(df, ["sequence", "peptide", "seq"])

if "sequence_length" in df.columns:
    df["sequence_length_plot"] = df["sequence_length"].astype(int)
elif "length" in df.columns:
    df["sequence_length_plot"] = df["length"].astype(int)
else:
    df["sequence_length_plot"] = df[seq_col].astype(str).str.len()

qc_split_col = pick_col(qc, ["assigned_split", "split"])

length_cap = None
for key in ["max_length", "sequence_max_length", "model_max_length", "length_cap"]:
    if key in contract:
        length_cap = int(contract[key])
        break
if length_cap is None:
    length_cap = int(df["sequence_length_plot"].max())

vocab_size = len(vocab)

split_order = ["train", "val", "test"]
df[split_col] = pd.Categorical(df[split_col], categories=split_order, ordered=True)
qc[qc_split_col] = pd.Categorical(qc[qc_split_col], categories=split_order, ordered=True)

# ============================================================
# PANEL C METRICS
# ============================================================
panel_c = qc[[qc_split_col]].copy()
panel_c["Valid condition ID"] = 1.0 - qc["fraction_primary_condition_id_missing"].astype(float)
panel_c["Non-truncated"] = 1.0 - qc["fraction_truncated"].astype(float)
panel_c["Known-token coverage"] = 1.0 - qc["unknown_token_fraction_nonpad"].astype(float)
panel_c = panel_c.rename(columns={qc_split_col: "split"})

# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Taller figure to give space for bottom legends
fig, axes = plt.subplots(1, 3, figsize=(17, 5.4), constrained_layout=False)
ax1, ax2, ax3 = axes

# ============================================================
# PANEL A: Length distribution and cap
# ============================================================
bins = np.arange(0, max(61, int(df["sequence_length_plot"].max()) + 2), 1)

for split in split_order:
    sub = df[df[split_col] == split]["sequence_length_plot"].dropna().astype(int).values
    if len(sub) == 0:
        continue
    ax1.hist(
        sub,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2.0,
        color=SPLIT_COLORS[split],
        label=split
    )

ax1.axvline(length_cap, color="black", linestyle="--", linewidth=2)
ax1.set_title("Length distribution and cap")
ax1.set_xlabel("Sequence length")
ax1.set_ylabel("Density")
ax1.grid(axis="y", alpha=0.25)

# Move legend BELOW panel a
ax1.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3
)

# ============================================================
# PANEL B: Token coverage
# ============================================================
unknown_by_split = (
    qc[[qc_split_col, "unknown_token_fraction_nonpad"]]
    .rename(columns={qc_split_col: "split"})
    .sort_values("split")
)

x_b = np.arange(len(split_order))
y_b = [
    float(unknown_by_split.loc[unknown_by_split["split"] == s, "unknown_token_fraction_nonpad"].iloc[0])
    for s in split_order
]

ax2.bar(
    x_b,
    y_b,
    color=[SPLIT_COLORS[s] for s in split_order],
    width=0.58
)
ax2.set_xticks(x_b)
ax2.set_xticklabels(split_order)
ax2.set_ylabel("Unknown-token fraction")
ax2.set_title("Token coverage")
upper_b = max(0.01, max(y_b) * 1.4 if max(y_b) > 0 else 0.01)
ax2.set_ylim(0, upper_b)
ax2.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
ax2.grid(axis="y", alpha=0.25)

for i, split in enumerate(split_order):
    ax2.text(
        i,
        upper_b * 0.03,
        f"V={vocab_size}",
        ha="center",
        va="bottom",
        fontsize=10
    )

# ============================================================
# PANEL C: Preprocessing integrity
# ============================================================
metrics = ["Valid condition ID", "Non-truncated", "Known-token coverage"]
x = np.arange(len(split_order))
width = 0.22

for j, metric in enumerate(metrics):
    vals = [
        float(panel_c.loc[panel_c["split"] == s, metric].iloc[0])
        for s in split_order
    ]
    bars = ax3.bar(
        x + (j - 1) * width,
        vals,
        width=width,
        label=metric,
        color=METRIC_COLORS[metric]
    )
    for b in bars:
        ax3.text(
            b.get_x() + b.get_width() / 2,
            min(1.03, b.get_height() + 0.01),
            f"{b.get_height():.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax3.set_xticks(x)
ax3.set_xticklabels(split_order)
ax3.set_ylim(0, 1.08)
ax3.set_ylabel("Fraction")
ax3.set_title("Preprocessing integrity")
ax3.grid(axis="y", alpha=0.25)

# Move legend BELOW panel c
ax3.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3
)

# ============================================================
# PANEL LABELS OUTSIDE AXES
# ============================================================
add_panel_label(ax1, "a")
add_panel_label(ax2, "b")
add_panel_label(ax3, "c")

# ============================================================
# FINAL SPACING
# ============================================================
fig.subplots_adjust(
    left=0.06,
    right=0.99,
    top=0.90,
    bottom=0.28,
    wspace=0.28
)

# ============================================================
# SAVE
# ============================================================
out_base = FIG_MAIN_DIR / OUT_BASENAME
save_figure(fig, out_base)
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import json

step4_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")

required_files = [
    step4_dir / "tables" / "step4_model_ready_sequences.csv",
    step4_dir / "tables" / "step4_baseline_features.csv",
    step4_dir / "artifacts" / "step4_preprocessing_contract.json",
    step4_dir / "artifacts" / "step4_vocab.json",
    step4_dir / "artifacts" / "step4_condition_id_mapping.json",
]

print("Checking required Step 4 files:\n")
for f in required_files:
    print(f"{'FOUND   ' if f.exists() else 'MISSING  '} {f}")

print("\n" + "="*100)
print("CSV schema preview\n")

for f in required_files[:2]:
    if f.exists():
        df = pd.read_csv(f)
        print("="*100)
        print(f"FILE: {f.name}")
        print(f"PATH: {f}")
        print(f"SHAPE: {df.shape}")
        print(f"COLUMNS:\n{list(df.columns)}\n")
        print(df.head(3))
        print()

print("\n" + "="*100)
print("JSON artifact preview\n")

json_files = required_files[2:]
for f in json_files:
    if f.exists():
        print("="*100)
        print(f"FILE: {f.name}")
        with open(f, "r", encoding="utf-8") as fh:
            obj = json.load(fh)
        if isinstance(obj, dict):
            print("TOP-LEVEL KEYS:", list(obj.keys())[:30])
        else:
            print("TYPE:", type(obj))
        print()

In [ ]:
from pathlib import Path
import pandas as pd

step4_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")

seq_df = pd.read_csv(step4_dir / "tables" / "step4_model_ready_sequences.csv")
feat_df = pd.read_csv(step4_dir / "tables" / "step4_baseline_features.csv")

print("Sequence table shape:", seq_df.shape)
print("Feature table shape :", feat_df.shape)
print()

common_cols = sorted(set(seq_df.columns).intersection(feat_df.columns))
print("Common columns between both tables:")
print(common_cols)
print()

candidate_id_cols = [c for c in common_cols if "id" in c.lower() or "seq" in c.lower()]
print("Candidate join columns:")
print(candidate_id_cols)

In [ ]:
from pathlib import Path
import pandas as pd

step4_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")

seq_df = pd.read_csv(step4_dir / "tables" / "step4_model_ready_sequences.csv")
feat_df = pd.read_csv(step4_dir / "tables" / "step4_baseline_features.csv")

seq_cols = set(seq_df.columns)
feat_cols = set(feat_df.columns)

baseline_only = sorted(feat_cols - seq_cols)

print("Baseline-feature-only columns:")
print(baseline_only)
print("\nCount:", len(baseline_only))

step: 5 new

In [ ]:
from pathlib import Path

step4_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")

print(f"Step 4 exists: {step4_dir.exists()}")
print(f"Step 4 path: {step4_dir}\n")

if step4_dir.exists():
    print("=== Top-level contents ===")
    for p in sorted(step4_dir.iterdir()):
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"{kind:4}  {p.name}")

    print("\n=== Full recursive file listing ===")
    for p in sorted(step4_dir.rglob("*")):
        rel = p.relative_to(step4_dir)
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"{kind:4}  {rel}")
else:
    print("Step 4 directory not found.")

In [ ]:
from pathlib import Path

step4_dir = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4")
wanted_suffixes = {".csv", ".json", ".npy", ".npz", ".txt"}

print("=== Candidate Step 5 input files from Step 4 ===")
for p in sorted(step4_dir.rglob("*")):
    if p.is_file() and p.suffix.lower() in wanted_suffixes:
        print(p)

updated step5: 

In [ ]:
# ================================
# Step 5 — Baseline predictive benchmarking
# Primary task: multiclass classification of primary_condition_id
# Inputs: frozen Step 4 outputs
# ================================

from __future__ import annotations

import json
import math
import hashlib
import inspect
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    log_loss,
)
from sklearn.inspection import permutation_importance

# ---------------------------------------------------------
# Reproducibility helpers
# ---------------------------------------------------------

GLOBAL_SEED = 42
RNG = np.random.default_rng(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


# ---------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------

@dataclass
class Step5Config:
    input_model_ready_csv: str
    input_baseline_features_csv: str
    input_preprocessing_contract_json: Optional[str] = None
    input_condition_id_mapping_json: Optional[str] = None
    out_dir_name: str = "step5"

    merge_key: str = "sequence_sha256"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"
    sequence_col: str = "sequence"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    random_seed: int = 42
    bootstrap_iterations: int = 500
    permutation_repeats: int = 10
    topk_values: Tuple[int, ...] = (1, 3)

    # Set to True if you want slightly faster testing with fewer bootstrap iterations
    fast_mode: bool = False


# ---------------------------------------------------------
# Small utilities
# ---------------------------------------------------------

def _safe_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def _sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _mkdirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables": base_dir / "tables",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "models": base_dir / "models",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def _project_root_from_input(input_csv: str) -> Path:
    """
    Given:
      /.../nature_computational2/step4/tables/step4_model_ready_sequences.csv
    return:
      /.../nature_computational2
    """
    p = Path(input_csv).resolve()
    # expected .../<project_root>/step4/tables/file.csv
    if p.parent.name == "tables" and p.parent.parent.name == "step4":
        return p.parent.parent.parent
    return p.parent


def _print_header(msg: str) -> None:
    print("=" * 100)
    print(msg)
    print("=" * 100)


# ---------------------------------------------------------
# Loading and validation
# ---------------------------------------------------------

def load_step4_inputs(cfg: Step5Config) -> Dict[str, Any]:
    seq_path = Path(cfg.input_model_ready_csv).resolve()
    feat_path = Path(cfg.input_baseline_features_csv).resolve()

    if not seq_path.exists():
        raise FileNotFoundError(f"Missing model-ready CSV: {seq_path}")
    if not feat_path.exists():
        raise FileNotFoundError(f"Missing baseline-features CSV: {feat_path}")

    seq_df = pd.read_csv(seq_path)
    feat_df = pd.read_csv(feat_path)

    contract = None
    condition_mapping = None

    if cfg.input_preprocessing_contract_json:
        contract_path = Path(cfg.input_preprocessing_contract_json).resolve()
        if not contract_path.exists():
            raise FileNotFoundError(f"Missing preprocessing contract JSON: {contract_path}")
        with open(contract_path, "r", encoding="utf-8") as f:
            contract = json.load(f)

    if cfg.input_condition_id_mapping_json:
        mapping_path = Path(cfg.input_condition_id_mapping_json).resolve()
        if not mapping_path.exists():
            raise FileNotFoundError(f"Missing condition ID mapping JSON: {mapping_path}")
        with open(mapping_path, "r", encoding="utf-8") as f:
            condition_mapping = json.load(f)

    return {
        "seq_df": seq_df,
        "feat_df": feat_df,
        "contract": contract,
        "condition_mapping": condition_mapping,
        "seq_path": seq_path,
        "feat_path": feat_path,
    }


def validate_and_merge_inputs(seq_df: pd.DataFrame, feat_df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    required_seq_cols = [cfg.merge_key, cfg.split_col, cfg.target_col, cfg.sequence_col]
    for col in required_seq_cols:
        if col not in seq_df.columns:
            raise ValueError(f"Required column '{col}' not found in sequence table.")

    if cfg.merge_key not in feat_df.columns:
        raise ValueError(f"Required merge key '{cfg.merge_key}' not found in baseline feature table.")

    if seq_df[cfg.merge_key].isna().any():
        raise ValueError(f"Merge key '{cfg.merge_key}' contains missing values in sequence table.")
    if feat_df[cfg.merge_key].isna().any():
        raise ValueError(f"Merge key '{cfg.merge_key}' contains missing values in feature table.")

    if not seq_df[cfg.merge_key].is_unique:
        dup_n = seq_df[cfg.merge_key].duplicated().sum()
        raise ValueError(f"Sequence table merge key is not unique. Duplicates: {dup_n}")
    if not feat_df[cfg.merge_key].is_unique:
        dup_n = feat_df[cfg.merge_key].duplicated().sum()
        raise ValueError(f"Feature table merge key is not unique. Duplicates: {dup_n}")

    # Keep all sequence metadata; add only feature columns not already present, plus merge key
    feat_extra_cols = [cfg.merge_key] + [c for c in feat_df.columns if c not in seq_df.columns and c != cfg.merge_key]
    merged = seq_df.merge(feat_df[feat_extra_cols], on=cfg.merge_key, how="inner", validate="one_to_one")

    if len(merged) != len(seq_df) or len(merged) != len(feat_df):
        raise ValueError(
            f"Merge row count mismatch: merged={len(merged)}, seq={len(seq_df)}, feat={len(feat_df)}"
        )

    expected_splits = {cfg.train_label, cfg.val_label, cfg.test_label}
    found_splits = set(merged[cfg.split_col].dropna().unique().tolist())
    if not expected_splits.issubset(found_splits):
        raise ValueError(f"Expected splits {expected_splits}, found {found_splits}")

    if merged[cfg.target_col].isna().any():
        n_missing = int(merged[cfg.target_col].isna().sum())
        raise ValueError(f"Target column '{cfg.target_col}' contains missing values: {n_missing}")

    return merged


# ---------------------------------------------------------
# Feature blocks
# ---------------------------------------------------------

def infer_feature_blocks(df: pd.DataFrame) -> Dict[str, List[str]]:
    aa_frac_raw = sorted([c for c in df.columns if c.startswith("aa_frac_") and not c.endswith("_z")])
    aa_frac_z = sorted([c for c in df.columns if c.startswith("aa_frac_") and c.endswith("_z")])

    global_raw_candidates = [
        "length", "net_charge_pH7", "hydrophobicity_KD", "entropy", "dominant_aa_frac"
    ]
    global_z_candidates = [
        "length_z", "net_charge_pH7_z", "hydrophobicity_KD_z", "entropy_z", "dominant_aa_frac_z"
    ]

    global_raw = [c for c in global_raw_candidates if c in df.columns]
    global_z = [c for c in global_z_candidates if c in df.columns]

    combined_raw = global_raw + aa_frac_raw
    combined_z = global_z + aa_frac_z
    combined_all = combined_raw + combined_z

    feature_blocks = {
        "global_raw": global_raw,
        "aa_frac_raw": aa_frac_raw,
        "global_z": global_z,
        "aa_frac_z": aa_frac_z,
        "combined_raw": combined_raw,
        "combined_z": combined_z,
        "combined_all": combined_all,
    }

    # Sanity checks
    for name, cols in feature_blocks.items():
        if len(cols) == 0:
            print(f"Warning: feature block '{name}' is empty.")
    return feature_blocks


# ---------------------------------------------------------
# Split preparation
# ---------------------------------------------------------

def prepare_splits(df: pd.DataFrame, feature_cols: List[str], cfg: Step5Config) -> Dict[str, Any]:
    use_cols = [cfg.merge_key, cfg.sequence_col, cfg.split_col, cfg.target_col] + feature_cols
    use_cols = [c for c in use_cols if c in df.columns]
    sub = df[use_cols].copy()

    train_df = sub[sub[cfg.split_col] == cfg.train_label].reset_index(drop=True)
    val_df = sub[sub[cfg.split_col] == cfg.val_label].reset_index(drop=True)
    test_df = sub[sub[cfg.split_col] == cfg.test_label].reset_index(drop=True)

    X_train = train_df[feature_cols].copy()
    y_train = train_df[cfg.target_col].astype(str).to_numpy()

    X_val = val_df[feature_cols].copy()
    y_val = val_df[cfg.target_col].astype(str).to_numpy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[cfg.target_col].astype(str).to_numpy()

    classes = np.array(sorted(pd.Series(y_train).unique().tolist()))
    if not set(pd.Series(y_val).unique()).issubset(set(classes)):
        raise ValueError("Validation set contains classes absent from train set.")
    if not set(pd.Series(y_test).unique()).issubset(set(classes)):
        raise ValueError("Test set contains classes absent from train set.")

    return {
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "classes": classes,
    }


# ---------------------------------------------------------
# Model definitions
# ---------------------------------------------------------

def build_model_registry(seed: int) -> Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]]:
    registry: Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]] = {}

    # Dummy majority baseline
    registry["dummy_majority"] = [
        (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]),
            {},
        )
    ]

    # Logistic regression
    lr_grid = []
    for C in [0.1, 1.0, 3.0, 10.0]:
        lr_grid.append((
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    C=C,
                    class_weight="balanced",
                    multi_class="multinomial",
                    solver="lbfgs",
                    max_iter=3000,
                    random_state=seed,
                )),
            ]),
            {"C": C},
        ))
    registry["logistic_regression"] = lr_grid

    # Random forest
    rf_grid = []
    for n_estimators in [300]:
        for max_depth in [None, 20]:
            for min_samples_leaf in [1, 3]:
                rf_grid.append((
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("model", RandomForestClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            class_weight="balanced_subsample",
                            n_jobs=-1,
                            random_state=seed,
                        )),
                    ]),
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                    },
                ))
    registry["random_forest"] = rf_grid

    # Extra trees
    et_grid = []
    for n_estimators in [300]:
        for max_depth in [None, 20]:
            for min_samples_leaf in [1, 3]:
                et_grid.append((
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("model", ExtraTreesClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            class_weight="balanced",
                            n_jobs=-1,
                            random_state=seed,
                        )),
                    ]),
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                    },
                ))
    registry["extra_trees"] = et_grid

    return registry


# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

def topk_accuracy(y_true: np.ndarray, proba: np.ndarray, classes: np.ndarray, k: int) -> float:
    class_to_idx = {c: i for i, c in enumerate(classes)}
    true_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    topk_idx = np.argsort(-proba, axis=1)[:, :k]
    hits = np.any(topk_idx == true_idx[:, None], axis=1)
    return float(np.mean(hits))


def safe_macro_auroc(y_true: np.ndarray, proba: np.ndarray, classes: np.ndarray) -> float:
    try:
        y_true_bin = label_binarize(y_true, classes=classes)
        if y_true_bin.shape[1] == 1:
            return float("nan")
        return float(roc_auc_score(y_true_bin, proba, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: np.ndarray,
    classes: np.ndarray,
    split_name: str,
    model_name: str,
    feature_block: str,
) -> Dict[str, Any]:
    metrics = {
        "split": split_name,
        "model_name": model_name,
        "feature_block": feature_block,
        "n_samples": int(len(y_true)),
        "n_classes": int(len(classes)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_auroc_ovr": safe_macro_auroc(y_true, proba, classes),
        "log_loss": float(log_loss(y_true, proba, labels=classes)) if proba is not None else float("nan"),
    }

    for k in [1, 3]:
        if proba is not None and k <= len(classes):
            metrics[f"top{k}_accuracy"] = topk_accuracy(y_true, proba, classes, k)
        else:
            metrics[f"top{k}_accuracy"] = float("nan")
    return metrics


def per_class_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray,
    model_name: str,
    feature_block: str,
) -> pd.DataFrame:
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    return pd.DataFrame({
        "model_name": model_name,
        "feature_block": feature_block,
        "class_label": classes,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "support": support.astype(int),
    })


# ---------------------------------------------------------
# Training / selection
# ---------------------------------------------------------

def fit_select_model(
    splits: Dict[str, Any],
    model_registry: Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]],
    feature_block_name: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    rows = []

    X_train = splits["X_train"]
    y_train = splits["y_train"]
    X_val = splits["X_val"]
    y_val = splits["y_val"]
    classes = splits["classes"]

    best_record = None
    best_score = -np.inf

    for model_name, candidates in model_registry.items():
        for candidate_pipeline, param_dict in candidates:
            model = clone(candidate_pipeline)
            model.fit(X_train, y_train)

            y_val_pred = model.predict(X_val)
            if hasattr(model, "predict_proba"):
                val_proba = model.predict_proba(X_val)
            else:
                # Fallback, though all configured models here support predict_proba
                val_proba = np.full((len(X_val), len(classes)), fill_value=1.0 / len(classes), dtype=float)

            metrics = evaluate_predictions(
                y_true=y_val,
                y_pred=y_val_pred,
                proba=val_proba,
                classes=classes,
                split_name="val",
                model_name=model_name,
                feature_block=feature_block_name,
            )
            metrics.update(param_dict)
            rows.append(metrics)

            score = metrics["macro_f1"]
            if score > best_score:
                best_score = score
                best_record = {
                    "model_name": model_name,
                    "params": param_dict,
                    "pipeline": model,
                    "val_metrics": metrics,
                }

    val_results_df = pd.DataFrame(rows).sort_values(
        by=["macro_f1", "balanced_accuracy", "top3_accuracy"], ascending=False
    ).reset_index(drop=True)

    if best_record is None:
        raise RuntimeError("No model could be selected on validation data.")

    return val_results_df, best_record


def refit_best_model_on_train_val(best_record: Dict[str, Any], splits: Dict[str, Any]) -> Pipeline:
    best_model = clone(best_record["pipeline"])

    X_train_val = pd.concat([splits["X_train"], splits["X_val"]], axis=0).reset_index(drop=True)
    y_train_val = np.concatenate([splits["y_train"], splits["y_val"]])

    best_model.fit(X_train_val, y_train_val)
    return best_model


# ---------------------------------------------------------
# Bootstrap confidence intervals
# ---------------------------------------------------------

def bootstrap_test_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: np.ndarray,
    classes: np.ndarray,
    n_iter: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []

    for i in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        pp = proba[idx]

        rows.append({
            "bootstrap_iter": i,
            "accuracy": accuracy_score(yt, yp),
            "balanced_accuracy": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "weighted_f1": f1_score(yt, yp, average="weighted"),
            "macro_auroc_ovr": safe_macro_auroc(yt, pp, classes),
            "top1_accuracy": topk_accuracy(yt, pp, classes, 1),
            "top3_accuracy": topk_accuracy(yt, pp, classes, 3) if len(classes) >= 3 else np.nan,
        })

    return pd.DataFrame(rows)


def summarize_bootstrap(df_boot: pd.DataFrame) -> pd.DataFrame:
    metrics = [c for c in df_boot.columns if c != "bootstrap_iter"]
    rows = []
    for m in metrics:
        vals = pd.Series(df_boot[m]).dropna()
        rows.append({
            "metric": m,
            "mean": float(vals.mean()),
            "median": float(vals.median()),
            "ci_lower_2.5": float(vals.quantile(0.025)),
            "ci_upper_97.5": float(vals.quantile(0.975)),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Interpretability
# ---------------------------------------------------------

def compute_permutation_importance_table(
    fitted_model: Pipeline,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    feature_cols: List[str],
    n_repeats: int,
    seed: int,
    scoring: str = "f1_macro",
) -> pd.DataFrame:
    result = permutation_importance(
        fitted_model,
        X_test,
        y_test,
        n_repeats=n_repeats,
        random_state=seed,
        n_jobs=-1,
        scoring=scoring,
    )
    out = pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return out


# ---------------------------------------------------------
# Plotting
# ---------------------------------------------------------

def save_model_comparison_figure(
    df_test_results: pd.DataFrame,
    out_path: Path,
    metric_col: str = "test_macro_f1",
) -> None:
    if metric_col not in df_test_results.columns:
        raise KeyError(f"Column '{metric_col}' not found in df_test_results.")

    plot_df = df_test_results[["model_name", metric_col]].copy()
    plot_df = plot_df.sort_values(metric_col, ascending=True)

    labels = plot_df["model_name"].tolist()
    values = plot_df[metric_col].tolist()

    plt.figure(figsize=(8, 5))
    plt.barh(labels, values)
    plt.xlabel("Test macro F1")
    plt.ylabel("Model")
    plt.title("Step 5 baseline model comparison (combined baseline features)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def save_feature_ablation_figure(df_ablation_results: pd.DataFrame, out_path: Path) -> None:
    plot_df = df_ablation_results.copy()
    plot_df = plot_df.sort_values("macro_f1", ascending=True)

    labels = plot_df["feature_block"].tolist()
    values = plot_df["macro_f1"].tolist()

    plt.figure(figsize=(8, 5))
    plt.barh(labels, values)
    plt.xlabel("Test macro F1")
    plt.ylabel("Feature block")
    plt.title("Step 5 feature-family ablation (best model family)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_confusion_matrix_figure(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray,
    out_path: Path,
    normalize: str = "true",
) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=classes, normalize=normalize)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, aspect="auto")
    plt.colorbar()
    plt.xticks(np.arange(len(classes)), classes, rotation=90)
    plt.yticks(np.arange(len(classes)), classes)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title("Normalized confusion matrix (best Step 5 model)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    return cm_df


def save_permutation_importance_figure(df_imp: pd.DataFrame, out_path: Path, top_n: int = 20) -> None:
    plot_df = df_imp.head(top_n).copy().sort_values("importance_mean", ascending=True)

    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["feature"], plot_df["importance_mean"])
    plt.xlabel("Permutation importance (mean decrease in macro F1)")
    plt.ylabel("Feature")
    plt.title(f"Top {top_n} permutation importances (best Step 5 model)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


# ---------------------------------------------------------
# Reporting
# ---------------------------------------------------------

def write_run_summary_report(
    out_path: Path,
    cfg: Step5Config,
    merged_df: pd.DataFrame,
    feature_blocks: Dict[str, List[str]],
    best_full_model_name: str,
    best_full_val_metrics: Dict[str, Any],
    best_test_metrics: Dict[str, Any],
) -> None:
    lines = []
    lines.append("Step 5 — Baseline predictive benchmarking summary")
    lines.append("")
    lines.append("Objective")
    lines.append(
        "Establish leakage-aware classical baselines for frozen Step 4 primary-condition classification "
        "using engineered peptide descriptors."
    )
    lines.append("")
    lines.append("Dataset")
    lines.append(f"- Total merged sequences: {len(merged_df):,}")
    lines.append(f"- Number of unique classes: {merged_df[cfg.target_col].nunique()}")
    lines.append(
        f"- Split counts: "
        f"{merged_df[cfg.split_col].value_counts().to_dict()}"
    )
    lines.append("")
    lines.append("Feature blocks")
    for k, v in feature_blocks.items():
        lines.append(f"- {k}: {len(v)} features")
    lines.append("")
    lines.append("Best full-feature validation model")
    lines.append(f"- Model: {best_full_model_name}")
    lines.append(f"- Validation macro F1: {best_full_val_metrics['macro_f1']:.4f}")
    lines.append(f"- Validation balanced accuracy: {best_full_val_metrics['balanced_accuracy']:.4f}")
    lines.append("")
    lines.append("Final frozen-test performance")
    lines.append(f"- Test macro F1: {best_test_metrics['macro_f1']:.4f}")
    lines.append(f"- Test weighted F1: {best_test_metrics['weighted_f1']:.4f}")
    lines.append(f"- Test balanced accuracy: {best_test_metrics['balanced_accuracy']:.4f}")
    lines.append(f"- Test top-1 accuracy: {best_test_metrics['top1_accuracy']:.4f}")
    lines.append(f"- Test top-3 accuracy: {best_test_metrics['top3_accuracy']:.4f}")
    lines.append("")
    lines.append("Recommended manuscript usage")
    lines.append("- Main paper Figure: baseline comparison")
    lines.append("- Main paper Figure: feature-family ablation")
    lines.append("- Supplementary Figure: normalized confusion matrix")
    lines.append("- Supplementary Figure: permutation importance summary")
    lines.append("")

    out_path.write_text("\n".join(lines), encoding="utf-8")


# ---------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------

def run_step5_pipeline(cfg: Step5Config) -> Dict[str, Any]:
    seed = cfg.random_seed
    n_boot = 100 if cfg.fast_mode else cfg.bootstrap_iterations
    n_perm = 5 if cfg.fast_mode else cfg.permutation_repeats

    project_root = _project_root_from_input(cfg.input_model_ready_csv)
    out_dir = project_root / cfg.out_dir_name
    dirs = _mkdirs(out_dir)

    _print_header("Loading Step 4 inputs")
    loaded = load_step4_inputs(cfg)
    seq_df = loaded["seq_df"]
    feat_df = loaded["feat_df"]

    _print_header("Validating and merging frozen Step 4 tables")
    merged_df = validate_and_merge_inputs(seq_df, feat_df, cfg)
    print(f"Merged dataset shape: {merged_df.shape}")

    _print_header("Building baseline feature blocks")
    feature_blocks = infer_feature_blocks(merged_df)
    feature_block_df = pd.DataFrame({
        "feature_block": list(feature_blocks.keys()),
        "n_features": [len(v) for v in feature_blocks.values()],
        "features": [json.dumps(v) for v in feature_blocks.values()],
    })
    feature_block_df.to_csv(dirs["tables"] / "step5_feature_blocks.csv", index=False)

    _print_header("Training and selecting baseline models on combined baseline features")
    combined_block_name = "combined_all"
    combined_features = feature_blocks[combined_block_name]
    if len(combined_features) == 0:
        raise ValueError("Combined baseline feature block is empty; cannot run Step 5.")

    combined_splits = prepare_splits(merged_df, combined_features, cfg)
    model_registry = build_model_registry(seed)

    val_results_df, best_full_record = fit_select_model(
        combined_splits,
        model_registry,
        feature_block_name=combined_block_name,
    )
    val_results_df.to_csv(dirs["tables"] / "step5_validation_model_comparison.csv", index=False)

    best_full_model_name = best_full_record["model_name"]
    best_full_val_metrics = best_full_record["val_metrics"]
    print(f"Selected best full-feature model: {best_full_model_name}")
    print(f"Validation macro F1: {best_full_val_metrics['macro_f1']:.4f}")

    _print_header("Refitting best full-feature model on train+val and evaluating on untouched test set")
    best_full_model = refit_best_model_on_train_val(best_full_record, combined_splits)

    X_test = combined_splits["X_test"]
    y_test = combined_splits["y_test"]
    classes = combined_splits["classes"]

    y_test_pred = best_full_model.predict(X_test)
    y_test_proba = best_full_model.predict_proba(X_test)

    best_test_metrics = evaluate_predictions(
        y_true=y_test,
        y_pred=y_test_pred,
        proba=y_test_proba,
        classes=classes,
        split_name="test",
        model_name=best_full_model_name,
        feature_block=combined_block_name,
    )
    best_test_metrics_df = pd.DataFrame([best_test_metrics])
    best_test_metrics_df.to_csv(dirs["tables"] / "step5_best_model_test_metrics.csv", index=False)

    # Per-class metrics
    per_class_df = per_class_metrics(
        y_true=y_test,
        y_pred=y_test_pred,
        classes=classes,
        model_name=best_full_model_name,
        feature_block=combined_block_name,
    ).sort_values("f1", ascending=False)
    per_class_df.to_csv(dirs["tables"] / "step5_per_class_metrics.csv", index=False)

    _print_header("Creating main baseline comparison table using full feature block")
    # For clean model comparison, refit top-selected model from each family on train+val, then test
    family_best_rows = []
    for model_name in val_results_df["model_name"].drop_duplicates().tolist():
        family_val = val_results_df[val_results_df["model_name"] == model_name].sort_values(
            by=["macro_f1", "balanced_accuracy", "top3_accuracy"], ascending=False
        )
        best_row = family_val.iloc[0].to_dict()

        # reconstruct matching pipeline from registry
        candidate = None
        for pipe, params in model_registry[model_name]:
            if all(best_row.get(k) == v for k, v in params.items()):
                candidate = clone(pipe)
                break
        if candidate is None:
            candidate = clone(model_registry[model_name][0][0])

        X_train_val = pd.concat([combined_splits["X_train"], combined_splits["X_val"]], axis=0).reset_index(drop=True)
        y_train_val = np.concatenate([combined_splits["y_train"], combined_splits["y_val"]])
        candidate.fit(X_train_val, y_train_val)

        pred = candidate.predict(X_test)
        proba = candidate.predict_proba(X_test)

        test_metrics = evaluate_predictions(
            y_true=y_test,
            y_pred=pred,
            proba=proba,
            classes=classes,
            split_name="test",
            model_name=model_name,
            feature_block=combined_block_name,
        )
        family_best_rows.append({**best_row, **{f"test_{k}": v for k, v in test_metrics.items() if k not in ["split", "model_name", "feature_block"]}})

    model_comparison_df = pd.DataFrame(family_best_rows).sort_values(
        by=["test_macro_f1", "test_balanced_accuracy", "test_top3_accuracy"], ascending=False
    ).reset_index(drop=True)
    model_comparison_df.to_csv(dirs["tables"] / "step5_model_comparison_test.csv", index=False)

    _print_header("Running feature-family ablation using best model family")
    best_family_name = best_full_model_name
    ablation_rows = []

    # select best validation params for this family separately within each feature block
    for feature_block_name, feature_cols in feature_blocks.items():
        if len(feature_cols) == 0:
            continue

        splits = prepare_splits(merged_df, feature_cols, cfg)

        family_candidates = {best_family_name: model_registry[best_family_name]}
        family_val_df, family_best_record = fit_select_model(
            splits,
            family_candidates,
            feature_block_name=feature_block_name,
        )

        family_best_model = refit_best_model_on_train_val(family_best_record, splits)
        pred = family_best_model.predict(splits["X_test"])
        proba = family_best_model.predict_proba(splits["X_test"])

        test_metrics = evaluate_predictions(
            y_true=splits["y_test"],
            y_pred=pred,
            proba=proba,
            classes=splits["classes"],
            split_name="test",
            model_name=best_family_name,
            feature_block=feature_block_name,
        )
        test_metrics["n_features"] = len(feature_cols)
        ablation_rows.append(test_metrics)

    ablation_df = pd.DataFrame(ablation_rows).sort_values(
        by=["macro_f1", "balanced_accuracy"], ascending=False
    ).reset_index(drop=True)
    ablation_df.to_csv(dirs["tables"] / "step5_feature_ablation_test.csv", index=False)

    _print_header("Bootstrapping confidence intervals for the best full-feature model")
    boot_df = bootstrap_test_metrics(
        y_true=y_test,
        y_pred=y_test_pred,
        proba=y_test_proba,
        classes=classes,
        n_iter=n_boot,
        seed=seed,
    )
    boot_df.to_csv(dirs["tables"] / "step5_bootstrap_samples.csv", index=False)

    boot_summary_df = summarize_bootstrap(boot_df)
    boot_summary_df.to_csv(dirs["tables"] / "step5_bootstrap_summary.csv", index=False)

    _print_header("Computing permutation importance for the best full-feature model")
    perm_imp_df = compute_permutation_importance_table(
        fitted_model=best_full_model,
        X_test=X_test,
        y_test=y_test,
        feature_cols=combined_features,
        n_repeats=n_perm,
        seed=seed,
        scoring="f1_macro",
    )
    perm_imp_df.to_csv(dirs["tables"] / "step5_permutation_importance.csv", index=False)

    _print_header("Saving figures")
    # Main figures
    save_model_comparison_figure(
        df_test_results=model_comparison_df.rename(columns=lambda c: c.replace("test_", "") if c.startswith("test_") else c)[
            ["model_name", "test_macro_f1"] if "test_macro_f1" in model_comparison_df.columns else ["model_name", "macro_f1"]
        ].rename(columns={"test_macro_f1": "macro_f1"}),
        out_path=dirs["figures_main"] / "step5_main_baseline_model_comparison.png",
    )

    save_feature_ablation_figure(
        df_ablation_results=ablation_df[["feature_block", "macro_f1"]].copy(),
        out_path=dirs["figures_main"] / "step5_main_feature_ablation.png",
    )

    # Supplementary figures
    cm_df = save_confusion_matrix_figure(
        y_true=y_test,
        y_pred=y_test_pred,
        classes=classes,
        out_path=dirs["figures_supplementary"] / "step5_supp_confusion_matrix.png",
        normalize="true",
    )
    cm_df.to_csv(dirs["tables"] / "step5_confusion_matrix_normalized.csv", index=True)

    save_permutation_importance_figure(
        df_imp=perm_imp_df,
        out_path=dirs["figures_supplementary"] / "step5_supp_permutation_importance.png",
        top_n=20,
    )

    _print_header("Writing reports and reproducibility artifacts")
    write_run_summary_report(
        out_path=dirs["reports"] / "step5_run_summary.txt",
        cfg=cfg,
        merged_df=merged_df,
        feature_blocks=feature_blocks,
        best_full_model_name=best_full_model_name,
        best_full_val_metrics=best_full_val_metrics,
        best_test_metrics=best_test_metrics,
    )

    # Save config
    _safe_json_dump(asdict(cfg), dirs["artifacts"] / "step5_config.json")

    # Save artifact hashes
    artifact_hashes = {}
    for f in sorted(dirs["root"].rglob("*")):
        if f.is_file():
            artifact_hashes[str(f.relative_to(dirs["root"]))] = _sha256_of_file(f)
    _safe_json_dump(artifact_hashes, dirs["artifacts"] / "step5_artifact_hashes.json")

    _print_header("Step 5 completed successfully")
    print(f"Output directory: {dirs['root']}")
    print("-" * 80)
    print("Best full-feature model:", best_full_model_name)
    print("Test macro F1:", round(best_test_metrics["macro_f1"], 4))
    print("Test balanced accuracy:", round(best_test_metrics["balanced_accuracy"], 4))
    print("Test top-1 accuracy:", round(best_test_metrics["top1_accuracy"], 4))
    print("Test top-3 accuracy:", round(best_test_metrics["top3_accuracy"], 4))

    return {
        "output_dir": str(dirs["root"]),
        "best_model_name": best_full_model_name,
        "best_test_metrics": best_test_metrics,
        "model_comparison_df": model_comparison_df,
        "ablation_df": ablation_df,
        "per_class_df": per_class_df,
        "bootstrap_summary_df": boot_summary_df,
        "permutation_importance_df": perm_imp_df,
        "feature_blocks": feature_blocks,
    }


# ---------------------------------------------------------
# Notebook entry point
# ---------------------------------------------------------

def main_notebook(
    input_model_ready_csv: str,
    input_baseline_features_csv: str,
    input_preprocessing_contract_json: Optional[str] = None,
    input_condition_id_mapping_json: Optional[str] = None,
    out_dir_name: str = "step5",
    fast_mode: bool = False,
    bootstrap_iterations: int = 500,
    permutation_repeats: int = 10,
    random_seed: int = 42,
):
    cfg = Step5Config(
        input_model_ready_csv=input_model_ready_csv,
        input_baseline_features_csv=input_baseline_features_csv,
        input_preprocessing_contract_json=input_preprocessing_contract_json,
        input_condition_id_mapping_json=input_condition_id_mapping_json,
        out_dir_name=out_dir_name,
        fast_mode=fast_mode,
        bootstrap_iterations=bootstrap_iterations,
        permutation_repeats=permutation_repeats,
        random_seed=random_seed,
    )
    return run_step5_pipeline(cfg)


# ---------------------------------------------------------
# Example run
# ---------------------------------------------------------

if __name__ == "__main__":
    results = main_notebook(
        input_model_ready_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv",
        input_baseline_features_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv",
        input_preprocessing_contract_json="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts/step4_preprocessing_contract.json",
        input_condition_id_mapping_json="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts/step4_condition_id_mapping.json",
        out_dir_name="step5",
        fast_mode=False,                 # set True for quick smoke test
        bootstrap_iterations=500,        # reduce to 100 for faster run if needed
        permutation_repeats=10,          # reduce to 5 for faster run if needed
        random_seed=42,
    )
    print("Step 5 finished.")

Step 5 — Baseline predictive benchmarking

In [ ]:
# ================================
# Step 5 — Baseline predictive benchmarking
# Primary task: multiclass classification of primary_condition_id
# Inputs: frozen Step 4 outputs
# ================================

from __future__ import annotations

import json
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    log_loss,
)
from sklearn.inspection import permutation_importance


# ---------------------------------------------------------
# Reproducibility helpers
# ---------------------------------------------------------

GLOBAL_SEED = 42
RNG = np.random.default_rng(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


# ---------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------

@dataclass
class Step5Config:
    input_model_ready_csv: str
    input_baseline_features_csv: str
    input_preprocessing_contract_json: Optional[str] = None
    input_condition_id_mapping_json: Optional[str] = None
    out_dir_name: str = "step5"

    merge_key: str = "sequence_sha256"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"
    sequence_col: str = "sequence"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    random_seed: int = 42
    bootstrap_iterations: int = 500
    permutation_repeats: int = 10
    topk_values: Tuple[int, ...] = (1, 3)

    fast_mode: bool = False


# ---------------------------------------------------------
# Small utilities
# ---------------------------------------------------------

def _safe_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def _sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _mkdirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables": base_dir / "tables",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "models": base_dir / "models",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def _project_root_from_input(input_csv: str) -> Path:
    """
    Given:
      /.../nature_computational2/step4/tables/step4_model_ready_sequences.csv
    return:
      /.../nature_computational2
    """
    p = Path(input_csv).resolve()
    if p.parent.name == "tables" and p.parent.parent.name == "step4":
        return p.parent.parent.parent
    return p.parent


def _print_header(msg: str) -> None:
    print("=" * 100)
    print(msg)
    print("=" * 100)


# ---------------------------------------------------------
# Loading and validation
# ---------------------------------------------------------

def load_step4_inputs(cfg: Step5Config) -> Dict[str, Any]:
    seq_path = Path(cfg.input_model_ready_csv).resolve()
    feat_path = Path(cfg.input_baseline_features_csv).resolve()

    if not seq_path.exists():
        raise FileNotFoundError(f"Missing model-ready CSV: {seq_path}")
    if not feat_path.exists():
        raise FileNotFoundError(f"Missing baseline-features CSV: {feat_path}")

    seq_df = pd.read_csv(seq_path)
    feat_df = pd.read_csv(feat_path)

    contract = None
    condition_mapping = None

    if cfg.input_preprocessing_contract_json:
        contract_path = Path(cfg.input_preprocessing_contract_json).resolve()
        if not contract_path.exists():
            raise FileNotFoundError(f"Missing preprocessing contract JSON: {contract_path}")
        with open(contract_path, "r", encoding="utf-8") as f:
            contract = json.load(f)

    if cfg.input_condition_id_mapping_json:
        mapping_path = Path(cfg.input_condition_id_mapping_json).resolve()
        if not mapping_path.exists():
            raise FileNotFoundError(f"Missing condition ID mapping JSON: {mapping_path}")
        with open(mapping_path, "r", encoding="utf-8") as f:
            condition_mapping = json.load(f)

    return {
        "seq_df": seq_df,
        "feat_df": feat_df,
        "contract": contract,
        "condition_mapping": condition_mapping,
        "seq_path": seq_path,
        "feat_path": feat_path,
    }


def validate_and_merge_inputs(seq_df: pd.DataFrame, feat_df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    required_seq_cols = [cfg.merge_key, cfg.split_col, cfg.target_col, cfg.sequence_col]
    for col in required_seq_cols:
        if col not in seq_df.columns:
            raise ValueError(f"Required column '{col}' not found in sequence table.")

    if cfg.merge_key not in feat_df.columns:
        raise ValueError(f"Required merge key '{cfg.merge_key}' not found in baseline feature table.")

    if seq_df[cfg.merge_key].isna().any():
        raise ValueError(f"Merge key '{cfg.merge_key}' contains missing values in sequence table.")
    if feat_df[cfg.merge_key].isna().any():
        raise ValueError(f"Merge key '{cfg.merge_key}' contains missing values in feature table.")

    if not seq_df[cfg.merge_key].is_unique:
        dup_n = seq_df[cfg.merge_key].duplicated().sum()
        raise ValueError(f"Sequence table merge key is not unique. Duplicates: {dup_n}")
    if not feat_df[cfg.merge_key].is_unique:
        dup_n = feat_df[cfg.merge_key].duplicated().sum()
        raise ValueError(f"Feature table merge key is not unique. Duplicates: {dup_n}")

    feat_extra_cols = [cfg.merge_key] + [c for c in feat_df.columns if c not in seq_df.columns and c != cfg.merge_key]
    merged = seq_df.merge(feat_df[feat_extra_cols], on=cfg.merge_key, how="inner", validate="one_to_one")

    if len(merged) != len(seq_df) or len(merged) != len(feat_df):
        raise ValueError(
            f"Merge row count mismatch: merged={len(merged)}, seq={len(seq_df)}, feat={len(feat_df)}"
        )

    expected_splits = {cfg.train_label, cfg.val_label, cfg.test_label}
    found_splits = set(merged[cfg.split_col].dropna().unique().tolist())
    if not expected_splits.issubset(found_splits):
        raise ValueError(f"Expected splits {expected_splits}, found {found_splits}")

    if merged[cfg.target_col].isna().any():
        n_missing = int(merged[cfg.target_col].isna().sum())
        raise ValueError(f"Target column '{cfg.target_col}' contains missing values: {n_missing}")

    return merged


# ---------------------------------------------------------
# Feature blocks
# ---------------------------------------------------------

def infer_feature_blocks(df: pd.DataFrame) -> Dict[str, List[str]]:
    aa_frac_raw = sorted([c for c in df.columns if c.startswith("aa_frac_") and not c.endswith("_z")])
    aa_frac_z = sorted([c for c in df.columns if c.startswith("aa_frac_") and c.endswith("_z")])

    global_raw_candidates = [
        "length", "net_charge_pH7", "hydrophobicity_KD", "entropy", "dominant_aa_frac"
    ]
    global_z_candidates = [
        "length_z", "net_charge_pH7_z", "hydrophobicity_KD_z", "entropy_z", "dominant_aa_frac_z"
    ]

    global_raw = [c for c in global_raw_candidates if c in df.columns]
    global_z = [c for c in global_z_candidates if c in df.columns]

    combined_raw = global_raw + aa_frac_raw
    combined_z = global_z + aa_frac_z
    combined_all = combined_raw + combined_z

    feature_blocks = {
        "global_raw": global_raw,
        "aa_frac_raw": aa_frac_raw,
        "global_z": global_z,
        "aa_frac_z": aa_frac_z,
        "combined_raw": combined_raw,
        "combined_z": combined_z,
        "combined_all": combined_all,
    }

    for name, cols in feature_blocks.items():
        if len(cols) == 0:
            print(f"Warning: feature block '{name}' is empty.")
    return feature_blocks


# ---------------------------------------------------------
# Split preparation
# ---------------------------------------------------------

def prepare_splits(df: pd.DataFrame, feature_cols: List[str], cfg: Step5Config) -> Dict[str, Any]:
    use_cols = [cfg.merge_key, cfg.sequence_col, cfg.split_col, cfg.target_col] + feature_cols
    use_cols = [c for c in use_cols if c in df.columns]
    sub = df[use_cols].copy()

    train_df = sub[sub[cfg.split_col] == cfg.train_label].reset_index(drop=True)
    val_df = sub[sub[cfg.split_col] == cfg.val_label].reset_index(drop=True)
    test_df = sub[sub[cfg.split_col] == cfg.test_label].reset_index(drop=True)

    X_train = train_df[feature_cols].copy()
    y_train = train_df[cfg.target_col].astype(str).to_numpy()

    X_val = val_df[feature_cols].copy()
    y_val = val_df[cfg.target_col].astype(str).to_numpy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[cfg.target_col].astype(str).to_numpy()

    classes = np.array(sorted(pd.Series(y_train).unique().tolist()))
    if not set(pd.Series(y_val).unique()).issubset(set(classes)):
        raise ValueError("Validation set contains classes absent from train set.")
    if not set(pd.Series(y_test).unique()).issubset(set(classes)):
        raise ValueError("Test set contains classes absent from train set.")

    return {
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "classes": classes,
    }


# ---------------------------------------------------------
# Model definitions
# ---------------------------------------------------------

def build_model_registry(seed: int) -> Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]]:
    registry: Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]] = {}

    registry["dummy_majority"] = [
        (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]),
            {},
        )
    ]

    lr_grid = []
    for C in [0.1, 1.0, 3.0, 10.0]:
        lr_grid.append((
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    C=C,
                    class_weight="balanced",
                    multi_class="multinomial",
                    solver="lbfgs",
                    max_iter=3000,
                    random_state=seed,
                )),
            ]),
            {"C": C},
        ))
    registry["logistic_regression"] = lr_grid

    rf_grid = []
    for n_estimators in [300]:
        for max_depth in [None, 20]:
            for min_samples_leaf in [1, 3]:
                rf_grid.append((
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("model", RandomForestClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            class_weight="balanced_subsample",
                            n_jobs=-1,
                            random_state=seed,
                        )),
                    ]),
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                    },
                ))
    registry["random_forest"] = rf_grid

    et_grid = []
    for n_estimators in [300]:
        for max_depth in [None, 20]:
            for min_samples_leaf in [1, 3]:
                et_grid.append((
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("model", ExtraTreesClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            class_weight="balanced",
                            n_jobs=-1,
                            random_state=seed,
                        )),
                    ]),
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                    },
                ))
    registry["extra_trees"] = et_grid

    return registry


# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

def topk_accuracy(y_true: np.ndarray, proba: np.ndarray, classes: np.ndarray, k: int) -> float:
    class_to_idx = {c: i for i, c in enumerate(classes)}
    true_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    topk_idx = np.argsort(-proba, axis=1)[:, :k]
    hits = np.any(topk_idx == true_idx[:, None], axis=1)
    return float(np.mean(hits))


def safe_macro_auroc(y_true: np.ndarray, proba: np.ndarray, classes: np.ndarray) -> float:
    try:
        y_true_bin = label_binarize(y_true, classes=classes)
        if y_true_bin.shape[1] == 1:
            return float("nan")
        return float(roc_auc_score(y_true_bin, proba, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: np.ndarray,
    classes: np.ndarray,
    split_name: str,
    model_name: str,
    feature_block: str,
) -> Dict[str, Any]:
    metrics = {
        "split": split_name,
        "model_name": model_name,
        "feature_block": feature_block,
        "n_samples": int(len(y_true)),
        "n_classes": int(len(classes)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_auroc_ovr": safe_macro_auroc(y_true, proba, classes),
        "log_loss": float(log_loss(y_true, proba, labels=classes)) if proba is not None else float("nan"),
    }

    for k in [1, 3]:
        if proba is not None and k <= len(classes):
            metrics[f"top{k}_accuracy"] = topk_accuracy(y_true, proba, classes, k)
        else:
            metrics[f"top{k}_accuracy"] = float("nan")
    return metrics


def per_class_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray,
    model_name: str,
    feature_block: str,
) -> pd.DataFrame:
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    return pd.DataFrame({
        "model_name": model_name,
        "feature_block": feature_block,
        "class_label": classes,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "support": support.astype(int),
    })


# ---------------------------------------------------------
# Training / selection
# ---------------------------------------------------------

def fit_select_model(
    splits: Dict[str, Any],
    model_registry: Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]],
    feature_block_name: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    rows = []

    X_train = splits["X_train"]
    y_train = splits["y_train"]
    X_val = splits["X_val"]
    y_val = splits["y_val"]
    classes = splits["classes"]

    best_record = None
    best_score = -np.inf

    for model_name, candidates in model_registry.items():
        for candidate_pipeline, param_dict in candidates:
            model = clone(candidate_pipeline)
            model.fit(X_train, y_train)

            y_val_pred = model.predict(X_val)
            if hasattr(model, "predict_proba"):
                val_proba = model.predict_proba(X_val)
            else:
                val_proba = np.full((len(X_val), len(classes)), fill_value=1.0 / len(classes), dtype=float)

            metrics = evaluate_predictions(
                y_true=y_val,
                y_pred=y_val_pred,
                proba=val_proba,
                classes=classes,
                split_name="val",
                model_name=model_name,
                feature_block=feature_block_name,
            )
            metrics.update(param_dict)
            rows.append(metrics)

            score = metrics["macro_f1"]
            if score > best_score:
                best_score = score
                best_record = {
                    "model_name": model_name,
                    "params": param_dict,
                    "pipeline": model,
                    "val_metrics": metrics,
                }

    val_results_df = pd.DataFrame(rows).sort_values(
        by=["macro_f1", "balanced_accuracy", "top3_accuracy"], ascending=False
    ).reset_index(drop=True)

    if best_record is None:
        raise RuntimeError("No model could be selected on validation data.")

    return val_results_df, best_record


def refit_best_model_on_train_val(best_record: Dict[str, Any], splits: Dict[str, Any]) -> Pipeline:
    best_model = clone(best_record["pipeline"])
    X_train_val = pd.concat([splits["X_train"], splits["X_val"]], axis=0).reset_index(drop=True)
    y_train_val = np.concatenate([splits["y_train"], splits["y_val"]])
    best_model.fit(X_train_val, y_train_val)
    return best_model


# ---------------------------------------------------------
# Bootstrap confidence intervals
# ---------------------------------------------------------

def bootstrap_test_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: np.ndarray,
    classes: np.ndarray,
    n_iter: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []

    for i in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        pp = proba[idx]

        rows.append({
            "bootstrap_iter": i,
            "accuracy": accuracy_score(yt, yp),
            "balanced_accuracy": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "weighted_f1": f1_score(yt, yp, average="weighted"),
            "macro_auroc_ovr": safe_macro_auroc(yt, pp, classes),
            "top1_accuracy": topk_accuracy(yt, pp, classes, 1),
            "top3_accuracy": topk_accuracy(yt, pp, classes, 3) if len(classes) >= 3 else np.nan,
        })

    return pd.DataFrame(rows)


def summarize_bootstrap(df_boot: pd.DataFrame) -> pd.DataFrame:
    metrics = [c for c in df_boot.columns if c != "bootstrap_iter"]
    rows = []
    for m in metrics:
        vals = pd.Series(df_boot[m]).dropna()
        rows.append({
            "metric": m,
            "mean": float(vals.mean()),
            "median": float(vals.median()),
            "ci_lower_2.5": float(vals.quantile(0.025)),
            "ci_upper_97.5": float(vals.quantile(0.975)),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Interpretability
# ---------------------------------------------------------

def compute_permutation_importance_table(
    fitted_model: Pipeline,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    feature_cols: List[str],
    n_repeats: int,
    seed: int,
    scoring: str = "f1_macro",
) -> pd.DataFrame:
    result = permutation_importance(
        fitted_model,
        X_test,
        y_test,
        n_repeats=n_repeats,
        random_state=seed,
        n_jobs=-1,
        scoring=scoring,
    )
    out = pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return out


# ---------------------------------------------------------
# Plotting
# ---------------------------------------------------------

def save_model_comparison_figure(
    df_test_results: pd.DataFrame,
    out_path: Path,
    metric_col: str = "test_macro_f1",
) -> None:
    if "model_name" not in df_test_results.columns:
        raise KeyError(f"'model_name' not found in df_test_results columns: {list(df_test_results.columns)}")
    if metric_col not in df_test_results.columns:
        raise KeyError(f"Column '{metric_col}' not found in df_test_results columns: {list(df_test_results.columns)}")

    plot_df = df_test_results[["model_name", metric_col]].copy()
    plot_df = plot_df.sort_values(metric_col, ascending=True)

    plt.figure(figsize=(8, 5))
    plt.barh(plot_df["model_name"], plot_df[metric_col])
    plt.xlabel("Test macro F1")
    plt.ylabel("Model")
    plt.title("Step 5 baseline model comparison (combined baseline features)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_feature_ablation_figure(df_ablation_results: pd.DataFrame, out_path: Path) -> None:
    if "feature_block" not in df_ablation_results.columns or "macro_f1" not in df_ablation_results.columns:
        raise KeyError(
            f"df_ablation_results must contain ['feature_block', 'macro_f1']; got {list(df_ablation_results.columns)}"
        )

    plot_df = df_ablation_results[["feature_block", "macro_f1"]].copy()
    plot_df = plot_df.sort_values("macro_f1", ascending=True)

    plt.figure(figsize=(8, 5))
    plt.barh(plot_df["feature_block"], plot_df["macro_f1"])
    plt.xlabel("Test macro F1")
    plt.ylabel("Feature block")
    plt.title("Step 5 feature-family ablation (best model family)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_confusion_matrix_figure(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray,
    out_path: Path,
    normalize: str = "true",
) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=classes, normalize=normalize)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, aspect="auto")
    plt.colorbar()
    plt.xticks(np.arange(len(classes)), classes, rotation=90)
    plt.yticks(np.arange(len(classes)), classes)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title("Normalized confusion matrix (best Step 5 model)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    return cm_df


def save_permutation_importance_figure(df_imp: pd.DataFrame, out_path: Path, top_n: int = 20) -> None:
    if "feature" not in df_imp.columns or "importance_mean" not in df_imp.columns:
        raise KeyError(
            f"df_imp must contain ['feature', 'importance_mean']; got {list(df_imp.columns)}"
        )

    plot_df = df_imp.head(top_n).copy().sort_values("importance_mean", ascending=True)

    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["feature"], plot_df["importance_mean"])
    plt.xlabel("Permutation importance (mean decrease in macro F1)")
    plt.ylabel("Feature")
    plt.title(f"Top {top_n} permutation importances (best Step 5 model)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


# ---------------------------------------------------------
# Reporting
# ---------------------------------------------------------

def write_run_summary_report(
    out_path: Path,
    cfg: Step5Config,
    merged_df: pd.DataFrame,
    feature_blocks: Dict[str, List[str]],
    best_full_model_name: str,
    best_full_val_metrics: Dict[str, Any],
    best_test_metrics: Dict[str, Any],
) -> None:
    lines = []
    lines.append("Step 5 — Baseline predictive benchmarking summary")
    lines.append("")
    lines.append("Objective")
    lines.append(
        "Establish leakage-aware classical baselines for frozen Step 4 primary-condition classification "
        "using engineered peptide descriptors."
    )
    lines.append("")
    lines.append("Dataset")
    lines.append(f"- Total merged sequences: {len(merged_df):,}")
    lines.append(f"- Number of unique classes: {merged_df[cfg.target_col].nunique()}")
    lines.append(f"- Split counts: {merged_df[cfg.split_col].value_counts().to_dict()}")
    lines.append("")
    lines.append("Feature blocks")
    for k, v in feature_blocks.items():
        lines.append(f"- {k}: {len(v)} features")
    lines.append("")
    lines.append("Best full-feature validation model")
    lines.append(f"- Model: {best_full_model_name}")
    lines.append(f"- Validation macro F1: {best_full_val_metrics['macro_f1']:.4f}")
    lines.append(f"- Validation balanced accuracy: {best_full_val_metrics['balanced_accuracy']:.4f}")
    lines.append("")
    lines.append("Final frozen-test performance")
    lines.append(f"- Test macro F1: {best_test_metrics['macro_f1']:.4f}")
    lines.append(f"- Test weighted F1: {best_test_metrics['weighted_f1']:.4f}")
    lines.append(f"- Test balanced accuracy: {best_test_metrics['balanced_accuracy']:.4f}")
    lines.append(f"- Test top-1 accuracy: {best_test_metrics['top1_accuracy']:.4f}")
    lines.append(f"- Test top-3 accuracy: {best_test_metrics['top3_accuracy']:.4f}")
    lines.append("")
    lines.append("Recommended manuscript usage")
    lines.append("- Main paper Figure: baseline comparison")
    lines.append("- Main paper Figure: feature-family ablation")
    lines.append("- Supplementary Figure: normalized confusion matrix")
    lines.append("- Supplementary Figure: permutation importance summary")
    lines.append("")

    out_path.write_text("\n".join(lines), encoding="utf-8")


# ---------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------

def run_step5_pipeline(cfg: Step5Config) -> Dict[str, Any]:
    seed = cfg.random_seed
    n_boot = 100 if cfg.fast_mode else cfg.bootstrap_iterations
    n_perm = 5 if cfg.fast_mode else cfg.permutation_repeats

    project_root = _project_root_from_input(cfg.input_model_ready_csv)
    out_dir = project_root / cfg.out_dir_name
    dirs = _mkdirs(out_dir)

    _print_header("Loading Step 4 inputs")
    loaded = load_step4_inputs(cfg)
    seq_df = loaded["seq_df"]
    feat_df = loaded["feat_df"]

    _print_header("Validating and merging frozen Step 4 tables")
    merged_df = validate_and_merge_inputs(seq_df, feat_df, cfg)
    print(f"Merged dataset shape: {merged_df.shape}")

    _print_header("Building baseline feature blocks")
    feature_blocks = infer_feature_blocks(merged_df)
    feature_block_df = pd.DataFrame({
        "feature_block": list(feature_blocks.keys()),
        "n_features": [len(v) for v in feature_blocks.values()],
        "features": [json.dumps(v) for v in feature_blocks.values()],
    })
    feature_block_df.to_csv(dirs["tables"] / "step5_feature_blocks.csv", index=False)

    _print_header("Training and selecting baseline models on combined baseline features")
    combined_block_name = "combined_all"
    combined_features = feature_blocks[combined_block_name]
    if len(combined_features) == 0:
        raise ValueError("Combined baseline feature block is empty; cannot run Step 5.")

    combined_splits = prepare_splits(merged_df, combined_features, cfg)
    model_registry = build_model_registry(seed)

    val_results_df, best_full_record = fit_select_model(
        combined_splits,
        model_registry,
        feature_block_name=combined_block_name,
    )
    val_results_df.to_csv(dirs["tables"] / "step5_validation_model_comparison.csv", index=False)

    best_full_model_name = best_full_record["model_name"]
    best_full_val_metrics = best_full_record["val_metrics"]
    print(f"Selected best full-feature model: {best_full_model_name}")
    print(f"Validation macro F1: {best_full_val_metrics['macro_f1']:.4f}")

    _print_header("Refitting best full-feature model on train+val and evaluating on untouched test set")
    best_full_model = refit_best_model_on_train_val(best_full_record, combined_splits)

    X_test = combined_splits["X_test"]
    y_test = combined_splits["y_test"]
    classes = combined_splits["classes"]

    y_test_pred = best_full_model.predict(X_test)
    y_test_proba = best_full_model.predict_proba(X_test)

    best_test_metrics = evaluate_predictions(
        y_true=y_test,
        y_pred=y_test_pred,
        proba=y_test_proba,
        classes=classes,
        split_name="test",
        model_name=best_full_model_name,
        feature_block=combined_block_name,
    )
    pd.DataFrame([best_test_metrics]).to_csv(dirs["tables"] / "step5_best_model_test_metrics.csv", index=False)

    per_class_df = per_class_metrics(
        y_true=y_test,
        y_pred=y_test_pred,
        classes=classes,
        model_name=best_full_model_name,
        feature_block=combined_block_name,
    ).sort_values("f1", ascending=False)
    per_class_df.to_csv(dirs["tables"] / "step5_per_class_metrics.csv", index=False)

    _print_header("Creating main baseline comparison table using full feature block")
    family_best_rows = []

    X_train_val = pd.concat([combined_splits["X_train"], combined_splits["X_val"]], axis=0).reset_index(drop=True)
    y_train_val = np.concatenate([combined_splits["y_train"], combined_splits["y_val"]])

    for model_name in val_results_df["model_name"].drop_duplicates().tolist():
        family_val = val_results_df[val_results_df["model_name"] == model_name].sort_values(
            by=["macro_f1", "balanced_accuracy", "top3_accuracy"], ascending=False
        )
        best_row = family_val.iloc[0].to_dict()

        candidate = None
        for pipe, params in model_registry[model_name]:
            if all(best_row.get(k) == v for k, v in params.items()):
                candidate = clone(pipe)
                break
        if candidate is None:
            candidate = clone(model_registry[model_name][0][0])

        candidate.fit(X_train_val, y_train_val)

        pred = candidate.predict(X_test)
        proba = candidate.predict_proba(X_test)

        test_metrics = evaluate_predictions(
            y_true=y_test,
            y_pred=pred,
            proba=proba,
            classes=classes,
            split_name="test",
            model_name=model_name,
            feature_block=combined_block_name,
        )

        row = {
            "model_name": model_name,
            "feature_block": combined_block_name,
            "val_macro_f1": best_row["macro_f1"],
            "val_balanced_accuracy": best_row["balanced_accuracy"],
            "val_top1_accuracy": best_row["top1_accuracy"],
            "val_top3_accuracy": best_row["top3_accuracy"],
            "test_accuracy": test_metrics["accuracy"],
            "test_balanced_accuracy": test_metrics["balanced_accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_macro_auroc_ovr": test_metrics["macro_auroc_ovr"],
            "test_log_loss": test_metrics["log_loss"],
            "test_top1_accuracy": test_metrics["top1_accuracy"],
            "test_top3_accuracy": test_metrics["top3_accuracy"],
        }

        for k in ["C", "n_estimators", "max_depth", "min_samples_leaf"]:
            if k in best_row:
                row[k] = best_row[k]

        family_best_rows.append(row)

    model_comparison_df = pd.DataFrame(family_best_rows).sort_values(
        by=["test_macro_f1", "test_balanced_accuracy", "test_top3_accuracy"], ascending=False
    ).reset_index(drop=True)
    model_comparison_df.to_csv(dirs["tables"] / "step5_model_comparison_test.csv", index=False)

    _print_header("Running feature-family ablation using best model family")
    best_family_name = best_full_model_name
    ablation_rows = []

    for feature_block_name, feature_cols in feature_blocks.items():
        if len(feature_cols) == 0:
            continue

        splits = prepare_splits(merged_df, feature_cols, cfg)
        family_candidates = {best_family_name: model_registry[best_family_name]}

        _, family_best_record = fit_select_model(
            splits,
            family_candidates,
            feature_block_name=feature_block_name,
        )

        family_best_model = refit_best_model_on_train_val(family_best_record, splits)
        pred = family_best_model.predict(splits["X_test"])
        proba = family_best_model.predict_proba(splits["X_test"])

        test_metrics = evaluate_predictions(
            y_true=splits["y_test"],
            y_pred=pred,
            proba=proba,
            classes=splits["classes"],
            split_name="test",
            model_name=best_family_name,
            feature_block=feature_block_name,
        )
        test_metrics["n_features"] = len(feature_cols)
        ablation_rows.append(test_metrics)

    ablation_df = pd.DataFrame(ablation_rows).sort_values(
        by=["macro_f1", "balanced_accuracy"], ascending=False
    ).reset_index(drop=True)
    ablation_df.to_csv(dirs["tables"] / "step5_feature_ablation_test.csv", index=False)

    _print_header("Bootstrapping confidence intervals for the best full-feature model")
    boot_df = bootstrap_test_metrics(
        y_true=y_test,
        y_pred=y_test_pred,
        proba=y_test_proba,
        classes=classes,
        n_iter=n_boot,
        seed=seed,
    )
    boot_df.to_csv(dirs["tables"] / "step5_bootstrap_samples.csv", index=False)

    boot_summary_df = summarize_bootstrap(boot_df)
    boot_summary_df.to_csv(dirs["tables"] / "step5_bootstrap_summary.csv", index=False)

    _print_header("Computing permutation importance for the best full-feature model")
    perm_imp_df = compute_permutation_importance_table(
        fitted_model=best_full_model,
        X_test=X_test,
        y_test=y_test,
        feature_cols=combined_features,
        n_repeats=n_perm,
        seed=seed,
        scoring="f1_macro",
    )
    perm_imp_df.to_csv(dirs["tables"] / "step5_permutation_importance.csv", index=False)

    _print_header("Saving figures")

    # Main figure 1
    save_model_comparison_figure(
        df_test_results=model_comparison_df,
        out_path=dirs["figures_main"] / "step5_main_baseline_model_comparison.png",
        metric_col="test_macro_f1",
    )

    # Main figure 2
    save_feature_ablation_figure(
        df_ablation_results=ablation_df[["feature_block", "macro_f1"]].copy(),
        out_path=dirs["figures_main"] / "step5_main_feature_ablation.png",
    )

    # Supplementary figure 1
    cm_df = save_confusion_matrix_figure(
        y_true=y_test,
        y_pred=y_test_pred,
        classes=classes,
        out_path=dirs["figures_supplementary"] / "step5_supp_confusion_matrix.png",
        normalize="true",
    )
    cm_df.to_csv(dirs["tables"] / "step5_confusion_matrix_normalized.csv", index=True)

    # Supplementary figure 2
    save_permutation_importance_figure(
        df_imp=perm_imp_df,
        out_path=dirs["figures_supplementary"] / "step5_supp_permutation_importance.png",
        top_n=20,
    )

    _print_header("Writing reports and reproducibility artifacts")
    write_run_summary_report(
        out_path=dirs["reports"] / "step5_run_summary.txt",
        cfg=cfg,
        merged_df=merged_df,
        feature_blocks=feature_blocks,
        best_full_model_name=best_full_model_name,
        best_full_val_metrics=best_full_val_metrics,
        best_test_metrics=best_test_metrics,
    )

    _safe_json_dump(asdict(cfg), dirs["artifacts"] / "step5_config.json")

    artifact_hashes = {}
    for f in sorted(dirs["root"].rglob("*")):
        if f.is_file():
            artifact_hashes[str(f.relative_to(dirs["root"]))] = _sha256_of_file(f)
    _safe_json_dump(artifact_hashes, dirs["artifacts"] / "step5_artifact_hashes.json")

    _print_header("Step 5 completed successfully")
    print(f"Output directory: {dirs['root']}")
    print("-" * 80)
    print("Best full-feature model:", best_full_model_name)
    print("Test macro F1:", round(best_test_metrics["macro_f1"], 4))
    print("Test balanced accuracy:", round(best_test_metrics["balanced_accuracy"], 4))
    print("Test top-1 accuracy:", round(best_test_metrics["top1_accuracy"], 4))
    print("Test top-3 accuracy:", round(best_test_metrics["top3_accuracy"], 4))

    return {
        "output_dir": str(dirs["root"]),
        "best_model_name": best_full_model_name,
        "best_test_metrics": best_test_metrics,
        "model_comparison_df": model_comparison_df,
        "ablation_df": ablation_df,
        "per_class_df": per_class_df,
        "bootstrap_summary_df": boot_summary_df,
        "permutation_importance_df": perm_imp_df,
        "feature_blocks": feature_blocks,
    }


# ---------------------------------------------------------
# Notebook entry point
# ---------------------------------------------------------

def main_notebook(
    input_model_ready_csv: str,
    input_baseline_features_csv: str,
    input_preprocessing_contract_json: Optional[str] = None,
    input_condition_id_mapping_json: Optional[str] = None,
    out_dir_name: str = "step5",
    fast_mode: bool = False,
    bootstrap_iterations: int = 500,
    permutation_repeats: int = 10,
    random_seed: int = 42,
):
    cfg = Step5Config(
        input_model_ready_csv=input_model_ready_csv,
        input_baseline_features_csv=input_baseline_features_csv,
        input_preprocessing_contract_json=input_preprocessing_contract_json,
        input_condition_id_mapping_json=input_condition_id_mapping_json,
        out_dir_name=out_dir_name,
        fast_mode=fast_mode,
        bootstrap_iterations=bootstrap_iterations,
        permutation_repeats=permutation_repeats,
        random_seed=random_seed,
    )
    return run_step5_pipeline(cfg)


# ---------------------------------------------------------
# Example run
# ---------------------------------------------------------

if __name__ == "__main__":
    results = main_notebook(
        input_model_ready_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv",
        input_baseline_features_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv",
        input_preprocessing_contract_json="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts/step4_preprocessing_contract.json",
        input_condition_id_mapping_json="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts/step4_condition_id_mapping.json",
        out_dir_name="step5",
        fast_mode=False,
        bootstrap_iterations=500,
        permutation_repeats=10,
        random_seed=42,
    )
    print("Step 5 finished.")

In [ ]:
import sys
import ipykernel
print("Python:", sys.executable)
print("ipykernel:", ipykernel.__version__)

In [ ]:
# ================================
# Step 5 — Upgraded baseline benchmarking + audit pipeline
# Primary task: multiclass classification of primary_condition_id
# Inputs: frozen Step 4 outputs
# ================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    log_loss,
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA


# ---------------------------------------------------------
# Global settings
# ---------------------------------------------------------

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

COLOR_MODELS = {
    "dummy_majority": "#999999",
    "logistic_regression": "#0072B2",
    "random_forest": "#D55E00",
    "extra_trees": "#009E73",
}

COLOR_BLOCKS = {
    "global_raw": "#0072B2",
    "global_z": "#56B4E9",
    "aa_frac_raw": "#009E73",
    "aa_frac_z": "#F0E442",
    "combined_raw": "#E69F00",
    "combined_z": "#CC79A7",
    "combined_all": "#D55E00",
    "condition_core_raw": "#0072B2",
    "condition_core_z": "#56B4E9",
    "minimal_condition_proxy": "#999999",
}


# ---------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------

@dataclass
class Step5Config:
    input_model_ready_csv: str
    input_baseline_features_csv: str
    input_preprocessing_contract_json: Optional[str] = None
    input_condition_id_mapping_json: Optional[str] = None
    out_dir_name: str = "step5"

    merge_key: str = "sequence_sha256"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"
    sequence_col: str = "sequence"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    random_seed: int = 42
    bootstrap_iterations: int = 500
    permutation_repeats: int = 10
    topk_values: Tuple[int, ...] = (1, 3)

    fast_mode: bool = False
    generate_audit_figures: bool = True
    generate_manuscript_figures: bool = True

    # Audit thresholds
    descriptor_trivial_threshold: float = 0.98
    reconstructability_gap_threshold: float = 0.02
    pca_variance_threshold: float = 0.80


# ---------------------------------------------------------
# Small utilities
# ---------------------------------------------------------

def _safe_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def _sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _mkdirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables": base_dir / "tables",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "figures_audit": base_dir / "figures_audit",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "models": base_dir / "models",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def _project_root_from_input(input_csv: str) -> Path:
    p = Path(input_csv).resolve()
    if p.parent.name == "tables" and p.parent.parent.name == "step4":
        return p.parent.parent.parent
    return p.parent


def _print_header(msg: str) -> None:
    print("=" * 100)
    print(msg)
    print("=" * 100)


def _savefig_both(fig: plt.Figure, out_base: Path) -> None:
    fig.savefig(out_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")


def _point_range_plot(
    df: pd.DataFrame,
    label_col: str,
    value_col: str,
    lower_col: Optional[str],
    upper_col: Optional[str],
    title: str,
    xlabel: str,
    out_base: Path,
    color_map: Optional[Dict[str, str]] = None,
    sort_ascending: bool = True,
):
    plot_df = df[[label_col, value_col] + ([lower_col, upper_col] if lower_col and upper_col else [])].copy()
    plot_df = plot_df.sort_values(value_col, ascending=sort_ascending).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    y = np.arange(len(plot_df))

    colors = None
    if color_map is not None:
        colors = [color_map.get(lbl, "#4C72B0") for lbl in plot_df[label_col]]

    ax.scatter(plot_df[value_col], y, s=60, c=colors if colors is not None else None)

    if lower_col and upper_col and lower_col in plot_df.columns and upper_col in plot_df.columns:
        for i, row in plot_df.iterrows():
            lo = row[lower_col]
            hi = row[upper_col]
            if pd.notna(lo) and pd.notna(hi):
                ax.plot([lo, hi], [i, i], linewidth=2, color=colors[i] if colors is not None else "black")

    ax.set_yticks(y)
    ax.set_yticklabels(plot_df[label_col])
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


# ---------------------------------------------------------
# Loading and validation
# ---------------------------------------------------------

def load_step4_inputs(cfg: Step5Config) -> Dict[str, Any]:
    seq_path = Path(cfg.input_model_ready_csv).resolve()
    feat_path = Path(cfg.input_baseline_features_csv).resolve()

    if not seq_path.exists():
        raise FileNotFoundError(f"Missing model-ready CSV: {seq_path}")
    if not feat_path.exists():
        raise FileNotFoundError(f"Missing baseline-features CSV: {feat_path}")

    seq_df = pd.read_csv(seq_path)
    feat_df = pd.read_csv(feat_path)

    contract = None
    condition_mapping = None

    if cfg.input_preprocessing_contract_json:
        contract_path = Path(cfg.input_preprocessing_contract_json).resolve()
        if not contract_path.exists():
            raise FileNotFoundError(f"Missing preprocessing contract JSON: {contract_path}")
        with open(contract_path, "r", encoding="utf-8") as f:
            contract = json.load(f)

    if cfg.input_condition_id_mapping_json:
        mapping_path = Path(cfg.input_condition_id_mapping_json).resolve()
        if not mapping_path.exists():
            raise FileNotFoundError(f"Missing condition ID mapping JSON: {mapping_path}")
        with open(mapping_path, "r", encoding="utf-8") as f:
            condition_mapping = json.load(f)

    return {
        "seq_df": seq_df,
        "feat_df": feat_df,
        "contract": contract,
        "condition_mapping": condition_mapping,
        "seq_path": seq_path,
        "feat_path": feat_path,
    }


def validate_and_merge_inputs(seq_df: pd.DataFrame, feat_df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    required_seq_cols = [cfg.merge_key, cfg.split_col, cfg.target_col, cfg.sequence_col]
    for col in required_seq_cols:
        if col not in seq_df.columns:
            raise ValueError(f"Required column '{col}' not found in sequence table.")

    if cfg.merge_key not in feat_df.columns:
        raise ValueError(f"Required merge key '{cfg.merge_key}' not found in baseline feature table.")

    if seq_df[cfg.merge_key].isna().any():
        raise ValueError(f"Merge key '{cfg.merge_key}' contains missing values in sequence table.")
    if feat_df[cfg.merge_key].isna().any():
        raise ValueError(f"Merge key '{cfg.merge_key}' contains missing values in feature table.")

    if not seq_df[cfg.merge_key].is_unique:
        raise ValueError(f"Sequence table merge key '{cfg.merge_key}' is not unique.")
    if not feat_df[cfg.merge_key].is_unique:
        raise ValueError(f"Feature table merge key '{cfg.merge_key}' is not unique.")

    feat_extra_cols = [cfg.merge_key] + [c for c in feat_df.columns if c not in seq_df.columns and c != cfg.merge_key]
    merged = seq_df.merge(feat_df[feat_extra_cols], on=cfg.merge_key, how="inner", validate="one_to_one")

    if len(merged) != len(seq_df) or len(merged) != len(feat_df):
        raise ValueError("Merged row count mismatch across Step 4 inputs.")

    expected_splits = {cfg.train_label, cfg.val_label, cfg.test_label}
    found_splits = set(merged[cfg.split_col].dropna().unique().tolist())
    if not expected_splits.issubset(found_splits):
        raise ValueError(f"Expected splits {expected_splits}, found {found_splits}")

    if merged[cfg.target_col].isna().any():
        n_missing = int(merged[cfg.target_col].isna().sum())
        raise ValueError(f"Target column '{cfg.target_col}' contains missing values: {n_missing}")

    return merged


def build_schema_report(df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    rows = []
    rows.append({
        "n_rows": len(df),
        "n_columns": df.shape[1],
        "n_classes": df[cfg.target_col].nunique(),
        "train_n": int((df[cfg.split_col] == cfg.train_label).sum()),
        "val_n": int((df[cfg.split_col] == cfg.val_label).sum()),
        "test_n": int((df[cfg.split_col] == cfg.test_label).sum()),
    })
    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Feature blocks
# ---------------------------------------------------------

def infer_feature_blocks(df: pd.DataFrame) -> Dict[str, List[str]]:
    aa_frac_raw = sorted([c for c in df.columns if c.startswith("aa_frac_") and not c.endswith("_z")])
    aa_frac_z = sorted([c for c in df.columns if c.startswith("aa_frac_") and c.endswith("_z")])

    global_raw_candidates = ["length", "net_charge_pH7", "hydrophobicity_KD", "entropy", "dominant_aa_frac"]
    global_z_candidates = ["length_z", "net_charge_pH7_z", "hydrophobicity_KD_z", "entropy_z", "dominant_aa_frac_z"]

    global_raw = [c for c in global_raw_candidates if c in df.columns]
    global_z = [c for c in global_z_candidates if c in df.columns]

    condition_core_raw = [c for c in ["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"] if c in df.columns]
    condition_core_z = [c for c in ["length_z", "net_charge_pH7_z", "hydrophobicity_KD_z", "entropy_z"] if c in df.columns]

    minimal_condition_proxy = [c for c in ["length", "net_charge_pH7", "hydrophobicity_KD"] if c in df.columns]
    if len(minimal_condition_proxy) == 0:
        minimal_condition_proxy = global_raw[:3]

    feature_blocks = {
        "global_raw": global_raw,
        "aa_frac_raw": aa_frac_raw,
        "global_z": global_z,
        "aa_frac_z": aa_frac_z,
        "combined_raw": global_raw + aa_frac_raw,
        "combined_z": global_z + aa_frac_z,
        "combined_all": global_raw + aa_frac_raw + global_z + aa_frac_z,
        "condition_core_raw": condition_core_raw,
        "condition_core_z": condition_core_z,
        "minimal_condition_proxy": minimal_condition_proxy,
    }

    for name, cols in feature_blocks.items():
        if len(cols) == 0:
            warnings.warn(f"Feature block '{name}' is empty.")
    return feature_blocks


def feature_block_manifest(feature_blocks: Dict[str, List[str]]) -> pd.DataFrame:
    rows = []
    for block, cols in feature_blocks.items():
        rows.append({
            "feature_block": block,
            "n_features": len(cols),
            "features_json": json.dumps(cols),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Split preparation
# ---------------------------------------------------------

def prepare_splits(df: pd.DataFrame, feature_cols: List[str], cfg: Step5Config) -> Dict[str, Any]:
    use_cols = [cfg.merge_key, cfg.sequence_col, cfg.split_col, cfg.target_col] + feature_cols
    use_cols = [c for c in use_cols if c in df.columns]
    sub = df[use_cols].copy()

    train_df = sub[sub[cfg.split_col] == cfg.train_label].reset_index(drop=True)
    val_df = sub[sub[cfg.split_col] == cfg.val_label].reset_index(drop=True)
    test_df = sub[sub[cfg.split_col] == cfg.test_label].reset_index(drop=True)

    X_train = train_df[feature_cols].copy()
    y_train = train_df[cfg.target_col].astype(str).to_numpy()

    X_val = val_df[feature_cols].copy()
    y_val = val_df[cfg.target_col].astype(str).to_numpy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[cfg.target_col].astype(str).to_numpy()

    classes = np.array(sorted(pd.Series(y_train).unique().tolist()))
    if not set(pd.Series(y_val).unique()).issubset(set(classes)):
        raise ValueError("Validation set contains classes absent from train set.")
    if not set(pd.Series(y_test).unique()).issubset(set(classes)):
        raise ValueError("Test set contains classes absent from train set.")

    return {
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "classes": classes,
    }


# ---------------------------------------------------------
# Models
# ---------------------------------------------------------

def build_model_registry(seed: int) -> Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]]:
    registry: Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]] = {}

    registry["dummy_majority"] = [
        (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]),
            {},
        )
    ]

    lr_grid = []
    for C in [0.1, 1.0, 3.0, 10.0]:
        lr_grid.append((
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    C=C,
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=3000,
                    random_state=seed,
                )),
            ]),
            {"C": C},
        ))
    registry["logistic_regression"] = lr_grid

    rf_grid = []
    for n_estimators in [300]:
        for max_depth in [None, 20]:
            for min_samples_leaf in [1, 3]:
                rf_grid.append((
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("model", RandomForestClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            class_weight="balanced_subsample",
                            n_jobs=-1,
                            random_state=seed,
                        )),
                    ]),
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                    },
                ))
    registry["random_forest"] = rf_grid

    et_grid = []
    for n_estimators in [300]:
        for max_depth in [None, 20]:
            for min_samples_leaf in [1, 3]:
                et_grid.append((
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("model", ExtraTreesClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            class_weight="balanced",
                            n_jobs=-1,
                            random_state=seed,
                        )),
                    ]),
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                    },
                ))
    registry["extra_trees"] = et_grid
    return registry


# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

def topk_accuracy(y_true: np.ndarray, proba: np.ndarray, classes: np.ndarray, k: int) -> float:
    class_to_idx = {c: i for i, c in enumerate(classes)}
    true_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    topk_idx = np.argsort(-proba, axis=1)[:, :k]
    return float(np.mean(np.any(topk_idx == true_idx[:, None], axis=1)))


def safe_macro_auroc(y_true: np.ndarray, proba: np.ndarray, classes: np.ndarray) -> float:
    try:
        y_true_bin = label_binarize(y_true, classes=classes)
        if y_true_bin.shape[1] <= 1:
            return float("nan")
        return float(roc_auc_score(y_true_bin, proba, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: np.ndarray,
    classes: np.ndarray,
    split_name: str,
    model_name: str,
    feature_block: str,
) -> Dict[str, Any]:
    metrics = {
        "split": split_name,
        "model_name": model_name,
        "feature_block": feature_block,
        "n_samples": int(len(y_true)),
        "n_classes": int(len(classes)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_auroc_ovr": safe_macro_auroc(y_true, proba, classes),
        "log_loss": float(log_loss(y_true, proba, labels=classes)) if proba is not None else float("nan"),
    }

    for k in [1, 3]:
        if proba is not None and k <= len(classes):
            metrics[f"top{k}_accuracy"] = topk_accuracy(y_true, proba, classes, k)
        else:
            metrics[f"top{k}_accuracy"] = float("nan")
    return metrics


def per_class_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray,
    model_name: str,
    feature_block: str,
) -> pd.DataFrame:
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    return pd.DataFrame({
        "model_name": model_name,
        "feature_block": feature_block,
        "class_label": classes,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "support": support.astype(int),
    })


# ---------------------------------------------------------
# Training and selection
# ---------------------------------------------------------

def fit_select_model(
    splits: Dict[str, Any],
    model_registry: Dict[str, List[Tuple[Pipeline, Dict[str, Any]]]],
    feature_block_name: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    rows = []
    X_train = splits["X_train"]
    y_train = splits["y_train"]
    X_val = splits["X_val"]
    y_val = splits["y_val"]
    classes = splits["classes"]

    best_record = None
    best_score = -np.inf

    for model_name, candidates in model_registry.items():
        for pipeline, param_dict in candidates:
            model = clone(pipeline)
            model.fit(X_train, y_train)

            y_val_pred = model.predict(X_val)
            val_proba = model.predict_proba(X_val)

            metrics = evaluate_predictions(
                y_true=y_val,
                y_pred=y_val_pred,
                proba=val_proba,
                classes=classes,
                split_name="val",
                model_name=model_name,
                feature_block=feature_block_name,
            )
            metrics.update(param_dict)
            rows.append(metrics)

            score = metrics["macro_f1"]
            if score > best_score:
                best_score = score
                best_record = {
                    "model_name": model_name,
                    "params": param_dict,
                    "pipeline": model,
                    "val_metrics": metrics,
                }

    val_results_df = pd.DataFrame(rows).sort_values(
        by=["macro_f1", "balanced_accuracy", "top3_accuracy"],
        ascending=False
    ).reset_index(drop=True)

    if best_record is None:
        raise RuntimeError("No model selected on validation data.")
    return val_results_df, best_record


def refit_best_model_on_train_val(best_record: Dict[str, Any], splits: Dict[str, Any]) -> Pipeline:
    model = clone(best_record["pipeline"])
    X_train_val = pd.concat([splits["X_train"], splits["X_val"]], axis=0).reset_index(drop=True)
    y_train_val = np.concatenate([splits["y_train"], splits["y_val"]])
    model.fit(X_train_val, y_train_val)
    return model


# ---------------------------------------------------------
# Bootstrap
# ---------------------------------------------------------

def bootstrap_test_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: np.ndarray,
    classes: np.ndarray,
    n_iter: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []

    for i in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        pp = proba[idx]
        rows.append({
            "bootstrap_iter": i,
            "accuracy": accuracy_score(yt, yp),
            "balanced_accuracy": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "weighted_f1": f1_score(yt, yp, average="weighted"),
            "macro_auroc_ovr": safe_macro_auroc(yt, pp, classes),
            "top1_accuracy": topk_accuracy(yt, pp, classes, 1),
            "top3_accuracy": topk_accuracy(yt, pp, classes, 3) if len(classes) >= 3 else np.nan,
        })
    return pd.DataFrame(rows)


def summarize_bootstrap(df_boot: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in [c for c in df_boot.columns if c != "bootstrap_iter"]:
        vals = pd.Series(df_boot[col]).dropna()
        rows.append({
            "metric": col,
            "mean": float(vals.mean()),
            "median": float(vals.median()),
            "ci_lower_2.5": float(vals.quantile(0.025)),
            "ci_upper_97.5": float(vals.quantile(0.975)),
        })
    return pd.DataFrame(rows)


def attach_ci_to_metrics(metric_df: pd.DataFrame, bootstrap_summary_df: pd.DataFrame, metric_name: str = "macro_f1", prefix: str = "test_") -> pd.DataFrame:
    df = metric_df.copy()
    bs = bootstrap_summary_df.copy()
    row = bs[bs["metric"] == metric_name]
    if len(row) == 1:
        df[f"{prefix}{metric_name}_ci_lower"] = float(row["ci_lower_2.5"].iloc[0])
        df[f"{prefix}{metric_name}_ci_upper"] = float(row["ci_upper_97.5"].iloc[0])
    else:
        df[f"{prefix}{metric_name}_ci_lower"] = np.nan
        df[f"{prefix}{metric_name}_ci_upper"] = np.nan
    return df


# ---------------------------------------------------------
# Interpretability
# ---------------------------------------------------------

def compute_permutation_importance_table(
    fitted_model: Pipeline,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    feature_cols: List[str],
    n_repeats: int,
    seed: int,
    scoring: str = "f1_macro",
) -> pd.DataFrame:
    result = permutation_importance(
        fitted_model,
        X_test,
        y_test,
        n_repeats=n_repeats,
        random_state=seed,
        n_jobs=-1,
        scoring=scoring,
    )
    out = pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return out


# ---------------------------------------------------------
# Audit helpers
# ---------------------------------------------------------

def compute_pca_projection(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    max_components: int = 2,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(X_train)
    X_train_scaled = scaler.fit_transform(X_train_imp)

    X_test_imp = imputer.transform(X_test)
    X_test_scaled = scaler.transform(X_test_imp)

    pca = PCA(n_components=max_components, random_state=GLOBAL_SEED)
    X_test_pca = pca.fit(X_train_scaled).transform(X_test_scaled)

    proj_df = pd.DataFrame(X_test_pca, columns=[f"PC{i+1}" for i in range(max_components)])
    proj_df["class_label"] = y_test

    meta = {
        "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
        "explained_variance_total_2d": float(np.sum(pca.explained_variance_ratio_)),
    }
    return proj_df, meta


def class_counts_by_split(df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    ct = (
        df.groupby([cfg.target_col, cfg.split_col])
        .size()
        .reset_index(name="count")
        .pivot(index=cfg.target_col, columns=cfg.split_col, values="count")
        .fillna(0)
        .reset_index()
    )
    for split_name in [cfg.train_label, cfg.val_label, cfg.test_label]:
        if split_name not in ct.columns:
            ct[split_name] = 0
    return ct


def descriptor_distribution_by_split(df: pd.DataFrame, cfg: Step5Config, descriptor_cols: List[str]) -> pd.DataFrame:
    rows = []
    for col in descriptor_cols:
        if col not in df.columns:
            continue
        for split_name in [cfg.train_label, cfg.val_label, cfg.test_label]:
            vals = df.loc[df[cfg.split_col] == split_name, col].dropna()
            rows.append({
                "descriptor": col,
                "split": split_name,
                "mean": float(vals.mean()) if len(vals) else np.nan,
                "std": float(vals.std()) if len(vals) else np.nan,
                "median": float(vals.median()) if len(vals) else np.nan,
                "min": float(vals.min()) if len(vals) else np.nan,
                "max": float(vals.max()) if len(vals) else np.nan,
            })
    return pd.DataFrame(rows)


def classify_audit_outcome(
    best_full_metrics: Dict[str, Any],
    reconstructability_df: pd.DataFrame,
    pca_meta: Dict[str, Any],
    cfg: Step5Config,
) -> Dict[str, Any]:
    full_f1 = float(best_full_metrics["macro_f1"])

    recon_df = reconstructability_df.copy()
    recon_df = recon_df.sort_values("macro_f1", ascending=False)

    best_non_full = recon_df[recon_df["feature_block"] != "combined_all"]
    best_non_full_f1 = float(best_non_full["macro_f1"].max()) if len(best_non_full) else np.nan

    best_condition_proxy = recon_df[recon_df["feature_block"].isin(["minimal_condition_proxy", "condition_core_raw", "condition_core_z"])]
    best_condition_proxy_f1 = float(best_condition_proxy["macro_f1"].max()) if len(best_condition_proxy) else np.nan

    pca_var = float(pca_meta.get("explained_variance_total_2d", np.nan))

    # Rules
    if pd.notna(best_condition_proxy_f1) and best_condition_proxy_f1 >= cfg.descriptor_trivial_threshold:
        if abs(full_f1 - best_condition_proxy_f1) <= cfg.reconstructability_gap_threshold:
            category = "C"
            conclusion = "Target appears largely reconstructable from condition-linked descriptors."
        else:
            category = "B"
            conclusion = "Benchmark is valid but descriptor-trivial: shallow descriptors recover most of the target."
    elif full_f1 >= cfg.descriptor_trivial_threshold and pca_var >= cfg.pca_variance_threshold:
        category = "B"
        conclusion = "Benchmark is valid but likely structurally easy in low-dimensional descriptor space."
    elif full_f1 >= 0.80:
        category = "A"
        conclusion = "Benchmark appears valid and non-trivial under current audits."
    else:
        category = "D"
        conclusion = "Benchmark remains unresolved or weak; interpretation should be cautious."

    return {
        "audit_category": category,
        "best_full_macro_f1": full_f1,
        "best_non_full_macro_f1": best_non_full_f1,
        "best_condition_proxy_macro_f1": best_condition_proxy_f1,
        "pca_explained_variance_total_2d": pca_var,
        "conclusion": conclusion,
    }


def manuscript_recommendation_from_audit(audit_category: str) -> Dict[str, str]:
    if audit_category == "A":
        return {
            "main_figure_5_1": "keep_in_main",
            "main_figure_5_2": "keep_in_main",
            "step5_role": "strong_baseline_anchor",
        }
    if audit_category == "B":
        return {
            "main_figure_5_1": "keep_in_main_with_conservative_framing",
            "main_figure_5_2": "keep_in_main_or_supplementary",
            "step5_role": "descriptor_trivial_but_valid_baseline",
        }
    if audit_category == "C":
        return {
            "main_figure_5_1": "downgrade_or_move_to_supplementary",
            "main_figure_5_2": "keep_if_it_explains_reconstructability",
            "step5_role": "sanity_control_or_target_reconstruction_result",
        }
    return {
        "main_figure_5_1": "do_not_lock",
        "main_figure_5_2": "do_not_lock",
        "step5_role": "revise_before_manuscript_use",
    }


# ---------------------------------------------------------
# Figure functions
# ---------------------------------------------------------

def save_workflow_schematic(out_base: Path, train_n: int, val_n: int, test_n: int):
    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.axis("off")

    boxes = [
        (0.02, 0.25, 0.22, 0.5, "Frozen Step 4 inputs\nmodel-ready sequences\nbaseline features"),
        (0.29, 0.25, 0.18, 0.5, f"Fixed split reuse\ntrain={train_n:,}\nval={val_n:,}\ntest={test_n:,}"),
        (0.52, 0.25, 0.18, 0.5, "Model selection\nvalidation macro F1"),
        (0.75, 0.25, 0.22, 0.5, "Final test evaluation\nbenchmark + audit"),
    ]

    for i, (x, y, w, h, text) in enumerate(boxes):
        rect = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.8)
        ax.add_patch(rect)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)
        if i < len(boxes) - 1:
            ax.annotate("", xy=(boxes[i + 1][0] - 0.01, y + h / 2), xytext=(x + w + 0.01, y + h / 2),
                        arrowprops=dict(arrowstyle="->", lw=1.8))
    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


def save_main_figure_5_1(
    train_n: int,
    val_n: int,
    test_n: int,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    out_base: Path,
):
    fig = plt.figure(figsize=(14, 4.5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.35, 1.0, 1.0])

    # panel a
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.axis("off")
    boxes = [
        (0.02, 0.25, 0.24, 0.5, "Frozen Step 4\ninputs"),
        (0.31, 0.25, 0.18, 0.5, f"Split reuse\n{train_n:,}/{val_n:,}/{test_n:,}"),
        (0.54, 0.25, 0.18, 0.5, "Validation\nselection"),
        (0.77, 0.25, 0.18, 0.5, "Untouched\ntest"),
    ]
    for i, (x, y, w, h, text) in enumerate(boxes):
        ax0.add_patch(plt.Rectangle((x, y), w, h, fill=False, linewidth=1.6))
        ax0.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)
        if i < len(boxes) - 1:
            ax0.annotate("", xy=(boxes[i + 1][0] - 0.01, y + h / 2), xytext=(x + w + 0.01, y + h / 2),
                         arrowprops=dict(arrowstyle="->", lw=1.6))
    ax0.text(-0.05, 1.02, "a", transform=ax0.transAxes, fontsize=13, fontweight="bold", va="top")

    # panel b
    ax1 = fig.add_subplot(gs[0, 1])
    vdf = val_df.sort_values("macro_f1", ascending=True).reset_index(drop=True)
    y = np.arange(len(vdf))
    colors = [COLOR_MODELS.get(x, "#4C72B0") for x in vdf["model_name"]]
    ax1.scatter(vdf["macro_f1"], y, s=60, c=colors)
    ax1.set_yticks(y)
    ax1.set_yticklabels(vdf["model_name"])
    ax1.set_xlabel("Validation macro F1")
    ax1.set_title("Model selection")
    ax1.text(-0.18, 1.02, "b", transform=ax1.transAxes, fontsize=13, fontweight="bold", va="top")

    # panel c
    ax2 = fig.add_subplot(gs[0, 2])
    tdf = test_df.sort_values("test_macro_f1", ascending=True).reset_index(drop=True)
    y = np.arange(len(tdf))
    colors = [COLOR_MODELS.get(x, "#4C72B0") for x in tdf["model_name"]]
    ax2.scatter(tdf["test_macro_f1"], y, s=60, c=colors)
    if "test_macro_f1_ci_lower" in tdf.columns and "test_macro_f1_ci_upper" in tdf.columns:
        for i, row in tdf.iterrows():
            lo = row["test_macro_f1_ci_lower"]
            hi = row["test_macro_f1_ci_upper"]
            if pd.notna(lo) and pd.notna(hi):
                ax2.plot([lo, hi], [i, i], linewidth=2, color=colors[i])
    ax2.set_yticks(y)
    ax2.set_yticklabels(tdf["model_name"])
    ax2.set_xlabel("Test macro F1")
    ax2.set_title("Untouched test performance")
    ax2.text(-0.18, 1.02, "c", transform=ax2.transAxes, fontsize=13, fontweight="bold", va="top")

    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


def save_main_figure_5_2(ablation_df: pd.DataFrame, out_base: Path):
    fig = plt.figure(figsize=(12, 4.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.0])

    # panel a: canonical feature ablation
    ax0 = fig.add_subplot(gs[0, 0])
    canonical_blocks = ["global_raw", "global_z", "aa_frac_raw", "aa_frac_z", "combined_raw", "combined_z", "combined_all"]
    df0 = ablation_df[ablation_df["feature_block"].isin(canonical_blocks)].copy()
    df0 = df0.sort_values("macro_f1", ascending=True).reset_index(drop=True)
    y = np.arange(len(df0))
    colors = [COLOR_BLOCKS.get(x, "#4C72B0") for x in df0["feature_block"]]
    ax0.scatter(df0["macro_f1"], y, s=60, c=colors)
    ax0.set_yticks(y)
    ax0.set_yticklabels(df0["feature_block"])
    ax0.set_xlabel("Test macro F1")
    ax0.set_title("Feature-family ablation")
    ax0.text(-0.12, 1.02, "a", transform=ax0.transAxes, fontsize=13, fontweight="bold", va="top")

    # panel b: audit-focused blocks
    ax1 = fig.add_subplot(gs[0, 1])
    audit_blocks = ["minimal_condition_proxy", "condition_core_raw", "condition_core_z", "combined_all"]
    df1 = ablation_df[ablation_df["feature_block"].isin(audit_blocks)].copy()
    df1 = df1.sort_values("macro_f1", ascending=True).reset_index(drop=True)
    y = np.arange(len(df1))
    colors = [COLOR_BLOCKS.get(x, "#4C72B0") for x in df1["feature_block"]]
    ax1.scatter(df1["macro_f1"], y, s=60, c=colors)
    ax1.set_yticks(y)
    ax1.set_yticklabels(df1["feature_block"])
    ax1.set_xlabel("Test macro F1")
    ax1.set_title("Condition-linked audit blocks")
    ax1.text(-0.12, 1.02, "b", transform=ax1.transAxes, fontsize=13, fontweight="bold", va="top")

    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


def save_reconstructability_audit_figure(df: pd.DataFrame, out_base: Path):
    plot_df = df.sort_values("macro_f1", ascending=True).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    y = np.arange(len(plot_df))
    colors = [COLOR_BLOCKS.get(x, "#4C72B0") for x in plot_df["feature_block"]]
    ax.scatter(plot_df["macro_f1"], y, s=70, c=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["feature_block"])
    ax.set_xlabel("Test macro F1")
    ax.set_title("Target reconstructability audit")
    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


def save_pca_separability_figure(proj_df: pd.DataFrame, pca_meta: Dict[str, Any], out_base: Path):
    fig, ax = plt.subplots(figsize=(7, 6))
    classes = sorted(proj_df["class_label"].unique().tolist())

    # For many classes, cycle default colors; readable enough for audit
    for cls in classes:
        sub = proj_df[proj_df["class_label"] == cls]
        ax.scatter(sub["PC1"], sub["PC2"], s=12, alpha=0.6, label=str(cls))

    ax.set_xlabel(f"PC1 ({100 * pca_meta['explained_variance_ratio'][0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({100 * pca_meta['explained_variance_ratio'][1]:.1f}%)")
    ax.set_title("Descriptor-space separability audit (PCA)")
    # legend omitted for many classes to keep plot readable
    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


def save_split_integrity_figure(class_count_df: pd.DataFrame, dist_df: pd.DataFrame, out_base: Path):
    fig = plt.figure(figsize=(12, 4.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.0])

    # panel a: class counts by split
    ax0 = fig.add_subplot(gs[0, 0])
    x = np.arange(len(class_count_df))
    width = 0.25
    for i, split in enumerate(["train", "val", "test"]):
        vals = class_count_df[split].to_numpy() if split in class_count_df.columns else np.zeros(len(class_count_df))
        ax0.bar(x + (i - 1) * width, vals, width=width, label=split)
    ax0.set_xticks(x)
    ax0.set_xticklabels(class_count_df["primary_condition_id"], rotation=90)
    ax0.set_ylabel("Count")
    ax0.set_title("Class counts by split")
    ax0.legend()

    # panel b: descriptor means by split for selected descriptors
    ax1 = fig.add_subplot(gs[0, 1])
    chosen = dist_df[dist_df["descriptor"].isin(["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"])]
    if len(chosen):
        pivot = chosen.pivot(index="descriptor", columns="split", values="mean")
        pivot = pivot.reindex(columns=["train", "val", "test"])
        im = ax1.imshow(pivot.values, aspect="auto")
        ax1.set_yticks(np.arange(len(pivot.index)))
        ax1.set_yticklabels(pivot.index)
        ax1.set_xticks(np.arange(len(pivot.columns)))
        ax1.set_xticklabels(pivot.columns)
        ax1.set_title("Descriptor means by split")
        fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    else:
        ax1.axis("off")
        ax1.text(0.5, 0.5, "No descriptor summary available", ha="center", va="center")

    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


def save_confusion_matrix_figure(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray,
    out_base: Path,
    normalize: str = "true",
) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=classes, normalize=normalize)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, aspect="auto")
    fig.colorbar(im, ax=ax)
    ax.set_xticks(np.arange(len(classes)))
    ax.set_xticklabels(classes, rotation=90)
    ax.set_yticks(np.arange(len(classes)))
    ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Normalized confusion matrix")
    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)
    return cm_df


def save_permutation_importance_figure(df_imp: pd.DataFrame, out_base: Path, top_n: int = 20):
    plot_df = df_imp.head(top_n).copy().sort_values("importance_mean", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(plot_df["feature"], plot_df["importance_mean"])
    ax.set_xlabel("Permutation importance (macro F1 decrease)")
    ax.set_ylabel("Feature")
    ax.set_title(f"Top {top_n} permutation importances")
    fig.tight_layout()
    _savefig_both(fig, out_base)
    plt.close(fig)


# ---------------------------------------------------------
# Reporting
# ---------------------------------------------------------

def write_text_report(lines: List[str], out_path: Path):
    out_path.write_text("\n".join(lines), encoding="utf-8")


def build_figure_manifest(audit_category: str) -> pd.DataFrame:
    rows = [
        {
            "figure_id": "Fig5_1",
            "filename_base": "step5_main_baseline_benchmark",
            "placement_candidate": "main",
            "scientific_purpose": "Establish the classical baseline ceiling under frozen Step 4 splits.",
            "status_rule": f"keep in main if audit category in ['A','B']; current category={audit_category}",
            "what_it_proves": "Strength of classical baselines on the Step 5 task.",
            "what_it_does_not_prove": "Biological mechanism or necessity of deep learning.",
        },
        {
            "figure_id": "Fig5_2",
            "filename_base": "step5_main_feature_ablation",
            "placement_candidate": "main_or_supplementary",
            "scientific_purpose": "Show which descriptor families drive performance and whether condition-linked blocks dominate.",
            "status_rule": f"main if explanatory; current category={audit_category}",
            "what_it_proves": "Relative value of feature blocks.",
            "what_it_does_not_prove": "Generalization beyond the current benchmark task.",
        },
        {
            "figure_id": "SuppFigS5_1",
            "filename_base": "step5_audit_reconstructability",
            "placement_candidate": "supplementary",
            "scientific_purpose": "Test whether target is reconstructable from condition-linked descriptors.",
            "status_rule": "mandatory supplementary audit figure",
            "what_it_proves": "Whether perfect performance may be descriptor-trivial or target-linked.",
            "what_it_does_not_prove": "Formal leakage by itself.",
        },
        {
            "figure_id": "SuppFigS5_2",
            "filename_base": "step5_audit_pca_separability",
            "placement_candidate": "supplementary",
            "scientific_purpose": "Assess low-dimensional separability in descriptor space.",
            "status_rule": "mandatory supplementary audit figure",
            "what_it_proves": "Whether shallow descriptor space is already highly separable.",
            "what_it_does_not_prove": "Causality or leakage.",
        },
        {
            "figure_id": "SuppFigS5_3",
            "filename_base": "step5_audit_split_integrity",
            "placement_candidate": "supplementary",
            "scientific_purpose": "Show class and descriptor stability across splits.",
            "status_rule": "mandatory supplementary audit figure",
            "what_it_proves": "Split composition and class completeness.",
            "what_it_does_not_prove": "Model validity on its own.",
        },
        {
            "figure_id": "SuppFigS5_4",
            "filename_base": "step5_supp_confusion_matrix",
            "placement_candidate": "supplementary",
            "scientific_purpose": "Show per-class error structure for the best model.",
            "status_rule": "supplementary diagnostic",
            "what_it_proves": "Error distribution across classes.",
            "what_it_does_not_prove": "Mechanistic interpretation.",
        },
        {
            "figure_id": "SuppFigS5_5",
            "filename_base": "step5_supp_permutation_importance",
            "placement_candidate": "supplementary",
            "scientific_purpose": "Rank influential engineered descriptors in the best model.",
            "status_rule": "supplementary diagnostic",
            "what_it_proves": "Predictive importance ranking.",
            "what_it_does_not_prove": "Biological mechanism.",
        },
    ]
    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------

def run_step5_pipeline(cfg: Step5Config) -> Dict[str, Any]:
    seed = cfg.random_seed
    n_boot = 100 if cfg.fast_mode else cfg.bootstrap_iterations
    n_perm = 5 if cfg.fast_mode else cfg.permutation_repeats

    project_root = _project_root_from_input(cfg.input_model_ready_csv)
    out_dir = project_root / cfg.out_dir_name
    dirs = _mkdirs(out_dir)

    _print_header("Loading Step 4 inputs")
    loaded = load_step4_inputs(cfg)
    seq_df = loaded["seq_df"]
    feat_df = loaded["feat_df"]

    _print_header("Validating and merging frozen Step 4 tables")
    merged_df = validate_and_merge_inputs(seq_df, feat_df, cfg)
    print(f"Merged dataset shape: {merged_df.shape}")

    schema_df = build_schema_report(merged_df, cfg)
    schema_df.to_csv(dirs["tables"] / "step5_schema_summary.csv", index=False)

    _print_header("Building feature blocks")
    feature_blocks = infer_feature_blocks(merged_df)
    feature_manifest_df = feature_block_manifest(feature_blocks)
    feature_manifest_df.to_csv(dirs["tables"] / "step5_feature_block_manifest.csv", index=False)
    _safe_json_dump({k: v for k, v in feature_blocks.items()}, dirs["artifacts"] / "step5_feature_block_manifest.json")

    _print_header("Training and selecting baseline models on combined baseline features")
    combined_block_name = "combined_all"
    combined_features = feature_blocks[combined_block_name]
    if len(combined_features) == 0:
        raise ValueError("Combined feature block is empty.")

    model_registry = build_model_registry(seed)
    combined_splits = prepare_splits(merged_df, combined_features, cfg)

    val_results_df, best_full_record = fit_select_model(
        combined_splits,
        model_registry,
        feature_block_name=combined_block_name,
    )
    val_results_df.to_csv(dirs["tables"] / "step5_validation_model_comparison.csv", index=False)

    best_full_model_name = best_full_record["model_name"]
    best_full_val_metrics = best_full_record["val_metrics"]
    print(f"Selected best full-feature model: {best_full_model_name}")
    print(f"Validation macro F1: {best_full_val_metrics['macro_f1']:.4f}")

    _print_header("Refitting best full-feature model on train+val and evaluating on untouched test set")
    best_full_model = refit_best_model_on_train_val(best_full_record, combined_splits)

    X_test = combined_splits["X_test"]
    y_test = combined_splits["y_test"]
    classes = combined_splits["classes"]

    y_test_pred = best_full_model.predict(X_test)
    y_test_proba = best_full_model.predict_proba(X_test)

    best_test_metrics = evaluate_predictions(
        y_true=y_test,
        y_pred=y_test_pred,
        proba=y_test_proba,
        classes=classes,
        split_name="test",
        model_name=best_full_model_name,
        feature_block=combined_block_name,
    )
    pd.DataFrame([best_test_metrics]).to_csv(dirs["tables"] / "step5_best_model_test_metrics.csv", index=False)

    per_class_df = per_class_metrics(
        y_true=y_test,
        y_pred=y_test_pred,
        classes=classes,
        model_name=best_full_model_name,
        feature_block=combined_block_name,
    ).sort_values("f1", ascending=False)
    per_class_df.to_csv(dirs["tables"] / "step5_per_class_metrics.csv", index=False)

    _print_header("Creating main baseline comparison table")
    family_best_rows = []
    X_train_val = pd.concat([combined_splits["X_train"], combined_splits["X_val"]], axis=0).reset_index(drop=True)
    y_train_val = np.concatenate([combined_splits["y_train"], combined_splits["y_val"]])

    for model_name in val_results_df["model_name"].drop_duplicates().tolist():
        family_val = val_results_df[val_results_df["model_name"] == model_name].sort_values(
            by=["macro_f1", "balanced_accuracy", "top3_accuracy"], ascending=False
        )
        best_row = family_val.iloc[0].to_dict()

        candidate = None
        for pipe, params in model_registry[model_name]:
            if all(best_row.get(k) == v for k, v in params.items()):
                candidate = clone(pipe)
                break
        if candidate is None:
            candidate = clone(model_registry[model_name][0][0])

        candidate.fit(X_train_val, y_train_val)
        pred = candidate.predict(X_test)
        proba = candidate.predict_proba(X_test)

        test_metrics = evaluate_predictions(
            y_true=y_test,
            y_pred=pred,
            proba=proba,
            classes=classes,
            split_name="test",
            model_name=model_name,
            feature_block=combined_block_name,
        )

        row = {
            "model_name": model_name,
            "feature_block": combined_block_name,
            "val_macro_f1": best_row["macro_f1"],
            "val_balanced_accuracy": best_row["balanced_accuracy"],
            "val_top1_accuracy": best_row["top1_accuracy"],
            "val_top3_accuracy": best_row["top3_accuracy"],
            "test_accuracy": test_metrics["accuracy"],
            "test_balanced_accuracy": test_metrics["balanced_accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_macro_auroc_ovr": test_metrics["macro_auroc_ovr"],
            "test_log_loss": test_metrics["log_loss"],
            "test_top1_accuracy": test_metrics["top1_accuracy"],
            "test_top3_accuracy": test_metrics["top3_accuracy"],
        }
        for k in ["C", "n_estimators", "max_depth", "min_samples_leaf"]:
            if k in best_row:
                row[k] = best_row[k]
        family_best_rows.append(row)

    model_comparison_df = pd.DataFrame(family_best_rows).sort_values(
        by=["test_macro_f1", "test_balanced_accuracy", "test_top3_accuracy"],
        ascending=False
    ).reset_index(drop=True)
    model_comparison_df.to_csv(dirs["tables"] / "step5_model_comparison_test.csv", index=False)

    _print_header("Running feature-family ablation using best model family")
    best_family_name = best_full_model_name
    ablation_rows = []

    for feature_block_name, feature_cols in feature_blocks.items():
        if len(feature_cols) == 0:
            continue
        splits = prepare_splits(merged_df, feature_cols, cfg)
        family_candidates = {best_family_name: model_registry[best_family_name]}

        _, family_best_record = fit_select_model(splits, family_candidates, feature_block_name)
        family_best_model = refit_best_model_on_train_val(family_best_record, splits)

        pred = family_best_model.predict(splits["X_test"])
        proba = family_best_model.predict_proba(splits["X_test"])

        test_metrics = evaluate_predictions(
            y_true=splits["y_test"],
            y_pred=pred,
            proba=proba,
            classes=splits["classes"],
            split_name="test",
            model_name=best_family_name,
            feature_block=feature_block_name,
        )
        test_metrics["n_features"] = len(feature_cols)
        ablation_rows.append(test_metrics)

    ablation_df = pd.DataFrame(ablation_rows).sort_values(
        by=["macro_f1", "balanced_accuracy"], ascending=False
    ).reset_index(drop=True)
    ablation_df.to_csv(dirs["tables"] / "step5_feature_ablation_test.csv", index=False)

    # Reconstructability audit is a focused view of ablation results
    reconstructability_df = ablation_df[
        ablation_df["feature_block"].isin(["minimal_condition_proxy", "condition_core_raw", "condition_core_z", "combined_all"])
    ].copy()
    reconstructability_df.to_csv(dirs["tables"] / "step5_audit_reconstructability.csv", index=False)

    _print_header("Bootstrapping confidence intervals for the best full-feature model")
    boot_df = bootstrap_test_metrics(
        y_true=y_test,
        y_pred=y_test_pred,
        proba=y_test_proba,
        classes=classes,
        n_iter=n_boot,
        seed=seed,
    )
    boot_df.to_csv(dirs["tables"] / "step5_bootstrap_samples.csv", index=False)

    boot_summary_df = summarize_bootstrap(boot_df)
    boot_summary_df.to_csv(dirs["tables"] / "step5_bootstrap_summary.csv", index=False)

    model_comparison_df = attach_ci_to_metrics(model_comparison_df, boot_summary_df, metric_name="macro_f1", prefix="test_")
    model_comparison_df.to_csv(dirs["tables"] / "step5_main_benchmark_table.csv", index=False)

    _print_header("Computing permutation importance for the best full-feature model")
    perm_imp_df = compute_permutation_importance_table(
        fitted_model=best_full_model,
        X_test=X_test,
        y_test=y_test,
        feature_cols=combined_features,
        n_repeats=n_perm,
        seed=seed,
        scoring="f1_macro",
    )
    perm_imp_df.to_csv(dirs["tables"] / "step5_permutation_importance.csv", index=False)

    _print_header("Running descriptor-space separability audit")
    proj_df, pca_meta = compute_pca_projection(
        X_train=combined_splits["X_train"],
        X_test=combined_splits["X_test"],
        y_test=y_test,
        max_components=2,
    )
    proj_df.to_csv(dirs["tables"] / "step5_audit_pca_projection.csv", index=False)
    _safe_json_dump(pca_meta, dirs["artifacts"] / "step5_audit_pca_meta.json")

    _print_header("Running split/class integrity audit")
    class_count_df = class_counts_by_split(merged_df, cfg)
    class_count_df.to_csv(dirs["tables"] / "step5_audit_class_counts_by_split.csv", index=False)

    dist_df = descriptor_distribution_by_split(
        merged_df,
        cfg,
        descriptor_cols=["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"],
    )
    dist_df.to_csv(dirs["tables"] / "step5_audit_descriptor_distribution_by_split.csv", index=False)

    _print_header("Classifying audit outcome")
    audit_summary = classify_audit_outcome(
        best_full_metrics=best_test_metrics,
        reconstructability_df=reconstructability_df,
        pca_meta=pca_meta,
        cfg=cfg,
    )
    audit_reco = manuscript_recommendation_from_audit(audit_summary["audit_category"])
    audit_summary.update(audit_reco)

    audit_summary_df = pd.DataFrame([audit_summary])
    audit_summary_df.to_csv(dirs["tables"] / "step5_audit_summary.csv", index=False)
    _safe_json_dump(audit_summary, dirs["artifacts"] / "step5_audit_summary.json")

    _print_header("Saving figures")
    if cfg.generate_manuscript_figures:
        save_main_figure_5_1(
            train_n=len(combined_splits["train_df"]),
            val_n=len(combined_splits["val_df"]),
            test_n=len(combined_splits["test_df"]),
            val_df=val_results_df.groupby("model_name", as_index=False).first()[["model_name", "macro_f1"]],
            test_df=model_comparison_df,
            out_base=dirs["figures_main"] / "step5_main_baseline_benchmark",
        )

        save_main_figure_5_2(
            ablation_df=ablation_df,
            out_base=dirs["figures_main"] / "step5_main_feature_ablation",
        )

    if cfg.generate_audit_figures:
        save_reconstructability_audit_figure(
            reconstructability_df,
            dirs["figures_audit"] / "step5_audit_reconstructability",
        )
        save_pca_separability_figure(
            proj_df,
            pca_meta,
            dirs["figures_audit"] / "step5_audit_pca_separability",
        )
        save_split_integrity_figure(
            class_count_df,
            dist_df,
            dirs["figures_audit"] / "step5_audit_split_integrity",
        )

    cm_df = save_confusion_matrix_figure(
        y_true=y_test,
        y_pred=y_test_pred,
        classes=classes,
        out_base=dirs["figures_supplementary"] / "step5_supp_confusion_matrix",
        normalize="true",
    )
    cm_df.to_csv(dirs["tables"] / "step5_confusion_matrix_normalized.csv", index=True)

    save_permutation_importance_figure(
        df_imp=perm_imp_df,
        out_base=dirs["figures_supplementary"] / "step5_supp_permutation_importance",
        top_n=20,
    )

    _print_header("Writing reports and reproducibility artifacts")
    figure_manifest_df = build_figure_manifest(audit_summary["audit_category"])
    figure_manifest_df.to_csv(dirs["tables"] / "step5_figure_manifest.csv", index=False)

    run_summary_lines = [
        "Step 5 — upgraded baseline benchmarking + audit pipeline",
        "",
        "Objective",
        "Establish leakage-aware classical baselines and audit whether perfect performance is meaningful, descriptor-trivial, or target-reconstructable.",
        "",
        "Dataset summary",
        f"- Total merged rows: {len(merged_df):,}",
        f"- Train rows: {len(combined_splits['train_df']):,}",
        f"- Validation rows: {len(combined_splits['val_df']):,}",
        f"- Test rows: {len(combined_splits['test_df']):,}",
        f"- Number of classes: {len(classes)}",
        "",
        "Best full-feature model",
        f"- Model: {best_full_model_name}",
        f"- Validation macro F1: {best_full_val_metrics['macro_f1']:.4f}",
        f"- Test macro F1: {best_test_metrics['macro_f1']:.4f}",
        f"- Test balanced accuracy: {best_test_metrics['balanced_accuracy']:.4f}",
        f"- Test top-1 accuracy: {best_test_metrics['top1_accuracy']:.4f}",
        f"- Test top-3 accuracy: {best_test_metrics['top3_accuracy']:.4f}",
        "",
        "Audit classification",
        f"- Category: {audit_summary['audit_category']}",
        f"- Conclusion: {audit_summary['conclusion']}",
        f"- Main figure 5.1 recommendation: {audit_summary['main_figure_5_1']}",
        f"- Main figure 5.2 recommendation: {audit_summary['main_figure_5_2']}",
        f"- Step 5 manuscript role: {audit_summary['step5_role']}",
        "",
    ]
    write_text_report(run_summary_lines, dirs["reports"] / "step5_run_summary.txt")

    audit_report_lines = [
        "Step 5 audit report",
        "",
        f"Audit category: {audit_summary['audit_category']}",
        f"Conclusion: {audit_summary['conclusion']}",
        "",
        "Key audit metrics",
        f"- Best full-feature macro F1: {audit_summary['best_full_macro_f1']:.4f}",
        f"- Best non-full macro F1: {audit_summary['best_non_full_macro_f1']:.4f}" if pd.notna(audit_summary['best_non_full_macro_f1']) else "- Best non-full macro F1: NA",
        f"- Best condition-proxy macro F1: {audit_summary['best_condition_proxy_macro_f1']:.4f}" if pd.notna(audit_summary['best_condition_proxy_macro_f1']) else "- Best condition-proxy macro F1: NA",
        f"- PCA explained variance total (2D): {audit_summary['pca_explained_variance_total_2d']:.4f}" if pd.notna(audit_summary['pca_explained_variance_total_2d']) else "- PCA explained variance total (2D): NA",
        "",
        "Manuscript recommendation",
        f"- Main Figure 5.1: {audit_summary['main_figure_5_1']}",
        f"- Main Figure 5.2: {audit_summary['main_figure_5_2']}",
        f"- Step 5 role: {audit_summary['step5_role']}",
        "",
        "Interpretation note",
        "If audit category is B or C, later deep-learning claims must be framed conservatively and explicitly justify value beyond descriptor-defined separability.",
    ]
    write_text_report(audit_report_lines, dirs["reports"] / "step5_audit_report.txt")

    next_step_df = pd.DataFrame([
        {
            "next_step": "Step 6",
            "purpose": "Use audit outcome to determine how Step 5 should be cited in the manuscript and whether deep-model gains must be framed conservatively.",
            "recommended_inputs": str(dirs["root"]),
        }
    ])
    next_step_df.to_csv(dirs["tables"] / "step5_next_step_inputs.csv", index=False)

    _safe_json_dump(asdict(cfg), dirs["artifacts"] / "step5_config.json")

    artifact_hashes = {}
    for f in sorted(dirs["root"].rglob("*")):
        if f.is_file():
            artifact_hashes[str(f.relative_to(dirs["root"]))] = _sha256_of_file(f)
    _safe_json_dump(artifact_hashes, dirs["artifacts"] / "step5_artifact_hashes.json")

    _print_header("Step 5 completed successfully")
    print(f"Output directory: {dirs['root']}")
    print("-" * 80)
    print("Best full-feature model:", best_full_model_name)
    print("Test macro F1:", round(best_test_metrics["macro_f1"], 4))
    print("Test balanced accuracy:", round(best_test_metrics["balanced_accuracy"], 4))
    print("Test top-1 accuracy:", round(best_test_metrics["top1_accuracy"], 4))
    print("Test top-3 accuracy:", round(best_test_metrics["top3_accuracy"], 4))
    print("Audit category:", audit_summary["audit_category"])
    print("Audit conclusion:", audit_summary["conclusion"])

    return {
        "output_dir": str(dirs["root"]),
        "best_model_name": best_full_model_name,
        "best_test_metrics": best_test_metrics,
        "model_comparison_df": model_comparison_df,
        "ablation_df": ablation_df,
        "per_class_df": per_class_df,
        "bootstrap_summary_df": boot_summary_df,
        "permutation_importance_df": perm_imp_df,
        "reconstructability_df": reconstructability_df,
        "pca_projection_df": proj_df,
        "pca_meta": pca_meta,
        "audit_summary": audit_summary,
        "feature_blocks": feature_blocks,
        "figure_manifest_df": figure_manifest_df,
    }


# ---------------------------------------------------------
# Notebook entry point
# ---------------------------------------------------------

def main_notebook(
    input_model_ready_csv: str,
    input_baseline_features_csv: str,
    input_preprocessing_contract_json: Optional[str] = None,
    input_condition_id_mapping_json: Optional[str] = None,
    out_dir_name: str = "step5",
    fast_mode: bool = False,
    bootstrap_iterations: int = 500,
    permutation_repeats: int = 10,
    random_seed: int = 42,
    generate_audit_figures: bool = True,
    generate_manuscript_figures: bool = True,
):
    cfg = Step5Config(
        input_model_ready_csv=input_model_ready_csv,
        input_baseline_features_csv=input_baseline_features_csv,
        input_preprocessing_contract_json=input_preprocessing_contract_json,
        input_condition_id_mapping_json=input_condition_id_mapping_json,
        out_dir_name=out_dir_name,
        fast_mode=fast_mode,
        bootstrap_iterations=bootstrap_iterations,
        permutation_repeats=permutation_repeats,
        random_seed=random_seed,
        generate_audit_figures=generate_audit_figures,
        generate_manuscript_figures=generate_manuscript_figures,
    )
    return run_step5_pipeline(cfg)


# ---------------------------------------------------------
# Example run
# ---------------------------------------------------------

if __name__ == "__main__":
    results = main_notebook(
        input_model_ready_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv",
        input_baseline_features_csv="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv",
        input_preprocessing_contract_json="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts/step4_preprocessing_contract.json",
        input_condition_id_mapping_json="/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts/step4_condition_id_mapping.json",
        out_dir_name="step5",
        fast_mode=False,
        bootstrap_iterations=500,
        permutation_repeats=10,
        random_seed=42,
        generate_audit_figures=True,
        generate_manuscript_figures=True,
    )
    print("Step 5 finished.")

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd


# =========================
# User setting
# =========================
# Set this to your project root or the folder that contains step4
SEARCH_ROOT = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2").resolve()

# How many rows to inspect per CSV
N_PREVIEW = 5


# =========================
# Helpers
# =========================
def safe_read_csv_head(path: Path, nrows: int = 5) -> Optional[pd.DataFrame]:
    try:
        return pd.read_csv(path, nrows=nrows)
    except Exception:
        return None


def safe_read_columns(path: Path) -> List[str]:
    try:
        return list(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return []


def score_file(path: Path, cols: List[str]) -> Dict[str, int]:
    """
    Heuristic scoring for likely Step 4 files.
    """
    name = path.name.lower()
    cols_l = [c.lower() for c in cols]

    score = {
        "model_ready": 0,
        "baseline_features": 0,
        "split_source": 0,
        "tokenizer_or_vocab": 0,
        "condition_map": 0,
    }

    # filename clues
    if "model_ready" in name:
        score["model_ready"] += 5
    if "baseline" in name and "feature" in name:
        score["baseline_features"] += 5
    if "split" in name:
        score["split_source"] += 4
    if "tokenizer" in name or "vocab" in name:
        score["tokenizer_or_vocab"] += 5
    if "condition" in name and "map" in name:
        score["condition_map"] += 5

    # column clues
    sequence_like = {"sequence", "peptide", "seq", "aa_sequence", "clean_sequence"}
    id_like = {"sequence_id", "id", "peptide_id", "record_id"}
    split_like = {"split", "data_split", "fold_split", "partition", "set"}
    condition_like = {"primary_condition_id", "condition_id", "condition", "label"}
    feature_like = {
        "length", "hydrophobicity", "charge", "mw", "boman",
        "aromaticity", "instability_index", "isoelectric_point"
    }

    if any(c in cols_l for c in sequence_like):
        score["model_ready"] += 3
    if any(c in cols_l for c in id_like):
        score["model_ready"] += 1
        score["baseline_features"] += 1
    if any(c in cols_l for c in split_like):
        score["split_source"] += 5
        score["model_ready"] += 1
    if any(c in cols_l for c in condition_like):
        score["model_ready"] += 2
        score["condition_map"] += 2
    if len(set(cols_l).intersection(feature_like)) >= 2:
        score["baseline_features"] += 5

    # many columns often means feature table
    if len(cols) >= 20:
        score["baseline_features"] += 2

    return score


def detect_special_columns(cols: List[str]) -> Dict[str, List[str]]:
    cols_l = [c.lower() for c in cols]

    patterns = {
        "sequence_columns": [r"sequence", r"peptide", r"seq"],
        "id_columns": [r"(^id$|_id$|^record_id$|^sequence_id$|^peptide_id$)"],
        "split_columns": [r"split", r"partition", r"set"],
        "condition_columns": [r"condition", r"label", r"class"],
        "length_columns": [r"length"],
    }

    found: Dict[str, List[str]] = {k: [] for k in patterns}
    for c in cols:
        cl = c.lower()
        for key, regs in patterns.items():
            if any(re.search(rgx, cl) for rgx in regs):
                found[key].append(c)
    return found


def find_candidate_files(root: Path) -> List[Path]:
    candidates = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".csv", ".json", ".txt", ".pkl", ".joblib", ".pt", ".yaml", ".yml"}:
            candidates.append(p)
    return candidates


# =========================
# Discovery
# =========================
def build_step4_inventory(search_root: Path) -> Dict:
    files = find_candidate_files(search_root)

    csv_files = [p for p in files if p.suffix.lower() == ".csv"]
    other_files = [p for p in files if p.suffix.lower() != ".csv"]

    csv_reports = []
    for csv_path in csv_files:
        cols = safe_read_columns(csv_path)
        scores = score_file(csv_path, cols)
        detected = detect_special_columns(cols)

        csv_reports.append({
            "path": str(csv_path),
            "name": csv_path.name,
            "n_columns": len(cols),
            "columns": cols,
            "scores": scores,
            "detected_columns": detected,
        })

    def best_match(key: str) -> Optional[Dict]:
        ranked = sorted(
            csv_reports,
            key=lambda x: (x["scores"][key], x["n_columns"]),
            reverse=True
        )
        if not ranked or ranked[0]["scores"][key] <= 0:
            return None
        return ranked[0]

    # non-CSV artifacts
    tokenizer_candidates = [
        str(p) for p in other_files
        if any(k in p.name.lower() for k in ["tokenizer", "vocab", "alphabet"])
    ]
    condition_map_candidates = [
        str(p) for p in other_files
        if any(k in p.name.lower() for k in ["condition", "label_map", "class_map"])
    ]

    inventory = {
        "search_root": str(search_root),
        "best_guess": {
            "model_ready_csv": best_match("model_ready"),
            "baseline_features_csv": best_match("baseline_features"),
            "split_source_csv": best_match("split_source"),
            "tokenizer_or_vocab_files": tokenizer_candidates,
            "condition_map_files": condition_map_candidates,
        },
        "all_csv_reports": csv_reports,
    }
    return inventory


def print_summary(inv: Dict) -> None:
    print("=" * 100)
    print("STEP 4 DISCOVERY SUMMARY")
    print("=" * 100)
    print(f"Search root: {inv['search_root']}\n")

    bg = inv["best_guess"]

    def print_guess(title: str, item):
        print(f"[{title}]")
        if item is None:
            print("  Not found\n")
            return
        if isinstance(item, list):
            if not item:
                print("  None\n")
                return
            for x in item[:10]:
                print(f"  - {x}")
            print()
            return
        print(f"  Path: {item['path']}")
        print(f"  Columns: {item['n_columns']}")
        dc = item["detected_columns"]
        for k, v in dc.items():
            if v:
                print(f"  {k}: {v}")
        print()

    print_guess("Likely model-ready CSV", bg["model_ready_csv"])
    print_guess("Likely baseline-features CSV", bg["baseline_features_csv"])
    print_guess("Likely split-source CSV", bg["split_source_csv"])
    print_guess("Tokenizer/vocab files", bg["tokenizer_or_vocab_files"])
    print_guess("Condition-map files", bg["condition_map_files"])

    print("=" * 100)
    print("TOP CSV CANDIDATES")
    print("=" * 100)

    ranked = sorted(
        inv["all_csv_reports"],
        key=lambda x: (
            x["scores"]["model_ready"]
            + x["scores"]["baseline_features"]
            + x["scores"]["split_source"]
        ),
        reverse=True
    )

    for row in ranked[:15]:
        print(f"- {row['path']}")
        print(f"  scores={row['scores']}")
        print(f"  detected={row['detected_columns']}")
        print()


def save_inventory(inv: Dict, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(inv, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    inventory = build_step4_inventory(SEARCH_ROOT)
    print_summary(inventory)

    out_json = SEARCH_ROOT / "step4_discovery_for_step5.json"
    save_inventory(inventory, out_json)
    print(f"Saved discovery JSON to:\n{out_json}")

nature step 5: 

In [ ]:
from __future__ import annotations

# ======================================================================================
# Step 5 — Publication-grade baseline benchmark notebook code
# Single Jupyter-ready Python script
# ======================================================================================

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import gc
import sys
import json
import time
import hashlib
import random
import platform
import warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


plt.rcParams.update(
    {
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.transparent": False,
    }
)

OKABE_ITO = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "yellow": "#F0E442",
    "black": "#000000",
    "gray": "#7F7F7F",
    "light_gray": "#D9D9D9",
}

MODEL_DISPLAY_NAMES = {
    "random_conditional": "Random conditional",
    "retrieval_reference": "Retrieval reference",
    "gru_unconditional": "GRU (unconditional)",
    "vae_unconditional": "VAE (unconditional)",
    "gru_conditional": "GRU (conditional)",
    "cvae_conditional": "CVAE (conditional)",
}

MODEL_COLORS = {
    "random_conditional": OKABE_ITO["gray"],
    "retrieval_reference": OKABE_ITO["sky"],
    "gru_unconditional": OKABE_ITO["blue"],
    "vae_unconditional": OKABE_ITO["orange"],
    "gru_conditional": OKABE_ITO["purple"],
    "cvae_conditional": OKABE_ITO["green"],
}

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)
AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_TINY = set("AGSC")
AA_SMALL = set("AGSCVTNDP")
AA_PROPERTIES = {
    "A": {"hydro": 1.8}, "C": {"hydro": 2.5}, "D": {"hydro": -3.5}, "E": {"hydro": -3.5},
    "F": {"hydro": 2.8}, "G": {"hydro": -0.4}, "H": {"hydro": -3.2}, "I": {"hydro": 4.5},
    "K": {"hydro": -3.9}, "L": {"hydro": 3.8}, "M": {"hydro": 1.9}, "N": {"hydro": -3.5},
    "P": {"hydro": -1.6}, "Q": {"hydro": -3.5}, "R": {"hydro": -4.5}, "S": {"hydro": -0.8},
    "T": {"hydro": -0.7}, "V": {"hydro": 4.2}, "W": {"hydro": -0.9}, "Y": {"hydro": -1.3},
}


@dataclass
class FigureExportConfig:
    pdf: bool = True
    png: bool = True
    tif: bool = False
    png_dpi: int = 300
    tif_dpi: int = 600
    single_col_width_in: float = 3.45
    double_col_width_in: float = 7.10
    base_fontsize: int = 8
    panel_fontsize: int = 10


@dataclass
class GeneratorTrainConfig:
    max_seq_len: int = 64
    batch_size: int = 64
    num_workers: int = 0
    emb_dim: int = 64
    hidden_dim: int = 128
    latent_dim: int = 32
    num_layers: int = 1
    dropout: float = 0.10
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs_fast: int = 5
    epochs_full: int = 20
    clip_grad_norm: float = 1.0
    early_stopping_patience: int = 4
    temperature: float = 1.0


@dataclass
class Step5Config:
    input_model_ready_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv"
    input_baseline_features_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv"
    out_dir_name: str = "step5"

    sequence_col: str = "sequence"
    merge_key: str = "sequence_sha256"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    main_random_seed: int = 42
    repeated_seeds: Tuple[int, ...] = (42, 52, 62)
    inner_cv_folds: int = 5
    bootstrap_iterations: int = 400
    permutation_repeats: int = 20
    topk_values: Tuple[int, ...] = (1, 3)
    shuffle_repeats: int = 10

    fast_mode: bool = False
    n_generated_per_condition: int = 100
    similarity_kmer: int = 3
    pairwise_diversity_sample_size: int = 250
    max_unique_conditions_for_figures: int = 12
    min_samples_per_class_for_cv: int = 5
    low_complexity_threshold: float = 0.70

    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)
    gen_train: GeneratorTrainConfig = field(default_factory=GeneratorTrainConfig)


def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)


def seed_all(seed: int, deterministic_torch: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if deterministic_torch:
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


def device_for_training() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def now_timestamp() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)


def add_panel_label(ax: plt.Axes, label: str, fontsize: int = 10) -> None:
    ax.text(-0.14, 1.04, label, transform=ax.transAxes, fontsize=fontsize, fontweight="bold", va="bottom")


def _pretty_feature_name(name: str) -> str:
    mapping = {
        "length": "Length",
        "net_charge_proxy": "Charge proxy",
        "mean_hydropathy": "Mean hydropathy",
        "sequence_entropy": "Sequence entropy",
        "max_residue_fraction": "Max residue fraction",
    }
    if name in mapping:
        return mapping[name]
    if name.startswith("aa_frac_"):
        return f"AA fraction ({name.split('_')[-1]})"
    if name.endswith("_frac"):
        return name.replace('_frac', ' fraction').replace('_', ' ').title()
    return name.replace('_', ' ').title()


def _dot_whisker(ax, labels, values, lows, highs, colors, xlabel=None, title=None):
    y = np.arange(len(labels))[::-1]
    for yi, val, lo, hi, color in zip(y, values, lows, highs, colors):
        ax.hlines(yi, lo, hi, color=color, linewidth=1.5)
        ax.scatter(val, yi, s=34, color=color, edgecolor='black', linewidth=0.4, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    if xlabel:
        ax.set_xlabel(xlabel)
    if title:
        ax.set_title(title)
    ax.grid(alpha=0.25, axis='x')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)


def _generator_order(summary_df: pd.DataFrame) -> list:
    preferred = ['retrieval_reference', 'gru_conditional', 'cvae_conditional', 'random_conditional', 'vae_unconditional', 'gru_unconditional']
    present = [g for g in preferred if g in summary_df['generator'].tolist()]
    remainder = [g for g in summary_df['generator'].tolist() if g not in present]
    return present + remainder


def project_root_from_input(input_csv: str) -> Path:
    p = Path(input_csv).resolve()
    for parent in p.parents:
        if parent.name.startswith("nature"):
            return parent
    return p.parent.parent if p.parent.name == "tables" else p.parent


def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "models": base_dir / "models",
        "samples": base_dir / "samples",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp": now_timestamp(),
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "torch": torch.__version__,
        "sklearn": __import__("sklearn").__version__,
        "cuda_available": torch.cuda.is_available(),
        "device": str(device_for_training()),
    }


def manifest_from_outputs(base_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append(
                {
                    "relative_path": str(path.relative_to(base_dir)),
                    "size_bytes": path.stat().st_size,
                    "sha256": sha256_of_file(path),
                }
            )
    return pd.DataFrame(rows)


def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join([c for c in seq if c in AA_SET])


def is_valid_peptide(seq: str) -> bool:
    return len(seq) > 0 and all(c in AA_SET for c in seq)


def aa_fraction(seq: str, aa: str) -> float:
    if len(seq) == 0:
        return 0.0
    return seq.count(aa) / len(seq)


def shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return 0.0
    counts = np.array([seq.count(a) for a in AA_ALPHABET], dtype=float)
    probs = counts[counts > 0] / counts.sum()
    return float(-(probs * np.log2(probs)).sum())


def max_residue_fraction(seq: str) -> float:
    if len(seq) == 0:
        return 0.0
    counts = Counter(seq)
    return max(counts.values()) / len(seq)


def is_low_complexity(seq: str, threshold: float = 0.70) -> bool:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return True
    return max_residue_fraction(seq) >= threshold


def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    n = len(seq)
    base = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
    if n == 0:
        base.update(
            {
                "length": 0.0,
                "hydrophobic_frac": 0.0,
                "aromatic_frac": 0.0,
                "positive_frac": 0.0,
                "negative_frac": 0.0,
                "polar_frac": 0.0,
                "tiny_frac": 0.0,
                "small_frac": 0.0,
                "net_charge_proxy": 0.0,
                "mean_hydropathy": 0.0,
                "sequence_entropy": 0.0,
                "max_residue_fraction": 0.0,
            }
        )
        return base

    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    hydros = [AA_PROPERTIES[a]["hydro"] for a in seq]
    desc.update(
        {
            "length": float(n),
            "hydrophobic_frac": sum(a in AA_HYDROPHOBIC for a in seq) / n,
            "aromatic_frac": sum(a in AA_AROMATIC for a in seq) / n,
            "positive_frac": sum(a in AA_POSITIVE for a in seq) / n,
            "negative_frac": sum(a in AA_NEGATIVE for a in seq) / n,
            "polar_frac": sum(a in AA_POLAR for a in seq) / n,
            "tiny_frac": sum(a in AA_TINY for a in seq) / n,
            "small_frac": sum(a in AA_SMALL for a in seq) / n,
            "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
            "mean_hydropathy": float(np.mean(hydros)),
            "sequence_entropy": shannon_entropy(seq),
            "max_residue_fraction": max_residue_fraction(seq),
        }
    )
    return desc


def add_computable_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    descriptor_df = pd.DataFrame([compute_sequence_descriptors(s) for s in df[seq_col].astype(str)])
    return pd.concat([df.reset_index(drop=True), descriptor_df.reset_index(drop=True)], axis=1)


def build_feature_blocks(df: pd.DataFrame, cfg: Step5Config) -> Dict[str, List[str]]:
    protected = {cfg.sequence_col, cfg.merge_key, cfg.split_col, cfg.target_col}
    all_numeric = [c for c in df.columns if c not in protected and pd.api.types.is_numeric_dtype(df[c])]
    computable = [
        c for c in all_numeric
        if c == "length"
        or c.startswith("aa_frac_")
        or c.endswith("_frac")
        or c in {"net_charge_proxy", "mean_hydropathy", "sequence_entropy", "max_residue_fraction"}
    ]
    aa_only = [c for c in computable if c.startswith("aa_frac_")]
    physicochem = [c for c in computable if c not in aa_only]

    if not computable:
        raise ValueError("No computable descriptor block could be constructed.")

    return {
        "Full numeric descriptor set": all_numeric,
        "Computable descriptors": computable,
        "AA composition only": aa_only,
        "Physicochemical only": physicochem,
    }


def load_step4_inputs(cfg: Step5Config) -> Dict[str, pd.DataFrame]:
    seq_df = pd.read_csv(cfg.input_model_ready_csv)
    feat_df = pd.read_csv(cfg.input_baseline_features_csv)
    return {"seq_df": seq_df, "feat_df": feat_df}


def validate_step4_contract(seq_df: pd.DataFrame, feat_df: pd.DataFrame, cfg: Step5Config) -> Dict[str, Any]:
    required_seq = {cfg.sequence_col, cfg.merge_key, cfg.split_col, cfg.target_col}
    required_feat = {cfg.sequence_col, cfg.merge_key, cfg.split_col, cfg.target_col}
    miss_seq = required_seq - set(seq_df.columns)
    miss_feat = required_feat - set(feat_df.columns)
    if miss_seq:
        raise ValueError(f"Missing required sequence-table columns: {sorted(miss_seq)}")
    if miss_feat:
        raise ValueError(f"Missing required feature-table columns: {sorted(miss_feat)}")

    for df_name, df in [("seq_df", seq_df), ("feat_df", feat_df)]:
        if df[cfg.merge_key].duplicated().any():
            dup_n = int(df[cfg.merge_key].duplicated().sum())
            raise ValueError(f"{df_name} has {dup_n} duplicated merge keys")
        bad_splits = set(df[cfg.split_col].dropna().unique()) - {cfg.train_label, cfg.val_label, cfg.test_label}
        if bad_splits:
            raise ValueError(f"Unexpected split labels in {df_name}: {sorted(bad_splits)}")

    merged = seq_df[[cfg.merge_key, cfg.target_col, cfg.split_col]].merge(
        feat_df[[cfg.merge_key, cfg.target_col, cfg.split_col]], on=cfg.merge_key, suffixes=("_seq", "_feat")
    )
    if len(merged) == 0:
        raise ValueError("No shared rows between sequence and feature tables")
    if not (merged[f"{cfg.target_col}_seq"] == merged[f"{cfg.target_col}_feat"]).all():
        raise ValueError("Target mismatch between sequence and feature tables")
    if not (merged[f"{cfg.split_col}_seq"] == merged[f"{cfg.split_col}_feat"]).all():
        raise ValueError("Split mismatch between sequence and feature tables")

    qc = {}
    seq_df = seq_df.copy()
    seq_df["clean_sequence"] = seq_df[cfg.sequence_col].astype(str).map(clean_sequence)
    seq_df["is_valid_clean"] = seq_df["clean_sequence"].map(is_valid_peptide)
    qc["n_total_rows"] = int(len(seq_df))
    qc["n_valid_clean_sequences"] = int(seq_df["is_valid_clean"].sum())
    qc["n_invalid_after_cleaning"] = int((~seq_df["is_valid_clean"]).sum())

    split_counts = seq_df[cfg.split_col].value_counts().to_dict()
    for split_name in [cfg.train_label, cfg.val_label, cfg.test_label]:
        if split_counts.get(split_name, 0) == 0:
            raise ValueError(f"Split {split_name!r} has zero rows.")

    clean_valid = seq_df[seq_df["is_valid_clean"]].copy()
    train_classes = set(clean_valid.loc[clean_valid[cfg.split_col] == cfg.train_label, cfg.target_col].astype(str))
    val_classes = set(clean_valid.loc[clean_valid[cfg.split_col] == cfg.val_label, cfg.target_col].astype(str))
    test_classes = set(clean_valid.loc[clean_valid[cfg.split_col] == cfg.test_label, cfg.target_col].astype(str))
    missing_from_train = sorted((val_classes | test_classes) - train_classes)
    if missing_from_train:
        raise ValueError(f"Validation/test contain classes absent from train: {missing_from_train}")

    train_counts = clean_valid.loc[clean_valid[cfg.split_col] == cfg.train_label, cfg.target_col].value_counts()
    if (train_counts < cfg.min_samples_per_class_for_cv).any():
        too_small = train_counts[train_counts < cfg.min_samples_per_class_for_cv].to_dict()
        raise ValueError(
            f"Some training classes have fewer than {cfg.min_samples_per_class_for_cv} samples, which breaks reliable CV: {too_small}"
        )
    qc["train_class_counts"] = {str(k): int(v) for k, v in train_counts.to_dict().items()}
    return qc


def build_model_registry(seed: int) -> Dict[str, Tuple[Pipeline, List[Dict[str, Any]]]]:
    simple_pre = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    registry = {
        "Dummy majority": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("clf", DummyClassifier(strategy="most_frequent"))]),
            [{}],
        ),
        "Logistic regression": (
            Pipeline(simple_pre + [("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=seed))]),
            [{"clf__C": c} for c in [0.25, 1.0, 4.0]],
        ),
        "Random forest": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("clf", RandomForestClassifier(random_state=seed, n_jobs=-1, class_weight="balanced_subsample")),
            ]),
            [
                {"clf__n_estimators": 300, "clf__max_depth": None, "clf__min_samples_leaf": 1},
                {"clf__n_estimators": 400, "clf__max_depth": 16, "clf__min_samples_leaf": 2},
            ],
        ),
        "Extra trees": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("clf", ExtraTreesClassifier(random_state=seed, n_jobs=-1, class_weight="balanced")),
            ]),
            [
                {"clf__n_estimators": 300, "clf__max_depth": None, "clf__min_samples_leaf": 1},
                {"clf__n_estimators": 400, "clf__max_depth": 18, "clf__min_samples_leaf": 2},
            ],
        ),
    }
    return registry


def topk_accuracy(y_true: np.ndarray, probas: np.ndarray, classes: np.ndarray, k: int) -> float:
    idx = np.argsort(-probas, axis=1)[:, :k]
    topk_labels = classes[idx]
    hits = [yt in set(topk_labels[i]) for i, yt in enumerate(y_true)]
    return float(np.mean(hits))


def compute_classification_metrics(
    y_true: np.ndarray,
    probas: np.ndarray,
    pred: np.ndarray,
    classes: np.ndarray,
    topk_values: Sequence[int],
) -> Dict[str, float]:
    metrics = {
        "macro_f1": float(f1_score(y_true, pred, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(y_true, pred)),
    }
    for k in topk_values:
        kk = min(k, probas.shape[1])
        metrics[f"top{kk}_acc"] = topk_accuracy(y_true, probas, classes, kk)
    try:
        if len(classes) > 2:
            y_onehot = pd.get_dummies(pd.Series(y_true), dtype=int).reindex(columns=classes, fill_value=0).values
            metrics["macro_ovr_auc"] = float(roc_auc_score(y_onehot, probas, average="macro", multi_class="ovr"))
        else:
            pos_label = classes[1]
            y_bin = (y_true == pos_label).astype(int)
            metrics["macro_ovr_auc"] = float(roc_auc_score(y_bin, probas[:, 1]))
    except Exception:
        metrics["macro_ovr_auc"] = float("nan")
    return metrics


def train_eval_family(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    seed: int,
    cfg: Step5Config,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    registry = build_model_registry(seed)
    cv = StratifiedKFold(n_splits=cfg.inner_cv_folds, shuffle=True, random_state=seed)
    rows = []
    fitted = {}

    for family, (pipe, grid) in registry.items():
        for params in grid:
            inner_scores = []
            for tr_idx, te_idx in cv.split(X_train, y_train):
                mdl = clone(pipe)
                mdl.set_params(**params)
                mdl.fit(X_train.iloc[tr_idx], y_train[tr_idx])
                pred = mdl.predict(X_train.iloc[te_idx])
                inner_scores.append(f1_score(y_train[te_idx], pred, average="macro"))
            mean_cv = float(np.mean(inner_scores))

            mdl = clone(pipe)
            mdl.set_params(**params)
            mdl.fit(X_train, y_train)
            val_probas = mdl.predict_proba(X_val)
            val_pred = mdl.predict(X_val)
            val_metrics = compute_classification_metrics(y_val, val_probas, val_pred, mdl.named_steps["clf"].classes_, cfg.topk_values)
            rows.append(
                {
                    "family": family,
                    "params": json.dumps(params, sort_keys=True),
                    "cv_macro_f1": mean_cv,
                    **{f"val_{k}": v for k, v in val_metrics.items()},
                }
            )
            fitted[(family, json.dumps(params, sort_keys=True))] = mdl

    results = pd.DataFrame(rows).sort_values(["val_macro_f1", "cv_macro_f1"], ascending=False).reset_index(drop=True)
    best = results.iloc[0]
    best_key = (best["family"], best["params"])
    return results, {"model": fitted[best_key], "family": best["family"], "params": json.loads(best["params"])}


def bootstrap_test_metrics(model: Pipeline, X_test: pd.DataFrame, y_test: np.ndarray, cfg: Step5Config, seed: int) -> Dict[str, Dict[str, float]]:
    rng = np.random.default_rng(seed)
    base_probas = model.predict_proba(X_test)
    base_pred = model.predict(X_test)
    classes = model.named_steps["clf"].classes_
    metric_names = ["macro_f1", "balanced_acc", "macro_ovr_auc"] + [f"top{min(k, base_probas.shape[1])}_acc" for k in cfg.topk_values]
    hist = defaultdict(list)
    n = len(y_test)
    for _ in range(cfg.bootstrap_iterations):
        idx = rng.integers(0, n, size=n)
        m = compute_classification_metrics(y_test[idx], base_probas[idx], base_pred[idx], classes, cfg.topk_values)
        for k in metric_names:
            hist[k].append(m.get(k, np.nan))
    out = {}
    for k, vals in hist.items():
        vals = np.asarray(vals, dtype=float)
        out[k] = {
            "mean": float(np.nanmean(vals)),
            "ci_low": float(np.nanpercentile(vals, 2.5)),
            "ci_high": float(np.nanpercentile(vals, 97.5)),
        }
    return out


def run_repeated_shuffle_control(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    cfg: Step5Config,
) -> pd.DataFrame:
    rows = []
    for rep in range(cfg.shuffle_repeats):
        seed = cfg.main_random_seed + 1000 + rep
        rng = np.random.default_rng(seed)
        y_train_shuf = y_train.copy()
        y_val_shuf = y_val.copy()
        rng.shuffle(y_train_shuf)
        rng.shuffle(y_val_shuf)

        _, best = train_eval_family(X_train, y_train_shuf, X_val, y_val_shuf, seed=seed, cfg=cfg)
        model = best["model"]
        X_tv = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
        y_tv_shuf = np.concatenate([y_train_shuf, y_val_shuf])
        model.fit(X_tv, y_tv_shuf)
        probas = model.predict_proba(X_test)
        pred = model.predict(X_test)
        metrics = compute_classification_metrics(y_test, probas, pred, model.named_steps["clf"].classes_, cfg.topk_values)
        rows.append({"shuffle_repeat": rep, **metrics})
    return pd.DataFrame(rows)


def run_step5a_descriptor_audit(data: Dict[str, pd.DataFrame], cfg: Step5Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Running Step 5A — descriptor benchmark and non-triviality audit")
    feat_df = add_computable_descriptors(data["feat_df"].copy(), cfg.sequence_col)
    feat_df[cfg.sequence_col] = feat_df[cfg.sequence_col].astype(str).map(clean_sequence)
    feat_df = feat_df[feat_df[cfg.sequence_col].map(is_valid_peptide)].reset_index(drop=True)
    blocks = build_feature_blocks(feat_df, cfg)

    split = feat_df[cfg.split_col]
    train_df = feat_df[split == cfg.train_label].reset_index(drop=True)
    val_df = feat_df[split == cfg.val_label].reset_index(drop=True)
    test_df = feat_df[split == cfg.test_label].reset_index(drop=True)

    if min(len(train_df), len(val_df), len(test_df)) == 0:
        raise ValueError("At least one split is empty after descriptor cleaning.")

    le = LabelEncoder()
    y_train = le.fit_transform(train_df[cfg.target_col].astype(str))
    y_val = le.transform(val_df[cfg.target_col].astype(str))
    y_test = le.transform(test_df[cfg.target_col].astype(str))

    feature_manifest_rows = []
    main_rows = []
    fitted_block_best = {}
    full_search_tables = {}

    for block_name, cols in blocks.items():
        if len(cols) == 0:
            continue
        feature_manifest_rows.append({"block": block_name, "n_features": len(cols), "features": ";".join(cols)})
        res_df, best = train_eval_family(train_df[cols], y_train, val_df[cols], y_val, seed=cfg.main_random_seed, cfg=cfg)
        full_search_tables[block_name] = res_df
        best_model = best["model"]

        X_tv = pd.concat([train_df[cols], val_df[cols]], axis=0).reset_index(drop=True)
        y_tv = np.concatenate([y_train, y_val])
        best_model.fit(X_tv, y_tv)
        probas = best_model.predict_proba(test_df[cols])
        pred = best_model.predict(test_df[cols])
        metrics = compute_classification_metrics(y_test, probas, pred, best_model.named_steps["clf"].classes_, cfg.topk_values)
        boot = bootstrap_test_metrics(best_model, test_df[cols], y_test, cfg=cfg, seed=cfg.main_random_seed + 111)

        row = {
            "block": block_name,
            "family": best["family"],
            "n_features": len(cols),
            **metrics,
        }
        for mk, stats in boot.items():
            row[f"{mk}_ci_low"] = stats["ci_low"]
            row[f"{mk}_ci_high"] = stats["ci_high"]
        main_rows.append(row)
        fitted_block_best[block_name] = {
            "model": best_model,
            "features": cols,
            "metrics": metrics,
            "bootstrap": boot,
            "family": best["family"],
        }

    summary_df = pd.DataFrame(main_rows).sort_values(["macro_f1", "balanced_acc"], ascending=False).reset_index(drop=True)
    summary_df.to_csv(dirs["tables_main"] / "table_5_1_step5a_descriptor_benchmark.csv", index=False)
    pd.DataFrame(feature_manifest_rows).to_csv(dirs["tables_supplementary"] / "table_s5_1_step5a_feature_manifest.csv", index=False)
    for block_name, table in full_search_tables.items():
        safe = block_name.lower().replace(" ", "_").replace("-", "_")
        table.to_csv(dirs["tables_supplementary"] / f"step5a_model_search_{safe}.csv", index=False)

    surrogate_key = "Computable descriptors" if "Computable descriptors" in fitted_block_best else summary_df.iloc[0]["block"]
    surrogate = fitted_block_best[surrogate_key]

    shuffle_df = run_repeated_shuffle_control(
        train_df[surrogate["features"]], y_train,
        val_df[surrogate["features"]], y_val,
        test_df[surrogate["features"]], y_test,
        cfg=cfg,
    )
    shuffle_df.to_csv(dirs["tables_supplementary"] / "table_s5_2_step5a_shuffle_control.csv", index=False)

    best_block = summary_df.iloc[0]["block"]
    best_obj = fitted_block_best[best_block]
    best_model = best_obj["model"]
    best_features = best_obj["features"]
    X_test_best = test_df[best_features]
    y_pred_best = best_model.predict(X_test_best)
    cm_raw = confusion_matrix(y_test, y_pred_best)
    cm_norm = confusion_matrix(y_test, y_pred_best, normalize="true")
    cm_raw_df = pd.DataFrame(cm_raw, index=le.classes_, columns=le.classes_)
    cm_norm_df = pd.DataFrame(cm_norm, index=le.classes_, columns=le.classes_)
    cm_raw_df.to_csv(dirs["tables_supplementary"] / "step5a_confusion_matrix_raw.csv")
    cm_norm_df.to_csv(dirs["tables_supplementary"] / "step5a_confusion_matrix_normalized.csv")

    perm_df = None
    try:
        perm = permutation_importance(
            best_model,
            X_test_best,
            y_test,
            n_repeats=cfg.permutation_repeats,
            random_state=cfg.main_random_seed,
            scoring="f1_macro",
        )
        perm_df = pd.DataFrame(
            {
                "feature": best_features,
                "importance_mean": perm.importances_mean,
                "importance_std": perm.importances_std,
            }
        ).sort_values("importance_mean", ascending=False)
        perm_df.to_csv(dirs["tables_supplementary"] / "step5a_permutation_importance.csv", index=False)
    except Exception as e:
        warnings.warn(f"Permutation importance failed: {e}")

    pca_cols = surrogate["features"]
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(imputer.fit_transform(train_df[pca_cols]))
    Xte = scaler.transform(imputer.transform(test_df[pca_cols]))
    pca = PCA(n_components=2, random_state=cfg.main_random_seed)
    Xte_2d = pca.fit(Xtr).transform(Xte)
    pca_df = pd.DataFrame({
        "pc1": Xte_2d[:, 0],
        "pc2": Xte_2d[:, 1],
        "condition": test_df[cfg.target_col].astype(str).values,
    })
    pca_df.to_csv(dirs["tables_supplementary"] / "step5a_pca_projection.csv", index=False)

    surrogate_payload = {
        "block": surrogate_key,
        "features": surrogate["features"],
        "label_classes": list(map(str, le.classes_)),
        "best_block_overall": best_block,
        "best_block_family": best_obj["family"],
    }
    save_json(surrogate_payload, dirs["artifacts"] / "step5a_surrogate_metadata.json")

    return {
        "summary_df": summary_df,
        "feature_manifest": pd.DataFrame(feature_manifest_rows),
        "shuffle_df": shuffle_df,
        "surrogate_model": surrogate["model"],
        "surrogate_features": surrogate["features"],
        "label_encoder": le,
        "best_block": best_block,
        "best_block_obj": best_obj,
        "confusion_matrix_raw": cm_raw_df,
        "confusion_matrix_norm": cm_norm_df,
        "perm_df": perm_df,
        "pca_df": pca_df,
        "train_df_with_desc": train_df,
        "val_df_with_desc": val_df,
        "test_df_with_desc": test_df,
    }


class PeptideTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"
        self.alphabet = [self.pad_token, self.bos_token, self.eos_token] + AA_ALPHABET
        self.stoi = {tok: i for i, tok in enumerate(self.alphabet)}
        self.itos = {i: tok for tok, i in self.stoi.items()}

    @property
    def pad_id(self) -> int:
        return self.stoi[self.pad_token]

    @property
    def bos_id(self) -> int:
        return self.stoi[self.bos_token]

    @property
    def eos_id(self) -> int:
        return self.stoi[self.eos_token]

    @property
    def vocab_size(self) -> int:
        return len(self.alphabet)

    def encode(self, seq: str, max_len: int) -> List[int]:
        seq = clean_sequence(seq)
        toks = [self.bos_id] + [self.stoi[s] for s in seq[: max_len - 2]] + [self.eos_id]
        toks = toks[:max_len]
        toks += [self.pad_id] * (max_len - len(toks))
        return toks

    def decode(self, ids: Sequence[int]) -> str:
        chars = []
        for i in ids:
            tok = self.itos[int(i)]
            if tok == self.eos_token:
                break
            if tok in {self.pad_token, self.bos_token}:
                continue
            chars.append(tok)
        return "".join(chars)


class SequenceDataset(Dataset):
    def __init__(self, sequences: Sequence[str], tokenizer: PeptideTokenizer, max_len: int, conditions: Optional[Sequence[int]] = None):
        self.seqs = [clean_sequence(s) for s in sequences]
        self.tok = tokenizer
        self.max_len = max_len
        self.conditions = None if conditions is None else list(conditions)

    def __len__(self) -> int:
        return len(self.seqs)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ids = self.tok.encode(self.seqs[idx], self.max_len)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        out = {"input_ids": x, "target_ids": y}
        if self.conditions is not None:
            out["condition_id"] = torch.tensor(self.conditions[idx], dtype=torch.long)
        return out


def make_dataloader(ds: Dataset, batch_size: int, shuffle: bool, num_workers: int, seed: int) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, generator=generator)


class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, num_layers: int, dropout: float, cond_vocab_size: int = 0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb = nn.Embedding(cond_vocab_size, hidden_dim) if cond_vocab_size > 0 else None
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids: torch.Tensor, condition_id: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = self.emb(input_ids)
        h0 = None
        if self.cond_emb is not None and condition_id is not None:
            h0 = self.cond_emb(condition_id).unsqueeze(0)
        out, _ = self.rnn(x, h0)
        logits = self.out(out)
        return logits


class SequenceVAE(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, latent_dim: int, num_layers: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.encoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)
        self.z_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.latent_dim = latent_dim

    def encode(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        emb = self.emb(input_ids)
        _, h = self.encoder(emb)
        h = h[-1]
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar

    def decode(self, input_ids: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        emb = self.emb(input_ids)
        h0 = self.z_to_hidden(z).unsqueeze(0)
        dec, _ = self.decoder(emb, h0)
        return self.out(dec)

    def forward(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z, mu, logvar = self.encode(input_ids)
        logits = self.decode(input_ids, z)
        return logits, mu, logvar


class ConditionalSequenceVAE(nn.Module):
    def __init__(self, vocab_size: int, cond_vocab_size: int, emb_dim: int, hidden_dim: int, latent_dim: int, num_layers: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb_enc = nn.Embedding(cond_vocab_size, emb_dim)
        self.cond_emb_dec = nn.Embedding(cond_vocab_size, hidden_dim)
        self.encoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)
        self.z_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.latent_dim = latent_dim

    def encode(self, input_ids: torch.Tensor, condition_id: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        emb = self.emb(input_ids) + self.cond_emb_enc(condition_id).unsqueeze(1)
        _, h = self.encoder(emb)
        h = h[-1]
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar

    def decode(self, input_ids: torch.Tensor, z: torch.Tensor, condition_id: torch.Tensor) -> torch.Tensor:
        emb = self.emb(input_ids)
        h0 = (self.z_to_hidden(z) + self.cond_emb_dec(condition_id)).unsqueeze(0)
        dec, _ = self.decoder(emb, h0)
        return self.out(dec)

    def forward(self, input_ids: torch.Tensor, condition_id: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z, mu, logvar = self.encode(input_ids, condition_id)
        logits = self.decode(input_ids, z, condition_id)
        return logits, mu, logvar


def cross_entropy_ignore_pad(logits: torch.Tensor, target_ids: torch.Tensor, pad_id: int) -> torch.Tensor:
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1), ignore_index=pad_id)


def train_gru_model(
    train_sequences: Sequence[str],
    val_sequences: Sequence[str],
    tokenizer: PeptideTokenizer,
    train_conditions: Optional[Sequence[int]],
    val_conditions: Optional[Sequence[int]],
    cond_vocab_size: int,
    seed: int,
    cfg: Step5Config,
) -> Tuple[nn.Module, pd.DataFrame]:
    seed_all(seed)
    gcfg = cfg.gen_train
    device = device_for_training()
    model = GRULanguageModel(tokenizer.vocab_size, gcfg.emb_dim, gcfg.hidden_dim, gcfg.num_layers, gcfg.dropout, cond_vocab_size=cond_vocab_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=gcfg.lr, weight_decay=gcfg.weight_decay)
    epochs = gcfg.epochs_fast if cfg.fast_mode else gcfg.epochs_full

    train_ds = SequenceDataset(train_sequences, tokenizer, gcfg.max_seq_len, train_conditions)
    val_ds = SequenceDataset(val_sequences, tokenizer, gcfg.max_seq_len, val_conditions)
    train_loader = make_dataloader(train_ds, gcfg.batch_size, True, gcfg.num_workers, seed)
    val_loader = make_dataloader(val_ds, gcfg.batch_size, False, gcfg.num_workers, seed + 1)

    best_state = None
    best_val = float("inf")
    bad = 0
    logs = []
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            x = batch["input_ids"].to(device)
            y = batch["target_ids"].to(device)
            c = batch.get("condition_id")
            c = c.to(device) if c is not None else None
            logits = model(x, c)
            loss = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gcfg.clip_grad_norm)
            opt.step()
            train_losses.append(float(loss.item()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_ids"].to(device)
                y = batch["target_ids"].to(device)
                c = batch.get("condition_id")
                c = c.to(device) if c is not None else None
                logits = model(x, c)
                val_losses.append(float(cross_entropy_ignore_pad(logits, y, tokenizer.pad_id).item()))
        train_loss = float(np.mean(train_losses)) if train_losses else np.nan
        val_loss = float(np.mean(val_losses)) if val_losses else np.inf
        logs.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= gcfg.early_stopping_patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(logs)


def train_vae_model(train_sequences: Sequence[str], val_sequences: Sequence[str], tokenizer: PeptideTokenizer, seed: int, cfg: Step5Config) -> Tuple[nn.Module, pd.DataFrame]:
    seed_all(seed)
    gcfg = cfg.gen_train
    device = device_for_training()
    model = SequenceVAE(tokenizer.vocab_size, gcfg.emb_dim, gcfg.hidden_dim, gcfg.latent_dim, gcfg.num_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=gcfg.lr, weight_decay=gcfg.weight_decay)
    epochs = gcfg.epochs_fast if cfg.fast_mode else gcfg.epochs_full

    train_ds = SequenceDataset(train_sequences, tokenizer, gcfg.max_seq_len)
    val_ds = SequenceDataset(val_sequences, tokenizer, gcfg.max_seq_len)
    train_loader = make_dataloader(train_ds, gcfg.batch_size, True, gcfg.num_workers, seed)
    val_loader = make_dataloader(val_ds, gcfg.batch_size, False, gcfg.num_workers, seed + 1)

    best_state = None
    best_val = float("inf")
    bad = 0
    logs = []
    for epoch in range(epochs):
        beta = min(1.0, (epoch + 1) / max(1, epochs / 2))
        model.train()
        train_total = []
        train_rec = []
        train_kl = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            x = batch["input_ids"].to(device)
            y = batch["target_ids"].to(device)
            logits, mu, logvar = model(x)
            rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = rec + beta * kl
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gcfg.clip_grad_norm)
            opt.step()
            train_total.append(float(loss.item()))
            train_rec.append(float(rec.item()))
            train_kl.append(float(kl.item()))

        model.eval()
        val_total = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_ids"].to(device)
                y = batch["target_ids"].to(device)
                logits, mu, logvar = model(x)
                rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                val_total.append(float((rec + kl).item()))
        train_loss = float(np.mean(train_total)) if train_total else np.nan
        val_loss = float(np.mean(val_total)) if val_total else np.inf
        logs.append(
            {
                "epoch": epoch + 1,
                "beta": beta,
                "train_loss": train_loss,
                "train_reconstruction": float(np.mean(train_rec)) if train_rec else np.nan,
                "train_kl": float(np.mean(train_kl)) if train_kl else np.nan,
                "val_loss": val_loss,
            }
        )
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= gcfg.early_stopping_patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(logs)


def train_cvae_model(
    train_sequences: Sequence[str],
    val_sequences: Sequence[str],
    tokenizer: PeptideTokenizer,
    train_conditions: Sequence[int],
    val_conditions: Sequence[int],
    cond_vocab_size: int,
    seed: int,
    cfg: Step5Config,
) -> Tuple[nn.Module, pd.DataFrame]:
    seed_all(seed)
    gcfg = cfg.gen_train
    device = device_for_training()
    model = ConditionalSequenceVAE(tokenizer.vocab_size, cond_vocab_size, gcfg.emb_dim, gcfg.hidden_dim, gcfg.latent_dim, gcfg.num_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=gcfg.lr, weight_decay=gcfg.weight_decay)
    epochs = gcfg.epochs_fast if cfg.fast_mode else gcfg.epochs_full

    train_ds = SequenceDataset(train_sequences, tokenizer, gcfg.max_seq_len, train_conditions)
    val_ds = SequenceDataset(val_sequences, tokenizer, gcfg.max_seq_len, val_conditions)
    train_loader = make_dataloader(train_ds, gcfg.batch_size, True, gcfg.num_workers, seed)
    val_loader = make_dataloader(val_ds, gcfg.batch_size, False, gcfg.num_workers, seed + 1)

    best_state = None
    best_val = float("inf")
    bad = 0
    logs = []
    for epoch in range(epochs):
        beta = min(1.0, (epoch + 1) / max(1, epochs / 2))
        model.train()
        train_total = []
        train_rec = []
        train_kl = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            x = batch["input_ids"].to(device)
            y = batch["target_ids"].to(device)
            c = batch["condition_id"].to(device)
            logits, mu, logvar = model(x, c)
            rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = rec + beta * kl
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gcfg.clip_grad_norm)
            opt.step()
            train_total.append(float(loss.item()))
            train_rec.append(float(rec.item()))
            train_kl.append(float(kl.item()))

        model.eval()
        val_total = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_ids"].to(device)
                y = batch["target_ids"].to(device)
                c = batch["condition_id"].to(device)
                logits, mu, logvar = model(x, c)
                rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                val_total.append(float((rec + kl).item()))
        train_loss = float(np.mean(train_total)) if train_total else np.nan
        val_loss = float(np.mean(val_total)) if val_total else np.inf
        logs.append(
            {
                "epoch": epoch + 1,
                "beta": beta,
                "train_loss": train_loss,
                "train_reconstruction": float(np.mean(train_rec)) if train_rec else np.nan,
                "train_kl": float(np.mean(train_kl)) if train_kl else np.nan,
                "val_loss": val_loss,
            }
        )
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= gcfg.early_stopping_patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(logs)


def sample_from_logits(logits: torch.Tensor, temperature: float = 1.0) -> int:
    probs = torch.softmax(logits / max(temperature, 1e-6), dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())


def generate_gru_sequences(
    model: GRULanguageModel,
    tokenizer: PeptideTokenizer,
    n: int,
    max_len: int,
    seed: int,
    temperature: float = 1.0,
    condition_id: Optional[int] = None,
) -> List[str]:
    seed_all(seed)
    device = device_for_training()
    model = model.to(device)
    model.eval()
    outputs = []
    with torch.no_grad():
        for _ in range(n):
            cur = [tokenizer.bos_id]
            hidden = None
            if model.cond_emb is not None and condition_id is not None:
                hidden = model.cond_emb(torch.tensor([condition_id], device=device)).unsqueeze(0)
            for _ in range(max_len - 1):
                x = torch.tensor([cur[-1:]], dtype=torch.long, device=device)
                emb = model.emb(x)
                out, hidden = model.rnn(emb, hidden)
                logits = model.out(out[:, -1, :]).squeeze(0)
                nxt = sample_from_logits(logits, temperature=temperature)
                if nxt == tokenizer.eos_id:
                    break
                if nxt == tokenizer.pad_id:
                    continue
                cur.append(nxt)
            outputs.append(clean_sequence(tokenizer.decode(cur[1:])))
    return outputs


def generate_vae_sequences(model: SequenceVAE, tokenizer: PeptideTokenizer, n: int, max_len: int, seed: int, temperature: float = 1.0) -> List[str]:
    seed_all(seed)
    device = device_for_training()
    model = model.to(device)
    model.eval()
    outputs = []
    with torch.no_grad():
        for _ in range(n):
            z = torch.randn(1, model.latent_dim, device=device)
            hidden = model.z_to_hidden(z).unsqueeze(0)
            cur_token = tokenizer.bos_id
            cur = []
            for _ in range(max_len - 1):
                x = torch.tensor([[cur_token]], dtype=torch.long, device=device)
                emb = model.emb(x)
                out, hidden = model.decoder(emb, hidden)
                logits = model.out(out[:, -1, :]).squeeze(0)
                nxt = sample_from_logits(logits, temperature=temperature)
                if nxt == tokenizer.eos_id:
                    break
                if nxt == tokenizer.pad_id:
                    continue
                cur.append(nxt)
                cur_token = nxt
            outputs.append(clean_sequence(tokenizer.decode(cur)))
    return outputs


def generate_cvae_sequences(
    model: ConditionalSequenceVAE,
    tokenizer: PeptideTokenizer,
    n: int,
    max_len: int,
    seed: int,
    condition_id: int,
    temperature: float = 1.0,
) -> List[str]:
    seed_all(seed)
    device = device_for_training()
    model = model.to(device)
    model.eval()
    outputs = []
    cond = torch.tensor([condition_id], device=device)
    with torch.no_grad():
        for _ in range(n):
            z = torch.randn(1, model.latent_dim, device=device)
            hidden = (model.z_to_hidden(z) + model.cond_emb_dec(cond)).unsqueeze(0)
            cur_token = tokenizer.bos_id
            cur = []
            for _ in range(max_len - 1):
                x = torch.tensor([[cur_token]], dtype=torch.long, device=device)
                emb = model.emb(x)
                out, hidden = model.decoder(emb, hidden)
                logits = model.out(out[:, -1, :]).squeeze(0)
                nxt = sample_from_logits(logits, temperature=temperature)
                if nxt == tokenizer.eos_id:
                    break
                if nxt == tokenizer.pad_id:
                    continue
                cur.append(nxt)
                cur_token = nxt
            outputs.append(clean_sequence(tokenizer.decode(cur)))
    return outputs


def build_random_conditional_generator(train_df: pd.DataFrame, cfg: Step5Config) -> Dict[str, Any]:
    grouped = {}
    for cond, g in train_df.groupby(cfg.target_col):
        seqs = [clean_sequence(s) for s in g[cfg.sequence_col].astype(str) if is_valid_peptide(clean_sequence(s))]
        lengths = [len(s) for s in seqs if len(s) > 0]
        aa_counts = Counter("".join(seqs))
        probs = np.array([aa_counts.get(a, 1) for a in AA_ALPHABET], dtype=float)
        probs = probs / probs.sum()
        grouped[str(cond)] = {"lengths": lengths, "aa_probs": probs.tolist()}
    return grouped


def generate_random_conditional(random_model: Dict[str, Any], condition: str, n: int, seed: int) -> List[str]:
    rng = np.random.default_rng(seed)
    spec = random_model[str(condition)]
    lengths = np.asarray(spec["lengths"], dtype=int)
    probs = np.asarray(spec["aa_probs"], dtype=float)
    out = []
    for _ in range(n):
        L = int(rng.choice(lengths)) if len(lengths) > 0 else int(rng.integers(8, 30))
        seq = "".join(rng.choice(np.array(AA_ALPHABET), size=L, p=probs))
        out.append(seq)
    return out


def build_retrieval_reference(train_df: pd.DataFrame, cfg: Step5Config) -> Dict[str, List[str]]:
    ref = {}
    for cond, g in train_df.groupby(cfg.target_col):
        ref[str(cond)] = [clean_sequence(s) for s in g[cfg.sequence_col].astype(str) if is_valid_peptide(clean_sequence(s))]
    return ref


def generate_retrieval_samples(retrieval_model: Dict[str, List[str]], condition: str, n: int, seed: int) -> List[str]:
    rng = np.random.default_rng(seed)
    pool = retrieval_model[str(condition)]
    if len(pool) == 0:
        return []
    idx = rng.integers(0, len(pool), size=n)
    return [pool[i] for i in idx]


def kmer_set(seq: str, k: int) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}


def jaccard_similarity(a: str, b: str, k: int) -> float:
    sa, sb = kmer_set(a, k), kmer_set(b, k)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def nearest_neighbor_similarity(seq: str, reference: Sequence[str], k: int) -> float:
    if len(reference) == 0:
        return 0.0
    return max(jaccard_similarity(seq, r, k) for r in reference)


def pairwise_diversity(sequences: Sequence[str], k: int, max_pairs: int, seed: int) -> float:
    seqs = [s for s in sequences if is_valid_peptide(s)]
    n = len(seqs)
    if n < 2:
        return 0.0
    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2
    if total_pairs <= max_pairs:
        idx_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    else:
        idx_pairs = []
        seen = set()
        while len(idx_pairs) < max_pairs:
            i, j = sorted(rng.choice(n, size=2, replace=False).tolist())
            if (i, j) not in seen:
                seen.add((i, j))
                idx_pairs.append((i, j))
    sims = [jaccard_similarity(seqs[i], seqs[j], k) for i, j in idx_pairs]
    return float(1.0 - np.mean(sims))


def build_descriptor_reference(train_sequences: Sequence[str]) -> Dict[str, Any]:
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in train_sequences])
    mu = desc_df.mean(axis=0)
    sd = desc_df.std(axis=0).replace(0, 1.0)
    return {"mean": mu, "std": sd, "columns": list(desc_df.columns)}


def support_rate(sequences: Sequence[str], ref: Dict[str, Any], z_threshold: float = 3.0) -> float:
    if len(sequences) == 0:
        return 0.0
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in sequences])
    desc_df = desc_df[ref["columns"]]
    z = (desc_df - ref["mean"]) / ref["std"]
    inside = (np.abs(z) <= z_threshold).all(axis=1)
    return float(np.mean(inside))


def bootstrap_ci(values: Sequence[float], n_boot: int, seed: int) -> Tuple[float, float, float]:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0]), float(vals[0])
    rng = np.random.default_rng(seed)
    hist = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(vals), size=len(vals))
        hist.append(np.mean(vals[idx]))
    hist = np.asarray(hist, dtype=float)
    return float(np.mean(vals)), float(np.percentile(hist, 2.5)), float(np.percentile(hist, 97.5))


def evaluate_generated_set(
    generated_df: pd.DataFrame,
    surrogate_model: Pipeline,
    surrogate_features: Sequence[str],
    label_encoder: LabelEncoder,
    train_sequences: Sequence[str],
    full_sequences: Sequence[str],
    train_by_condition: Dict[str, List[str]],
    descriptor_ref: Dict[str, Any],
    cfg: Step5Config,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    records = []
    candidate_rows = []
    train_set = set(train_sequences)
    full_set = set(full_sequences)

    for (generator, seed, requested_condition), g in generated_df.groupby(["generator", "seed", "requested_condition"]):
        requested_condition = str(requested_condition)
        raw_seqs = [clean_sequence(s) for s in g["generated_sequence"].astype(str)]
        valid_mask = [is_valid_peptide(s) for s in raw_seqs]
        valid_seqs = [s for s, ok in zip(raw_seqs, valid_mask) if ok]
        seen = set()
        unique_valid = []
        duplicate_flags = []
        for s in valid_seqs:
            duplicate = s in seen
            duplicate_flags.append(duplicate)
            if not duplicate:
                seen.add(s)
                unique_valid.append(s)

        desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in valid_seqs]) if valid_seqs else pd.DataFrame(columns=surrogate_features)
        if len(desc_df) > 0:
            desc_df = desc_df.reindex(columns=surrogate_features, fill_value=0.0)
            probas = surrogate_model.predict_proba(desc_df)
            pred_idx = np.argmax(probas, axis=1)
            pred_labels = label_encoder.inverse_transform(pred_idx)
            target_label_index = np.where(label_encoder.classes_ == requested_condition)[0]
            target_probs = probas[:, target_label_index[0]] if len(target_label_index) > 0 else np.zeros(len(valid_seqs), dtype=float)
        else:
            pred_labels = np.array([], dtype=object)
            target_probs = np.array([], dtype=float)

        train_same_cond_set = set(train_by_condition.get(requested_condition, []))
        nn_train = [nearest_neighbor_similarity(s, train_sequences, cfg.similarity_kmer) for s in valid_seqs]
        nn_train_same_cond = [nearest_neighbor_similarity(s, train_by_condition.get(requested_condition, []), cfg.similarity_kmer) for s in valid_seqs]
        exact_novel_train = [s not in train_set for s in valid_seqs]
        exact_novel_full = [s not in full_set for s in valid_seqs]
        exact_novel_same_cond_train = [s not in train_same_cond_set for s in valid_seqs]
        low_complex_flags = [is_low_complexity(s, cfg.low_complexity_threshold) for s in valid_seqs]

        desc_cache = [compute_sequence_descriptors(s) for s in valid_seqs]
        for i, s in enumerate(valid_seqs):
            d = desc_cache[i]
            candidate_rows.append(
                {
                    "generator": generator,
                    "seed": int(seed),
                    "requested_condition": requested_condition,
                    "generated_sequence": s,
                    "valid_flag": 1,
                    "duplicate_flag": int(duplicate_flags[i]),
                    "low_complexity_flag": int(low_complex_flags[i]),
                    "length": d["length"],
                    "net_charge_proxy": d["net_charge_proxy"],
                    "mean_hydropathy": d["mean_hydropathy"],
                    "sequence_entropy": d["sequence_entropy"],
                    "exact_novel_train": int(exact_novel_train[i]),
                    "exact_novel_full": int(exact_novel_full[i]),
                    "exact_novel_same_condition_train": int(exact_novel_same_cond_train[i]),
                    "nn_similarity_to_train": float(nn_train[i]),
                    "nn_similarity_to_requested_condition_train": float(nn_train_same_cond[i]),
                    "surrogate_predicted_condition": pred_labels[i] if len(pred_labels) > 0 else np.nan,
                    "surrogate_target_probability": float(target_probs[i]) if len(target_probs) > 0 else np.nan,
                }
            )

        metrics = {
            "generator": generator,
            "seed": int(seed),
            "requested_condition": requested_condition,
            "n_requested": int(len(raw_seqs)),
            "valid_rate": float(np.mean(valid_mask)) if len(valid_mask) else 0.0,
            "unique_rate_among_valid": float(len(unique_valid) / max(len(valid_seqs), 1)),
            "duplicate_rate_within_valid": 1.0 - float(len(unique_valid) / max(len(valid_seqs), 1)),
            "exact_novelty_vs_train": float(np.mean(exact_novel_train)) if len(exact_novel_train) else 0.0,
            "exact_novelty_vs_full": float(np.mean(exact_novel_full)) if len(exact_novel_full) else 0.0,
            "exact_novelty_vs_same_condition_train": float(np.mean(exact_novel_same_cond_train)) if len(exact_novel_same_cond_train) else 0.0,
            "mean_nn_similarity_to_train": float(np.mean(nn_train)) if nn_train else 0.0,
            "mean_nn_similarity_to_same_condition_train": float(np.mean(nn_train_same_cond)) if nn_train_same_cond else 0.0,
            "prop_nn_similarity_lt_080": float(np.mean(np.asarray(nn_train) < 0.80)) if nn_train else 0.0,
            "prop_nn_similarity_lt_070": float(np.mean(np.asarray(nn_train) < 0.70)) if nn_train else 0.0,
            "pairwise_diversity": pairwise_diversity(unique_valid, cfg.similarity_kmer, cfg.pairwise_diversity_sample_size, seed=seed),
            "support_rate": support_rate(unique_valid, descriptor_ref),
            "surrogate_condition_hit_rate": float(np.mean(pred_labels == requested_condition)) if len(pred_labels) else 0.0,
            "surrogate_target_probability": float(np.mean(target_probs)) if len(target_probs) else 0.0,
            "low_complexity_rate": float(np.mean(low_complex_flags)) if len(low_complex_flags) else 0.0,
            "mean_length": float(np.mean([d["length"] for d in desc_cache])) if desc_cache else 0.0,
            "mean_charge_proxy": float(np.mean([d["net_charge_proxy"] for d in desc_cache])) if desc_cache else 0.0,
            "mean_hydropathy": float(np.mean([d["mean_hydropathy"] for d in desc_cache])) if desc_cache else 0.0,
        }
        records.append(metrics)

    per_condition_df = pd.DataFrame(records)
    candidate_df = pd.DataFrame(candidate_rows)
    return per_condition_df, candidate_df


def summarize_generator_metrics(per_condition_df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    metric_cols = [
        c for c in per_condition_df.columns
        if c not in {"generator", "requested_condition", "seed", "n_requested"}
    ]
    rows = []
    for generator, g in per_condition_df.groupby("generator"):
        row = {"generator": generator, "generator_display": MODEL_DISPLAY_NAMES.get(generator, generator)}
        for c in metric_cols:
            stable_seed = cfg.main_random_seed + sum(ord(ch) for ch in f"{generator}|{c}")
            mean_val, ci_low, ci_high = bootstrap_ci(g[c].astype(float).values, cfg.bootstrap_iterations, stable_seed)
            row[c] = mean_val
            row[f"{c}_ci_low"] = ci_low
            row[f"{c}_ci_high"] = ci_high
            row[f"{c}_sd"] = float(np.std(g[c].astype(float).values, ddof=1)) if len(g) > 1 else 0.0
        row["n_generator_condition_runs"] = int(len(g))
        rows.append(row)
    summary = pd.DataFrame(rows).sort_values(["surrogate_condition_hit_rate", "exact_novelty_vs_train"], ascending=False).reset_index(drop=True)
    return summary


def run_step5b_generator_benchmark(data: Dict[str, pd.DataFrame], step5a: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Running Step 5B — baseline generator benchmark")
    seq_df = data["seq_df"].copy()
    seq_df[cfg.sequence_col] = seq_df[cfg.sequence_col].astype(str).map(clean_sequence)
    seq_df = seq_df[seq_df[cfg.sequence_col].map(is_valid_peptide)].reset_index(drop=True)

    train_df = seq_df[seq_df[cfg.split_col] == cfg.train_label].reset_index(drop=True)
    val_df = seq_df[seq_df[cfg.split_col] == cfg.val_label].reset_index(drop=True)
    _ = seq_df[seq_df[cfg.split_col] == cfg.test_label].reset_index(drop=True)

    tokenizer = PeptideTokenizer()
    cond_encoder = LabelEncoder()
    cond_train = cond_encoder.fit_transform(train_df[cfg.target_col].astype(str))
    cond_val = cond_encoder.transform(val_df[cfg.target_col].astype(str))
    unique_conditions = list(map(str, cond_encoder.classes_))

    save_json({"condition_classes": unique_conditions}, dirs["artifacts"] / "step5b_condition_classes.json")

    random_model = build_random_conditional_generator(train_df, cfg)
    retrieval_model = build_retrieval_reference(train_df, cfg)
    train_by_condition = {str(k): v for k, v in retrieval_model.items()}

    training_manifest_rows = []
    wrappers: List[Dict[str, Any]] = []
    for seed in cfg.repeated_seeds:
        print(f"Training seed {seed} baselines...")

        gru_u, gru_u_log = train_gru_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            None,
            None,
            0,
            seed,
            cfg,
        )
        torch.save(gru_u.state_dict(), dirs["models"] / f"gru_unconditional_seed{seed}.pt")
        gru_u_log.to_csv(dirs["logs"] / f"gru_unconditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "gru_unconditional", "kind": "gru_unconditional", "seed": seed, "model": gru_u})
        training_manifest_rows.append({"generator": "gru_unconditional", "seed": seed, "best_val_loss": float(gru_u_log["val_loss"].min())})

        vae_u, vae_u_log = train_vae_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            seed,
            cfg,
        )
        torch.save(vae_u.state_dict(), dirs["models"] / f"vae_unconditional_seed{seed}.pt")
        vae_u_log.to_csv(dirs["logs"] / f"vae_unconditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "vae_unconditional", "kind": "vae_unconditional", "seed": seed, "model": vae_u})
        training_manifest_rows.append({"generator": "vae_unconditional", "seed": seed, "best_val_loss": float(vae_u_log["val_loss"].min())})

        gru_c, gru_c_log = train_gru_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            cond_train,
            cond_val,
            len(unique_conditions),
            seed,
            cfg,
        )
        torch.save(gru_c.state_dict(), dirs["models"] / f"gru_conditional_seed{seed}.pt")
        gru_c_log.to_csv(dirs["logs"] / f"gru_conditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "gru_conditional", "kind": "gru_conditional", "seed": seed, "model": gru_c})
        training_manifest_rows.append({"generator": "gru_conditional", "seed": seed, "best_val_loss": float(gru_c_log["val_loss"].min())})

        cvae_c, cvae_c_log = train_cvae_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            cond_train,
            cond_val,
            len(unique_conditions),
            seed,
            cfg,
        )
        torch.save(cvae_c.state_dict(), dirs["models"] / f"cvae_conditional_seed{seed}.pt")
        cvae_c_log.to_csv(dirs["logs"] / f"cvae_conditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "cvae_conditional", "kind": "cvae_conditional", "seed": seed, "model": cvae_c})
        training_manifest_rows.append({"generator": "cvae_conditional", "seed": seed, "best_val_loss": float(cvae_c_log["val_loss"].min())})

        wrappers.append({"name": "random_conditional", "kind": "random_conditional", "seed": seed, "model": random_model})
        wrappers.append({"name": "retrieval_reference", "kind": "retrieval_reference", "seed": seed, "model": retrieval_model})

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    training_manifest_df = pd.DataFrame(training_manifest_rows)
    training_manifest_df.to_csv(dirs["tables_supplementary"] / "table_s5_3_generator_training_summary.csv", index=False)

    all_generated_rows = []
    budget = cfg.n_generated_per_condition
    for wrapper in wrappers:
        for cond in unique_conditions:
            if wrapper["kind"] == "random_conditional":
                seqs = generate_random_conditional(wrapper["model"], cond, budget, seed=wrapper["seed"] + 17)
            elif wrapper["kind"] == "retrieval_reference":
                seqs = generate_retrieval_samples(wrapper["model"], cond, budget, seed=wrapper["seed"] + 23)
            elif wrapper["kind"] == "gru_unconditional":
                seqs = generate_gru_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 29, temperature=cfg.gen_train.temperature)
            elif wrapper["kind"] == "vae_unconditional":
                seqs = generate_vae_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 31, temperature=cfg.gen_train.temperature)
            elif wrapper["kind"] == "gru_conditional":
                cond_id = int(cond_encoder.transform([cond])[0])
                seqs = generate_gru_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 37, temperature=cfg.gen_train.temperature, condition_id=cond_id)
            elif wrapper["kind"] == "cvae_conditional":
                cond_id = int(cond_encoder.transform([cond])[0])
                seqs = generate_cvae_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 41, condition_id=cond_id, temperature=cfg.gen_train.temperature)
            else:
                raise ValueError(f"Unsupported generator kind: {wrapper['kind']}")

            for s in seqs:
                all_generated_rows.append(
                    {
                        "generator": wrapper["name"],
                        "seed": int(wrapper["seed"]),
                        "requested_condition": str(cond),
                        "generated_sequence": clean_sequence(s),
                    }
                )

    generated_df = pd.DataFrame(all_generated_rows)
    generated_df.to_csv(dirs["samples"] / "step5b_generated_candidates_raw.csv", index=False)

    train_sequences = train_df[cfg.sequence_col].astype(str).map(clean_sequence).tolist()
    full_sequences = seq_df[cfg.sequence_col].astype(str).map(clean_sequence).tolist()
    descriptor_ref = build_descriptor_reference(train_sequences)

    per_condition_df, candidate_df = evaluate_generated_set(
        generated_df=generated_df,
        surrogate_model=step5a["surrogate_model"],
        surrogate_features=step5a["surrogate_features"],
        label_encoder=step5a["label_encoder"],
        train_sequences=train_sequences,
        full_sequences=full_sequences,
        train_by_condition=train_by_condition,
        descriptor_ref=descriptor_ref,
        cfg=cfg,
    )
    summary_df = summarize_generator_metrics(per_condition_df, cfg)

    per_condition_df.to_csv(dirs["tables_supplementary"] / "table_s5_4_step5b_per_condition_metrics.csv", index=False)
    candidate_df.to_csv(dirs["samples"] / "step5b_generated_candidates_evaluated.csv", index=False)
    summary_df.to_csv(dirs["tables_main"] / "table_5_2_step5b_generator_benchmark.csv", index=False)

    return {
        "generated_df": generated_df,
        "per_condition_df": per_condition_df,
        "candidate_df": candidate_df,
        "summary_df": summary_df,
        "condition_classes": unique_conditions,
        "training_manifest_df": training_manifest_df,
    }


def make_step5_main_figure(step5a: Dict[str, Any], step5b: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> None:
    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.6))
    gs = GridSpec(2, 2, figure=fig, hspace=0.50, wspace=0.42)

    # a) Descriptor recoverability audit
    ax = fig.add_subplot(gs[0, 0])
    df = step5a["summary_df"].copy()
    labels = list(df["block"]) + ["Shuffled-label control"]
    values = list(df["macro_f1"]) + [float(step5a["shuffle_df"]["macro_f1"].mean())]
    lows = list(df["macro_f1_ci_low"]) + [float(step5a["shuffle_df"]["macro_f1"].quantile(0.025))]
    highs = list(df["macro_f1_ci_high"]) + [float(step5a["shuffle_df"]["macro_f1"].quantile(0.975))]
    colors = [OKABE_ITO["blue"]] * len(df) + [OKABE_ITO["gray"]]
    _dot_whisker(ax, labels, values, lows, highs, colors, xlabel="Macro F1", title="Descriptor-based condition recoverability")
    ax.set_xlim(left=min(lows) - 0.03, right=min(1.02, max(highs) + 0.03))
    add_panel_label(ax, "a", cfg.figure_export.panel_fontsize)

    # b) Generator fidelity benchmark
    ax = fig.add_subplot(gs[0, 1])
    gdf = step5b["summary_df"].copy()
    order = _generator_order(gdf)
    gdf = gdf.set_index('generator').loc[order].reset_index()
    labels = [MODEL_DISPLAY_NAMES.get(g, g) for g in gdf['generator']]
    values = list(gdf['surrogate_condition_hit_rate'])
    lows = list(gdf['surrogate_condition_hit_rate_ci_low'])
    highs = list(gdf['surrogate_condition_hit_rate_ci_high'])
    colors = [MODEL_COLORS.get(g, OKABE_ITO['gray']) for g in gdf['generator']]
    _dot_whisker(ax, labels, values, lows, highs, colors, xlabel="Condition-hit rate", title="Baseline generator fidelity")
    ax.set_xlim(left=0.0, right=min(1.02, max(highs) + 0.05))
    add_panel_label(ax, "b", cfg.figure_export.panel_fontsize)

    # c) Fidelity–novelty tradeoff
    ax = fig.add_subplot(gs[1, 0])
    for _, row in gdf.iterrows():
        ax.scatter(
            row['exact_novelty_vs_train'],
            row['surrogate_condition_hit_rate'],
            s=58,
            color=MODEL_COLORS.get(row['generator'], OKABE_ITO['gray']),
            edgecolor='black',
            linewidth=0.4,
            zorder=3,
        )
        dx = -0.03 if row['exact_novelty_vs_train'] > 0.9 else 0.012
        dy = 0.008 if row['surrogate_condition_hit_rate'] < 0.4 else 0.0
        ax.text(
            row['exact_novelty_vs_train'] + dx,
            row['surrogate_condition_hit_rate'] + dy,
            MODEL_DISPLAY_NAMES.get(row['generator'], row['generator']),
            fontsize=6.3,
        )
    ax.set_xlabel("Exact novelty versus training set")
    ax.set_ylabel("Condition-hit rate")
    ax.set_title("Fidelity–novelty tradeoff")
    ax.set_xlim(-0.03, 1.03)
    ymax = max(0.5, float(gdf['surrogate_condition_hit_rate_ci_high'].max()) + 0.05)
    ax.set_ylim(0.0, ymax)
    ax.grid(alpha=0.25)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, "c", cfg.figure_export.panel_fontsize)

    # d) Similarity-to-train distribution
    ax = fig.add_subplot(gs[1, 1])
    per_cond = step5b['per_condition_df'].copy()
    order = _generator_order(step5b['summary_df'])
    data_plot = [per_cond.loc[per_cond['generator'] == g, 'mean_nn_similarity_to_train'].astype(float).values for g in order]
    positions = np.arange(1, len(order) + 1)
    parts = ax.violinplot(data_plot, positions=positions, showmeans=False, showmedians=True, showextrema=False)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(MODEL_COLORS.get(order[i], OKABE_ITO['gray']))
        body.set_edgecolor('black')
        body.set_alpha(0.60)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(1.0)
    for pos, vals in zip(positions, data_plot):
        if len(vals):
            ax.scatter(np.repeat(pos, len(vals)), vals, s=10, color='black', alpha=0.18, zorder=2)
    ax.set_xticks(positions)
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=20, ha='right')
    ax.set_ylabel("Nearest-neighbour similarity to training set")
    ax.set_title("Similarity-to-train profile")
    ax.grid(alpha=0.25, axis='y')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, "d", cfg.figure_export.panel_fontsize)

    export_figure(fig, dirs['figures_main'] / 'Fig_5_step5_baseline_benchmark_redesigned', cfg.figure_export)


def make_step5_supplementary_figures(step5a: Dict[str, Any], step5b: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> None:
    # Supplementary figure A: descriptor audit diagnostics
    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 7.2))
    gs = GridSpec(2, 2, figure=fig, hspace=0.48, wspace=0.40)

    ax = fig.add_subplot(gs[0, 0])
    split_counts = pd.concat([
        step5a['train_df_with_desc'][cfg.target_col].value_counts().rename('train'),
        step5a['val_df_with_desc'][cfg.target_col].value_counts().rename('val'),
        step5a['test_df_with_desc'][cfg.target_col].value_counts().rename('test'),
    ], axis=1).fillna(0)
    split_counts = split_counts.loc[split_counts.sum(axis=1).sort_values(ascending=False).index]
    sub = split_counts.head(cfg.max_unique_conditions_for_figures)
    width = 0.25
    x = np.arange(len(sub))
    ax.bar(x - width, sub['train'], width=width, color=OKABE_ITO['blue'], label='train')
    ax.bar(x, sub['val'], width=width, color=OKABE_ITO['orange'], label='val')
    ax.bar(x + width, sub['test'], width=width, color=OKABE_ITO['green'], label='test')
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index.astype(str), rotation=25, ha='right')
    ax.set_ylabel('Sequences')
    ax.set_title('Condition support across frozen splits')
    ax.legend(frameon=False)
    ax.grid(alpha=0.25, axis='y')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, 'a', cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[0, 1])
    cm_df = step5a['confusion_matrix_norm'].copy()
    if cm_df.shape[0] > 12:
        keep = list(cm_df.sum(axis=1).sort_values(ascending=False).head(12).index)
        cm_df = cm_df.loc[keep, keep]
    im = ax.imshow(cm_df.values, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(cm_df.columns)))
    ax.set_yticks(np.arange(len(cm_df.index)))
    ax.set_xticklabels([str(x) for x in cm_df.columns], rotation=45, ha='right')
    ax.set_yticklabels([str(x) for x in cm_df.index])
    ax.set_title('Normalized confusion matrix')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    add_panel_label(ax, 'b', cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    perm_df = step5a['perm_df']
    if perm_df is not None and len(perm_df) > 0:
        top = perm_df.head(10).copy().sort_values('importance_mean')
        ax.barh(np.arange(len(top)), top['importance_mean'], xerr=top['importance_std'], color=OKABE_ITO['orange'])
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels([_pretty_feature_name(x) for x in top['feature']])
        ax.set_xlabel('Permutation importance (macro F1)')
        ax.set_title('Top descriptor contributors')
        ax.grid(alpha=0.25, axis='x')
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
    else:
        ax.text(0.5, 0.5, 'Permutation importance unavailable', ha='center', va='center')
        ax.set_axis_off()
    add_panel_label(ax, 'c', cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 1])
    pca_df = step5a['pca_df'].copy()
    top_conditions = pca_df['condition'].value_counts().head(min(6, cfg.max_unique_conditions_for_figures)).index
    sub = pca_df[pca_df['condition'].isin(top_conditions)]
    for cond, g in sub.groupby('condition'):
        ax.scatter(g['pc1'], g['pc2'], s=12, alpha=0.55, label=str(cond))
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('Descriptor-space projection of test peptides')
    ax.legend(frameon=False, fontsize=6, ncol=2)
    ax.grid(alpha=0.25)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, 'd', cfg.figure_export.panel_fontsize)

    export_figure(fig, dirs['figures_supplementary'] / 'Fig_S5A_step5a_diagnostics_redesigned', cfg.figure_export)

    # Supplementary figure B: generator robustness and QC
    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 7.2))
    gs = GridSpec(2, 2, figure=fig, hspace=0.48, wspace=0.40)
    summary = step5b['summary_df'].copy().set_index('generator')
    order = _generator_order(step5b['summary_df'])
    summary = summary.loc[order].reset_index()
    per_cond = step5b['per_condition_df'].copy()
    candidate = step5b['candidate_df'].copy()

    ax = fig.add_subplot(gs[0, 0])
    metrics = ['valid_rate', 'exact_novelty_vs_train', 'unique_rate_among_valid']
    display_metrics = ['Validity', 'Novelty', 'Uniqueness']
    x = np.arange(len(order))
    width = 0.22
    for i, (metric, metric_name) in enumerate(zip(metrics, display_metrics)):
        vals = [summary.loc[summary['generator'] == g, metric].iloc[0] for g in order]
        ax.bar(x + (i - 1) * width, vals, width=width, label=metric_name)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=20, ha='right')
    ax.set_ylim(0.0, 1.05)
    ax.set_title('Validity, novelty and uniqueness')
    ax.legend(frameon=False)
    ax.grid(alpha=0.25, axis='y')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, 'a', cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[0, 1])
    data_plot = [candidate.loc[candidate['generator'] == g, 'nn_similarity_to_train'].astype(float).values for g in order]
    positions = np.arange(1, len(order) + 1)
    parts = ax.violinplot(data_plot, positions=positions, showmeans=False, showmedians=True, showextrema=False)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(MODEL_COLORS.get(order[i], OKABE_ITO['gray']))
        body.set_edgecolor('black')
        body.set_alpha(0.60)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(1.0)
    ax.set_xticks(positions)
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=20, ha='right')
    ax.set_ylabel('Per-sequence NN similarity to training set')
    ax.set_title('Per-sequence memorization profile')
    ax.grid(alpha=0.25, axis='y')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, 'b', cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    for i, g in enumerate(order):
        vals = per_cond.loc[per_cond['generator'] == g, 'surrogate_condition_hit_rate'].astype(float).values
        ax.scatter(np.repeat(i, len(vals)), vals, color=MODEL_COLORS.get(g, OKABE_ITO['gray']), s=16, alpha=0.65)
        if len(vals):
            ax.hlines(np.mean(vals), i - 0.23, i + 0.23, colors='black', linewidth=1.1)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=20, ha='right')
    ax.set_ylabel('Per-condition condition-hit rate')
    ax.set_title('Condition-level heterogeneity')
    ax.grid(alpha=0.25, axis='y')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    add_panel_label(ax, 'c', cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 1])
    var_cols = ['surrogate_condition_hit_rate_sd', 'exact_novelty_vs_train_sd', 'pairwise_diversity_sd']
    heat = summary[['generator'] + var_cols].set_index('generator')
    im = ax.imshow(heat.values, aspect='auto', cmap='Oranges')
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in heat.index])
    ax.set_xticks(np.arange(len(var_cols)))
    ax.set_xticklabels(['Fidelity SD', 'Novelty SD', 'Diversity SD'], rotation=20, ha='right')
    ax.set_title('Seed-to-seed variability')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    add_panel_label(ax, 'd', cfg.figure_export.panel_fontsize)

    export_figure(fig, dirs['figures_supplementary'] / 'Fig_S5B_step5b_diagnostics_redesigned', cfg.figure_export)


def write_step5_report(step5a: Dict[str, Any], step5b: Dict[str, Any], qc: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> None:
    a = step5a["summary_df"].copy()
    b = step5b["summary_df"].copy().sort_values(["surrogate_condition_hit_rate", "exact_novelty_vs_train"], ascending=False).reset_index(drop=True)
    best_a = a.iloc[0]
    best_b = b.iloc[0]

    lines = []
    lines.append("Step 5 report — descriptor audit and baseline generator benchmark\n\n")
    lines.append("Input and contract audit\n")
    lines.append(f"Timestamp: {now_timestamp()}\n")
    lines.append(f"Total Step 4 rows: {qc['n_total_rows']}\n")
    lines.append(f"Valid cleaned sequences: {qc['n_valid_clean_sequences']}\n")
    lines.append(f"Invalid after cleaning: {qc['n_invalid_after_cleaning']}\n\n")

    lines.append("Step 5A — descriptor audit\n")
    lines.append(f"Best descriptor block: {best_a['block']} using {best_a['family']}\n")
    lines.append(f"Locked-test macro F1: {best_a['macro_f1']:.3f} [{best_a['macro_f1_ci_low']:.3f}, {best_a['macro_f1_ci_high']:.3f}]\n")
    lines.append(f"Locked-test balanced accuracy: {best_a['balanced_acc']:.3f}\n")
    lines.append(f"Mean shuffled-label macro F1: {step5a['shuffle_df']['macro_f1'].mean():.3f}\n\n")

    lines.append("Step 5B — generator benchmark\n")
    lines.append(f"Top generator by surrogate fidelity: {MODEL_DISPLAY_NAMES.get(best_b['generator'], best_b['generator'])}\n")
    lines.append(f"Surrogate condition-hit rate: {best_b['surrogate_condition_hit_rate']:.3f} [{best_b['surrogate_condition_hit_rate_ci_low']:.3f}, {best_b['surrogate_condition_hit_rate_ci_high']:.3f}]\n")
    lines.append(f"Exact novelty vs train: {best_b['exact_novelty_vs_train']:.3f} [{best_b['exact_novelty_vs_train_ci_low']:.3f}, {best_b['exact_novelty_vs_train_ci_high']:.3f}]\n")
    lines.append(f"Pairwise diversity: {best_b['pairwise_diversity']:.3f} [{best_b['pairwise_diversity_ci_low']:.3f}, {best_b['pairwise_diversity_ci_high']:.3f}]\n")
    lines.append(f"Support rate: {best_b['support_rate']:.3f} [{best_b['support_rate_ci_low']:.3f}, {best_b['support_rate_ci_high']:.3f}]\n")
    lines.append(f"Low-complexity rate: {best_b['low_complexity_rate']:.3f} [{best_b['low_complexity_rate_ci_low']:.3f}, {best_b['low_complexity_rate_ci_high']:.3f}]\n")

    (dirs["reports"] / "step5_summary_report.txt").write_text("".join(lines), encoding="utf-8")

    methods = (
        "Step 5 comprised a descriptor-based audit benchmark (Step 5A) and a frozen-split baseline generator benchmark (Step 5B). "
        "In Step 5A, we benchmarked multiclass predictors of primary_condition_id from peptide descriptors using the fixed Step 4 train/validation/test partition, "
        "with feature blocks spanning the full numeric descriptor space, a computable descriptor subset, amino-acid composition alone, and physicochemical descriptors alone. "
        "Candidate surrogate families included a majority dummy baseline, multinomial logistic regression, random forest, and extra-trees classifiers. "
        "Model-family selection used train-only inner cross-validation and validation macro F1, followed by refitting on train+validation and locked-test evaluation with bootstrap confidence intervals. "
        "We additionally ran repeated shuffled-label controls, exported normalized confusion matrices, estimated permutation importance, and projected the computable descriptor space by PCA. "
        "In Step 5B, we benchmarked same-condition retrieval reference sampling, a random conditional baseline, an unconditional GRU language model, an unconditional sequence VAE, a conditional GRU, and a conditional VAE, all trained on the train split only and evaluated under matched generation budgets across requested conditions and repeated random seeds. "
        "Generated peptides were assessed using surrogate condition-hit rate and surrogate target probability as proxy controllability metrics, alongside exact novelty versus the training split and full corpus, nearest-neighbour 3-mer Jaccard similarity to the train split and same-condition train subset, pairwise diversity, within-set duplication, descriptor-space support, low-complexity rate, and basic physicochemical summary statistics."
    )
    (dirs["reports"] / "step5_methods_paragraph.txt").write_text(methods, encoding="utf-8")


def run_step5_pipeline(cfg: Step5Config) -> Dict[str, Any]:
    seed_all(cfg.main_random_seed)
    project_root = project_root_from_input(cfg.input_model_ready_csv)
    out_root = project_root / cfg.out_dir_name
    dirs = make_output_dirs(out_root)

    save_json(asdict(cfg), dirs["artifacts"] / "step5_config.json")
    save_json(environment_manifest(), dirs["artifacts"] / "environment_manifest.json")
    save_json(
        {
            "input_model_ready_csv": cfg.input_model_ready_csv,
            "input_baseline_features_csv": cfg.input_baseline_features_csv,
            "sha256_model_ready": sha256_of_file(Path(cfg.input_model_ready_csv)),
            "sha256_baseline_features": sha256_of_file(Path(cfg.input_baseline_features_csv)),
        },
        dirs["artifacts"] / "input_manifest.json",
    )

    data = load_step4_inputs(cfg)
    qc = validate_step4_contract(data["seq_df"], data["feat_df"], cfg)
    save_json(qc, dirs["artifacts"] / "step4_contract_qc.json")

    step5a = run_step5a_descriptor_audit(data, cfg, dirs)
    step5b = run_step5b_generator_benchmark(data, step5a, cfg, dirs)

    make_step5_main_figure(step5a, step5b, cfg, dirs)
    make_step5_supplementary_figures(step5a, step5b, cfg, dirs)
    write_step5_report(step5a, step5b, qc, cfg, dirs)

    artifact_df = manifest_from_outputs(out_root)
    artifact_df.to_csv(dirs["artifacts"] / "artifact_manifest.csv", index=False)

    return {
        "project_root": str(project_root),
        "out_root": str(out_root),
        "step5a_summary": step5a["summary_df"],
        "step5b_summary": step5b["summary_df"],
        "artifact_manifest": artifact_df,
    }


def main_notebook(
    input_model_ready_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv",
    input_baseline_features_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv",
    out_dir_name: str = "step5",
    fast_mode: bool = False,
    n_generated_per_condition: int = 100,
    repeated_seeds: Tuple[int, ...] = (42, 52, 62),
) -> Dict[str, Any]:
    cfg = Step5Config(
        input_model_ready_csv=input_model_ready_csv,
        input_baseline_features_csv=input_baseline_features_csv,
        out_dir_name=out_dir_name,
        fast_mode=fast_mode,
        n_generated_per_condition=n_generated_per_condition,
        repeated_seeds=repeated_seeds,
    )
    return run_step5_pipeline(cfg)


if __name__ == "__main__":
    results = main_notebook()
    print("Step 5 completed.")
    print(f"Output directory: {results['out_root']}")


In [ ]:
from __future__ import annotations

# ======================================================================================
# Step 5 — Publication-grade baseline benchmark notebook code
# Single Jupyter-ready Python script
# ======================================================================================

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import gc
import sys
import json
import time
import hashlib
import random
import platform
import warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


plt.rcParams.update(
    {
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.transparent": False,
    }
)

OKABE_ITO = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "yellow": "#F0E442",
    "black": "#000000",
    "gray": "#7F7F7F",
    "light_gray": "#D9D9D9",
}

MODEL_DISPLAY_NAMES = {
    "random_conditional": "Random conditional",
    "retrieval_reference": "Retrieval reference",
    "gru_unconditional": "GRU (unconditional)",
    "vae_unconditional": "VAE (unconditional)",
    "gru_conditional": "GRU (conditional)",
    "cvae_conditional": "CVAE (conditional)",
}

MODEL_COLORS = {
    "random_conditional": OKABE_ITO["gray"],
    "retrieval_reference": OKABE_ITO["sky"],
    "gru_unconditional": OKABE_ITO["blue"],
    "vae_unconditional": OKABE_ITO["orange"],
    "gru_conditional": OKABE_ITO["purple"],
    "cvae_conditional": OKABE_ITO["green"],
}

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)
AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_TINY = set("AGSC")
AA_SMALL = set("AGSCVTNDP")
AA_PROPERTIES = {
    "A": {"hydro": 1.8}, "C": {"hydro": 2.5}, "D": {"hydro": -3.5}, "E": {"hydro": -3.5},
    "F": {"hydro": 2.8}, "G": {"hydro": -0.4}, "H": {"hydro": -3.2}, "I": {"hydro": 4.5},
    "K": {"hydro": -3.9}, "L": {"hydro": 3.8}, "M": {"hydro": 1.9}, "N": {"hydro": -3.5},
    "P": {"hydro": -1.6}, "Q": {"hydro": -3.5}, "R": {"hydro": -4.5}, "S": {"hydro": -0.8},
    "T": {"hydro": -0.7}, "V": {"hydro": 4.2}, "W": {"hydro": -0.9}, "Y": {"hydro": -1.3},
}


@dataclass
class FigureExportConfig:
    pdf: bool = True
    png: bool = True
    tif: bool = False
    png_dpi: int = 300
    tif_dpi: int = 600
    single_col_width_in: float = 3.45
    double_col_width_in: float = 7.10
    base_fontsize: int = 8
    panel_fontsize: int = 10


@dataclass
class GeneratorTrainConfig:
    max_seq_len: int = 64
    batch_size: int = 64
    num_workers: int = 0
    emb_dim: int = 64
    hidden_dim: int = 128
    latent_dim: int = 32
    num_layers: int = 1
    dropout: float = 0.10
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs_fast: int = 5
    epochs_full: int = 20
    clip_grad_norm: float = 1.0
    early_stopping_patience: int = 4
    temperature: float = 1.0


@dataclass
class Step5Config:
    input_model_ready_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv"
    input_baseline_features_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv"
    out_dir_name: str = "step5"

    sequence_col: str = "sequence"
    merge_key: str = "sequence_sha256"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    main_random_seed: int = 42
    repeated_seeds: Tuple[int, ...] = (42, 52, 62)
    inner_cv_folds: int = 5
    bootstrap_iterations: int = 400
    permutation_repeats: int = 20
    topk_values: Tuple[int, ...] = (1, 3)
    shuffle_repeats: int = 10

    fast_mode: bool = False
    n_generated_per_condition: int = 100
    similarity_kmer: int = 3
    pairwise_diversity_sample_size: int = 250
    max_unique_conditions_for_figures: int = 12
    min_samples_per_class_for_cv: int = 5
    low_complexity_threshold: float = 0.70

    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)
    gen_train: GeneratorTrainConfig = field(default_factory=GeneratorTrainConfig)


def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)


def seed_all(seed: int, deterministic_torch: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if deterministic_torch:
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


def device_for_training() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def now_timestamp() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)


def add_panel_label(ax: plt.Axes, label: str, fontsize: int = 10) -> None:
    ax.text(-0.14, 1.04, label, transform=ax.transAxes, fontsize=fontsize, fontweight="bold", va="bottom")


def project_root_from_input(input_csv: str) -> Path:
    p = Path(input_csv).resolve()
    for parent in p.parents:
        if parent.name.startswith("nature"):
            return parent
    return p.parent.parent if p.parent.name == "tables" else p.parent


def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "models": base_dir / "models",
        "samples": base_dir / "samples",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp": now_timestamp(),
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "torch": torch.__version__,
        "sklearn": __import__("sklearn").__version__,
        "cuda_available": torch.cuda.is_available(),
        "device": str(device_for_training()),
    }


def manifest_from_outputs(base_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append(
                {
                    "relative_path": str(path.relative_to(base_dir)),
                    "size_bytes": path.stat().st_size,
                    "sha256": sha256_of_file(path),
                }
            )
    return pd.DataFrame(rows)


def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join([c for c in seq if c in AA_SET])


def is_valid_peptide(seq: str) -> bool:
    return len(seq) > 0 and all(c in AA_SET for c in seq)


def aa_fraction(seq: str, aa: str) -> float:
    if len(seq) == 0:
        return 0.0
    return seq.count(aa) / len(seq)


def shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return 0.0
    counts = np.array([seq.count(a) for a in AA_ALPHABET], dtype=float)
    probs = counts[counts > 0] / counts.sum()
    return float(-(probs * np.log2(probs)).sum())


def max_residue_fraction(seq: str) -> float:
    if len(seq) == 0:
        return 0.0
    counts = Counter(seq)
    return max(counts.values()) / len(seq)


def is_low_complexity(seq: str, threshold: float = 0.70) -> bool:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return True
    return max_residue_fraction(seq) >= threshold


def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    n = len(seq)
    base = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
    if n == 0:
        base.update(
            {
                "length": 0.0,
                "hydrophobic_frac": 0.0,
                "aromatic_frac": 0.0,
                "positive_frac": 0.0,
                "negative_frac": 0.0,
                "polar_frac": 0.0,
                "tiny_frac": 0.0,
                "small_frac": 0.0,
                "net_charge_proxy": 0.0,
                "mean_hydropathy": 0.0,
                "sequence_entropy": 0.0,
                "max_residue_fraction": 0.0,
            }
        )
        return base

    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    hydros = [AA_PROPERTIES[a]["hydro"] for a in seq]
    desc.update(
        {
            "length": float(n),
            "hydrophobic_frac": sum(a in AA_HYDROPHOBIC for a in seq) / n,
            "aromatic_frac": sum(a in AA_AROMATIC for a in seq) / n,
            "positive_frac": sum(a in AA_POSITIVE for a in seq) / n,
            "negative_frac": sum(a in AA_NEGATIVE for a in seq) / n,
            "polar_frac": sum(a in AA_POLAR for a in seq) / n,
            "tiny_frac": sum(a in AA_TINY for a in seq) / n,
            "small_frac": sum(a in AA_SMALL for a in seq) / n,
            "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
            "mean_hydropathy": float(np.mean(hydros)),
            "sequence_entropy": shannon_entropy(seq),
            "max_residue_fraction": max_residue_fraction(seq),
        }
    )
    return desc


def add_computable_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    descriptor_df = pd.DataFrame([compute_sequence_descriptors(s) for s in df[seq_col].astype(str)])
    return pd.concat([df.reset_index(drop=True), descriptor_df.reset_index(drop=True)], axis=1)


def build_feature_blocks(df: pd.DataFrame, cfg: Step5Config) -> Dict[str, List[str]]:
    protected = {cfg.sequence_col, cfg.merge_key, cfg.split_col, cfg.target_col}
    all_numeric = [c for c in df.columns if c not in protected and pd.api.types.is_numeric_dtype(df[c])]
    computable = [
        c for c in all_numeric
        if c == "length"
        or c.startswith("aa_frac_")
        or c.endswith("_frac")
        or c in {"net_charge_proxy", "mean_hydropathy", "sequence_entropy", "max_residue_fraction"}
    ]
    aa_only = [c for c in computable if c.startswith("aa_frac_")]
    physicochem = [c for c in computable if c not in aa_only]

    if not computable:
        raise ValueError("No computable descriptor block could be constructed.")

    return {
        "Full numeric descriptor set": all_numeric,
        "Computable descriptors": computable,
        "AA composition only": aa_only,
        "Physicochemical only": physicochem,
    }


def load_step4_inputs(cfg: Step5Config) -> Dict[str, pd.DataFrame]:
    seq_df = pd.read_csv(cfg.input_model_ready_csv)
    feat_df = pd.read_csv(cfg.input_baseline_features_csv)
    return {"seq_df": seq_df, "feat_df": feat_df}


def validate_step4_contract(seq_df: pd.DataFrame, feat_df: pd.DataFrame, cfg: Step5Config) -> Dict[str, Any]:
    required_seq = {cfg.sequence_col, cfg.merge_key, cfg.split_col, cfg.target_col}
    required_feat = {cfg.sequence_col, cfg.merge_key, cfg.split_col, cfg.target_col}
    miss_seq = required_seq - set(seq_df.columns)
    miss_feat = required_feat - set(feat_df.columns)
    if miss_seq:
        raise ValueError(f"Missing required sequence-table columns: {sorted(miss_seq)}")
    if miss_feat:
        raise ValueError(f"Missing required feature-table columns: {sorted(miss_feat)}")

    for df_name, df in [("seq_df", seq_df), ("feat_df", feat_df)]:
        if df[cfg.merge_key].duplicated().any():
            dup_n = int(df[cfg.merge_key].duplicated().sum())
            raise ValueError(f"{df_name} has {dup_n} duplicated merge keys")
        bad_splits = set(df[cfg.split_col].dropna().unique()) - {cfg.train_label, cfg.val_label, cfg.test_label}
        if bad_splits:
            raise ValueError(f"Unexpected split labels in {df_name}: {sorted(bad_splits)}")

    merged = seq_df[[cfg.merge_key, cfg.target_col, cfg.split_col]].merge(
        feat_df[[cfg.merge_key, cfg.target_col, cfg.split_col]], on=cfg.merge_key, suffixes=("_seq", "_feat")
    )
    if len(merged) == 0:
        raise ValueError("No shared rows between sequence and feature tables")
    if not (merged[f"{cfg.target_col}_seq"] == merged[f"{cfg.target_col}_feat"]).all():
        raise ValueError("Target mismatch between sequence and feature tables")
    if not (merged[f"{cfg.split_col}_seq"] == merged[f"{cfg.split_col}_feat"]).all():
        raise ValueError("Split mismatch between sequence and feature tables")

    qc = {}
    seq_df = seq_df.copy()
    seq_df["clean_sequence"] = seq_df[cfg.sequence_col].astype(str).map(clean_sequence)
    seq_df["is_valid_clean"] = seq_df["clean_sequence"].map(is_valid_peptide)
    qc["n_total_rows"] = int(len(seq_df))
    qc["n_valid_clean_sequences"] = int(seq_df["is_valid_clean"].sum())
    qc["n_invalid_after_cleaning"] = int((~seq_df["is_valid_clean"]).sum())

    split_counts = seq_df[cfg.split_col].value_counts().to_dict()
    for split_name in [cfg.train_label, cfg.val_label, cfg.test_label]:
        if split_counts.get(split_name, 0) == 0:
            raise ValueError(f"Split {split_name!r} has zero rows.")

    clean_valid = seq_df[seq_df["is_valid_clean"]].copy()
    train_classes = set(clean_valid.loc[clean_valid[cfg.split_col] == cfg.train_label, cfg.target_col].astype(str))
    val_classes = set(clean_valid.loc[clean_valid[cfg.split_col] == cfg.val_label, cfg.target_col].astype(str))
    test_classes = set(clean_valid.loc[clean_valid[cfg.split_col] == cfg.test_label, cfg.target_col].astype(str))
    missing_from_train = sorted((val_classes | test_classes) - train_classes)
    if missing_from_train:
        raise ValueError(f"Validation/test contain classes absent from train: {missing_from_train}")

    train_counts = clean_valid.loc[clean_valid[cfg.split_col] == cfg.train_label, cfg.target_col].value_counts()
    if (train_counts < cfg.min_samples_per_class_for_cv).any():
        too_small = train_counts[train_counts < cfg.min_samples_per_class_for_cv].to_dict()
        raise ValueError(
            f"Some training classes have fewer than {cfg.min_samples_per_class_for_cv} samples, which breaks reliable CV: {too_small}"
        )
    qc["train_class_counts"] = {str(k): int(v) for k, v in train_counts.to_dict().items()}
    return qc


def build_model_registry(seed: int) -> Dict[str, Tuple[Pipeline, List[Dict[str, Any]]]]:
    simple_pre = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    registry = {
        "Dummy majority": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("clf", DummyClassifier(strategy="most_frequent"))]),
            [{}],
        ),
        "Logistic regression": (
            Pipeline(simple_pre + [("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=seed))]),
            [{"clf__C": c} for c in [0.25, 1.0, 4.0]],
        ),
        "Random forest": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("clf", RandomForestClassifier(random_state=seed, n_jobs=-1, class_weight="balanced_subsample")),
            ]),
            [
                {"clf__n_estimators": 300, "clf__max_depth": None, "clf__min_samples_leaf": 1},
                {"clf__n_estimators": 400, "clf__max_depth": 16, "clf__min_samples_leaf": 2},
            ],
        ),
        "Extra trees": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("clf", ExtraTreesClassifier(random_state=seed, n_jobs=-1, class_weight="balanced")),
            ]),
            [
                {"clf__n_estimators": 300, "clf__max_depth": None, "clf__min_samples_leaf": 1},
                {"clf__n_estimators": 400, "clf__max_depth": 18, "clf__min_samples_leaf": 2},
            ],
        ),
    }
    return registry


def topk_accuracy(y_true: np.ndarray, probas: np.ndarray, classes: np.ndarray, k: int) -> float:
    idx = np.argsort(-probas, axis=1)[:, :k]
    topk_labels = classes[idx]
    hits = [yt in set(topk_labels[i]) for i, yt in enumerate(y_true)]
    return float(np.mean(hits))


def compute_classification_metrics(
    y_true: np.ndarray,
    probas: np.ndarray,
    pred: np.ndarray,
    classes: np.ndarray,
    topk_values: Sequence[int],
) -> Dict[str, float]:
    metrics = {
        "macro_f1": float(f1_score(y_true, pred, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(y_true, pred)),
    }
    for k in topk_values:
        kk = min(k, probas.shape[1])
        metrics[f"top{kk}_acc"] = topk_accuracy(y_true, probas, classes, kk)
    try:
        if len(classes) > 2:
            y_onehot = pd.get_dummies(pd.Series(y_true), dtype=int).reindex(columns=classes, fill_value=0).values
            metrics["macro_ovr_auc"] = float(roc_auc_score(y_onehot, probas, average="macro", multi_class="ovr"))
        else:
            pos_label = classes[1]
            y_bin = (y_true == pos_label).astype(int)
            metrics["macro_ovr_auc"] = float(roc_auc_score(y_bin, probas[:, 1]))
    except Exception:
        metrics["macro_ovr_auc"] = float("nan")
    return metrics


def train_eval_family(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    seed: int,
    cfg: Step5Config,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    registry = build_model_registry(seed)
    cv = StratifiedKFold(n_splits=cfg.inner_cv_folds, shuffle=True, random_state=seed)
    rows = []
    fitted = {}

    for family, (pipe, grid) in registry.items():
        for params in grid:
            inner_scores = []
            for tr_idx, te_idx in cv.split(X_train, y_train):
                mdl = clone(pipe)
                mdl.set_params(**params)
                mdl.fit(X_train.iloc[tr_idx], y_train[tr_idx])
                pred = mdl.predict(X_train.iloc[te_idx])
                inner_scores.append(f1_score(y_train[te_idx], pred, average="macro"))
            mean_cv = float(np.mean(inner_scores))

            mdl = clone(pipe)
            mdl.set_params(**params)
            mdl.fit(X_train, y_train)
            val_probas = mdl.predict_proba(X_val)
            val_pred = mdl.predict(X_val)
            val_metrics = compute_classification_metrics(y_val, val_probas, val_pred, mdl.named_steps["clf"].classes_, cfg.topk_values)
            rows.append(
                {
                    "family": family,
                    "params": json.dumps(params, sort_keys=True),
                    "cv_macro_f1": mean_cv,
                    **{f"val_{k}": v for k, v in val_metrics.items()},
                }
            )
            fitted[(family, json.dumps(params, sort_keys=True))] = mdl

    results = pd.DataFrame(rows).sort_values(["val_macro_f1", "cv_macro_f1"], ascending=False).reset_index(drop=True)
    best = results.iloc[0]
    best_key = (best["family"], best["params"])
    return results, {"model": fitted[best_key], "family": best["family"], "params": json.loads(best["params"])}


def bootstrap_test_metrics(model: Pipeline, X_test: pd.DataFrame, y_test: np.ndarray, cfg: Step5Config, seed: int) -> Dict[str, Dict[str, float]]:
    rng = np.random.default_rng(seed)
    base_probas = model.predict_proba(X_test)
    base_pred = model.predict(X_test)
    classes = model.named_steps["clf"].classes_
    metric_names = ["macro_f1", "balanced_acc", "macro_ovr_auc"] + [f"top{min(k, base_probas.shape[1])}_acc" for k in cfg.topk_values]
    hist = defaultdict(list)
    n = len(y_test)
    for _ in range(cfg.bootstrap_iterations):
        idx = rng.integers(0, n, size=n)
        m = compute_classification_metrics(y_test[idx], base_probas[idx], base_pred[idx], classes, cfg.topk_values)
        for k in metric_names:
            hist[k].append(m.get(k, np.nan))
    out = {}
    for k, vals in hist.items():
        vals = np.asarray(vals, dtype=float)
        out[k] = {
            "mean": float(np.nanmean(vals)),
            "ci_low": float(np.nanpercentile(vals, 2.5)),
            "ci_high": float(np.nanpercentile(vals, 97.5)),
        }
    return out


def run_repeated_shuffle_control(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    cfg: Step5Config,
) -> pd.DataFrame:
    rows = []
    for rep in range(cfg.shuffle_repeats):
        seed = cfg.main_random_seed + 1000 + rep
        rng = np.random.default_rng(seed)
        y_train_shuf = y_train.copy()
        y_val_shuf = y_val.copy()
        rng.shuffle(y_train_shuf)
        rng.shuffle(y_val_shuf)

        _, best = train_eval_family(X_train, y_train_shuf, X_val, y_val_shuf, seed=seed, cfg=cfg)
        model = best["model"]
        X_tv = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
        y_tv_shuf = np.concatenate([y_train_shuf, y_val_shuf])
        model.fit(X_tv, y_tv_shuf)
        probas = model.predict_proba(X_test)
        pred = model.predict(X_test)
        metrics = compute_classification_metrics(y_test, probas, pred, model.named_steps["clf"].classes_, cfg.topk_values)
        rows.append({"shuffle_repeat": rep, **metrics})
    return pd.DataFrame(rows)


def run_step5a_descriptor_audit(data: Dict[str, pd.DataFrame], cfg: Step5Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Running Step 5A — descriptor benchmark and non-triviality audit")
    feat_df = add_computable_descriptors(data["feat_df"].copy(), cfg.sequence_col)
    feat_df[cfg.sequence_col] = feat_df[cfg.sequence_col].astype(str).map(clean_sequence)
    feat_df = feat_df[feat_df[cfg.sequence_col].map(is_valid_peptide)].reset_index(drop=True)
    blocks = build_feature_blocks(feat_df, cfg)

    split = feat_df[cfg.split_col]
    train_df = feat_df[split == cfg.train_label].reset_index(drop=True)
    val_df = feat_df[split == cfg.val_label].reset_index(drop=True)
    test_df = feat_df[split == cfg.test_label].reset_index(drop=True)

    if min(len(train_df), len(val_df), len(test_df)) == 0:
        raise ValueError("At least one split is empty after descriptor cleaning.")

    le = LabelEncoder()
    y_train = le.fit_transform(train_df[cfg.target_col].astype(str))
    y_val = le.transform(val_df[cfg.target_col].astype(str))
    y_test = le.transform(test_df[cfg.target_col].astype(str))

    feature_manifest_rows = []
    main_rows = []
    fitted_block_best = {}
    full_search_tables = {}

    for block_name, cols in blocks.items():
        if len(cols) == 0:
            continue
        feature_manifest_rows.append({"block": block_name, "n_features": len(cols), "features": ";".join(cols)})
        res_df, best = train_eval_family(train_df[cols], y_train, val_df[cols], y_val, seed=cfg.main_random_seed, cfg=cfg)
        full_search_tables[block_name] = res_df
        best_model = best["model"]

        X_tv = pd.concat([train_df[cols], val_df[cols]], axis=0).reset_index(drop=True)
        y_tv = np.concatenate([y_train, y_val])
        best_model.fit(X_tv, y_tv)
        probas = best_model.predict_proba(test_df[cols])
        pred = best_model.predict(test_df[cols])
        metrics = compute_classification_metrics(y_test, probas, pred, best_model.named_steps["clf"].classes_, cfg.topk_values)
        boot = bootstrap_test_metrics(best_model, test_df[cols], y_test, cfg=cfg, seed=cfg.main_random_seed + 111)

        row = {
            "block": block_name,
            "family": best["family"],
            "n_features": len(cols),
            **metrics,
        }
        for mk, stats in boot.items():
            row[f"{mk}_ci_low"] = stats["ci_low"]
            row[f"{mk}_ci_high"] = stats["ci_high"]
        main_rows.append(row)
        fitted_block_best[block_name] = {
            "model": best_model,
            "features": cols,
            "metrics": metrics,
            "bootstrap": boot,
            "family": best["family"],
        }

    summary_df = pd.DataFrame(main_rows).sort_values(["macro_f1", "balanced_acc"], ascending=False).reset_index(drop=True)
    summary_df.to_csv(dirs["tables_main"] / "table_5_1_step5a_descriptor_benchmark.csv", index=False)
    pd.DataFrame(feature_manifest_rows).to_csv(dirs["tables_supplementary"] / "table_s5_1_step5a_feature_manifest.csv", index=False)
    for block_name, table in full_search_tables.items():
        safe = block_name.lower().replace(" ", "_").replace("-", "_")
        table.to_csv(dirs["tables_supplementary"] / f"step5a_model_search_{safe}.csv", index=False)

    surrogate_key = "Computable descriptors" if "Computable descriptors" in fitted_block_best else summary_df.iloc[0]["block"]
    surrogate = fitted_block_best[surrogate_key]

    shuffle_df = run_repeated_shuffle_control(
        train_df[surrogate["features"]], y_train,
        val_df[surrogate["features"]], y_val,
        test_df[surrogate["features"]], y_test,
        cfg=cfg,
    )
    shuffle_df.to_csv(dirs["tables_supplementary"] / "table_s5_2_step5a_shuffle_control.csv", index=False)

    best_block = summary_df.iloc[0]["block"]
    best_obj = fitted_block_best[best_block]
    best_model = best_obj["model"]
    best_features = best_obj["features"]
    X_test_best = test_df[best_features]
    y_pred_best = best_model.predict(X_test_best)
    cm_raw = confusion_matrix(y_test, y_pred_best)
    cm_norm = confusion_matrix(y_test, y_pred_best, normalize="true")
    cm_raw_df = pd.DataFrame(cm_raw, index=le.classes_, columns=le.classes_)
    cm_norm_df = pd.DataFrame(cm_norm, index=le.classes_, columns=le.classes_)
    cm_raw_df.to_csv(dirs["tables_supplementary"] / "step5a_confusion_matrix_raw.csv")
    cm_norm_df.to_csv(dirs["tables_supplementary"] / "step5a_confusion_matrix_normalized.csv")

    perm_df = None
    try:
        perm = permutation_importance(
            best_model,
            X_test_best,
            y_test,
            n_repeats=cfg.permutation_repeats,
            random_state=cfg.main_random_seed,
            scoring="f1_macro",
        )
        perm_df = pd.DataFrame(
            {
                "feature": best_features,
                "importance_mean": perm.importances_mean,
                "importance_std": perm.importances_std,
            }
        ).sort_values("importance_mean", ascending=False)
        perm_df.to_csv(dirs["tables_supplementary"] / "step5a_permutation_importance.csv", index=False)
    except Exception as e:
        warnings.warn(f"Permutation importance failed: {e}")

    pca_cols = surrogate["features"]
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(imputer.fit_transform(train_df[pca_cols]))
    Xte = scaler.transform(imputer.transform(test_df[pca_cols]))
    pca = PCA(n_components=2, random_state=cfg.main_random_seed)
    Xte_2d = pca.fit(Xtr).transform(Xte)
    pca_df = pd.DataFrame({
        "pc1": Xte_2d[:, 0],
        "pc2": Xte_2d[:, 1],
        "condition": test_df[cfg.target_col].astype(str).values,
    })
    pca_df.to_csv(dirs["tables_supplementary"] / "step5a_pca_projection.csv", index=False)

    surrogate_payload = {
        "block": surrogate_key,
        "features": surrogate["features"],
        "label_classes": list(map(str, le.classes_)),
        "best_block_overall": best_block,
        "best_block_family": best_obj["family"],
    }
    save_json(surrogate_payload, dirs["artifacts"] / "step5a_surrogate_metadata.json")

    return {
        "summary_df": summary_df,
        "feature_manifest": pd.DataFrame(feature_manifest_rows),
        "shuffle_df": shuffle_df,
        "surrogate_model": surrogate["model"],
        "surrogate_features": surrogate["features"],
        "label_encoder": le,
        "best_block": best_block,
        "best_block_obj": best_obj,
        "confusion_matrix_raw": cm_raw_df,
        "confusion_matrix_norm": cm_norm_df,
        "perm_df": perm_df,
        "pca_df": pca_df,
        "train_df_with_desc": train_df,
        "val_df_with_desc": val_df,
        "test_df_with_desc": test_df,
    }


class PeptideTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"
        self.alphabet = [self.pad_token, self.bos_token, self.eos_token] + AA_ALPHABET
        self.stoi = {tok: i for i, tok in enumerate(self.alphabet)}
        self.itos = {i: tok for tok, i in self.stoi.items()}

    @property
    def pad_id(self) -> int:
        return self.stoi[self.pad_token]

    @property
    def bos_id(self) -> int:
        return self.stoi[self.bos_token]

    @property
    def eos_id(self) -> int:
        return self.stoi[self.eos_token]

    @property
    def vocab_size(self) -> int:
        return len(self.alphabet)

    def encode(self, seq: str, max_len: int) -> List[int]:
        seq = clean_sequence(seq)
        toks = [self.bos_id] + [self.stoi[s] for s in seq[: max_len - 2]] + [self.eos_id]
        toks = toks[:max_len]
        toks += [self.pad_id] * (max_len - len(toks))
        return toks

    def decode(self, ids: Sequence[int]) -> str:
        chars = []
        for i in ids:
            tok = self.itos[int(i)]
            if tok == self.eos_token:
                break
            if tok in {self.pad_token, self.bos_token}:
                continue
            chars.append(tok)
        return "".join(chars)


class SequenceDataset(Dataset):
    def __init__(self, sequences: Sequence[str], tokenizer: PeptideTokenizer, max_len: int, conditions: Optional[Sequence[int]] = None):
        self.seqs = [clean_sequence(s) for s in sequences]
        self.tok = tokenizer
        self.max_len = max_len
        self.conditions = None if conditions is None else list(conditions)

    def __len__(self) -> int:
        return len(self.seqs)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ids = self.tok.encode(self.seqs[idx], self.max_len)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        out = {"input_ids": x, "target_ids": y}
        if self.conditions is not None:
            out["condition_id"] = torch.tensor(self.conditions[idx], dtype=torch.long)
        return out


def make_dataloader(ds: Dataset, batch_size: int, shuffle: bool, num_workers: int, seed: int) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, generator=generator)


class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, num_layers: int, dropout: float, cond_vocab_size: int = 0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb = nn.Embedding(cond_vocab_size, hidden_dim) if cond_vocab_size > 0 else None
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids: torch.Tensor, condition_id: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = self.emb(input_ids)
        h0 = None
        if self.cond_emb is not None and condition_id is not None:
            h0 = self.cond_emb(condition_id).unsqueeze(0)
        out, _ = self.rnn(x, h0)
        logits = self.out(out)
        return logits


class SequenceVAE(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, latent_dim: int, num_layers: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.encoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)
        self.z_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.latent_dim = latent_dim

    def encode(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        emb = self.emb(input_ids)
        _, h = self.encoder(emb)
        h = h[-1]
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar

    def decode(self, input_ids: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        emb = self.emb(input_ids)
        h0 = self.z_to_hidden(z).unsqueeze(0)
        dec, _ = self.decoder(emb, h0)
        return self.out(dec)

    def forward(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z, mu, logvar = self.encode(input_ids)
        logits = self.decode(input_ids, z)
        return logits, mu, logvar


class ConditionalSequenceVAE(nn.Module):
    def __init__(self, vocab_size: int, cond_vocab_size: int, emb_dim: int, hidden_dim: int, latent_dim: int, num_layers: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb_enc = nn.Embedding(cond_vocab_size, emb_dim)
        self.cond_emb_dec = nn.Embedding(cond_vocab_size, hidden_dim)
        self.encoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)
        self.z_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.latent_dim = latent_dim

    def encode(self, input_ids: torch.Tensor, condition_id: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        emb = self.emb(input_ids) + self.cond_emb_enc(condition_id).unsqueeze(1)
        _, h = self.encoder(emb)
        h = h[-1]
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar

    def decode(self, input_ids: torch.Tensor, z: torch.Tensor, condition_id: torch.Tensor) -> torch.Tensor:
        emb = self.emb(input_ids)
        h0 = (self.z_to_hidden(z) + self.cond_emb_dec(condition_id)).unsqueeze(0)
        dec, _ = self.decoder(emb, h0)
        return self.out(dec)

    def forward(self, input_ids: torch.Tensor, condition_id: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z, mu, logvar = self.encode(input_ids, condition_id)
        logits = self.decode(input_ids, z, condition_id)
        return logits, mu, logvar


def cross_entropy_ignore_pad(logits: torch.Tensor, target_ids: torch.Tensor, pad_id: int) -> torch.Tensor:
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1), ignore_index=pad_id)


def train_gru_model(
    train_sequences: Sequence[str],
    val_sequences: Sequence[str],
    tokenizer: PeptideTokenizer,
    train_conditions: Optional[Sequence[int]],
    val_conditions: Optional[Sequence[int]],
    cond_vocab_size: int,
    seed: int,
    cfg: Step5Config,
) -> Tuple[nn.Module, pd.DataFrame]:
    seed_all(seed)
    gcfg = cfg.gen_train
    device = device_for_training()
    model = GRULanguageModel(tokenizer.vocab_size, gcfg.emb_dim, gcfg.hidden_dim, gcfg.num_layers, gcfg.dropout, cond_vocab_size=cond_vocab_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=gcfg.lr, weight_decay=gcfg.weight_decay)
    epochs = gcfg.epochs_fast if cfg.fast_mode else gcfg.epochs_full

    train_ds = SequenceDataset(train_sequences, tokenizer, gcfg.max_seq_len, train_conditions)
    val_ds = SequenceDataset(val_sequences, tokenizer, gcfg.max_seq_len, val_conditions)
    train_loader = make_dataloader(train_ds, gcfg.batch_size, True, gcfg.num_workers, seed)
    val_loader = make_dataloader(val_ds, gcfg.batch_size, False, gcfg.num_workers, seed + 1)

    best_state = None
    best_val = float("inf")
    bad = 0
    logs = []
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            x = batch["input_ids"].to(device)
            y = batch["target_ids"].to(device)
            c = batch.get("condition_id")
            c = c.to(device) if c is not None else None
            logits = model(x, c)
            loss = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gcfg.clip_grad_norm)
            opt.step()
            train_losses.append(float(loss.item()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_ids"].to(device)
                y = batch["target_ids"].to(device)
                c = batch.get("condition_id")
                c = c.to(device) if c is not None else None
                logits = model(x, c)
                val_losses.append(float(cross_entropy_ignore_pad(logits, y, tokenizer.pad_id).item()))
        train_loss = float(np.mean(train_losses)) if train_losses else np.nan
        val_loss = float(np.mean(val_losses)) if val_losses else np.inf
        logs.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= gcfg.early_stopping_patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(logs)


def train_vae_model(train_sequences: Sequence[str], val_sequences: Sequence[str], tokenizer: PeptideTokenizer, seed: int, cfg: Step5Config) -> Tuple[nn.Module, pd.DataFrame]:
    seed_all(seed)
    gcfg = cfg.gen_train
    device = device_for_training()
    model = SequenceVAE(tokenizer.vocab_size, gcfg.emb_dim, gcfg.hidden_dim, gcfg.latent_dim, gcfg.num_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=gcfg.lr, weight_decay=gcfg.weight_decay)
    epochs = gcfg.epochs_fast if cfg.fast_mode else gcfg.epochs_full

    train_ds = SequenceDataset(train_sequences, tokenizer, gcfg.max_seq_len)
    val_ds = SequenceDataset(val_sequences, tokenizer, gcfg.max_seq_len)
    train_loader = make_dataloader(train_ds, gcfg.batch_size, True, gcfg.num_workers, seed)
    val_loader = make_dataloader(val_ds, gcfg.batch_size, False, gcfg.num_workers, seed + 1)

    best_state = None
    best_val = float("inf")
    bad = 0
    logs = []
    for epoch in range(epochs):
        beta = min(1.0, (epoch + 1) / max(1, epochs / 2))
        model.train()
        train_total = []
        train_rec = []
        train_kl = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            x = batch["input_ids"].to(device)
            y = batch["target_ids"].to(device)
            logits, mu, logvar = model(x)
            rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = rec + beta * kl
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gcfg.clip_grad_norm)
            opt.step()
            train_total.append(float(loss.item()))
            train_rec.append(float(rec.item()))
            train_kl.append(float(kl.item()))

        model.eval()
        val_total = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_ids"].to(device)
                y = batch["target_ids"].to(device)
                logits, mu, logvar = model(x)
                rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                val_total.append(float((rec + kl).item()))
        train_loss = float(np.mean(train_total)) if train_total else np.nan
        val_loss = float(np.mean(val_total)) if val_total else np.inf
        logs.append(
            {
                "epoch": epoch + 1,
                "beta": beta,
                "train_loss": train_loss,
                "train_reconstruction": float(np.mean(train_rec)) if train_rec else np.nan,
                "train_kl": float(np.mean(train_kl)) if train_kl else np.nan,
                "val_loss": val_loss,
            }
        )
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= gcfg.early_stopping_patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(logs)


def train_cvae_model(
    train_sequences: Sequence[str],
    val_sequences: Sequence[str],
    tokenizer: PeptideTokenizer,
    train_conditions: Sequence[int],
    val_conditions: Sequence[int],
    cond_vocab_size: int,
    seed: int,
    cfg: Step5Config,
) -> Tuple[nn.Module, pd.DataFrame]:
    seed_all(seed)
    gcfg = cfg.gen_train
    device = device_for_training()
    model = ConditionalSequenceVAE(tokenizer.vocab_size, cond_vocab_size, gcfg.emb_dim, gcfg.hidden_dim, gcfg.latent_dim, gcfg.num_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=gcfg.lr, weight_decay=gcfg.weight_decay)
    epochs = gcfg.epochs_fast if cfg.fast_mode else gcfg.epochs_full

    train_ds = SequenceDataset(train_sequences, tokenizer, gcfg.max_seq_len, train_conditions)
    val_ds = SequenceDataset(val_sequences, tokenizer, gcfg.max_seq_len, val_conditions)
    train_loader = make_dataloader(train_ds, gcfg.batch_size, True, gcfg.num_workers, seed)
    val_loader = make_dataloader(val_ds, gcfg.batch_size, False, gcfg.num_workers, seed + 1)

    best_state = None
    best_val = float("inf")
    bad = 0
    logs = []
    for epoch in range(epochs):
        beta = min(1.0, (epoch + 1) / max(1, epochs / 2))
        model.train()
        train_total = []
        train_rec = []
        train_kl = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            x = batch["input_ids"].to(device)
            y = batch["target_ids"].to(device)
            c = batch["condition_id"].to(device)
            logits, mu, logvar = model(x, c)
            rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = rec + beta * kl
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gcfg.clip_grad_norm)
            opt.step()
            train_total.append(float(loss.item()))
            train_rec.append(float(rec.item()))
            train_kl.append(float(kl.item()))

        model.eval()
        val_total = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_ids"].to(device)
                y = batch["target_ids"].to(device)
                c = batch["condition_id"].to(device)
                logits, mu, logvar = model(x, c)
                rec = cross_entropy_ignore_pad(logits, y, tokenizer.pad_id)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                val_total.append(float((rec + kl).item()))
        train_loss = float(np.mean(train_total)) if train_total else np.nan
        val_loss = float(np.mean(val_total)) if val_total else np.inf
        logs.append(
            {
                "epoch": epoch + 1,
                "beta": beta,
                "train_loss": train_loss,
                "train_reconstruction": float(np.mean(train_rec)) if train_rec else np.nan,
                "train_kl": float(np.mean(train_kl)) if train_kl else np.nan,
                "val_loss": val_loss,
            }
        )
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= gcfg.early_stopping_patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(logs)


def sample_from_logits(logits: torch.Tensor, temperature: float = 1.0) -> int:
    probs = torch.softmax(logits / max(temperature, 1e-6), dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())


def generate_gru_sequences(
    model: GRULanguageModel,
    tokenizer: PeptideTokenizer,
    n: int,
    max_len: int,
    seed: int,
    temperature: float = 1.0,
    condition_id: Optional[int] = None,
) -> List[str]:
    seed_all(seed)
    device = device_for_training()
    model = model.to(device)
    model.eval()
    outputs = []
    with torch.no_grad():
        for _ in range(n):
            cur = [tokenizer.bos_id]
            hidden = None
            if model.cond_emb is not None and condition_id is not None:
                hidden = model.cond_emb(torch.tensor([condition_id], device=device)).unsqueeze(0)
            for _ in range(max_len - 1):
                x = torch.tensor([cur[-1:]], dtype=torch.long, device=device)
                emb = model.emb(x)
                out, hidden = model.rnn(emb, hidden)
                logits = model.out(out[:, -1, :]).squeeze(0)
                nxt = sample_from_logits(logits, temperature=temperature)
                if nxt == tokenizer.eos_id:
                    break
                if nxt == tokenizer.pad_id:
                    continue
                cur.append(nxt)
            outputs.append(clean_sequence(tokenizer.decode(cur[1:])))
    return outputs


def generate_vae_sequences(model: SequenceVAE, tokenizer: PeptideTokenizer, n: int, max_len: int, seed: int, temperature: float = 1.0) -> List[str]:
    seed_all(seed)
    device = device_for_training()
    model = model.to(device)
    model.eval()
    outputs = []
    with torch.no_grad():
        for _ in range(n):
            z = torch.randn(1, model.latent_dim, device=device)
            hidden = model.z_to_hidden(z).unsqueeze(0)
            cur_token = tokenizer.bos_id
            cur = []
            for _ in range(max_len - 1):
                x = torch.tensor([[cur_token]], dtype=torch.long, device=device)
                emb = model.emb(x)
                out, hidden = model.decoder(emb, hidden)
                logits = model.out(out[:, -1, :]).squeeze(0)
                nxt = sample_from_logits(logits, temperature=temperature)
                if nxt == tokenizer.eos_id:
                    break
                if nxt == tokenizer.pad_id:
                    continue
                cur.append(nxt)
                cur_token = nxt
            outputs.append(clean_sequence(tokenizer.decode(cur)))
    return outputs


def generate_cvae_sequences(
    model: ConditionalSequenceVAE,
    tokenizer: PeptideTokenizer,
    n: int,
    max_len: int,
    seed: int,
    condition_id: int,
    temperature: float = 1.0,
) -> List[str]:
    seed_all(seed)
    device = device_for_training()
    model = model.to(device)
    model.eval()
    outputs = []
    cond = torch.tensor([condition_id], device=device)
    with torch.no_grad():
        for _ in range(n):
            z = torch.randn(1, model.latent_dim, device=device)
            hidden = (model.z_to_hidden(z) + model.cond_emb_dec(cond)).unsqueeze(0)
            cur_token = tokenizer.bos_id
            cur = []
            for _ in range(max_len - 1):
                x = torch.tensor([[cur_token]], dtype=torch.long, device=device)
                emb = model.emb(x)
                out, hidden = model.decoder(emb, hidden)
                logits = model.out(out[:, -1, :]).squeeze(0)
                nxt = sample_from_logits(logits, temperature=temperature)
                if nxt == tokenizer.eos_id:
                    break
                if nxt == tokenizer.pad_id:
                    continue
                cur.append(nxt)
                cur_token = nxt
            outputs.append(clean_sequence(tokenizer.decode(cur)))
    return outputs


def build_random_conditional_generator(train_df: pd.DataFrame, cfg: Step5Config) -> Dict[str, Any]:
    grouped = {}
    for cond, g in train_df.groupby(cfg.target_col):
        seqs = [clean_sequence(s) for s in g[cfg.sequence_col].astype(str) if is_valid_peptide(clean_sequence(s))]
        lengths = [len(s) for s in seqs if len(s) > 0]
        aa_counts = Counter("".join(seqs))
        probs = np.array([aa_counts.get(a, 1) for a in AA_ALPHABET], dtype=float)
        probs = probs / probs.sum()
        grouped[str(cond)] = {"lengths": lengths, "aa_probs": probs.tolist()}
    return grouped


def generate_random_conditional(random_model: Dict[str, Any], condition: str, n: int, seed: int) -> List[str]:
    rng = np.random.default_rng(seed)
    spec = random_model[str(condition)]
    lengths = np.asarray(spec["lengths"], dtype=int)
    probs = np.asarray(spec["aa_probs"], dtype=float)
    out = []
    for _ in range(n):
        L = int(rng.choice(lengths)) if len(lengths) > 0 else int(rng.integers(8, 30))
        seq = "".join(rng.choice(np.array(AA_ALPHABET), size=L, p=probs))
        out.append(seq)
    return out


def build_retrieval_reference(train_df: pd.DataFrame, cfg: Step5Config) -> Dict[str, List[str]]:
    ref = {}
    for cond, g in train_df.groupby(cfg.target_col):
        ref[str(cond)] = [clean_sequence(s) for s in g[cfg.sequence_col].astype(str) if is_valid_peptide(clean_sequence(s))]
    return ref


def generate_retrieval_samples(retrieval_model: Dict[str, List[str]], condition: str, n: int, seed: int) -> List[str]:
    rng = np.random.default_rng(seed)
    pool = retrieval_model[str(condition)]
    if len(pool) == 0:
        return []
    idx = rng.integers(0, len(pool), size=n)
    return [pool[i] for i in idx]


def kmer_set(seq: str, k: int) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}


def jaccard_similarity(a: str, b: str, k: int) -> float:
    sa, sb = kmer_set(a, k), kmer_set(b, k)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def nearest_neighbor_similarity(seq: str, reference: Sequence[str], k: int) -> float:
    if len(reference) == 0:
        return 0.0
    return max(jaccard_similarity(seq, r, k) for r in reference)


def pairwise_diversity(sequences: Sequence[str], k: int, max_pairs: int, seed: int) -> float:
    seqs = [s for s in sequences if is_valid_peptide(s)]
    n = len(seqs)
    if n < 2:
        return 0.0
    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2
    if total_pairs <= max_pairs:
        idx_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    else:
        idx_pairs = []
        seen = set()
        while len(idx_pairs) < max_pairs:
            i, j = sorted(rng.choice(n, size=2, replace=False).tolist())
            if (i, j) not in seen:
                seen.add((i, j))
                idx_pairs.append((i, j))
    sims = [jaccard_similarity(seqs[i], seqs[j], k) for i, j in idx_pairs]
    return float(1.0 - np.mean(sims))


def build_descriptor_reference(train_sequences: Sequence[str]) -> Dict[str, Any]:
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in train_sequences])
    mu = desc_df.mean(axis=0)
    sd = desc_df.std(axis=0).replace(0, 1.0)
    return {"mean": mu, "std": sd, "columns": list(desc_df.columns)}


def support_rate(sequences: Sequence[str], ref: Dict[str, Any], z_threshold: float = 3.0) -> float:
    if len(sequences) == 0:
        return 0.0
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in sequences])
    desc_df = desc_df[ref["columns"]]
    z = (desc_df - ref["mean"]) / ref["std"]
    inside = (np.abs(z) <= z_threshold).all(axis=1)
    return float(np.mean(inside))


def bootstrap_ci(values: Sequence[float], n_boot: int, seed: int) -> Tuple[float, float, float]:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0]), float(vals[0])
    rng = np.random.default_rng(seed)
    hist = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(vals), size=len(vals))
        hist.append(np.mean(vals[idx]))
    hist = np.asarray(hist, dtype=float)
    return float(np.mean(vals)), float(np.percentile(hist, 2.5)), float(np.percentile(hist, 97.5))


def evaluate_generated_set(
    generated_df: pd.DataFrame,
    surrogate_model: Pipeline,
    surrogate_features: Sequence[str],
    label_encoder: LabelEncoder,
    train_sequences: Sequence[str],
    full_sequences: Sequence[str],
    train_by_condition: Dict[str, List[str]],
    descriptor_ref: Dict[str, Any],
    cfg: Step5Config,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    records = []
    candidate_rows = []
    train_set = set(train_sequences)
    full_set = set(full_sequences)

    for (generator, seed, requested_condition), g in generated_df.groupby(["generator", "seed", "requested_condition"]):
        requested_condition = str(requested_condition)
        raw_seqs = [clean_sequence(s) for s in g["generated_sequence"].astype(str)]
        valid_mask = [is_valid_peptide(s) for s in raw_seqs]
        valid_seqs = [s for s, ok in zip(raw_seqs, valid_mask) if ok]
        seen = set()
        unique_valid = []
        duplicate_flags = []
        for s in valid_seqs:
            duplicate = s in seen
            duplicate_flags.append(duplicate)
            if not duplicate:
                seen.add(s)
                unique_valid.append(s)

        desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in valid_seqs]) if valid_seqs else pd.DataFrame(columns=surrogate_features)
        if len(desc_df) > 0:
            desc_df = desc_df.reindex(columns=surrogate_features, fill_value=0.0)
            probas = surrogate_model.predict_proba(desc_df)
            pred_idx = np.argmax(probas, axis=1)
            pred_labels = label_encoder.inverse_transform(pred_idx)
            target_label_index = np.where(label_encoder.classes_ == requested_condition)[0]
            target_probs = probas[:, target_label_index[0]] if len(target_label_index) > 0 else np.zeros(len(valid_seqs), dtype=float)
        else:
            pred_labels = np.array([], dtype=object)
            target_probs = np.array([], dtype=float)

        train_same_cond_set = set(train_by_condition.get(requested_condition, []))
        nn_train = [nearest_neighbor_similarity(s, train_sequences, cfg.similarity_kmer) for s in valid_seqs]
        nn_train_same_cond = [nearest_neighbor_similarity(s, train_by_condition.get(requested_condition, []), cfg.similarity_kmer) for s in valid_seqs]
        exact_novel_train = [s not in train_set for s in valid_seqs]
        exact_novel_full = [s not in full_set for s in valid_seqs]
        exact_novel_same_cond_train = [s not in train_same_cond_set for s in valid_seqs]
        low_complex_flags = [is_low_complexity(s, cfg.low_complexity_threshold) for s in valid_seqs]

        desc_cache = [compute_sequence_descriptors(s) for s in valid_seqs]
        for i, s in enumerate(valid_seqs):
            d = desc_cache[i]
            candidate_rows.append(
                {
                    "generator": generator,
                    "seed": int(seed),
                    "requested_condition": requested_condition,
                    "generated_sequence": s,
                    "valid_flag": 1,
                    "duplicate_flag": int(duplicate_flags[i]),
                    "low_complexity_flag": int(low_complex_flags[i]),
                    "length": d["length"],
                    "net_charge_proxy": d["net_charge_proxy"],
                    "mean_hydropathy": d["mean_hydropathy"],
                    "sequence_entropy": d["sequence_entropy"],
                    "exact_novel_train": int(exact_novel_train[i]),
                    "exact_novel_full": int(exact_novel_full[i]),
                    "exact_novel_same_condition_train": int(exact_novel_same_cond_train[i]),
                    "nn_similarity_to_train": float(nn_train[i]),
                    "nn_similarity_to_requested_condition_train": float(nn_train_same_cond[i]),
                    "surrogate_predicted_condition": pred_labels[i] if len(pred_labels) > 0 else np.nan,
                    "surrogate_target_probability": float(target_probs[i]) if len(target_probs) > 0 else np.nan,
                }
            )

        metrics = {
            "generator": generator,
            "seed": int(seed),
            "requested_condition": requested_condition,
            "n_requested": int(len(raw_seqs)),
            "valid_rate": float(np.mean(valid_mask)) if len(valid_mask) else 0.0,
            "unique_rate_among_valid": float(len(unique_valid) / max(len(valid_seqs), 1)),
            "duplicate_rate_within_valid": 1.0 - float(len(unique_valid) / max(len(valid_seqs), 1)),
            "exact_novelty_vs_train": float(np.mean(exact_novel_train)) if len(exact_novel_train) else 0.0,
            "exact_novelty_vs_full": float(np.mean(exact_novel_full)) if len(exact_novel_full) else 0.0,
            "exact_novelty_vs_same_condition_train": float(np.mean(exact_novel_same_cond_train)) if len(exact_novel_same_cond_train) else 0.0,
            "mean_nn_similarity_to_train": float(np.mean(nn_train)) if nn_train else 0.0,
            "mean_nn_similarity_to_same_condition_train": float(np.mean(nn_train_same_cond)) if nn_train_same_cond else 0.0,
            "prop_nn_similarity_lt_080": float(np.mean(np.asarray(nn_train) < 0.80)) if nn_train else 0.0,
            "prop_nn_similarity_lt_070": float(np.mean(np.asarray(nn_train) < 0.70)) if nn_train else 0.0,
            "pairwise_diversity": pairwise_diversity(unique_valid, cfg.similarity_kmer, cfg.pairwise_diversity_sample_size, seed=seed),
            "support_rate": support_rate(unique_valid, descriptor_ref),
            "surrogate_condition_hit_rate": float(np.mean(pred_labels == requested_condition)) if len(pred_labels) else 0.0,
            "surrogate_target_probability": float(np.mean(target_probs)) if len(target_probs) else 0.0,
            "low_complexity_rate": float(np.mean(low_complex_flags)) if len(low_complex_flags) else 0.0,
            "mean_length": float(np.mean([d["length"] for d in desc_cache])) if desc_cache else 0.0,
            "mean_charge_proxy": float(np.mean([d["net_charge_proxy"] for d in desc_cache])) if desc_cache else 0.0,
            "mean_hydropathy": float(np.mean([d["mean_hydropathy"] for d in desc_cache])) if desc_cache else 0.0,
        }
        records.append(metrics)

    per_condition_df = pd.DataFrame(records)
    candidate_df = pd.DataFrame(candidate_rows)
    return per_condition_df, candidate_df


def summarize_generator_metrics(per_condition_df: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    metric_cols = [
        c for c in per_condition_df.columns
        if c not in {"generator", "requested_condition", "seed", "n_requested"}
    ]
    rows = []
    for generator, g in per_condition_df.groupby("generator"):
        row = {"generator": generator, "generator_display": MODEL_DISPLAY_NAMES.get(generator, generator)}
        for c in metric_cols:
            stable_seed = cfg.main_random_seed + sum(ord(ch) for ch in f"{generator}|{c}")
            mean_val, ci_low, ci_high = bootstrap_ci(g[c].astype(float).values, cfg.bootstrap_iterations, stable_seed)
            row[c] = mean_val
            row[f"{c}_ci_low"] = ci_low
            row[f"{c}_ci_high"] = ci_high
            row[f"{c}_sd"] = float(np.std(g[c].astype(float).values, ddof=1)) if len(g) > 1 else 0.0
        row["n_generator_condition_runs"] = int(len(g))
        rows.append(row)
    summary = pd.DataFrame(rows).sort_values(["surrogate_condition_hit_rate", "exact_novelty_vs_train"], ascending=False).reset_index(drop=True)
    return summary


def run_step5b_generator_benchmark(data: Dict[str, pd.DataFrame], step5a: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Running Step 5B — baseline generator benchmark")
    seq_df = data["seq_df"].copy()
    seq_df[cfg.sequence_col] = seq_df[cfg.sequence_col].astype(str).map(clean_sequence)
    seq_df = seq_df[seq_df[cfg.sequence_col].map(is_valid_peptide)].reset_index(drop=True)

    train_df = seq_df[seq_df[cfg.split_col] == cfg.train_label].reset_index(drop=True)
    val_df = seq_df[seq_df[cfg.split_col] == cfg.val_label].reset_index(drop=True)
    _ = seq_df[seq_df[cfg.split_col] == cfg.test_label].reset_index(drop=True)

    tokenizer = PeptideTokenizer()
    cond_encoder = LabelEncoder()
    cond_train = cond_encoder.fit_transform(train_df[cfg.target_col].astype(str))
    cond_val = cond_encoder.transform(val_df[cfg.target_col].astype(str))
    unique_conditions = list(map(str, cond_encoder.classes_))

    save_json({"condition_classes": unique_conditions}, dirs["artifacts"] / "step5b_condition_classes.json")

    random_model = build_random_conditional_generator(train_df, cfg)
    retrieval_model = build_retrieval_reference(train_df, cfg)
    train_by_condition = {str(k): v for k, v in retrieval_model.items()}

    training_manifest_rows = []
    wrappers: List[Dict[str, Any]] = []
    for seed in cfg.repeated_seeds:
        print(f"Training seed {seed} baselines...")

        gru_u, gru_u_log = train_gru_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            None,
            None,
            0,
            seed,
            cfg,
        )
        torch.save(gru_u.state_dict(), dirs["models"] / f"gru_unconditional_seed{seed}.pt")
        gru_u_log.to_csv(dirs["logs"] / f"gru_unconditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "gru_unconditional", "kind": "gru_unconditional", "seed": seed, "model": gru_u})
        training_manifest_rows.append({"generator": "gru_unconditional", "seed": seed, "best_val_loss": float(gru_u_log["val_loss"].min())})

        vae_u, vae_u_log = train_vae_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            seed,
            cfg,
        )
        torch.save(vae_u.state_dict(), dirs["models"] / f"vae_unconditional_seed{seed}.pt")
        vae_u_log.to_csv(dirs["logs"] / f"vae_unconditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "vae_unconditional", "kind": "vae_unconditional", "seed": seed, "model": vae_u})
        training_manifest_rows.append({"generator": "vae_unconditional", "seed": seed, "best_val_loss": float(vae_u_log["val_loss"].min())})

        gru_c, gru_c_log = train_gru_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            cond_train,
            cond_val,
            len(unique_conditions),
            seed,
            cfg,
        )
        torch.save(gru_c.state_dict(), dirs["models"] / f"gru_conditional_seed{seed}.pt")
        gru_c_log.to_csv(dirs["logs"] / f"gru_conditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "gru_conditional", "kind": "gru_conditional", "seed": seed, "model": gru_c})
        training_manifest_rows.append({"generator": "gru_conditional", "seed": seed, "best_val_loss": float(gru_c_log["val_loss"].min())})

        cvae_c, cvae_c_log = train_cvae_model(
            train_df[cfg.sequence_col].tolist(),
            val_df[cfg.sequence_col].tolist(),
            tokenizer,
            cond_train,
            cond_val,
            len(unique_conditions),
            seed,
            cfg,
        )
        torch.save(cvae_c.state_dict(), dirs["models"] / f"cvae_conditional_seed{seed}.pt")
        cvae_c_log.to_csv(dirs["logs"] / f"cvae_conditional_seed{seed}_training.csv", index=False)
        wrappers.append({"name": "cvae_conditional", "kind": "cvae_conditional", "seed": seed, "model": cvae_c})
        training_manifest_rows.append({"generator": "cvae_conditional", "seed": seed, "best_val_loss": float(cvae_c_log["val_loss"].min())})

        wrappers.append({"name": "random_conditional", "kind": "random_conditional", "seed": seed, "model": random_model})
        wrappers.append({"name": "retrieval_reference", "kind": "retrieval_reference", "seed": seed, "model": retrieval_model})

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    training_manifest_df = pd.DataFrame(training_manifest_rows)
    training_manifest_df.to_csv(dirs["tables_supplementary"] / "table_s5_3_generator_training_summary.csv", index=False)

    all_generated_rows = []
    budget = cfg.n_generated_per_condition
    for wrapper in wrappers:
        for cond in unique_conditions:
            if wrapper["kind"] == "random_conditional":
                seqs = generate_random_conditional(wrapper["model"], cond, budget, seed=wrapper["seed"] + 17)
            elif wrapper["kind"] == "retrieval_reference":
                seqs = generate_retrieval_samples(wrapper["model"], cond, budget, seed=wrapper["seed"] + 23)
            elif wrapper["kind"] == "gru_unconditional":
                seqs = generate_gru_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 29, temperature=cfg.gen_train.temperature)
            elif wrapper["kind"] == "vae_unconditional":
                seqs = generate_vae_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 31, temperature=cfg.gen_train.temperature)
            elif wrapper["kind"] == "gru_conditional":
                cond_id = int(cond_encoder.transform([cond])[0])
                seqs = generate_gru_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 37, temperature=cfg.gen_train.temperature, condition_id=cond_id)
            elif wrapper["kind"] == "cvae_conditional":
                cond_id = int(cond_encoder.transform([cond])[0])
                seqs = generate_cvae_sequences(wrapper["model"], tokenizer, budget, cfg.gen_train.max_seq_len, seed=wrapper["seed"] + 41, condition_id=cond_id, temperature=cfg.gen_train.temperature)
            else:
                raise ValueError(f"Unsupported generator kind: {wrapper['kind']}")

            for s in seqs:
                all_generated_rows.append(
                    {
                        "generator": wrapper["name"],
                        "seed": int(wrapper["seed"]),
                        "requested_condition": str(cond),
                        "generated_sequence": clean_sequence(s),
                    }
                )

    generated_df = pd.DataFrame(all_generated_rows)
    generated_df.to_csv(dirs["samples"] / "step5b_generated_candidates_raw.csv", index=False)

    train_sequences = train_df[cfg.sequence_col].astype(str).map(clean_sequence).tolist()
    full_sequences = seq_df[cfg.sequence_col].astype(str).map(clean_sequence).tolist()
    descriptor_ref = build_descriptor_reference(train_sequences)

    per_condition_df, candidate_df = evaluate_generated_set(
        generated_df=generated_df,
        surrogate_model=step5a["surrogate_model"],
        surrogate_features=step5a["surrogate_features"],
        label_encoder=step5a["label_encoder"],
        train_sequences=train_sequences,
        full_sequences=full_sequences,
        train_by_condition=train_by_condition,
        descriptor_ref=descriptor_ref,
        cfg=cfg,
    )
    summary_df = summarize_generator_metrics(per_condition_df, cfg)

    per_condition_df.to_csv(dirs["tables_supplementary"] / "table_s5_4_step5b_per_condition_metrics.csv", index=False)
    candidate_df.to_csv(dirs["samples"] / "step5b_generated_candidates_evaluated.csv", index=False)
    summary_df.to_csv(dirs["tables_main"] / "table_5_2_step5b_generator_benchmark.csv", index=False)

    return {
        "generated_df": generated_df,
        "per_condition_df": per_condition_df,
        "candidate_df": candidate_df,
        "summary_df": summary_df,
        "condition_classes": unique_conditions,
        "training_manifest_df": training_manifest_df,
    }


def make_step5_main_figure(step5a: Dict[str, Any], step5b: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> None:
    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.8))
    gs = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    df = step5a["summary_df"].copy()
    labels = list(df["block"])
    y = list(df["macro_f1"])
    ylow = list(df["macro_f1"] - df["macro_f1_ci_low"])
    yhigh = list(df["macro_f1_ci_high"] - df["macro_f1"])
    shuffle_mean = float(step5a["shuffle_df"]["macro_f1"].mean())
    shuffle_low = float(step5a["shuffle_df"]["macro_f1"].quantile(0.025))
    shuffle_high = float(step5a["shuffle_df"]["macro_f1"].quantile(0.975))
    labels_all = labels + ["Shuffled-label control"]
    y_all = y + [shuffle_mean]
    ylow_all = ylow + [shuffle_mean - shuffle_low]
    yhigh_all = yhigh + [shuffle_high - shuffle_mean]
    xpos = np.arange(len(labels_all))
    colors = [OKABE_ITO["blue"]] * len(labels) + [OKABE_ITO["gray"]]
    ax.errorbar(xpos, y_all, yerr=[ylow_all, yhigh_all], fmt="o", capsize=3, color=OKABE_ITO["black"])
    ax.scatter(xpos, y_all, s=35, c=colors, zorder=3)
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels_all, rotation=22, ha="right")
    ax.set_ylabel("Macro F1 on held-out test set")
    ax.set_title("Condition recoverability is above shuffled control")
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "a", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[0, 1])
    gdf = step5b["summary_df"].copy().sort_values("surrogate_condition_hit_rate", ascending=False).reset_index(drop=True)
    xpos = np.arange(len(gdf))
    for i, row in gdf.iterrows():
        ax.errorbar(
            i,
            row["surrogate_condition_hit_rate"],
            yerr=[[row["surrogate_condition_hit_rate"] - row["surrogate_condition_hit_rate_ci_low"]], [row["surrogate_condition_hit_rate_ci_high"] - row["surrogate_condition_hit_rate"]]],
            fmt="o",
            color=MODEL_COLORS.get(row["generator"], OKABE_ITO["gray"]),
            capsize=3,
        )
    ax.set_xticks(xpos)
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in gdf["generator"]], rotation=22, ha="right")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_title("Conditional generators improve surrogate fidelity")
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "b", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    for _, row in gdf.iterrows():
        ax.scatter(
            row["exact_novelty_vs_train"],
            row["surrogate_condition_hit_rate"],
            s=55,
            color=MODEL_COLORS.get(row["generator"], OKABE_ITO["gray"]),
            edgecolor="black",
            linewidth=0.4,
        )
        ax.text(
            row["exact_novelty_vs_train"] + 0.004,
            row["surrogate_condition_hit_rate"] + 0.004,
            MODEL_DISPLAY_NAMES.get(row["generator"], row["generator"]),
            fontsize=6.5,
        )
    ax.set_xlabel("Exact novelty versus training set")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_title("Fidelity–novelty tradeoff")
    ax.grid(alpha=0.25)
    add_panel_label(ax, "c", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 1])
    per_cond = step5b["per_condition_df"].copy()
    order = list(gdf["generator"])
    data_plot = [per_cond.loc[per_cond["generator"] == g, "mean_nn_similarity_to_train"].astype(float).values for g in order]
    parts = ax.violinplot(data_plot, positions=np.arange(1, len(order) + 1), showmeans=False, showmedians=True, showextrema=False)
    for i, body in enumerate(parts["bodies"]):
        body.set_facecolor(MODEL_COLORS.get(order[i], OKABE_ITO["gray"]))
        body.set_edgecolor("black")
        body.set_alpha(0.60)
    if "cmedians" in parts:
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(1.0)
    ax.set_xticks(np.arange(1, len(order) + 1))
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=22, ha="right")
    ax.set_ylabel("Nearest-neighbour similarity to training set")
    ax.set_title("Similarity-to-train distribution")
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "d", cfg.figure_export.panel_fontsize)

    export_figure(fig, dirs["figures_main"] / "Fig_5_step5_baseline_benchmark", cfg.figure_export)


def make_step5_supplementary_figures(step5a: Dict[str, Any], step5b: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> None:
    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 7.4))
    gs = GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    split_counts = pd.concat(
        [
            step5a["train_df_with_desc"][cfg.target_col].value_counts().rename("train"),
            step5a["val_df_with_desc"][cfg.target_col].value_counts().rename("val"),
            step5a["test_df_with_desc"][cfg.target_col].value_counts().rename("test"),
        ],
        axis=1,
    ).fillna(0)
    split_counts = split_counts.loc[split_counts.sum(axis=1).sort_values(ascending=False).index]
    sub = split_counts.head(cfg.max_unique_conditions_for_figures)
    x = np.arange(len(sub))
    width = 0.25
    ax.bar(x - width, sub["train"], width=width, label="train", color=OKABE_ITO["blue"])
    ax.bar(x, sub["val"], width=width, label="val", color=OKABE_ITO["orange"])
    ax.bar(x + width, sub["test"], width=width, label="test", color=OKABE_ITO["green"])
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index.astype(str), rotation=25, ha="right")
    ax.set_ylabel("Sequences")
    ax.set_title("Condition support across frozen splits")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "a", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[0, 1])
    cm = step5a["confusion_matrix_norm"].values
    im = ax.imshow(cm, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    labels = list(step5a["confusion_matrix_norm"].index.astype(str))
    if len(labels) > 12:
        show_idx = np.arange(min(12, len(labels)))
        ax.set_xticks(show_idx)
        ax.set_yticks(show_idx)
        ax.set_xticklabels(labels[: len(show_idx)], rotation=45, ha="right")
        ax.set_yticklabels(labels[: len(show_idx)])
    else:
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_yticklabels(labels)
    ax.set_title("Normalized confusion matrix")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    add_panel_label(ax, "b", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    perm_df = step5a["perm_df"]
    if perm_df is not None and len(perm_df) > 0:
        top = perm_df.head(12).sort_values("importance_mean")
        ax.barh(np.arange(len(top)), top["importance_mean"], xerr=top["importance_std"], color=OKABE_ITO["orange"])
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels(top["feature"])
        ax.set_xlabel("Permutation importance (macro F1)")
        ax.set_title("Top descriptor contributors")
    else:
        ax.text(0.5, 0.5, "Permutation importance unavailable", ha="center", va="center")
        ax.set_axis_off()
    add_panel_label(ax, "c", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 1])
    pca_df = step5a["pca_df"].copy()
    top_conditions = pca_df["condition"].value_counts().head(cfg.max_unique_conditions_for_figures).index
    sub = pca_df[pca_df["condition"].isin(top_conditions)]
    for cond, g in sub.groupby("condition"):
        ax.scatter(g["pc1"], g["pc2"], s=12, alpha=0.7, label=str(cond))
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Test-set projection in computable descriptor space")
    ax.legend(fontsize=6, ncol=2, frameon=False)
    ax.grid(alpha=0.25)
    add_panel_label(ax, "d", cfg.figure_export.panel_fontsize)

    export_figure(fig, dirs["figures_supplementary"] / "Fig_S5A_step5a_diagnostics", cfg.figure_export)

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 7.4))
    gs = GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)
    summary = step5b["summary_df"].copy().sort_values("surrogate_condition_hit_rate", ascending=False).reset_index(drop=True)
    per_cond = step5b["per_condition_df"].copy()
    candidate = step5b["candidate_df"].copy()
    order = list(summary["generator"])
    x = np.arange(len(order))

    ax = fig.add_subplot(gs[0, 0])
    width = 0.22
    metric_map = [
        ("valid_rate", "Validity"),
        ("exact_novelty_vs_train", "Novelty"),
        ("unique_rate_among_valid", "Uniqueness"),
    ]
    for i, (col, lbl) in enumerate(metric_map):
        vals = [summary.loc[summary["generator"] == g, col].iloc[0] for g in order]
        ax.bar(x + (i - 1) * width, vals, width=width, label=lbl)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=22, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title("Validity, novelty, and uniqueness")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "a", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[0, 1])
    data_plot = [candidate.loc[candidate["generator"] == g, "nn_similarity_to_train"].astype(float).values for g in order]
    parts = ax.violinplot(data_plot, positions=np.arange(1, len(order) + 1), showmeans=False, showmedians=True, showextrema=False)
    for i, body in enumerate(parts["bodies"]):
        body.set_facecolor(MODEL_COLORS.get(order[i], OKABE_ITO["gray"]))
        body.set_edgecolor("black")
        body.set_alpha(0.60)
    if "cmedians" in parts:
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(1.0)
    ax.set_xticks(np.arange(1, len(order) + 1))
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=22, ha="right")
    ax.set_ylabel("Per-sequence NN similarity to training set")
    ax.set_title("Per-sequence memorization profile")
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "b", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    for i, g in enumerate(order):
        vals = per_cond.loc[per_cond["generator"] == g, "surrogate_condition_hit_rate"].astype(float).values
        ax.scatter(np.repeat(i, len(vals)), vals, color=MODEL_COLORS.get(g, OKABE_ITO["gray"]), s=16, alpha=0.7)
        mean_val = np.mean(vals) if len(vals) else np.nan
        ax.hlines(mean_val, i - 0.25, i + 0.25, colors="black", linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in order], rotation=22, ha="right")
    ax.set_ylabel("Per-condition surrogate hit rate")
    ax.set_title("Condition-level heterogeneity")
    ax.grid(alpha=0.25, axis="y")
    add_panel_label(ax, "c", cfg.figure_export.panel_fontsize)

    ax = fig.add_subplot(gs[1, 1])
    keep = ["surrogate_condition_hit_rate_sd", "exact_novelty_vs_train_sd", "pairwise_diversity_sd"]
    heat = summary[["generator"] + keep].set_index("generator")
    im = ax.imshow(heat.values, aspect="auto", cmap="Oranges")
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels([MODEL_DISPLAY_NAMES.get(g, g) for g in heat.index])
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(["Fidelity SD", "Novelty SD", "Diversity SD"], rotation=22, ha="right")
    ax.set_title("Seed-to-seed variability")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    add_panel_label(ax, "d", cfg.figure_export.panel_fontsize)

    export_figure(fig, dirs["figures_supplementary"] / "Fig_S5B_step5b_diagnostics", cfg.figure_export)


def write_step5_report(step5a: Dict[str, Any], step5b: Dict[str, Any], qc: Dict[str, Any], cfg: Step5Config, dirs: Dict[str, Path]) -> None:
    a = step5a["summary_df"].copy()
    b = step5b["summary_df"].copy().sort_values(["surrogate_condition_hit_rate", "exact_novelty_vs_train"], ascending=False).reset_index(drop=True)
    best_a = a.iloc[0]
    best_b = b.iloc[0]

    lines = []
    lines.append("Step 5 report — descriptor audit and baseline generator benchmark\n\n")
    lines.append("Input and contract audit\n")
    lines.append(f"Timestamp: {now_timestamp()}\n")
    lines.append(f"Total Step 4 rows: {qc['n_total_rows']}\n")
    lines.append(f"Valid cleaned sequences: {qc['n_valid_clean_sequences']}\n")
    lines.append(f"Invalid after cleaning: {qc['n_invalid_after_cleaning']}\n\n")

    lines.append("Step 5A — descriptor audit\n")
    lines.append(f"Best descriptor block: {best_a['block']} using {best_a['family']}\n")
    lines.append(f"Locked-test macro F1: {best_a['macro_f1']:.3f} [{best_a['macro_f1_ci_low']:.3f}, {best_a['macro_f1_ci_high']:.3f}]\n")
    lines.append(f"Locked-test balanced accuracy: {best_a['balanced_acc']:.3f}\n")
    lines.append(f"Mean shuffled-label macro F1: {step5a['shuffle_df']['macro_f1'].mean():.3f}\n\n")

    lines.append("Step 5B — generator benchmark\n")
    lines.append(f"Top generator by surrogate fidelity: {MODEL_DISPLAY_NAMES.get(best_b['generator'], best_b['generator'])}\n")
    lines.append(f"Surrogate condition-hit rate: {best_b['surrogate_condition_hit_rate']:.3f} [{best_b['surrogate_condition_hit_rate_ci_low']:.3f}, {best_b['surrogate_condition_hit_rate_ci_high']:.3f}]\n")
    lines.append(f"Exact novelty vs train: {best_b['exact_novelty_vs_train']:.3f} [{best_b['exact_novelty_vs_train_ci_low']:.3f}, {best_b['exact_novelty_vs_train_ci_high']:.3f}]\n")
    lines.append(f"Pairwise diversity: {best_b['pairwise_diversity']:.3f} [{best_b['pairwise_diversity_ci_low']:.3f}, {best_b['pairwise_diversity_ci_high']:.3f}]\n")
    lines.append(f"Support rate: {best_b['support_rate']:.3f} [{best_b['support_rate_ci_low']:.3f}, {best_b['support_rate_ci_high']:.3f}]\n")
    lines.append(f"Low-complexity rate: {best_b['low_complexity_rate']:.3f} [{best_b['low_complexity_rate_ci_low']:.3f}, {best_b['low_complexity_rate_ci_high']:.3f}]\n")

    (dirs["reports"] / "step5_summary_report.txt").write_text("".join(lines), encoding="utf-8")

    methods = (
        "Step 5 comprised a descriptor-based audit benchmark (Step 5A) and a frozen-split baseline generator benchmark (Step 5B). "
        "In Step 5A, we benchmarked multiclass predictors of primary_condition_id from peptide descriptors using the fixed Step 4 train/validation/test partition, "
        "with feature blocks spanning the full numeric descriptor space, a computable descriptor subset, amino-acid composition alone, and physicochemical descriptors alone. "
        "Candidate surrogate families included a majority dummy baseline, multinomial logistic regression, random forest, and extra-trees classifiers. "
        "Model-family selection used train-only inner cross-validation and validation macro F1, followed by refitting on train+validation and locked-test evaluation with bootstrap confidence intervals. "
        "We additionally ran repeated shuffled-label controls, exported normalized confusion matrices, estimated permutation importance, and projected the computable descriptor space by PCA. "
        "In Step 5B, we benchmarked same-condition retrieval reference sampling, a random conditional baseline, an unconditional GRU language model, an unconditional sequence VAE, a conditional GRU, and a conditional VAE, all trained on the train split only and evaluated under matched generation budgets across requested conditions and repeated random seeds. "
        "Generated peptides were assessed using surrogate condition-hit rate and surrogate target probability as proxy controllability metrics, alongside exact novelty versus the training split and full corpus, nearest-neighbour 3-mer Jaccard similarity to the train split and same-condition train subset, pairwise diversity, within-set duplication, descriptor-space support, low-complexity rate, and basic physicochemical summary statistics."
    )
    (dirs["reports"] / "step5_methods_paragraph.txt").write_text(methods, encoding="utf-8")


def run_step5_pipeline(cfg: Step5Config) -> Dict[str, Any]:
    seed_all(cfg.main_random_seed)
    project_root = project_root_from_input(cfg.input_model_ready_csv)
    out_root = project_root / cfg.out_dir_name
    dirs = make_output_dirs(out_root)

    save_json(asdict(cfg), dirs["artifacts"] / "step5_config.json")
    save_json(environment_manifest(), dirs["artifacts"] / "environment_manifest.json")
    save_json(
        {
            "input_model_ready_csv": cfg.input_model_ready_csv,
            "input_baseline_features_csv": cfg.input_baseline_features_csv,
            "sha256_model_ready": sha256_of_file(Path(cfg.input_model_ready_csv)),
            "sha256_baseline_features": sha256_of_file(Path(cfg.input_baseline_features_csv)),
        },
        dirs["artifacts"] / "input_manifest.json",
    )

    data = load_step4_inputs(cfg)
    qc = validate_step4_contract(data["seq_df"], data["feat_df"], cfg)
    save_json(qc, dirs["artifacts"] / "step4_contract_qc.json")

    step5a = run_step5a_descriptor_audit(data, cfg, dirs)
    step5b = run_step5b_generator_benchmark(data, step5a, cfg, dirs)

    make_step5_main_figure(step5a, step5b, cfg, dirs)
    make_step5_supplementary_figures(step5a, step5b, cfg, dirs)
    write_step5_report(step5a, step5b, qc, cfg, dirs)

    artifact_df = manifest_from_outputs(out_root)
    artifact_df.to_csv(dirs["artifacts"] / "artifact_manifest.csv", index=False)

    return {
        "project_root": str(project_root),
        "out_root": str(out_root),
        "step5a_summary": step5a["summary_df"],
        "step5b_summary": step5b["summary_df"],
        "artifact_manifest": artifact_df,
    }


def main_notebook(
    input_model_ready_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv",
    input_baseline_features_csv: str = "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_baseline_features.csv",
    out_dir_name: str = "step5",
    fast_mode: bool = False,
    n_generated_per_condition: int = 100,
    repeated_seeds: Tuple[int, ...] = (42, 52, 62),
) -> Dict[str, Any]:
    cfg = Step5Config(
        input_model_ready_csv=input_model_ready_csv,
        input_baseline_features_csv=input_baseline_features_csv,
        out_dir_name=out_dir_name,
        fast_mode=fast_mode,
        n_generated_per_condition=n_generated_per_condition,
        repeated_seeds=repeated_seeds,
    )
    return run_step5_pipeline(cfg)


if __name__ == "__main__":
    results = main_notebook()
    print("Step 5 completed.")
    print(f"Output directory: {results['out_root']}")

redeisgn step 5 figures:

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =============================================================================
# User paths
# =============================================================================
DEFAULT_STEP5_ROOT = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5"
)
DEFAULT_STEP4_MODEL_READY = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv"
)


# =============================================================================
# Style constants
# =============================================================================
MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "retrieval_reference": "Retrieval reference",
    "cvae_conditional": "CVAE (conditional)",
    "gru_conditional": "GRU (conditional)",
    "random_conditional": "Random conditional",
    "gru_unconditional": "GRU (unconditional)",
    "vae_unconditional": "VAE (unconditional)",
}

MODEL_COLORS = {
    "retrieval_reference": "#56B4E9",
    "cvae_conditional": "#009E73",
    "gru_conditional": "#CC79A7",
    "random_conditional": "#7F7F7F",
    "gru_unconditional": "#0072B2",
    "vae_unconditional": "#E69F00",
}

DESCRIPTOR_ORDER = [
    "Full numeric descriptor set",
    "Computable descriptors",
    "Physicochemical only",
    "AA composition only",
    "Shuffled-label control",
]

DESCRIPTOR_LABELS = {
    "Full numeric descriptor set": "Full numeric",
    "Computable descriptors": "Computable",
    "Physicochemical only": "Physicochemical",
    "AA composition only": "AA composition",
    "Shuffled-label control": "Shuffle control",
}

DESCRIPTOR_COLORS = {
    "Full numeric descriptor set": "#1f77b4",
    "Computable descriptors": "#377eb8",
    "Physicochemical only": "#6baed6",
    "AA composition only": "#9ecae1",
    "Shuffled-label control": "#7F7F7F",
}


# =============================================================================
# Config
# =============================================================================
@dataclass(frozen=True)
class FigureRedesignConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    step4_model_ready_csv: Path = DEFAULT_STEP4_MODEL_READY
    main_name: str = "fig_5_step5_main_benchmark_redesign"
    supp1_name: str = "fig_s5a_step5_descriptor_audit_redesign"
    supp2_name: str = "fig_s5b_step5_generator_diagnostics_redesign"
    export_png: bool = True
    export_pdf: bool = True
    dpi: int = 300


# =============================================================================
# Global style
# =============================================================================
def apply_global_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.titlesize": 12,
            "axes.labelsize": 11,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 9,
            "figure.titlesize": 13,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": False,
            "savefig.dpi": 300,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


# =============================================================================
# I/O discovery
# =============================================================================
def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None



def discover_step5_inputs(cfg: FigureRedesignConfig) -> Dict[str, Optional[Path]]:
    root = cfg.step5_root
    if not root.exists():
        raise FileNotFoundError(f"Step 5 root does not exist: {root}")

    model_ready_path: Optional[Path] = None
    if cfg.step4_model_ready_csv.exists():
        model_ready_path = cfg.step4_model_ready_csv
    else:
        model_ready_path = _find_first(
            root,
            [
                "step4_model_ready_sequences.csv",
                "../step4/tables/step4_model_ready_sequences.csv",
                "../../step4/tables/step4_model_ready_sequences.csv",
            ],
        )

    inputs = {
        "descriptor_benchmark": _find_first(root, ["table_5_1_step5a_descriptor_benchmark.csv"]),
        "generator_benchmark": _find_first(root, ["table_5_2_step5b_generator_benchmark.csv"]),
        "training_summary": _find_first(root, ["table_s5_3_generator_training_summary.csv"]),
        "per_condition_metrics": _find_first(root, ["table_s5_4_step5b_per_condition_metrics.csv"]),
        "candidate_metrics": _find_first(root, ["step5b_generated_candidates_evaluated.csv"]),
        "confusion_matrix_norm": _find_first(root, ["step5a_confusion_matrix_normalized.csv"]),
        "permutation_importance": _find_first(root, ["step5a_permutation_importance.csv"]),
        "pca_projection": _find_first(root, ["step5a_pca_projection.csv"]),
        "model_ready_sequences": model_ready_path,
    }

    missing = [
        key for key in ("descriptor_benchmark", "generator_benchmark") if inputs[key] is None
    ]
    if missing:
        raise FileNotFoundError(
            "Missing required Step 5 files: " + ", ".join(missing)
        )
    return inputs



def load_csv_if_exists(path: Optional[Path]) -> Optional[pd.DataFrame]:
    if path is None or not path.exists():
        return None
    return pd.read_csv(path)



def load_step5_tables(cfg: FigureRedesignConfig) -> Dict[str, Optional[pd.DataFrame]]:
    paths = discover_step5_inputs(cfg)
    tables: Dict[str, Optional[pd.DataFrame]] = {
        key: load_csv_if_exists(path) for key, path in paths.items()
    }
    tables["_paths"] = paths  # type: ignore[assignment]
    return tables


# =============================================================================
# Helpers
# =============================================================================
def ensure_output_dirs(step5_root: Path) -> Dict[str, Path]:
    out_main = step5_root / "figures_main"
    out_supp = step5_root / "figures_supplementary"
    out_main.mkdir(parents=True, exist_ok=True)
    out_supp.mkdir(parents=True, exist_ok=True)
    return {"main": out_main, "supp": out_supp}



def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="bottom",
        ha="left",
    )



def style_axis(ax: plt.Axes, ygrid: bool = True) -> None:
    if ygrid:
        ax.yaxis.grid(True, color="#d9d9d9", linewidth=0.8, alpha=0.8)
    else:
        ax.grid(False)
    ax.set_axisbelow(True)



def rename_generator_display(values: Iterable[str]) -> list[str]:
    return [MODEL_LABELS.get(v, v) for v in values]



def rename_descriptor_display(values: Iterable[str]) -> list[str]:
    return [DESCRIPTOR_LABELS.get(v, v) for v in values]



def safe_label_offsets(yvals: Sequence[float], magnitude: float = 0.012) -> Dict[int, float]:
    offsets: Dict[int, float] = {}
    used: list[float] = []
    for i, y in enumerate(yvals):
        offset = magnitude
        for prev in used:
            if abs((y + offset) - prev) < 0.03:
                offset += magnitude
        offsets[i] = offset
        used.append(y + offset)
    return offsets



def clean_feature_name(name: str) -> str:
    mapping = {
        "net_charge_proxy": "Net charge proxy",
        "net_charge_pH7": "Net charge (pH 7)",
        "net_charge_pH7_z": "Net charge (pH 7, z)",
        "mean_hydropathy": "Mean hydropathy",
        "hydrophobicity_KD": "Hydropathy (Kyte–Doolittle)",
        "hydrophobicity_KD_z": "Hydropathy (KD, z)",
        "sequence_entropy": "Sequence entropy",
        "entropy_z": "Sequence entropy (z)",
        "dominant_aa_frac_z": "Dominant residue fraction",
        "length": "Length",
        "length_z": "Length (z)",
        "primary_condition_id_idx": "Condition index",
        "aa_frac_Y_z": "Tyr fraction (z)",
        "aa_frac_W_z": "Trp fraction (z)",
        "aa_frac_V_z": "Val fraction (z)",
        "mean_length": "Mean length",
    }
    if name in mapping:
        return mapping[name]
    name = name.replace("aa_frac_", "").replace("_z", " (z)")
    name = name.replace("_", " ")
    return name.title()



def require_columns(df: pd.DataFrame, required: Sequence[str], name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")



def resolve_column(df: pd.DataFrame, candidates: Sequence[str], *, required: bool = True) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"None of the candidate columns were found: {candidates}")
    return None



def point_ci_plot(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    low: np.ndarray,
    high: np.ndarray,
    colors: Sequence[str],
    markersize: float = 8.0,
) -> None:
    for xi, yi, lo, hi, color in zip(x, y, low, high, colors):
        ax.errorbar(
            xi,
            yi,
            yerr=[[yi - lo], [hi - yi]],
            fmt="o",
            color=color,
            ecolor=color,
            elinewidth=1.8,
            capsize=4,
            markersize=markersize,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=3,
        )



def save_figure(fig: plt.Figure, output_base: Path, cfg: FigureRedesignConfig) -> None:
    if cfg.export_pdf:
        fig.savefig(output_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(output_base.with_suffix(".png"), bbox_inches="tight", dpi=cfg.dpi)
    plt.close(fig)


# =============================================================================
# Figure builders
# =============================================================================
def make_main_figure(
    tables: Dict[str, Optional[pd.DataFrame]],
    output_path: Path,
    cfg: FigureRedesignConfig,
) -> None:
    desc = tables["descriptor_benchmark"]
    gen = tables["generator_benchmark"]
    cand = tables.get("candidate_metrics")

    assert desc is not None and gen is not None
    desc = desc.copy()
    gen = gen.copy()
    cand = None if cand is None else cand.copy()

    require_columns(
        desc,
        ["block", "macro_f1", "macro_f1_ci_low", "macro_f1_ci_high"],
        "descriptor benchmark",
    )
    require_columns(
        gen,
        [
            "generator",
            "surrogate_condition_hit_rate",
            "surrogate_condition_hit_rate_ci_low",
            "surrogate_condition_hit_rate_ci_high",
            "exact_novelty_vs_train",
            "exact_novelty_vs_train_ci_low",
            "exact_novelty_vs_train_ci_high",
            "mean_nn_similarity_to_train",
            "mean_nn_similarity_to_train_ci_low",
            "mean_nn_similarity_to_train_ci_high",
        ],
        "generator benchmark",
    )

    desc = desc[desc["block"].isin(DESCRIPTOR_ORDER)].copy()
    desc["block"] = pd.Categorical(desc["block"], categories=DESCRIPTOR_ORDER, ordered=True)
    desc = desc.sort_values("block").reset_index(drop=True)

    gen = gen[gen["generator"].isin(MODEL_ORDER)].copy()
    gen["generator"] = pd.Categorical(gen["generator"], categories=MODEL_ORDER, ordered=True)
    gen = gen.sort_values("generator").reset_index(drop=True)

    fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.2), constrained_layout=True)

    # a
    ax = axes[0, 0]
    x = np.arange(len(desc))
    point_ci_plot(
        ax,
        x=x,
        y=desc["macro_f1"].to_numpy(float),
        low=desc["macro_f1_ci_low"].to_numpy(float),
        high=desc["macro_f1_ci_high"].to_numpy(float),
        colors=[DESCRIPTOR_COLORS[b] for b in desc["block"]],
    )
    ax.set_xticks(x)
    ax.set_xticklabels(rename_descriptor_display(desc["block"].astype(str)), rotation=22, ha="right")
    ax.set_ylabel("Macro F1 on held-out test set")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Condition recoverability exceeds shuffled control", pad=10)
    style_axis(ax)
    panel_label(ax, "a")

    # b
    ax = axes[0, 1]
    x = np.arange(len(gen))
    point_ci_plot(
        ax,
        x=x,
        y=gen["surrogate_condition_hit_rate"].to_numpy(float),
        low=gen["surrogate_condition_hit_rate_ci_low"].to_numpy(float),
        high=gen["surrogate_condition_hit_rate_ci_high"].to_numpy(float),
        colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
    )
    ax.set_xticks(x)
    ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_ylim(0.0, max(0.70, float(gen["surrogate_condition_hit_rate_ci_high"].max()) + 0.05))
    ax.set_title("Conditional generators improve surrogate fidelity", pad=10)
    style_axis(ax)
    panel_label(ax, "b")

    # c
    ax = axes[1, 0]
    offsets = safe_label_offsets(gen["surrogate_condition_hit_rate"].to_numpy(float), magnitude=0.010)
    for i, row in gen.iterrows():
        generator = str(row["generator"])
        ax.errorbar(
            row["exact_novelty_vs_train"],
            row["surrogate_condition_hit_rate"],
            xerr=[[row["exact_novelty_vs_train"] - row["exact_novelty_vs_train_ci_low"]],
                  [row["exact_novelty_vs_train_ci_high"] - row["exact_novelty_vs_train"]]],
            yerr=[[row["surrogate_condition_hit_rate"] - row["surrogate_condition_hit_rate_ci_low"]],
                  [row["surrogate_condition_hit_rate_ci_high"] - row["surrogate_condition_hit_rate"]]],
            fmt="o",
            color=MODEL_COLORS[generator],
            ecolor=MODEL_COLORS[generator],
            capsize=3,
            elinewidth=1.6,
            markersize=8,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=3,
        )
        ax.text(
            float(row["exact_novelty_vs_train"]) + 0.01,
            float(row["surrogate_condition_hit_rate"]) + offsets[i],
            MODEL_LABELS[generator],
            fontsize=9,
            color="#222222",
        )
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(0.0, max(0.68, float(gen["surrogate_condition_hit_rate_ci_high"].max()) + 0.05))
    ax.set_xlabel("Exact novelty versus training set")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_title("Fidelity–novelty tradeoff", pad=10)
    style_axis(ax)
    panel_label(ax, "c")

    # d
    ax = axes[1, 1]
    if cand is not None and {"generator", "nn_similarity_to_train"}.issubset(cand.columns):
        datasets: list[np.ndarray] = []
        positions: list[int] = []
        colors: list[str] = []
        for i, generator in enumerate(MODEL_ORDER, start=1):
            values = pd.to_numeric(
                cand.loc[cand["generator"] == generator, "nn_similarity_to_train"],
                errors="coerce",
            ).dropna().to_numpy()
            if len(values) == 0:
                continue
            datasets.append(values)
            positions.append(i)
            colors.append(MODEL_COLORS[generator])

        violin = ax.violinplot(
            datasets,
            positions=positions,
            widths=0.82,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body, color in zip(violin["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("#444444")
            body.set_alpha(0.45)
            body.set_linewidth(0.8)

        box = ax.boxplot(
            datasets,
            positions=positions,
            widths=0.26,
            patch_artist=True,
            showfliers=False,
            medianprops={"color": "#222222", "linewidth": 1.5},
            boxprops={"linewidth": 1.0},
            whiskerprops={"linewidth": 0.9},
            capprops={"linewidth": 0.9},
        )
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor("white")
            patch.set_edgecolor(color)
            patch.set_alpha(0.95)

        shown_generators = [g for g in MODEL_ORDER if len(pd.to_numeric(cand.loc[cand['generator'] == g, 'nn_similarity_to_train'], errors='coerce').dropna()) > 0]
        ax.set_xticks(range(1, len(shown_generators) + 1))
        ax.set_xticklabels(rename_generator_display(shown_generators), rotation=25, ha="right")
    else:
        point_ci_plot(
            ax,
            x=np.arange(len(gen)),
            y=gen["mean_nn_similarity_to_train"].to_numpy(float),
            low=gen["mean_nn_similarity_to_train_ci_low"].to_numpy(float),
            high=gen["mean_nn_similarity_to_train_ci_high"].to_numpy(float),
            colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
        )
        ax.set_xticks(np.arange(len(gen)))
        ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Nearest-neighbour similarity to training set")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Similarity-to-train distribution", pad=10)
    style_axis(ax)
    panel_label(ax, "d")

    save_figure(fig, output_path, cfg)



def make_supplementary_figure_s5a(
    tables: Dict[str, Optional[pd.DataFrame]],
    output_path: Path,
    cfg: FigureRedesignConfig,
) -> None:
    model_ready = tables.get("model_ready_sequences")
    confusion = tables.get("confusion_matrix_norm")
    perm = tables.get("permutation_importance")
    pca = tables.get("pca_projection")

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.8), constrained_layout=True)

    # a support heatmap
    ax = axes[0, 0]
    if model_ready is not None and {"split", "primary_condition_id"}.issubset(model_ready.columns):
        support = (
            model_ready.groupby(["primary_condition_id", "split"]).size().unstack(fill_value=0)
        )
        split_order = [s for s in ["train", "val", "test"] if s in support.columns]
        support = support.reindex(columns=split_order)
        support = support.sort_values(split_order[0], ascending=False)
        im = ax.imshow(support.to_numpy(), aspect="auto", cmap="Blues")
        ax.set_xticks(np.arange(support.shape[1]))
        ax.set_xticklabels(split_order)
        n_rows = support.shape[0]
        if n_rows <= 18:
            yticks = np.arange(n_rows)
        else:
            yticks = np.arange(0, n_rows, max(1, n_rows // 12))
        ax.set_yticks(yticks)
        ax.set_yticklabels(support.index[yticks])
        ax.set_ylabel("Condition")
        ax.set_title("Condition support across frozen splits", pad=10)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Count", rotation=90)
    else:
        ax.text(0.5, 0.5, "Model-ready split table unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "a")

    # b confusion matrix
    ax = axes[0, 1]
    if confusion is not None:
        if np.issubdtype(confusion.iloc[:, 0].dtype, np.number):
            arr = confusion.to_numpy(float)
            row_labels = [str(i) for i in range(arr.shape[0])]
            col_labels = [str(i) for i in range(arr.shape[1])]
        else:
            arr = confusion.iloc[:, 1:].to_numpy(float)
            row_labels = confusion.iloc[:, 0].astype(str).tolist()
            col_labels = confusion.columns[1:].astype(str).tolist()
        im = ax.imshow(arr, aspect="equal", cmap="Blues", vmin=0.0, vmax=1.0)
        n = arr.shape[0]
        ticks = np.arange(0, n, max(1, n // 10))
        ax.set_xticks(ticks)
        ax.set_xticklabels([col_labels[i] for i in ticks], rotation=45, ha="right")
        ax.set_yticks(ticks)
        ax.set_yticklabels([row_labels[i] for i in ticks])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Normalized confusion matrix", pad=10)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Row-normalized fraction", rotation=90)
    else:
        ax.text(0.5, 0.5, "Confusion matrix unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "b")

    # c permutation importance
    ax = axes[1, 0]
    if perm is not None:
        feature_col = resolve_column(perm, ["feature", "descriptor", "variable"]) or perm.columns[0]
        value_col = resolve_column(
            perm,
            ["importance", "permutation_importance", "macro_f1_importance", "value"],
            required=False,
        )
        if value_col is None:
            numeric = perm.select_dtypes(include=[np.number]).columns.tolist()
            if not numeric:
                raise ValueError("Permutation importance file contains no numeric columns.")
            value_col = numeric[0]
        plot_df = perm[[feature_col, value_col]].copy().sort_values(value_col, ascending=False).head(10).iloc[::-1]
        ax.barh(
            np.arange(len(plot_df)),
            plot_df[value_col].to_numpy(float),
            color="#D8A12A",
            edgecolor="#8C6D1F",
            linewidth=0.6,
        )
        ax.set_yticks(np.arange(len(plot_df)))
        ax.set_yticklabels([clean_feature_name(str(x)) for x in plot_df[feature_col]])
        ax.set_xlabel("Permutation importance (macro F1)")
        ax.set_title("Top descriptor contributors", pad=10)
        style_axis(ax)
    else:
        ax.text(0.5, 0.5, "Permutation importance unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "c")

    # d PCA structure
    ax = axes[1, 1]
    if pca is not None and {"PC1", "PC2"}.issubset(pca.columns):
        cond_col = resolve_column(pca, ["primary_condition_id", "condition", "label"], required=False)
        if cond_col is None:
            non_pc = [c for c in pca.columns if c not in {"PC1", "PC2"}]
            cond_col = non_pc[0] if non_pc else None
        if cond_col is not None:
            top_conditions = pca[cond_col].astype(str).value_counts().head(8).index.tolist()
            sub = pca[pca[cond_col].astype(str).isin(top_conditions)].copy()
            palette = plt.cm.tab10(np.linspace(0, 1, len(top_conditions)))
            for color, cond in zip(palette, top_conditions):
                tmp = sub[sub[cond_col].astype(str) == cond]
                ax.scatter(tmp["PC1"], tmp["PC2"], s=14, alpha=0.45, color=color, label=cond, edgecolors="none")
            ax.legend(frameon=False, fontsize=7, ncol=2, loc="best")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            ax.set_title("Descriptor-space structure (top conditions)", pad=10)
            style_axis(ax)
        else:
            ax.text(0.5, 0.5, "PCA condition labels unavailable", ha="center", va="center")
            ax.set_axis_off()
    else:
        ax.text(0.5, 0.5, "PCA projection unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "d")

    save_figure(fig, output_path, cfg)



def make_supplementary_figure_s5b(
    tables: Dict[str, Optional[pd.DataFrame]],
    output_path: Path,
    cfg: FigureRedesignConfig,
) -> None:
    gen = tables["generator_benchmark"]
    per_cond = tables.get("per_condition_metrics")
    cand = tables.get("candidate_metrics")
    train_sum = tables.get("training_summary")

    assert gen is not None
    gen = gen.copy()
    gen = gen[gen["generator"].isin(MODEL_ORDER)].copy()
    gen["generator"] = pd.Categorical(gen["generator"], categories=MODEL_ORDER, ordered=True)
    gen = gen.sort_values("generator").reset_index(drop=True)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.8), constrained_layout=True)

    # a quality triad
    ax = axes[0, 0]
    metrics = [
        ("valid_rate", "Validity", "o"),
        ("exact_novelty_vs_train", "Novelty", "s"),
        ("unique_rate_among_valid", "Uniqueness", "^"),
    ]
    x = np.arange(len(gen))
    for j, (metric_col, label, marker) in enumerate(metrics):
        require_columns(gen, [metric_col, f"{metric_col}_ci_low", f"{metric_col}_ci_high"], "generator benchmark")
        xj = x + (j - 1) * 0.10
        for xi, generator, y, lo, hi in zip(
            xj,
            gen["generator"].astype(str),
            gen[metric_col].to_numpy(float),
            gen[f"{metric_col}_ci_low"].to_numpy(float),
            gen[f"{metric_col}_ci_high"].to_numpy(float),
        ):
            ax.errorbar(
                xi,
                y,
                yerr=[[y - lo], [hi - y]],
                fmt=marker,
                color=MODEL_COLORS[generator],
                ecolor=MODEL_COLORS[generator],
                capsize=3,
                markersize=6.5,
                markeredgecolor="white",
                markeredgewidth=0.7,
            )
    ax.set_xticks(x)
    ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Rate")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Validity, novelty, and uniqueness", pad=10)
    handles = [Line2D([0], [0], marker=m, color="black", linestyle="None", markersize=6, label=l) for _, l, m in metrics]
    ax.legend(handles=handles, frameon=False, loc="lower right")
    style_axis(ax)
    panel_label(ax, "a")

    # b memorization profile
    ax = axes[0, 1]
    if cand is not None and {"generator", "nn_similarity_to_train"}.issubset(cand.columns):
        datasets: list[np.ndarray] = []
        positions: list[int] = []
        colors: list[str] = []
        shown_generators: list[str] = []
        for i, generator in enumerate(MODEL_ORDER, start=1):
            values = pd.to_numeric(
                cand.loc[cand["generator"] == generator, "nn_similarity_to_train"],
                errors="coerce",
            ).dropna().to_numpy()
            if len(values) == 0:
                continue
            datasets.append(values)
            positions.append(i)
            colors.append(MODEL_COLORS[generator])
            shown_generators.append(generator)
        violin = ax.violinplot(
            datasets,
            positions=positions,
            widths=0.82,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body, color in zip(violin["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("#444444")
            body.set_alpha(0.45)
            body.set_linewidth(0.8)
        box = ax.boxplot(
            datasets,
            positions=positions,
            widths=0.26,
            patch_artist=True,
            showfliers=False,
            medianprops={"color": "#222222", "linewidth": 1.5},
            boxprops={"linewidth": 1.0},
            whiskerprops={"linewidth": 0.9},
            capprops={"linewidth": 0.9},
        )
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor("white")
            patch.set_edgecolor(color)
        ax.set_xticks(positions)
        ax.set_xticklabels(rename_generator_display(shown_generators), rotation=25, ha="right")
    else:
        point_ci_plot(
            ax,
            x=np.arange(len(gen)),
            y=gen["mean_nn_similarity_to_train"].to_numpy(float),
            low=gen["mean_nn_similarity_to_train_ci_low"].to_numpy(float),
            high=gen["mean_nn_similarity_to_train_ci_high"].to_numpy(float),
            colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
        )
        ax.set_xticks(np.arange(len(gen)))
        ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Per-sequence NN similarity to training set")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Per-sequence memorization profile", pad=10)
    style_axis(ax)
    panel_label(ax, "b")

    # c condition heterogeneity
    ax = axes[1, 0]
    if per_cond is not None and {"generator", "surrogate_condition_hit_rate"}.issubset(per_cond.columns):
        rng = np.random.default_rng(7)
        positions = np.arange(1, len(MODEL_ORDER) + 1)
        for position, generator in zip(positions, MODEL_ORDER):
            values = pd.to_numeric(
                per_cond.loc[per_cond["generator"] == generator, "surrogate_condition_hit_rate"],
                errors="coerce",
            ).dropna().to_numpy()
            if len(values) == 0:
                continue
            jitter = rng.normal(0, 0.04, size=len(values))
            ax.scatter(np.full(len(values), position) + jitter, values, s=22, color=MODEL_COLORS[generator], alpha=0.65)
            median = float(np.median(values))
            ax.plot([position - 0.22, position + 0.22], [median, median], color="black", linewidth=1.6)
        ax.set_xticks(positions)
        ax.set_xticklabels(rename_generator_display(MODEL_ORDER), rotation=25, ha="right")
    else:
        point_ci_plot(
            ax,
            x=np.arange(len(gen)),
            y=gen["surrogate_condition_hit_rate"].to_numpy(float),
            low=gen["surrogate_condition_hit_rate_ci_low"].to_numpy(float),
            high=gen["surrogate_condition_hit_rate_ci_high"].to_numpy(float),
            colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
        )
        ax.set_xticks(np.arange(len(gen)))
        ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Per-condition surrogate hit rate")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Condition-level heterogeneity", pad=10)
    style_axis(ax)
    panel_label(ax, "c")

    # d seed variability
    ax = axes[1, 1]
    variability_rows: list[pd.DataFrame] = []
    if train_sum is not None and {"generator", "seed", "best_val_loss"}.issubset(train_sum.columns):
        tmp = train_sum.groupby("generator")["best_val_loss"].agg(std="std").reset_index()
        tmp["Metric"] = "Best validation loss SD"
        tmp["Value"] = tmp["std"]
        variability_rows.append(tmp[["generator", "Metric", "Value"]])
    for metric_name, metric_col in [
        ("Fidelity SD", "surrogate_condition_hit_rate_sd"),
        ("Novelty SD", "exact_novelty_vs_train_sd"),
        ("Diversity SD", "pairwise_diversity_sd"),
    ]:
        if metric_col in gen.columns:
            tmp = gen[["generator", metric_col]].copy()
            tmp["Metric"] = metric_name
            tmp["Value"] = pd.to_numeric(tmp[metric_col], errors="coerce")
            variability_rows.append(tmp[["generator", "Metric", "Value"]])
    if variability_rows:
        long = pd.concat(variability_rows, ignore_index=True)
        filtered = long[long["Metric"].isin(["Fidelity SD", "Novelty SD", "Diversity SD"])]
        if not filtered.empty:
            long = filtered
        metrics_order = list(dict.fromkeys(long["Metric"].tolist()))
        ypos = np.arange(len(MODEL_ORDER))
        markers = ["o", "s", "^"]
        for j, metric in enumerate(metrics_order):
            sub = long[long["Metric"] == metric].copy()
            sub["generator"] = pd.Categorical(sub["generator"], categories=MODEL_ORDER, ordered=True)
            sub = sub.sort_values("generator")
            yy = ypos + (j - (len(metrics_order) - 1) / 2) * 0.18
            for xv, yv, generator in zip(sub["Value"].to_numpy(float), yy, sub["generator"].astype(str)):
                ax.scatter(xv, yv, color=MODEL_COLORS[generator], s=36, marker=markers[j % len(markers)], alpha=0.9)
        ax.set_yticks(ypos)
        ax.set_yticklabels(rename_generator_display(MODEL_ORDER))
        ax.set_xlabel("Across-seed variability")
        ax.set_title("Seed variability summary", pad=10)
        handles = [Line2D([0], [0], marker=markers[j % len(markers)], color="black", linestyle="None", label=m) for j, m in enumerate(metrics_order)]
        ax.legend(handles=handles, frameon=False, loc="best")
        style_axis(ax, ygrid=False)
        ax.xaxis.grid(True, color="#d9d9d9", linewidth=0.8, alpha=0.8)
    else:
        ax.text(0.5, 0.5, "Seed-variability data unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "d")

    save_figure(fig, output_path, cfg)


# =============================================================================
# Public entry point
# =============================================================================
def run_step5_figure_redesign(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    step4_model_ready_csv: str | Path = DEFAULT_STEP4_MODEL_READY,
    main_name: str = "fig_5_step5_main_benchmark_redesign",
    supp1_name: str = "fig_s5a_step5_descriptor_audit_redesign",
    supp2_name: str = "fig_s5b_step5_generator_diagnostics_redesign",
    export_png: bool = True,
    export_pdf: bool = True,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureRedesignConfig(
        step5_root=Path(step5_root),
        step4_model_ready_csv=Path(step4_model_ready_csv),
        main_name=main_name,
        supp1_name=supp1_name,
        supp2_name=supp2_name,
        export_png=export_png,
        export_pdf=export_pdf,
    )
    apply_global_style()
    outdirs = ensure_output_dirs(cfg.step5_root)
    tables = load_step5_tables(cfg)

    main_path = outdirs["main"] / cfg.main_name
    supp1_path = outdirs["supp"] / cfg.supp1_name
    supp2_path = outdirs["supp"] / cfg.supp2_name

    make_main_figure(tables, main_path, cfg)
    make_supplementary_figure_s5a(tables, supp1_path, cfg)
    make_supplementary_figure_s5b(tables, supp2_path, cfg)

    outputs = {
        "main_pdf": str(main_path.with_suffix(".pdf")),
        "main_png": str(main_path.with_suffix(".png")),
        "supp1_pdf": str(supp1_path.with_suffix(".pdf")),
        "supp1_png": str(supp1_path.with_suffix(".png")),
        "supp2_pdf": str(supp2_path.with_suffix(".pdf")),
        "supp2_png": str(supp2_path.with_suffix(".png")),
    }
    if verbose:
        print("Step 5 figure redesign completed.")
        print(f"step5_root: {cfg.step5_root}")
        print(f"step4_model_ready_csv: {cfg.step4_model_ready_csv}")
        print("Detected input files:")
        for key, path in tables["_paths"].items():
            print(f"  {key}: {path}")
        print("Output files:")
        for key, value in outputs.items():
            print(f"  {key}: {value}")
    return outputs


if __name__ == "__main__":
    run_step5_figure_redesign()

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =============================================================================
# User paths
# =============================================================================
DEFAULT_STEP5_ROOT = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5"
)
DEFAULT_STEP4_MODEL_READY = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv"
)


# =============================================================================
# Style constants
# =============================================================================
MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "retrieval_reference": "Retrieval reference",
    "cvae_conditional": "CVAE (conditional)",
    "gru_conditional": "GRU (conditional)",
    "random_conditional": "Random conditional",
    "gru_unconditional": "GRU (unconditional)",
    "vae_unconditional": "VAE (unconditional)",
}

MODEL_COLORS = {
    "retrieval_reference": "#56B4E9",
    "cvae_conditional": "#009E73",
    "gru_conditional": "#CC79A7",
    "random_conditional": "#7F7F7F",
    "gru_unconditional": "#0072B2",
    "vae_unconditional": "#E69F00",
}

DESCRIPTOR_ORDER = [
    "Full numeric descriptor set",
    "Computable descriptors",
    "Physicochemical only",
    "AA composition only",
    "Shuffled-label control",
]

DESCRIPTOR_LABELS = {
    "Full numeric descriptor set": "Full numeric",
    "Computable descriptors": "Computable",
    "Physicochemical only": "Physicochemical",
    "AA composition only": "AA composition",
    "Shuffled-label control": "Shuffle control",
}

DESCRIPTOR_COLORS = {
    "Full numeric descriptor set": "#1f77b4",
    "Computable descriptors": "#377eb8",
    "Physicochemical only": "#6baed6",
    "AA composition only": "#9ecae1",
    "Shuffled-label control": "#7F7F7F",
}


# =============================================================================
# Config
# =============================================================================
@dataclass(frozen=True)
class FigureRedesignConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    step4_model_ready_csv: Path = DEFAULT_STEP4_MODEL_READY
    main_name: str = "fig_5_step5_main_benchmark_redesign"
    supp1_name: str = "fig_s5a_step5_descriptor_audit_redesign"
    supp2_name: str = "fig_s5b_step5_generator_diagnostics_redesign"
    export_png: bool = True
    export_pdf: bool = True
    dpi: int = 300


# =============================================================================
# Global style
# =============================================================================
def apply_global_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.titlesize": 12,
            "axes.labelsize": 11,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 9,
            "figure.titlesize": 13,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": False,
            "savefig.dpi": 300,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


# =============================================================================
# I/O discovery
# =============================================================================
def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None



def discover_step5_inputs(cfg: FigureRedesignConfig) -> Dict[str, Optional[Path]]:
    root = cfg.step5_root
    if not root.exists():
        raise FileNotFoundError(f"Step 5 root does not exist: {root}")

    model_ready_path: Optional[Path] = None
    if cfg.step4_model_ready_csv.exists():
        model_ready_path = cfg.step4_model_ready_csv
    else:
        model_ready_path = _find_first(
            root,
            [
                "step4_model_ready_sequences.csv",
                "../step4/tables/step4_model_ready_sequences.csv",
                "../../step4/tables/step4_model_ready_sequences.csv",
            ],
        )

    inputs = {
        "descriptor_benchmark": _find_first(root, ["table_5_1_step5a_descriptor_benchmark.csv"]),
        "generator_benchmark": _find_first(root, ["table_5_2_step5b_generator_benchmark.csv"]),
        "training_summary": _find_first(root, ["table_s5_3_generator_training_summary.csv"]),
        "per_condition_metrics": _find_first(root, ["table_s5_4_step5b_per_condition_metrics.csv"]),
        "candidate_metrics": _find_first(root, ["step5b_generated_candidates_evaluated.csv"]),
        "confusion_matrix_norm": _find_first(root, ["step5a_confusion_matrix_normalized.csv"]),
        "permutation_importance": _find_first(root, ["step5a_permutation_importance.csv"]),
        "pca_projection": _find_first(root, ["step5a_pca_projection.csv"]),
        "model_ready_sequences": model_ready_path,
    }

    missing = [
        key for key in ("descriptor_benchmark", "generator_benchmark") if inputs[key] is None
    ]
    if missing:
        raise FileNotFoundError(
            "Missing required Step 5 files: " + ", ".join(missing)
        )
    return inputs



def load_csv_if_exists(path: Optional[Path]) -> Optional[pd.DataFrame]:
    if path is None or not path.exists():
        return None
    return pd.read_csv(path)



def load_step5_tables(cfg: FigureRedesignConfig) -> Dict[str, Optional[pd.DataFrame]]:
    paths = discover_step5_inputs(cfg)
    tables: Dict[str, Optional[pd.DataFrame]] = {
        key: load_csv_if_exists(path) for key, path in paths.items()
    }
    tables["_paths"] = paths  # type: ignore[assignment]
    return tables


# =============================================================================
# Helpers
# =============================================================================
def ensure_output_dirs(step5_root: Path) -> Dict[str, Path]:
    out_main = step5_root / "figures_main"
    out_supp = step5_root / "figures_supplementary"
    out_main.mkdir(parents=True, exist_ok=True)
    out_supp.mkdir(parents=True, exist_ok=True)
    return {"main": out_main, "supp": out_supp}



def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="bottom",
        ha="left",
    )



def style_axis(ax: plt.Axes, ygrid: bool = True) -> None:
    if ygrid:
        ax.yaxis.grid(True, color="#d9d9d9", linewidth=0.8, alpha=0.8)
    else:
        ax.grid(False)
    ax.set_axisbelow(True)



def rename_generator_display(values: Iterable[str]) -> list[str]:
    return [MODEL_LABELS.get(v, v) for v in values]



def rename_descriptor_display(values: Iterable[str]) -> list[str]:
    return [DESCRIPTOR_LABELS.get(v, v) for v in values]



def safe_label_offsets(yvals: Sequence[float], magnitude: float = 0.012) -> Dict[int, float]:
    offsets: Dict[int, float] = {}
    used: list[float] = []
    for i, y in enumerate(yvals):
        offset = magnitude
        for prev in used:
            if abs((y + offset) - prev) < 0.03:
                offset += magnitude
        offsets[i] = offset
        used.append(y + offset)
    return offsets



def clean_feature_name(name: str) -> str:
    mapping = {
        "net_charge_proxy": "Net charge proxy",
        "net_charge_pH7": "Net charge (pH 7)",
        "net_charge_pH7_z": "Net charge (pH 7, z)",
        "mean_hydropathy": "Mean hydropathy",
        "hydrophobicity_KD": "Hydropathy (Kyte–Doolittle)",
        "hydrophobicity_KD_z": "Hydropathy (KD, z)",
        "sequence_entropy": "Sequence entropy",
        "entropy_z": "Sequence entropy (z)",
        "dominant_aa_frac_z": "Dominant residue fraction",
        "length": "Length",
        "length_z": "Length (z)",
        "primary_condition_id_idx": "Condition index",
        "aa_frac_Y_z": "Tyr fraction (z)",
        "aa_frac_W_z": "Trp fraction (z)",
        "aa_frac_V_z": "Val fraction (z)",
        "mean_length": "Mean length",
    }
    if name in mapping:
        return mapping[name]
    name = name.replace("aa_frac_", "").replace("_z", " (z)")
    name = name.replace("_", " ")
    return name.title()



def require_columns(df: pd.DataFrame, required: Sequence[str], name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")



def resolve_column(df: pd.DataFrame, candidates: Sequence[str], *, required: bool = True) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"None of the candidate columns were found: {candidates}")
    return None



def point_ci_plot(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    low: np.ndarray,
    high: np.ndarray,
    colors: Sequence[str],
    markersize: float = 8.0,
) -> None:
    for xi, yi, lo, hi, color in zip(x, y, low, high, colors):
        ax.errorbar(
            xi,
            yi,
            yerr=[[yi - lo], [hi - yi]],
            fmt="o",
            color=color,
            ecolor=color,
            elinewidth=1.8,
            capsize=4,
            markersize=markersize,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=3,
        )



def save_figure(fig: plt.Figure, output_base: Path, cfg: FigureRedesignConfig) -> None:
    if cfg.export_pdf:
        fig.savefig(output_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(output_base.with_suffix(".png"), bbox_inches="tight", dpi=cfg.dpi)
    plt.close(fig)


# =============================================================================
# Figure builders
# =============================================================================
def make_main_figure(
    tables: Dict[str, Optional[pd.DataFrame]],
    output_path: Path,
    cfg: FigureRedesignConfig,
) -> None:
    desc = tables["descriptor_benchmark"]
    gen = tables["generator_benchmark"]
    cand = tables.get("candidate_metrics")

    assert desc is not None and gen is not None
    desc = desc.copy()
    gen = gen.copy()
    cand = None if cand is None else cand.copy()

    require_columns(
        desc,
        ["block", "macro_f1", "macro_f1_ci_low", "macro_f1_ci_high"],
        "descriptor benchmark",
    )
    require_columns(
        gen,
        [
            "generator",
            "surrogate_condition_hit_rate",
            "surrogate_condition_hit_rate_ci_low",
            "surrogate_condition_hit_rate_ci_high",
            "exact_novelty_vs_train",
            "exact_novelty_vs_train_ci_low",
            "exact_novelty_vs_train_ci_high",
            "mean_nn_similarity_to_train",
            "mean_nn_similarity_to_train_ci_low",
            "mean_nn_similarity_to_train_ci_high",
        ],
        "generator benchmark",
    )

    desc = desc[desc["block"].isin(DESCRIPTOR_ORDER)].copy()
    desc["block"] = pd.Categorical(desc["block"], categories=DESCRIPTOR_ORDER, ordered=True)
    desc = desc.sort_values("block").reset_index(drop=True)

    gen = gen[gen["generator"].isin(MODEL_ORDER)].copy()
    gen["generator"] = pd.Categorical(gen["generator"], categories=MODEL_ORDER, ordered=True)
    gen = gen.sort_values("generator").reset_index(drop=True)

    fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.2), constrained_layout=True)

    # a
    ax = axes[0, 0]
    x = np.arange(len(desc))
    point_ci_plot(
        ax,
        x=x,
        y=desc["macro_f1"].to_numpy(float),
        low=desc["macro_f1_ci_low"].to_numpy(float),
        high=desc["macro_f1_ci_high"].to_numpy(float),
        colors=[DESCRIPTOR_COLORS[b] for b in desc["block"]],
    )
    ax.set_xticks(x)
    ax.set_xticklabels(rename_descriptor_display(desc["block"].astype(str)), rotation=22, ha="right")
    ax.set_ylabel("Macro F1 on held-out test set")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Condition recoverability exceeds shuffled control", pad=10)
    style_axis(ax)
    panel_label(ax, "a")

    # b
    ax = axes[0, 1]
    x = np.arange(len(gen))
    point_ci_plot(
        ax,
        x=x,
        y=gen["surrogate_condition_hit_rate"].to_numpy(float),
        low=gen["surrogate_condition_hit_rate_ci_low"].to_numpy(float),
        high=gen["surrogate_condition_hit_rate_ci_high"].to_numpy(float),
        colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
    )
    ax.set_xticks(x)
    ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_ylim(0.0, max(0.70, float(gen["surrogate_condition_hit_rate_ci_high"].max()) + 0.05))
    ax.set_title("Conditional generators improve surrogate fidelity", pad=10)
    style_axis(ax)
    panel_label(ax, "b")

    # c
    ax = axes[1, 0]
    offsets = safe_label_offsets(gen["surrogate_condition_hit_rate"].to_numpy(float), magnitude=0.010)
    for i, row in gen.iterrows():
        generator = str(row["generator"])
        ax.errorbar(
            row["exact_novelty_vs_train"],
            row["surrogate_condition_hit_rate"],
            xerr=[[row["exact_novelty_vs_train"] - row["exact_novelty_vs_train_ci_low"]],
                  [row["exact_novelty_vs_train_ci_high"] - row["exact_novelty_vs_train"]]],
            yerr=[[row["surrogate_condition_hit_rate"] - row["surrogate_condition_hit_rate_ci_low"]],
                  [row["surrogate_condition_hit_rate_ci_high"] - row["surrogate_condition_hit_rate"]]],
            fmt="o",
            color=MODEL_COLORS[generator],
            ecolor=MODEL_COLORS[generator],
            capsize=3,
            elinewidth=1.6,
            markersize=8,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=3,
        )
        ax.text(
            float(row["exact_novelty_vs_train"]) + 0.01,
            float(row["surrogate_condition_hit_rate"]) + offsets[i],
            MODEL_LABELS[generator],
            fontsize=9,
            color="#222222",
        )
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(0.0, max(0.68, float(gen["surrogate_condition_hit_rate_ci_high"].max()) + 0.05))
    ax.set_xlabel("Exact novelty versus training set")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_title("Fidelity–novelty tradeoff", pad=10)
    style_axis(ax)
    panel_label(ax, "c")

    # d
    ax = axes[1, 1]
    if cand is not None and {"generator", "nn_similarity_to_train"}.issubset(cand.columns):
        datasets: list[np.ndarray] = []
        positions: list[int] = []
        colors: list[str] = []
        for i, generator in enumerate(MODEL_ORDER, start=1):
            values = pd.to_numeric(
                cand.loc[cand["generator"] == generator, "nn_similarity_to_train"],
                errors="coerce",
            ).dropna().to_numpy()
            if len(values) == 0:
                continue
            datasets.append(values)
            positions.append(i)
            colors.append(MODEL_COLORS[generator])

        violin = ax.violinplot(
            datasets,
            positions=positions,
            widths=0.82,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body, color in zip(violin["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("#444444")
            body.set_alpha(0.45)
            body.set_linewidth(0.8)

        box = ax.boxplot(
            datasets,
            positions=positions,
            widths=0.26,
            patch_artist=True,
            showfliers=False,
            medianprops={"color": "#222222", "linewidth": 1.5},
            boxprops={"linewidth": 1.0},
            whiskerprops={"linewidth": 0.9},
            capprops={"linewidth": 0.9},
        )
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor("white")
            patch.set_edgecolor(color)
            patch.set_alpha(0.95)

        shown_generators = [g for g in MODEL_ORDER if len(pd.to_numeric(cand.loc[cand['generator'] == g, 'nn_similarity_to_train'], errors='coerce').dropna()) > 0]
        ax.set_xticks(range(1, len(shown_generators) + 1))
        ax.set_xticklabels(rename_generator_display(shown_generators), rotation=25, ha="right")
    else:
        point_ci_plot(
            ax,
            x=np.arange(len(gen)),
            y=gen["mean_nn_similarity_to_train"].to_numpy(float),
            low=gen["mean_nn_similarity_to_train_ci_low"].to_numpy(float),
            high=gen["mean_nn_similarity_to_train_ci_high"].to_numpy(float),
            colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
        )
        ax.set_xticks(np.arange(len(gen)))
        ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Nearest-neighbour similarity to training set")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Similarity-to-train distribution", pad=10)
    style_axis(ax)
    panel_label(ax, "d")

    save_figure(fig, output_path, cfg)



def make_supplementary_figure_s5a(
    tables: Dict[str, Optional[pd.DataFrame]],
    output_path: Path,
    cfg: FigureRedesignConfig,
) -> None:
    model_ready = tables.get("model_ready_sequences")
    confusion = tables.get("confusion_matrix_norm")
    perm = tables.get("permutation_importance")
    pca = tables.get("pca_projection")

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.8), constrained_layout=True)

    # a support heatmap
    ax = axes[0, 0]
    if model_ready is not None and {"split", "primary_condition_id"}.issubset(model_ready.columns):
        support = (
            model_ready.groupby(["primary_condition_id", "split"]).size().unstack(fill_value=0)
        )
        split_order = [s for s in ["train", "val", "test"] if s in support.columns]
        support = support.reindex(columns=split_order)
        support = support.sort_values(split_order[0], ascending=False)
        im = ax.imshow(support.to_numpy(), aspect="auto", cmap="Blues")
        ax.set_xticks(np.arange(support.shape[1]))
        ax.set_xticklabels(split_order)
        n_rows = support.shape[0]
        if n_rows <= 18:
            yticks = np.arange(n_rows)
        else:
            yticks = np.arange(0, n_rows, max(1, n_rows // 12))
        ax.set_yticks(yticks)
        ax.set_yticklabels(support.index[yticks])
        ax.set_ylabel("Condition")
        ax.set_title("Condition support across frozen splits", pad=10)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Count", rotation=90)
    else:
        ax.text(0.5, 0.5, "Model-ready split table unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "a")

    # b confusion matrix
    ax = axes[0, 1]
    if confusion is not None:
        if np.issubdtype(confusion.iloc[:, 0].dtype, np.number):
            arr = confusion.to_numpy(float)
            row_labels = [str(i) for i in range(arr.shape[0])]
            col_labels = [str(i) for i in range(arr.shape[1])]
        else:
            arr = confusion.iloc[:, 1:].to_numpy(float)
            row_labels = confusion.iloc[:, 0].astype(str).tolist()
            col_labels = confusion.columns[1:].astype(str).tolist()
        im = ax.imshow(arr, aspect="equal", cmap="Blues", vmin=0.0, vmax=1.0)
        n = arr.shape[0]
        ticks = np.arange(0, n, max(1, n // 10))
        ax.set_xticks(ticks)
        ax.set_xticklabels([col_labels[i] for i in ticks], rotation=45, ha="right")
        ax.set_yticks(ticks)
        ax.set_yticklabels([row_labels[i] for i in ticks])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Normalized confusion matrix", pad=10)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Row-normalized fraction", rotation=90)
    else:
        ax.text(0.5, 0.5, "Confusion matrix unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "b")

    # c permutation importance
    ax = axes[1, 0]
    if perm is not None:
        feature_col = resolve_column(perm, ["feature", "descriptor", "variable"]) or perm.columns[0]
        value_col = resolve_column(
            perm,
            ["importance", "permutation_importance", "macro_f1_importance", "value"],
            required=False,
        )
        if value_col is None:
            numeric = perm.select_dtypes(include=[np.number]).columns.tolist()
            if not numeric:
                raise ValueError("Permutation importance file contains no numeric columns.")
            value_col = numeric[0]
        plot_df = perm[[feature_col, value_col]].copy().sort_values(value_col, ascending=False).head(10).iloc[::-1]
        ax.barh(
            np.arange(len(plot_df)),
            plot_df[value_col].to_numpy(float),
            color="#D8A12A",
            edgecolor="#8C6D1F",
            linewidth=0.6,
        )
        ax.set_yticks(np.arange(len(plot_df)))
        ax.set_yticklabels([clean_feature_name(str(x)) for x in plot_df[feature_col]])
        ax.set_xlabel("Permutation importance (macro F1)")
        ax.set_title("Top descriptor contributors", pad=10)
        style_axis(ax)
    else:
        ax.text(0.5, 0.5, "Permutation importance unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "c")

    # d PCA structure
    ax = axes[1, 1]
    if pca is not None and {"PC1", "PC2"}.issubset(pca.columns):
        cond_col = resolve_column(pca, ["primary_condition_id", "condition", "label"], required=False)
        if cond_col is None:
            non_pc = [c for c in pca.columns if c not in {"PC1", "PC2"}]
            cond_col = non_pc[0] if non_pc else None
        if cond_col is not None:
            top_conditions = pca[cond_col].astype(str).value_counts().head(8).index.tolist()
            sub = pca[pca[cond_col].astype(str).isin(top_conditions)].copy()
            palette = plt.cm.tab10(np.linspace(0, 1, len(top_conditions)))
            for color, cond in zip(palette, top_conditions):
                tmp = sub[sub[cond_col].astype(str) == cond]
                ax.scatter(tmp["PC1"], tmp["PC2"], s=14, alpha=0.45, color=color, label=cond, edgecolors="none")
            ax.legend(frameon=False, fontsize=7, ncol=2, loc="best")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            ax.set_title("Descriptor-space structure (top conditions)", pad=10)
            style_axis(ax)
        else:
            ax.text(0.5, 0.5, "PCA condition labels unavailable", ha="center", va="center")
            ax.set_axis_off()
    else:
        ax.text(0.5, 0.5, "PCA projection unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "d")

    save_figure(fig, output_path, cfg)



def make_supplementary_figure_s5b(
    tables: Dict[str, Optional[pd.DataFrame]],
    output_path: Path,
    cfg: FigureRedesignConfig,
) -> None:
    gen = tables["generator_benchmark"]
    per_cond = tables.get("per_condition_metrics")
    cand = tables.get("candidate_metrics")
    train_sum = tables.get("training_summary")

    assert gen is not None
    gen = gen.copy()
    gen = gen[gen["generator"].isin(MODEL_ORDER)].copy()
    gen["generator"] = pd.Categorical(gen["generator"], categories=MODEL_ORDER, ordered=True)
    gen = gen.sort_values("generator").reset_index(drop=True)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.8), constrained_layout=True)

    # a quality triad
    ax = axes[0, 0]
    metrics = [
        ("valid_rate", "Validity", "o"),
        ("exact_novelty_vs_train", "Novelty", "s"),
        ("unique_rate_among_valid", "Uniqueness", "^"),
    ]
    x = np.arange(len(gen))
    for j, (metric_col, label, marker) in enumerate(metrics):
        require_columns(gen, [metric_col, f"{metric_col}_ci_low", f"{metric_col}_ci_high"], "generator benchmark")
        xj = x + (j - 1) * 0.10
        for xi, generator, y, lo, hi in zip(
            xj,
            gen["generator"].astype(str),
            gen[metric_col].to_numpy(float),
            gen[f"{metric_col}_ci_low"].to_numpy(float),
            gen[f"{metric_col}_ci_high"].to_numpy(float),
        ):
            ax.errorbar(
                xi,
                y,
                yerr=[[y - lo], [hi - y]],
                fmt=marker,
                color=MODEL_COLORS[generator],
                ecolor=MODEL_COLORS[generator],
                capsize=3,
                markersize=6.5,
                markeredgecolor="white",
                markeredgewidth=0.7,
            )
    ax.set_xticks(x)
    ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Rate")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Validity, novelty, and uniqueness", pad=10)
    handles = [Line2D([0], [0], marker=m, color="black", linestyle="None", markersize=6, label=l) for _, l, m in metrics]
    ax.legend(handles=handles, frameon=False, loc="lower right")
    style_axis(ax)
    panel_label(ax, "a")

    # b memorization profile
    ax = axes[0, 1]
    if cand is not None and {"generator", "nn_similarity_to_train"}.issubset(cand.columns):
        datasets: list[np.ndarray] = []
        positions: list[int] = []
        colors: list[str] = []
        shown_generators: list[str] = []
        for i, generator in enumerate(MODEL_ORDER, start=1):
            values = pd.to_numeric(
                cand.loc[cand["generator"] == generator, "nn_similarity_to_train"],
                errors="coerce",
            ).dropna().to_numpy()
            if len(values) == 0:
                continue
            datasets.append(values)
            positions.append(i)
            colors.append(MODEL_COLORS[generator])
            shown_generators.append(generator)
        violin = ax.violinplot(
            datasets,
            positions=positions,
            widths=0.82,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body, color in zip(violin["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("#444444")
            body.set_alpha(0.45)
            body.set_linewidth(0.8)
        box = ax.boxplot(
            datasets,
            positions=positions,
            widths=0.26,
            patch_artist=True,
            showfliers=False,
            medianprops={"color": "#222222", "linewidth": 1.5},
            boxprops={"linewidth": 1.0},
            whiskerprops={"linewidth": 0.9},
            capprops={"linewidth": 0.9},
        )
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor("white")
            patch.set_edgecolor(color)
        ax.set_xticks(positions)
        ax.set_xticklabels(rename_generator_display(shown_generators), rotation=25, ha="right")
    else:
        point_ci_plot(
            ax,
            x=np.arange(len(gen)),
            y=gen["mean_nn_similarity_to_train"].to_numpy(float),
            low=gen["mean_nn_similarity_to_train_ci_low"].to_numpy(float),
            high=gen["mean_nn_similarity_to_train_ci_high"].to_numpy(float),
            colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
        )
        ax.set_xticks(np.arange(len(gen)))
        ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Per-sequence NN similarity to training set")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Per-sequence memorization profile", pad=10)
    style_axis(ax)
    panel_label(ax, "b")

    # c condition heterogeneity
    ax = axes[1, 0]
    if per_cond is not None and {"generator", "surrogate_condition_hit_rate"}.issubset(per_cond.columns):
        rng = np.random.default_rng(7)
        positions = np.arange(1, len(MODEL_ORDER) + 1)
        for position, generator in zip(positions, MODEL_ORDER):
            values = pd.to_numeric(
                per_cond.loc[per_cond["generator"] == generator, "surrogate_condition_hit_rate"],
                errors="coerce",
            ).dropna().to_numpy()
            if len(values) == 0:
                continue
            jitter = rng.normal(0, 0.04, size=len(values))
            ax.scatter(np.full(len(values), position) + jitter, values, s=22, color=MODEL_COLORS[generator], alpha=0.65)
            median = float(np.median(values))
            ax.plot([position - 0.22, position + 0.22], [median, median], color="black", linewidth=1.6)
        ax.set_xticks(positions)
        ax.set_xticklabels(rename_generator_display(MODEL_ORDER), rotation=25, ha="right")
    else:
        point_ci_plot(
            ax,
            x=np.arange(len(gen)),
            y=gen["surrogate_condition_hit_rate"].to_numpy(float),
            low=gen["surrogate_condition_hit_rate_ci_low"].to_numpy(float),
            high=gen["surrogate_condition_hit_rate_ci_high"].to_numpy(float),
            colors=[MODEL_COLORS[str(g)] for g in gen["generator"]],
        )
        ax.set_xticks(np.arange(len(gen)))
        ax.set_xticklabels(rename_generator_display(gen["generator"].astype(str)), rotation=25, ha="right")
    ax.set_ylabel("Per-condition surrogate hit rate")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Condition-level heterogeneity", pad=10)
    style_axis(ax)
    panel_label(ax, "c")

    # d seed variability
    ax = axes[1, 1]
    variability_rows: list[pd.DataFrame] = []
    if train_sum is not None and {"generator", "seed", "best_val_loss"}.issubset(train_sum.columns):
        tmp = train_sum.groupby("generator")["best_val_loss"].agg(std="std").reset_index()
        tmp["Metric"] = "Best validation loss SD"
        tmp["Value"] = tmp["std"]
        variability_rows.append(tmp[["generator", "Metric", "Value"]])
    for metric_name, metric_col in [
        ("Fidelity SD", "surrogate_condition_hit_rate_sd"),
        ("Novelty SD", "exact_novelty_vs_train_sd"),
        ("Diversity SD", "pairwise_diversity_sd"),
    ]:
        if metric_col in gen.columns:
            tmp = gen[["generator", metric_col]].copy()
            tmp["Metric"] = metric_name
            tmp["Value"] = pd.to_numeric(tmp[metric_col], errors="coerce")
            variability_rows.append(tmp[["generator", "Metric", "Value"]])
    if variability_rows:
        long = pd.concat(variability_rows, ignore_index=True)
        filtered = long[long["Metric"].isin(["Fidelity SD", "Novelty SD", "Diversity SD"])]
        if not filtered.empty:
            long = filtered
        metrics_order = list(dict.fromkeys(long["Metric"].tolist()))
        ypos = np.arange(len(MODEL_ORDER))
        markers = ["o", "s", "^"]
        for j, metric in enumerate(metrics_order):
            sub = long[long["Metric"] == metric].copy()
            sub["generator"] = pd.Categorical(sub["generator"], categories=MODEL_ORDER, ordered=True)
            sub = sub.sort_values("generator")
            yy = ypos + (j - (len(metrics_order) - 1) / 2) * 0.18
            for xv, yv, generator in zip(sub["Value"].to_numpy(float), yy, sub["generator"].astype(str)):
                ax.scatter(xv, yv, color=MODEL_COLORS[generator], s=36, marker=markers[j % len(markers)], alpha=0.9)
        ax.set_yticks(ypos)
        ax.set_yticklabels(rename_generator_display(MODEL_ORDER))
        ax.set_xlabel("Across-seed variability")
        ax.set_title("Seed variability summary", pad=10)
        handles = [Line2D([0], [0], marker=markers[j % len(markers)], color="black", linestyle="None", label=m) for j, m in enumerate(metrics_order)]
        ax.legend(handles=handles, frameon=False, loc="best")
        style_axis(ax, ygrid=False)
        ax.xaxis.grid(True, color="#d9d9d9", linewidth=0.8, alpha=0.8)
    else:
        ax.text(0.5, 0.5, "Seed-variability data unavailable", ha="center", va="center")
        ax.set_axis_off()
    panel_label(ax, "d")

    save_figure(fig, output_path, cfg)


# =============================================================================
# Public entry point
# =============================================================================
def run_step5_figure_redesign(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    step4_model_ready_csv: str | Path = DEFAULT_STEP4_MODEL_READY,
    main_name: str = "fig_5_step5_main_benchmark_redesign",
    supp1_name: str = "fig_s5a_step5_descriptor_audit_redesign",
    supp2_name: str = "fig_s5b_step5_generator_diagnostics_redesign",
    export_png: bool = True,
    export_pdf: bool = True,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureRedesignConfig(
        step5_root=Path(step5_root),
        step4_model_ready_csv=Path(step4_model_ready_csv),
        main_name=main_name,
        supp1_name=supp1_name,
        supp2_name=supp2_name,
        export_png=export_png,
        export_pdf=export_pdf,
    )
    apply_global_style()
    outdirs = ensure_output_dirs(cfg.step5_root)
    tables = load_step5_tables(cfg)

    main_path = outdirs["main"] / cfg.main_name
    supp1_path = outdirs["supp"] / cfg.supp1_name
    supp2_path = outdirs["supp"] / cfg.supp2_name

    make_main_figure(tables, main_path, cfg)
    make_supplementary_figure_s5a(tables, supp1_path, cfg)
    make_supplementary_figure_s5b(tables, supp2_path, cfg)

    outputs = {
        "main_pdf": str(main_path.with_suffix(".pdf")),
        "main_png": str(main_path.with_suffix(".png")),
        "supp1_pdf": str(supp1_path.with_suffix(".pdf")),
        "supp1_png": str(supp1_path.with_suffix(".png")),
        "supp2_pdf": str(supp2_path.with_suffix(".pdf")),
        "supp2_png": str(supp2_path.with_suffix(".png")),
    }
    if verbose:
        print("Step 5 figure redesign completed.")
        print(f"step5_root: {cfg.step5_root}")
        print(f"step4_model_ready_csv: {cfg.step4_model_ready_csv}")
        print("Detected input files:")
        for key, path in tables["_paths"].items():
            print(f"  {key}: {path}")
        print("Output files:")
        for key, value in outputs.items():
            print(f"  {key}: {value}")
    return outputs


if __name__ == "__main__":
    run_step5_figure_redesign()

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

DEFAULT_STEP5_ROOT = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5")
DEFAULT_STEP4_MODEL_READY = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv")

HYDRAMP_RED_1 = "#B80018"
HYDRAMP_RED_2 = "#D91115"
HYDRAMP_RED_3 = "#660708"
HYDRAMP_NAVY = "#1D3557"
HYDRAMP_PALE_BLUE = "#B4C5E4"
HYDRAMP_TEAL = "#0F5257"
HYDRAMP_YELLOW = "#FFD23F"
HYDRAMP_PURPLE = "#540D6E"
CONTROL_GREY = "#7F7F7F"

MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]
GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS_FULL = {
    "retrieval_reference": "Retrieval reference",
    "cvae_conditional": "CVAE (conditional)",
    "gru_conditional": "GRU (conditional)",
    "random_conditional": "Random conditional",
    "gru_unconditional": "GRU (unconditional)",
    "vae_unconditional": "VAE (unconditional)",
}
MODEL_LABELS_SHORT = {
    "retrieval_reference": "Retrieval",
    "cvae_conditional": "CVAE-cond",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
}
MODEL_MARKERS = {
    "retrieval_reference": "D",
    "cvae_conditional": "o",
    "gru_conditional": "s",
    "random_conditional": "P",
    "gru_unconditional": "^",
    "vae_unconditional": "v",
}
MODEL_COLORS = {
    "retrieval_reference": HYDRAMP_PALE_BLUE,
    "cvae_conditional": HYDRAMP_RED_1,
    "gru_conditional": HYDRAMP_RED_2,
    "random_conditional": HYDRAMP_TEAL,
    "gru_unconditional": HYDRAMP_NAVY,
    "vae_unconditional": HYDRAMP_RED_3,
}

DESCRIPTOR_ORDER = [
    "Full numeric descriptor set",
    "Computable descriptors",
    "Physicochemical only",
    "AA composition only",
    "Shuffled-label control",
]
DESCRIPTOR_LABELS_SHORT = {
    "Full numeric descriptor set": "Full numeric",
    "Computable descriptors": "Computable",
    "Physicochemical only": "Physicochemical",
    "AA composition only": "AA composition",
    "Shuffled-label control": "Shuffled control",
}
DESCRIPTOR_COLORS = {
    "Full numeric descriptor set": HYDRAMP_NAVY,
    "Computable descriptors": HYDRAMP_TEAL,
    "Physicochemical only": HYDRAMP_YELLOW,
    "AA composition only": HYDRAMP_PURPLE,
    "Shuffled-label control": CONTROL_GREY,
}

@dataclass(frozen=True)
class FigureConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    step4_model_ready_csv: Path = DEFAULT_STEP4_MODEL_READY
    manuscript_mode: bool = True
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    main_name: str = "fig_5_step5_main_benchmark_redesign_v9"
    supp_a_name: str = "fig_s5a_step5_descriptor_audit_redesign_v9"
    supp_b_name: str = "fig_s5b_step5_generator_diagnostics_redesign_v9"
    support_table_name: str = "table_s5a_condition_support_lookup_v9"

def apply_global_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9.5,
            "axes.titlesize": 11.0,
            "axes.labelsize": 10.8,
            "xtick.labelsize": 8.9,
            "ytick.labelsize": 8.9,
            "legend.fontsize": 8.1,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.9,
            "xtick.major.width": 0.9,
            "ytick.major.width": 0.9,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )

def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color="#D9D9D9", linewidth=0.8, alpha=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color="#D9D9D9", linewidth=0.8, alpha=0.9)
    elif grid_axis == "both":
        ax.grid(True, color="#D9D9D9", linewidth=0.8, alpha=0.9)

def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.14, 1.065, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=15.5, fontweight="bold")

def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

def discover_inputs(cfg: FigureConfig) -> Dict[str, Optional[Path]]:
    root = cfg.step5_root
    if not root.exists():
        raise FileNotFoundError(f"Step 5 root does not exist: {root}")
    model_ready_path = cfg.step4_model_ready_csv if cfg.step4_model_ready_csv.exists() else _find_first(
        root,
        [
            "step4_model_ready_sequences.csv",
            "../step4/tables/step4_model_ready_sequences.csv",
            "../../step4/tables/step4_model_ready_sequences.csv",
        ],
    )
    return {
        "descriptor_benchmark": _find_first(root, ["table_5_1_step5a_descriptor_benchmark.csv"]),
        "generator_benchmark": _find_first(root, ["table_5_2_step5b_generator_benchmark.csv"]),
        "per_condition_metrics": _find_first(root, ["table_s5_4_step5b_per_condition_metrics.csv"]),
        "candidate_metrics": _find_first(root, ["step5b_generated_candidates_evaluated.csv"]),
        "confusion_matrix_norm": _find_first(root, ["step5a_confusion_matrix_normalized.csv"]),
        "permutation_importance": _find_first(root, ["step5a_permutation_importance.csv"]),
        "model_ready_sequences": model_ready_path,
    }

def read_csv_or_none(path: Optional[Path]) -> Optional[pd.DataFrame]:
    if path is None or not path.exists():
        return None
    return pd.read_csv(path)

def load_tables(cfg: FigureConfig) -> Dict[str, Optional[pd.DataFrame]]:
    paths = discover_inputs(cfg)
    tables = {k: read_csv_or_none(v) for k, v in paths.items()}
    tables["_paths"] = paths
    return tables

def ensure_dirs(step5_root: Path) -> Dict[str, Path]:
    figs_main = step5_root / "figures_main"
    figs_supp = step5_root / "figures_supplementary"
    src_data = step5_root / "source_data_figures"
    tables_dir = step5_root / "tables"
    for p in [figs_main, figs_supp, src_data, tables_dir]:
        p.mkdir(parents=True, exist_ok=True)
    return {"main": figs_main, "supp": figs_supp, "source": src_data, "tables": tables_dir}

def save_figure(fig: plt.Figure, out_base: Path, cfg: FigureConfig) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    if cfg.export_pdf:
        pdf_path = out_base.with_suffix(".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        outputs["pdf"] = str(pdf_path)
    if cfg.export_png:
        png_path = out_base.with_suffix(".png")
        fig.savefig(png_path, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs["png"] = str(png_path)
    if cfg.export_tiff:
        tiff_path = out_base.with_suffix(".tiff")
        fig.savefig(tiff_path, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
        outputs["tiff"] = str(tiff_path)
    plt.close(fig)
    return outputs

def export_source_data(df: pd.DataFrame, out_dir: Path, name: str) -> str:
    out_path = out_dir / f"{name}.csv"
    df.to_csv(out_path, index=False)
    return str(out_path)

def require_df(df: Optional[pd.DataFrame], name: str) -> pd.DataFrame:
    if df is None:
        raise FileNotFoundError(f"Required table missing: {name}")
    return df.copy()

def require_columns(df: pd.DataFrame, required: Sequence[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")

def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"Could not resolve any of columns: {candidates}")
    return None

def verify_ci_columns(df: pd.DataFrame, metric: str, name: str) -> None:
    cols = [metric, f"{metric}_ci_low", f"{metric}_ci_high"]
    require_columns(df, cols, name)
    bad = df[
        (pd.to_numeric(df[f"{metric}_ci_low"], errors="coerce") > pd.to_numeric(df[metric], errors="coerce"))
        | (pd.to_numeric(df[metric], errors="coerce") > pd.to_numeric(df[f"{metric}_ci_high"], errors="coerce"))
    ]
    if len(bad) > 0:
        raise ValueError(f"{name}: invalid CI bounds for metric '{metric}'.")

def recode_condition_to_short_ids(values: Sequence[str]) -> Tuple[Dict[str, str], List[str]]:
    unique_vals = list(dict.fromkeys([str(v) for v in values]))
    mapping = {val: f"C{i+1:02d}" for i, val in enumerate(unique_vals)}
    ordered_ids = [mapping[v] for v in unique_vals]
    return mapping, ordered_ids

def resolve_split_column(df: pd.DataFrame) -> Optional[str]:
    return resolve_column(df, ["split", "Split", "dataset_split", "set", "subset", "partition", "fold", "split_name", "data_split", "group"], required=False)

def infer_split_values_from_any_column(df: pd.DataFrame) -> Optional[pd.Series]:
    for col in df.columns:
        series = df[col].astype(str).str.strip().str.lower()
        mapped = []
        found = False
        for v in series:
            label = None
            if "train" in v:
                label = "train"; found = True
            elif "val" in v or "valid" in v or "dev" in v:
                label = "val"; found = True
            elif "test" in v or "hold" in v:
                label = "test"; found = True
            mapped.append(label)
        if found:
            out = pd.Series(mapped, index=df.index, dtype="object")
            if out.notna().sum() > 0:
                return out
    return None

def prep_descriptor_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["block", "macro_f1", "macro_f1_ci_low", "macro_f1_ci_high"], "descriptor_benchmark")
    verify_ci_columns(df, "macro_f1", "descriptor_benchmark")
    out = df[df["block"].isin(DESCRIPTOR_ORDER)].copy()
    out["block"] = pd.Categorical(out["block"], categories=DESCRIPTOR_ORDER, ordered=True)
    out = out.sort_values("block").reset_index(drop=True)
    out["block_short"] = out["block"].astype(str).map(DESCRIPTOR_LABELS_SHORT)
    return out

def prep_generator_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    needed = [
        "generator","surrogate_condition_hit_rate","surrogate_condition_hit_rate_ci_low","surrogate_condition_hit_rate_ci_high",
        "exact_novelty_vs_train","exact_novelty_vs_train_ci_low","exact_novelty_vs_train_ci_high",
        "mean_nn_similarity_to_train","mean_nn_similarity_to_train_ci_low","mean_nn_similarity_to_train_ci_high",
        "valid_rate","valid_rate_ci_low","valid_rate_ci_high",
        "unique_rate_among_valid","unique_rate_among_valid_ci_low","unique_rate_among_valid_ci_high",
    ]
    require_columns(df, needed, "generator_benchmark")
    verify_ci_columns(df, "surrogate_condition_hit_rate", "generator_benchmark")
    verify_ci_columns(df, "exact_novelty_vs_train", "generator_benchmark")
    verify_ci_columns(df, "mean_nn_similarity_to_train", "generator_benchmark")
    verify_ci_columns(df, "valid_rate", "generator_benchmark")
    verify_ci_columns(df, "unique_rate_among_valid", "generator_benchmark")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    missing = [m for m in MODEL_ORDER if m not in set(out["generator"].astype(str))]
    if missing:
        raise ValueError(f"generator_benchmark missing expected generators: {missing}")
    out["generator"] = pd.Categorical(out["generator"], categories=MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["label_short"] = out["generator"].astype(str).map(MODEL_LABELS_SHORT)
    out["label_full"] = out["generator"].astype(str).map(MODEL_LABELS_FULL)
    return out

def prep_candidate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "nn_similarity_to_train"], "candidate_metrics")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["nn_similarity_to_train"] = pd.to_numeric(out["nn_similarity_to_train"], errors="coerce")
    out = out.dropna(subset=["nn_similarity_to_train"]).reset_index(drop=True)
    if out.empty:
        raise ValueError("candidate_metrics contains no valid nn_similarity_to_train values.")
    return out

def prep_per_condition_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "surrogate_condition_hit_rate"], "per_condition_metrics")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["surrogate_condition_hit_rate"] = pd.to_numeric(out["surrogate_condition_hit_rate"], errors="coerce")
    out = out.dropna(subset=["surrogate_condition_hit_rate"]).reset_index(drop=True)
    return out

def prep_model_ready(df: pd.DataFrame) -> pd.DataFrame:
    cond_col = resolve_column(df, ["primary_condition_id", "condition", "condition_id", "label", "primary_condition", "cond_id"], required=True)
    split_col = resolve_split_column(df)
    out = df.copy()
    if split_col is None:
        inferred = infer_split_values_from_any_column(out)
        if inferred is not None:
            out["split"] = inferred
            split_col = "split"
    if split_col is None:
        out["split"] = "all"
        split_col = "split"
    out = out.rename(columns={cond_col: "primary_condition_id", split_col: "split"}).copy()
    out["primary_condition_id"] = out["primary_condition_id"].astype(str)
    out["split"] = out["split"].astype(str).str.strip().str.lower()
    split_map = {
        "training": "train","train": "train",
        "valid": "val","validation": "val","val": "val","dev": "val",
        "testing": "test","test": "test","holdout": "test","heldout": "test","held-out": "test",
        "all": "all",
    }
    out["split"] = out["split"].map(lambda x: split_map.get(x, x))
    return out

def prep_confusion(df: pd.DataFrame) -> Tuple[np.ndarray, List[str], List[str]]:
    if np.issubdtype(df.iloc[:, 0].dtype, np.number):
        arr = df.to_numpy(float)
        rows = [f"C{i+1:02d}" for i in range(arr.shape[0])]
        cols = [f"C{i+1:02d}" for i in range(arr.shape[1])]
        return arr, rows, cols
    arr = df.iloc[:, 1:].to_numpy(float)
    row_labels = df.iloc[:, 0].astype(str).tolist()
    col_labels = df.columns[1:].astype(str).tolist()
    return arr, row_labels, col_labels

def prep_permutation_importance(df: pd.DataFrame) -> pd.DataFrame:
    feature_col = resolve_column(df, ["feature", "descriptor", "variable"], required=True)
    value_col = resolve_column(df, ["importance", "permutation_importance", "macro_f1_importance", "value"], required=False)
    if value_col is None:
        numeric = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric:
            raise ValueError("permutation_importance contains no numeric columns.")
        value_col = numeric[0]
    out = df[[feature_col, value_col]].copy()
    out.columns = ["feature", "importance"]
    out["importance"] = pd.to_numeric(out["importance"], errors="coerce")
    out = out.dropna(subset=["importance"]).sort_values("importance", ascending=False)
    return out

def clean_feature_name(name: str) -> str:
    mapping = {
        "net_charge_proxy": "Net charge proxy",
        "net_charge_pH7": "Net charge (pH 7)",
        "net_charge_pH7_z": "Net charge (pH 7)",
        "mean_hydropathy": "Mean hydropathy",
        "hydrophobicity_KD": "Hydropathy (Kyte–Doolittle)",
        "hydrophobicity_KD_z": "Hydropathy (Kyte–Doolittle)",
        "sequence_entropy": "Entropy",
        "entropy_z": "Entropy",
        "dominant_aa_frac_z": "Dominant Aa frac",
        "length": "Length",
        "length_z": "Length (z)",
        "primary_condition_id_idx": "Condition index",
        "aa_frac_E_z": "E (z)",
        "aa_frac_F_z": "F (z)",
        "aa_frac_A_z": "A (z)",
        "aa_frac_C_z": "C (z)",
    }
    if name in mapping:
        return mapping[name]
    return name.replace("aa_frac_", "").replace("_z", " (z)").replace("_", " ").title()

def point_interval(ax, x, y, low, high, color, marker="o", markersize=7.5):
    ax.errorbar([x], [y], yerr=[[y-low], [high-y]], fmt=marker, color=color, ecolor=color,
                elinewidth=1.7, capsize=3.0, markersize=markersize,
                markeredgecolor="white", markeredgewidth=0.85, zorder=3)

def horizontal_point_interval(ax, x, y, low, high, color, marker="o", markersize=7.0):
    ax.errorbar([x], [y], xerr=[[x-low], [high-x]], fmt=marker, color=color, ecolor=color,
                elinewidth=1.55, capsize=2.6, markersize=markersize,
                markeredgecolor="white", markeredgewidth=0.8, zorder=3)

def draw_violin_box(ax, series, positions, colors, widths=0.72):
    violin = ax.violinplot(series, positions=positions, widths=widths, showmeans=False, showmedians=False, showextrema=False)
    for body, color in zip(violin["bodies"], colors):
        body.set_facecolor(color); body.set_edgecolor(color); body.set_alpha(0.16); body.set_linewidth(0.85)
    box = ax.boxplot(series, positions=positions, widths=0.22, patch_artist=True, showfliers=False,
                     medianprops={"color": "#111111", "linewidth": 1.4},
                     whiskerprops={"linewidth": 0.9}, capprops={"linewidth": 0.9}, boxprops={"linewidth": 0.95})
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor("white"); patch.set_edgecolor(color); patch.set_alpha(1.0)

def build_support_lookup_table(model_ready, confusion_arr, confusion_rows, out_dir, name):
    support = model_ready.groupby(["primary_condition_id", "split"]).size().unstack(fill_value=0)
    cond_map, _ = recode_condition_to_short_ids(support.index.astype(str).tolist())
    support_reset = support.reset_index().rename(columns={"primary_condition_id": "condition_full"})
    support_reset["condition_id"] = support_reset["condition_full"].map(cond_map)
    numeric_cols = [c for c in support.columns]
    support_reset["support_total"] = support_reset[numeric_cols].sum(axis=1)
    diag_lookup = pd.DataFrame({"condition_full": confusion_rows, "diagonal_fraction": np.diag(confusion_arr)})
    out = support_reset.merge(diag_lookup, on="condition_full", how="left")
    out = out[["condition_id", "condition_full"] + numeric_cols + ["support_total", "diagonal_fraction"]]
    path = out_dir / f"{name}.csv"
    out.to_csv(path, index=False)
    return str(path)

def export_main_sources(desc, gen, cand, out_dir):
    outputs = {}
    panel_a = gen[["generator", "label_full", "label_short", "surrogate_condition_hit_rate", "surrogate_condition_hit_rate_ci_low", "surrogate_condition_hit_rate_ci_high"]].copy()
    outputs["main_panel_a_condition_fidelity"] = export_source_data(panel_a, out_dir, "main_panel_a_condition_fidelity")
    panel_b_rows = []
    for _, row in gen.iterrows():
        panel_b_rows.append({"generator": row["generator"], "label_full": row["label_full"], "label_short": row["label_short"], "metric": "Novelty", "value": row["exact_novelty_vs_train"], "ci_low": row["exact_novelty_vs_train_ci_low"], "ci_high": row["exact_novelty_vs_train_ci_high"]})
        panel_b_rows.append({"generator": row["generator"], "label_full": row["label_full"], "label_short": row["label_short"], "metric": "Fidelity", "value": row["surrogate_condition_hit_rate"], "ci_low": row["surrogate_condition_hit_rate_ci_low"], "ci_high": row["surrogate_condition_hit_rate_ci_high"]})
    outputs["main_panel_b_fidelity_vs_novelty"] = export_source_data(pd.DataFrame(panel_b_rows), out_dir, "main_panel_b_fidelity_vs_novelty")
    memory_rows = []
    for g in MODEL_ORDER:
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        memory_rows.append({"generator": g, "label_full": MODEL_LABELS_FULL[g], "label_short": MODEL_LABELS_SHORT[g], "n": int(len(vals)), "median": float(np.median(vals)), "q1": float(np.quantile(vals, 0.25)), "q3": float(np.quantile(vals, 0.75)), "mean": float(np.mean(vals))})
    outputs["main_panel_c_similarity_summary"] = export_source_data(pd.DataFrame(memory_rows), out_dir, "main_panel_c_similarity_summary")
    panel_d = desc[["block", "block_short", "macro_f1", "macro_f1_ci_low", "macro_f1_ci_high"]].copy()
    outputs["main_panel_d_descriptor_recoverability"] = export_source_data(panel_d, out_dir, "main_panel_d_descriptor_recoverability")
    return outputs

def build_main_figure(desc, gen, cand, cfg, source_dir, out_base):
    source_paths = export_main_sources(desc, gen, cand, source_dir)
    fig = plt.figure(figsize=(10.8, 7.4))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.0, 0.97], height_ratios=[1.0, 0.95], wspace=0.35, hspace=0.46)

    axa = fig.add_subplot(gs[0, 0])
    x_pos = np.array([0.0, 1.1, 1.9, 3.0, 4.1, 4.9]); x_map = dict(zip(MODEL_ORDER, x_pos))
    for _, row in gen.iterrows():
        g = str(row["generator"])
        point_interval(axa, float(x_map[g]), float(row["surrogate_condition_hit_rate"]),
                       float(row["surrogate_condition_hit_rate_ci_low"]), float(row["surrogate_condition_hit_rate_ci_high"]),
                       MODEL_COLORS[g], MODEL_MARKERS[g], 8.1 if g != "retrieval_reference" else 7.4)
    for xpos in [0.55, 2.45, 3.55]:
        axa.axvline(xpos, color="#D6D6D6", linestyle="--", linewidth=0.75)
    y_max = max(0.72, float(gen["surrogate_condition_hit_rate_ci_high"].max()) + 0.06)
    axa.set_ylim(0.0, y_max)
    axa.text(0.00, 1.01, "Reference", transform=axa.transAxes, ha="left", va="bottom", fontsize=8.0, color="#666666")
    axa.text(0.33, 1.01, "Conditional", transform=axa.transAxes, ha="center", va="bottom", fontsize=8.0, color="#666666")
    axa.text(0.63, 1.01, "Control", transform=axa.transAxes, ha="center", va="bottom", fontsize=8.0, color="#666666")
    axa.text(0.87, 1.01, "Unconditional", transform=axa.transAxes, ha="center", va="bottom", fontsize=8.0, color="#666666")
    axa.set_xlim(-0.35, 5.3)
    axa.set_xticks(x_pos); axa.set_xticklabels([MODEL_LABELS_SHORT[g] for g in MODEL_ORDER], rotation=15, ha="right")
    axa.set_ylabel("Surrogate condition-hit rate")
    axa.set_title("Conditional models improve fidelity", pad=12)
    style_axis(axa, "y"); add_panel_label(axa, "a")

    gsb = gs[0, 1].subgridspec(1, 2, width_ratios=[0.34, 0.66], wspace=0.12)
    axb_n = fig.add_subplot(gsb[0, 0]); axb_f = fig.add_subplot(gsb[0, 1], sharey=axb_n)
    y_positions = np.arange(len(MODEL_ORDER))[::-1]; y_map = dict(zip(MODEL_ORDER, y_positions))
    for _, row in gen.iterrows():
        g = str(row["generator"]); y = y_map[g]
        horizontal_point_interval(axb_n, float(row["exact_novelty_vs_train"]), float(y),
                                  float(row["exact_novelty_vs_train_ci_low"]), float(row["exact_novelty_vs_train_ci_high"]),
                                  MODEL_COLORS[g], MODEL_MARKERS[g], 7.0)
        horizontal_point_interval(axb_f, float(row["surrogate_condition_hit_rate"]), float(y),
                                  float(row["surrogate_condition_hit_rate_ci_low"]), float(row["surrogate_condition_hit_rate_ci_high"]),
                                  MODEL_COLORS[g], MODEL_MARKERS[g], 7.0)
    axb_n.set_xlim(0.90, 1.01); axb_f.set_xlim(0.0, 0.72)
    axb_n.set_yticks(y_positions); axb_n.set_yticklabels([MODEL_LABELS_SHORT[g] for g in MODEL_ORDER])
    axb_f.tick_params(axis="y", left=False, labelleft=False)
    axb_n.set_xlabel("Novelty"); axb_f.set_xlabel("Fidelity")
    axb_n.set_title("Fidelity versus novelty", pad=12)
    style_axis(axb_n, "x"); style_axis(axb_f, "x")
    axb_n.axvspan(0.90, 1.01, color="#F8F8F8", zorder=0); axb_f.axvspan(0.0, 0.72, color="#F8F8F8", zorder=0)
    axb_n.text(0.98, 1.01, "Generators saturate near-complete novelty", transform=axb_n.transAxes, ha="right", va="bottom", fontsize=7.5, color="#666666")
    add_panel_label(axb_n, "b")

    gsc = gs[1, 0].subgridspec(1, 2, width_ratios=[0.83, 0.17], wspace=0.035)
    axc = fig.add_subplot(gsc[0, 0]); axc_r = fig.add_subplot(gsc[0, 1])
    datasets, positions, colors, labels = [], [], [], []
    for i, g in enumerate(GENERATOR_ONLY_ORDER, start=1):
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            if cfg.manuscript_mode: raise ValueError(f"Missing per-sequence similarity values for generator: {g}")
            continue
        datasets.append(vals); positions.append(i); colors.append(MODEL_COLORS[g]); labels.append(MODEL_LABELS_SHORT[g])
    draw_violin_box(axc, datasets, positions, colors, widths=0.68)
    axc.set_xticks(positions); axc.set_xticklabels(labels, rotation=15, ha="right")
    axc.set_ylabel("NN similarity to train"); axc.set_ylim(0.0, 0.45)
    axc.set_title("Similarity to training set", pad=12)
    style_axis(axc, "y"); add_panel_label(axc, "c")
    retrieval_vals = cand.loc[cand["generator"] == "retrieval_reference", "nn_similarity_to_train"].dropna().to_numpy()
    if len(retrieval_vals) > 0:
        draw_violin_box(axc_r, [retrieval_vals], [1], [MODEL_COLORS["retrieval_reference"]], widths=0.48)
        axc_r.set_xlim(0.65, 1.35); axc_r.set_ylim(0.75, 1.02)
        axc_r.set_xticks([1]); axc_r.set_xticklabels(["Reference"], rotation=15, ha="right")
        axc_r.set_yticks([0.8, 0.9, 1.0]); style_axis(axc_r, "y")
        axc_r.tick_params(axis="y", labelsize=8.0)
        axc_r.text(0.5, 1.01, "Retrieval", transform=axc_r.transAxes, ha="center", va="bottom", fontsize=7.7, color="#666666")
    else:
        axc_r.axis("off")

    axd = fig.add_subplot(gs[1, 1]); y_pos = np.arange(len(desc))[::-1]
    for yp, (_, row) in zip(y_pos, desc.iterrows()):
        b = str(row["block"]); val = float(row["macro_f1"]); lo = float(row["macro_f1_ci_low"]); hi = float(row["macro_f1_ci_high"])
        axd.hlines(yp, 0, val, color=DESCRIPTOR_COLORS[b], linewidth=2.25, alpha=0.95)
        axd.errorbar([val], [yp], xerr=[[val-lo], [hi-val]], fmt="o", color=DESCRIPTOR_COLORS[b], ecolor=DESCRIPTOR_COLORS[b],
                     markersize=8.1, capsize=3.3, markeredgecolor="white", markeredgewidth=0.8, linewidth=1.15)
    if "Shuffled-label control" in desc["block"].astype(str).tolist():
        shuf_val = float(desc.loc[desc["block"].astype(str) == "Shuffled-label control", "macro_f1"].iloc[0])
        axd.axvspan(0, shuf_val, color="#E6E6E6", alpha=0.42, zorder=0)
    axd.set_yticks(y_pos); axd.set_yticklabels(desc["block_short"].tolist())
    axd.set_xlim(0.0, 1.05); axd.set_xlabel("Macro F1 on held-out test set")
    axd.set_title("Descriptor recoverability", pad=12)
    style_axis(axd, "x"); add_panel_label(axd, "d")

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs

def build_supp_descriptor_figure(model_ready, confusion_df, perm_df, cfg, source_dir, out_base, tables_dir):
    support = model_ready.groupby(["primary_condition_id", "split"]).size().unstack(fill_value=0)
    cond_map, _ = recode_condition_to_short_ids(support.index.astype(str).tolist())
    valid_real_splits = any(col in {"train", "val", "test"} for col in support.columns)
    plot_support = support.copy()
    if valid_real_splits:
        preferred = [c for c in ["train", "val", "test"] if c in plot_support.columns]
        plot_support = plot_support.reindex(columns=preferred)
    plot_support["support_total"] = plot_support.sum(axis=1)
    plot_support.index = [cond_map[str(i)] for i in plot_support.index]
    plot_support = plot_support.sort_values("support_total", ascending=False)

    arr, row_labels, _ = prep_confusion(confusion_df)
    lookup_path = build_support_lookup_table(model_ready, arr, row_labels, tables_dir, cfg.support_table_name)

    perm_top = perm_df.head(8).copy()
    perm_top["feature_clean"] = perm_top["feature"].astype(str).map(clean_feature_name)

    rank_support = support.sum(axis=1).sort_values(ascending=False).reset_index()
    rank_support.columns = ["condition_full", "support_count"]
    rank_support["condition_id"] = rank_support["condition_full"].map(cond_map)
    rank_support["rank"] = np.arange(1, len(rank_support) + 1)
    rank_support = rank_support.sort_values("rank")
    rank_support["cum_support_frac"] = rank_support["support_count"].cumsum() / rank_support["support_count"].sum()
    rank_support["cum_condition_frac"] = rank_support["rank"] / len(rank_support)

    src_paths = {
        "supp_s5a_condition_support": export_source_data(plot_support.reset_index().rename(columns={"index": "condition_id"}), source_dir, "supp_s5a_condition_support"),
        "supp_s5a_support_concentration": export_source_data(rank_support, source_dir, "supp_s5a_support_concentration"),
        "supp_s5a_permutation_importance": export_source_data(perm_top[["feature", "feature_clean", "importance"]], source_dir, "supp_s5a_permutation_importance"),
        "support_lookup_table": lookup_path,
    }

    fig = plt.figure(figsize=(10.8, 7.0))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.02, 0.98], height_ratios=[1.0, 0.88], wspace=0.42, hspace=0.46)

    axa = fig.add_subplot(gs[:, 0]); plot_df = plot_support.iloc[::-1].copy()
    totals = plot_df["support_total"].to_numpy(float)
    axa.barh(plot_df.index.tolist(), totals, color=HYDRAMP_TEAL, edgecolor="white", linewidth=0.4)
    if valid_real_splits:
        if "train" in plot_df.columns:
            train_vals = plot_df["train"].to_numpy(float)
            axa.barh(plot_df.index.tolist(), train_vals, color=HYDRAMP_NAVY, alpha=0.42, edgecolor="white", linewidth=0.2, label="train")
        if "test" in plot_df.columns:
            test_vals = plot_df["test"].to_numpy(float)
            axa.barh(plot_df.index.tolist(), test_vals, left=totals - test_vals, color=HYDRAMP_RED_2, edgecolor="white", linewidth=0.2, label="test")
        axa.legend(frameon=False, loc="lower right")
    axa.set_xlabel("Count"); axa.set_ylabel("Condition ID")
    axa.set_title("Condition support", pad=12)
    style_axis(axa, "x"); add_panel_label(axa, "a")

    axb = fig.add_subplot(gs[0, 1])
    axb.plot(rank_support["cum_condition_frac"], rank_support["cum_support_frac"], color=HYDRAMP_TEAL, linewidth=2.0)
    axb.plot([0, 1], [0, 1], color="#BFBFBF", linewidth=1.0, linestyle="--")
    checkpoints = [0.25, 0.50, 0.75, 1.00]
    for q in checkpoints:
        idx = min(max(int(np.ceil(len(rank_support) * q)) - 1, 0), len(rank_support) - 1)
        row = rank_support.iloc[idx]
        axb.scatter([row["cum_condition_frac"]], [row["cum_support_frac"]], s=32, color=HYDRAMP_TEAL, edgecolors="white", linewidths=0.7, zorder=3)
    q50_idx = min(max(int(np.ceil(len(rank_support) * 0.50)) - 1, 0), len(rank_support) - 1)
    q50_row = rank_support.iloc[q50_idx]
    axb.text(0.52, 0.18, f"Top 50% of conditions\ncover {q50_row['cum_support_frac']:.0%} of support",
             transform=axb.transAxes, ha="left", va="center", fontsize=7.8, color="#555555")
    axb.set_xlim(0.0, 1.0); axb.set_ylim(0.0, 1.0)
    axb.set_xlabel("Cumulative fraction of conditions"); axb.set_ylabel("Cumulative fraction of support")
    axb.set_title("Support concentration", pad=12)
    style_axis(axb, "both"); add_panel_label(axb, "b")

    axc = fig.add_subplot(gs[1, 1]); plot_imp = perm_top.iloc[::-1].copy()
    accent_colors = [HYDRAMP_YELLOW] * len(plot_imp)
    if len(plot_imp) >= 2:
        accent_colors[-1] = HYDRAMP_RED_1; accent_colors[-2] = HYDRAMP_RED_1
    for yi, (_, row), col in zip(np.arange(len(plot_imp)), plot_imp.iterrows(), accent_colors):
        axc.hlines(yi, 0, float(row["importance"]), color=col, linewidth=2.0, alpha=0.95)
        axc.scatter(float(row["importance"]), yi, color=col,
                    edgecolors="#7A5C00" if col == HYDRAMP_YELLOW else "white",
                    s=54, linewidths=0.8, zorder=3)
    x_max = float(plot_imp["importance"].max()) * 1.03 if len(plot_imp) else 0.2
    axc.set_xlim(-0.001, max(0.10, x_max))
    axc.set_yticks(np.arange(len(plot_imp))); axc.set_yticklabels(plot_imp["feature_clean"].tolist())
    axc.set_xlabel("Permutation importance (macro F1)")
    axc.set_title("Descriptor importance", pad=12)
    style_axis(axc, "x"); add_panel_label(axc, "c")

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in src_paths.items()})
    return outputs

def build_supp_generator_figure(gen, cand, per_cond, cfg, source_dir, out_base):
    quality_rows = []
    for metric, label in [("valid_rate", "Validity"), ("exact_novelty_vs_train", "Novelty"), ("unique_rate_among_valid", "Uniqueness")]:
        tmp = gen[["generator", "label_full", "label_short", metric, f"{metric}_ci_low", f"{metric}_ci_high"]].copy()
        tmp["metric"] = label
        tmp = tmp.rename(columns={metric: "value", f"{metric}_ci_low": "ci_low", f"{metric}_ci_high": "ci_high"})
        quality_rows.append(tmp)
    quality_df = pd.concat(quality_rows, ignore_index=True)
    memory_rows = []
    for g in MODEL_ORDER:
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        memory_rows.append({"generator": g, "label_full": MODEL_LABELS_FULL[g], "label_short": MODEL_LABELS_SHORT[g], "n": int(len(vals)), "median": float(np.median(vals)), "q1": float(np.quantile(vals, 0.25)), "q3": float(np.quantile(vals, 0.75)), "mean": float(np.mean(vals))})
    memory_df = pd.DataFrame(memory_rows)
    src_paths = {
        "supp_s5b_generation_quality": export_source_data(quality_df, source_dir, "supp_s5b_generation_quality"),
        "supp_s5b_memory_summary": export_source_data(memory_df, source_dir, "supp_s5b_memory_summary"),
        "supp_s5b_condition_heterogeneity": export_source_data(per_cond, source_dir, "supp_s5b_condition_heterogeneity"),
    }

    fig = plt.figure(figsize=(11.0, 7.2))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.0, 1.0], height_ratios=[0.56, 1.0], wspace=0.34, hspace=0.42)

    axa = fig.add_subplot(gs[0, :])
    metric_order = ["Validity", "Novelty", "Uniqueness"]; metric_x = {m: i for i, m in enumerate(metric_order)}
    model_y = {g: len(MODEL_ORDER) - 1 - i for i, g in enumerate(MODEL_ORDER)}
    for _, row in quality_df.iterrows():
        g = str(row["generator"]); x = metric_x[str(row["metric"])]; y = model_y[g]; val = float(row["value"])
        axa.plot([x, x], [y - 0.28, y + 0.28], color="#E5E5E5", linewidth=1.0, zorder=0)
        axa.scatter([x], [y], s=56, color=MODEL_COLORS[g], marker=MODEL_MARKERS[g], edgecolors="white", linewidths=0.8, zorder=3)
        axa.text(x + 0.13, y, f"{val:.2f}", va="center", ha="left", fontsize=7.9, color="#444444")
    axa.set_xlim(-0.45, len(metric_order) - 0.05); axa.set_ylim(-0.6, len(MODEL_ORDER) - 0.4)
    axa.set_xticks(range(len(metric_order))); axa.set_xticklabels(metric_order)
    axa.set_yticks(range(len(MODEL_ORDER))); axa.set_yticklabels([MODEL_LABELS_SHORT[g] for g in MODEL_ORDER[::-1]])
    axa.set_title("Generation quality", pad=12); style_axis(axa, "x"); add_panel_label(axa, "a")

    gsb = gs[1, 0].subgridspec(1, 2, width_ratios=[0.82, 0.18], wspace=0.05)
    axb = fig.add_subplot(gsb[0, 0]); axb_r = fig.add_subplot(gsb[0, 1])
    datasets, positions, colors, labels = [], [], [], []
    for i, g in enumerate(GENERATOR_ONLY_ORDER, start=1):
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            if cfg.manuscript_mode: raise ValueError(f"Missing per-sequence similarity values for generator: {g}")
            continue
        datasets.append(vals); positions.append(i); colors.append(MODEL_COLORS[g]); labels.append(MODEL_LABELS_SHORT[g])
    draw_violin_box(axb, datasets, positions, colors, widths=0.70)
    axb.set_xticks(positions); axb.set_xticklabels(labels, rotation=18, ha="right")
    axb.set_ylabel("Per-sequence NN similarity to train"); axb.set_ylim(0.0, 0.45)
    axb.set_title("Memorization profile", pad=12); style_axis(axb, "y"); add_panel_label(axb, "b")
    retrieval_vals = cand.loc[cand["generator"] == "retrieval_reference", "nn_similarity_to_train"].dropna().to_numpy()
    if len(retrieval_vals) > 0:
        draw_violin_box(axb_r, [retrieval_vals], [1], [MODEL_COLORS["retrieval_reference"]], widths=0.54)
        axb_r.set_xticks([1]); axb_r.set_xticklabels(["Retrieval"], rotation=18, ha="right")
        axb_r.set_xlim(0.6, 1.4); style_axis(axb_r, "y"); axb_r.tick_params(axis="y", left=False, labelleft=False); axb_r.spines["left"].set_visible(False)
    else:
        axb_r.axis("off")

    axc = fig.add_subplot(gs[1, 1]); rng = np.random.default_rng(7)
    medians = []
    for g in MODEL_ORDER:
        vals = per_cond.loc[per_cond["generator"] == g, "surrogate_condition_hit_rate"].dropna().to_numpy()
        if len(vals):
            medians.append((g, float(np.median(vals))))
    ordered_models = [g for g, _ in sorted(medians, key=lambda x: x[1], reverse=True)]
    for i, g in enumerate(ordered_models, start=1):
        vals = per_cond.loc[per_cond["generator"] == g, "surrogate_condition_hit_rate"].dropna().to_numpy()
        violin = axc.violinplot([vals], positions=[i], widths=0.66, showmeans=False, showmedians=False, showextrema=False)
        body = violin["bodies"][0]
        body.set_facecolor(MODEL_COLORS[g]); body.set_edgecolor(MODEL_COLORS[g]); body.set_alpha(0.15); body.set_linewidth(0.8)
        jitter = rng.normal(0, 0.032, size=len(vals))
        axc.scatter(np.full(len(vals), i) + jitter, vals, s=13, color=MODEL_COLORS[g], alpha=0.48, edgecolors="none")
        med = float(np.median(vals)); q1 = float(np.quantile(vals, 0.25)); q3 = float(np.quantile(vals, 0.75))
        axc.plot([i - 0.15, i + 0.15], [med, med], color="#111111", linewidth=1.35)
        axc.vlines(i, q1, q3, color="#111111", linewidth=0.95)
    axc.set_xticks(np.arange(1, len(ordered_models) + 1)); axc.set_xticklabels([MODEL_LABELS_SHORT[g] for g in ordered_models], rotation=18, ha="right")
    axc.set_ylabel("Per-condition surrogate hit rate"); axc.set_ylim(0.0, 1.05)
    axc.set_title("Condition heterogeneity", pad=12); style_axis(axc, "y"); add_panel_label(axc, "c")

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in src_paths.items()})
    return outputs

def run_step5_figure_redesign_v9(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    step4_model_ready_csv: str | Path = DEFAULT_STEP4_MODEL_READY,
    manuscript_mode: bool = True,
    export_png: bool = True,
    export_pdf: bool = True,
    export_tiff: bool = True,
    dpi_png: int = 300,
    dpi_tiff: int = 600,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureConfig(
        step5_root=Path(step5_root),
        step4_model_ready_csv=Path(step4_model_ready_csv),
        manuscript_mode=manuscript_mode,
        export_png=export_png,
        export_pdf=export_pdf,
        export_tiff=export_tiff,
        dpi_png=dpi_png,
        dpi_tiff=dpi_tiff,
    )
    apply_global_style()
    outdirs = ensure_dirs(cfg.step5_root)
    tables = load_tables(cfg)
    desc = prep_descriptor_benchmark(require_df(tables["descriptor_benchmark"], "descriptor_benchmark"))
    gen = prep_generator_benchmark(require_df(tables["generator_benchmark"], "generator_benchmark"))
    cand = prep_candidate_metrics(require_df(tables["candidate_metrics"], "candidate_metrics"))
    per_cond = prep_per_condition_metrics(require_df(tables["per_condition_metrics"], "per_condition_metrics"))
    model_ready = prep_model_ready(require_df(tables["model_ready_sequences"], "model_ready_sequences"))
    confusion_df = require_df(tables["confusion_matrix_norm"], "confusion_matrix_norm")
    perm_df = prep_permutation_importance(require_df(tables["permutation_importance"], "permutation_importance"))

    outputs: Dict[str, str] = {}
    outputs.update({f"main_{k}": v for k, v in build_main_figure(desc, gen, cand, cfg, outdirs["source"], outdirs["main"] / cfg.main_name).items()})
    outputs.update({f"supp_a_{k}": v for k, v in build_supp_descriptor_figure(model_ready, confusion_df, perm_df, cfg, outdirs["source"], outdirs["supp"] / cfg.supp_a_name, outdirs["tables"]).items()})
    outputs.update({f"supp_b_{k}": v for k, v in build_supp_generator_figure(gen, cand, per_cond, cfg, outdirs["source"], outdirs["supp"] / cfg.supp_b_name).items()})

    manifest_path = cfg.step5_root / "figure_redesign_v9_manifest.txt"
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write("Step 5 figure redesign v9 outputs\n")
        f.write(f"step5_root: {cfg.step5_root}\n")
        f.write(f"step4_model_ready_csv: {cfg.step4_model_ready_csv}\n\n")
        f.write("Detected inputs\n")
        for key, path in tables["_paths"].items():
            f.write(f"  {key}: {path}\n")
        f.write("\nOutputs\n")
        for key, value in outputs.items():
            f.write(f"  {key}: {value}\n")
    outputs["manifest"] = str(manifest_path)

    if verbose:
        print("Step 5 figure redesign v9 completed.")
        print(f"step5_root: {cfg.step5_root}")
        print(f"step4_model_ready_csv: {cfg.step4_model_ready_csv}")
        print("Detected input files:")
        for key, path in tables["_paths"].items():
            print(f"  {key}: {path}")
        print("Outputs:")
        for key, value in outputs.items():
            print(f"  {key}: {value}")

    return outputs

if __name__ == "__main__":
    run_step5_figure_redesign_v9()

Main fig 5

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# =============================================================================
# CONFIG
# =============================================================================
DEFAULT_STEP5_ROOT = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5")

MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]
GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "retrieval_reference": "Retrieval",
    "cvae_conditional": "CVAE-cond",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
}

MODEL_COLORS = {
    "retrieval_reference": "#B4C5E4",
    "cvae_conditional": "#B80018",
    "gru_conditional": "#E80C12",
    "random_conditional": "#165C63",
    "gru_unconditional": "#223E68",
    "vae_unconditional": "#8B0000",
}


@dataclass(frozen=True)
class FigureConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    output_name: str = "fig_5_step5_reference_style_v4"


# =============================================================================
# STYLE
# =============================================================================
def apply_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9.4,
            "axes.titlesize": 11.0,
            "axes.labelsize": 10.8,
            "xtick.labelsize": 8.7,
            "ytick.labelsize": 8.8,
            "legend.fontsize": 8.2,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.85,
            "xtick.major.width": 0.85,
            "ytick.major.width": 0.85,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.facecolor": "white",
        }
    )


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=0.9)
    elif grid_axis == "both":
        ax.grid(True, color="#D9D9D9", linewidth=0.75, alpha=0.9)


def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=15,
        fontweight="bold",
    )

def add_aligned_panel_label(fig: plt.Figure, ref_ax: plt.Axes, target_ax: plt.Axes, label: str) -> None:
    ref_pos = ref_ax.get_position()
    tgt_pos = target_ax.get_position()
    x = ref_pos.x0 - 0.04
    y = tgt_pos.y1 + 0.01
    fig.text(x, y, label, ha="left", va="bottom", fontsize=15, fontweight="bold")


# =============================================================================
# IO
# =============================================================================
def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


def ensure_dirs(step5_root: Path) -> Dict[str, Path]:
    main_dir = step5_root / "figures_main"
    src_dir = step5_root / "source_data_figures"
    main_dir.mkdir(parents=True, exist_ok=True)
    src_dir.mkdir(parents=True, exist_ok=True)
    return {"main": main_dir, "source": src_dir}


def save_figure(fig: plt.Figure, out_base: Path, cfg: FigureConfig) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs["pdf"] = str(p)
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs["png"] = str(p)
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
        outputs["tiff"] = str(p)
    plt.close(fig)
    return outputs


def export_source_data(df: pd.DataFrame, out_dir: Path, name: str) -> str:
    p = out_dir / f"{name}.csv"
    df.to_csv(p, index=False)
    return str(p)


def require_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")


def load_inputs(step5_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    gen_path = _find_first(step5_root, ["table_5_2_step5b_generator_benchmark.csv"])
    cand_path = _find_first(step5_root, ["step5b_generated_candidates_evaluated.csv"])
    if not gen_path or not cand_path:
        raise FileNotFoundError("Missing one or more required Step 5 inputs.")

    gen = pd.read_csv(gen_path)
    cand = pd.read_csv(cand_path)

    require_columns(
        gen,
        [
            "generator",
            "surrogate_condition_hit_rate",
            "surrogate_condition_hit_rate_ci_low",
            "surrogate_condition_hit_rate_ci_high",
            "exact_novelty_vs_train",
            "exact_novelty_vs_train_ci_low",
            "exact_novelty_vs_train_ci_high",
        ],
        "generator_benchmark",
    )
    require_columns(cand, ["generator", "nn_similarity_to_train"], "candidate_metrics")

    gen = gen[gen["generator"].isin(MODEL_ORDER)].copy()
    gen["generator"] = pd.Categorical(gen["generator"], MODEL_ORDER, ordered=True)
    gen = gen.sort_values("generator").reset_index(drop=True)
    gen["label_short"] = gen["generator"].astype(str).map(MODEL_LABELS)

    cand = cand[cand["generator"].isin(MODEL_ORDER)].copy()
    cand["nn_similarity_to_train"] = pd.to_numeric(cand["nn_similarity_to_train"], errors="coerce")
    cand = cand.dropna(subset=["nn_similarity_to_train"]).reset_index(drop=True)

    return gen, cand


# =============================================================================
# SUMMARIES
# =============================================================================
def summarize_similarity(cand: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    rows = []
    retrieval = {}
    for g in MODEL_ORDER:
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        median = float(np.median(vals))
        q1 = float(np.quantile(vals, 0.25))
        q3 = float(np.quantile(vals, 0.75))
        row = {
            "generator": g,
            "label_short": MODEL_LABELS[g],
            "median_nn_similarity": median,
            "q1": q1,
            "q3": q3,
            "iqr_low_err": median - q1,
            "iqr_high_err": q3 - median,
            "n": int(len(vals)),
        }
        if g == "retrieval_reference":
            retrieval = row
        else:
            rows.append(row)
    out = pd.DataFrame(rows)
    out["generator"] = pd.Categorical(out["generator"], GENERATOR_ONLY_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    return out, retrieval


# =============================================================================
# HELPERS
# =============================================================================
def bar_with_error(
    ax: plt.Axes,
    x: np.ndarray,
    heights: np.ndarray,
    yerr_low: np.ndarray,
    yerr_high: np.ndarray,
    colors: List[str],
    width: float = 0.68,
) -> None:
    ax.bar(x, heights, color=colors, width=width, edgecolor="none", zorder=2)
    ax.errorbar(
        x,
        heights,
        yerr=np.vstack([yerr_low, yerr_high]),
        fmt="none",
        ecolor="#444444",
        elinewidth=1.0,
        capsize=3.0,
        zorder=3,
    )


def label_bars(
    ax: plt.Axes,
    x: np.ndarray,
    heights: np.ndarray,
    yerr_high: np.ndarray,
    fmt: str = "{:.2f}",
    pad_frac: float = 0.02,
) -> None:
    ylim = ax.get_ylim()[1]
    for xi, hi, eh in zip(x, heights, yerr_high):
        ax.text(
            xi,
            hi + eh + pad_frac * ylim,
            fmt.format(hi),
            ha="center",
            va="bottom",
            fontsize=8.0,
            color="#333333",
        )


# =============================================================================
# FIGURE
# =============================================================================
def build_main_figure(gen: pd.DataFrame, cand: pd.DataFrame, cfg: FigureConfig, source_dir: Path, out_base: Path) -> Dict[str, str]:
    sim, retrieval_ref = summarize_similarity(cand)

    source_paths = {
        "panel_a_fidelity": export_source_data(
            gen[["generator", "label_short", "surrogate_condition_hit_rate", "surrogate_condition_hit_rate_ci_low", "surrogate_condition_hit_rate_ci_high"]],
            source_dir, "mainfig_reference_v4_panel_a_fidelity",
        ),
        "panel_b_fidelity_novelty": export_source_data(
            gen[[
                "generator", "label_short",
                "surrogate_condition_hit_rate", "surrogate_condition_hit_rate_ci_low", "surrogate_condition_hit_rate_ci_high",
                "exact_novelty_vs_train", "exact_novelty_vs_train_ci_low", "exact_novelty_vs_train_ci_high"
            ]],
            source_dir, "mainfig_reference_v4_panel_b_fidelity_novelty",
        ),
        "panel_c_similarity": export_source_data(
            sim,
            source_dir, "mainfig_reference_v4_panel_c_similarity",
        ),
    }

    fig = plt.figure(figsize=(11.0, 7.2))
    gs = GridSpec(
        2, 2,
        figure=fig,
        width_ratios=[1.0, 1.02],
        height_ratios=[1.0, 1.05],
        wspace=0.22,
        hspace=0.50
    )

    # ------------------------------------------------------------------
    # Panel a
    # ------------------------------------------------------------------
    axa = fig.add_subplot(gs[0, 0])

    x_a = np.array([0.0, 1.35, 2.20, 3.75, 5.15, 6.00])
    heights_a = gen["surrogate_condition_hit_rate"].to_numpy(float)
    yerr_low_a = heights_a - gen["surrogate_condition_hit_rate_ci_low"].to_numpy(float)
    yerr_high_a = gen["surrogate_condition_hit_rate_ci_high"].to_numpy(float) - heights_a
    colors_a = [MODEL_COLORS[g] for g in gen["generator"].astype(str)]

    bar_with_error(axa, x_a, heights_a, yerr_low_a, yerr_high_a, colors_a, width=0.76)
    axa.set_xlim(-0.6, 6.7)
    axa.set_ylim(0.0, max(0.80, float(np.max(heights_a + yerr_high_a)) + 0.10))
    style_axis(axa, "y")
    label_bars(axa, x_a, heights_a, yerr_high_a, fmt="{:.2f}", pad_frac=0.018)
    axa.set_ylabel("Fidelity")
    axa.set_xticks(x_a)
    axa.set_xticklabels([""] * len(x_a))
    axa.set_title("Conditional models improve fidelity", pad=14)
    for xpos in [0.7, 2.95, 4.45]:
        axa.axvline(xpos, color="#D8D8D8", linewidth=0.7)
    add_panel_label(axa, "a")

    # ------------------------------------------------------------------
    # Panel b (more space between fidelity and novelty)
    # ------------------------------------------------------------------
    gsb = gs[0, 1].subgridspec(1, 2, width_ratios=[1.0, 1.0], wspace=0.30)
    axb1 = fig.add_subplot(gsb[0, 0])
    axb2 = fig.add_subplot(gsb[0, 1])

    x_b = np.arange(len(MODEL_ORDER))
    colors_b = [MODEL_COLORS[g] for g in MODEL_ORDER]
    df_index = gen.set_index("generator").loc[MODEL_ORDER]

    # Fidelity
    heights_b1 = df_index["surrogate_condition_hit_rate"].to_numpy(float)
    yerr_low_b1 = heights_b1 - df_index["surrogate_condition_hit_rate_ci_low"].to_numpy(float)
    yerr_high_b1 = df_index["surrogate_condition_hit_rate_ci_high"].to_numpy(float) - heights_b1

    bar_with_error(axb1, x_b, heights_b1, yerr_low_b1, yerr_high_b1, colors_b, width=0.78)
    axb1.set_ylim(0.0, max(0.80, float(np.max(heights_b1 + yerr_high_b1)) + 0.10))
    style_axis(axb1, "y")
    label_bars(axb1, x_b, heights_b1, yerr_high_b1, fmt="{:.2f}", pad_frac=0.018)
    axb1.set_ylabel("Fraction")
    axb1.set_xticks(x_b)
    axb1.set_xticklabels([""] * len(x_b))
    axb1.set_title("Fidelity", pad=10)
    add_panel_label(axb1, "b")

    # Novelty
    heights_b2 = df_index["exact_novelty_vs_train"].to_numpy(float)
    yerr_low_b2 = heights_b2 - df_index["exact_novelty_vs_train_ci_low"].to_numpy(float)
    yerr_high_b2 = df_index["exact_novelty_vs_train_ci_high"].to_numpy(float) - heights_b2

    bar_with_error(axb2, x_b, heights_b2, yerr_low_b2, yerr_high_b2, colors_b, width=0.78)
    axb2.set_ylim(0.0, 1.05)
    style_axis(axb2, "y")
    label_bars(axb2, x_b, heights_b2, yerr_high_b2, fmt="{:.2f}", pad_frac=0.010)
    axb2.set_xticks(x_b)
    axb2.set_xticklabels([""] * len(x_b))
    axb2.set_title("Novelty", pad=10)

    # ------------------------------------------------------------------
    # Panel c
    # ------------------------------------------------------------------
    axc = fig.add_subplot(gs[1, :])

    x_c = np.array([0.0, 1.35, 2.70, 4.05, 5.40])
    heights_c = sim["median_nn_similarity"].to_numpy(float)
    yerr_low_c = sim["iqr_low_err"].to_numpy(float)
    yerr_high_c = sim["iqr_high_err"].to_numpy(float)
    colors_c = [MODEL_COLORS[g] for g in sim["generator"].astype(str)]

    bar_with_error(axc, x_c, heights_c, yerr_low_c, yerr_high_c, colors_c, width=0.98)
    axc.set_ylim(0.0, max(0.34, float(np.max(heights_c + yerr_high_c)) + 0.08))
    axc.set_xlim(-0.8, 6.2)
    style_axis(axc, "y")
    label_bars(axc, x_c, heights_c, yerr_high_c, fmt="{:.2f}", pad_frac=0.020)
    axc.set_ylabel("NN similarity")
    axc.set_xticks(x_c)
    axc.set_xticklabels([""] * len(x_c))
    axc.set_title("Similarity to training set", pad=16)
    add_aligned_panel_label(fig, axa, axc, "c")

    # retrieval note moved inside panel c, removed upper title
    if retrieval_ref:
        med = float(retrieval_ref["median_nn_similarity"])
        q1 = float(retrieval_ref["q1"])
        q3 = float(retrieval_ref["q3"])
        axc.axhline(med, color=MODEL_COLORS["retrieval_reference"], linestyle="--", linewidth=1.1, alpha=0.95)
        axc.text(
            0.995,
            0.98,
            f"Retrieval ref. median {med:.2f}",
            transform=axc.transAxes,
            ha="right",
            va="top",
            fontsize=8.0,
            color="#666666",
        )

    # ------------------------------------------------------------------
    # Shared boxed legend like reference
    # ------------------------------------------------------------------
    legend_handles = [
        Patch(facecolor=MODEL_COLORS[g], edgecolor="none", label=MODEL_LABELS[g])
        for g in MODEL_ORDER
    ]
    leg = fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, -0.015),
        columnspacing=1.5,
        handlelength=1.5,
        handleheight=0.8,
        fancybox=True,
    )
    frame = leg.get_frame()
    frame.set_facecolor("#F5F5F5")
    frame.set_edgecolor("#BFBFBF")
    frame.set_linewidth(1.0)
    frame.set_alpha(1.0)

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs


def run_step5_mainfig_reference_style_v4(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    export_png: bool = True,
    export_pdf: bool = True,
    export_tiff: bool = True,
    dpi_png: int = 300,
    dpi_tiff: int = 600,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureConfig(
        step5_root=Path(step5_root),
        export_png=export_png,
        export_pdf=export_pdf,
        export_tiff=export_tiff,
        dpi_png=dpi_png,
        dpi_tiff=dpi_tiff,
    )
    apply_style()
    outdirs = ensure_dirs(cfg.step5_root)
    gen, cand = load_inputs(cfg.step5_root)

    outputs = build_main_figure(
        gen=gen,
        cand=cand,
        cfg=cfg,
        source_dir=outdirs["source"],
        out_base=outdirs["main"] / cfg.output_name,
    )

    manifest_path = cfg.step5_root / "mainfig_reference_style_v4_manifest.txt"
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write("Main Fig. 5 reference-style v4 outputs\n")
        f.write(f"step5_root: {cfg.step5_root}\n")
        for k, v in outputs.items():
            f.write(f"{k}: {v}\n")
    outputs["manifest"] = str(manifest_path)

    if verbose:
        print("Main Fig. 5 reference-style v3 completed.")
        for k, v in outputs.items():
            print(f"{k}: {v}")
    return outputs


if __name__ == "__main__":
    run_step5_mainfig_reference_style_v4()

amazing deisgn for fig 5 main 

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# =============================================================================
# CONFIG
# =============================================================================
DEFAULT_STEP5_ROOT = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5")

MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]
GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "retrieval_reference": "Retrieval",
    "cvae_conditional": "CVAE-cond",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
}

MODEL_COLORS = {
    "retrieval_reference": "#B4C5E4",
    "cvae_conditional": "#B80018",
    "gru_conditional": "#E80C12",
    "random_conditional": "#165C63",
    "gru_unconditional": "#223E68",
    "vae_unconditional": "#8B0000",
}


@dataclass(frozen=True)
class FigureConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    output_name: str = "fig_5_step5_reference_style_v5"


# =============================================================================
# STYLE
# =============================================================================
def apply_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9.4,
            "axes.titlesize": 10.8,
            "axes.labelsize": 10.6,
            "xtick.labelsize": 8.6,
            "ytick.labelsize": 8.8,
            "legend.fontsize": 8.1,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.85,
            "xtick.major.width": 0.85,
            "ytick.major.width": 0.85,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.facecolor": "white",
        }
    )


def style_axis(ax: plt.Axes, grid_axis: str = "y", grid_alpha: float = 0.9) -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "both":
        ax.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)


def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=15,
        fontweight="bold",
    )


def add_aligned_panel_label(fig: plt.Figure, ref_ax: plt.Axes, target_ax: plt.Axes, label: str) -> None:
    ref_pos = ref_ax.get_position()
    tgt_pos = target_ax.get_position()
    x = ref_pos.x0 - 0.04
    y = tgt_pos.y1 + 0.01
    fig.text(x, y, label, ha="left", va="bottom", fontsize=15, fontweight="bold")


# =============================================================================
# IO
# =============================================================================
def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


def ensure_dirs(step5_root: Path) -> Dict[str, Path]:
    main_dir = step5_root / "figures_main"
    src_dir = step5_root / "source_data_figures"
    main_dir.mkdir(parents=True, exist_ok=True)
    src_dir.mkdir(parents=True, exist_ok=True)
    return {"main": main_dir, "source": src_dir}


def save_figure(fig: plt.Figure, out_base: Path, cfg: FigureConfig) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs["pdf"] = str(p)
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs["png"] = str(p)
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
        outputs["tiff"] = str(p)
    plt.close(fig)
    return outputs


def export_source_data(df: pd.DataFrame, out_dir: Path, name: str) -> str:
    p = out_dir / f"{name}.csv"
    df.to_csv(p, index=False)
    return str(p)


def require_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")


def load_inputs(step5_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    gen_path = _find_first(step5_root, ["table_5_2_step5b_generator_benchmark.csv"])
    cand_path = _find_first(step5_root, ["step5b_generated_candidates_evaluated.csv"])
    if not gen_path or not cand_path:
        raise FileNotFoundError("Missing one or more required Step 5 inputs.")

    gen = pd.read_csv(gen_path)
    cand = pd.read_csv(cand_path)

    require_columns(
        gen,
        [
            "generator",
            "surrogate_condition_hit_rate",
            "surrogate_condition_hit_rate_ci_low",
            "surrogate_condition_hit_rate_ci_high",
            "exact_novelty_vs_train",
            "exact_novelty_vs_train_ci_low",
            "exact_novelty_vs_train_ci_high",
        ],
        "generator_benchmark",
    )
    require_columns(cand, ["generator", "nn_similarity_to_train"], "candidate_metrics")

    gen = gen[gen["generator"].isin(MODEL_ORDER)].copy()
    gen["generator"] = pd.Categorical(gen["generator"], MODEL_ORDER, ordered=True)
    gen = gen.sort_values("generator").reset_index(drop=True)
    gen["label_short"] = gen["generator"].astype(str).map(MODEL_LABELS)

    cand = cand[cand["generator"].isin(MODEL_ORDER)].copy()
    cand["nn_similarity_to_train"] = pd.to_numeric(cand["nn_similarity_to_train"], errors="coerce")
    cand = cand.dropna(subset=["nn_similarity_to_train"]).reset_index(drop=True)

    return gen, cand


# =============================================================================
# SUMMARIES
# =============================================================================
def summarize_similarity(cand: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    rows = []
    retrieval = {}
    for g in MODEL_ORDER:
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        median = float(np.median(vals))
        q1 = float(np.quantile(vals, 0.25))
        q3 = float(np.quantile(vals, 0.75))
        row = {
            "generator": g,
            "label_short": MODEL_LABELS[g],
            "median_nn_similarity": median,
            "q1": q1,
            "q3": q3,
            "iqr_low_err": median - q1,
            "iqr_high_err": q3 - median,
            "n": int(len(vals)),
        }
        if g == "retrieval_reference":
            retrieval = row
        else:
            rows.append(row)
    out = pd.DataFrame(rows)
    out["generator"] = pd.Categorical(out["generator"], GENERATOR_ONLY_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    return out, retrieval


# =============================================================================
# HELPERS
# =============================================================================
def bar_with_error(
    ax: plt.Axes,
    x: np.ndarray,
    heights: np.ndarray,
    yerr_low: np.ndarray,
    yerr_high: np.ndarray,
    colors: List[str],
    width: float = 0.68,
) -> None:
    ax.bar(x, heights, color=colors, width=width, edgecolor="none", zorder=2)
    ax.errorbar(
        x,
        heights,
        yerr=np.vstack([yerr_low, yerr_high]),
        fmt="none",
        ecolor="#444444",
        elinewidth=1.0,
        capsize=3.0,
        zorder=3,
    )


def label_bars(
    ax: plt.Axes,
    x: np.ndarray,
    heights: np.ndarray,
    yerr_high: np.ndarray,
    fmt: str = "{:.2f}",
    pad_frac: float = 0.018,
    fontsize: float = 8.0,
) -> None:
    ylim = ax.get_ylim()[1]
    for xi, hi, eh in zip(x, heights, yerr_high):
        ax.text(
            xi,
            hi + eh + pad_frac * ylim,
            fmt.format(hi),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#333333",
        )


# =============================================================================
# FIGURE
# =============================================================================
def build_main_figure(gen: pd.DataFrame, cand: pd.DataFrame, cfg: FigureConfig, source_dir: Path, out_base: Path) -> Dict[str, str]:
    sim, retrieval_ref = summarize_similarity(cand)

    source_paths = {
        "panel_a_fidelity": export_source_data(
            gen[["generator", "label_short", "surrogate_condition_hit_rate", "surrogate_condition_hit_rate_ci_low", "surrogate_condition_hit_rate_ci_high"]],
            source_dir, "mainfig_reference_v5_panel_a_fidelity",
        ),
        "panel_b_fidelity_novelty": export_source_data(
            gen[[
                "generator", "label_short",
                "surrogate_condition_hit_rate", "surrogate_condition_hit_rate_ci_low", "surrogate_condition_hit_rate_ci_high",
                "exact_novelty_vs_train", "exact_novelty_vs_train_ci_low", "exact_novelty_vs_train_ci_high"
            ]],
            source_dir, "mainfig_reference_v5_panel_b_fidelity_novelty",
        ),
        "panel_c_similarity": export_source_data(
            sim,
            source_dir, "mainfig_reference_v5_panel_c_similarity",
        ),
    }

    fig = plt.figure(figsize=(11.0, 7.15))
    gs = GridSpec(
        2, 2,
        figure=fig,
        width_ratios=[1.0, 1.02],
        height_ratios=[1.0, 1.05],
        wspace=0.22,
        hspace=0.50
    )

    # ------------------------------------------------------------------
    # Panel a
    # ------------------------------------------------------------------
    axa = fig.add_subplot(gs[0, 0])

    x_a = np.array([0.0, 1.35, 2.20, 3.75, 5.15, 6.00])
    heights_a = gen["surrogate_condition_hit_rate"].to_numpy(float)
    yerr_low_a = heights_a - gen["surrogate_condition_hit_rate_ci_low"].to_numpy(float)
    yerr_high_a = gen["surrogate_condition_hit_rate_ci_high"].to_numpy(float) - heights_a
    colors_a = [MODEL_COLORS[g] for g in gen["generator"].astype(str)]

    bar_with_error(axa, x_a, heights_a, yerr_low_a, yerr_high_a, colors_a, width=0.76)
    axa.set_xlim(-0.6, 6.7)
    axa.set_ylim(0.0, max(0.80, float(np.max(heights_a + yerr_high_a)) + 0.10))
    style_axis(axa, "y")
    label_bars(axa, x_a, heights_a, yerr_high_a, fmt="{:.2f}", pad_frac=0.016)
    axa.set_ylabel("Fidelity")
    axa.set_xticks(x_a)
    axa.set_xticklabels([""] * len(x_a))
    axa.set_title("Fidelity benchmark", pad=14)
    add_panel_label(axa, "a")

    # ------------------------------------------------------------------
    # Panel b
    # ------------------------------------------------------------------
    gsb = gs[0, 1].subgridspec(1, 2, width_ratios=[1.0, 1.0], wspace=0.30)
    axb1 = fig.add_subplot(gsb[0, 0])
    axb2 = fig.add_subplot(gsb[0, 1])

    x_b = np.arange(len(MODEL_ORDER))
    colors_b = [MODEL_COLORS[g] for g in MODEL_ORDER]
    df_index = gen.set_index("generator").loc[MODEL_ORDER]

    heights_b1 = df_index["surrogate_condition_hit_rate"].to_numpy(float)
    yerr_low_b1 = heights_b1 - df_index["surrogate_condition_hit_rate_ci_low"].to_numpy(float)
    yerr_high_b1 = df_index["surrogate_condition_hit_rate_ci_high"].to_numpy(float) - heights_b1

    bar_with_error(axb1, x_b, heights_b1, yerr_low_b1, yerr_high_b1, colors_b, width=0.78)
    axb1.set_ylim(0.0, max(0.80, float(np.max(heights_b1 + yerr_high_b1)) + 0.10))
    style_axis(axb1, "y")
    label_bars(axb1, x_b, heights_b1, yerr_high_b1, fmt="{:.2f}", pad_frac=0.016)
    axb1.set_ylabel("Fraction")
    axb1.set_xticks(x_b)
    axb1.set_xticklabels([""] * len(x_b))
    axb1.set_title("Fidelity", pad=10)
    add_panel_label(axb1, "b")

    heights_b2 = df_index["exact_novelty_vs_train"].to_numpy(float)
    yerr_low_b2 = heights_b2 - df_index["exact_novelty_vs_train_ci_low"].to_numpy(float)
    yerr_high_b2 = df_index["exact_novelty_vs_train_ci_high"].to_numpy(float) - heights_b2

    bar_with_error(axb2, x_b, heights_b2, yerr_low_b2, yerr_high_b2, colors_b, width=0.78)
    axb2.set_ylim(0.0, 1.05)
    style_axis(axb2, "y", grid_alpha=0.6)
    label_bars(axb2, x_b, heights_b2, yerr_high_b2, fmt="{:.2f}", pad_frac=0.008, fontsize=7.8)
    axb2.set_xticks(x_b)
    axb2.set_xticklabels([""] * len(x_b))
    axb2.set_title("Novelty", pad=10)

    # ------------------------------------------------------------------
    # Panel c
    # ------------------------------------------------------------------
    axc = fig.add_subplot(gs[1, :])

    x_c = np.array([0.0, 1.35, 2.70, 4.05, 5.40])
    heights_c = sim["median_nn_similarity"].to_numpy(float)
    yerr_low_c = sim["iqr_low_err"].to_numpy(float)
    yerr_high_c = sim["iqr_high_err"].to_numpy(float)
    colors_c = [MODEL_COLORS[g] for g in sim["generator"].astype(str)]

    bar_with_error(axc, x_c, heights_c, yerr_low_c, yerr_high_c, colors_c, width=0.98)
    axc.set_ylim(0.0, max(0.34, float(np.max(heights_c + yerr_high_c)) + 0.08))
    axc.set_xlim(-0.8, 6.2)
    style_axis(axc, "y")
    label_bars(axc, x_c, heights_c, yerr_high_c, fmt="{:.2f}", pad_frac=0.018)
    axc.set_ylabel("NN similarity")
    axc.set_xticks(x_c)
    axc.set_xticklabels([""] * len(x_c))
    axc.set_title("Similarity to training set", pad=16)
    add_aligned_panel_label(fig, axa, axc, "c")

    if retrieval_ref:
        med = float(retrieval_ref["median_nn_similarity"])
        axc.axhline(med, color=MODEL_COLORS["retrieval_reference"], linestyle="--", linewidth=1.1, alpha=0.95)
        axc.text(
            0.995,
            0.94,
            f"Retrieval ref. = {med:.2f}",
            transform=axc.transAxes,
            ha="right",
            va="top",
            fontsize=8.0,
            color="#666666",
        )

    # ------------------------------------------------------------------
    # Shared boxed legend
    # ------------------------------------------------------------------
    legend_handles = [
        Patch(facecolor=MODEL_COLORS[g], edgecolor="none", label=MODEL_LABELS[g])
        for g in MODEL_ORDER
    ]
    leg = fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, -0.015),
        columnspacing=1.35,
        handlelength=1.4,
        handleheight=0.75,
        borderpad=0.35,
        labelspacing=0.45,
        fancybox=True,
    )
    frame = leg.get_frame()
    frame.set_facecolor("#F5F5F5")
    frame.set_edgecolor("#BFBFBF")
    frame.set_linewidth(1.0)
    frame.set_alpha(1.0)

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs


def run_step5_mainfig_reference_style_v5(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    export_png: bool = True,
    export_pdf: bool = True,
    export_tiff: bool = True,
    dpi_png: int = 300,
    dpi_tiff: int = 600,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureConfig(
        step5_root=Path(step5_root),
        export_png=export_png,
        export_pdf=export_pdf,
        export_tiff=export_tiff,
        dpi_png=dpi_png,
        dpi_tiff=dpi_tiff,
    )
    apply_style()
    outdirs = ensure_dirs(cfg.step5_root)
    gen, cand = load_inputs(cfg.step5_root)

    outputs = build_main_figure(
        gen=gen,
        cand=cand,
        cfg=cfg,
        source_dir=outdirs["source"],
        out_base=outdirs["main"] / cfg.output_name,
    )

    manifest_path = cfg.step5_root / "mainfig_reference_style_v5_manifest.txt"
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write("Main Fig. 5 reference-style v5 outputs\n")
        f.write(f"step5_root: {cfg.step5_root}\n")
        for k, v in outputs.items():
            f.write(f"{k}: {v}\n")
    outputs["manifest"] = str(manifest_path)

    if verbose:
        print("Main Fig. 5 reference-style v5 completed.")
        for k, v in outputs.items():
            print(f"{k}: {v}")
    return outputs


if __name__ == "__main__":
    run_step5_mainfig_reference_style_v5()

new sub fig 5 deisgn

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# =============================================================================
# CONFIG
# =============================================================================
DEFAULT_STEP5_ROOT = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5")
DEFAULT_STEP4_MODEL_READY = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv")

MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]
GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "retrieval_reference": "Retrieval",
    "cvae_conditional": "CVAE-cond",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
}

MODEL_COLORS = {
    "retrieval_reference": "#B4C5E4",
    "cvae_conditional": "#B80018",
    "gru_conditional": "#E80C12",
    "random_conditional": "#165C63",
    "gru_unconditional": "#223E68",
    "vae_unconditional": "#8B0000",
}

DESCRIPTOR_ORDER = [
    "Full numeric descriptor set",
    "Computable descriptors",
    "Physicochemical only",
    "AA composition only",
    "Shuffled-label control",
]
DESCRIPTOR_LABELS = {
    "Full numeric descriptor set": "Full numeric",
    "Computable descriptors": "Computable",
    "Physicochemical only": "Physicochemical",
    "AA composition only": "AA composition",
    "Shuffled-label control": "Shuffled control",
}
DESCRIPTOR_COLORS = {
    "Full numeric descriptor set": "#223E68",
    "Computable descriptors": "#165C63",
    "Physicochemical only": "#F2C230",
    "AA composition only": "#6A1B8A",
    "Shuffled-label control": "#7F7F7F",
}


@dataclass(frozen=True)
class FigureConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    step4_model_ready_csv: Path = DEFAULT_STEP4_MODEL_READY
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    supp_a_name: str = "fig_s5a_reference_style_v1"
    supp_b_name: str = "fig_s5b_reference_style_v1"


# =============================================================================
# STYLE
# =============================================================================
def apply_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9.3,
            "axes.titlesize": 10.8,
            "axes.labelsize": 10.6,
            "xtick.labelsize": 8.6,
            "ytick.labelsize": 8.7,
            "legend.fontsize": 8.1,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.85,
            "xtick.major.width": 0.85,
            "ytick.major.width": 0.85,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.facecolor": "white",
        }
    )


def style_axis(ax: plt.Axes, grid_axis: str = "y", grid_alpha: float = 0.9) -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "both":
        ax.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)


def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.12, 1.05, label,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=15, fontweight="bold",
    )


def add_aligned_panel_label(fig: plt.Figure, ref_ax: plt.Axes, target_ax: plt.Axes, label: str) -> None:
    ref_pos = ref_ax.get_position()
    tgt_pos = target_ax.get_position()
    x = ref_pos.x0 - 0.04
    y = tgt_pos.y1 + 0.01
    fig.text(x, y, label, ha="left", va="bottom", fontsize=15, fontweight="bold")


# =============================================================================
# IO
# =============================================================================
def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


def ensure_dirs(step5_root: Path) -> Dict[str, Path]:
    supp_dir = step5_root / "figures_supplementary"
    src_dir = step5_root / "source_data_figures"
    tables_dir = step5_root / "tables"
    supp_dir.mkdir(parents=True, exist_ok=True)
    src_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)
    return {"supp": supp_dir, "source": src_dir, "tables": tables_dir}


def save_figure(fig: plt.Figure, out_base: Path, cfg: FigureConfig) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs["pdf"] = str(p)
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs["png"] = str(p)
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
        outputs["tiff"] = str(p)
    plt.close(fig)
    return outputs


def export_source_data(df: pd.DataFrame, out_dir: Path, name: str) -> str:
    p = out_dir / f"{name}.csv"
    df.to_csv(p, index=False)
    return str(p)


def read_csv_or_none(path: Optional[Path]) -> Optional[pd.DataFrame]:
    if path is None or not path.exists():
        return None
    return pd.read_csv(path)


def require_df(df: Optional[pd.DataFrame], name: str) -> pd.DataFrame:
    if df is None:
        raise FileNotFoundError(f"Required table missing: {name}")
    return df.copy()


def require_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"Could not resolve any of columns: {candidates}")
    return None


def load_tables(cfg: FigureConfig) -> Dict[str, Optional[pd.DataFrame]]:
    root = cfg.step5_root
    model_ready_path = cfg.step4_model_ready_csv if cfg.step4_model_ready_csv.exists() else _find_first(
        root,
        [
            "step4_model_ready_sequences.csv",
            "../step4/tables/step4_model_ready_sequences.csv",
            "../../step4/tables/step4_model_ready_sequences.csv",
        ],
    )
    paths = {
        "generator_benchmark": _find_first(root, ["table_5_2_step5b_generator_benchmark.csv"]),
        "per_condition_metrics": _find_first(root, ["table_s5_4_step5b_per_condition_metrics.csv"]),
        "candidate_metrics": _find_first(root, ["step5b_generated_candidates_evaluated.csv"]),
        "confusion_matrix_norm": _find_first(root, ["step5a_confusion_matrix_normalized.csv"]),
        "permutation_importance": _find_first(root, ["step5a_permutation_importance.csv"]),
        "model_ready_sequences": model_ready_path,
    }
    tables = {k: read_csv_or_none(v) for k, v in paths.items()}
    tables["_paths"] = paths
    return tables


# =============================================================================
# PREP
# =============================================================================
def prep_generator_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(
        df,
        [
            "generator",
            "surrogate_condition_hit_rate",
            "surrogate_condition_hit_rate_ci_low",
            "surrogate_condition_hit_rate_ci_high",
            "exact_novelty_vs_train",
            "exact_novelty_vs_train_ci_low",
            "exact_novelty_vs_train_ci_high",
            "valid_rate",
            "valid_rate_ci_low",
            "valid_rate_ci_high",
            "unique_rate_among_valid",
            "unique_rate_among_valid_ci_low",
            "unique_rate_among_valid_ci_high",
        ],
        "generator_benchmark",
    )
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["generator"] = pd.Categorical(out["generator"], MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["label_short"] = out["generator"].astype(str).map(MODEL_LABELS)
    return out


def prep_candidate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "nn_similarity_to_train"], "candidate_metrics")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["nn_similarity_to_train"] = pd.to_numeric(out["nn_similarity_to_train"], errors="coerce")
    out = out.dropna(subset=["nn_similarity_to_train"]).reset_index(drop=True)
    return out


def prep_per_condition_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "surrogate_condition_hit_rate"], "per_condition_metrics")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["surrogate_condition_hit_rate"] = pd.to_numeric(out["surrogate_condition_hit_rate"], errors="coerce")
    out = out.dropna(subset=["surrogate_condition_hit_rate"]).reset_index(drop=True)
    return out


def infer_split_values_from_any_column(df: pd.DataFrame) -> Optional[pd.Series]:
    for col in df.columns:
        series = df[col].astype(str).str.strip().str.lower()
        mapped = []
        found = False
        for v in series:
            label = None
            if "train" in v:
                label = "train"; found = True
            elif "val" in v or "valid" in v or "dev" in v:
                label = "val"; found = True
            elif "test" in v or "hold" in v:
                label = "test"; found = True
            mapped.append(label)
        if found:
            out = pd.Series(mapped, index=df.index, dtype="object")
            if out.notna().sum() > 0:
                return out
    return None


def prep_model_ready(df: pd.DataFrame) -> pd.DataFrame:
    cond_col = resolve_column(df, ["primary_condition_id", "condition", "condition_id", "label", "primary_condition"], required=True)
    split_col = resolve_column(df, ["split", "subset", "partition", "group", "data_split"], required=False)
    out = df.copy()
    if split_col is None:
        inferred = infer_split_values_from_any_column(out)
        if inferred is not None:
            out["split"] = inferred
            split_col = "split"
    if split_col is None:
        out["split"] = "all"
        split_col = "split"
    out = out.rename(columns={cond_col: "primary_condition_id", split_col: "split"}).copy()
    out["primary_condition_id"] = out["primary_condition_id"].astype(str)
    out["split"] = out["split"].astype(str).str.strip().str.lower()
    split_map = {
        "training": "train", "train": "train",
        "valid": "val", "validation": "val", "val": "val", "dev": "val",
        "testing": "test", "test": "test", "holdout": "test", "heldout": "test", "held-out": "test",
        "all": "all",
    }
    out["split"] = out["split"].map(lambda x: split_map.get(x, x))
    return out


def prep_confusion(df: pd.DataFrame) -> Tuple[np.ndarray, List[str], List[str]]:
    if np.issubdtype(df.iloc[:, 0].dtype, np.number):
        arr = df.to_numpy(float)
        rows = [f"C{i+1:02d}" for i in range(arr.shape[0])]
        cols = [f"C{i+1:02d}" for i in range(arr.shape[1])]
        return arr, rows, cols
    arr = df.iloc[:, 1:].to_numpy(float)
    row_labels = df.iloc[:, 0].astype(str).tolist()
    col_labels = df.columns[1:].astype(str).tolist()
    return arr, row_labels, col_labels


def prep_permutation_importance(df: pd.DataFrame) -> pd.DataFrame:
    feature_col = resolve_column(df, ["feature", "descriptor", "variable"], required=True)
    value_col = resolve_column(df, ["importance", "permutation_importance", "macro_f1_importance", "value"], required=False)
    if value_col is None:
        numeric = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric:
            raise ValueError("permutation_importance contains no numeric columns.")
        value_col = numeric[0]
    out = df[[feature_col, value_col]].copy()
    out.columns = ["feature", "importance"]
    out["importance"] = pd.to_numeric(out["importance"], errors="coerce")
    out = out.dropna(subset=["importance"]).sort_values("importance", ascending=False).reset_index(drop=True)
    return out


def clean_feature_name(name: str) -> str:
    mapping = {
        "net_charge_proxy": "Net charge proxy",
        "net_charge_pH7": "Net charge (pH 7)",
        "net_charge_pH7_z": "Net charge (pH 7)",
        "mean_hydropathy": "Mean hydropathy",
        "hydrophobicity_KD": "Hydropathy (Kyte–Doolittle)",
        "hydrophobicity_KD_z": "Hydropathy (Kyte–Doolittle)",
        "sequence_entropy": "Entropy",
        "entropy_z": "Entropy",
        "dominant_aa_frac_z": "Dominant Aa frac",
        "length": "Length",
        "length_z": "Length (z)",
        "primary_condition_id_idx": "Condition index",
        "aa_frac_E_z": "E (z)",
        "aa_frac_F_z": "F (z)",
        "aa_frac_A_z": "A (z)",
        "aa_frac_C_z": "C (z)",
    }
    if name in mapping:
        return mapping[name]
    return name.replace("aa_frac_", "").replace("_z", " (z)").replace("_", " ").title()


def recode_condition_to_short_ids(values: Sequence[str]) -> Tuple[Dict[str, str], List[str]]:
    unique_vals = list(dict.fromkeys([str(v) for v in values]))
    mapping = {val: f"C{i+1:02d}" for i, val in enumerate(unique_vals)}
    ordered_ids = [mapping[v] for v in unique_vals]
    return mapping, ordered_ids


# =============================================================================
# SUMMARY HELPERS
# =============================================================================
def summarize_metric_bars(gen: pd.DataFrame, metric_specs: List[Tuple[str, str]]) -> pd.DataFrame:
    rows = []
    for metric, label in metric_specs:
        for _, row in gen.iterrows():
            rows.append(
                {
                    "generator": str(row["generator"]),
                    "label_short": row["label_short"],
                    "metric": label,
                    "value": float(row[metric]),
                    "ci_low": float(row[f"{metric}_ci_low"]),
                    "ci_high": float(row[f"{metric}_ci_high"]),
                }
            )
    out = pd.DataFrame(rows)
    return out


def summarize_similarity(cand: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in MODEL_ORDER:
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        median = float(np.median(vals))
        q1 = float(np.quantile(vals, 0.25))
        q3 = float(np.quantile(vals, 0.75))
        rows.append(
            {
                "generator": g,
                "label_short": MODEL_LABELS[g],
                "median": median,
                "q1": q1,
                "q3": q3,
                "iqr_low_err": median - q1,
                "iqr_high_err": q3 - median,
                "n": int(len(vals)),
            }
        )
    out = pd.DataFrame(rows)
    out["generator"] = pd.Categorical(out["generator"], MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    return out


def summarize_condition_heterogeneity(per_cond: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in MODEL_ORDER:
        vals = per_cond.loc[per_cond["generator"] == g, "surrogate_condition_hit_rate"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        median = float(np.median(vals))
        q1 = float(np.quantile(vals, 0.25))
        q3 = float(np.quantile(vals, 0.75))
        rows.append(
            {
                "generator": g,
                "label_short": MODEL_LABELS[g],
                "median": median,
                "q1": q1,
                "q3": q3,
                "iqr_low_err": median - q1,
                "iqr_high_err": q3 - median,
                "n": int(len(vals)),
            }
        )
    out = pd.DataFrame(rows)
    out["generator"] = pd.Categorical(out["generator"], MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    return out


# =============================================================================
# PLOT HELPERS
# =============================================================================
def bar_with_error(ax: plt.Axes, x: np.ndarray, heights: np.ndarray, yerr_low: np.ndarray, yerr_high: np.ndarray, colors: List[str], width: float = 0.72) -> None:
    ax.bar(x, heights, color=colors, width=width, edgecolor="none", zorder=2)
    ax.errorbar(
        x,
        heights,
        yerr=np.vstack([yerr_low, yerr_high]),
        fmt="none",
        ecolor="#444444",
        elinewidth=1.0,
        capsize=3.0,
        zorder=3,
    )


def barh_with_error(ax: plt.Axes, y: np.ndarray, widths: np.ndarray, xerr_low: np.ndarray, xerr_high: np.ndarray, colors: List[str], height: float = 0.62) -> None:
    ax.barh(y, widths, color=colors, height=height, edgecolor="none", zorder=2)
    ax.errorbar(
        widths,
        y,
        xerr=np.vstack([xerr_low, xerr_high]),
        fmt="none",
        ecolor="#444444",
        elinewidth=1.0,
        capsize=3.0,
        zorder=3,
    )


def label_bars(ax: plt.Axes, x: np.ndarray, heights: np.ndarray, yerr_high: np.ndarray, fmt: str = "{:.2f}", pad_frac: float = 0.018, fontsize: float = 8.0) -> None:
    ylim = ax.get_ylim()[1]
    for xi, hi, eh in zip(x, heights, yerr_high):
        ax.text(
            xi,
            hi + eh + pad_frac * ylim,
            fmt.format(hi),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#333333",
        )


def label_bars_h(ax: plt.Axes, y: np.ndarray, widths: np.ndarray, xerr_high: np.ndarray, fmt: str = "{:.2f}", pad_frac: float = 0.015, fontsize: float = 8.0) -> None:
    xlim = ax.get_xlim()[1]
    for yi, wi, eh in zip(y, widths, xerr_high):
        ax.text(
            wi + eh + pad_frac * xlim,
            yi,
            fmt.format(wi),
            ha="left",
            va="center",
            fontsize=fontsize,
            color="#333333",
        )


# =============================================================================
# SOURCE EXPORTS
# =============================================================================
def export_supp_a_sources(plot_support: pd.DataFrame, rank_support: pd.DataFrame, perm_top: pd.DataFrame, source_dir: Path) -> Dict[str, str]:
    return {
        "s5a_panel_a_condition_support": export_source_data(plot_support.reset_index().rename(columns={"index": "condition_id"}), source_dir, "s5a_panel_a_condition_support_v1"),
        "s5a_panel_b_support_concentration": export_source_data(rank_support, source_dir, "s5a_panel_b_support_concentration_v1"),
        "s5a_panel_c_descriptor_importance": export_source_data(perm_top, source_dir, "s5a_panel_c_descriptor_importance_v1"),
    }


def export_supp_b_sources(quality_df: pd.DataFrame, memory_df: pd.DataFrame, heter_df: pd.DataFrame, source_dir: Path) -> Dict[str, str]:
    return {
        "s5b_panel_a_generation_quality": export_source_data(quality_df, source_dir, "s5b_panel_a_generation_quality_v1"),
        "s5b_panel_b_memory_summary": export_source_data(memory_df, source_dir, "s5b_panel_b_memory_summary_v1"),
        "s5b_panel_c_condition_heterogeneity": export_source_data(heter_df, source_dir, "s5b_panel_c_condition_heterogeneity_v1"),
    }


# =============================================================================
# FIGURE BUILDERS
# =============================================================================
def build_supp_descriptor_figure(model_ready: pd.DataFrame, confusion_df: pd.DataFrame, perm_df: pd.DataFrame, cfg: FigureConfig, source_dir: Path, out_base: Path) -> Dict[str, str]:
    support = model_ready.groupby(["primary_condition_id", "split"]).size().unstack(fill_value=0)
    cond_map, _ = recode_condition_to_short_ids(support.index.astype(str).tolist())
    valid_real_splits = any(col in {"train", "val", "test"} for col in support.columns)

    plot_support = support.copy()
    if valid_real_splits:
        preferred = [c for c in ["train", "val", "test"] if c in plot_support.columns]
        plot_support = plot_support.reindex(columns=preferred)
    plot_support["support_total"] = plot_support.sum(axis=1)
    plot_support.index = [cond_map[str(i)] for i in plot_support.index]
    plot_support = plot_support.sort_values("support_total", ascending=False)

    arr, row_labels, _ = prep_confusion(confusion_df)

    rank_support = support.sum(axis=1).sort_values(ascending=False).reset_index()
    rank_support.columns = ["condition_full", "support_count"]
    rank_support["condition_id"] = rank_support["condition_full"].map(cond_map)
    rank_support["rank"] = np.arange(1, len(rank_support) + 1)
    rank_support = rank_support.sort_values("rank")
    rank_support["cum_support_frac"] = rank_support["support_count"].cumsum() / rank_support["support_count"].sum()
    rank_support["cum_condition_frac"] = rank_support["rank"] / len(rank_support)

    perm_top = perm_df.head(8).copy()
    perm_top["feature_clean"] = perm_top["feature"].astype(str).map(clean_feature_name)

    source_paths = export_supp_a_sources(
        plot_support,
        rank_support,
        perm_top[["feature", "feature_clean", "importance"]],
        source_dir,
    )

    fig = plt.figure(figsize=(10.8, 7.0))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.04, 0.96], height_ratios=[1.0, 0.88], wspace=0.42, hspace=0.44)

    # a) condition support
    axa = fig.add_subplot(gs[:, 0])
    plot_df = plot_support.iloc[::-1].copy()
    totals = plot_df["support_total"].to_numpy(float)
    y = np.arange(len(plot_df))

    axa.barh(y, totals, color=MODEL_COLORS["random_conditional"], edgecolor="none", height=0.78, zorder=2)
    if valid_real_splits and "test" in plot_df.columns:
        test_vals = plot_df["test"].to_numpy(float)
        axa.barh(y, test_vals, left=totals - test_vals, color="#E80C12", edgecolor="none", height=0.78, zorder=3)
        leg = axa.legend(
            handles=[Patch(facecolor=MODEL_COLORS["random_conditional"], edgecolor="none", label="total"),
                     Patch(facecolor="#E80C12", edgecolor="none", label="test")],
            loc="lower right", frameon=False, ncol=2
        )
    axa.set_yticks(y)
    axa.set_yticklabels(plot_df.index.tolist())
    axa.set_xlabel("Count")
    axa.set_ylabel("Condition ID")
    axa.set_title("Condition support", pad=12)
    style_axis(axa, "x")
    add_panel_label(axa, "a")

    # b) support concentration
    axb = fig.add_subplot(gs[0, 1])
    axb.plot(rank_support["cum_condition_frac"], rank_support["cum_support_frac"], color=MODEL_COLORS["random_conditional"], linewidth=2.0)
    axb.plot([0, 1], [0, 1], color="#BFBFBF", linewidth=1.0, linestyle="--")
    checkpoints = [0.25, 0.50, 0.75, 1.00]
    for q in checkpoints:
        idx = min(max(int(np.ceil(len(rank_support) * q)) - 1, 0), len(rank_support) - 1)
        row = rank_support.iloc[idx]
        axb.scatter([row["cum_condition_frac"]], [row["cum_support_frac"]], s=34, color=MODEL_COLORS["random_conditional"], edgecolors="white", linewidths=0.7, zorder=3)
    axb.set_xlim(0.0, 1.0)
    axb.set_ylim(0.0, 1.0)
    axb.set_xlabel("Cumulative fraction of conditions")
    axb.set_ylabel("Cumulative fraction of support")
    axb.set_title("Support concentration", pad=12)
    style_axis(axb, "both")
    add_panel_label(axb, "b")

    # c) descriptor importance as horizontal bars
    axc = fig.add_subplot(gs[1, 1])
    plot_imp = perm_top.iloc[::-1].copy()
    widths = plot_imp["importance"].to_numpy(float)
    colors = [DESCRIPTOR_COLORS["Physicochemical only"]] * len(plot_imp)
    if len(plot_imp) >= 2:
        colors[-1] = MODEL_COLORS["cvae_conditional"]
        colors[-2] = MODEL_COLORS["cvae_conditional"]
    y2 = np.arange(len(plot_imp))
    axc.barh(y2, widths, color=colors, edgecolor="none", height=0.58, zorder=2)
    for yi, wi in zip(y2, widths):
        axc.text(wi + max(widths) * 0.015, yi, f"{wi:.3f}" if wi < 0.1 else f"{wi:.2f}", va="center", ha="left", fontsize=7.8, color="#333333")
    axc.set_yticks(y2)
    axc.set_yticklabels(plot_imp["feature_clean"].tolist())
    axc.set_xlim(0.0, max(0.22, float(widths.max()) * 1.05))
    axc.set_xlabel("Permutation importance (macro F1)")
    axc.set_title("Descriptor importance", pad=12)
    style_axis(axc, "x")
    add_panel_label(axc, "c")

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs


def build_supp_generator_figure(gen: pd.DataFrame, cand: pd.DataFrame, per_cond: pd.DataFrame, cfg: FigureConfig, source_dir: Path, out_base: Path) -> Dict[str, str]:
    quality_df = summarize_metric_bars(
        gen,
        [
            ("valid_rate", "Validity"),
            ("exact_novelty_vs_train", "Novelty"),
            ("unique_rate_among_valid", "Uniqueness"),
        ],
    )
    memory_df = summarize_similarity(cand)
    heter_df = summarize_condition_heterogeneity(per_cond)

    source_paths = export_supp_b_sources(quality_df, memory_df, heter_df, source_dir)

    fig = plt.figure(figsize=(11.0, 7.2))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.0, 1.0], height_ratios=[0.92, 1.0], wspace=0.32, hspace=0.42)

    # a) generation quality — grouped bars
    axa = fig.add_subplot(gs[0, :])
    metric_order = ["Validity", "Novelty", "Uniqueness"]
    centers = np.array([0.0, 2.0, 4.0])
    n_models = len(MODEL_ORDER)
    bar_w = 0.16
    offsets = np.linspace(-(n_models - 1) / 2 * bar_w, (n_models - 1) / 2 * bar_w, n_models)

    for mi, metric in enumerate(metric_order):
        sub = quality_df[quality_df["metric"] == metric].set_index("generator").loc[MODEL_ORDER].reset_index()
        xs = centers[mi] + offsets
        heights = sub["value"].to_numpy(float)
        lows = heights - sub["ci_low"].to_numpy(float)
        highs = sub["ci_high"].to_numpy(float) - heights
        colors = [MODEL_COLORS[g] for g in sub["generator"].astype(str)]
        bar_with_error(axa, xs, heights, lows, highs, colors, width=bar_w * 0.92)
        label_bars(axa, xs, heights, highs, fmt="{:.2f}", pad_frac=0.010, fontsize=7.4)

    axa.set_xlim(-0.9, 4.9)
    axa.set_ylim(0.0, 1.06)
    axa.set_xticks(centers)
    axa.set_xticklabels(metric_order)
    axa.set_ylabel("Fraction")
    axa.set_title("Generation quality", pad=12)
    style_axis(axa, "y")
    add_panel_label(axa, "a")

    # b) memorization profile — bars + IQR, retrieval as reference
    axb = fig.add_subplot(gs[1, 0])
    mem_gen = memory_df[memory_df["generator"].astype(str) != "retrieval_reference"].copy()
    x_b = np.arange(len(mem_gen))
    heights_b = mem_gen["median"].to_numpy(float)
    lows_b = mem_gen["iqr_low_err"].to_numpy(float)
    highs_b = mem_gen["iqr_high_err"].to_numpy(float)
    colors_b = [MODEL_COLORS[g] for g in mem_gen["generator"].astype(str)]
    bar_with_error(axb, x_b, heights_b, lows_b, highs_b, colors_b, width=0.74)
    axb.set_ylim(0.0, max(0.34, float(np.max(heights_b + highs_b)) + 0.08))
    style_axis(axb, "y")
    label_bars(axb, x_b, heights_b, highs_b, fmt="{:.2f}", pad_frac=0.018)
    axb.set_ylabel("NN similarity")
    axb.set_xticks(x_b)
    axb.set_xticklabels([""] * len(x_b))
    axb.set_title("Memorization profile", pad=12)
    add_panel_label(axb, "b")

    retrieval_row = memory_df[memory_df["generator"].astype(str) == "retrieval_reference"]
    if len(retrieval_row) == 1:
        med = float(retrieval_row["median"].iloc[0])
        axb.axhline(med, color=MODEL_COLORS["retrieval_reference"], linestyle="--", linewidth=1.1, alpha=0.95)
        axb.text(
            0.99, 0.96,
            f"Retrieval ref. = {med:.2f}",
            transform=axb.transAxes,
            ha="right", va="top",
            fontsize=8.0, color="#666666",
        )

    # c) condition heterogeneity — bars + IQR, includes retrieval
    axc = fig.add_subplot(gs[1, 1])
    x_c = np.arange(len(heter_df))
    heights_c = heter_df["median"].to_numpy(float)
    lows_c = heter_df["iqr_low_err"].to_numpy(float)
    highs_c = heter_df["iqr_high_err"].to_numpy(float)
    colors_c = [MODEL_COLORS[g] for g in heter_df["generator"].astype(str)]
    bar_with_error(axc, x_c, heights_c, lows_c, highs_c, colors_c, width=0.74)
    axc.set_ylim(0.0, 1.05)
    style_axis(axc, "y")
    label_bars(axc, x_c, heights_c, highs_c, fmt="{:.2f}", pad_frac=0.012)
    axc.set_ylabel("Per-condition surrogate hit rate")
    axc.set_xticks(x_c)
    axc.set_xticklabels([""] * len(x_c))
    axc.set_title("Condition heterogeneity", pad=12)
    add_aligned_panel_label(fig, axa, axc, "c")

    # shared boxed legend
    legend_handles = [
        Patch(facecolor=MODEL_COLORS[g], edgecolor="none", label=MODEL_LABELS[g])
        for g in MODEL_ORDER
    ]
    leg = fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02),
        columnspacing=1.35,
        handlelength=1.4,
        handleheight=0.75,
        borderpad=0.35,
        labelspacing=0.45,
        fancybox=True,
    )
    frame = leg.get_frame()
    frame.set_facecolor("#F5F5F5")
    frame.set_edgecolor("#BFBFBF")
    frame.set_linewidth(1.0)
    frame.set_alpha(1.0)

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs


# =============================================================================
# RUNNER
# =============================================================================
def run_step5_supp_reference_style_v1(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    step4_model_ready_csv: str | Path = DEFAULT_STEP4_MODEL_READY,
    export_png: bool = True,
    export_pdf: bool = True,
    export_tiff: bool = True,
    dpi_png: int = 300,
    dpi_tiff: int = 600,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureConfig(
        step5_root=Path(step5_root),
        step4_model_ready_csv=Path(step4_model_ready_csv),
        export_png=export_png,
        export_pdf=export_pdf,
        export_tiff=export_tiff,
        dpi_png=dpi_png,
        dpi_tiff=dpi_tiff,
    )

    apply_style()
    outdirs = ensure_dirs(cfg.step5_root)
    tables = load_tables(cfg)

    gen = prep_generator_benchmark(require_df(tables["generator_benchmark"], "generator_benchmark"))
    cand = prep_candidate_metrics(require_df(tables["candidate_metrics"], "candidate_metrics"))
    per_cond = prep_per_condition_metrics(require_df(tables["per_condition_metrics"], "per_condition_metrics"))
    model_ready = prep_model_ready(require_df(tables["model_ready_sequences"], "model_ready_sequences"))
    confusion_df = require_df(tables["confusion_matrix_norm"], "confusion_matrix_norm")
    perm_df = prep_permutation_importance(require_df(tables["permutation_importance"], "permutation_importance"))

    outputs: Dict[str, str] = {}
    outputs.update(
        {
            f"supp_a_{k}": v
            for k, v in build_supp_descriptor_figure(
                model_ready=model_ready,
                confusion_df=confusion_df,
                perm_df=perm_df,
                cfg=cfg,
                source_dir=outdirs["source"],
                out_base=outdirs["supp"] / cfg.supp_a_name,
            ).items()
        }
    )
    outputs.update(
        {
            f"supp_b_{k}": v
            for k, v in build_supp_generator_figure(
                gen=gen,
                cand=cand,
                per_cond=per_cond,
                cfg=cfg,
                source_dir=outdirs["source"],
                out_base=outdirs["supp"] / cfg.supp_b_name,
            ).items()
        }
    )

    manifest_path = cfg.step5_root / "step5_supp_reference_style_v1_manifest.txt"
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write("Step 5 supplementary figures reference-style v1 outputs\n")
        f.write(f"step5_root: {cfg.step5_root}\n")
        f.write(f"step4_model_ready_csv: {cfg.step4_model_ready_csv}\n\n")
        for key, path in tables["_paths"].items():
            f.write(f"input_{key}: {path}\n")
        for key, value in outputs.items():
            f.write(f"{key}: {value}\n")
    outputs["manifest"] = str(manifest_path)

    if verbose:
        print("Step 5 supplementary figures reference-style v1 completed.")
        for key, value in outputs.items():
            print(f"{key}: {value}")
    return outputs


if __name__ == "__main__":
    run_step5_supp_reference_style_v1()

S2.5.2 v2

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# =============================================================================
# CONFIG
# =============================================================================
DEFAULT_STEP5_ROOT = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5")
DEFAULT_STEP4_MODEL_READY = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv")

MODEL_ORDER = [
    "retrieval_reference",
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]
GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "retrieval_reference": "Retrieval",
    "cvae_conditional": "CVAE-cond",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
}

MODEL_COLORS = {
    "retrieval_reference": "#B4C5E4",
    "cvae_conditional": "#B80018",
    "gru_conditional": "#E80C12",
    "random_conditional": "#165C63",
    "gru_unconditional": "#223E68",
    "vae_unconditional": "#8B0000",
}

DESCRIPTOR_COLORS = {
    "major": "#B80018",
    "minor": "#F2C230",
    "support": "#165C63",
    "support_test": "#E80C12",
}


@dataclass(frozen=True)
class FigureConfig:
    step5_root: Path = DEFAULT_STEP5_ROOT
    step4_model_ready_csv: Path = DEFAULT_STEP4_MODEL_READY
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    supp_a_name: str = "fig_s5a_reference_style_v8"
    supp_b_name: str = "fig_s5b_reference_style_v8"


# =============================================================================
# STYLE
# =============================================================================
def apply_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9.3,
            "axes.titlesize": 10.8,
            "axes.labelsize": 10.6,
            "xtick.labelsize": 8.6,
            "ytick.labelsize": 8.7,
            "legend.fontsize": 8.1,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.85,
            "xtick.major.width": 0.85,
            "ytick.major.width": 0.85,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.facecolor": "white",
        }
    )


def style_axis(ax: plt.Axes, grid_axis: str = "y", grid_alpha: float = 0.9) -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "both":
        ax.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.12, y: float = 1.05) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=15, fontweight="bold",
    )


def add_aligned_panel_label(fig: plt.Figure, x_anchor: float, target_ax: plt.Axes, label: str, y_offset: float = 0.01) -> None:
    tgt_pos = target_ax.get_position()
    x = x_anchor
    y = tgt_pos.y1 + y_offset
    fig.text(x, y, label, ha="left", va="bottom", fontsize=15, fontweight="bold")

def add_two_left_labels(fig: plt.Figure, top_ax: plt.Axes, bottom_ax: plt.Axes, x_ref_ax: plt.Axes) -> None:
    fig.canvas.draw()
    x_anchor = x_ref_ax.get_position().x0 - 0.045
    fig.text(x_anchor, top_ax.get_position().y1 + 0.01, "a", ha="left", va="bottom", fontsize=15, fontweight="bold")
    fig.text(x_anchor, bottom_ax.get_position().y1 + 0.01, "b", ha="left", va="bottom", fontsize=15, fontweight="bold")


def _find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


def ensure_dirs(step5_root: Path) -> Dict[str, Path]:
    supp_dir = step5_root / "figures_supplementary"
    src_dir = step5_root / "source_data_figures"
    tables_dir = step5_root / "tables"
    supp_dir.mkdir(parents=True, exist_ok=True)
    src_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)
    return {"supp": supp_dir, "source": src_dir, "tables": tables_dir}


def save_figure(fig: plt.Figure, out_base: Path, cfg: FigureConfig) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs["pdf"] = str(p)
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs["png"] = str(p)
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
        outputs["tiff"] = str(p)
    plt.close(fig)
    return outputs


def export_source_data(df: pd.DataFrame, out_dir: Path, name: str) -> str:
    p = out_dir / f"{name}.csv"
    df.to_csv(p, index=False)
    return str(p)


def read_csv_or_none(path: Optional[Path]) -> Optional[pd.DataFrame]:
    if path is None or not path.exists():
        return None
    return pd.read_csv(path)


def require_df(df: Optional[pd.DataFrame], name: str) -> pd.DataFrame:
    if df is None:
        raise FileNotFoundError(f"Required table missing: {name}")
    return df.copy()


def require_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"Could not resolve any of columns: {candidates}")
    return None


def load_tables(cfg: FigureConfig) -> Dict[str, Optional[pd.DataFrame]]:
    root = cfg.step5_root
    model_ready_path = cfg.step4_model_ready_csv if cfg.step4_model_ready_csv.exists() else _find_first(
        root,
        [
            "step4_model_ready_sequences.csv",
            "../step4/tables/step4_model_ready_sequences.csv",
            "../../step4/tables/step4_model_ready_sequences.csv",
        ],
    )
    paths = {
        "generator_benchmark": _find_first(root, ["table_5_2_step5b_generator_benchmark.csv"]),
        "per_condition_metrics": _find_first(root, ["table_s5_4_step5b_per_condition_metrics.csv"]),
        "candidate_metrics": _find_first(root, ["step5b_generated_candidates_evaluated.csv"]),
        "confusion_matrix_norm": _find_first(root, ["step5a_confusion_matrix_normalized.csv"]),
        "permutation_importance": _find_first(root, ["step5a_permutation_importance.csv"]),
        "model_ready_sequences": model_ready_path,
    }
    tables = {k: read_csv_or_none(v) for k, v in paths.items()}
    tables["_paths"] = paths
    return tables


# =============================================================================
# PREP
# =============================================================================
def prep_generator_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(
        df,
        [
            "generator",
            "surrogate_condition_hit_rate",
            "surrogate_condition_hit_rate_ci_low",
            "surrogate_condition_hit_rate_ci_high",
            "exact_novelty_vs_train",
            "exact_novelty_vs_train_ci_low",
            "exact_novelty_vs_train_ci_high",
            "valid_rate",
            "valid_rate_ci_low",
            "valid_rate_ci_high",
            "unique_rate_among_valid",
            "unique_rate_among_valid_ci_low",
            "unique_rate_among_valid_ci_high",
        ],
        "generator_benchmark",
    )
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["generator"] = pd.Categorical(out["generator"], MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["label_short"] = out["generator"].astype(str).map(MODEL_LABELS)
    return out


def prep_candidate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "nn_similarity_to_train"], "candidate_metrics")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["nn_similarity_to_train"] = pd.to_numeric(out["nn_similarity_to_train"], errors="coerce")
    out = out.dropna(subset=["nn_similarity_to_train"]).reset_index(drop=True)
    return out


def prep_per_condition_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "surrogate_condition_hit_rate"], "per_condition_metrics")
    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    out["surrogate_condition_hit_rate"] = pd.to_numeric(out["surrogate_condition_hit_rate"], errors="coerce")
    out = out.dropna(subset=["surrogate_condition_hit_rate"]).reset_index(drop=True)
    return out


def infer_split_values_from_any_column(df: pd.DataFrame) -> Optional[pd.Series]:
    for col in df.columns:
        series = df[col].astype(str).str.strip().str.lower()
        mapped = []
        found = False
        for v in series:
            label = None
            if "train" in v:
                label = "train"; found = True
            elif "val" in v or "valid" in v or "dev" in v:
                label = "val"; found = True
            elif "test" in v or "hold" in v:
                label = "test"; found = True
            mapped.append(label)
        if found:
            out = pd.Series(mapped, index=df.index, dtype="object")
            if out.notna().sum() > 0:
                return out
    return None


def prep_model_ready(df: pd.DataFrame) -> pd.DataFrame:
    cond_col = resolve_column(df, ["primary_condition_id", "condition", "condition_id", "label", "primary_condition"], required=True)
    split_col = resolve_column(df, ["split", "subset", "partition", "group", "data_split"], required=False)
    out = df.copy()
    if split_col is None:
        inferred = infer_split_values_from_any_column(out)
        if inferred is not None:
            out["split"] = inferred
            split_col = "split"
    if split_col is None:
        out["split"] = "all"
        split_col = "split"
    out = out.rename(columns={cond_col: "primary_condition_id", split_col: "split"}).copy()
    out["primary_condition_id"] = out["primary_condition_id"].astype(str)
    out["split"] = out["split"].astype(str).str.strip().str.lower()
    split_map = {
        "training": "train", "train": "train",
        "valid": "val", "validation": "val", "val": "val", "dev": "val",
        "testing": "test", "test": "test", "holdout": "test", "heldout": "test", "held-out": "test",
        "all": "all",
    }
    out["split"] = out["split"].map(lambda x: split_map.get(x, x))
    return out


def prep_confusion(df: pd.DataFrame) -> Tuple[np.ndarray, List[str], List[str]]:
    if np.issubdtype(df.iloc[:, 0].dtype, np.number):
        arr = df.to_numpy(float)
        rows = [f"C{i+1:02d}" for i in range(arr.shape[0])]
        cols = [f"C{i+1:02d}" for i in range(arr.shape[1])]
        return arr, rows, cols
    arr = df.iloc[:, 1:].to_numpy(float)
    row_labels = df.iloc[:, 0].astype(str).tolist()
    col_labels = df.columns[1:].astype(str).tolist()
    return arr, row_labels, col_labels


def prep_permutation_importance(df: pd.DataFrame) -> pd.DataFrame:
    feature_col = resolve_column(df, ["feature", "descriptor", "variable"], required=True)
    value_col = resolve_column(df, ["importance", "permutation_importance", "macro_f1_importance", "value"], required=False)
    if value_col is None:
        numeric = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric:
            raise ValueError("permutation_importance contains no numeric columns.")
        value_col = numeric[0]
    out = df[[feature_col, value_col]].copy()
    out.columns = ["feature", "importance"]
    out["importance"] = pd.to_numeric(out["importance"], errors="coerce")
    out = out.dropna(subset=["importance"]).sort_values("importance", ascending=False).reset_index(drop=True)
    return out


def clean_feature_name(name: str) -> str:
    mapping = {
        "net_charge_proxy": "Net charge proxy",
        "net_charge_pH7": "Net charge (pH 7)",
        "net_charge_pH7_z": "Net charge (pH 7)",
        "mean_hydropathy": "Mean hydropathy",
        "hydrophobicity_KD": "Hydropathy (Kyte–Doolittle)",
        "hydrophobicity_KD_z": "Hydropathy (Kyte–Doolittle)",
        "sequence_entropy": "Entropy",
        "entropy_z": "Entropy",
        "dominant_aa_frac_z": "Dominant Aa frac",
        "length": "Length",
        "length_z": "Length (z)",
        "primary_condition_id_idx": "Condition index",
        "aa_frac_E_z": "E (z)",
        "aa_frac_F_z": "F (z)",
        "aa_frac_A_z": "A (z)",
        "aa_frac_C_z": "C (z)",
    }
    if name in mapping:
        return mapping[name]
    return name.replace("aa_frac_", "").replace("_z", " (z)").replace("_", " ").title()


def recode_condition_to_short_ids(values: Sequence[str]) -> Tuple[Dict[str, str], List[str]]:
    unique_vals = list(dict.fromkeys([str(v) for v in values]))
    mapping = {val: f"C{i+1:02d}" for i, val in enumerate(unique_vals)}
    ordered_ids = [mapping[v] for v in unique_vals]
    return mapping, ordered_ids


# =============================================================================
# SUMMARIES
# =============================================================================
def summarize_metric_bars(gen: pd.DataFrame, metric_specs: List[Tuple[str, str]]) -> pd.DataFrame:
    rows = []
    for metric, label in metric_specs:
        for _, row in gen.iterrows():
            rows.append(
                {
                    "generator": str(row["generator"]),
                    "label_short": row["label_short"],
                    "metric": label,
                    "value": float(row[metric]),
                    "ci_low": float(row[f"{metric}_ci_low"]),
                    "ci_high": float(row[f"{metric}_ci_high"]),
                }
            )
    return pd.DataFrame(rows)


def summarize_similarity(cand: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in MODEL_ORDER:
        vals = cand.loc[cand["generator"] == g, "nn_similarity_to_train"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        median = float(np.median(vals))
        q1 = float(np.quantile(vals, 0.25))
        q3 = float(np.quantile(vals, 0.75))
        rows.append(
            {
                "generator": g,
                "label_short": MODEL_LABELS[g],
                "median": median,
                "q1": q1,
                "q3": q3,
                "iqr_low_err": median - q1,
                "iqr_high_err": q3 - median,
                "n": int(len(vals)),
            }
        )
    out = pd.DataFrame(rows)
    out["generator"] = pd.Categorical(out["generator"], MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    return out


def summarize_condition_heterogeneity(per_cond: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in MODEL_ORDER:
        vals = per_cond.loc[per_cond["generator"] == g, "surrogate_condition_hit_rate"].dropna().to_numpy()
        if len(vals) == 0:
            continue
        median = float(np.median(vals))
        q1 = float(np.quantile(vals, 0.25))
        q3 = float(np.quantile(vals, 0.75))
        rows.append(
            {
                "generator": g,
                "label_short": MODEL_LABELS[g],
                "median": median,
                "q1": q1,
                "q3": q3,
                "iqr_low_err": median - q1,
                "iqr_high_err": q3 - median,
                "n": int(len(vals)),
            }
        )
    out = pd.DataFrame(rows)
    out["generator"] = pd.Categorical(out["generator"], MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    return out


# =============================================================================
# PLOT HELPERS
# =============================================================================
def bar_with_error(ax: plt.Axes, x: np.ndarray, heights: np.ndarray, yerr_low: np.ndarray, yerr_high: np.ndarray, colors: List[str], width: float = 0.72) -> None:
    ax.bar(x, heights, color=colors, width=width, edgecolor="none", zorder=2)
    ax.errorbar(
        x,
        heights,
        yerr=np.vstack([yerr_low, yerr_high]),
        fmt="none",
        ecolor="#444444",
        elinewidth=0.95,
        capsize=3.0,
        zorder=3,
    )


def label_bars(ax: plt.Axes, x: np.ndarray, heights: np.ndarray, yerr_high: np.ndarray, fmt: str = "{:.2f}", pad_frac: float = 0.018, fontsize: float = 8.0, informative_only: bool = False) -> None:
    ylim = ax.get_ylim()[1]
    for xi, hi, eh in zip(x, heights, yerr_high):
        if informative_only and abs(hi - 1.0) < 1e-8:
            continue
        ax.text(
            xi,
            hi + eh + pad_frac * ylim,
            fmt.format(hi),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#333333",
        )


# =============================================================================
# FIGURE BUILDERS
# =============================================================================
def build_supp_descriptor_figure(model_ready: pd.DataFrame, confusion_df: pd.DataFrame, perm_df: pd.DataFrame, cfg: FigureConfig, source_dir: Path, out_base: Path) -> Dict[str, str]:
    support = model_ready.groupby(["primary_condition_id", "split"]).size().unstack(fill_value=0)
    cond_map, _ = recode_condition_to_short_ids(support.index.astype(str).tolist())
    valid_real_splits = any(col in {"train", "val", "test"} for col in support.columns)

    plot_support = support.copy()
    if valid_real_splits:
        preferred = [c for c in ["train", "val", "test"] if c in plot_support.columns]
        plot_support = plot_support.reindex(columns=preferred)
    plot_support["support_total"] = plot_support.sum(axis=1)
    plot_support.index = [cond_map[str(i)] for i in plot_support.index]
    plot_support = plot_support.sort_values("support_total", ascending=False)

    rank_support = support.sum(axis=1).sort_values(ascending=False).reset_index()
    rank_support.columns = ["condition_full", "support_count"]
    rank_support["condition_id"] = rank_support["condition_full"].map(cond_map)
    rank_support["rank"] = np.arange(1, len(rank_support) + 1)
    rank_support = rank_support.sort_values("rank")
    rank_support["cum_support_frac"] = rank_support["support_count"].cumsum() / rank_support["support_count"].sum()
    rank_support["cum_condition_frac"] = rank_support["rank"] / len(rank_support)

    perm_top = perm_df.head(8).copy()
    perm_top["feature_clean"] = perm_top["feature"].astype(str).map(clean_feature_name)

    source_paths = {
        "s5a_panel_a_condition_support": export_source_data(plot_support.reset_index().rename(columns={"index": "condition_id"}), source_dir, "s5a_panel_a_condition_support_v8"),
        "s5a_panel_b_support_concentration": export_source_data(rank_support, source_dir, "s5a_panel_b_support_concentration_v8"),
        "s5a_panel_c_descriptor_importance": export_source_data(perm_top[["feature", "feature_clean", "importance"]], source_dir, "s5a_panel_c_descriptor_importance_v8"),
    }

    fig = plt.figure(figsize=(10.8, 7.0))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.04, 0.96], height_ratios=[1.0, 0.88], wspace=0.42, hspace=0.44)

    axa = fig.add_subplot(gs[:, 0])
    plot_df = plot_support.iloc[::-1].copy()
    totals = plot_df["support_total"].to_numpy(float)
    y = np.arange(len(plot_df))
    axa.barh(y, totals, color=DESCRIPTOR_COLORS["support"], edgecolor="none", height=0.78, zorder=2)
    if valid_real_splits and "test" in plot_df.columns:
        test_vals = plot_df["test"].to_numpy(float)
        axa.barh(y, test_vals, left=totals - test_vals, color=DESCRIPTOR_COLORS["support_test"], edgecolor="none", height=0.78, zorder=3)
        axa.legend(
            handles=[
                Patch(facecolor=DESCRIPTOR_COLORS["support"], edgecolor="none", label="total"),
                Patch(facecolor=DESCRIPTOR_COLORS["support_test"], edgecolor="none", label="test"),
            ],
            loc="lower right", frameon=False, ncol=2
        )
    axa.set_yticks(y)
    axa.set_yticklabels(plot_df.index.tolist())
    axa.set_xlabel("Count")
    axa.set_ylabel("Condition ID")
    axa.set_title("Condition support", pad=12)
    style_axis(axa, "x")

    axb = fig.add_subplot(gs[0, 1])
    axb.plot(rank_support["cum_condition_frac"], rank_support["cum_support_frac"], color=DESCRIPTOR_COLORS["support"], linewidth=2.0)
    axb.plot([0, 1], [0, 1], color="#BFBFBF", linewidth=1.0, linestyle="--")
    for q in [0.25, 0.50, 0.75, 1.00]:
        idx = min(max(int(np.ceil(len(rank_support) * q)) - 1, 0), len(rank_support) - 1)
        row = rank_support.iloc[idx]
        axb.scatter([row["cum_condition_frac"]], [row["cum_support_frac"]], s=34, color=DESCRIPTOR_COLORS["support"], edgecolors="white", linewidths=0.7, zorder=3)
    axb.set_xlim(0.0, 1.0)
    axb.set_ylim(0.0, 1.0)
    axb.set_xlabel("Cumulative fraction of conditions")
    axb.set_ylabel("Cumulative fraction of support")
    axb.set_title("Support concentration", pad=12)
    style_axis(axb, "both")

    axc = fig.add_subplot(gs[1, 1])
    plot_imp = perm_top.iloc[::-1].copy()
    widths = plot_imp["importance"].to_numpy(float)
    colors = [DESCRIPTOR_COLORS["minor"]] * len(plot_imp)
    if len(plot_imp) >= 2:
        colors[-1] = DESCRIPTOR_COLORS["major"]
        colors[-2] = DESCRIPTOR_COLORS["major"]
    y2 = np.arange(len(plot_imp))
    axc.barh(y2, widths, color=colors, edgecolor="none", height=0.58, zorder=2)
    for yi, wi in zip(y2, widths):
        axc.text(wi + max(widths) * 0.015, yi, f"{wi:.3f}" if wi < 0.1 else f"{wi:.2f}", va="center", ha="left", fontsize=7.8, color="#333333")
    axc.set_yticks(y2)
    axc.set_yticklabels(plot_imp["feature_clean"].tolist())
    axc.set_xlim(0.0, max(0.22, float(widths.max()) * 1.05))
    axc.set_xlabel("Permutation importance (macro F1)")
    axc.set_title("Descriptor importance", pad=12)
    style_axis(axc, "x")
    add_panel_label(axc, "c", x=-0.12, y=1.04)
    add_two_left_labels(fig, axa, axb, axb)

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs


def build_supp_generator_figure(gen: pd.DataFrame, cand: pd.DataFrame, per_cond: pd.DataFrame, cfg: FigureConfig, source_dir: Path, out_base: Path) -> Dict[str, str]:
    quality_df = summarize_metric_bars(
        gen,
        [
            ("valid_rate", "Validity"),
            ("exact_novelty_vs_train", "Novelty"),
            ("unique_rate_among_valid", "Uniqueness"),
        ],
    )
    memory_df = summarize_similarity(cand)
    heter_df = summarize_condition_heterogeneity(per_cond)

    source_paths = {
        "s5b_panel_a_generation_quality": export_source_data(quality_df, source_dir, "s5b_panel_a_generation_quality_v8"),
        "s5b_panel_b_memory_summary": export_source_data(memory_df, source_dir, "s5b_panel_b_memory_summary_v8"),
        "s5b_panel_c_condition_heterogeneity": export_source_data(heter_df, source_dir, "s5b_panel_c_condition_heterogeneity_v8"),
    }

    fig = plt.figure(figsize=(11.0, 7.2))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.0, 1.0], height_ratios=[0.92, 1.0], wspace=0.32, hspace=0.42)

    # a) generation quality
    axa = fig.add_subplot(gs[0, :])
    metric_order = ["Validity", "Novelty", "Uniqueness"]
    centers = np.array([0.0, 2.0, 4.0])
    n_models = len(MODEL_ORDER)
    bar_w = 0.16
    offsets = np.linspace(-(n_models - 1) / 2 * bar_w, (n_models - 1) / 2 * bar_w, n_models)

    for mi, metric in enumerate(metric_order):
        sub = quality_df[quality_df["metric"] == metric].set_index("generator").loc[MODEL_ORDER].reset_index()
        xs = centers[mi] + offsets
        heights = sub["value"].to_numpy(float)
        lows = heights - sub["ci_low"].to_numpy(float)
        highs = sub["ci_high"].to_numpy(float) - heights
        colors = [MODEL_COLORS[g] for g in sub["generator"].astype(str)]
        bar_with_error(axa, xs, heights, lows, highs, colors, width=bar_w * 0.92)

        if metric == "Validity":
            # keep only retrieval plus non-saturated values to reduce clutter
            for xi, hi, eh, g in zip(xs, heights, highs, sub["generator"].astype(str)):
                if g == "retrieval_reference" or hi < 0.999:
                    axa.text(xi, hi + eh + 0.008 * axa.get_ylim()[1], f"{hi:.2f}", ha="center", va="bottom", fontsize=7.4, color="#333333")
        elif metric == "Novelty":
            # only the informative retrieval 0.00
            for xi, hi, eh, g in zip(xs, heights, highs, sub["generator"].astype(str)):
                if g == "retrieval_reference":
                    axa.text(xi, hi + eh + 0.008 * axa.get_ylim()[1], f"{hi:.2f}", ha="center", va="bottom", fontsize=7.4, color="#333333")
        elif metric == "Uniqueness":
            # only retrieval 0.92; all others are saturated at 1.00
            for xi, hi, eh, g in zip(xs, heights, highs, sub["generator"].astype(str)):
                if g == "retrieval_reference":
                    axa.text(xi, hi + eh + 0.008 * axa.get_ylim()[1], f"{hi:.2f}", ha="center", va="bottom", fontsize=7.4, color="#333333")

    axa.set_xlim(-0.95, 4.95)
    axa.set_ylim(0.0, 1.06)
    axa.set_xticks(centers)
    axa.set_xticklabels(metric_order)
    axa.set_ylabel("Fraction")
    axa.set_title("Generation quality", pad=12)
    style_axis(axa, "y")

    # b) memorization profile
    axb = fig.add_subplot(gs[1, 0])
    mem_gen = memory_df[memory_df["generator"].astype(str) != "retrieval_reference"].copy()
    x_b = np.arange(len(mem_gen))
    heights_b = mem_gen["median"].to_numpy(float)
    lows_b = mem_gen["iqr_low_err"].to_numpy(float)
    highs_b = mem_gen["iqr_high_err"].to_numpy(float)
    colors_b = [MODEL_COLORS[g] for g in mem_gen["generator"].astype(str)]
    bar_with_error(axb, x_b, heights_b, lows_b, highs_b, colors_b, width=0.74)
    axb.set_ylim(0.0, max(0.34, float(np.max(heights_b + highs_b)) + 0.08))
    style_axis(axb, "y")
    label_bars(axb, x_b, heights_b, highs_b, fmt="{:.2f}", pad_frac=0.016)
    axb.set_ylabel("NN similarity")
    axb.set_xticks(x_b)
    axb.set_xticklabels([""] * len(x_b))
    axb.set_title("Memorization profile", pad=12)

    retrieval_row = memory_df[memory_df["generator"].astype(str) == "retrieval_reference"]
    if len(retrieval_row) == 1:
        med = float(retrieval_row["median"].iloc[0])
        axb.axhline(med, color=MODEL_COLORS["retrieval_reference"], linestyle="--", linewidth=1.0, alpha=0.9)
        axb.text(
            0.99, 0.95,
            f"Retrieval ref. = {med:.2f}",
            transform=axb.transAxes,
            ha="right", va="top",
            fontsize=8.0, color="#666666",
        )

    # c) condition heterogeneity
    axc = fig.add_subplot(gs[1, 1])
    x_c = np.arange(len(heter_df))
    heights_c = heter_df["median"].to_numpy(float)
    lows_c = heter_df["iqr_low_err"].to_numpy(float)
    highs_c = heter_df["iqr_high_err"].to_numpy(float)
    colors_c = [MODEL_COLORS[g] for g in heter_df["generator"].astype(str)]
    bar_with_error(axc, x_c, heights_c, lows_c, highs_c, colors_c, width=0.72)
    axc.set_ylim(0.0, 1.05)
    style_axis(axc, "y", grid_alpha=0.8)
    label_bars(axc, x_c, heights_c, highs_c, fmt="{:.2f}", pad_frac=0.010)
    axc.set_ylabel("Per-condition surrogate hit rate")
    axc.set_xticks(x_c)
    axc.set_xticklabels([""] * len(x_c))
    axc.set_title("Condition heterogeneity", pad=12)
    add_panel_label(axc, "c", x=-0.12, y=1.04)
    add_two_left_labels(fig, axa, axb, axb)

    # shared boxed legend
    legend_handles = [
        Patch(facecolor=MODEL_COLORS[g], edgecolor="none", label=MODEL_LABELS[g])
        for g in MODEL_ORDER
    ]
    leg = fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02),
        columnspacing=1.35,
        handlelength=1.4,
        handleheight=0.75,
        borderpad=0.35,
        labelspacing=0.45,
        fancybox=True,
    )
    frame = leg.get_frame()
    frame.set_facecolor("#F5F5F5")
    frame.set_edgecolor("#BFBFBF")
    frame.set_linewidth(1.0)
    frame.set_alpha(1.0)

    outputs = save_figure(fig, out_base, cfg)
    outputs.update({f"source_{k}": v for k, v in source_paths.items()})
    return outputs


# =============================================================================
# RUNNER
# =============================================================================
def run_step5_supp_reference_style_v8(
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    step4_model_ready_csv: str | Path = DEFAULT_STEP4_MODEL_READY,
    export_png: bool = True,
    export_pdf: bool = True,
    export_tiff: bool = True,
    dpi_png: int = 300,
    dpi_tiff: int = 600,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = FigureConfig(
        step5_root=Path(step5_root),
        step4_model_ready_csv=Path(step4_model_ready_csv),
        export_png=export_png,
        export_pdf=export_pdf,
        export_tiff=export_tiff,
        dpi_png=dpi_png,
        dpi_tiff=dpi_tiff,
    )

    apply_style()
    outdirs = ensure_dirs(cfg.step5_root)
    tables = load_tables(cfg)

    gen = prep_generator_benchmark(require_df(tables["generator_benchmark"], "generator_benchmark"))
    cand = prep_candidate_metrics(require_df(tables["candidate_metrics"], "candidate_metrics"))
    per_cond = prep_per_condition_metrics(require_df(tables["per_condition_metrics"], "per_condition_metrics"))
    model_ready = prep_model_ready(require_df(tables["model_ready_sequences"], "model_ready_sequences"))
    confusion_df = require_df(tables["confusion_matrix_norm"], "confusion_matrix_norm")
    perm_df = prep_permutation_importance(require_df(tables["permutation_importance"], "permutation_importance"))

    outputs: Dict[str, str] = {}
    outputs.update(
        {
            f"supp_a_{k}": v
            for k, v in build_supp_descriptor_figure(
                model_ready=model_ready,
                confusion_df=confusion_df,
                perm_df=perm_df,
                cfg=cfg,
                source_dir=outdirs["source"],
                out_base=outdirs["supp"] / cfg.supp_a_name,
            ).items()
        }
    )
    outputs.update(
        {
            f"supp_b_{k}": v
            for k, v in build_supp_generator_figure(
                gen=gen,
                cand=cand,
                per_cond=per_cond,
                cfg=cfg,
                source_dir=outdirs["source"],
                out_base=outdirs["supp"] / cfg.supp_b_name,
            ).items()
        }
    )

    manifest_path = cfg.step5_root / "step5_supp_reference_style_v8_manifest.txt"
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write("Step 5 supplementary figures reference-style v8 outputs\n")
        f.write(f"step5_root: {cfg.step5_root}\n")
        f.write(f"step4_model_ready_csv: {cfg.step4_model_ready_csv}\n\n")
        for key, path in tables["_paths"].items():
            f.write(f"input_{key}: {path}\n")
        for key, value in outputs.items():
            f.write(f"{key}: {value}\n")
    outputs["manifest"] = str(manifest_path)

    if verbose:
        print("Step 5 supplementary figures reference-style v2 completed.")
        for key, value in outputs.items():
            print(f"{key}: {value}")
    return outputs


if __name__ == "__main__":
    run_step5_supp_reference_style_v8()

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional

# -----------------------------------------------------------------------------
# Edit this if your project root is different
# -----------------------------------------------------------------------------
SEARCH_ROOTS = [
    Path("/home/data3/mohamed/peponco"),
    Path("/mnt/data"),
]

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def find_first(root: Path, patterns: Iterable[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

def find_all(root: Path, patterns: Iterable[str], limit: int = 20) -> list[Path]:
    found: list[Path] = []
    seen: set[Path] = set()
    for pattern in patterns:
        for p in sorted(root.rglob(pattern)):
            if p not in seen:
                seen.add(p)
                found.append(p)
    return found[:limit]

def existing_dirs(paths: Iterable[Path]) -> list[Path]:
    return [p for p in paths if p.exists()]

# -----------------------------------------------------------------------------
# Candidate patterns
# -----------------------------------------------------------------------------
STEP4_MODEL_READY_PATTERNS = [
    "step4_model_ready_sequences.csv",
    "*model_ready*sequences*.csv",
    "*step4*tables*model_ready*.csv",
]

STEP4_ARTIFACT_DIR_PATTERNS = [
    "*step4/artifacts",
    "*step4/arrays",
    "*step4/encoded*",
    "*step4/artifact*",
]

STEP5_ROOT_PATTERNS = [
    "*nature_computational2/step5",
    "*step5",
]

STEP6_ROOT_PATTERNS = [
    "*nature_computational2/step6",
    "*step6",
]

STEP5_BENCHMARK_PATTERNS = [
    "table_5_2_step5b_generator_benchmark.csv",
    "table_5_1_step5a_descriptor_benchmark.csv",
    "step5b_generated_candidates_evaluated.csv",
]

EXISTING_PEPCVAE_CODE_PATTERNS = [
    "*PepCVAE*.py",
    "*pepcvae*.py",
    "*PepCVAE*.ipynb",
    "*pepcvae*.ipynb",
    "*step6*.py",
    "*step6*.ipynb",
]

# -----------------------------------------------------------------------------
# Search
# -----------------------------------------------------------------------------
results = []

for root in existing_dirs(SEARCH_ROOTS):
    step4_model_ready = find_first(root, STEP4_MODEL_READY_PATTERNS)
    step4_artifact_dirs = find_all(root, STEP4_ARTIFACT_DIR_PATTERNS, limit=10)
    step5_root_candidates = [p for p in find_all(root, STEP5_ROOT_PATTERNS, limit=20) if p.is_dir()]
    step6_root_candidates = [p for p in find_all(root, STEP6_ROOT_PATTERNS, limit=20) if p.is_dir()]
    step5_benchmark_files = find_all(root, STEP5_BENCHMARK_PATTERNS, limit=20)
    existing_pepcvae_code = find_all(root, EXISTING_PEPCVAE_CODE_PATTERNS, limit=20)

    results.append(
        {
            "search_root": root,
            "step4_model_ready_csv": step4_model_ready,
            "step4_artifact_dirs": step4_artifact_dirs,
            "step5_root_candidates": step5_root_candidates,
            "step6_root_candidates": step6_root_candidates,
            "step5_benchmark_files": step5_benchmark_files,
            "existing_pepcvae_code": existing_pepcvae_code,
        }
    )

# -----------------------------------------------------------------------------
# Print nicely
# -----------------------------------------------------------------------------
for block in results:
    print("=" * 100)
    print(f"SEARCH ROOT: {block['search_root']}")
    print()

    print("1) step4_model_ready_csv")
    print(block["step4_model_ready_csv"] or "NOT FOUND")
    print()

    print("2) step4_artifact_dirs")
    if block["step4_artifact_dirs"]:
        for p in block["step4_artifact_dirs"]:
            print(" -", p)
    else:
        print("NOT FOUND")
    print()

    print("3) step5_root_candidates")
    if block["step5_root_candidates"]:
        for p in block["step5_root_candidates"]:
            print(" -", p)
    else:
        print("NOT FOUND")
    print()

    print("4) step6_root_candidates")
    if block["step6_root_candidates"]:
        for p in block["step6_root_candidates"]:
            print(" -", p)
    else:
        print("NOT FOUND")
    print()

    print("5) step5_benchmark_files")
    if block["step5_benchmark_files"]:
        for p in block["step5_benchmark_files"]:
            print(" -", p)
    else:
        print("NOT FOUND")
    print()

    print("6) existing_pepcvae_code")
    if block["existing_pepcvae_code"]:
        for p in block["existing_pepcvae_code"]:
            print(" -", p)
    else:
        print("NOT FOUND")
    print()

# -----------------------------------------------------------------------------
# Optional: build a copy-paste summary
# -----------------------------------------------------------------------------
best = results[0] if results else None
if best:
    step5_root = best["step5_root_candidates"][0] if best["step5_root_candidates"] else None
    step6_root = best["step6_root_candidates"][0] if best["step6_root_candidates"] else None
    artifact_dir = best["step4_artifact_dirs"][0] if best["step4_artifact_dirs"] else None
    code_file = best["existing_pepcvae_code"][0] if best["existing_pepcvae_code"] else None

    print("=" * 100)
    print("COPY-PASTE SUMMARY")
    print(f"step4_model_ready_csv = {best['step4_model_ready_csv']}")
    print(f"step4_artifacts_dir = {artifact_dir}")
    print(f"step5_root = {step5_root}")
    print(f"step6_root = {step6_root}")
    print(f"existing_pepcvae_code = {code_file}")

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional

# -----------------------------------------------------------------------------
# Edit these roots only if needed
# -----------------------------------------------------------------------------
SEARCH_ROOTS = [
    Path("/home/data3/mohamed/peponco"),
    Path("/mnt/data"),
]

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def find_first(root: Path, patterns: Iterable[str]) -> Optional[Path]:
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

def find_all(root: Path, patterns: Iterable[str], limit: int = 100) -> list[Path]:
    found: list[Path] = []
    seen: set[Path] = set()
    for pattern in patterns:
        for p in sorted(root.rglob(pattern)):
            if p not in seen:
                seen.add(p)
                found.append(p)
    return found[:limit]

def existing_dirs(paths: Iterable[Path]) -> list[Path]:
    return [p for p in paths if p.exists()]

def print_block(title: str, items: list[Path] | Path | None) -> None:
    print(title)
    if items is None:
        print("NOT FOUND\n")
        return
    if isinstance(items, Path):
        print(items, "\n")
        return
    if len(items) == 0:
        print("NOT FOUND\n")
        return
    for p in items:
        print(" -", p)
    print()

# -----------------------------------------------------------------------------
# Patterns
# -----------------------------------------------------------------------------
STEP6_ROOT_PATTERNS = [
    "*nature_computational2/step6",
    "*clean dataset*/step6",
    "*step6",
]

PEPCVAE_NOTEBOOK_PATTERNS = [
    "*pepcvae.ipynb",
    "*PepCVAE*.ipynb",
    "*pepcvae*.py",
    "*PepCVAE*.py",
]

CHECKPOINT_PATTERNS = [
    "*.ckpt",
    "*.pt",
    "*.pth",
    "*checkpoint*",
    "*checkpoints*",
    "*best_model*",
    "*best*.pt",
    "*best*.pth",
]

TRAINING_LOG_PATTERNS = [
    "*metrics.csv",
    "*history.csv",
    "*training_log*.csv",
    "*train_log*.csv",
    "*loss*.csv",
    "*wandb-summary.json",
    "*events.out.tfevents*",
    "*trainer_state.json",
]

CONFIG_PATTERNS = [
    "*config*.yaml",
    "*config*.yml",
    "*config*.json",
    "*hparams*.yaml",
    "*hparams*.yml",
    "*args.json",
]

RUN_DIR_HINT_PATTERNS = [
    "*run*",
    "*runs*",
    "*experiment*",
    "*experiments*",
    "*trial*",
    "*seed*",
]

# -----------------------------------------------------------------------------
# Search
# -----------------------------------------------------------------------------
all_results = []

for root in existing_dirs(SEARCH_ROOTS):
    step6_root_candidates = [p for p in find_all(root, STEP6_ROOT_PATTERNS, limit=50) if p.is_dir()]
    pepcvae_code = find_all(root, PEPCVAE_NOTEBOOK_PATTERNS, limit=20)
    checkpoints = find_all(root, CHECKPOINT_PATTERNS, limit=100)
    logs = find_all(root, TRAINING_LOG_PATTERNS, limit=100)
    configs = find_all(root, CONFIG_PATTERNS, limit=100)
    run_dir_hints = [p for p in find_all(root, RUN_DIR_HINT_PATTERNS, limit=100) if p.is_dir()]

    all_results.append(
        {
            "search_root": root,
            "step6_root_candidates": step6_root_candidates,
            "pepcvae_code": pepcvae_code,
            "checkpoints": checkpoints,
            "logs": logs,
            "configs": configs,
            "run_dir_hints": run_dir_hints,
        }
    )

# -----------------------------------------------------------------------------
# Show results
# -----------------------------------------------------------------------------
for block in all_results:
    print("=" * 100)
    print(f"SEARCH ROOT: {block['search_root']}\n")

    print_block("1) step6_root_candidates", block["step6_root_candidates"])
    print_block("2) pepcvae_code", block["pepcvae_code"])
    print_block("3) checkpoints", block["checkpoints"][:25])
    print_block("4) training_logs", block["logs"][:25])
    print_block("5) configs", block["configs"][:25])
    print_block("6) run_dir_hints", block["run_dir_hints"][:25])

# -----------------------------------------------------------------------------
# Infer likely run directories from discovered checkpoints/logs/configs
# -----------------------------------------------------------------------------
def parent_dirs(paths: list[Path], max_depth_up: int = 2) -> list[Path]:
    out: list[Path] = []
    seen: set[Path] = set()
    for p in paths:
        cur = p.parent
        for _ in range(max_depth_up):
            if cur not in seen and cur.exists():
                seen.add(cur)
                out.append(cur)
            cur = cur.parent
    return out

best = all_results[0] if all_results else None
if best:
    candidate_run_dirs = parent_dirs(best["checkpoints"][:20] + best["logs"][:20] + best["configs"][:20], max_depth_up=2)
    print("=" * 100)
    print("LIKELY TRAINED-RUN DIRECTORIES")
    if candidate_run_dirs:
        for p in candidate_run_dirs[:30]:
            print(" -", p)
    else:
        print("NOT FOUND")
    print()

    # Choose best guesses
    step6_root = best["step6_root_candidates"][0] if best["step6_root_candidates"] else Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step6")
    pepcvae_code = best["pepcvae_code"][0] if best["pepcvae_code"] else None
    trained_run_dir = candidate_run_dirs[0] if candidate_run_dirs else None

    print("=" * 100)
    print("COPY-PASTE SUMMARY")
    print(f"step6_root = {step6_root}")
    print(f"pepcvae_code = {pepcvae_code}")
    print(f"trained_run_dir = {trained_run_dir}")

Step 6 title

Training diagnostics, latent-space auditing, and visualization hardening

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
import json
import math
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


# =============================================================================
# CONFIG
# =============================================================================
DEFAULT_STEP4_MODEL_READY_CSV = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv"
)
DEFAULT_STEP4_ARTIFACTS_DIR = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts"
)
DEFAULT_STEP5_ROOT = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5"
)
DEFAULT_STEP5_MODELS_DIR = DEFAULT_STEP5_ROOT / "models"
DEFAULT_STEP6_ROOT = Path(
    "/home/data3/mohamed/peponco/clean dataset/nature_computational2/step6"
)
DEFAULT_PEPCVAE_CODE = Path("/home/data3/mohamed/peponco/pepcvae.ipynb")

MODEL_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "cvae_conditional": "CVAE-cond",
    "gru_conditional": "GRU-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
}

MODEL_COLORS = {
    "cvae_conditional": "#B80018",
    "gru_conditional": "#E80C12",
    "gru_unconditional": "#223E68",
    "vae_unconditional": "#8B0000",
}


@dataclass(frozen=True)
class Step6Config:
    step4_model_ready_csv: Path = DEFAULT_STEP4_MODEL_READY_CSV
    step4_artifacts_dir: Path = DEFAULT_STEP4_ARTIFACTS_DIR
    step5_root: Path = DEFAULT_STEP5_ROOT
    step5_models_dir: Path = DEFAULT_STEP5_MODELS_DIR
    step6_root: Path = DEFAULT_STEP6_ROOT
    pepcvae_code: Path = DEFAULT_PEPCVAE_CODE
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def ensure_dirs(step6_root: Path) -> Dict[str, Path]:
    dirs = {
        "root": step6_root,
        "tables": step6_root / "tables",
        "figures_main": step6_root / "figures_main",
        "figures_supplementary": step6_root / "figures_supplementary",
        "artifacts": step6_root / "artifacts",
        "arrays": step6_root / "arrays",
        "logs": step6_root / "logs",
        "reports": step6_root / "reports",
        "checkpoints": step6_root / "checkpoints",
        "source_data": step6_root / "source_data_figures",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def save_figure(fig: plt.Figure, out_base: Path, cfg: Step6Config) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs["pdf"] = str(p)
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs["png"] = str(p)
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
        outputs["tiff"] = str(p)
    plt.close(fig)
    return outputs


def export_csv(df: pd.DataFrame, out_path: Path) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return str(out_path)


def style_axis(ax: plt.Axes, grid_axis: str = "y", grid_alpha: float = 0.85) -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "both":
        ax.grid(True, color="#D9D9D9", linewidth=0.75, alpha=grid_alpha)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.12, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=15, fontweight="bold")


def coerce_scalar(x: Any) -> Optional[float]:
    try:
        if x is None:
            return None
        if isinstance(x, torch.Tensor):
            if x.numel() == 1:
                return float(x.detach().cpu().item())
            return None
        if isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
        return float(x)
    except Exception:
        return None


def flatten_keys(d: Dict[str, Any], prefix: str = "", limit: int = 300) -> List[str]:
    keys: List[str] = []
    for k, v in d.items():
        key = f"{prefix}.{k}" if prefix else str(k)
        keys.append(key)
        if len(keys) >= limit:
            return keys[:limit]
        if isinstance(v, dict):
            keys.extend(flatten_keys(v, key, max(0, limit - len(keys))))
            if len(keys) >= limit:
                return keys[:limit]
    return keys[:limit]


# =============================================================================
# DISCOVERY
# =============================================================================
RUN_RE = re.compile(r"(?P<model>.+?)_seed(?P<seed>\d+)\.(pt|pth|ckpt)$")


def parse_run_name(path: Path) -> Tuple[str, Optional[int]]:
    m = RUN_RE.match(path.name)
    if m:
        return m.group("model"), int(m.group("seed"))
    stem = path.stem
    return stem, None


def discover_checkpoints(models_dir: Path) -> List[Path]:
    if not models_dir.exists():
        return []
    hits = []
    for pattern in ("*.pt", "*.pth", "*.ckpt"):
        hits.extend(sorted(models_dir.glob(pattern)))
    return sorted(set(hits))


def discover_artifact_files(step4_artifacts_dir: Path) -> List[Path]:
    if not step4_artifacts_dir.exists():
        return []
    hits: List[Path] = []
    for pattern in ("*.npz", "*.npy", "*.json", "*.csv", "*.pt", "*.pth", "*.pkl"):
        hits.extend(sorted(step4_artifacts_dir.glob(pattern)))
    return sorted(set(hits))


# =============================================================================
# CHECKPOINT INSPECTION
# =============================================================================
HISTORY_KEY_CANDIDATES = [
    "history", "train_history", "training_history", "metrics_history", "log_history",
    "loss_history", "trainer_history",
]
STATE_DICT_KEYS = ["state_dict", "model_state_dict", "model", "network", "module"]


def safe_torch_load(path: Path) -> Any:
    return torch.load(path, map_location="cpu")


def extract_history_from_obj(obj: Any) -> Optional[pd.DataFrame]:
    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, dict):
        for key in HISTORY_KEY_CANDIDATES:
            if key in obj:
                value = obj[key]
                if isinstance(value, pd.DataFrame):
                    return value.copy()
                if isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
                    return pd.DataFrame(value)
                if isinstance(value, dict):
                    try:
                        df = pd.DataFrame(value)
                        if not df.empty:
                            return df
                    except Exception:
                        pass
        # flat dict of lists
        try:
            df = pd.DataFrame(obj)
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if len(numeric_cols) > 0 and len(df) > 1:
                return df
        except Exception:
            pass
    return None


def extract_state_dict_keys(obj: Any) -> List[str]:
    if isinstance(obj, dict):
        for key in STATE_DICT_KEYS:
            if key in obj and isinstance(obj[key], dict):
                return list(obj[key].keys())[:50]
        if all(isinstance(v, torch.Tensor) for v in obj.values()):
            return list(obj.keys())[:50]
    return []


def estimate_active_units_from_state_keys(keys: List[str]) -> Optional[int]:
    # Weak heuristic: infer latent dim from layers that commonly encode mu/logvar.
    candidates = [k for k in keys if any(tok in k.lower() for tok in ["mu", "logvar", "latent", "z_mean", "z_logvar"])]
    if not candidates:
        return None
    return len(candidates)


def inspect_checkpoint(path: Path) -> Dict[str, Any]:
    model_name, seed = parse_run_name(path)
    record: Dict[str, Any] = {
        "checkpoint_path": str(path),
        "checkpoint_name": path.name,
        "model_name": model_name,
        "seed": seed,
        "exists": path.exists(),
        "file_size_mb": round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else None,
    }

    try:
        obj = safe_torch_load(path)
        record["load_ok"] = True
        record["top_type"] = type(obj).__name__

        if isinstance(obj, dict):
            keys = flatten_keys(obj, limit=200)
            record["top_keys_preview"] = " | ".join(keys[:25])
            state_keys = extract_state_dict_keys(obj)
            record["state_key_preview"] = " | ".join(state_keys[:15]) if state_keys else None
            record["epoch"] = coerce_scalar(obj.get("epoch")) if "epoch" in obj else None
            record["global_step"] = coerce_scalar(obj.get("global_step")) if "global_step" in obj else None
            record["best_val_loss"] = coerce_scalar(obj.get("best_val_loss")) if "best_val_loss" in obj else None
            record["val_loss"] = coerce_scalar(obj.get("val_loss")) if "val_loss" in obj else None
            record["train_loss"] = coerce_scalar(obj.get("train_loss")) if "train_loss" in obj else None
            record["active_units_estimate"] = estimate_active_units_from_state_keys(state_keys)
        else:
            record["top_keys_preview"] = None
            record["state_key_preview"] = None
            record["epoch"] = None
            record["global_step"] = None
            record["best_val_loss"] = None
            record["val_loss"] = None
            record["train_loss"] = None
            record["active_units_estimate"] = None

        history_df = extract_history_from_obj(obj)
        record["history_available"] = history_df is not None
        if history_df is not None:
            record["history_rows"] = len(history_df)
            record["history_cols"] = " | ".join([str(c) for c in history_df.columns[:20]])
        else:
            record["history_rows"] = None
            record["history_cols"] = None

    except Exception as e:
        record["load_ok"] = False
        record["top_type"] = None
        record["top_keys_preview"] = None
        record["state_key_preview"] = None
        record["epoch"] = None
        record["global_step"] = None
        record["best_val_loss"] = None
        record["val_loss"] = None
        record["train_loss"] = None
        record["active_units_estimate"] = None
        record["history_available"] = False
        record["history_rows"] = None
        record["history_cols"] = None
        record["load_error"] = repr(e)

    return record


# =============================================================================
# STEP 4 / STEP 5 CONTEXT
# =============================================================================
def load_step4_context(step4_model_ready_csv: Path, step4_artifacts_dir: Path) -> Dict[str, Any]:
    ctx: Dict[str, Any] = {}

    if step4_model_ready_csv.exists():
        df = pd.read_csv(step4_model_ready_csv)
        ctx["step4_model_ready_rows"] = len(df)
        ctx["step4_model_ready_cols"] = len(df.columns)
        ctx["step4_model_ready_columns"] = " | ".join(df.columns.astype(str).tolist()[:40])

        # useful split/condition summaries if present
        split_col = None
        for cand in ["split", "subset", "partition", "group"]:
            if cand in df.columns:
                split_col = cand
                break
        cond_col = None
        for cand in ["primary_condition_id", "condition", "condition_id", "label", "primary_condition"]:
            if cand in df.columns:
                cond_col = cand
                break

        if split_col is not None:
            split_counts = df[split_col].astype(str).value_counts().to_dict()
            ctx["step4_split_counts_json"] = json.dumps(split_counts, ensure_ascii=False)
        if cond_col is not None:
            ctx["step4_n_conditions"] = int(df[cond_col].astype(str).nunique())

    artifact_files = discover_artifact_files(step4_artifacts_dir)
    ctx["step4_artifacts_dir_exists"] = step4_artifacts_dir.exists()
    ctx["step4_artifact_file_count"] = len(artifact_files)
    ctx["step4_artifact_preview"] = " | ".join([p.name for p in artifact_files[:30]]) if artifact_files else None
    return ctx


def load_step5_benchmark_context(step5_root: Path) -> Dict[str, Any]:
    ctx: Dict[str, Any] = {}
    table = step5_root / "tables_main" / "table_5_2_step5b_generator_benchmark.csv"
    if table.exists():
        df = pd.read_csv(table)
        ctx["step5_generator_benchmark_rows"] = len(df)
        ctx["step5_generator_benchmark_columns"] = " | ".join(df.columns.astype(str).tolist()[:40])
        if "generator" in df.columns and "surrogate_condition_hit_rate" in df.columns:
            preview = df[["generator", "surrogate_condition_hit_rate"]].copy()
            ctx["step5_fidelity_preview_json"] = preview.to_json(orient="records")
    return ctx


# =============================================================================
# FIGURES
# =============================================================================
def build_seed_inventory_figure(inv_df: pd.DataFrame, out_base: Path, cfg: Step6Config) -> Dict[str, str]:
    if inv_df.empty:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.text(0.5, 0.5, "No checkpoints discovered", ha="center", va="center")
        ax.axis("off")
        return save_figure(fig, out_base, cfg)

    counts = (
        inv_df.groupby("model_name")["seed"]
        .count()
        .reindex([m for m in MODEL_ORDER if m in set(inv_df["model_name"].astype(str))])
        .dropna()
    )
    if counts.empty:
        counts = inv_df.groupby("model_name")["checkpoint_name"].count()

    labels = [MODEL_LABELS.get(str(i), str(i)) for i in counts.index]
    colors = [MODEL_COLORS.get(str(i), "#666666") for i in counts.index]
    vals = counts.to_numpy(float)

    fig, ax = plt.subplots(figsize=(8.5, 4.0))
    x = np.arange(len(vals))
    ax.bar(x, vals, color=colors, width=0.72, edgecolor="none")
    for xi, v in zip(x, vals):
        ax.text(xi, v + 0.05, f"{int(v)}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=0)
    ax.set_ylabel("Checkpoint count")
    ax.set_title("Discovered trained runs")
    style_axis(ax, "y")
    return save_figure(fig, out_base, cfg)


def build_best_val_loss_figure(inv_df: pd.DataFrame, out_base: Path, cfg: Step6Config) -> Dict[str, str]:
    plot_df = inv_df.dropna(subset=["best_val_loss"]).copy()
    if plot_df.empty:
        fig, ax = plt.subplots(figsize=(8.5, 3.5))
        ax.text(
            0.5,
            0.5,
            "Best validation loss was not stored in the discovered checkpoints.\nThis diagnostic requires checkpoints or logs with explicit validation metrics.",
            ha="center",
            va="center",
            fontsize=9,
        )
        ax.axis("off")
        return save_figure(fig, out_base, cfg)

    fig, ax = plt.subplots(figsize=(9.0, 4.2))
    models = [m for m in MODEL_ORDER if m in set(plot_df["model_name"].astype(str))]
    data = []
    for m in models:
        vals = plot_df.loc[plot_df["model_name"] == m, "best_val_loss"].astype(float).to_numpy()
        if len(vals):
            data.append((m, vals))

    xs = np.arange(len(data))
    means = np.array([np.mean(vals) for _, vals in data])
    sds = np.array([np.std(vals, ddof=0) for _, vals in data])
    colors = [MODEL_COLORS.get(m, "#666666") for m, _ in data]

    ax.bar(xs, means, color=colors, width=0.72, edgecolor="none")
    ax.errorbar(xs, means, yerr=sds, fmt="none", ecolor="#444444", elinewidth=1.0, capsize=3.0)
    for xi, m, vals in zip(xs, [m for m, _ in data], [v for _, v in data]):
        jitter = np.linspace(-0.10, 0.10, len(vals)) if len(vals) > 1 else np.array([0.0])
        ax.scatter(np.full(len(vals), xi) + jitter, vals, s=20, color="white", edgecolors="#333333", linewidths=0.6, zorder=3)

    ax.set_xticks(xs)
    ax.set_xticklabels([MODEL_LABELS.get(m, m) for m, _ in data], rotation=0)
    ax.set_ylabel("Best validation loss")
    ax.set_title("Validation-loss summary across seeds")
    style_axis(ax, "y")
    return save_figure(fig, out_base, cfg)


def build_active_units_figure(inv_df: pd.DataFrame, out_base: Path, cfg: Step6Config) -> Dict[str, str]:
    plot_df = inv_df.dropna(subset=["active_units_estimate"]).copy()
    if plot_df.empty:
        fig, ax = plt.subplots(figsize=(8.5, 3.5))
        ax.text(
            0.5,
            0.5,
            "Checkpoint contents do not expose latent-dimension statistics directly.\nActive-unit auditing will require saved posterior means/log-variances or model-specific encode hooks.",
            ha="center",
            va="center",
            fontsize=9,
        )
        ax.axis("off")
        return save_figure(fig, out_base, cfg)

    fig, ax = plt.subplots(figsize=(8.5, 4.0))
    counts = (
        plot_df.groupby("model_name")["active_units_estimate"]
        .mean()
        .reindex([m for m in MODEL_ORDER if m in set(plot_df["model_name"].astype(str))])
        .dropna()
    )
    x = np.arange(len(counts))
    vals = counts.to_numpy(float)
    colors = [MODEL_COLORS.get(str(i), "#666666") for i in counts.index]
    ax.bar(x, vals, color=colors, width=0.72, edgecolor="none")
    for xi, v in zip(x, vals):
        ax.text(xi, v + 0.05, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS.get(str(i), str(i)) for i in counts.index], rotation=0)
    ax.set_ylabel("Estimated active units")
    ax.set_title("Latent-activity proxy from checkpoint contents")
    style_axis(ax, "y")
    return save_figure(fig, out_base, cfg)


# =============================================================================
# REPORTING
# =============================================================================
def write_report(report_path: Path, lines: List[str]) -> str:
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return str(report_path)


# =============================================================================
# MAIN RUNNER
# =============================================================================
def run_step6_diagnostics_v1(
    step4_model_ready_csv: str | Path = DEFAULT_STEP4_MODEL_READY_CSV,
    step4_artifacts_dir: str | Path = DEFAULT_STEP4_ARTIFACTS_DIR,
    step5_root: str | Path = DEFAULT_STEP5_ROOT,
    step5_models_dir: str | Path = DEFAULT_STEP5_MODELS_DIR,
    step6_root: str | Path = DEFAULT_STEP6_ROOT,
    pepcvae_code: str | Path = DEFAULT_PEPCVAE_CODE,
    export_png: bool = True,
    export_pdf: bool = True,
    export_tiff: bool = True,
    dpi_png: int = 300,
    dpi_tiff: int = 600,
    verbose: bool = True,
) -> Dict[str, str]:
    cfg = Step6Config(
        step4_model_ready_csv=Path(step4_model_ready_csv),
        step4_artifacts_dir=Path(step4_artifacts_dir),
        step5_root=Path(step5_root),
        step5_models_dir=Path(step5_models_dir),
        step6_root=Path(step6_root),
        pepcvae_code=Path(pepcvae_code),
        export_png=export_png,
        export_pdf=export_pdf,
        export_tiff=export_tiff,
        dpi_png=dpi_png,
        dpi_tiff=dpi_tiff,
    )

    outdirs = ensure_dirs(cfg.step6_root)

    outputs: Dict[str, str] = {}

    # 1) Context inventory
    step4_ctx = load_step4_context(cfg.step4_model_ready_csv, cfg.step4_artifacts_dir)
    step5_ctx = load_step5_benchmark_context(cfg.step5_root)

    # 2) Checkpoint discovery and inspection
    checkpoints = discover_checkpoints(cfg.step5_models_dir)
    inv_records = [inspect_checkpoint(p) for p in checkpoints]
    inv_df = pd.DataFrame(inv_records)

    if inv_df.empty:
        warnings.warn(f"No checkpoints discovered in {cfg.step5_models_dir}")

    # 3) Export tables
    outputs["checkpoint_inventory_csv"] = export_csv(inv_df, outdirs["tables"] / "step6_checkpoint_inventory.csv")

    ctx_rows = []
    for k, v in {**step4_ctx, **step5_ctx}.items():
        ctx_rows.append({"key": k, "value": v})
    ctx_df = pd.DataFrame(ctx_rows)
    outputs["context_summary_csv"] = export_csv(ctx_df, outdirs["tables"] / "step6_context_summary.csv")

    # Seed summary table
    if not inv_df.empty:
        seed_summary = (
            inv_df.groupby("model_name", dropna=False)
            .agg(
                n_checkpoints=("checkpoint_name", "count"),
                n_load_ok=("load_ok", lambda x: int(np.sum(np.asarray(x) == True))),
                mean_best_val_loss=("best_val_loss", "mean"),
                sd_best_val_loss=("best_val_loss", "std"),
                mean_epoch=("epoch", "mean"),
                mean_active_units_estimate=("active_units_estimate", "mean"),
            )
            .reset_index()
        )
    else:
        seed_summary = pd.DataFrame(
            columns=[
                "model_name",
                "n_checkpoints",
                "n_load_ok",
                "mean_best_val_loss",
                "sd_best_val_loss",
                "mean_epoch",
                "mean_active_units_estimate",
            ]
        )
    outputs["seed_training_summary_csv"] = export_csv(seed_summary, outdirs["tables"] / "step6_seed_training_summary.csv")

    # 4) Figures
    outputs.update({f"fig_seed_inventory_{k}": v for k, v in build_seed_inventory_figure(inv_df, outdirs["figures_main"] / "fig_6a_step6_discovered_runs", cfg).items()})
    outputs.update({f"fig_best_val_loss_{k}": v for k, v in build_best_val_loss_figure(inv_df, outdirs["figures_main"] / "fig_6b_step6_validation_loss", cfg).items()})
    outputs.update({f"fig_active_units_{k}": v for k, v in build_active_units_figure(inv_df, outdirs["figures_supplementary"] / "fig_s6_1_step6_active_units_proxy", cfg).items()})

    # 5) Manifest and narrative report
    manifest = {
        "step4_model_ready_csv": str(cfg.step4_model_ready_csv),
        "step4_artifacts_dir": str(cfg.step4_artifacts_dir),
        "step5_root": str(cfg.step5_root),
        "step5_models_dir": str(cfg.step5_models_dir),
        "step6_root": str(cfg.step6_root),
        "pepcvae_code": str(cfg.pepcvae_code),
        "n_checkpoints_discovered": int(len(checkpoints)),
        "checkpoint_preview": [str(p) for p in checkpoints[:20]],
        "outputs": outputs,
    }
    manifest_path = outdirs["root"] / "step6_diagnostics_v1_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    outputs["manifest_json"] = str(manifest_path)

    report_lines = [
        "Step 6 diagnostics v1 summary",
        "",
        f"Step 4 model-ready CSV: {cfg.step4_model_ready_csv}",
        f"Step 4 artifacts dir: {cfg.step4_artifacts_dir}",
        f"Step 5 root: {cfg.step5_root}",
        f"Step 5 models dir: {cfg.step5_models_dir}",
        f"Step 6 root: {cfg.step6_root}",
        f"PepCVAE code reference: {cfg.pepcvae_code}",
        "",
        f"Discovered checkpoints: {len(checkpoints)}",
        "",
        "Key observations",
        "- This implementation treats the Step 5 model directory as the trained-run source for Step 6 diagnostics.",
        "- Validation-loss summaries are computed only when checkpoints store explicit validation metrics.",
        "- Latent-activity summaries are computed only when checkpoint contents expose latent-related keys.",
        "- If training-curve CSV logs were not saved separately, full per-epoch stability plots cannot be reconstructed from checkpoints alone.",
        "",
        "Recommended next data additions (optional but helpful)",
        "- per-epoch training logs (CSV or JSON) for each seed",
        "- saved posterior mean / log-variance arrays on validation or test data",
        "- saved reconstruction metrics per seed",
        "",
        "Primary exported tables",
        f"- {outputs['checkpoint_inventory_csv']}",
        f"- {outputs['context_summary_csv']}",
        f"- {outputs['seed_training_summary_csv']}",
        "",
        "Primary exported figures",
        f"- {outputs.get('fig_seed_inventory_png', outputs.get('fig_seed_inventory_pdf', 'N/A'))}",
        f"- {outputs.get('fig_best_val_loss_png', outputs.get('fig_best_val_loss_pdf', 'N/A'))}",
        f"- {outputs.get('fig_active_units_png', outputs.get('fig_active_units_pdf', 'N/A'))}",
    ]
    outputs["summary_report_txt"] = write_report(outdirs["reports"] / "step6_diagnostics_v1_summary.txt", report_lines)

    if verbose:
        print("Step 6 diagnostics v1 completed.")
        print(f"Discovered checkpoints: {len(checkpoints)}")
        for key, value in outputs.items():
            print(f"{key}: {value}")

    return outputs


if __name__ == "__main__":
    run_step6_diagnostics_v1()

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
from collections import Counter

# -----------------------------------------------------------------------------
# Edit only if needed
# -----------------------------------------------------------------------------
PEPCVAE_NOTEBOOK = Path("/home/data3/mohamed/peponco/pepcvae.ipynb")
STEP4_ARTIFACTS_DIR = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/artifacts")
STEP4_MODEL_READY_CSV = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step4/tables/step4_model_ready_sequences.csv")
STEP5_MODELS_DIR = Path("/home/data3/mohamed/peponco/clean dataset/nature_computational2/step5/models")

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def read_notebook_cells(nb_path: Path) -> list[dict]:
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    return nb.get("cells", [])

def cell_source(cell: dict) -> str:
    src = cell.get("source", [])
    if isinstance(src, list):
        return "".join(src)
    return str(src)

def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def grep_lines(text: str, patterns: list[str], max_hits: int = 40) -> list[str]:
    hits = []
    lines = text.splitlines()
    for i, line in enumerate(lines, start=1):
        lower = line.lower()
        if any(p.lower() in lower for p in patterns):
            hits.append(f"[L{i}] {line}")
        if len(hits) >= max_hits:
            break
    return hits

def find_python_defs(text: str) -> dict[str, list[str]]:
    out = {
        "classes": re.findall(r"^\s*class\s+([A-Za-z_][A-Za-z0-9_]*)", text, flags=re.M),
        "functions": re.findall(r"^\s*def\s+([A-Za-z_][A-Za-z0-9_]*)", text, flags=re.M),
    }
    return out

def guess_hparams(text: str) -> dict[str, list[str]]:
    keys = [
        "latent_dim", "z_dim", "embedding_dim", "embed_dim", "hidden_dim",
        "hidden_size", "num_layers", "dropout", "batch_size", "lr",
        "learning_rate", "weight_decay", "epochs", "max_epochs",
        "kl_weight", "beta", "anneal", "teacher_forcing", "seed",
        "temperature", "patience"
    ]
    results = {}
    lines = text.splitlines()
    for key in keys:
        matched = []
        for i, line in enumerate(lines, start=1):
            if key.lower() in line.lower():
                matched.append(f"[L{i}] {line}")
            if len(matched) >= 8:
                break
        if matched:
            results[key] = matched
    return results

def list_artifacts(dir_path: Path, limit: int = 100) -> list[Path]:
    if not dir_path.exists():
        return []
    out = []
    for p in sorted(dir_path.rglob("*")):
        if p.is_file():
            out.append(p)
        if len(out) >= limit:
            break
    return out

# -----------------------------------------------------------------------------
# Load notebook
# -----------------------------------------------------------------------------
cells = read_notebook_cells(PEPCVAE_NOTEBOOK)
code_cells = [c for c in cells if c.get("cell_type") == "code"]
md_cells = [c for c in cells if c.get("cell_type") == "markdown"]

full_code = "\n\n".join(cell_source(c) for c in code_cells)
full_md = "\n\n".join(cell_source(c) for c in md_cells)

# -----------------------------------------------------------------------------
# 1) High-level notebook summary
# -----------------------------------------------------------------------------
print_header("1) NOTEBOOK SUMMARY")
print(f"Notebook path: {PEPCVAE_NOTEBOOK}")
print(f"Total cells: {len(cells)}")
print(f"Code cells: {len(code_cells)}")
print(f"Markdown cells: {len(md_cells)}")

defs = find_python_defs(full_code)
print("\nClasses found:")
for x in defs["classes"][:50]:
    print(" -", x)

print("\nFunctions found:")
for x in defs["functions"][:80]:
    print(" -", x)

# -----------------------------------------------------------------------------
# 2) Architecture clues
# -----------------------------------------------------------------------------
print_header("2) ARCHITECTURE CLUES")
arch_patterns = [
    "encoder", "decoder", "vae", "cvae", "latent", "reparameter", "mu", "logvar",
    "embedding", "gru", "lstm", "rnn", "teacher forcing", "condition", "conditioning"
]
for line in grep_lines(full_code, arch_patterns, max_hits=80):
    print(line)

# -----------------------------------------------------------------------------
# 3) Hyperparameter clues
# -----------------------------------------------------------------------------
print_header("3) HYPERPARAMETER CLUES")
hparams = guess_hparams(full_code)
for key, lines in hparams.items():
    print(f"\n[{key}]")
    for line in lines:
        print(line)

# -----------------------------------------------------------------------------
# 4) Training pipeline clues
# -----------------------------------------------------------------------------
print_header("4) TRAINING PIPELINE CLUES")
train_patterns = [
    "optimizer", "adam", "scheduler", "train_loader", "val_loader", "test_loader",
    "criterion", "loss", "kl", "anneal", "beta", "fit(", "epoch", "early stopping",
    "patience", "save", "checkpoint", "best model", "torch.save"
]
for line in grep_lines(full_code, train_patterns, max_hits=120):
    print(line)

# -----------------------------------------------------------------------------
# 5) Data pipeline clues
# -----------------------------------------------------------------------------
print_header("5) DATA PIPELINE CLUES")
data_patterns = [
    "token", "vocab", "stoi", "itos", "pad", "mask", "collate", "dataset",
    "dataloader", "condition_id", "primary_condition_id", "sequence", "max_len",
    "encode", "decode"
]
for line in grep_lines(full_code, data_patterns, max_hits=120):
    print(line)

# -----------------------------------------------------------------------------
# 6) Evaluation / generation clues
# -----------------------------------------------------------------------------
print_header("6) EVALUATION / GENERATION CLUES")
eval_patterns = [
    "sample", "generate", "reconstruct", "novelty", "validity", "uniqueness",
    "similarity", "nearest", "nn_similarity", "condition-hit", "hit rate",
    "surrogate", "bootstrap", "confidence interval", "ci"
]
for line in grep_lines(full_code, eval_patterns, max_hits=120):
    print(line)

# -----------------------------------------------------------------------------
# 7) Markdown / narrative clues
# -----------------------------------------------------------------------------
print_header("7) MARKDOWN / NOTEBOOK NARRATIVE CLUES")
for line in grep_lines(full_md, ["step 6", "vae", "cvae", "latent", "training", "generation", "evaluation"], max_hits=80):
    print(line)

# -----------------------------------------------------------------------------
# 8) Step 4 artifacts preview
# -----------------------------------------------------------------------------
print_header("8) STEP 4 ARTIFACTS PREVIEW")
artifact_files = list_artifacts(STEP4_ARTIFACTS_DIR, limit=120)
if artifact_files:
    for p in artifact_files:
        print(" -", p)
else:
    print("No files found.")

# -----------------------------------------------------------------------------
# 9) Step 4 CSV column preview
# -----------------------------------------------------------------------------
print_header("9) STEP 4 MODEL-READY CSV PREVIEW")
if STEP4_MODEL_READY_CSV.exists():
    import pandas as pd
    df = pd.read_csv(STEP4_MODEL_READY_CSV, nrows=5)
    print("Columns:")
    for c in df.columns:
        print(" -", c)
    print("\nHead:")
    print(df.head())
else:
    print("CSV not found.")

# -----------------------------------------------------------------------------
# 10) Step 5 checkpoint names
# -----------------------------------------------------------------------------
print_header("10) STEP 5 CHECKPOINTS")
if STEP5_MODELS_DIR.exists():
    ckpts = sorted([p for p in STEP5_MODELS_DIR.iterdir() if p.is_file()])
    for p in ckpts:
        print(" -", p.name)
else:
    print("Step 5 models dir not found.")

# -----------------------------------------------------------------------------
# 11) Compact copy-paste summary
# -----------------------------------------------------------------------------
print_header("11) COPY-PASTE SUMMARY FOR CHATGPT")

top_classes = defs["classes"][:15]
top_functions = defs["functions"][:25]

summary = {
    "notebook_path": str(PEPCVAE_NOTEBOOK),
    "n_code_cells": len(code_cells),
    "n_markdown_cells": len(md_cells),
    "top_classes": top_classes,
    "top_functions": top_functions,
    "artifact_files_preview": [str(p) for p in artifact_files[:20]],
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

main step 6 training: 

In [ ]:
from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE — Step 6 v4
# Full executable manuscript-facing conditional generative pipeline
# =============================================================================

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import gc
import io
import re
import json
import math
import time
import uuid
import copy
import random
import hashlib
import warnings
import platform
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import PCA


# =============================================================================
# Project paths
# =============================================================================
PROJECT_ROOT = Path("/home/data3/Moe/nature_computational2").resolve()

DEFAULT_STEP4_MODEL_READY_CSV = PROJECT_ROOT / "step4" / "tables" / "step4_model_ready_sequences.csv"
DEFAULT_STEP4_ARTIFACTS_DIR = PROJECT_ROOT / "step4" / "artifacts"
DEFAULT_STEP5_ROOT = PROJECT_ROOT / "step5"
DEFAULT_STEP6_ROOT = PROJECT_ROOT / "step6"


# =============================================================================
# Constants
# =============================================================================
AA = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA)

KD = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8, "G": -0.4, "H": -3.2,
    "I": 4.5, "K": -3.9, "L": 3.8, "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5,
    "R": -4.5, "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

PALETTE = {
    "full_cvae": "#B80018",
    "conditional_gru_lm": "#8B0000",
    "retrieval": "#B4C5E4",
    "no_condition": "#E80C12",
    "small_latent": "#165C63",
    "no_warmup": "#223E68",
    "gray": "#7F7F7F",
    "grid": "#D9D9D9",
    "black": "#333333",
    "train": "#223E68",
    "val": "#B80018",
    "test": "#165C63",
}

MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "retrieval",
    "no_condition",
    "small_latent",
    "no_warmup",
]

MODEL_LABELS = {
    "full_cvae": "Full CVAE",
    "conditional_gru_lm": "Cond. GRU LM",
    "retrieval": "Retrieval",
    "no_condition": "No conditioning",
    "small_latent": "Small latent",
    "no_warmup": "No warmup",
}

MODEL_COLORS = {k: PALETTE[k] for k in MODEL_LABELS if k in PALETTE}


# =============================================================================
# Configuration
# =============================================================================
@dataclass(frozen=True)
class FigureExportConfig:
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600


@dataclass(frozen=True)
class GeneratorTrainConfig:
    max_seq_len: int = 64
    batch_size: int = 64
    num_workers: int = 0

    emb_dim: int = 96
    hidden_dim: int = 192
    latent_dim: int = 32
    num_layers: int = 1
    dropout: float = 0.10

    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 35
    clip_grad_norm: float = 1.0
    early_stopping_patience: int = 7

    beta_max: float = 1.0
    beta_warmup_fraction: float = 0.50

    min_length: int = 5
    max_decode_len: int = 60

    generation_per_condition: int = 50
    generation_per_condition_support_scaled: bool = True
    min_generation_per_condition: int = 25
    max_generation_per_condition_cap: int = 100

    sample_temperature: float = 1.0
    decoding_temperatures: Tuple[float, ...] = (0.8, 1.0, 1.2)
    top_k: Optional[int] = None
    nucleus_p: Optional[float] = None

    reconstruction_sample_limit: int = 400


@dataclass(frozen=True)
class EvaluatorTrainConfig:
    emb_dim: int = 64
    hidden_dim: int = 128
    num_layers: int = 1
    dropout: float = 0.10
    batch_size: int = 64
    num_workers: int = 0
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 20
    early_stopping_patience: int = 5
    clip_grad_norm: float = 1.0


@dataclass(frozen=True)
class AblationConfig:
    run_full_cvae: bool = True
    run_no_condition: bool = True
    run_small_latent: bool = True
    run_no_warmup: bool = True
    run_conditional_gru_lm: bool = True
    run_retrieval_baseline: bool = True
    small_latent_dim: int = 16


@dataclass(frozen=True)
class QAConfig:
    low_complexity_repeat_threshold: int = 5
    min_unique_amino_acids: int = 4
    max_single_aa_fraction: float = 0.60
    early_eos_length_threshold: int = 3
    collapse_duplicate_rate_threshold: float = 0.50
    kl_collapse_threshold: float = 0.02
    unseen_condition_exclusion_warn_threshold: float = 0.05


@dataclass(frozen=True)
class RuntimeConfig:
    skip_completed_runs: bool = True
    no_figure_mode: bool = False
    no_generation_mode: bool = False
    run_only_model_tags: Tuple[str, ...] = tuple()
    run_only_seeds: Tuple[int, ...] = tuple()
    verbose: bool = True


@dataclass(frozen=True)
class Step6Config:
    step4_model_ready_csv: str = str(DEFAULT_STEP4_MODEL_READY_CSV)
    step4_artifacts_dir: str = str(DEFAULT_STEP4_ARTIFACTS_DIR)
    step5_root: str = str(DEFAULT_STEP5_ROOT)
    step6_root: str = str(DEFAULT_STEP6_ROOT)

    sequence_col: str = "sequence"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    repeated_seeds: Tuple[int, ...] = (42, 52, 62)
    bootstrap_iterations: int = 400

    active_unit_variance_threshold: float = 1e-2
    active_unit_thresholds: Tuple[float, ...] = (1e-3, 1e-2, 5e-2)

    jaccard_k: int = 3
    nn_similarity_sample_limit: Optional[int] = 2000
    pairwise_diversity_pair_limit: int = 5000

    generated_eval_temperature_default: float = 1.0

    train_cfg: GeneratorTrainConfig = field(default_factory=GeneratorTrainConfig)
    evaluator_cfg: EvaluatorTrainConfig = field(default_factory=EvaluatorTrainConfig)
    ablation_cfg: AblationConfig = field(default_factory=AblationConfig)
    qa_cfg: QAConfig = field(default_factory=QAConfig)
    runtime_cfg: RuntimeConfig = field(default_factory=RuntimeConfig)
    fig_cfg: FigureExportConfig = field(default_factory=FigureExportConfig)


# =============================================================================
# Utilities
# =============================================================================
def print_header(msg: str):
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)


def seed_all(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def device_for_training() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def clean_sequence(seq: str) -> str:
    return "".join([a for a in str(seq).strip().upper() if a in AA_SET])


def contains_noncanonical(seq: str) -> bool:
    s = str(seq).strip().upper()
    return any(ch not in AA_SET for ch in s)


def is_valid_peptide(seq: str, min_len: int = 5, max_len: int = 60) -> bool:
    s = clean_sequence(seq)
    return min_len <= len(s) <= max_len and all(a in AA_SET for a in s)


def style_axis(ax, grid_axis: str = "y", grid_alpha: float = 0.85):
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PALETTE["grid"], linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PALETTE["grid"], linewidth=0.75, alpha=grid_alpha)
    else:
        ax.grid(True, color=PALETTE["grid"], linewidth=0.75, alpha=grid_alpha)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_label(ax, label: str):
    ax.text(-0.12, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=15, fontweight="bold")


def ensure_dirs(root: Path) -> Dict[str, Path]:
    d = {k: root / k for k in [
        "tables",
        "figures_main",
        "figures_supplementary",
        "artifacts",
        "arrays",
        "logs",
        "reports",
        "models",
        "source_data_figures",
        "qa",
    ]}
    d["root"] = root
    for p in d.values():
        p.mkdir(parents=True, exist_ok=True)
    return d


def save_figure(fig, out_base: Path, fc: FigureExportConfig):
    out = {}
    if fc.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        out["pdf"] = str(p)
    if fc.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=fc.dpi_png)
        out["png"] = str(p)
    if fc.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=fc.dpi_tiff, format="tiff")
        out["tiff"] = str(p)
    plt.close(fig)
    return out


def export_csv(df: pd.DataFrame, out_path: Path) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return str(out_path)


def export_json(data: Any, out_path: Path) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=_json_default)
    return str(out_path)


def _json_default(obj: Any):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Unsupported type for JSON serialization: {type(obj)}")


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns:
            return c
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"Could not resolve any of columns: {candidates}")
    return None


def config_hash(cfg: Step6Config) -> str:
    return hashlib.sha256(json.dumps(asdict(cfg), sort_keys=True, default=_json_default).encode()).hexdigest()[:16]


def environment_snapshot() -> Dict[str, Any]:
    return {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
    }


def should_run_variant_seed(cfg: Step6Config, model_tag: str, seed: int) -> bool:
    if cfg.runtime_cfg.run_only_model_tags and model_tag not in cfg.runtime_cfg.run_only_model_tags:
        return False
    if cfg.runtime_cfg.run_only_seeds and seed not in cfg.runtime_cfg.run_only_seeds:
        return False
    return True


def safe_float(x: Any) -> float:
    try:
        return float(x)
    except Exception:
        return float("nan")


def mean_or_nan(vals: Sequence[float]) -> float:
    vals = [float(v) for v in vals if pd.notnull(v)]
    return float(np.mean(vals)) if vals else float("nan")


def std_or_nan(vals: Sequence[float]) -> float:
    vals = [float(v) for v in vals if pd.notnull(v)]
    return float(np.std(vals, ddof=1)) if len(vals) > 1 else float("nan")


def bootstrap_mean_ci(vals: Sequence[float], iterations: int = 400, seed: int = 42, alpha: float = 0.05) -> Dict[str, float]:
    vals = np.array([float(v) for v in vals if pd.notnull(v)], dtype=float)
    if vals.size == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0}
    if vals.size == 1:
        v = float(vals[0])
        return {"mean": v, "ci_low": v, "ci_high": v, "n": 1}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(iterations):
        sample = rng.choice(vals, size=vals.size, replace=True)
        boots.append(np.mean(sample))
    boots = np.array(boots, dtype=float)
    return {
        "mean": float(np.mean(vals)),
        "ci_low": float(np.quantile(boots, alpha / 2.0)),
        "ci_high": float(np.quantile(boots, 1.0 - alpha / 2.0)),
        "n": int(vals.size),
    }


# =============================================================================
# Tokenizer / data
# =============================================================================
class PeptideTokenizer:
    def __init__(self, vocab_json: Optional[Path] = None, special_tokens_json: Optional[Path] = None):
        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"

        if special_tokens_json and special_tokens_json.exists():
            try:
                d = json.loads(special_tokens_json.read_text())
                self.pad_token = d.get("pad_token", self.pad_token)
                self.bos_token = d.get("bos_token", self.bos_token)
                self.eos_token = d.get("eos_token", self.eos_token)
            except Exception:
                pass

        if vocab_json and vocab_json.exists():
            try:
                d = json.loads(vocab_json.read_text())
                if isinstance(d, dict):
                    vocab = [k for k, _ in sorted(d.items(), key=lambda kv: kv[1])]
                else:
                    vocab = list(d)
                special = [self.pad_token, self.bos_token, self.eos_token]
                self.alphabet = special + [x for x in vocab if x not in special]
            except Exception:
                self.alphabet = [self.pad_token, self.bos_token, self.eos_token] + AA
        else:
            self.alphabet = [self.pad_token, self.bos_token, self.eos_token] + AA

        seen = set()
        clean = []
        for t in self.alphabet:
            if t not in seen:
                seen.add(t)
                clean.append(t)
        self.alphabet = clean
        self.stoi = {t: i for i, t in enumerate(self.alphabet)}
        self.itos = {i: t for t, i in self.stoi.items()}

    @property
    def pad_id(self):
        return self.stoi[self.pad_token]

    @property
    def bos_id(self):
        return self.stoi[self.bos_token]

    @property
    def eos_id(self):
        return self.stoi[self.eos_token]

    @property
    def vocab_size(self):
        return len(self.alphabet)

    def encode_unpadded(self, seq: str, max_len: int) -> List[int]:
        toks = [self.bos_id]
        for s in clean_sequence(seq)[: max_len - 2]:
            if s in self.stoi:
                toks.append(self.stoi[s])
        toks.append(self.eos_id)
        return toks[:max_len]

    def decode(self, ids: Sequence[int]) -> str:
        chars = []
        for i in ids:
            tok = self.itos[int(i)]
            if tok == self.eos_token:
                break
            if tok in {self.pad_token, self.bos_token}:
                continue
            chars.append(tok)
        return "".join(chars)


class SequenceDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_len, conditions=None):
        self.seqs = [clean_sequence(s) for s in sequences]
        self.tok = tokenizer
        self.max_len = max_len
        self.conditions = None if conditions is None else list(conditions)

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = self.tok.encode_unpadded(self.seqs[idx], self.max_len)
        x = ids[:-1]
        y = ids[1:]
        d = {
            "input_ids": torch.tensor(x, dtype=torch.long),
            "target_ids": torch.tensor(y, dtype=torch.long),
            "length": len(x),
            "sequence": self.seqs[idx],
        }
        if self.conditions is not None:
            d["condition_id"] = int(self.conditions[idx])
        return d


class ClassifierDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_len, labels):
        self.seqs = [clean_sequence(s) for s in sequences]
        self.tok = tokenizer
        self.max_len = max_len
        self.labels = list(labels)

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = self.tok.encode_unpadded(self.seqs[idx], self.max_len)
        return {
            "input_ids": torch.tensor(ids[:-1], dtype=torch.long),
            "length": len(ids[:-1]),
            "sequence": self.seqs[idx],
            "label": int(self.labels[idx]),
        }


def collate_sequence_batch(batch, pad_id: int):
    lengths = torch.tensor([b["length"] for b in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    bs = len(batch)

    input_ids = torch.full((bs, max_len), pad_id, dtype=torch.long)
    seqs = []
    for i, b in enumerate(batch):
        n = b["length"]
        input_ids[i, :n] = b["input_ids"]
        seqs.append(b["sequence"])

    out = {
        "input_ids": input_ids,
        "lengths": lengths,
        "sequence": seqs,
    }

    if "target_ids" in batch[0]:
        target_ids = torch.full((bs, max_len), pad_id, dtype=torch.long)
        for i, b in enumerate(batch):
            n = b["length"]
            target_ids[i, :n] = b["target_ids"]
        out["target_ids"] = target_ids

    if "condition_id" in batch[0]:
        out["condition_id"] = torch.tensor([int(b["condition_id"]) for b in batch], dtype=torch.long)

    if "label" in batch[0]:
        out["label"] = torch.tensor([int(b["label"]) for b in batch], dtype=torch.long)

    return out


def make_dataloader(ds, batch_size, shuffle, num_workers, seed, pad_id):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        generator=g,
        collate_fn=lambda b: collate_sequence_batch(b, pad_id),
    )


# =============================================================================
# Models
# =============================================================================
class ConditionalSequenceVAE(nn.Module):
    def __init__(self, vocab_size, cond_vocab_size, emb_dim, hidden_dim, latent_dim, num_layers, dropout=0.0, use_condition=True):
        super().__init__()
        self.use_condition = use_condition
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb_enc = nn.Embedding(cond_vocab_size, emb_dim) if use_condition else None
        self.cond_emb_dec = nn.Embedding(cond_vocab_size, hidden_dim) if use_condition else None
        self.encoder = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)
        self.z_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.latent_dim = latent_dim
        self.num_layers = num_layers

    def encode(self, input_ids, lengths, condition_id=None):
        emb = self.emb(input_ids)
        if self.use_condition and condition_id is not None:
            emb = emb + self.cond_emb_enc(condition_id).unsqueeze(1)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.encoder(packed)
        h = h[-1]
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        return z, mu, logvar

    def _decoder_hidden(self, z, condition_id=None):
        h0 = self.z_to_hidden(z)
        if self.use_condition and condition_id is not None:
            h0 = h0 + self.cond_emb_dec(condition_id)
        return h0.unsqueeze(0).repeat(self.num_layers, 1, 1)

    def decode_teacher_forced(self, input_ids, z, condition_id=None):
        dec, _ = self.decoder(self.emb(input_ids), self._decoder_hidden(z, condition_id))
        return self.out(dec)

    def forward(self, input_ids, lengths, condition_id=None):
        z, mu, logvar = self.encode(input_ids, lengths, condition_id)
        logits = self.decode_teacher_forced(input_ids, z, condition_id)
        return logits, mu, logvar

    def decode_from_z(self, z, input_ids, condition_id=None):
        return self.decode_teacher_forced(input_ids, z, condition_id)

    def greedy_decode(self, z, bos_id, eos_id, pad_id, max_decode_len, condition_id=None):
        device = z.device
        bs = z.size(0)
        h = self._decoder_hidden(z, condition_id)
        cur = torch.full((bs, 1), bos_id, dtype=torch.long, device=device)
        generated = []
        finished = torch.zeros(bs, dtype=torch.bool, device=device)
        for _ in range(max_decode_len):
            dec, h = self.decoder(self.emb(cur[:, -1:]), h)
            nxt = self.out(dec[:, -1, :]).argmax(dim=-1)
            generated.append(nxt)
            cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            finished = finished | nxt.eq(eos_id)
            if finished.all():
                break
        if not generated:
            return torch.full((bs, 1), eos_id, dtype=torch.long, device=device)
        return _mask_after_eos(torch.stack(generated, dim=1), eos_id, pad_id)

    def sample_sequences(self, n, bos_id, eos_id, pad_id, max_decode_len, temperature=1.0, top_k=None, nucleus_p=None, condition_id=None, device=None):
        device = device or next(self.parameters()).device
        z = torch.randn(n, self.latent_dim, device=device)
        h = self._decoder_hidden(z, condition_id)
        cur = torch.full((n, 1), bos_id, dtype=torch.long, device=device)
        generated = []
        finished = torch.zeros(n, dtype=torch.bool, device=device)
        for _ in range(max_decode_len):
            dec, h = self.decoder(self.emb(cur[:, -1:]), h)
            logits = apply_sampling_constraints(self.out(dec[:, -1, :]) / max(temperature, 1e-6), top_k, nucleus_p)
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).squeeze(1)
            generated.append(nxt)
            cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            finished = finished | nxt.eq(eos_id)
            if finished.all():
                break
        if not generated:
            return torch.full((n, 1), eos_id, dtype=torch.long, device=device)
        return _mask_after_eos(torch.stack(generated, dim=1), eos_id, pad_id)


class ConditionalGRULM(nn.Module):
    def __init__(self, vocab_size, cond_vocab_size, emb_dim, hidden_dim, num_layers, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb = nn.Embedding(cond_vocab_size, hidden_dim)
        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.num_layers = num_layers

    def forward(self, input_ids, condition_id):
        h0 = self.cond_emb(condition_id).unsqueeze(0).repeat(self.num_layers, 1, 1)
        out, _ = self.rnn(self.emb(input_ids), h0)
        return self.out(out)

    def sample_sequences(self, n, bos_id, eos_id, pad_id, max_decode_len, condition_id, temperature=1.0, top_k=None, nucleus_p=None, device=None):
        device = device or next(self.parameters()).device
        h = self.cond_emb(condition_id.to(device)).unsqueeze(0).repeat(self.num_layers, 1, 1)
        cur = torch.full((n, 1), bos_id, dtype=torch.long, device=device)
        generated = []
        finished = torch.zeros(n, dtype=torch.bool, device=device)
        for _ in range(max_decode_len):
            dec, h = self.rnn(self.emb(cur[:, -1:]), h)
            logits = apply_sampling_constraints(self.out(dec[:, -1, :]) / max(temperature, 1e-6), top_k, nucleus_p)
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).squeeze(1)
            generated.append(nxt)
            cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            finished = finished | nxt.eq(eos_id)
            if finished.all():
                break
        if not generated:
            return torch.full((n, 1), eos_id, dtype=torch.long, device=device)
        return _mask_after_eos(torch.stack(generated, dim=1), eos_id, pad_id)


class SequenceConditionClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim, hidden_dim, num_layers, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.encoder = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, n_classes)

    def forward(self, input_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(input_ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.encoder(packed)
        return self.out(h[-1])


# =============================================================================
# Sequence helpers / metrics
# =============================================================================
def _mask_after_eos(out, eos_id, pad_id):
    out = out.clone()
    for i in range(out.size(0)):
        pos = (out[i] == eos_id).nonzero(as_tuple=False)
        if len(pos) > 0:
            first = int(pos[0].item())
            if first + 1 < out.size(1):
                out[i, first + 1:] = pad_id
    return out


def apply_sampling_constraints(logits, top_k=None, nucleus_p=None):
    logits = logits.clone()

    if top_k is not None and top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        thresh = v[:, -1].unsqueeze(-1)
        logits[logits < thresh] = -float("inf")

    if nucleus_p is not None and 0 < nucleus_p < 1:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumulative = torch.cumsum(probs, dim=-1)
        remove = cumulative > nucleus_p
        remove[:, 0] = False
        sorted_logits[remove] = -float("inf")
        new_logits = torch.full_like(logits, -float("inf"))
        new_logits.scatter_(1, sorted_idx, sorted_logits)
        logits = new_logits

    return logits


def cross_entropy_ignore_pad(logits, target_ids, pad_id):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1), ignore_index=pad_id)


def token_accuracy_ignore_pad(logits, target_ids, pad_id):
    pred = logits.argmax(dim=-1)
    mask = target_ids.ne(pad_id)
    denom = int(mask.sum().item())
    if denom == 0:
        return float("nan")
    return int((pred.eq(target_ids) & mask).sum().item()) / denom


def classification_accuracy(logits, labels):
    return float((logits.argmax(dim=-1).eq(labels)).float().mean().item())


def normalized_edit_distance(a, b):
    if a == b:
        return 0.0
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 0.0
    dp = np.zeros((la + 1, lb + 1), dtype=np.int32)
    dp[:, 0] = np.arange(la + 1)
    dp[0, :] = np.arange(lb + 1)
    for i in range(1, la + 1):
        ca = a[i - 1]
        for j in range(1, lb + 1):
            cb = b[j - 1]
            cost = 0 if ca == cb else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )
    return float(dp[la, lb]) / max(la, lb, 1)


def simple_charge_proxy(seq):
    seq = clean_sequence(seq)
    return sum(1 for a in seq if a in {"K", "R", "H"}) - sum(1 for a in seq if a in {"D", "E"})


def mean_hydropathy(seq):
    seq = clean_sequence(seq)
    return float(np.mean([KD[a] for a in seq])) if seq else float("nan")


def sequence_entropy(seq):
    seq = clean_sequence(seq)
    if not seq:
        return float("nan")
    c = pd.Series(list(seq)).value_counts(normalize=True).to_numpy(float)
    return float(-(c * np.log2(c)).sum())


def low_complexity_flags(seq, qa: QAConfig):
    seq = clean_sequence(seq)
    if not seq:
        return {
            "homopolymer_flag": 1,
            "low_unique_aa_flag": 1,
            "dominant_aa_flag": 1,
            "low_complexity_any_flag": 1,
        }
    max_run = 1
    cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 1
    h = int(max_run >= qa.low_complexity_repeat_threshold)
    u = int(len(set(seq)) < qa.min_unique_amino_acids)
    d = int(max(seq.count(a) for a in set(seq)) / len(seq) > qa.max_single_aa_fraction)
    return {
        "homopolymer_flag": h,
        "low_unique_aa_flag": u,
        "dominant_aa_flag": d,
        "low_complexity_any_flag": int(h or u or d),
    }


def kmer_set(seq, k=3):
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}


def jaccard_similarity(a, b, k=3):
    sa = kmer_set(a, k)
    sb = kmer_set(b, k)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def nearest_neighbor_similarity(seq, refs, k=3, sample_limit=None, seed=42):
    if not refs:
        return float("nan")
    if sample_limit is not None and len(refs) > sample_limit:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(refs), size=sample_limit, replace=False)
        refs_eff = [refs[i] for i in idx]
    else:
        refs_eff = refs
    return max(jaccard_similarity(seq, r, k=k) for r in refs_eff)


def pairwise_diversity_summary(seqs, k=3, pair_limit=5000, seed=42):
    seqs = [clean_sequence(s) for s in seqs if clean_sequence(s)]
    n = len(seqs)
    if n <= 1:
        return {"mean_pairwise_jaccard": np.nan, "mean_pairwise_diversity": np.nan}

    rng = np.random.default_rng(seed)
    pairs = []
    max_pairs = n * (n - 1) // 2

    if max_pairs <= pair_limit:
        for i in range(n):
            for j in range(i + 1, n):
                pairs.append((i, j))
    else:
        seen = set()
        while len(pairs) < pair_limit:
            i = int(rng.integers(0, n))
            j = int(rng.integers(0, n))
            if i >= j or (i, j) in seen:
                continue
            seen.add((i, j))
            pairs.append((i, j))

    sims = [jaccard_similarity(seqs[i], seqs[j], k=k) for i, j in pairs]
    return {
        "mean_pairwise_jaccard": float(np.mean(sims)),
        "mean_pairwise_diversity": float(1.0 - np.mean(sims)),
    }


def compute_kl(mu, logvar):
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    return kl.sum(dim=1).mean()


def beta_for_epoch(epoch_idx: int, epochs: int, beta_max: float, warmup_fraction: float, variant_tag: str) -> float:
    if variant_tag == "no_warmup":
        return beta_max
    warmup_epochs = max(1, int(round(epochs * warmup_fraction)))
    if epoch_idx >= warmup_epochs:
        return beta_max
    return beta_max * float(epoch_idx + 1) / float(warmup_epochs)


def compute_sequence_descriptors(seqs: Sequence[str]) -> pd.DataFrame:
    rows = []
    for s in seqs:
        s = clean_sequence(s)
        rows.append({
            "sequence": s,
            "length": len(s),
            "charge_proxy": simple_charge_proxy(s),
            "mean_hydropathy": mean_hydropathy(s),
            "entropy": sequence_entropy(s),
        })
    return pd.DataFrame(rows)


def descriptor_summary(df_desc: pd.DataFrame, prefix: str = "") -> Dict[str, float]:
    out = {}
    for c in ["length", "charge_proxy", "mean_hydropathy", "entropy"]:
        vals = df_desc[c].dropna().to_numpy(dtype=float) if c in df_desc.columns else np.array([], dtype=float)
        out[f"{prefix}{c}_mean"] = float(np.mean(vals)) if vals.size else np.nan
        out[f"{prefix}{c}_std"] = float(np.std(vals, ddof=1)) if vals.size > 1 else np.nan
    return out


def descriptor_shift_score(ref_df: pd.DataFrame, gen_df: pd.DataFrame) -> Dict[str, float]:
    out = {}
    for c in ["length", "charge_proxy", "mean_hydropathy", "entropy"]:
        if c not in ref_df.columns or c not in gen_df.columns:
            out[f"{c}_mean_abs_diff"] = np.nan
            continue
        a = ref_df[c].dropna().to_numpy(dtype=float)
        b = gen_df[c].dropna().to_numpy(dtype=float)
        if a.size == 0 or b.size == 0:
            out[f"{c}_mean_abs_diff"] = np.nan
        else:
            out[f"{c}_mean_abs_diff"] = float(abs(np.mean(a) - np.mean(b)))
    diffs = [v for v in out.values() if pd.notnull(v)]
    out["descriptor_mean_abs_diff_mean"] = float(np.mean(diffs)) if diffs else np.nan
    return out


# =============================================================================
# Data loading / validation
# =============================================================================
def load_tokenizer_from_artifacts(step4_artifacts_dir: Path) -> PeptideTokenizer:
    vocab_candidates = [
        step4_artifacts_dir / "vocab.json",
        step4_artifacts_dir / "tokenizer_vocab.json",
        step4_artifacts_dir / "vocabulary.json",
    ]
    special_candidates = [
        step4_artifacts_dir / "special_tokens.json",
        step4_artifacts_dir / "tokenizer_special_tokens.json",
    ]

    vocab_json = next((p for p in vocab_candidates if p.exists()), None)
    special_json = next((p for p in special_candidates if p.exists()), None)
    return PeptideTokenizer(vocab_json=vocab_json, special_tokens_json=special_json)


def infer_condition_column(df: pd.DataFrame, preferred: str) -> str:
    candidates = [
        preferred,
        "primary_condition_id",
        "condition_id",
        "condition",
        "target_condition",
        "conditioning_id",
        "conditioning_label",
    ]
    return resolve_column(df, candidates, required=True)


def normalize_split_labels(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    s = s.replace({
        "validation": "val",
        "valid": "val",
        "dev": "val",
        "training": "train",
        "testing": "test",
    })
    return s


def load_step4_inputs(cfg: Step6Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    csv_path = Path(cfg.step4_model_ready_csv)
    artifacts_dir = Path(cfg.step4_artifacts_dir)

    if not csv_path.exists():
        raise FileNotFoundError(f"Step 4 model-ready CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    seq_col = resolve_column(df, [cfg.sequence_col, "sequence", "peptide_sequence", "clean_sequence"])
    split_col = resolve_column(df, [cfg.split_col, "assigned_split", "split", "data_split"])
    target_col = infer_condition_column(df, cfg.target_col)

    work = df.copy()
    work[seq_col] = work[seq_col].astype(str)
    work["sequence_raw"] = work[seq_col].astype(str)
    work["sequence"] = work[seq_col].map(clean_sequence)
    work["split"] = normalize_split_labels(work[split_col])
    work["condition_raw"] = work[target_col].astype(str)

    work["contains_noncanonical_raw"] = work["sequence_raw"].map(contains_noncanonical).astype(int)
    work["valid_after_clean"] = work["sequence"].map(lambda x: is_valid_peptide(x, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)).astype(int)

    removed_df = work.loc[work["valid_after_clean"] == 0].copy()
    work = work.loc[work["valid_after_clean"] == 1].copy()

    allowed_splits = {cfg.train_label, cfg.val_label, cfg.test_label}
    unknown_splits = sorted(set(work["split"].unique()) - allowed_splits)
    if unknown_splits:
        raise ValueError(f"Unexpected split labels found: {unknown_splits}")

    if work["sequence"].duplicated().any():
        dup_count = int(work["sequence"].duplicated().sum())
        warnings.warn(
            f"Detected {dup_count} duplicate cleaned sequences in Step 4 table. "
            "Step 2 should already have resolved uniqueness at split unit level. "
            "This runner keeps rows as provided but records the duplication."
        )

    split_counts = work["split"].value_counts().to_dict()
    for req in [cfg.train_label, cfg.val_label, cfg.test_label]:
        if split_counts.get(req, 0) == 0:
            raise ValueError(f"Empty required split detected: {req}")

    train_conditions = set(work.loc[work["split"] == cfg.train_label, "condition_raw"].astype(str))
    work["condition_seen_in_train"] = work["condition_raw"].isin(train_conditions).astype(int)

    unseen_eval = work.loc[
        (work["split"].isin([cfg.val_label, cfg.test_label])) & (work["condition_seen_in_train"] == 0)
    ].copy()

    unseen_rate = float(len(unseen_eval) / max(1, len(work.loc[work["split"].isin([cfg.val_label, cfg.test_label])])))
    if unseen_rate > cfg.qa_cfg.unseen_condition_exclusion_warn_threshold:
        warnings.warn(
            f"Unseen validation/test conditions exceed warning threshold: {unseen_rate:.3f}. "
            "These rows will be excluded from condition-dependent evaluation."
        )

    kept = work.loc[
        (work["split"] == cfg.train_label) |
        ((work["split"].isin([cfg.val_label, cfg.test_label])) & (work["condition_seen_in_train"] == 1))
    ].copy()

    condition_values = sorted(set(kept.loc[kept["split"] == cfg.train_label, "condition_raw"].astype(str)))
    condition_to_id = {c: i for i, c in enumerate(condition_values)}
    kept["condition_id"] = kept["condition_raw"].map(condition_to_id).astype(int)

    train_df = kept.loc[kept["split"] == cfg.train_label].copy().reset_index(drop=True)
    val_df = kept.loc[kept["split"] == cfg.val_label].copy().reset_index(drop=True)
    test_df = kept.loc[kept["split"] == cfg.test_label].copy().reset_index(drop=True)

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError("At least one split became empty after train-condition support filtering.")

    tokenizer = load_tokenizer_from_artifacts(artifacts_dir)

    split_summary = pd.DataFrame([
        {
            "split": "removed_invalid",
            "n": len(removed_df),
            "n_unique_sequences": removed_df["sequence"].nunique() if len(removed_df) else 0,
            "n_conditions": removed_df["condition_raw"].nunique() if len(removed_df) else 0,
        },
        {
            "split": "unseen_eval_excluded",
            "n": len(unseen_eval),
            "n_unique_sequences": unseen_eval["sequence"].nunique() if len(unseen_eval) else 0,
            "n_conditions": unseen_eval["condition_raw"].nunique() if len(unseen_eval) else 0,
        },
        {
            "split": cfg.train_label,
            "n": len(train_df),
            "n_unique_sequences": train_df["sequence"].nunique(),
            "n_conditions": train_df["condition_raw"].nunique(),
        },
        {
            "split": cfg.val_label,
            "n": len(val_df),
            "n_unique_sequences": val_df["sequence"].nunique(),
            "n_conditions": val_df["condition_raw"].nunique(),
        },
        {
            "split": cfg.test_label,
            "n": len(test_df),
            "n_unique_sequences": test_df["sequence"].nunique(),
            "n_conditions": test_df["condition_raw"].nunique(),
        },
    ])

    condition_support = (
        kept.groupby(["split", "condition_raw"], as_index=False)
        .size()
        .rename(columns={"size": "n_sequences"})
        .sort_values(["split", "n_sequences", "condition_raw"], ascending=[True, False, True])
        .reset_index(drop=True)
    )
    condition_support["condition_id"] = condition_support["condition_raw"].map(condition_to_id)

    export_csv(split_summary, dirs["tables"] / "step6_input_split_summary.csv")
    export_csv(condition_support, dirs["tables"] / "step6_condition_support_summary.csv")
    export_csv(removed_df[["sequence_raw", "sequence", "split", "condition_raw", "contains_noncanonical_raw", "valid_after_clean"]], dirs["qa"] / "step6_removed_invalid_sequences.csv")
    export_csv(unseen_eval[["sequence", "split", "condition_raw", "condition_seen_in_train"]], dirs["qa"] / "step6_unseen_eval_conditions_excluded.csv")

    info = {
        "seq_col": seq_col,
        "split_col": split_col,
        "target_col": target_col,
        "condition_to_id": condition_to_id,
        "id_to_condition": {v: k for k, v in condition_to_id.items()},
        "tokenizer": tokenizer,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "kept_df": kept,
        "removed_df": removed_df,
        "unseen_eval_df": unseen_eval,
        "split_summary": split_summary,
        "condition_support": condition_support,
        "step4_csv_path": str(csv_path),
        "step4_artifacts_dir": str(artifacts_dir),
    }
    return info


def build_sequence_dataloaders(df_train, df_val, df_test, tokenizer, cfg: Step6Config, seed: int, use_condition: bool) -> Dict[str, DataLoader]:
    train_ds = SequenceDataset(
        df_train["sequence"].tolist(),
        tokenizer,
        cfg.train_cfg.max_seq_len,
        conditions=df_train["condition_id"].tolist() if use_condition else None,
    )
    val_ds = SequenceDataset(
        df_val["sequence"].tolist(),
        tokenizer,
        cfg.train_cfg.max_seq_len,
        conditions=df_val["condition_id"].tolist() if use_condition else None,
    )
    test_ds = SequenceDataset(
        df_test["sequence"].tolist(),
        tokenizer,
        cfg.train_cfg.max_seq_len,
        conditions=df_test["condition_id"].tolist() if use_condition else None,
    )

    return {
        "train": make_dataloader(train_ds, cfg.train_cfg.batch_size, True, cfg.train_cfg.num_workers, seed, tokenizer.pad_id),
        "val": make_dataloader(val_ds, cfg.train_cfg.batch_size, False, cfg.train_cfg.num_workers, seed, tokenizer.pad_id),
        "test": make_dataloader(test_ds, cfg.train_cfg.batch_size, False, cfg.train_cfg.num_workers, seed, tokenizer.pad_id),
    }


def build_classifier_dataloaders(df_train, df_val, df_test, tokenizer, cfg: Step6Config, seed: int) -> Dict[str, DataLoader]:
    train_ds = ClassifierDataset(df_train["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_train["condition_id"].tolist())
    val_ds = ClassifierDataset(df_val["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_val["condition_id"].tolist())
    test_ds = ClassifierDataset(df_test["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_test["condition_id"].tolist())

    return {
        "train": make_dataloader(train_ds, cfg.evaluator_cfg.batch_size, True, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id),
        "val": make_dataloader(val_ds, cfg.evaluator_cfg.batch_size, False, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id),
        "test": make_dataloader(test_ds, cfg.evaluator_cfg.batch_size, False, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id),
    }


# =============================================================================
# Evaluator training
# =============================================================================
def train_condition_classifier(df_train, df_val, df_test, tokenizer, n_classes: int, cfg: Step6Config, seed: int, device: torch.device, dirs: Dict[str, Path]) -> Dict[str, Any]:
    seed_all(seed)

    loaders = build_classifier_dataloaders(df_train, df_val, df_test, tokenizer, cfg, seed)
    model = SequenceConditionClassifier(
        vocab_size=tokenizer.vocab_size,
        n_classes=n_classes,
        emb_dim=cfg.evaluator_cfg.emb_dim,
        hidden_dim=cfg.evaluator_cfg.hidden_dim,
        num_layers=cfg.evaluator_cfg.num_layers,
        dropout=cfg.evaluator_cfg.dropout,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.evaluator_cfg.lr, weight_decay=cfg.evaluator_cfg.weight_decay)

    best_state = None
    best_metric = float("inf")
    patience = 0
    history = []

    for epoch in range(cfg.evaluator_cfg.epochs):
        model.train()
        train_loss_sum = 0.0
        train_n = 0
        train_acc_sum = 0.0

        for batch in loaders["train"]:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.evaluator_cfg.clip_grad_norm)
            optimizer.step()

            bs = labels.size(0)
            train_loss_sum += float(loss.item()) * bs
            train_acc_sum += classification_accuracy(logits, labels) * bs
            train_n += bs

        train_loss = train_loss_sum / max(1, train_n)
        train_acc = train_acc_sum / max(1, train_n)

        val_metrics = evaluate_condition_classifier(model, loaders["val"], device)
        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        if val_metrics["loss"] < best_metric:
            best_metric = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= cfg.evaluator_cfg.early_stopping_patience:
                break

    if best_state is None:
        raise RuntimeError("Condition evaluator failed to produce a valid checkpoint.")

    model.load_state_dict(best_state)

    val_final = evaluate_condition_classifier(model, loaders["val"], device)
    test_final = evaluate_condition_classifier(model, loaders["test"], device)

    ckpt_path = dirs["models"] / f"step6_condition_evaluator_seed{seed}.pt"
    torch.save(
        {
            "seed": seed,
            "state_dict": model.state_dict(),
            "config": asdict(cfg.evaluator_cfg),
            "n_classes": n_classes,
            "tokenizer_vocab_size": tokenizer.vocab_size,
        },
        ckpt_path,
    )

    hist_df = pd.DataFrame(history)
    export_csv(hist_df, dirs["logs"] / f"step6_condition_evaluator_history_seed{seed}.csv")

    return {
        "model": model,
        "checkpoint_path": str(ckpt_path),
        "history_df": hist_df,
        "val_metrics": val_final,
        "test_metrics": test_final,
    }


@torch.no_grad()
def evaluate_condition_classifier(model, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    loss_sum = 0.0
    acc_sum = 0.0
    n = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, lengths)
        loss = F.cross_entropy(logits, labels)

        bs = labels.size(0)
        loss_sum += float(loss.item()) * bs
        acc_sum += classification_accuracy(logits, labels) * bs
        n += bs

    return {
        "loss": loss_sum / max(1, n),
        "acc": acc_sum / max(1, n),
        "n": n,
    }


@torch.no_grad()
def predict_conditions_for_sequences(model, sequences: Sequence[str], tokenizer: PeptideTokenizer, cfg: Step6Config, seed: int, device: torch.device) -> np.ndarray:
    ds = ClassifierDataset(sequences, tokenizer, cfg.train_cfg.max_seq_len, labels=[0] * len(sequences))
    loader = make_dataloader(ds, cfg.evaluator_cfg.batch_size, False, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id)
    preds = []
    model.eval()
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        logits = model(input_ids, lengths)
        preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


# =============================================================================
# Training / evaluation loops
# =============================================================================
def make_model_for_variant(model_tag: str, tokenizer: PeptideTokenizer, n_conditions: int, cfg: Step6Config):
    if model_tag in {"full_cvae", "no_condition", "small_latent", "no_warmup"}:
        latent_dim = cfg.train_cfg.latent_dim
        use_condition = True
        if model_tag == "small_latent":
            latent_dim = cfg.ablation_cfg.small_latent_dim
        if model_tag == "no_condition":
            use_condition = False
        model = ConditionalSequenceVAE(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=cfg.train_cfg.emb_dim,
            hidden_dim=cfg.train_cfg.hidden_dim,
            latent_dim=latent_dim,
            num_layers=cfg.train_cfg.num_layers,
            dropout=cfg.train_cfg.dropout,
            use_condition=use_condition,
        )
        family = "cvae"
    elif model_tag == "conditional_gru_lm":
        model = ConditionalGRULM(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=cfg.train_cfg.emb_dim,
            hidden_dim=cfg.train_cfg.hidden_dim,
            num_layers=cfg.train_cfg.num_layers,
            dropout=cfg.train_cfg.dropout,
        )
        family = "gru"
    else:
        raise ValueError(f"Unsupported model tag for trainable model: {model_tag}")
    return model, family


def get_run_done_flag(dirs: Dict[str, Path], model_tag: str, seed: int) -> Path:
    return dirs["artifacts"] / f"step6_done_{model_tag}_seed{seed}.json"


def variant_enabled(cfg: Step6Config, model_tag: str) -> bool:
    ab = cfg.ablation_cfg
    mapping = {
        "full_cvae": ab.run_full_cvae,
        "no_condition": ab.run_no_condition,
        "small_latent": ab.run_small_latent,
        "no_warmup": ab.run_no_warmup,
        "conditional_gru_lm": ab.run_conditional_gru_lm,
        "retrieval": ab.run_retrieval_baseline,
    }
    return bool(mapping.get(model_tag, False))


def train_one_variant(
    model_tag: str,
    seed: int,
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    df_test: pd.DataFrame,
    tokenizer: PeptideTokenizer,
    n_conditions: int,
    evaluator_model: SequenceConditionClassifier,
    cfg: Step6Config,
    dirs: Dict[str, Path],
    device: torch.device,
) -> Dict[str, Any]:
    if model_tag == "retrieval":
        return run_retrieval_baseline(
            model_tag=model_tag,
            seed=seed,
            df_train=df_train,
            df_val=df_val,
            df_test=df_test,
            tokenizer=tokenizer,
            evaluator_model=evaluator_model,
            cfg=cfg,
            dirs=dirs,
            device=device,
        )

    seed_all(seed)

    model, family = make_model_for_variant(model_tag, tokenizer, n_conditions, cfg)
    model = model.to(device)

    use_condition = (model_tag != "no_condition")
    loaders = build_sequence_dataloaders(df_train, df_val, df_test, tokenizer, cfg, seed, use_condition=use_condition)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train_cfg.lr, weight_decay=cfg.train_cfg.weight_decay)

    best_state = None
    best_metric = float("inf")
    best_epoch = None
    patience = 0
    history = []

    for epoch in range(cfg.train_cfg.epochs):
        beta = beta_for_epoch(epoch, cfg.train_cfg.epochs, cfg.train_cfg.beta_max, cfg.train_cfg.beta_warmup_fraction, model_tag)

        train_metrics = train_epoch_generator(
            model=model,
            loader=loaders["train"],
            optimizer=optimizer,
            tokenizer=tokenizer,
            beta=beta,
            device=device,
            family=family,
            use_condition=use_condition,
            clip_grad_norm=cfg.train_cfg.clip_grad_norm,
        )

        val_metrics = evaluate_generator_loader(
            model=model,
            loader=loaders["val"],
            tokenizer=tokenizer,
            beta=beta,
            device=device,
            family=family,
            use_condition=use_condition,
        )

        row = {
            "model_tag": model_tag,
            "seed": seed,
            "epoch": epoch + 1,
            "beta": beta,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        val_select = val_metrics["elbo"] if family == "cvae" else val_metrics["loss"]
        if val_select < best_metric:
            best_metric = val_select
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            patience = 0
        else:
            patience += 1
            if patience >= cfg.train_cfg.early_stopping_patience:
                break

    if best_state is None:
        raise RuntimeError(f"Training failed to produce a valid checkpoint for {model_tag} seed={seed}")

    model.load_state_dict(best_state)

    ckpt_path = dirs["models"] / f"step6_{model_tag}_seed{seed}.pt"
    torch.save(
        {
            "seed": seed,
            "model_tag": model_tag,
            "state_dict": model.state_dict(),
            "family": family,
            "config": asdict(cfg.train_cfg),
            "n_conditions": n_conditions,
            "vocab_size": tokenizer.vocab_size,
            "best_epoch": best_epoch,
        },
        ckpt_path,
    )

    history_df = pd.DataFrame(history)
    export_csv(history_df, dirs["logs"] / f"step6_training_history_{model_tag}_seed{seed}.csv")

    val_eval = evaluate_full_variant(
        model=model,
        family=family,
        model_tag=model_tag,
        split_name="val",
        df_split=df_val,
        loader=loaders["val"],
        tokenizer=tokenizer,
        evaluator_model=evaluator_model,
        cfg=cfg,
        seed=seed,
        device=device,
        use_condition=use_condition,
    )
    test_eval = evaluate_full_variant(
        model=model,
        family=family,
        model_tag=model_tag,
        split_name="test",
        df_split=df_test,
        loader=loaders["test"],
        tokenizer=tokenizer,
        evaluator_model=evaluator_model,
        cfg=cfg,
        seed=seed,
        device=device,
        use_condition=use_condition,
    )

    gen_eval = None
    temp_eval_df = None
    generated_df = None

    if not cfg.runtime_cfg.no_generation_mode:
        gen_eval, temp_eval_df, generated_df = evaluate_generation_for_variant(
            model=model,
            family=family,
            model_tag=model_tag,
            df_train=df_train,
            df_ref=df_test,
            tokenizer=tokenizer,
            evaluator_model=evaluator_model,
            cfg=cfg,
            seed=seed,
            device=device,
            use_condition=use_condition,
        )

    out = {
        "model_tag": model_tag,
        "seed": seed,
        "family": family,
        "best_epoch": best_epoch,
        "checkpoint_path": str(ckpt_path),
        "history_df": history_df,
        "val_eval": val_eval,
        "test_eval": test_eval,
        "generation_eval": gen_eval,
        "temperature_eval_df": temp_eval_df,
        "generated_df": generated_df,
    }

    export_json(
        {
            "model_tag": model_tag,
            "seed": seed,
            "family": family,
            "best_epoch": best_epoch,
            "checkpoint_path": str(ckpt_path),
            "val_eval": val_eval,
            "test_eval": test_eval,
            "generation_eval": gen_eval,
        },
        get_run_done_flag(dirs, model_tag, seed),
    )
    return out


def train_epoch_generator(model, loader, optimizer, tokenizer, beta, device, family, use_condition, clip_grad_norm):
    model.train()
    loss_sum = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    acc_sum = 0.0
    n = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)
        lengths = batch["lengths"].to(device)
        condition_id = batch["condition_id"].to(device) if use_condition else None

        optimizer.zero_grad(set_to_none=True)

        if family == "cvae":
            logits, mu, logvar = model(input_ids, lengths, condition_id=condition_id)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)
            kl = compute_kl(mu, logvar)
            loss = recon + beta * kl
            kl_value = float(kl.item())
        else:
            logits = model(input_ids, condition_id=condition_id)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)
            kl = torch.tensor(0.0, device=device)
            loss = recon
            kl_value = 0.0

        if not torch.isfinite(loss):
            raise FloatingPointError("Encountered non-finite training loss.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)
        optimizer.step()

        bs = input_ids.size(0)
        acc = token_accuracy_ignore_pad(logits, target_ids, tokenizer.pad_id)
        loss_sum += float(loss.item()) * bs
        recon_sum += float(recon.item()) * bs
        kl_sum += kl_value * bs
        acc_sum += float(acc) * bs if pd.notnull(acc) else 0.0
        n += bs

    return {
        "loss": loss_sum / max(1, n),
        "recon_loss": recon_sum / max(1, n),
        "kl_loss": kl_sum / max(1, n),
        "elbo": loss_sum / max(1, n),
        "token_acc": acc_sum / max(1, n),
        "n": n,
    }


@torch.no_grad()
def evaluate_generator_loader(model, loader, tokenizer, beta, device, family, use_condition):
    model.eval()
    loss_sum = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    acc_sum = 0.0
    n = 0

    all_mu = []
    all_logvar = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)
        lengths = batch["lengths"].to(device)
        condition_id = batch["condition_id"].to(device) if use_condition else None

        if family == "cvae":
            logits, mu, logvar = model(input_ids, lengths, condition_id=condition_id)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)
            kl = compute_kl(mu, logvar)
            loss = recon + beta * kl
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            kl_value = float(kl.item())
        else:
            logits = model(input_ids, condition_id=condition_id)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)
            kl = torch.tensor(0.0, device=device)
            loss = recon
            kl_value = 0.0

        bs = input_ids.size(0)
        acc = token_accuracy_ignore_pad(logits, target_ids, tokenizer.pad_id)
        loss_sum += float(loss.item()) * bs
        recon_sum += float(recon.item()) * bs
        kl_sum += kl_value * bs
        acc_sum += float(acc) * bs if pd.notnull(acc) else 0.0
        n += bs

    metrics = {
        "loss": loss_sum / max(1, n),
        "recon_loss": recon_sum / max(1, n),
        "kl_loss": kl_sum / max(1, n),
        "elbo": loss_sum / max(1, n),
        "token_acc": acc_sum / max(1, n),
        "n": n,
    }

    if family == "cvae" and all_mu:
        mu = torch.cat(all_mu, dim=0).numpy()
        var_per_dim = np.var(mu, axis=0)
        metrics["active_units"] = int(np.sum(var_per_dim > 1e-2))
        metrics["mean_mu_abs"] = float(np.mean(np.abs(mu)))
        metrics["mean_latent_var"] = float(np.mean(var_per_dim))
    else:
        metrics["active_units"] = np.nan
        metrics["mean_mu_abs"] = np.nan
        metrics["mean_latent_var"] = np.nan

    return metrics


@torch.no_grad()
def reconstruct_subset_sequences(model, family, loader, tokenizer, seed, device, use_condition, sample_limit=400):
    model.eval()
    rng = np.random.default_rng(seed)
    rows = []
    seen = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        target_ids = batch["target_ids"].to(device)
        cond = batch["condition_id"].to(device) if use_condition and "condition_id" in batch else None
        seq_true = batch["sequence"]

        if family == "cvae":
            z, mu, logvar = model.encode(input_ids, lengths, condition_id=cond)
            decoded = model.greedy_decode(
                z=mu,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=target_ids.size(1),
                condition_id=cond,
            )
            seq_pred = [tokenizer.decode(row.cpu().tolist()) for row in decoded]
        else:
            logits = model(input_ids, condition_id=cond)
            pred = logits.argmax(dim=-1)
            seq_pred = [tokenizer.decode(row.cpu().tolist()) for row in pred]

        idx = np.arange(len(seq_true))
        rng.shuffle(idx)
        for i in idx:
            rows.append({
                "sequence_true": clean_sequence(seq_true[i]),
                "sequence_recon": clean_sequence(seq_pred[i]),
                "normalized_edit_distance": normalized_edit_distance(clean_sequence(seq_true[i]), clean_sequence(seq_pred[i])),
                "exact_reconstruction": int(clean_sequence(seq_true[i]) == clean_sequence(seq_pred[i])),
            })
            seen += 1
            if seen >= sample_limit:
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)


@torch.no_grad()
def collect_latent_codes(model, loader, device, use_condition):
    model.eval()
    mus = []
    labels = []
    seqs = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        cond = batch["condition_id"].to(device) if use_condition and "condition_id" in batch else None
        z, mu, logvar = model.encode(input_ids, lengths, condition_id=cond)
        mus.append(mu.cpu().numpy())
        if cond is not None:
            labels.extend(cond.cpu().numpy().tolist())
        else:
            labels.extend([-1] * input_ids.size(0))
        seqs.extend(batch["sequence"])

    if not mus:
        return pd.DataFrame(columns=["pc1", "pc2", "condition_id", "sequence"])
    X = np.concatenate(mus, axis=0)
    pca = PCA(n_components=2, random_state=0)
    pcs = pca.fit_transform(X)
    out = pd.DataFrame({
        "pc1": pcs[:, 0],
        "pc2": pcs[:, 1],
        "condition_id": labels,
        "sequence": seqs,
    })
    return out


def active_units_from_loader(model, loader, device, use_condition, thresholds: Sequence[float]) -> pd.DataFrame:
    model.eval()
    mus = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            cond = batch["condition_id"].to(device) if use_condition and "condition_id" in batch else None
            z, mu, logvar = model.encode(input_ids, lengths, condition_id=cond)
            mus.append(mu.cpu().numpy())
    if not mus:
        return pd.DataFrame({"threshold": list(thresholds), "active_units": [np.nan] * len(thresholds)})
    mu_all = np.concatenate(mus, axis=0)
    var_per_dim = np.var(mu_all, axis=0)
    rows = []
    for th in thresholds:
        rows.append({"threshold": th, "active_units": int(np.sum(var_per_dim > th))})
    return pd.DataFrame(rows)


def evaluate_full_variant(
    model,
    family,
    model_tag,
    split_name,
    df_split,
    loader,
    tokenizer,
    evaluator_model,
    cfg,
    seed,
    device,
    use_condition,
):
    beta = cfg.train_cfg.beta_max
    loader_metrics = evaluate_generator_loader(
        model=model,
        loader=loader,
        tokenizer=tokenizer,
        beta=beta,
        device=device,
        family=family,
        use_condition=use_condition,
    )

    recon_df = reconstruct_subset_sequences(
        model=model,
        family=family,
        loader=loader,
        tokenizer=tokenizer,
        seed=seed,
        device=device,
        use_condition=use_condition,
        sample_limit=cfg.train_cfg.reconstruction_sample_limit,
    )

    cond_fidelity = np.nan
    if len(recon_df):
        preds = predict_conditions_for_sequences(
            evaluator_model,
            recon_df["sequence_recon"].tolist(),
            tokenizer,
            cfg,
            seed=seed,
            device=device,
        )
        true_labels = df_split.iloc[:len(recon_df)]["condition_id"].to_numpy(dtype=int)
        cond_fidelity = float(np.mean(preds == true_labels)) if len(true_labels) == len(preds) else np.nan

    out = {
        "model_tag": model_tag,
        "split": split_name,
        **loader_metrics,
        "recon_subset_n": int(len(recon_df)),
        "recon_edit_distance_mean": float(recon_df["normalized_edit_distance"].mean()) if len(recon_df) else np.nan,
        "recon_exact_rate": float(recon_df["exact_reconstruction"].mean()) if len(recon_df) else np.nan,
        "recon_condition_fidelity": cond_fidelity,
    }
    return out


# =============================================================================
# Generation / QA evaluation
# =============================================================================
def build_condition_generation_plan(df_train: pd.DataFrame, cfg: Step6Config) -> pd.DataFrame:
    support = (
        df_train.groupby("condition_id", as_index=False)
        .size()
        .rename(columns={"size": "n_train"})
        .sort_values("condition_id")
        .reset_index(drop=True)
    )
    base = cfg.train_cfg.generation_per_condition
    if cfg.train_cfg.generation_per_condition_support_scaled:
        scale = support["n_train"] / support["n_train"].median()
        n_gen = np.round(base * scale).astype(int)
        n_gen = np.clip(n_gen, cfg.train_cfg.min_generation_per_condition, cfg.train_cfg.max_generation_per_condition_cap)
    else:
        n_gen = np.repeat(base, len(support))
    support["n_generate"] = n_gen.astype(int)
    return support


def generate_sequences_for_variant(model, family, model_tag, df_train, tokenizer, cfg, seed, device, use_condition):
    plan = build_condition_generation_plan(df_train, cfg)
    rows = []

    train_ref_by_condition = (
        df_train.groupby("condition_id")["sequence"]
        .apply(list)
        .to_dict()
    )

    for _, r in plan.iterrows():
        cond_id = int(r["condition_id"])
        n_gen = int(r["n_generate"])

        if family == "cvae":
            cond_tensor = torch.full((n_gen,), cond_id, dtype=torch.long, device=device) if use_condition else None
            toks = model.sample_sequences(
                n=n_gen,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=cfg.train_cfg.max_decode_len,
                temperature=cfg.generated_eval_temperature_default,
                top_k=cfg.train_cfg.top_k,
                nucleus_p=cfg.train_cfg.nucleus_p,
                condition_id=cond_tensor,
                device=device,
            )
            seqs = [tokenizer.decode(t.cpu().tolist()) for t in toks]
            raw_lens = [int((t != tokenizer.pad_id).sum().item()) for t in toks]
        elif family == "gru":
            cond_tensor = torch.full((n_gen,), cond_id, dtype=torch.long, device=device)
            toks = model.sample_sequences(
                n=n_gen,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=cfg.train_cfg.max_decode_len,
                condition_id=cond_tensor,
                temperature=cfg.generated_eval_temperature_default,
                top_k=cfg.train_cfg.top_k,
                nucleus_p=cfg.train_cfg.nucleus_p,
                device=device,
            )
            seqs = [tokenizer.decode(t.cpu().tolist()) for t in toks]
            raw_lens = [int((t != tokenizer.pad_id).sum().item()) for t in toks]
        else:
            raise ValueError("This function is only for trainable generation models.")

        for s, raw_len in zip(seqs, raw_lens):
            rows.append({
                "model_tag": model_tag,
                "seed": seed,
                "condition_id": cond_id,
                "sequence": clean_sequence(s),
                "raw_decode_len_tokens": raw_len,
                "source": "generated",
            })

    gen_df = pd.DataFrame(rows)
    train_refs = set(df_train["sequence"].tolist())
    gen_df["valid_peptide"] = gen_df["sequence"].map(lambda s: int(is_valid_peptide(s, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)))
    gen_df["exact_train_overlap"] = gen_df["sequence"].map(lambda s: int(s in train_refs))
    lowc = gen_df["sequence"].map(lambda s: low_complexity_flags(s, cfg.qa_cfg))
    lowc_df = pd.DataFrame(list(lowc))
    gen_df = pd.concat([gen_df, lowc_df], axis=1)
    gen_df["early_eos_flag"] = (gen_df["raw_decode_len_tokens"] <= cfg.qa_cfg.early_eos_length_threshold).astype(int)
    gen_df["length"] = gen_df["sequence"].map(len)
    gen_df["charge_proxy"] = gen_df["sequence"].map(simple_charge_proxy)
    gen_df["mean_hydropathy"] = gen_df["sequence"].map(mean_hydropathy)
    gen_df["entropy"] = gen_df["sequence"].map(sequence_entropy)
    return gen_df


def evaluate_generation_table(gen_df, df_train, df_ref, evaluator_model, tokenizer, cfg, seed, device):
    out = {
        "n_generated": int(len(gen_df)),
    }

    if len(gen_df) == 0:
        out.update({
            "valid_rate": np.nan,
            "unique_rate": np.nan,
            "duplicate_rate": np.nan,
            "low_complexity_rate": np.nan,
            "early_eos_rate": np.nan,
            "exact_train_overlap_rate": np.nan,
            "classifier_condition_fidelity": np.nan,
            "descriptor_mean_abs_diff_mean": np.nan,
            "mean_pairwise_diversity": np.nan,
            "mean_pairwise_jaccard": np.nan,
            "collapse_duplicate_flag": np.nan,
        })
        return out

    out["valid_rate"] = float(gen_df["valid_peptide"].mean())
    out["unique_rate"] = float(gen_df["sequence"].nunique() / max(1, len(gen_df)))
    out["duplicate_rate"] = 1.0 - out["unique_rate"]
    out["low_complexity_rate"] = float(gen_df["low_complexity_any_flag"].mean())
    out["early_eos_rate"] = float(gen_df["early_eos_flag"].mean())
    out["exact_train_overlap_rate"] = float(gen_df["exact_train_overlap"].mean())
    out["collapse_duplicate_flag"] = int(out["duplicate_rate"] >= cfg.qa_cfg.collapse_duplicate_rate_threshold)

    pred_cond = predict_conditions_for_sequences(
        evaluator_model,
        gen_df["sequence"].tolist(),
        tokenizer,
        cfg,
        seed=seed,
        device=device,
    )
    out["classifier_condition_fidelity"] = float(np.mean(pred_cond == gen_df["condition_id"].to_numpy(dtype=int)))

    ref_desc = compute_sequence_descriptors(df_ref["sequence"].tolist())
    gen_desc = gen_df[["sequence", "length", "charge_proxy", "mean_hydropathy", "entropy"]].copy()
    out.update(descriptor_shift_score(ref_desc, gen_desc))

    div = pairwise_diversity_summary(
        gen_df["sequence"].tolist(),
        k=cfg.jaccard_k,
        pair_limit=cfg.pairwise_diversity_pair_limit,
        seed=seed,
    )
    out.update(div)

    out.update(descriptor_summary(gen_desc, prefix="gen_"))
    return out


def evaluate_generation_for_variant(model, family, model_tag, df_train, df_ref, tokenizer, evaluator_model, cfg, seed, device, use_condition):
    gen_df = generate_sequences_for_variant(
        model=model,
        family=family,
        model_tag=model_tag,
        df_train=df_train,
        tokenizer=tokenizer,
        cfg=cfg,
        seed=seed,
        device=device,
        use_condition=use_condition,
    )

    temp_rows = []
    for temp in cfg.train_cfg.decoding_temperatures:
        tmp_df = generate_sequences_for_variant_at_temperature(
            model=model,
            family=family,
            model_tag=model_tag,
            df_train=df_train,
            tokenizer=tokenizer,
            cfg=cfg,
            seed=seed,
            device=device,
            use_condition=use_condition,
            temperature=temp,
        )
        tmp_eval = evaluate_generation_table(
            tmp_df,
            df_train=df_train,
            df_ref=df_ref,
            evaluator_model=evaluator_model,
            tokenizer=tokenizer,
            cfg=cfg,
            seed=seed,
            device=device,
        )
        temp_rows.append({"temperature": temp, **tmp_eval})
    temp_eval_df = pd.DataFrame(temp_rows)

    gen_eval = evaluate_generation_table(
        gen_df,
        df_train=df_train,
        df_ref=df_ref,
        evaluator_model=evaluator_model,
        tokenizer=tokenizer,
        cfg=cfg,
        seed=seed,
        device=device,
    )

    return gen_eval, temp_eval_df, gen_df


def generate_sequences_for_variant_at_temperature(model, family, model_tag, df_train, tokenizer, cfg, seed, device, use_condition, temperature):
    plan = build_condition_generation_plan(df_train, cfg)
    rows = []

    for _, r in plan.iterrows():
        cond_id = int(r["condition_id"])
        n_gen = int(r["n_generate"])

        if family == "cvae":
            cond_tensor = torch.full((n_gen,), cond_id, dtype=torch.long, device=device) if use_condition else None
            toks = model.sample_sequences(
                n=n_gen,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=cfg.train_cfg.max_decode_len,
                temperature=temperature,
                top_k=cfg.train_cfg.top_k,
                nucleus_p=cfg.train_cfg.nucleus_p,
                condition_id=cond_tensor,
                device=device,
            )
            seqs = [tokenizer.decode(t.cpu().tolist()) for t in toks]
            raw_lens = [int((t != tokenizer.pad_id).sum().item()) for t in toks]
        elif family == "gru":
            cond_tensor = torch.full((n_gen,), cond_id, dtype=torch.long, device=device)
            toks = model.sample_sequences(
                n=n_gen,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=cfg.train_cfg.max_decode_len,
                condition_id=cond_tensor,
                temperature=temperature,
                top_k=cfg.train_cfg.top_k,
                nucleus_p=cfg.train_cfg.nucleus_p,
                device=device,
            )
            seqs = [tokenizer.decode(t.cpu().tolist()) for t in toks]
            raw_lens = [int((t != tokenizer.pad_id).sum().item()) for t in toks]
        else:
            raise ValueError("Temperature sweep only applies to trainable generators.")

        for s, raw_len in zip(seqs, raw_lens):
            rows.append({
                "model_tag": model_tag,
                "seed": seed,
                "condition_id": cond_id,
                "sequence": clean_sequence(s),
                "raw_decode_len_tokens": raw_len,
            })

    gen_df = pd.DataFrame(rows)
    train_refs = set(df_train["sequence"].tolist())
    gen_df["valid_peptide"] = gen_df["sequence"].map(lambda s: int(is_valid_peptide(s, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)))
    gen_df["exact_train_overlap"] = gen_df["sequence"].map(lambda s: int(s in train_refs))
    lowc = gen_df["sequence"].map(lambda s: low_complexity_flags(s, cfg.qa_cfg))
    lowc_df = pd.DataFrame(list(lowc))
    gen_df = pd.concat([gen_df, lowc_df], axis=1)
    gen_df["early_eos_flag"] = (gen_df["raw_decode_len_tokens"] <= cfg.qa_cfg.early_eos_length_threshold).astype(int)
    gen_df["length"] = gen_df["sequence"].map(len)
    gen_df["charge_proxy"] = gen_df["sequence"].map(simple_charge_proxy)
    gen_df["mean_hydropathy"] = gen_df["sequence"].map(mean_hydropathy)
    gen_df["entropy"] = gen_df["sequence"].map(sequence_entropy)
    return gen_df


# =============================================================================
# Retrieval baseline
# =============================================================================
def build_train_reference_by_condition(df_train: pd.DataFrame) -> Dict[int, List[str]]:
    return df_train.groupby("condition_id")["sequence"].apply(list).to_dict()


def retrieval_match_sequence(query_seq: str, refs: List[str], k: int = 3) -> Tuple[str, float]:
    if not refs:
        return "", np.nan
    best_ref = None
    best_sim = -1.0
    for r in refs:
        sim = jaccard_similarity(query_seq, r, k=k)
        if sim > best_sim:
            best_sim = sim
            best_ref = r
    return best_ref, best_sim


def run_retrieval_baseline(model_tag, seed, df_train, df_val, df_test, tokenizer, evaluator_model, cfg, dirs, device):
    seed_all(seed)
    refs_by_cond = build_train_reference_by_condition(df_train)

    val_rows = []
    for _, row in df_val.iterrows():
        refs = refs_by_cond.get(int(row["condition_id"]), [])
        hit, sim = retrieval_match_sequence(row["sequence"], refs, k=cfg.jaccard_k)
        val_rows.append({
            "sequence_true": row["sequence"],
            "condition_id": int(row["condition_id"]),
            "sequence_retrieved": hit,
            "jaccard_to_retrieved": sim,
            "normalized_edit_distance": normalized_edit_distance(row["sequence"], hit),
            "exact_reconstruction": int(row["sequence"] == hit),
        })
    val_recon_df = pd.DataFrame(val_rows)

    test_rows = []
    for _, row in df_test.iterrows():
        refs = refs_by_cond.get(int(row["condition_id"]), [])
        hit, sim = retrieval_match_sequence(row["sequence"], refs, k=cfg.jaccard_k)
        test_rows.append({
            "sequence_true": row["sequence"],
            "condition_id": int(row["condition_id"]),
            "sequence_retrieved": hit,
            "jaccard_to_retrieved": sim,
            "normalized_edit_distance": normalized_edit_distance(row["sequence"], hit),
            "exact_reconstruction": int(row["sequence"] == hit),
        })
    test_recon_df = pd.DataFrame(test_rows)

    def summarize_recon(df_recon, split_name):
        preds = predict_conditions_for_sequences(
            evaluator_model,
            df_recon["sequence_retrieved"].tolist(),
            tokenizer,
            cfg,
            seed=seed,
            device=device,
        )
        cond_fidelity = float(np.mean(preds == df_recon["condition_id"].to_numpy(dtype=int))) if len(df_recon) else np.nan
        return {
            "model_tag": model_tag,
            "split": split_name,
            "loss": np.nan,
            "recon_loss": np.nan,
            "kl_loss": 0.0,
            "elbo": np.nan,
            "token_acc": np.nan,
            "active_units": np.nan,
            "mean_mu_abs": np.nan,
            "mean_latent_var": np.nan,
            "recon_subset_n": int(len(df_recon)),
            "recon_edit_distance_mean": float(df_recon["normalized_edit_distance"].mean()) if len(df_recon) else np.nan,
            "recon_exact_rate": float(df_recon["exact_reconstruction"].mean()) if len(df_recon) else np.nan,
            "recon_condition_fidelity": cond_fidelity,
        }

    val_eval = summarize_recon(val_recon_df, "val")
    test_eval = summarize_recon(test_recon_df, "test")

    gen_df = []
    rng = np.random.default_rng(seed)
    gen_plan = build_condition_generation_plan(df_train, cfg)
    for _, r in gen_plan.iterrows():
        cond_id = int(r["condition_id"])
        n_gen = int(r["n_generate"])
        refs = refs_by_cond.get(cond_id, [])
        if not refs:
            continue
        picks = rng.choice(len(refs), size=n_gen, replace=True)
        for idx in picks:
            seq = refs[int(idx)]
            gen_df.append({
                "model_tag": model_tag,
                "seed": seed,
                "condition_id": cond_id,
                "sequence": seq,
                "raw_decode_len_tokens": len(seq),
                "source": "retrieval_train_reference",
            })
    gen_df = pd.DataFrame(gen_df)
    if len(gen_df):
        gen_df["valid_peptide"] = gen_df["sequence"].map(lambda s: int(is_valid_peptide(s, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)))
        gen_df["exact_train_overlap"] = 1
        lowc = gen_df["sequence"].map(lambda s: low_complexity_flags(s, cfg.qa_cfg))
        lowc_df = pd.DataFrame(list(lowc))
        gen_df = pd.concat([gen_df, lowc_df], axis=1)
        gen_df["early_eos_flag"] = 0
        gen_df["length"] = gen_df["sequence"].map(len)
        gen_df["charge_proxy"] = gen_df["sequence"].map(simple_charge_proxy)
        gen_df["mean_hydropathy"] = gen_df["sequence"].map(mean_hydropathy)
        gen_df["entropy"] = gen_df["sequence"].map(sequence_entropy)

    gen_eval = evaluate_generation_table(
        gen_df=gen_df,
        df_train=df_train,
        df_ref=df_test,
        evaluator_model=evaluator_model,
        tokenizer=tokenizer,
        cfg=cfg,
        seed=seed,
        device=device,
    ) if len(gen_df) else None

    temp_eval_df = pd.DataFrame({
        "temperature": list(cfg.train_cfg.decoding_temperatures),
        "n_generated": [len(gen_df)] * len(cfg.train_cfg.decoding_temperatures),
        "valid_rate": [gen_eval["valid_rate"] if gen_eval else np.nan] * len(cfg.train_cfg.decoding_temperatures),
        "unique_rate": [gen_eval["unique_rate"] if gen_eval else np.nan] * len(cfg.train_cfg.decoding_temperatures),
        "duplicate_rate": [gen_eval["duplicate_rate"] if gen_eval else np.nan] * len(cfg.train_cfg.decoding_temperatures),
        "low_complexity_rate": [gen_eval["low_complexity_rate"] if gen_eval else np.nan] * len(cfg.train_cfg.decoding_temperatures),
        "exact_train_overlap_rate": [gen_eval["exact_train_overlap_rate"] if gen_eval else np.nan] * len(cfg.train_cfg.decoding_temperatures),
        "classifier_condition_fidelity": [gen_eval["classifier_condition_fidelity"] if gen_eval else np.nan] * len(cfg.train_cfg.decoding_temperatures),
    })

    done_path = dirs["artifacts"] / f"step6_done_{model_tag}_seed{seed}.json"
    export_json(
        {
            "model_tag": model_tag,
            "seed": seed,
            "family": "retrieval",
            "val_eval": val_eval,
            "test_eval": test_eval,
            "generation_eval": gen_eval,
        },
        done_path,
    )

    return {
        "model_tag": model_tag,
        "seed": seed,
        "family": "retrieval",
        "best_epoch": 0,
        "checkpoint_path": "",
        "history_df": pd.DataFrame(),
        "val_eval": val_eval,
        "test_eval": test_eval,
        "generation_eval": gen_eval,
        "temperature_eval_df": temp_eval_df,
        "generated_df": gen_df,
    }


# =============================================================================
# Aggregation / reporting
# =============================================================================
def flatten_result_record(res: Dict[str, Any]) -> Dict[str, Any]:
    row = {
        "model_tag": res["model_tag"],
        "seed": res["seed"],
        "family": res["family"],
        "best_epoch": res.get("best_epoch", np.nan),
        "checkpoint_path": res.get("checkpoint_path", ""),
    }
    for prefix, block in [("val", res.get("val_eval")), ("test", res.get("test_eval")), ("gen", res.get("generation_eval"))]:
        if block is None:
            continue
        for k, v in block.items():
            if k in {"model_tag", "split"}:
                continue
            row[f"{prefix}_{k}"] = v
    return row


def aggregate_seed_metrics(per_seed_df: pd.DataFrame, cfg: Step6Config) -> pd.DataFrame:
    metric_cols = [c for c in per_seed_df.columns if c not in {"model_tag", "seed", "family", "checkpoint_path"} and pd.api.types.is_numeric_dtype(per_seed_df[c])]
    rows = []
    for model_tag, g in per_seed_df.groupby("model_tag"):
        for metric in metric_cols:
            vals = g[metric].dropna().tolist()
            stat = bootstrap_mean_ci(vals, iterations=cfg.bootstrap_iterations, seed=42)
            rows.append({
                "model_tag": model_tag,
                "metric": metric,
                **stat,
            })
    return pd.DataFrame(rows)


def build_summary_tables(results: List[Dict[str, Any]], cfg: Step6Config, dirs: Dict[str, Path]) -> Dict[str, pd.DataFrame]:
    per_seed_rows = [flatten_result_record(r) for r in results]
    per_seed_df = pd.DataFrame(per_seed_rows).sort_values(["model_tag", "seed"]).reset_index(drop=True)
    export_csv(per_seed_df, dirs["tables"] / "step6_per_seed_summary.csv")

    agg_df = aggregate_seed_metrics(per_seed_df, cfg)
    export_csv(agg_df, dirs["tables"] / "step6_aggregated_summary.csv")

    temp_rows = []
    for r in results:
        df = r.get("temperature_eval_df")
        if df is not None and len(df):
            tmp = df.copy()
            tmp["model_tag"] = r["model_tag"]
            tmp["seed"] = r["seed"]
            temp_rows.append(tmp)
    temp_df = pd.concat(temp_rows, axis=0, ignore_index=True) if temp_rows else pd.DataFrame()
    if len(temp_df):
        export_csv(temp_df, dirs["tables"] / "step6_temperature_sensitivity_summary.csv")

    gen_rows = []
    for r in results:
        g = r.get("generated_df")
        if g is not None and len(g):
            gen_rows.append(g.copy())
    gen_df_all = pd.concat(gen_rows, axis=0, ignore_index=True) if gen_rows else pd.DataFrame()
    if len(gen_df_all):
        export_csv(gen_df_all, dirs["tables"] / "step6_all_generated_sequences.csv")

    return {
        "per_seed_df": per_seed_df,
        "agg_df": agg_df,
        "temp_df": temp_df,
        "all_generated_df": gen_df_all,
    }


# =============================================================================
# Figures
# =============================================================================
def get_metric_summary(agg_df: pd.DataFrame, model_tag: str, metric: str) -> Dict[str, float]:
    x = agg_df.loc[(agg_df["model_tag"] == model_tag) & (agg_df["metric"] == metric)]
    if len(x) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    row = x.iloc[0]
    return {"mean": float(row["mean"]), "ci_low": float(row["ci_low"]), "ci_high": float(row["ci_high"])}


def build_main_figure(history_df_all: pd.DataFrame, per_seed_df: pd.DataFrame, agg_df: pd.DataFrame, dirs: Dict[str, Path], cfg: Step6Config):
    fig = plt.figure(figsize=(13.5, 9.0))
    gs = GridSpec(2, 2, figure=fig)

    # Panel a: Full CVAE validation curves across seeds
    ax = fig.add_subplot(gs[0, 0])
    hx = history_df_all.loc[history_df_all["model_tag"] == "full_cvae"].copy()
    if len(hx):
        curve = hx.groupby("epoch", as_index=False)[["val_recon_loss", "val_kl_loss"]].mean()
        ax.plot(curve["epoch"], curve["val_recon_loss"], label="Val reconstruction", linewidth=2)
        ax.plot(curve["epoch"], curve["val_kl_loss"], label="Val KL", linewidth=2)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss component")
        ax.legend(frameon=False, loc="best")
    ax.set_title("Full CVAE validation dynamics")
    style_axis(ax, "y")
    add_panel_label(ax, "a")

    # Panel b: test reconstruction edit distance across variants
    ax = fig.add_subplot(gs[0, 1])
    xs = []
    ys = []
    lo = []
    hi = []
    for i, m in enumerate(MODEL_ORDER):
        stat = get_metric_summary(agg_df, m, "test_recon_edit_distance_mean")
        xs.append(i)
        ys.append(stat["mean"])
        lo.append(stat["mean"] - stat["ci_low"] if pd.notnull(stat["ci_low"]) and pd.notnull(stat["mean"]) else 0.0)
        hi.append(stat["ci_high"] - stat["mean"] if pd.notnull(stat["ci_high"]) and pd.notnull(stat["mean"]) else 0.0)
    ax.errorbar(xs, ys, yerr=[lo, hi], fmt="o", capsize=4, linewidth=1.5)
    ax.set_xticks(xs)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=30, ha="right")
    ax.set_ylabel("Test reconstruction edit distance")
    ax.set_title("Held-out reconstruction summary")
    style_axis(ax, "y")
    add_panel_label(ax, "b")

    # Panel c: conditional fidelity
    ax = fig.add_subplot(gs[1, 0])
    xs = []
    ys = []
    lo = []
    hi = []
    for i, m in enumerate(MODEL_ORDER):
        stat = get_metric_summary(agg_df, m, "gen_classifier_condition_fidelity")
        xs.append(i)
        ys.append(stat["mean"])
        lo.append(stat["mean"] - stat["ci_low"] if pd.notnull(stat["ci_low"]) and pd.notnull(stat["mean"]) else 0.0)
        hi.append(stat["ci_high"] - stat["mean"] if pd.notnull(stat["ci_high"]) and pd.notnull(stat["mean"]) else 0.0)
    ax.errorbar(xs, ys, yerr=[lo, hi], fmt="o", capsize=4, linewidth=1.5)
    ax.set_xticks(xs)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=30, ha="right")
    ax.set_ylabel("Generated condition fidelity")
    ax.set_title("Condition fidelity of generated peptides")
    style_axis(ax, "y")
    add_panel_label(ax, "c")

    # Panel d: generation diversity
    ax = fig.add_subplot(gs[1, 1])
    xs = []
    ys = []
    lo = []
    hi = []
    for i, m in enumerate(MODEL_ORDER):
        stat = get_metric_summary(agg_df, m, "gen_mean_pairwise_diversity")
        xs.append(i)
        ys.append(stat["mean"])
        lo.append(stat["mean"] - stat["ci_low"] if pd.notnull(stat["ci_low"]) and pd.notnull(stat["mean"]) else 0.0)
        hi.append(stat["ci_high"] - stat["mean"] if pd.notnull(stat["ci_high"]) and pd.notnull(stat["mean"]) else 0.0)
    ax.errorbar(xs, ys, yerr=[lo, hi], fmt="o", capsize=4, linewidth=1.5)
    ax.set_xticks(xs)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=30, ha="right")
    ax.set_ylabel("Pairwise diversity")
    ax.set_title("Generated-set diversity")
    style_axis(ax, "y")
    add_panel_label(ax, "d")

    fig.tight_layout()
    out = save_figure(fig, dirs["figures_main"] / "Fig6_step6_training_ablation_summary", cfg.fig_cfg)

    # source data
    src = []
    for m in MODEL_ORDER:
        src.append({
            "model_tag": m,
            "test_recon_edit_distance_mean": get_metric_summary(agg_df, m, "test_recon_edit_distance_mean")["mean"],
            "gen_classifier_condition_fidelity_mean": get_metric_summary(agg_df, m, "gen_classifier_condition_fidelity")["mean"],
            "gen_mean_pairwise_diversity_mean": get_metric_summary(agg_df, m, "gen_mean_pairwise_diversity")["mean"],
        })
    export_csv(pd.DataFrame(src), dirs["source_data_figures"] / "Fig6_step6_training_ablation_summary_source_data.csv")
    return out


def build_supp_figure_latent(results: List[Dict[str, Any]], df_test: pd.DataFrame, tokenizer: PeptideTokenizer, cfg: Step6Config, dirs: Dict[str, Path], device: torch.device):
    candidate = next((r for r in results if r["model_tag"] == "full_cvae"), None)
    if candidate is None or not candidate.get("checkpoint_path"):
        return None

    model = ConditionalSequenceVAE(
        vocab_size=tokenizer.vocab_size,
        cond_vocab_size=int(df_test["condition_id"].nunique() if len(df_test) else 1),
        emb_dim=cfg.train_cfg.emb_dim,
        hidden_dim=cfg.train_cfg.hidden_dim,
        latent_dim=cfg.train_cfg.latent_dim,
        num_layers=cfg.train_cfg.num_layers,
        dropout=cfg.train_cfg.dropout,
        use_condition=True,
    ).to(device)

    ckpt = torch.load(candidate["checkpoint_path"], map_location=device)
    model.load_state_dict(ckpt["state_dict"])

    loader = build_sequence_dataloaders(df_test, df_test.iloc[:0].copy(), df_test.iloc[:0].copy(), tokenizer, cfg, candidate["seed"], use_condition=True)["train"]
    latent_df = collect_latent_codes(model, loader, device=device, use_condition=True)
    active_df = active_units_from_loader(model, loader, device=device, use_condition=True, thresholds=cfg.active_unit_thresholds)

    fig = plt.figure(figsize=(12.0, 5.5))
    gs = GridSpec(1, 2, figure=fig)

    ax = fig.add_subplot(gs[0, 0])
    if len(latent_df):
        sc = ax.scatter(latent_df["pc1"], latent_df["pc2"], c=latent_df["condition_id"], s=10, alpha=0.75)
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
    ax.set_title("Test-set latent means (PCA)")
    style_axis(ax, "both")
    add_panel_label(ax, "a")

    ax = fig.add_subplot(gs[0, 1])
    if len(active_df):
        ax.plot(active_df["threshold"], active_df["active_units"], marker="o", linewidth=2)
        ax.set_xscale("log")
        ax.set_xlabel("Variance threshold")
        ax.set_ylabel("Active latent units")
    ax.set_title("Latent activity sensitivity")
    style_axis(ax, "y")
    add_panel_label(ax, "b")

    fig.tight_layout()
    out = save_figure(fig, dirs["figures_supplementary"] / "FigS_step6_latent_diagnostics", cfg.fig_cfg)
    export_csv(latent_df, dirs["source_data_figures"] / "FigS_step6_latent_diagnostics_pca_source_data.csv")
    export_csv(active_df, dirs["source_data_figures"] / "FigS_step6_latent_diagnostics_active_units_source_data.csv")
    return out


def build_supp_figure_temperature(temp_df: pd.DataFrame, all_generated_df: pd.DataFrame, cfg: Step6Config, dirs: Dict[str, Path]):
    if temp_df is None or len(temp_df) == 0:
        return None

    fig = plt.figure(figsize=(13.0, 5.5))
    gs = GridSpec(1, 2, figure=fig)

    ax = fig.add_subplot(gs[0, 0])
    tx = (
        temp_df.groupby(["model_tag", "temperature"], as_index=False)[["classifier_condition_fidelity", "unique_rate"]]
        .mean()
    )
    for model_tag in ["full_cvae", "conditional_gru_lm", "no_condition", "small_latent", "no_warmup", "retrieval"]:
        part = tx.loc[tx["model_tag"] == model_tag]
        if len(part) == 0:
            continue
        ax.plot(part["temperature"], part["classifier_condition_fidelity"], marker="o", label=MODEL_LABELS.get(model_tag, model_tag))
    ax.set_xlabel("Sampling temperature")
    ax.set_ylabel("Condition fidelity")
    ax.set_title("Decoding temperature sensitivity")
    ax.legend(frameon=False, fontsize=8)
    style_axis(ax, "y")
    add_panel_label(ax, "a")

    ax = fig.add_subplot(gs[0, 1])
    if all_generated_df is not None and len(all_generated_df):
        part = (
            all_generated_df.groupby("model_tag", as_index=False)
            .agg(
                valid_rate=("valid_peptide", "mean"),
                low_complexity_rate=("low_complexity_any_flag", "mean"),
                overlap_rate=("exact_train_overlap", "mean"),
            )
        )
        xs = np.arange(len(part))
        ax.plot(xs, part["valid_rate"], marker="o", linewidth=2, label="Valid rate")
        ax.plot(xs, part["low_complexity_rate"], marker="o", linewidth=2, label="Low-complexity rate")
        ax.plot(xs, part["overlap_rate"], marker="o", linewidth=2, label="Exact train overlap")
        ax.set_xticks(xs)
        ax.set_xticklabels([MODEL_LABELS.get(x, x) for x in part["model_tag"]], rotation=30, ha="right")
        ax.legend(frameon=False, fontsize=8)
    ax.set_ylabel("Rate")
    ax.set_title("Generation-time QA summary")
    style_axis(ax, "y")
    add_panel_label(ax, "b")

    fig.tight_layout()
    out = save_figure(fig, dirs["figures_supplementary"] / "FigS_step6_temperature_and_qa", cfg.fig_cfg)
    export_csv(temp_df, dirs["source_data_figures"] / "FigS_step6_temperature_and_qa_temperature_source_data.csv")
    if all_generated_df is not None and len(all_generated_df):
        qa_src = (
            all_generated_df.groupby("model_tag", as_index=False)
            .agg(
                valid_rate=("valid_peptide", "mean"),
                low_complexity_rate=("low_complexity_any_flag", "mean"),
                overlap_rate=("exact_train_overlap", "mean"),
            )
        )
        export_csv(qa_src, dirs["source_data_figures"] / "FigS_step6_temperature_and_qa_generation_source_data.csv")
    return out


# =============================================================================
# Manifest / report
# =============================================================================
def write_step6_report(
    cfg: Step6Config,
    input_info: Dict[str, Any],
    evaluator_res: Dict[str, Any],
    results: List[Dict[str, Any]],
    summary_tables: Dict[str, pd.DataFrame],
    dirs: Dict[str, Path],
):
    per_seed_df = summary_tables["per_seed_df"]
    agg_df = summary_tables["agg_df"]

    key_metrics = []
    for m in MODEL_ORDER:
        row = {"model_tag": m}
        for metric in [
            "test_recon_edit_distance_mean",
            "test_recon_exact_rate",
            "test_recon_condition_fidelity",
            "gen_classifier_condition_fidelity",
            "gen_mean_pairwise_diversity",
            "gen_exact_train_overlap_rate",
            "gen_low_complexity_rate",
        ]:
            stat = get_metric_summary(agg_df, m, metric)
            row[f"{metric}_mean"] = stat["mean"]
            row[f"{metric}_ci_low"] = stat["ci_low"]
            row[f"{metric}_ci_high"] = stat["ci_high"]
        key_metrics.append(row)

    report = {
        "run_id": str(uuid.uuid4()),
        "timestamp_unix": time.time(),
        "config_hash": config_hash(cfg),
        "environment": environment_snapshot(),
        "config": asdict(cfg),
        "step4_input": {
            "csv_path": input_info["step4_csv_path"],
            "artifacts_dir": input_info["step4_artifacts_dir"],
        },
        "split_summary_path": str(dirs["tables"] / "step6_input_split_summary.csv"),
        "condition_support_path": str(dirs["tables"] / "step6_condition_support_summary.csv"),
        "evaluator": {
            "checkpoint_path": evaluator_res["checkpoint_path"],
            "val_metrics": evaluator_res["val_metrics"],
            "test_metrics": evaluator_res["test_metrics"],
        },
        "n_variant_runs": len(results),
        "key_metrics": key_metrics,
        "output_paths": {
            "per_seed_summary": str(dirs["tables"] / "step6_per_seed_summary.csv"),
            "aggregated_summary": str(dirs["tables"] / "step6_aggregated_summary.csv"),
            "temperature_summary": str(dirs["tables"] / "step6_temperature_sensitivity_summary.csv"),
            "all_generated_sequences": str(dirs["tables"] / "step6_all_generated_sequences.csv"),
        },
    }
    export_json(report, dirs["reports"] / "step6_run_report.json")
    return report


# =============================================================================
# Runner
# =============================================================================
def run_step6_v4(
    step4_model_ready_csv=DEFAULT_STEP4_MODEL_READY_CSV,
    step4_artifacts_dir=DEFAULT_STEP4_ARTIFACTS_DIR,
    step5_root=DEFAULT_STEP5_ROOT,
    step6_root=DEFAULT_STEP6_ROOT,
    export_png=True,
    export_pdf=True,
    export_tiff=True,
    dpi_png=300,
    dpi_tiff=600,
    verbose=True,
):
    cfg = Step6Config(
        step4_model_ready_csv=str(step4_model_ready_csv),
        step4_artifacts_dir=str(step4_artifacts_dir),
        step5_root=str(step5_root),
        step6_root=str(step6_root),
        fig_cfg=FigureExportConfig(
            export_png=export_png,
            export_pdf=export_pdf,
            export_tiff=export_tiff,
            dpi_png=dpi_png,
            dpi_tiff=dpi_tiff,
        ),
        runtime_cfg=RuntimeConfig(verbose=verbose),
    )

    root = Path(cfg.step6_root).resolve()
    dirs = ensure_dirs(root)
    device = device_for_training()

    if cfg.runtime_cfg.verbose:
        print_header("Step 6 v4 — loading inputs")

    export_json(environment_snapshot(), dirs["artifacts"] / "step6_environment_snapshot.json")
    export_json(asdict(cfg), dirs["artifacts"] / "step6_config.json")

    input_info = load_step4_inputs(cfg, dirs)
    tokenizer = input_info["tokenizer"]
    train_df = input_info["train_df"]
    val_df = input_info["val_df"]
    test_df = input_info["test_df"]

    n_conditions = int(train_df["condition_id"].nunique())
    if cfg.runtime_cfg.verbose:
        print(f"Device: {device}")
        print(f"Train / Val / Test sizes: {len(train_df)} / {len(val_df)} / {len(test_df)}")
        print(f"Number of train-supported conditions: {n_conditions}")
        print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

    # -------------------------------------------------------------------------
    # Train condition evaluator using the first seed for deterministic support
    # -------------------------------------------------------------------------
    evaluator_seed = int(cfg.repeated_seeds[0])
    if cfg.runtime_cfg.verbose:
        print_header(f"Training condition evaluator (seed={evaluator_seed})")

    evaluator_res = train_condition_classifier(
        df_train=train_df,
        df_val=val_df,
        df_test=test_df,
        tokenizer=tokenizer,
        n_classes=n_conditions,
        cfg=cfg,
        seed=evaluator_seed,
        device=device,
        dirs=dirs,
    )
    evaluator_model = evaluator_res["model"]

    # -------------------------------------------------------------------------
    # Train / evaluate variants
    # -------------------------------------------------------------------------
    results = []
    history_frames = []

    for model_tag in MODEL_ORDER:
        if not variant_enabled(cfg, model_tag):
            continue

        for seed in cfg.repeated_seeds:
            if not should_run_variant_seed(cfg, model_tag, seed):
                continue

            done_flag = get_run_done_flag(dirs, model_tag, seed)
            if cfg.runtime_cfg.skip_completed_runs and done_flag.exists():
                if cfg.runtime_cfg.verbose:
                    print(f"[skip] {model_tag} seed={seed} because done flag exists: {done_flag}")

            if cfg.runtime_cfg.verbose:
                print_header(f"Running variant={model_tag} seed={seed}")

            res = train_one_variant(
                model_tag=model_tag,
                seed=int(seed),
                df_train=train_df,
                df_val=val_df,
                df_test=test_df,
                tokenizer=tokenizer,
                n_conditions=n_conditions,
                evaluator_model=evaluator_model,
                cfg=cfg,
                dirs=dirs,
                device=device,
            )
            results.append(res)

            hdf = res.get("history_df")
            if hdf is not None and len(hdf):
                history_frames.append(hdf.copy())

            # per-run exports
            if res.get("generated_df") is not None and len(res["generated_df"]):
                export_csv(res["generated_df"], dirs["qa"] / f"step6_generated_sequences_{model_tag}_seed{seed}.csv")
            if res.get("temperature_eval_df") is not None and len(res["temperature_eval_df"]):
                export_csv(res["temperature_eval_df"], dirs["qa"] / f"step6_temperature_sensitivity_{model_tag}_seed{seed}.csv")

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if len(results) == 0:
        raise RuntimeError("No Step 6 runs were executed. Check runtime filters or ablation flags.")

    history_df_all = pd.concat(history_frames, axis=0, ignore_index=True) if history_frames else pd.DataFrame()
    if len(history_df_all):
        export_csv(history_df_all, dirs["logs"] / "step6_all_training_histories.csv")

    summary_tables = build_summary_tables(results, cfg, dirs)

    # -------------------------------------------------------------------------
    # Figures
    # -------------------------------------------------------------------------
    figure_manifest = {}
    if not cfg.runtime_cfg.no_figure_mode:
        if cfg.runtime_cfg.verbose:
            print_header("Building figures")

        fig_main = build_main_figure(
            history_df_all=history_df_all,
            per_seed_df=summary_tables["per_seed_df"],
            agg_df=summary_tables["agg_df"],
            dirs=dirs,
            cfg=cfg,
        )
        figure_manifest["main_figure"] = fig_main

        fig_s1 = build_supp_figure_latent(
            results=results,
            df_test=test_df,
            tokenizer=tokenizer,
            cfg=cfg,
            dirs=dirs,
            device=device,
        )
        figure_manifest["supp_latent"] = fig_s1

        fig_s2 = build_supp_figure_temperature(
            temp_df=summary_tables["temp_df"],
            all_generated_df=summary_tables["all_generated_df"],
            cfg=cfg,
            dirs=dirs,
        )
        figure_manifest["supp_temperature_qa"] = fig_s2

        export_json(figure_manifest, dirs["reports"] / "step6_figure_manifest.json")

    # -------------------------------------------------------------------------
    # Final report
    # -------------------------------------------------------------------------
    report = write_step6_report(
        cfg=cfg,
        input_info=input_info,
        evaluator_res=evaluator_res,
        results=results,
        summary_tables=summary_tables,
        dirs=dirs,
    )

    if cfg.runtime_cfg.verbose:
        print_header("Step 6 v4 completed")
        print(f"Per-seed summary: {dirs['tables'] / 'step6_per_seed_summary.csv'}")
        print(f"Aggregated summary: {dirs['tables'] / 'step6_aggregated_summary.csv'}")
        print(f"Run report: {dirs['reports'] / 'step6_run_report.json'}")
        if not cfg.runtime_cfg.no_figure_mode:
            print(f"Main figures dir: {dirs['figures_main']}")
            print(f"Supplementary figures dir: {dirs['figures_supplementary']}")

    return {
        "config": cfg,
        "dirs": dirs,
        "input_info": input_info,
        "evaluator_res": evaluator_res,
        "results": results,
        "summary_tables": summary_tables,
        "report": report,
        "figure_manifest": figure_manifest,
    }


# =============================================================================
# Execute
# =============================================================================
if __name__ == "__main__":
    out = run_step6_v4(
        step4_model_ready_csv=DEFAULT_STEP4_MODEL_READY_CSV,
        step4_artifacts_dir=DEFAULT_STEP4_ARTIFACTS_DIR,
        step5_root=DEFAULT_STEP5_ROOT,
        step6_root=DEFAULT_STEP6_ROOT,
        export_png=True,
        export_pdf=True,
        export_tiff=True,
        dpi_png=300,
        dpi_tiff=600,
        verbose=True,
    )

new code step 6

In [ ]:
from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE — Step 6 v5
# Collapse-resistant conditional VAE training + benchmark-style figure package
# =============================================================================

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import gc
import json
import time
import uuid
import math
import copy
import random
import hashlib
import warnings
import platform
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import PCA


# =============================================================================
# Paths
# =============================================================================
PROJECT_ROOT = Path("/home/data3/Moe/nature_computational2").resolve()

DEFAULT_STEP4_MODEL_READY_CSV = PROJECT_ROOT / "step4" / "tables" / "step4_model_ready_sequences.csv"
DEFAULT_STEP4_ARTIFACTS_DIR = PROJECT_ROOT / "step4" / "artifacts"
DEFAULT_STEP5_ROOT = PROJECT_ROOT / "step5"
DEFAULT_STEP6_ROOT = PROJECT_ROOT / "step6_v5"


# =============================================================================
# Constants
# =============================================================================
AA = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA)

KD = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8, "G": -0.4, "H": -3.2,
    "I": 4.5, "K": -3.9, "L": 3.8, "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5,
    "R": -4.5, "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

PALETTE = {
    "retrieval": "#B4C5E4",
    "full_cvae": "#C0001A",
    "conditional_gru_lm": "#FF1A1A",
    "random_condition": "#1F6B72",
    "no_condition": "#2C4B7C",
    "small_latent": "#8B0000",
    "gray": "#7F7F7F",
    "grid": "#D9D9D9",
    "black": "#333333",
}

MODEL_ORDER = [
    "retrieval",
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "retrieval": "Retrieval",
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

MODEL_COLORS = {k: PALETTE[k] for k in MODEL_ORDER}


# =============================================================================
# Config
# =============================================================================
@dataclass(frozen=True)
class FigureExportConfig:
    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600


@dataclass(frozen=True)
class GeneratorTrainConfig:
    max_seq_len: int = 64
    batch_size: int = 64
    num_workers: int = 0

    emb_dim: int = 96
    enc_hidden_dim: int = 192
    dec_hidden_dim: int = 128
    latent_dim: int = 32
    num_layers: int = 1
    dropout: float = 0.15

    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 40
    clip_grad_norm: float = 1.0
    early_stopping_patience: int = 8

    beta_max: float = 1.0
    beta_warmup_fraction: float = 0.40
    kl_schedule: str = "cyclical"  # ["linear", "cyclical", "constant"]
    kl_cycle_length: int = 10
    free_bits_per_dim: float = 0.02

    aux_condition_loss_weight: float = 0.50
    decoder_input_dropout: float = 0.20

    min_length: int = 5
    max_decode_len: int = 60

    generation_per_condition: int = 80
    generation_per_condition_support_scaled: bool = True
    min_generation_per_condition: int = 40
    max_generation_per_condition_cap: int = 120

    sample_temperature: float = 1.0
    decoding_temperatures: Tuple[float, ...] = (0.8, 1.0, 1.2)
    top_k: Optional[int] = None
    nucleus_p: Optional[float] = None

    reconstruction_sample_limit: int = 500


@dataclass(frozen=True)
class EvaluatorTrainConfig:
    emb_dim: int = 64
    hidden_dim: int = 128
    num_layers: int = 1
    dropout: float = 0.10
    batch_size: int = 64
    num_workers: int = 0
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 20
    early_stopping_patience: int = 5
    clip_grad_norm: float = 1.0


@dataclass(frozen=True)
class AblationConfig:
    run_retrieval_baseline: bool = True
    run_full_cvae: bool = True
    run_conditional_gru_lm: bool = True
    run_random_condition: bool = True
    run_no_condition: bool = True
    run_small_latent: bool = True
    small_latent_dim: int = 8


@dataclass(frozen=True)
class QAConfig:
    low_complexity_repeat_threshold: int = 5
    min_unique_amino_acids: int = 4
    max_single_aa_fraction: float = 0.60
    early_eos_length_threshold: int = 3
    collapse_duplicate_rate_threshold: float = 0.50
    kl_collapse_threshold: float = 0.02
    unseen_condition_exclusion_warn_threshold: float = 0.05


@dataclass(frozen=True)
class RuntimeConfig:
    skip_completed_runs: bool = False
    no_figure_mode: bool = False
    no_generation_mode: bool = False
    run_only_model_tags: Tuple[str, ...] = tuple()
    run_only_seeds: Tuple[int, ...] = tuple()
    verbose: bool = True


@dataclass(frozen=True)
class Step6Config:
    step4_model_ready_csv: str = str(DEFAULT_STEP4_MODEL_READY_CSV)
    step4_artifacts_dir: str = str(DEFAULT_STEP4_ARTIFACTS_DIR)
    step5_root: str = str(DEFAULT_STEP5_ROOT)
    step6_root: str = str(DEFAULT_STEP6_ROOT)

    sequence_col: str = "sequence"
    split_col: str = "assigned_split"
    target_col: str = "primary_condition_id"

    train_label: str = "train"
    val_label: str = "val"
    test_label: str = "test"

    repeated_seeds: Tuple[int, ...] = (42, 52, 62)
    bootstrap_iterations: int = 400

    active_unit_variance_threshold: float = 1e-2
    active_unit_thresholds: Tuple[float, ...] = (1e-3, 1e-2, 5e-2)

    jaccard_k: int = 3
    nn_similarity_sample_limit: Optional[int] = 2000
    pairwise_diversity_pair_limit: int = 5000

    generated_eval_temperature_default: float = 1.0

    train_cfg: GeneratorTrainConfig = field(default_factory=GeneratorTrainConfig)
    evaluator_cfg: EvaluatorTrainConfig = field(default_factory=EvaluatorTrainConfig)
    ablation_cfg: AblationConfig = field(default_factory=AblationConfig)
    qa_cfg: QAConfig = field(default_factory=QAConfig)
    runtime_cfg: RuntimeConfig = field(default_factory=RuntimeConfig)
    fig_cfg: FigureExportConfig = field(default_factory=FigureExportConfig)


# =============================================================================
# Utilities
# =============================================================================
def print_header(msg: str):
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)


def seed_all(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def device_for_training() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def clean_sequence(seq: str) -> str:
    return "".join([a for a in str(seq).strip().upper() if a in AA_SET])


def contains_noncanonical(seq: str) -> bool:
    s = str(seq).strip().upper()
    return any(ch not in AA_SET for ch in s)


def is_valid_peptide(seq: str, min_len: int = 5, max_len: int = 60) -> bool:
    s = clean_sequence(seq)
    return min_len <= len(s) <= max_len and all(a in AA_SET for a in s)


def style_axis(ax, grid_axis: str = "y", grid_alpha: float = 0.85):
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PALETTE["grid"], linewidth=0.75, alpha=grid_alpha)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PALETTE["grid"], linewidth=0.75, alpha=grid_alpha)
    else:
        ax.grid(True, color=PALETTE["grid"], linewidth=0.75, alpha=grid_alpha)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_label(ax, label: str):
    ax.text(-0.12, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=16, fontweight="bold")


def ensure_dirs(root: Path) -> Dict[str, Path]:
    d = {
        k: root / k for k in [
            "tables",
            "figures_main",
            "figures_supplementary",
            "artifacts",
            "arrays",
            "logs",
            "reports",
            "models",
            "source_data_figures",
            "qa",
        ]
    }
    d["root"] = root
    for p in d.values():
        p.mkdir(parents=True, exist_ok=True)
    return d


def save_figure(fig, out_base: Path, fc: FigureExportConfig):
    out = {}
    if fc.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        out["pdf"] = str(p)
    if fc.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=fc.dpi_png)
        out["png"] = str(p)
    if fc.export_tiff:
        p = out_base.with_suffix(".tiff")
        fig.savefig(p, bbox_inches="tight", dpi=fc.dpi_tiff, format="tiff")
        out["tiff"] = str(p)
    plt.close(fig)
    return out


def export_csv(df: pd.DataFrame, out_path: Path) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return str(out_path)


def export_json(data: Any, out_path: Path) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=_json_default)
    return str(out_path)


def _json_default(obj: Any):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Unsupported JSON type: {type(obj)}")


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns:
            return c
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"Could not resolve any of columns: {candidates}")
    return None


def config_hash(cfg: Step6Config) -> str:
    return hashlib.sha256(json.dumps(asdict(cfg), sort_keys=True, default=_json_default).encode()).hexdigest()[:16]


def environment_snapshot() -> Dict[str, Any]:
    return {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
    }


def should_run_variant_seed(cfg: Step6Config, model_tag: str, seed: int) -> bool:
    if cfg.runtime_cfg.run_only_model_tags and model_tag not in cfg.runtime_cfg.run_only_model_tags:
        return False
    if cfg.runtime_cfg.run_only_seeds and seed not in cfg.runtime_cfg.run_only_seeds:
        return False
    return True


def safe_mean(x: Sequence[float]) -> float:
    x = [float(v) for v in x if pd.notnull(v)]
    return float(np.mean(x)) if len(x) else float("nan")


def bootstrap_mean_ci(vals: Sequence[float], iterations: int = 400, seed: int = 42, alpha: float = 0.05) -> Dict[str, float]:
    vals = np.array([float(v) for v in vals if pd.notnull(v)], dtype=float)
    if vals.size == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0}
    if vals.size == 1:
        v = float(vals[0])
        return {"mean": v, "ci_low": v, "ci_high": v, "n": 1}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(iterations):
        sample = rng.choice(vals, size=vals.size, replace=True)
        boots.append(np.mean(sample))
    boots = np.array(boots, dtype=float)
    return {
        "mean": float(np.mean(vals)),
        "ci_low": float(np.quantile(boots, alpha / 2)),
        "ci_high": float(np.quantile(boots, 1 - alpha / 2)),
        "n": int(vals.size),
    }


# =============================================================================
# Sequence features
# =============================================================================
def simple_charge_proxy(seq):
    seq = clean_sequence(seq)
    return sum(1 for a in seq if a in {"K", "R", "H"}) - sum(1 for a in seq if a in {"D", "E"})


def mean_hydropathy(seq):
    seq = clean_sequence(seq)
    return float(np.mean([KD[a] for a in seq])) if seq else float("nan")


def sequence_entropy(seq):
    seq = clean_sequence(seq)
    if not seq:
        return float("nan")
    c = pd.Series(list(seq)).value_counts(normalize=True).to_numpy(float)
    return float(-(c * np.log2(c)).sum())


def low_complexity_flags(seq, qa: QAConfig):
    seq = clean_sequence(seq)
    if not seq:
        return {
            "homopolymer_flag": 1,
            "low_unique_aa_flag": 1,
            "dominant_aa_flag": 1,
            "low_complexity_any_flag": 1,
        }
    max_run = 1
    cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 1
    h = int(max_run >= qa.low_complexity_repeat_threshold)
    u = int(len(set(seq)) < qa.min_unique_amino_acids)
    d = int(max(seq.count(a) for a in set(seq)) / len(seq) > qa.max_single_aa_fraction)
    return {
        "homopolymer_flag": h,
        "low_unique_aa_flag": u,
        "dominant_aa_flag": d,
        "low_complexity_any_flag": int(h or u or d),
    }


def kmer_set(seq, k=3):
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}


def jaccard_similarity(a, b, k=3):
    sa = kmer_set(a, k)
    sb = kmer_set(b, k)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def nearest_neighbor_similarity(seq, refs, k=3, sample_limit=None, seed=42):
    if not refs:
        return float("nan")
    if sample_limit is not None and len(refs) > sample_limit:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(refs), size=sample_limit, replace=False)
        refs_eff = [refs[i] for i in idx]
    else:
        refs_eff = refs
    return max(jaccard_similarity(seq, r, k=k) for r in refs_eff)


def pairwise_diversity_summary(seqs, k=3, pair_limit=5000, seed=42):
    seqs = [clean_sequence(s) for s in seqs if clean_sequence(s)]
    n = len(seqs)
    if n <= 1:
        return {"mean_pairwise_jaccard": np.nan, "mean_pairwise_diversity": np.nan}
    rng = np.random.default_rng(seed)
    pairs = []
    max_pairs = n * (n - 1) // 2
    if max_pairs <= pair_limit:
        for i in range(n):
            for j in range(i + 1, n):
                pairs.append((i, j))
    else:
        seen = set()
        while len(pairs) < pair_limit:
            i = int(rng.integers(0, n))
            j = int(rng.integers(0, n))
            if i >= j or (i, j) in seen:
                continue
            seen.add((i, j))
            pairs.append((i, j))
    sims = [jaccard_similarity(seqs[i], seqs[j], k=k) for i, j in pairs]
    return {
        "mean_pairwise_jaccard": float(np.mean(sims)),
        "mean_pairwise_diversity": float(1.0 - np.mean(sims)),
    }


def normalized_edit_distance(a, b):
    if a == b:
        return 0.0
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 0.0
    dp = np.zeros((la + 1, lb + 1), dtype=np.int32)
    dp[:, 0] = np.arange(la + 1)
    dp[0, :] = np.arange(lb + 1)
    for i in range(1, la + 1):
        ca = a[i - 1]
        for j in range(1, lb + 1):
            cb = b[j - 1]
            cost = 0 if ca == cb else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
    return float(dp[la, lb]) / max(la, lb, 1)


# =============================================================================
# Tokenizer / datasets
# =============================================================================
class PeptideTokenizer:
    def __init__(self, vocab_json: Optional[Path] = None, special_tokens_json: Optional[Path] = None):
        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"

        if special_tokens_json and special_tokens_json.exists():
            try:
                d = json.loads(special_tokens_json.read_text())
                self.pad_token = d.get("pad_token", self.pad_token)
                self.bos_token = d.get("bos_token", self.bos_token)
                self.eos_token = d.get("eos_token", self.eos_token)
            except Exception:
                pass

        if vocab_json and vocab_json.exists():
            try:
                d = json.loads(vocab_json.read_text())
                if isinstance(d, dict):
                    vocab = [k for k, _ in sorted(d.items(), key=lambda kv: kv[1])]
                else:
                    vocab = list(d)
                special = [self.pad_token, self.bos_token, self.eos_token]
                self.alphabet = special + [x for x in vocab if x not in special]
            except Exception:
                self.alphabet = [self.pad_token, self.bos_token, self.eos_token] + AA
        else:
            self.alphabet = [self.pad_token, self.bos_token, self.eos_token] + AA

        seen = set()
        clean = []
        for tok in self.alphabet:
            if tok not in seen:
                seen.add(tok)
                clean.append(tok)
        self.alphabet = clean
        self.stoi = {t: i for i, t in enumerate(self.alphabet)}
        self.itos = {i: t for t, i in self.stoi.items()}

    @property
    def pad_id(self):
        return self.stoi[self.pad_token]

    @property
    def bos_id(self):
        return self.stoi[self.bos_token]

    @property
    def eos_id(self):
        return self.stoi[self.eos_token]

    @property
    def vocab_size(self):
        return len(self.alphabet)

    def encode_unpadded(self, seq: str, max_len: int) -> List[int]:
        toks = [self.bos_id]
        for s in clean_sequence(seq)[: max_len - 2]:
            if s in self.stoi:
                toks.append(self.stoi[s])
        toks.append(self.eos_id)
        return toks[:max_len]

    def decode(self, ids: Sequence[int]) -> str:
        chars = []
        for i in ids:
            tok = self.itos[int(i)]
            if tok == self.eos_token:
                break
            if tok in {self.pad_token, self.bos_token}:
                continue
            chars.append(tok)
        return "".join(chars)


class SequenceDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_len, conditions=None):
        self.seqs = [clean_sequence(s) for s in sequences]
        self.tok = tokenizer
        self.max_len = max_len
        self.conditions = None if conditions is None else list(conditions)

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = self.tok.encode_unpadded(self.seqs[idx], self.max_len)
        x = ids[:-1]
        y = ids[1:]
        out = {
            "input_ids": torch.tensor(x, dtype=torch.long),
            "target_ids": torch.tensor(y, dtype=torch.long),
            "length": len(x),
            "sequence": self.seqs[idx],
        }
        if self.conditions is not None:
            out["condition_id"] = int(self.conditions[idx])
        return out


class ClassifierDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_len, labels):
        self.seqs = [clean_sequence(s) for s in sequences]
        self.tok = tokenizer
        self.max_len = max_len
        self.labels = list(labels)

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = self.tok.encode_unpadded(self.seqs[idx], self.max_len)
        return {
            "input_ids": torch.tensor(ids[:-1], dtype=torch.long),
            "length": len(ids[:-1]),
            "sequence": self.seqs[idx],
            "label": int(self.labels[idx]),
        }


def collate_sequence_batch(batch, pad_id: int):
    lengths = torch.tensor([b["length"] for b in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    bs = len(batch)

    input_ids = torch.full((bs, max_len), pad_id, dtype=torch.long)
    seqs = []

    for i, b in enumerate(batch):
        n = b["length"]
        input_ids[i, :n] = b["input_ids"]
        seqs.append(b["sequence"])

    out = {
        "input_ids": input_ids,
        "lengths": lengths,
        "sequence": seqs,
    }

    if "target_ids" in batch[0]:
        target_ids = torch.full((bs, max_len), pad_id, dtype=torch.long)
        for i, b in enumerate(batch):
            n = b["length"]
            target_ids[i, :n] = b["target_ids"]
        out["target_ids"] = target_ids

    if "condition_id" in batch[0]:
        out["condition_id"] = torch.tensor([int(b["condition_id"]) for b in batch], dtype=torch.long)

    if "label" in batch[0]:
        out["label"] = torch.tensor([int(b["label"]) for b in batch], dtype=torch.long)

    return out


def make_dataloader(ds, batch_size, shuffle, num_workers, seed, pad_id):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        generator=g,
        collate_fn=lambda b: collate_sequence_batch(b, pad_id),
    )


# =============================================================================
# Models
# =============================================================================
def token_dropout(input_ids: torch.Tensor, lengths: torch.Tensor, pad_id: int, bos_id: int, eos_id: int, p: float) -> torch.Tensor:
    if p <= 0.0:
        return input_ids
    keep = torch.ones_like(input_ids, dtype=torch.bool)
    rand_mask = torch.rand_like(input_ids.float()) < p
    special = input_ids.eq(pad_id) | input_ids.eq(bos_id) | input_ids.eq(eos_id)
    keep = ~(rand_mask & ~special)
    out = input_ids.clone()
    out[~keep] = pad_id
    return out


class ConditionalSequenceVAE(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        cond_vocab_size: int,
        emb_dim: int,
        enc_hidden_dim: int,
        dec_hidden_dim: int,
        latent_dim: int,
        num_layers: int,
        dropout: float = 0.0,
        use_condition: bool = True,
        aux_condition_head: bool = True,
    ):
        super().__init__()
        self.use_condition = use_condition
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        self.emb = nn.Embedding(vocab_size, emb_dim)

        self.cond_emb_enc = nn.Embedding(cond_vocab_size, emb_dim) if use_condition else None
        self.cond_emb_dec = nn.Embedding(cond_vocab_size, dec_hidden_dim) if use_condition else None

        self.encoder = nn.GRU(
            emb_dim,
            enc_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.to_mu = nn.Linear(enc_hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(enc_hidden_dim, latent_dim)

        self.z_to_hidden = nn.Linear(latent_dim, dec_hidden_dim)

        self.decoder = nn.GRU(
            emb_dim,
            dec_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(dec_hidden_dim, vocab_size)
        self.aux_head = nn.Linear(latent_dim, cond_vocab_size) if aux_condition_head else None

    def encode(self, input_ids, lengths, condition_id=None):
        emb = self.emb(input_ids)
        if self.use_condition and condition_id is not None:
            emb = emb + self.cond_emb_enc(condition_id).unsqueeze(1)

        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.encoder(packed)
        h = h[-1]
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        return z, mu, logvar

    def _decoder_hidden(self, z, condition_id=None):
        h0 = self.z_to_hidden(z)
        if self.use_condition and condition_id is not None:
            h0 = h0 + self.cond_emb_dec(condition_id)
        return h0.unsqueeze(0).repeat(self.num_layers, 1, 1)

    def decode_teacher_forced(self, input_ids, z, condition_id=None):
        dec, _ = self.decoder(self.emb(input_ids), self._decoder_hidden(z, condition_id))
        return self.out(dec)

    def forward(self, input_ids, lengths, condition_id=None):
        z, mu, logvar = self.encode(input_ids, lengths, condition_id)
        logits = self.decode_teacher_forced(input_ids, z, condition_id)
        aux_logits = self.aux_head(mu) if self.aux_head is not None else None
        return logits, mu, logvar, aux_logits

    def greedy_decode(self, z, bos_id, eos_id, pad_id, max_decode_len, condition_id=None):
        device = z.device
        bs = z.size(0)
        h = self._decoder_hidden(z, condition_id)
        cur = torch.full((bs, 1), bos_id, dtype=torch.long, device=device)
        generated = []
        finished = torch.zeros(bs, dtype=torch.bool, device=device)

        for _ in range(max_decode_len):
            dec, h = self.decoder(self.emb(cur[:, -1:]), h)
            nxt = self.out(dec[:, -1, :]).argmax(dim=-1)
            generated.append(nxt)
            cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            finished = finished | nxt.eq(eos_id)
            if finished.all():
                break

        if not generated:
            return torch.full((bs, 1), eos_id, dtype=torch.long, device=device)

        return _mask_after_eos(torch.stack(generated, dim=1), eos_id, pad_id)

    def sample_sequences(self, n, bos_id, eos_id, pad_id, max_decode_len, temperature=1.0, top_k=None, nucleus_p=None, condition_id=None, device=None):
        device = device or next(self.parameters()).device
        z = torch.randn(n, self.latent_dim, device=device)
        h = self._decoder_hidden(z, condition_id)
        cur = torch.full((n, 1), bos_id, dtype=torch.long, device=device)
        generated = []
        finished = torch.zeros(n, dtype=torch.bool, device=device)

        for _ in range(max_decode_len):
            dec, h = self.decoder(self.emb(cur[:, -1:]), h)
            logits = apply_sampling_constraints(self.out(dec[:, -1, :]) / max(temperature, 1e-6), top_k, nucleus_p)
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).squeeze(1)
            generated.append(nxt)
            cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            finished = finished | nxt.eq(eos_id)
            if finished.all():
                break

        if not generated:
            return torch.full((n, 1), eos_id, dtype=torch.long, device=device)

        return _mask_after_eos(torch.stack(generated, dim=1), eos_id, pad_id)


class ConditionalGRULM(nn.Module):
    def __init__(self, vocab_size, cond_vocab_size, emb_dim, hidden_dim, num_layers, dropout=0.0, use_condition=True):
        super().__init__()
        self.use_condition = use_condition
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cond_emb = nn.Embedding(cond_vocab_size, hidden_dim) if use_condition else None
        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.num_layers = num_layers

    def forward(self, input_ids, condition_id=None):
        if self.use_condition and condition_id is not None:
            h0 = self.cond_emb(condition_id).unsqueeze(0).repeat(self.num_layers, 1, 1)
        else:
            h0 = None
        out, _ = self.rnn(self.emb(input_ids), h0)
        return self.out(out)

    def sample_sequences(self, n, bos_id, eos_id, pad_id, max_decode_len, condition_id=None, temperature=1.0, top_k=None, nucleus_p=None, device=None):
        device = device or next(self.parameters()).device
        if self.use_condition and condition_id is not None:
            h = self.cond_emb(condition_id.to(device)).unsqueeze(0).repeat(self.num_layers, 1, 1)
        else:
            h = None
        cur = torch.full((n, 1), bos_id, dtype=torch.long, device=device)
        generated = []
        finished = torch.zeros(n, dtype=torch.bool, device=device)

        for _ in range(max_decode_len):
            dec, h = self.rnn(self.emb(cur[:, -1:]), h)
            logits = apply_sampling_constraints(self.out(dec[:, -1, :]) / max(temperature, 1e-6), top_k, nucleus_p)
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).squeeze(1)
            generated.append(nxt)
            cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            finished = finished | nxt.eq(eos_id)
            if finished.all():
                break

        if not generated:
            return torch.full((n, 1), eos_id, dtype=torch.long, device=device)

        return _mask_after_eos(torch.stack(generated, dim=1), eos_id, pad_id)


class SequenceConditionClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim, hidden_dim, num_layers, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.encoder = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, n_classes)

    def forward(self, input_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(input_ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.encoder(packed)
        return self.out(h[-1])


# =============================================================================
# Model helpers
# =============================================================================
def _mask_after_eos(out, eos_id, pad_id):
    out = out.clone()
    for i in range(out.size(0)):
        pos = (out[i] == eos_id).nonzero(as_tuple=False)
        if len(pos) > 0:
            first = int(pos[0].item())
            if first + 1 < out.size(1):
                out[i, first + 1:] = pad_id
    return out


def apply_sampling_constraints(logits, top_k=None, nucleus_p=None):
    logits = logits.clone()
    if top_k is not None and top_k > 0:
        vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        thresh = vals[:, -1].unsqueeze(-1)
        logits[logits < thresh] = -float("inf")
    if nucleus_p is not None and 0 < nucleus_p < 1:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = torch.softmax(sorted_logits, dim=-1)
        cum = torch.cumsum(probs, dim=-1)
        remove = cum > nucleus_p
        remove[:, 0] = False
        sorted_logits[remove] = -float("inf")
        new_logits = torch.full_like(logits, -float("inf"))
        new_logits.scatter_(1, sorted_idx, sorted_logits)
        logits = new_logits
    return logits


def cross_entropy_ignore_pad(logits, target_ids, pad_id):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1), ignore_index=pad_id)


def token_accuracy_ignore_pad(logits, target_ids, pad_id):
    pred = logits.argmax(dim=-1)
    mask = target_ids.ne(pad_id)
    denom = int(mask.sum().item())
    if denom == 0:
        return float("nan")
    return int((pred.eq(target_ids) & mask).sum().item()) / denom


def classification_accuracy(logits, labels):
    return float((logits.argmax(dim=-1).eq(labels)).float().mean().item())


def kl_per_dim_from_mu_logvar(mu, logvar):
    # shape: [batch, latent_dim]
    return -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())


def free_bits_kl(mu, logvar, free_bits_per_dim: float):
    kl_dim = kl_per_dim_from_mu_logvar(mu, logvar)  # [B, D]
    mean_per_dim = kl_dim.mean(dim=0)
    clipped = torch.clamp(mean_per_dim, min=free_bits_per_dim)
    return clipped.sum(), mean_per_dim


def raw_kl(mu, logvar):
    kl = kl_per_dim_from_mu_logvar(mu, logvar)
    return kl.sum(dim=1).mean(), kl.mean(dim=0)


def beta_for_epoch(epoch_idx: int, cfg: GeneratorTrainConfig) -> float:
    if cfg.kl_schedule == "constant":
        return cfg.beta_max
    if cfg.kl_schedule == "linear":
        warm = max(1, int(round(cfg.epochs * cfg.beta_warmup_fraction)))
        if epoch_idx >= warm:
            return cfg.beta_max
        return cfg.beta_max * float(epoch_idx + 1) / float(warm)
    if cfg.kl_schedule == "cyclical":
        cycle_len = max(2, cfg.kl_cycle_length)
        pos = epoch_idx % cycle_len
        warm = max(1, int(round(cycle_len * cfg.beta_warmup_fraction)))
        if pos >= warm:
            return cfg.beta_max
        return cfg.beta_max * float(pos + 1) / float(warm)
    raise ValueError(f"Unknown KL schedule: {cfg.kl_schedule}")


# =============================================================================
# Data loading
# =============================================================================
def load_tokenizer_from_artifacts(step4_artifacts_dir: Path) -> PeptideTokenizer:
    vocab_candidates = [
        step4_artifacts_dir / "vocab.json",
        step4_artifacts_dir / "tokenizer_vocab.json",
        step4_artifacts_dir / "vocabulary.json",
    ]
    special_candidates = [
        step4_artifacts_dir / "special_tokens.json",
        step4_artifacts_dir / "tokenizer_special_tokens.json",
    ]

    vocab_json = next((p for p in vocab_candidates if p.exists()), None)
    special_json = next((p for p in special_candidates if p.exists()), None)
    return PeptideTokenizer(vocab_json=vocab_json, special_tokens_json=special_json)


def infer_condition_column(df: pd.DataFrame, preferred: str) -> str:
    return resolve_column(
        df,
        [preferred, "primary_condition_id", "condition_id", "condition", "target_condition", "conditioning_id"],
        required=True,
    )


def normalize_split_labels(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    s = s.replace({
        "validation": "val",
        "valid": "val",
        "dev": "val",
        "training": "train",
        "testing": "test",
    })
    return s


def load_step4_inputs(cfg: Step6Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    csv_path = Path(cfg.step4_model_ready_csv)
    artifacts_dir = Path(cfg.step4_artifacts_dir)

    if not csv_path.exists():
        raise FileNotFoundError(f"Step 4 model-ready CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    seq_col = resolve_column(df, [cfg.sequence_col, "sequence", "peptide_sequence", "clean_sequence"])
    split_col = resolve_column(df, [cfg.split_col, "assigned_split", "split", "data_split"])
    target_col = infer_condition_column(df, cfg.target_col)

    work = df.copy()
    work["sequence_raw"] = work[seq_col].astype(str)
    work["sequence"] = work["sequence_raw"].map(clean_sequence)
    work["split"] = normalize_split_labels(work[split_col])
    work["condition_raw"] = work[target_col].astype(str)

    work["contains_noncanonical_raw"] = work["sequence_raw"].map(contains_noncanonical).astype(int)
    work["valid_after_clean"] = work["sequence"].map(lambda x: is_valid_peptide(x, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)).astype(int)

    removed_df = work.loc[work["valid_after_clean"] == 0].copy()
    work = work.loc[work["valid_after_clean"] == 1].copy()

    allowed_splits = {cfg.train_label, cfg.val_label, cfg.test_label}
    unknown_splits = sorted(set(work["split"].unique()) - allowed_splits)
    if unknown_splits:
        raise ValueError(f"Unexpected split labels found: {unknown_splits}")

    split_counts = work["split"].value_counts().to_dict()
    for req in [cfg.train_label, cfg.val_label, cfg.test_label]:
        if split_counts.get(req, 0) == 0:
            raise ValueError(f"Empty required split detected: {req}")

    train_conditions = set(work.loc[work["split"] == cfg.train_label, "condition_raw"].astype(str))
    work["condition_seen_in_train"] = work["condition_raw"].isin(train_conditions).astype(int)

    unseen_eval = work.loc[
        (work["split"].isin([cfg.val_label, cfg.test_label])) &
        (work["condition_seen_in_train"] == 0)
    ].copy()

    unseen_rate = float(len(unseen_eval) / max(1, len(work.loc[work["split"].isin([cfg.val_label, cfg.test_label])])))
    if unseen_rate > cfg.qa_cfg.unseen_condition_exclusion_warn_threshold:
        warnings.warn(
            f"Unseen validation/test conditions exceed threshold: {unseen_rate:.3f}. "
            "They will be excluded from condition-dependent evaluation."
        )

    kept = work.loc[
        (work["split"] == cfg.train_label) |
        ((work["split"].isin([cfg.val_label, cfg.test_label])) & (work["condition_seen_in_train"] == 1))
    ].copy()

    condition_values = sorted(set(kept.loc[kept["split"] == cfg.train_label, "condition_raw"].astype(str)))
    condition_to_id = {c: i for i, c in enumerate(condition_values)}
    kept["condition_id"] = kept["condition_raw"].map(condition_to_id).astype(int)

    train_df = kept.loc[kept["split"] == cfg.train_label].copy().reset_index(drop=True)
    val_df = kept.loc[kept["split"] == cfg.val_label].copy().reset_index(drop=True)
    test_df = kept.loc[kept["split"] == cfg.test_label].copy().reset_index(drop=True)

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError("At least one split became empty after train-condition support filtering.")

    tokenizer = load_tokenizer_from_artifacts(artifacts_dir)

    split_summary = pd.DataFrame([
        {"split": "removed_invalid", "n": len(removed_df), "n_unique_sequences": removed_df["sequence"].nunique() if len(removed_df) else 0, "n_conditions": removed_df["condition_raw"].nunique() if len(removed_df) else 0},
        {"split": "unseen_eval_excluded", "n": len(unseen_eval), "n_unique_sequences": unseen_eval["sequence"].nunique() if len(unseen_eval) else 0, "n_conditions": unseen_eval["condition_raw"].nunique() if len(unseen_eval) else 0},
        {"split": cfg.train_label, "n": len(train_df), "n_unique_sequences": train_df["sequence"].nunique(), "n_conditions": train_df["condition_raw"].nunique()},
        {"split": cfg.val_label, "n": len(val_df), "n_unique_sequences": val_df["sequence"].nunique(), "n_conditions": val_df["condition_raw"].nunique()},
        {"split": cfg.test_label, "n": len(test_df), "n_unique_sequences": test_df["sequence"].nunique(), "n_conditions": test_df["condition_raw"].nunique()},
    ])

    condition_support = (
        kept.groupby(["split", "condition_raw"], as_index=False)
        .size()
        .rename(columns={"size": "n_sequences"})
        .sort_values(["split", "n_sequences", "condition_raw"], ascending=[True, False, True])
        .reset_index(drop=True)
    )
    condition_support["condition_id"] = condition_support["condition_raw"].map(condition_to_id)

    export_csv(split_summary, dirs["tables"] / "step6_input_split_summary.csv")
    export_csv(condition_support, dirs["tables"] / "step6_condition_support_summary.csv")
    export_csv(removed_df[["sequence_raw", "sequence", "split", "condition_raw", "contains_noncanonical_raw", "valid_after_clean"]], dirs["qa"] / "step6_removed_invalid_sequences.csv")
    export_csv(unseen_eval[["sequence", "split", "condition_raw", "condition_seen_in_train"]], dirs["qa"] / "step6_unseen_eval_conditions_excluded.csv")

    return {
        "seq_col": seq_col,
        "split_col": split_col,
        "target_col": target_col,
        "condition_to_id": condition_to_id,
        "id_to_condition": {v: k for k, v in condition_to_id.items()},
        "tokenizer": tokenizer,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "kept_df": kept,
        "removed_df": removed_df,
        "unseen_eval_df": unseen_eval,
        "split_summary": split_summary,
        "condition_support": condition_support,
        "step4_csv_path": str(csv_path),
        "step4_artifacts_dir": str(artifacts_dir),
    }


def build_sequence_dataloaders(df_train, df_val, df_test, tokenizer, cfg: Step6Config, seed: int, with_condition: bool) -> Dict[str, DataLoader]:
    train_ds = SequenceDataset(df_train["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_train["condition_id"].tolist() if with_condition else None)
    val_ds = SequenceDataset(df_val["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_val["condition_id"].tolist() if with_condition else None)
    test_ds = SequenceDataset(df_test["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_test["condition_id"].tolist() if with_condition else None)

    return {
        "train": make_dataloader(train_ds, cfg.train_cfg.batch_size, True, cfg.train_cfg.num_workers, seed, tokenizer.pad_id),
        "val": make_dataloader(val_ds, cfg.train_cfg.batch_size, False, cfg.train_cfg.num_workers, seed, tokenizer.pad_id),
        "test": make_dataloader(test_ds, cfg.train_cfg.batch_size, False, cfg.train_cfg.num_workers, seed, tokenizer.pad_id),
    }


def build_classifier_dataloaders(df_train, df_val, df_test, tokenizer, cfg: Step6Config, seed: int) -> Dict[str, DataLoader]:
    train_ds = ClassifierDataset(df_train["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_train["condition_id"].tolist())
    val_ds = ClassifierDataset(df_val["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_val["condition_id"].tolist())
    test_ds = ClassifierDataset(df_test["sequence"].tolist(), tokenizer, cfg.train_cfg.max_seq_len, df_test["condition_id"].tolist())
    return {
        "train": make_dataloader(train_ds, cfg.evaluator_cfg.batch_size, True, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id),
        "val": make_dataloader(val_ds, cfg.evaluator_cfg.batch_size, False, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id),
        "test": make_dataloader(test_ds, cfg.evaluator_cfg.batch_size, False, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id),
    }


# =============================================================================
# Condition evaluator
# =============================================================================
@torch.no_grad()
def evaluate_condition_classifier(model, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    loss_sum, acc_sum, n = 0.0, 0.0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        labels = batch["label"].to(device)
        logits = model(input_ids, lengths)
        loss = F.cross_entropy(logits, labels)

        bs = labels.size(0)
        loss_sum += float(loss.item()) * bs
        acc_sum += classification_accuracy(logits, labels) * bs
        n += bs

    return {
        "loss": loss_sum / max(1, n),
        "acc": acc_sum / max(1, n),
        "n": n,
    }


def train_condition_classifier(df_train, df_val, df_test, tokenizer, n_classes: int, cfg: Step6Config, seed: int, device: torch.device, dirs: Dict[str, Path]) -> Dict[str, Any]:
    seed_all(seed)
    loaders = build_classifier_dataloaders(df_train, df_val, df_test, tokenizer, cfg, seed)

    model = SequenceConditionClassifier(
        vocab_size=tokenizer.vocab_size,
        n_classes=n_classes,
        emb_dim=cfg.evaluator_cfg.emb_dim,
        hidden_dim=cfg.evaluator_cfg.hidden_dim,
        num_layers=cfg.evaluator_cfg.num_layers,
        dropout=cfg.evaluator_cfg.dropout,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.evaluator_cfg.lr, weight_decay=cfg.evaluator_cfg.weight_decay)

    best_state = None
    best_metric = float("inf")
    patience = 0
    hist = []

    for epoch in range(cfg.evaluator_cfg.epochs):
        model.train()
        train_loss_sum, train_acc_sum, n = 0.0, 0.0, 0

        for batch in loaders["train"]:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.evaluator_cfg.clip_grad_norm)
            optimizer.step()

            bs = labels.size(0)
            train_loss_sum += float(loss.item()) * bs
            train_acc_sum += classification_accuracy(logits, labels) * bs
            n += bs

        train_loss = train_loss_sum / max(1, n)
        train_acc = train_acc_sum / max(1, n)
        val_metrics = evaluate_condition_classifier(model, loaders["val"], device)

        hist.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        })

        if val_metrics["loss"] < best_metric:
            best_metric = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= cfg.evaluator_cfg.early_stopping_patience:
                break

    if best_state is None:
        raise RuntimeError("Condition evaluator failed to produce a checkpoint.")

    model.load_state_dict(best_state)

    val_final = evaluate_condition_classifier(model, loaders["val"], device)
    test_final = evaluate_condition_classifier(model, loaders["test"], device)

    ckpt_path = dirs["models"] / f"step6_condition_evaluator_seed{seed}.pt"
    torch.save({
        "seed": seed,
        "state_dict": model.state_dict(),
        "config": asdict(cfg.evaluator_cfg),
        "n_classes": n_classes,
        "tokenizer_vocab_size": tokenizer.vocab_size,
    }, ckpt_path)

    hist_df = pd.DataFrame(hist)
    export_csv(hist_df, dirs["logs"] / f"step6_condition_evaluator_history_seed{seed}.csv")

    return {
        "model": model,
        "checkpoint_path": str(ckpt_path),
        "history_df": hist_df,
        "val_metrics": val_final,
        "test_metrics": test_final,
    }


@torch.no_grad()
def predict_conditions_for_sequences(model, sequences: Sequence[str], tokenizer: PeptideTokenizer, cfg: Step6Config, seed: int, device: torch.device) -> np.ndarray:
    ds = ClassifierDataset(sequences, tokenizer, cfg.train_cfg.max_seq_len, [0] * len(sequences))
    loader = make_dataloader(ds, cfg.evaluator_cfg.batch_size, False, cfg.evaluator_cfg.num_workers, seed, tokenizer.pad_id)
    preds = []
    model.eval()
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        logits = model(input_ids, lengths)
        preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


# =============================================================================
# Training / evaluation
# =============================================================================
def make_model_for_variant(model_tag: str, tokenizer: PeptideTokenizer, n_conditions: int, cfg: Step6Config):
    tc = cfg.train_cfg
    if model_tag == "full_cvae":
        return ConditionalSequenceVAE(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=tc.emb_dim,
            enc_hidden_dim=tc.enc_hidden_dim,
            dec_hidden_dim=tc.dec_hidden_dim,
            latent_dim=tc.latent_dim,
            num_layers=tc.num_layers,
            dropout=tc.dropout,
            use_condition=True,
            aux_condition_head=True,
        ), "cvae", True
    if model_tag == "small_latent":
        return ConditionalSequenceVAE(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=tc.emb_dim,
            enc_hidden_dim=tc.enc_hidden_dim,
            dec_hidden_dim=tc.dec_hidden_dim,
            latent_dim=cfg.ablation_cfg.small_latent_dim,
            num_layers=tc.num_layers,
            dropout=tc.dropout,
            use_condition=False,
            aux_condition_head=False,
        ), "cvae", False
    if model_tag == "conditional_gru_lm":
        return ConditionalGRULM(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=tc.emb_dim,
            hidden_dim=tc.dec_hidden_dim,
            num_layers=tc.num_layers,
            dropout=tc.dropout,
            use_condition=True,
        ), "gru", True
    if model_tag == "no_condition":
        return ConditionalGRULM(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=tc.emb_dim,
            hidden_dim=tc.dec_hidden_dim,
            num_layers=tc.num_layers,
            dropout=tc.dropout,
            use_condition=False,
        ), "gru", False
    if model_tag == "random_condition":
        return ConditionalGRULM(
            vocab_size=tokenizer.vocab_size,
            cond_vocab_size=n_conditions,
            emb_dim=tc.emb_dim,
            hidden_dim=tc.dec_hidden_dim,
            num_layers=tc.num_layers,
            dropout=tc.dropout,
            use_condition=True,
        ), "gru_random_condition", True
    raise ValueError(f"Unsupported model tag: {model_tag}")


def variant_enabled(cfg: Step6Config, model_tag: str) -> bool:
    ab = cfg.ablation_cfg
    mapping = {
        "retrieval": ab.run_retrieval_baseline,
        "full_cvae": ab.run_full_cvae,
        "conditional_gru_lm": ab.run_conditional_gru_lm,
        "random_condition": ab.run_random_condition,
        "no_condition": ab.run_no_condition,
        "small_latent": ab.run_small_latent,
    }
    return bool(mapping.get(model_tag, False))


def get_done_flag(dirs: Dict[str, Path], model_tag: str, seed: int) -> Path:
    return dirs["artifacts"] / f"step6_done_{model_tag}_seed{seed}.json"


def active_units_from_mu(mu_np: np.ndarray, thresholds: Sequence[float]) -> Dict[float, int]:
    if mu_np.size == 0:
        return {float(th): 0 for th in thresholds}
    var_per_dim = np.var(mu_np, axis=0)
    return {float(th): int(np.sum(var_per_dim > th)) for th in thresholds}


def train_epoch_generator(model, loader, optimizer, tokenizer, cfg: Step6Config, device, family: str, use_condition: bool, randomize_condition: bool = False):
    tc = cfg.train_cfg
    model.train()

    loss_sum = recon_sum = raw_kl_sum = free_kl_sum = aux_sum = acc_sum = 0.0
    n = 0

    mu_collect = []
    kl_dim_collect = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)
        lengths = batch["lengths"].to(device)
        condition_id = batch["condition_id"].to(device) if "condition_id" in batch else None

        if randomize_condition and condition_id is not None:
            perm = torch.randperm(condition_id.size(0), device=condition_id.device)
            condition_id = condition_id[perm]

        input_ids_drop = token_dropout(
            input_ids=input_ids,
            lengths=lengths,
            pad_id=tokenizer.pad_id,
            bos_id=tokenizer.bos_id,
            eos_id=tokenizer.eos_id,
            p=tc.decoder_input_dropout,
        )

        optimizer.zero_grad(set_to_none=True)
        beta = beta_for_epoch(train_epoch_generator.current_epoch, tc)

        if family == "cvae":
            logits, mu, logvar, aux_logits = model(input_ids_drop, lengths, condition_id=condition_id if use_condition else None)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)

            raw_kl_value, kl_mean_dim = raw_kl(mu, logvar)
            free_kl_value, kl_mean_dim_free = free_bits_kl(mu, logvar, tc.free_bits_per_dim)

            aux_loss = F.cross_entropy(aux_logits, condition_id) if (aux_logits is not None and use_condition and condition_id is not None) else torch.tensor(0.0, device=device)

            loss = recon + beta * free_kl_value + tc.aux_condition_loss_weight * aux_loss

            mu_collect.append(mu.detach().cpu().numpy())
            kl_dim_collect.append(kl_mean_dim.detach().cpu().numpy())

            raw_kl_num = float(raw_kl_value.item())
            free_kl_num = float(free_kl_value.item())
            aux_num = float(aux_loss.item())
        else:
            logits = model(input_ids_drop, condition_id=condition_id if use_condition else None)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)
            raw_kl_num = 0.0
            free_kl_num = 0.0
            aux_num = 0.0
            loss = recon

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss encountered.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), tc.clip_grad_norm)
        optimizer.step()

        bs = input_ids.size(0)
        acc = token_accuracy_ignore_pad(logits, target_ids, tokenizer.pad_id)

        loss_sum += float(loss.item()) * bs
        recon_sum += float(recon.item()) * bs
        raw_kl_sum += raw_kl_num * bs
        free_kl_sum += free_kl_num * bs
        aux_sum += aux_num * bs
        acc_sum += float(acc) * bs if pd.notnull(acc) else 0.0
        n += bs

    if len(mu_collect):
        mu_np = np.concatenate(mu_collect, axis=0)
        var_per_dim = np.var(mu_np, axis=0)
        active_units = active_units_from_mu(mu_np, cfg.active_unit_thresholds)
        mean_latent_var = float(np.mean(var_per_dim))
        mean_mu_abs = float(np.mean(np.abs(mu_np)))
    else:
        var_per_dim = np.array([])
        active_units = {float(th): np.nan for th in cfg.active_unit_thresholds}
        mean_latent_var = np.nan
        mean_mu_abs = np.nan

    if len(kl_dim_collect):
        kl_dim_np = np.stack(kl_dim_collect, axis=0).mean(axis=0)
        mean_kl_per_dim = float(np.mean(kl_dim_np))
    else:
        kl_dim_np = np.array([])
        mean_kl_per_dim = np.nan

    out = {
        "loss": loss_sum / max(1, n),
        "recon_loss": recon_sum / max(1, n),
        "raw_kl": raw_kl_sum / max(1, n),
        "free_kl": free_kl_sum / max(1, n),
        "aux_condition_loss": aux_sum / max(1, n),
        "token_acc": acc_sum / max(1, n),
        "active_units_main": active_units.get(cfg.active_unit_variance_threshold, np.nan),
        "mean_latent_var": mean_latent_var,
        "mean_mu_abs": mean_mu_abs,
        "mean_kl_per_dim": mean_kl_per_dim,
        "beta": beta,
        "n": n,
    }

    for th, au in active_units.items():
        out[f"active_units_thr_{th:g}"] = au
    if kl_dim_np.size:
        for i, v in enumerate(kl_dim_np):
            out[f"kl_dim_{i:02d}"] = float(v)
    return out


train_epoch_generator.current_epoch = 0


@torch.no_grad()
def evaluate_generator_loader(model, loader, tokenizer, cfg: Step6Config, device, family: str, use_condition: bool, randomize_condition: bool = False):
    tc = cfg.train_cfg
    model.eval()

    loss_sum = recon_sum = raw_kl_sum = free_kl_sum = aux_sum = acc_sum = 0.0
    n = 0

    mu_collect = []
    kl_dim_collect = []

    beta = beta_for_epoch(train_epoch_generator.current_epoch, tc)

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)
        lengths = batch["lengths"].to(device)
        condition_id = batch["condition_id"].to(device) if "condition_id" in batch else None

        if randomize_condition and condition_id is not None:
            perm = torch.randperm(condition_id.size(0), device=condition_id.device)
            condition_id = condition_id[perm]

        if family == "cvae":
            logits, mu, logvar, aux_logits = model(input_ids, lengths, condition_id=condition_id if use_condition else None)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)

            raw_kl_value, kl_mean_dim = raw_kl(mu, logvar)
            free_kl_value, kl_mean_dim_free = free_bits_kl(mu, logvar, tc.free_bits_per_dim)
            aux_loss = F.cross_entropy(aux_logits, condition_id) if (aux_logits is not None and use_condition and condition_id is not None) else torch.tensor(0.0, device=device)

            loss = recon + beta * free_kl_value + tc.aux_condition_loss_weight * aux_loss

            mu_collect.append(mu.detach().cpu().numpy())
            kl_dim_collect.append(kl_mean_dim.detach().cpu().numpy())

            raw_kl_num = float(raw_kl_value.item())
            free_kl_num = float(free_kl_value.item())
            aux_num = float(aux_loss.item())
        else:
            logits = model(input_ids, condition_id=condition_id if use_condition else None)
            recon = cross_entropy_ignore_pad(logits, target_ids, tokenizer.pad_id)
            raw_kl_num = 0.0
            free_kl_num = 0.0
            aux_num = 0.0
            loss = recon

        bs = input_ids.size(0)
        acc = token_accuracy_ignore_pad(logits, target_ids, tokenizer.pad_id)

        loss_sum += float(loss.item()) * bs
        recon_sum += float(recon.item()) * bs
        raw_kl_sum += raw_kl_num * bs
        free_kl_sum += free_kl_num * bs
        aux_sum += aux_num * bs
        acc_sum += float(acc) * bs if pd.notnull(acc) else 0.0
        n += bs

    if len(mu_collect):
        mu_np = np.concatenate(mu_collect, axis=0)
        var_per_dim = np.var(mu_np, axis=0)
        active_units = active_units_from_mu(mu_np, cfg.active_unit_thresholds)
        mean_latent_var = float(np.mean(var_per_dim))
        mean_mu_abs = float(np.mean(np.abs(mu_np)))
    else:
        active_units = {float(th): np.nan for th in cfg.active_unit_thresholds}
        mean_latent_var = np.nan
        mean_mu_abs = np.nan

    if len(kl_dim_collect):
        kl_dim_np = np.stack(kl_dim_collect, axis=0).mean(axis=0)
        mean_kl_per_dim = float(np.mean(kl_dim_np))
    else:
        kl_dim_np = np.array([])
        mean_kl_per_dim = np.nan

    out = {
        "loss": loss_sum / max(1, n),
        "recon_loss": recon_sum / max(1, n),
        "raw_kl": raw_kl_sum / max(1, n),
        "free_kl": free_kl_sum / max(1, n),
        "aux_condition_loss": aux_sum / max(1, n),
        "token_acc": acc_sum / max(1, n),
        "active_units_main": active_units.get(cfg.active_unit_variance_threshold, np.nan),
        "mean_latent_var": mean_latent_var,
        "mean_mu_abs": mean_mu_abs,
        "mean_kl_per_dim": mean_kl_per_dim,
        "beta": beta,
        "n": n,
    }
    for th, au in active_units.items():
        out[f"active_units_thr_{th:g}"] = au
    if kl_dim_np.size:
        for i, v in enumerate(kl_dim_np):
            out[f"kl_dim_{i:02d}"] = float(v)
    return out


@torch.no_grad()
def reconstruct_subset_sequences(model, family, loader, tokenizer, seed, device, use_condition, randomize_condition=False, sample_limit=500):
    model.eval()
    rng = np.random.default_rng(seed)
    rows = []
    seen = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        target_ids = batch["target_ids"].to(device)
        cond = batch["condition_id"].to(device) if use_condition and "condition_id" in batch else None
        if randomize_condition and cond is not None:
            perm = torch.randperm(cond.size(0), device=cond.device)
            cond = cond[perm]
        seq_true = batch["sequence"]

        if family == "cvae":
            z, mu, logvar = model.encode(input_ids, lengths, condition_id=cond)
            decoded = model.greedy_decode(
                z=mu,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=target_ids.size(1),
                condition_id=cond,
            )
            seq_pred = [tokenizer.decode(row.cpu().tolist()) for row in decoded]
        else:
            logits = model(input_ids, condition_id=cond if use_condition else None)
            pred = logits.argmax(dim=-1)
            seq_pred = [tokenizer.decode(row.cpu().tolist()) for row in pred]

        idx = np.arange(len(seq_true))
        rng.shuffle(idx)
        for i in idx:
            rows.append({
                "sequence_true": clean_sequence(seq_true[i]),
                "sequence_recon": clean_sequence(seq_pred[i]),
                "normalized_edit_distance": normalized_edit_distance(clean_sequence(seq_true[i]), clean_sequence(seq_pred[i])),
                "exact_reconstruction": int(clean_sequence(seq_true[i]) == clean_sequence(seq_pred[i])),
            })
            seen += 1
            if seen >= sample_limit:
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)


@torch.no_grad()
def collect_latent_codes(model, loader, device, use_condition):
    model.eval()
    mus = []
    labels = []
    seqs = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)
        cond = batch["condition_id"].to(device) if use_condition and "condition_id" in batch else None
        z, mu, logvar = model.encode(input_ids, lengths, condition_id=cond)
        mus.append(mu.cpu().numpy())
        if cond is not None:
            labels.extend(cond.cpu().numpy().tolist())
        else:
            labels.extend([-1] * input_ids.size(0))
        seqs.extend(batch["sequence"])

    if not mus:
        return pd.DataFrame(columns=["pc1", "pc2", "condition_id", "sequence"])

    X = np.concatenate(mus, axis=0)
    pca = PCA(n_components=2, random_state=0)
    pcs = pca.fit_transform(X)
    return pd.DataFrame({
        "pc1": pcs[:, 0],
        "pc2": pcs[:, 1],
        "condition_id": labels,
        "sequence": seqs,
    })


@torch.no_grad()
def predict_condition_fidelity_table(evaluator_model, generated_df: pd.DataFrame, tokenizer, cfg, seed, device):
    if len(generated_df) == 0:
        return pd.DataFrame(columns=["condition_id", "n", "surrogate_hit_rate"])
    preds = predict_conditions_for_sequences(
        evaluator_model,
        generated_df["sequence"].tolist(),
        tokenizer,
        cfg,
        seed=seed,
        device=device,
    )
    tab = generated_df[["condition_id"]].copy()
    tab["pred_condition_id"] = preds
    tab["hit"] = (tab["condition_id"].to_numpy(dtype=int) == tab["pred_condition_id"].to_numpy(dtype=int)).astype(int)
    out = tab.groupby("condition_id", as_index=False).agg(
        n=("hit", "size"),
        surrogate_hit_rate=("hit", "mean"),
    )
    return out


def evaluate_full_variant(model, family, model_tag, split_name, df_split, loader, tokenizer, evaluator_model, cfg, seed, device, use_condition, randomize_condition=False):
    eval_metrics = evaluate_generator_loader(
        model=model,
        loader=loader,
        tokenizer=tokenizer,
        cfg=cfg,
        device=device,
        family=family,
        use_condition=use_condition,
        randomize_condition=randomize_condition,
    )

    recon_df = reconstruct_subset_sequences(
        model=model,
        family=family,
        loader=loader,
        tokenizer=tokenizer,
        seed=seed,
        device=device,
        use_condition=use_condition,
        randomize_condition=randomize_condition,
        sample_limit=cfg.train_cfg.reconstruction_sample_limit,
    )

    cond_fidelity = np.nan
    if len(recon_df):
        true_labels = df_split.iloc[:len(recon_df)]["condition_id"].to_numpy(dtype=int)
        preds = predict_conditions_for_sequences(
            evaluator_model,
            recon_df["sequence_recon"].tolist(),
            tokenizer,
            cfg,
            seed=seed,
            device=device,
        )
        if len(preds) == len(true_labels):
            cond_fidelity = float(np.mean(preds == true_labels))

    return {
        "model_tag": model_tag,
        "split": split_name,
        **eval_metrics,
        "recon_subset_n": int(len(recon_df)),
        "recon_edit_distance_mean": float(recon_df["normalized_edit_distance"].mean()) if len(recon_df) else np.nan,
        "recon_exact_rate": float(recon_df["exact_reconstruction"].mean()) if len(recon_df) else np.nan,
        "recon_condition_fidelity": cond_fidelity,
    }


# =============================================================================
# Generation benchmarking
# =============================================================================
def build_condition_generation_plan(df_train: pd.DataFrame, cfg: Step6Config) -> pd.DataFrame:
    support = (
        df_train.groupby("condition_id", as_index=False)
        .size()
        .rename(columns={"size": "n_train"})
        .sort_values("condition_id")
        .reset_index(drop=True)
    )
    base = cfg.train_cfg.generation_per_condition
    if cfg.train_cfg.generation_per_condition_support_scaled:
        scale = support["n_train"] / support["n_train"].median()
        n_gen = np.round(base * scale).astype(int)
        n_gen = np.clip(n_gen, cfg.train_cfg.min_generation_per_condition, cfg.train_cfg.max_generation_per_condition_cap)
    else:
        n_gen = np.repeat(base, len(support))
    support["n_generate"] = n_gen.astype(int)
    return support


def generate_sequences_for_variant(model, family, model_tag, df_train, tokenizer, cfg, seed, device, use_condition, randomize_condition=False, temperature=None):
    if temperature is None:
        temperature = cfg.generated_eval_temperature_default

    plan = build_condition_generation_plan(df_train, cfg)
    rows = []

    for _, r in plan.iterrows():
        cond_id = int(r["condition_id"])
        n_gen = int(r["n_generate"])

        if family == "cvae":
            if use_condition:
                cond_tensor = torch.full((n_gen,), cond_id, dtype=torch.long, device=device)
                if randomize_condition:
                    rand_cond = torch.randint(low=0, high=int(df_train["condition_id"].nunique()), size=(n_gen,), device=device)
                    cond_tensor = rand_cond
            else:
                cond_tensor = None
            toks = model.sample_sequences(
                n=n_gen,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=cfg.train_cfg.max_decode_len,
                temperature=temperature,
                top_k=cfg.train_cfg.top_k,
                nucleus_p=cfg.train_cfg.nucleus_p,
                condition_id=cond_tensor,
                device=device,
            )
            seqs = [tokenizer.decode(t.cpu().tolist()) for t in toks]
            raw_lens = [int((t != tokenizer.pad_id).sum().item()) for t in toks]

        else:
            if use_condition:
                cond_tensor = torch.full((n_gen,), cond_id, dtype=torch.long, device=device)
                if randomize_condition:
                    rand_cond = torch.randint(low=0, high=int(df_train["condition_id"].nunique()), size=(n_gen,), device=device)
                    cond_tensor = rand_cond
            else:
                cond_tensor = None
            toks = model.sample_sequences(
                n=n_gen,
                bos_id=tokenizer.bos_id,
                eos_id=tokenizer.eos_id,
                pad_id=tokenizer.pad_id,
                max_decode_len=cfg.train_cfg.max_decode_len,
                condition_id=cond_tensor,
                temperature=temperature,
                top_k=cfg.train_cfg.top_k,
                nucleus_p=cfg.train_cfg.nucleus_p,
                device=device,
            )
            seqs = [tokenizer.decode(t.cpu().tolist()) for t in toks]
            raw_lens = [int((t != tokenizer.pad_id).sum().item()) for t in toks]

        for s, raw_len in zip(seqs, raw_lens):
            rows.append({
                "model_tag": model_tag,
                "seed": seed,
                "condition_id": cond_id,
                "sequence": clean_sequence(s),
                "raw_decode_len_tokens": raw_len,
                "source": "generated",
            })

    gen_df = pd.DataFrame(rows)
    train_refs = set(df_train["sequence"].tolist())
    train_ref_list = df_train["sequence"].tolist()

    gen_df["valid_peptide"] = gen_df["sequence"].map(lambda s: int(is_valid_peptide(s, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)))
    gen_df["exact_train_overlap"] = gen_df["sequence"].map(lambda s: int(s in train_refs))
    gen_df["novelty_exact"] = 1 - gen_df["exact_train_overlap"]

    lowc = gen_df["sequence"].map(lambda s: low_complexity_flags(s, cfg.qa_cfg))
    gen_df = pd.concat([gen_df, pd.DataFrame(list(lowc))], axis=1)

    gen_df["early_eos_flag"] = (gen_df["raw_decode_len_tokens"] <= cfg.qa_cfg.early_eos_length_threshold).astype(int)
    gen_df["length"] = gen_df["sequence"].map(len)
    gen_df["charge_proxy"] = gen_df["sequence"].map(simple_charge_proxy)
    gen_df["mean_hydropathy"] = gen_df["sequence"].map(mean_hydropathy)
    gen_df["entropy"] = gen_df["sequence"].map(sequence_entropy)
    gen_df["nn_similarity_train"] = gen_df["sequence"].map(
        lambda s: nearest_neighbor_similarity(s, train_ref_list, k=cfg.jaccard_k, sample_limit=cfg.nn_similarity_sample_limit, seed=seed)
    )

    return gen_df


def evaluate_generation_table(gen_df, df_train, evaluator_model, tokenizer, cfg, seed, device):
    if len(gen_df) == 0:
        return {
            "n_generated": 0,
            "valid_rate": np.nan,
            "novelty_rate": np.nan,
            "uniqueness_rate": np.nan,
            "low_complexity_rate": np.nan,
            "exact_train_overlap_rate": np.nan,
            "nn_similarity_train_mean": np.nan,
            "condition_fidelity": np.nan,
            "per_condition_surrogate_hit_mean": np.nan,
            "pairwise_diversity": np.nan,
        }

    preds = predict_conditions_for_sequences(
        evaluator_model,
        gen_df["sequence"].tolist(),
        tokenizer,
        cfg,
        seed=seed,
        device=device,
    )
    cond_fid = float(np.mean(preds == gen_df["condition_id"].to_numpy(dtype=int)))
    per_cond_tab = predict_condition_fidelity_table(evaluator_model, gen_df, tokenizer, cfg, seed, device)
    hetero = float(per_cond_tab["surrogate_hit_rate"].mean()) if len(per_cond_tab) else np.nan

    div = pairwise_diversity_summary(
        gen_df["sequence"].tolist(),
        k=cfg.jaccard_k,
        pair_limit=cfg.pairwise_diversity_pair_limit,
        seed=seed,
    )

    return {
        "n_generated": int(len(gen_df)),
        "valid_rate": float(gen_df["valid_peptide"].mean()),
        "novelty_rate": float(gen_df["novelty_exact"].mean()),
        "uniqueness_rate": float(gen_df["sequence"].nunique() / max(1, len(gen_df))),
        "low_complexity_rate": float(gen_df["low_complexity_any_flag"].mean()),
        "exact_train_overlap_rate": float(gen_df["exact_train_overlap"].mean()),
        "nn_similarity_train_mean": float(gen_df["nn_similarity_train"].mean()),
        "condition_fidelity": cond_fid,
        "per_condition_surrogate_hit_mean": hetero,
        "pairwise_diversity": float(div["mean_pairwise_diversity"]),
    }


def evaluate_generation_for_variant(model, family, model_tag, df_train, tokenizer, evaluator_model, cfg, seed, device, use_condition, randomize_condition=False):
    gen_df = generate_sequences_for_variant(
        model=model,
        family=family,
        model_tag=model_tag,
        df_train=df_train,
        tokenizer=tokenizer,
        cfg=cfg,
        seed=seed,
        device=device,
        use_condition=use_condition,
        randomize_condition=randomize_condition,
        temperature=cfg.generated_eval_temperature_default,
    )

    main_eval = evaluate_generation_table(
        gen_df=gen_df,
        df_train=df_train,
        evaluator_model=evaluator_model,
        tokenizer=tokenizer,
        cfg=cfg,
        seed=seed,
        device=device,
    )

    temp_rows = []
    for temp in cfg.train_cfg.decoding_temperatures:
        temp_df = generate_sequences_for_variant(
            model=model,
            family=family,
            model_tag=model_tag,
            df_train=df_train,
            tokenizer=tokenizer,
            cfg=cfg,
            seed=seed,
            device=device,
            use_condition=use_condition,
            randomize_condition=randomize_condition,
            temperature=temp,
        )
        temp_eval = evaluate_generation_table(
            gen_df=temp_df,
            df_train=df_train,
            evaluator_model=evaluator_model,
            tokenizer=tokenizer,
            cfg=cfg,
            seed=seed,
            device=device,
        )
        temp_rows.append({"temperature": temp, **temp_eval})

    temp_eval_df = pd.DataFrame(temp_rows)
    per_condition_df = predict_condition_fidelity_table(evaluator_model, gen_df, tokenizer, cfg, seed, device)
    return main_eval, temp_eval_df, gen_df, per_condition_df


# =============================================================================
# Retrieval baseline
# =============================================================================
def build_train_reference_by_condition(df_train: pd.DataFrame) -> Dict[int, List[str]]:
    return df_train.groupby("condition_id")["sequence"].apply(list).to_dict()


def retrieval_match_sequence(query_seq: str, refs: List[str], k: int = 3) -> Tuple[str, float]:
    if not refs:
        return "", np.nan
    best_ref = None
    best_sim = -1.0
    for r in refs:
        sim = jaccard_similarity(query_seq, r, k=k)
        if sim > best_sim:
            best_sim = sim
            best_ref = r
    return best_ref, best_sim


def run_retrieval_baseline(model_tag, seed, df_train, df_val, df_test, tokenizer, evaluator_model, cfg, dirs, device):
    seed_all(seed)
    refs_by_cond = build_train_reference_by_condition(df_train)

    def summarize_retrieval(split_df, split_name):
        rows = []
        for _, row in split_df.iterrows():
            refs = refs_by_cond.get(int(row["condition_id"]), [])
            hit, sim = retrieval_match_sequence(row["sequence"], refs, k=cfg.jaccard_k)
            rows.append({
                "sequence_true": row["sequence"],
                "condition_id": int(row["condition_id"]),
                "sequence_retrieved": hit,
                "jaccard_to_retrieved": sim,
                "normalized_edit_distance": normalized_edit_distance(row["sequence"], hit),
                "exact_reconstruction": int(row["sequence"] == hit),
            })
        recon_df = pd.DataFrame(rows)

        preds = predict_conditions_for_sequences(
            evaluator_model,
            recon_df["sequence_retrieved"].tolist(),
            tokenizer,
            cfg,
            seed=seed,
            device=device,
        )
        cond_fid = float(np.mean(preds == recon_df["condition_id"].to_numpy(dtype=int))) if len(recon_df) else np.nan

        return {
            "model_tag": model_tag,
            "split": split_name,
            "loss": np.nan,
            "recon_loss": np.nan,
            "raw_kl": 0.0,
            "free_kl": 0.0,
            "aux_condition_loss": np.nan,
            "token_acc": np.nan,
            "active_units_main": np.nan,
            "mean_latent_var": np.nan,
            "mean_mu_abs": np.nan,
            "mean_kl_per_dim": np.nan,
            "recon_subset_n": int(len(recon_df)),
            "recon_edit_distance_mean": float(recon_df["normalized_edit_distance"].mean()) if len(recon_df) else np.nan,
            "recon_exact_rate": float(recon_df["exact_reconstruction"].mean()) if len(recon_df) else np.nan,
            "recon_condition_fidelity": cond_fid,
        }, recon_df

    val_eval, val_recon_df = summarize_retrieval(df_val, "val")
    test_eval, test_recon_df = summarize_retrieval(df_test, "test")

    gen_rows = []
    rng = np.random.default_rng(seed)
    gen_plan = build_condition_generation_plan(df_train, cfg)
    for _, r in gen_plan.iterrows():
        cond_id = int(r["condition_id"])
        n_gen = int(r["n_generate"])
        refs = refs_by_cond.get(cond_id, [])
        if not refs:
            continue
        picks = rng.choice(len(refs), size=n_gen, replace=True)
        for idx in picks:
            seq = refs[int(idx)]
            gen_rows.append({
                "model_tag": model_tag,
                "seed": seed,
                "condition_id": cond_id,
                "sequence": seq,
                "raw_decode_len_tokens": len(seq),
                "source": "retrieval_reference",
            })

    gen_df = pd.DataFrame(gen_rows)
    if len(gen_df):
        gen_df["valid_peptide"] = gen_df["sequence"].map(lambda s: int(is_valid_peptide(s, cfg.train_cfg.min_length, cfg.train_cfg.max_decode_len)))
        gen_df["exact_train_overlap"] = 1
        gen_df["novelty_exact"] = 0
        lowc = gen_df["sequence"].map(lambda s: low_complexity_flags(s, cfg.qa_cfg))
        gen_df = pd.concat([gen_df, pd.DataFrame(list(lowc))], axis=1)
        gen_df["early_eos_flag"] = 0
        gen_df["length"] = gen_df["sequence"].map(len)
        gen_df["charge_proxy"] = gen_df["sequence"].map(simple_charge_proxy)
        gen_df["mean_hydropathy"] = gen_df["sequence"].map(mean_hydropathy)
        gen_df["entropy"] = gen_df["sequence"].map(sequence_entropy)
        gen_df["nn_similarity_train"] = 1.0

    gen_eval = evaluate_generation_table(
        gen_df=gen_df,
        df_train=df_train,
        evaluator_model=evaluator_model,
        tokenizer=tokenizer,
        cfg=cfg,
        seed=seed,
        device=device,
    )

    temp_eval_df = pd.DataFrame({
        "temperature": list(cfg.train_cfg.decoding_temperatures),
        "n_generated": [len(gen_df)] * len(cfg.train_cfg.decoding_temperatures),
        "valid_rate": [gen_eval["valid_rate"]] * len(cfg.train_cfg.decoding_temperatures),
        "novelty_rate": [gen_eval["novelty_rate"]] * len(cfg.train_cfg.decoding_temperatures),
        "uniqueness_rate": [gen_eval["uniqueness_rate"]] * len(cfg.train_cfg.decoding_temperatures),
        "low_complexity_rate": [gen_eval["low_complexity_rate"]] * len(cfg.train_cfg.decoding_temperatures),
        "exact_train_overlap_rate": [gen_eval["exact_train_overlap_rate"]] * len(cfg.train_cfg.decoding_temperatures),
        "nn_similarity_train_mean": [gen_eval["nn_similarity_train_mean"]] * len(cfg.train_cfg.decoding_temperatures),
        "condition_fidelity": [gen_eval["condition_fidelity"]] * len(cfg.train_cfg.decoding_temperatures),
        "per_condition_surrogate_hit_mean": [gen_eval["per_condition_surrogate_hit_mean"]] * len(cfg.train_cfg.decoding_temperatures),
        "pairwise_diversity": [gen_eval["pairwise_diversity"]] * len(cfg.train_cfg.decoding_temperatures),
    })

    per_condition_df = predict_condition_fidelity_table(evaluator_model, gen_df, tokenizer, cfg, seed, device)

    export_json({
        "model_tag": model_tag,
        "seed": seed,
        "family": "retrieval",
        "val_eval": val_eval,
        "test_eval": test_eval,
        "generation_eval": gen_eval,
    }, get_done_flag(dirs, model_tag, seed))

    return {
        "model_tag": model_tag,
        "seed": seed,
        "family": "retrieval",
        "best_epoch": 0,
        "checkpoint_path": "",
        "history_df": pd.DataFrame(),
        "val_eval": val_eval,
        "test_eval": test_eval,
        "generation_eval": gen_eval,
        "temperature_eval_df": temp_eval_df,
        "generated_df": gen_df,
        "per_condition_df": per_condition_df,
    }


# =============================================================================
# Variant runner
# =============================================================================
def train_one_variant(model_tag, seed, df_train, df_val, df_test, tokenizer, n_conditions, evaluator_model, cfg, dirs, device):
    if model_tag == "retrieval":
        return run_retrieval_baseline(model_tag, seed, df_train, df_val, df_test, tokenizer, evaluator_model, cfg, dirs, device)

    seed_all(seed)

    model, family, use_condition = make_model_for_variant(model_tag, tokenizer, n_conditions, cfg)
    model = model.to(device)

    randomize_condition = (model_tag == "random_condition")

    loaders = build_sequence_dataloaders(df_train, df_val, df_test, tokenizer, cfg, seed, with_condition=(use_condition or family in {"gru_random_condition"}))
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train_cfg.lr, weight_decay=cfg.train_cfg.weight_decay)

    best_state = None
    best_metric = float("inf")
    best_epoch = None
    patience = 0
    hist = []

    for epoch in range(cfg.train_cfg.epochs):
        train_epoch_generator.current_epoch = epoch

        tr = train_epoch_generator(
            model=model,
            loader=loaders["train"],
            optimizer=optimizer,
            tokenizer=tokenizer,
            cfg=cfg,
            device=device,
            family="gru" if family == "gru_random_condition" else family,
            use_condition=use_condition,
            randomize_condition=randomize_condition,
        )
        va = evaluate_generator_loader(
            model=model,
            loader=loaders["val"],
            tokenizer=tokenizer,
            cfg=cfg,
            device=device,
            family="gru" if family == "gru_random_condition" else family,
            use_condition=use_condition,
            randomize_condition=randomize_condition,
        )

        row = {
            "model_tag": model_tag,
            "seed": seed,
            "epoch": epoch + 1,
            **{f"train_{k}": v for k, v in tr.items()},
            **{f"val_{k}": v for k, v in va.items()},
        }
        hist.append(row)

        if family in {"cvae"}:
            val_select = va["recon_loss"] - 0.10 * va["free_kl"] - 0.05 * va["active_units_main"]
        else:
            val_select = va["recon_loss"]

        if val_select < best_metric:
            best_metric = val_select
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            patience = 0
        else:
            patience += 1
            if patience >= cfg.train_cfg.early_stopping_patience:
                break

    if best_state is None:
        raise RuntimeError(f"Training failed for {model_tag} seed={seed}")

    model.load_state_dict(best_state)

    ckpt_path = dirs["models"] / f"step6_{model_tag}_seed{seed}.pt"
    torch.save({
        "seed": seed,
        "model_tag": model_tag,
        "family": family,
        "state_dict": model.state_dict(),
        "config": asdict(cfg.train_cfg),
        "n_conditions": n_conditions,
        "vocab_size": tokenizer.vocab_size,
        "best_epoch": best_epoch,
    }, ckpt_path)

    hist_df = pd.DataFrame(hist)
    export_csv(hist_df, dirs["logs"] / f"step6_training_history_{model_tag}_seed{seed}.csv")

    val_eval = evaluate_full_variant(
        model=model,
        family="gru" if family == "gru_random_condition" else family,
        model_tag=model_tag,
        split_name="val",
        df_split=df_val,
        loader=loaders["val"],
        tokenizer=tokenizer,
        evaluator_model=evaluator_model,
        cfg=cfg,
        seed=seed,
        device=device,
        use_condition=use_condition,
        randomize_condition=randomize_condition,
    )

    test_eval = evaluate_full_variant(
        model=model,
        family="gru" if family == "gru_random_condition" else family,
        model_tag=model_tag,
        split_name="test",
        df_split=df_test,
        loader=loaders["test"],
        tokenizer=tokenizer,
        evaluator_model=evaluator_model,
        cfg=cfg,
        seed=seed,
        device=device,
        use_condition=use_condition,
        randomize_condition=randomize_condition,
    )

    if cfg.runtime_cfg.no_generation_mode:
        gen_eval = None
        temp_eval_df = pd.DataFrame()
        generated_df = pd.DataFrame()
        per_condition_df = pd.DataFrame()
    else:
        gen_eval, temp_eval_df, generated_df, per_condition_df = evaluate_generation_for_variant(
            model=model,
            family="gru" if family == "gru_random_condition" else family,
            model_tag=model_tag,
            df_train=df_train,
            tokenizer=tokenizer,
            evaluator_model=evaluator_model,
            cfg=cfg,
            seed=seed,
            device=device,
            use_condition=use_condition,
            randomize_condition=randomize_condition,
        )

    out = {
        "model_tag": model_tag,
        "seed": seed,
        "family": family,
        "best_epoch": best_epoch,
        "checkpoint_path": str(ckpt_path),
        "history_df": hist_df,
        "val_eval": val_eval,
        "test_eval": test_eval,
        "generation_eval": gen_eval,
        "temperature_eval_df": temp_eval_df,
        "generated_df": generated_df,
        "per_condition_df": per_condition_df,
    }

    export_json({
        "model_tag": model_tag,
        "seed": seed,
        "family": family,
        "best_epoch": best_epoch,
        "checkpoint_path": str(ckpt_path),
        "val_eval": val_eval,
        "test_eval": test_eval,
        "generation_eval": gen_eval,
    }, get_done_flag(dirs, model_tag, seed))

    return out


# =============================================================================
# Aggregation
# =============================================================================
def flatten_result_record(res: Dict[str, Any]) -> Dict[str, Any]:
    row = {
        "model_tag": res["model_tag"],
        "seed": res["seed"],
        "family": res["family"],
        "best_epoch": res.get("best_epoch", np.nan),
        "checkpoint_path": res.get("checkpoint_path", ""),
    }
    for prefix, block in [("val", res.get("val_eval")), ("test", res.get("test_eval")), ("gen", res.get("generation_eval"))]:
        if block is None:
            continue
        for k, v in block.items():
            if k in {"model_tag", "split"}:
                continue
            row[f"{prefix}_{k}"] = v
    return row


def aggregate_seed_metrics(per_seed_df: pd.DataFrame, cfg: Step6Config) -> pd.DataFrame:
    metric_cols = [c for c in per_seed_df.columns if c not in {"model_tag", "seed", "family", "checkpoint_path"} and pd.api.types.is_numeric_dtype(per_seed_df[c])]
    rows = []
    for model_tag, g in per_seed_df.groupby("model_tag"):
        for metric in metric_cols:
            vals = g[metric].dropna().tolist()
            stat = bootstrap_mean_ci(vals, iterations=cfg.bootstrap_iterations, seed=42)
            rows.append({"model_tag": model_tag, "metric": metric, **stat})
    return pd.DataFrame(rows)


def build_summary_tables(results: List[Dict[str, Any]], cfg: Step6Config, dirs: Dict[str, Path]) -> Dict[str, pd.DataFrame]:
    per_seed_df = pd.DataFrame([flatten_result_record(r) for r in results]).sort_values(["model_tag", "seed"]).reset_index(drop=True)
    export_csv(per_seed_df, dirs["tables"] / "step6_per_seed_summary.csv")

    agg_df = aggregate_seed_metrics(per_seed_df, cfg)
    export_csv(agg_df, dirs["tables"] / "step6_aggregated_summary.csv")

    history_frames = [r["history_df"] for r in results if r.get("history_df") is not None and len(r["history_df"])]
    hist_df = pd.concat(history_frames, axis=0, ignore_index=True) if len(history_frames) else pd.DataFrame()
    if len(hist_df):
        export_csv(hist_df, dirs["logs"] / "step6_all_training_histories.csv")

    temp_frames = []
    for r in results:
        df = r.get("temperature_eval_df")
        if df is not None and len(df):
            tmp = df.copy()
            tmp["model_tag"] = r["model_tag"]
            tmp["seed"] = r["seed"]
            temp_frames.append(tmp)
    temp_df = pd.concat(temp_frames, axis=0, ignore_index=True) if len(temp_frames) else pd.DataFrame()
    if len(temp_df):
        export_csv(temp_df, dirs["tables"] / "step6_temperature_sensitivity_summary.csv")

    gen_frames = []
    for r in results:
        df = r.get("generated_df")
        if df is not None and len(df):
            gen_frames.append(df.copy())
    gen_df = pd.concat(gen_frames, axis=0, ignore_index=True) if len(gen_frames) else pd.DataFrame()
    if len(gen_df):
        export_csv(gen_df, dirs["tables"] / "step6_all_generated_sequences.csv")

    per_cond_frames = []
    for r in results:
        df = r.get("per_condition_df")
        if df is not None and len(df):
            tmp = df.copy()
            tmp["model_tag"] = r["model_tag"]
            tmp["seed"] = r["seed"]
            per_cond_frames.append(tmp)
    per_cond_df = pd.concat(per_cond_frames, axis=0, ignore_index=True) if len(per_cond_frames) else pd.DataFrame()
    if len(per_cond_df):
        export_csv(per_cond_df, dirs["tables"] / "step6_per_condition_fidelity_summary.csv")

    return {
        "per_seed_df": per_seed_df,
        "agg_df": agg_df,
        "hist_df": hist_df,
        "temp_df": temp_df,
        "gen_df": gen_df,
        "per_cond_df": per_cond_df,
    }


# =============================================================================
# Figure helpers
# =============================================================================
def get_metric_summary(agg_df: pd.DataFrame, model_tag: str, metric: str) -> Dict[str, float]:
    x = agg_df.loc[(agg_df["model_tag"] == model_tag) & (agg_df["metric"] == metric)]
    if len(x) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    row = x.iloc[0]
    return {"mean": float(row["mean"]), "ci_low": float(row["ci_low"]), "ci_high": float(row["ci_high"])}


def draw_bars(ax, model_tags, means, lows, highs, ylabel, title, ylim=None, value_fmt="{:.2f}"):
    xs = np.arange(len(model_tags))
    colors = [MODEL_COLORS[m] for m in model_tags]
    ax.bar(xs, means, color=colors, edgecolor="none")
    yerr = np.vstack([np.array(means) - np.array(lows), np.array(highs) - np.array(means)])
    ax.errorbar(xs, means, yerr=yerr, fmt="none", ecolor=PALETTE["black"], capsize=4, linewidth=1.2)
    ax.set_xticks(xs)
    ax.set_xticklabels([MODEL_LABELS[m] for m in model_tags], rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    style_axis(ax, "y")
    y0, y1 = ax.get_ylim()
    dy = (y1 - y0) * 0.02
    for x, v in zip(xs, means):
        if pd.notnull(v):
            ax.text(x, v + dy, value_fmt.format(v), ha="center", va="bottom", fontsize=10)


# =============================================================================
# Figures
# =============================================================================
def build_main_benchmark_figure(agg_df: pd.DataFrame, dirs: Dict[str, Path], cfg: Step6Config):
    # Figure 1: Fidelity benchmark + novelty + similarity
    fig = plt.figure(figsize=(14.5, 8.5))
    gs = GridSpec(2, 3, figure=fig, height_ratios=[1.0, 1.1], width_ratios=[1.3, 1.0, 0.9])

    # a: fidelity benchmark
    ax = fig.add_subplot(gs[0, 0])
    means, lows, highs = [], [], []
    subset = MODEL_ORDER
    for m in subset:
        stat = get_metric_summary(agg_df, m, "gen_condition_fidelity")
        means.append(stat["mean"]); lows.append(stat["ci_low"]); highs.append(stat["ci_high"])
    draw_bars(ax, subset, means, lows, highs, ylabel="Fidelity", title="Fidelity benchmark", ylim=(0, 0.8))
    add_panel_label(ax, "a")

    # b-left: fidelity
    ax = fig.add_subplot(gs[0, 1])
    draw_bars(ax, subset, means, lows, highs, ylabel="Fraction", title="Fidelity", ylim=(0, 0.8))
    ax.set_xticklabels([])
    add_panel_label(ax, "b")

    # b-right: novelty
    ax = fig.add_subplot(gs[0, 2])
    means, lows, highs = [], [], []
    for m in subset:
        stat = get_metric_summary(agg_df, m, "gen_novelty_rate")
        means.append(stat["mean"]); lows.append(stat["ci_low"]); highs.append(stat["ci_high"])
    draw_bars(ax, subset, means, lows, highs, ylabel="", title="Novelty", ylim=(0, 1.05))
    ax.set_xticklabels([])
    ax.set_ylabel("")

    # c: similarity to training set
    ax = fig.add_subplot(gs[1, :])
    subset_c = [m for m in MODEL_ORDER if m != "retrieval"]
    means, lows, highs = [], [], []
    for m in subset_c:
        stat = get_metric_summary(agg_df, m, "gen_nn_similarity_train_mean")
        means.append(stat["mean"]); lows.append(stat["ci_low"]); highs.append(stat["ci_high"])
    draw_bars(ax, subset_c, means, lows, highs, ylabel="NN similarity", title="Similarity to training set", ylim=(0, 0.34))
    ax.text(0.995, 0.98, "Retrieval ref. = 1.00", transform=ax.transAxes, ha="right", va="top", fontsize=11, color=PALETTE["gray"])
    add_panel_label(ax, "c")

    # shared legend
    handles = [plt.Rectangle((0, 0), 1, 1, color=MODEL_COLORS[m]) for m in MODEL_ORDER]
    labels = [MODEL_LABELS[m] for m in MODEL_ORDER]
    fig.legend(handles, labels, ncol=6, loc="lower center", frameon=True, bbox_to_anchor=(0.5, -0.02))

    fig.tight_layout(rect=(0, 0.05, 1, 1))
    out = save_figure(fig, dirs["figures_main"] / "Fig6_step6_benchmark_main", cfg.fig_cfg)

    # source data
    rows = []
    for m in MODEL_ORDER:
        rows.append({
            "model_tag": m,
            "condition_fidelity_mean": get_metric_summary(agg_df, m, "gen_condition_fidelity")["mean"],
            "novelty_rate_mean": get_metric_summary(agg_df, m, "gen_novelty_rate")["mean"],
            "nn_similarity_train_mean": get_metric_summary(agg_df, m, "gen_nn_similarity_train_mean")["mean"],
        })
    export_csv(pd.DataFrame(rows), dirs["source_data_figures"] / "Fig6_step6_benchmark_main_source_data.csv")
    return out


def build_main_generation_quality_figure(agg_df: pd.DataFrame, dirs: Dict[str, Path], cfg: Step6Config):
    # Figure 2: generation quality + memorization profile + condition heterogeneity
    fig = plt.figure(figsize=(14.0, 8.0))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.0], width_ratios=[1.0, 1.0])

    # a: generation quality grouped bars
    ax = fig.add_subplot(gs[0, :])
    metrics = [
        ("gen_valid_rate", "Validity"),
        ("gen_novelty_rate", "Novelty"),
        ("gen_uniqueness_rate", "Uniqueness"),
    ]
    width = 0.12
    xs = np.arange(len(metrics))
    for i, mtag in enumerate(MODEL_ORDER):
        vals = [get_metric_summary(agg_df, mtag, met)["mean"] for met, _ in metrics]
        ax.bar(xs + (i - (len(MODEL_ORDER)-1)/2) * width, vals, width=width, color=MODEL_COLORS[mtag], label=MODEL_LABELS[mtag])
        for j, v in enumerate(vals):
            if pd.notnull(v) and i == 0:
                ax.text(xs[j] + (i - (len(MODEL_ORDER)-1)/2) * width, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(xs)
    ax.set_xticklabels([lab for _, lab in metrics])
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, 1.05)
    ax.set_title("Generation quality")
    style_axis(ax, "y")
    add_panel_label(ax, "a")

    # b: memorization profile
    ax = fig.add_subplot(gs[1, 0])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    means, lows, highs = [], [], []
    for m in subset:
        stat = get_metric_summary(agg_df, m, "gen_nn_similarity_train_mean")
        means.append(stat["mean"]); lows.append(stat["ci_low"]); highs.append(stat["ci_high"])
    draw_bars(ax, subset, means, lows, highs, ylabel="NN similarity", title="Memorization profile", ylim=(0, 0.34))
    ax.text(0.995, 0.98, "Retrieval ref. = 1.00", transform=ax.transAxes, ha="right", va="top", fontsize=11, color=PALETTE["gray"])
    add_panel_label(ax, "b")

    # c: condition heterogeneity
    ax = fig.add_subplot(gs[1, 1])
    means, lows, highs = [], [], []
    for m in MODEL_ORDER:
        stat = get_metric_summary(agg_df, m, "gen_per_condition_surrogate_hit_mean")
        means.append(stat["mean"]); lows.append(stat["ci_low"]); highs.append(stat["ci_high"])
    draw_bars(ax, MODEL_ORDER, means, lows, highs, ylabel="Per-condition surrogate hit rate", title="Condition heterogeneity", ylim=(0, 1.05))
    add_panel_label(ax, "c")

    handles = [plt.Rectangle((0, 0), 1, 1, color=MODEL_COLORS[m]) for m in MODEL_ORDER]
    labels = [MODEL_LABELS[m] for m in MODEL_ORDER]
    fig.legend(handles, labels, ncol=6, loc="lower center", frameon=True, bbox_to_anchor=(0.5, -0.02))

    fig.tight_layout(rect=(0, 0.05, 1, 1))
    out = save_figure(fig, dirs["figures_main"] / "Fig6_step6_generation_quality", cfg.fig_cfg)

    rows = []
    for m in MODEL_ORDER:
        rows.append({
            "model_tag": m,
            "valid_rate_mean": get_metric_summary(agg_df, m, "gen_valid_rate")["mean"],
            "novelty_rate_mean": get_metric_summary(agg_df, m, "gen_novelty_rate")["mean"],
            "uniqueness_rate_mean": get_metric_summary(agg_df, m, "gen_uniqueness_rate")["mean"],
            "nn_similarity_train_mean": get_metric_summary(agg_df, m, "gen_nn_similarity_train_mean")["mean"],
            "per_condition_surrogate_hit_mean": get_metric_summary(agg_df, m, "gen_per_condition_surrogate_hit_mean")["mean"],
        })
    export_csv(pd.DataFrame(rows), dirs["source_data_figures"] / "Fig6_step6_generation_quality_source_data.csv")
    return out


def build_supp_latent_figure(results: List[Dict[str, Any]], df_test: pd.DataFrame, tokenizer: PeptideTokenizer, cfg: Step6Config, dirs: Dict[str, Path], device: torch.device):
    candidate = next((r for r in results if r["model_tag"] == "full_cvae"), None)
    if candidate is None or not candidate.get("checkpoint_path"):
        return None

    tc = cfg.train_cfg
    model = ConditionalSequenceVAE(
        vocab_size=tokenizer.vocab_size,
        cond_vocab_size=int(df_test["condition_id"].nunique()),
        emb_dim=tc.emb_dim,
        enc_hidden_dim=tc.enc_hidden_dim,
        dec_hidden_dim=tc.dec_hidden_dim,
        latent_dim=tc.latent_dim,
        num_layers=tc.num_layers,
        dropout=tc.dropout,
        use_condition=True,
        aux_condition_head=True,
    ).to(device)
    ckpt = torch.load(candidate["checkpoint_path"], map_location=device)
    model.load_state_dict(ckpt["state_dict"])

    loader = build_sequence_dataloaders(df_test, df_test.iloc[:0].copy(), df_test.iloc[:0].copy(), tokenizer, cfg, candidate["seed"], with_condition=True)["train"]
    latent_df = collect_latent_codes(model, loader, device=device, use_condition=True)

    hist_df = candidate["history_df"].copy()
    active_cols = [c for c in hist_df.columns if c.startswith("val_active_units_thr_")]
    latent_curve = hist_df[["epoch", "val_raw_kl", "val_free_kl", "val_active_units_main", "val_mean_latent_var"]].copy()

    fig = plt.figure(figsize=(15, 5.5))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1.2, 1.0, 1.0])

    ax = fig.add_subplot(gs[0, 0])
    if len(latent_df):
        ax.scatter(latent_df["pc1"], latent_df["pc2"], c=latent_df["condition_id"], s=10, alpha=0.75)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Test-set latent means (PCA)")
    style_axis(ax, "both")
    add_panel_label(ax, "a")

    ax = fig.add_subplot(gs[0, 1])
    ax.plot(latent_curve["epoch"], latent_curve["val_raw_kl"], label="Val raw KL", linewidth=2)
    ax.plot(latent_curve["epoch"], latent_curve["val_free_kl"], label="Val free-bits KL", linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("KL")
    ax.set_title("Latent KL dynamics")
    ax.legend(frameon=False)
    style_axis(ax, "y")
    add_panel_label(ax, "b")

    ax = fig.add_subplot(gs[0, 2])
    ax.plot(latent_curve["epoch"], latent_curve["val_active_units_main"], label="Active units", linewidth=2)
    ax.plot(latent_curve["epoch"], latent_curve["val_mean_latent_var"], label="Mean latent variance", linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Value")
    ax.set_title("Latent activity summary")
    ax.legend(frameon=False)
    style_axis(ax, "y")
    add_panel_label(ax, "c")

    fig.tight_layout()
    out = save_figure(fig, dirs["figures_supplementary"] / "FigS_step6_latent_diagnostics", cfg.fig_cfg)

    export_csv(latent_df, dirs["source_data_figures"] / "FigS_step6_latent_diagnostics_pca_source_data.csv")
    export_csv(latent_curve, dirs["source_data_figures"] / "FigS_step6_latent_diagnostics_curve_source_data.csv")
    return out


def build_supp_temperature_figure(temp_df: pd.DataFrame, dirs: Dict[str, Path], cfg: Step6Config):
    if temp_df is None or len(temp_df) == 0:
        return None

    fig = plt.figure(figsize=(14.0, 5.5))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.0])

    ax = fig.add_subplot(gs[0, 0])
    temp_mean = temp_df.groupby(["model_tag", "temperature"], as_index=False)[["condition_fidelity"]].mean()
    for m in MODEL_ORDER:
        part = temp_mean.loc[temp_mean["model_tag"] == m]
        if len(part) == 0:
            continue
        ax.plot(part["temperature"], part["condition_fidelity"], marker="o", linewidth=2, label=MODEL_LABELS[m], color=MODEL_COLORS[m])
    ax.set_xlabel("Sampling temperature")
    ax.set_ylabel("Condition fidelity")
    ax.set_ylim(0, 1.0)
    ax.set_title("Decoding temperature sensitivity")
    style_axis(ax, "y")
    ax.legend(frameon=False, fontsize=9)
    add_panel_label(ax, "a")

    ax = fig.add_subplot(gs[0, 1])
    qa_mean = temp_df.groupby(["model_tag"], as_index=False)[["valid_rate", "low_complexity_rate", "exact_train_overlap_rate"]].mean()
    xs = np.arange(len(qa_mean))
    ax.plot(xs, qa_mean["valid_rate"], marker="o", linewidth=2, label="Valid rate")
    ax.plot(xs, qa_mean["low_complexity_rate"], marker="o", linewidth=2, label="Low-complexity rate")
    ax.plot(xs, qa_mean["exact_train_overlap_rate"], marker="o", linewidth=2, label="Exact train overlap")
    ax.set_xticks(xs)
    ax.set_xticklabels([MODEL_LABELS[m] for m in qa_mean["model_tag"]], rotation=30, ha="right")
    ax.set_ylabel("Rate")
    ax.set_title("Generation-time QA summary")
    style_axis(ax, "y")
    ax.legend(frameon=False, fontsize=9)
    add_panel_label(ax, "b")

    fig.tight_layout()
    out = save_figure(fig, dirs["figures_supplementary"] / "FigS_step6_temperature_and_qa", cfg.fig_cfg)
    export_csv(temp_df, dirs["source_data_figures"] / "FigS_step6_temperature_and_qa_source_data.csv")
    return out


# =============================================================================
# Report
# =============================================================================
def write_step6_report(cfg: Step6Config, input_info: Dict[str, Any], evaluator_res: Dict[str, Any], results: List[Dict[str, Any]], summary_tables: Dict[str, pd.DataFrame], dirs: Dict[str, Path]):
    agg_df = summary_tables["agg_df"]
    key_metrics = []
    for m in MODEL_ORDER:
        row = {"model_tag": m}
        for metric in [
            "test_recon_edit_distance_mean",
            "test_recon_exact_rate",
            "gen_condition_fidelity",
            "gen_novelty_rate",
            "gen_uniqueness_rate",
            "gen_valid_rate",
            "gen_nn_similarity_train_mean",
            "gen_per_condition_surrogate_hit_mean",
        ]:
            stat = get_metric_summary(agg_df, m, metric)
            row[f"{metric}_mean"] = stat["mean"]
            row[f"{metric}_ci_low"] = stat["ci_low"]
            row[f"{metric}_ci_high"] = stat["ci_high"]
        key_metrics.append(row)

    report = {
        "run_id": str(uuid.uuid4()),
        "timestamp_unix": time.time(),
        "config_hash": config_hash(cfg),
        "environment": environment_snapshot(),
        "config": asdict(cfg),
        "step4_input": {
            "csv_path": input_info["step4_csv_path"],
            "artifacts_dir": input_info["step4_artifacts_dir"],
        },
        "evaluator": {
            "checkpoint_path": evaluator_res["checkpoint_path"],
            "val_metrics": evaluator_res["val_metrics"],
            "test_metrics": evaluator_res["test_metrics"],
        },
        "n_variant_runs": len(results),
        "key_metrics": key_metrics,
    }
    export_json(report, dirs["reports"] / "step6_run_report.json")
    return report


# =============================================================================
# Runner
# =============================================================================
def run_step6_v5(
    step4_model_ready_csv=DEFAULT_STEP4_MODEL_READY_CSV,
    step4_artifacts_dir=DEFAULT_STEP4_ARTIFACTS_DIR,
    step5_root=DEFAULT_STEP5_ROOT,
    step6_root=DEFAULT_STEP6_ROOT,
    export_png=True,
    export_pdf=True,
    export_tiff=True,
    dpi_png=300,
    dpi_tiff=600,
    verbose=True,
):
    cfg = Step6Config(
        step4_model_ready_csv=str(step4_model_ready_csv),
        step4_artifacts_dir=str(step4_artifacts_dir),
        step5_root=str(step5_root),
        step6_root=str(step6_root),
        fig_cfg=FigureExportConfig(
            export_png=export_png,
            export_pdf=export_pdf,
            export_tiff=export_tiff,
            dpi_png=dpi_png,
            dpi_tiff=dpi_tiff,
        ),
        runtime_cfg=RuntimeConfig(verbose=verbose),
    )

    dirs = ensure_dirs(Path(cfg.step6_root).resolve())
    device = device_for_training()

    export_json(environment_snapshot(), dirs["artifacts"] / "step6_environment_snapshot.json")
    export_json(asdict(cfg), dirs["artifacts"] / "step6_config.json")

    if cfg.runtime_cfg.verbose:
        print_header("Step 6 v5 — loading inputs")

    input_info = load_step4_inputs(cfg, dirs)
    tokenizer = input_info["tokenizer"]
    train_df = input_info["train_df"]
    val_df = input_info["val_df"]
    test_df = input_info["test_df"]
    n_conditions = int(train_df["condition_id"].nunique())

    if cfg.runtime_cfg.verbose:
        print(f"Device: {device}")
        print(f"Train / Val / Test sizes: {len(train_df)} / {len(val_df)} / {len(test_df)}")
        print(f"Number of train-supported conditions: {n_conditions}")
        print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

    evaluator_seed = int(cfg.repeated_seeds[0])
    if cfg.runtime_cfg.verbose:
        print_header(f"Training condition evaluator (seed={evaluator_seed})")

    evaluator_res = train_condition_classifier(
        df_train=train_df,
        df_val=val_df,
        df_test=test_df,
        tokenizer=tokenizer,
        n_classes=n_conditions,
        cfg=cfg,
        seed=evaluator_seed,
        device=device,
        dirs=dirs,
    )
    evaluator_model = evaluator_res["model"]

    results = []
    for model_tag in MODEL_ORDER:
        if not variant_enabled(cfg, model_tag):
            continue
        for seed in cfg.repeated_seeds:
            if not should_run_variant_seed(cfg, model_tag, seed):
                continue

            done_flag = get_done_flag(dirs, model_tag, seed)
            if cfg.runtime_cfg.skip_completed_runs and done_flag.exists():
                if cfg.runtime_cfg.verbose:
                    print(f"[skip] {model_tag} seed={seed}")
                continue

            if cfg.runtime_cfg.verbose:
                print_header(f"Running variant={model_tag} seed={seed}")

            res = train_one_variant(
                model_tag=model_tag,
                seed=int(seed),
                df_train=train_df,
                df_val=val_df,
                df_test=test_df,
                tokenizer=tokenizer,
                n_conditions=n_conditions,
                evaluator_model=evaluator_model,
                cfg=cfg,
                dirs=dirs,
                device=device,
            )
            results.append(res)

            if res.get("generated_df") is not None and len(res["generated_df"]):
                export_csv(res["generated_df"], dirs["qa"] / f"step6_generated_sequences_{model_tag}_seed{seed}.csv")
            if res.get("temperature_eval_df") is not None and len(res["temperature_eval_df"]):
                export_csv(res["temperature_eval_df"], dirs["qa"] / f"step6_temperature_sensitivity_{model_tag}_seed{seed}.csv")
            if res.get("per_condition_df") is not None and len(res["per_condition_df"]):
                export_csv(res["per_condition_df"], dirs["qa"] / f"step6_per_condition_{model_tag}_seed{seed}.csv")

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if len(results) == 0:
        raise RuntimeError("No Step 6 runs executed.")

    summary_tables = build_summary_tables(results, cfg, dirs)

    figure_manifest = {}
    if not cfg.runtime_cfg.no_figure_mode:
        if cfg.runtime_cfg.verbose:
            print_header("Building figures")

        figure_manifest["main_benchmark"] = build_main_benchmark_figure(summary_tables["agg_df"], dirs, cfg)
        figure_manifest["main_generation_quality"] = build_main_generation_quality_figure(summary_tables["agg_df"], dirs, cfg)
        figure_manifest["supp_latent"] = build_supp_latent_figure(results, test_df, tokenizer, cfg, dirs, device)
        figure_manifest["supp_temperature"] = build_supp_temperature_figure(summary_tables["temp_df"], dirs, cfg)
        export_json(figure_manifest, dirs["reports"] / "step6_figure_manifest.json")

    report = write_step6_report(cfg, input_info, evaluator_res, results, summary_tables, dirs)

    if cfg.runtime_cfg.verbose:
        print_header("Step 6 v5 completed")
        print(f"Per-seed summary: {dirs['tables'] / 'step6_per_seed_summary.csv'}")
        print(f"Aggregated summary: {dirs['tables'] / 'step6_aggregated_summary.csv'}")
        print(f"All generated sequences: {dirs['tables'] / 'step6_all_generated_sequences.csv'}")
        print(f"Run report: {dirs['reports'] / 'step6_run_report.json'}")

    return {
        "config": cfg,
        "dirs": dirs,
        "input_info": input_info,
        "evaluator_res": evaluator_res,
        "results": results,
        "summary_tables": summary_tables,
        "figure_manifest": figure_manifest,
        "report": report,
    }


# =============================================================================
# Execute
# =============================================================================
if __name__ == "__main__":
    out = run_step6_v5(
        step4_model_ready_csv=DEFAULT_STEP4_MODEL_READY_CSV,
        step4_artifacts_dir=DEFAULT_STEP4_ARTIFACTS_DIR,
        step5_root=DEFAULT_STEP5_ROOT,
        step6_root=DEFAULT_STEP6_ROOT,
        export_png=True,
        export_pdf=True,
        export_tiff=True,
        dpi_png=300,
        dpi_tiff=600,
        verbose=True,
    )

redesign figures

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# =============================================================================
# PATHS
# =============================================================================
STEP6_ROOT = Path("/home/data3/Moe/nature_computational2/step6_v5").resolve()

TABLES_DIR = STEP6_ROOT / "tables"
LOGS_DIR = STEP6_ROOT / "logs"

FIG_MAIN_DIR = STEP6_ROOT / "figures_main_redesign_v8"
FIG_SUPP_DIR = STEP6_ROOT / "figures_supplementary_redesign_v8"
SOURCE_DATA_DIR = STEP6_ROOT / "source_data_figures_redesign_v8"

FIG_MAIN_DIR.mkdir(parents=True, exist_ok=True)
FIG_SUPP_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# STYLE
# =============================================================================
MODEL_ORDER = [
    "retrieval",
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "retrieval": "Retrieval",
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

COLORS = {
    "retrieval": "#B4C5E4",
    "full_cvae": "#C0001A",
    "conditional_gru_lm": "#FF1A1A",
    "random_condition": "#1F6B72",
    "no_condition": "#2C4B7C",
    "small_latent": "#8B0000",
    "grid": "#D3D3D3",
    "text": "#000000",
    "spine": "#000000",
    "edge": "#000000",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 15,
    "figure.titlesize": 17,
})

EXPORT_PNG = True
EXPORT_PDF = True
DPI_PNG = 300

# =============================================================================
# HELPERS
# =============================================================================
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

def save_figure(fig, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=DPI_PNG)
    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    else:
        ax.grid(True, color=COLORS["grid"], linewidth=0.9)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.9)
        ax.spines[side].set_color(COLORS["spine"])

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=1.1, colors=COLORS["text"])

def add_panel_label_fig(fig, ax, label: str, x_shift: float = -0.006, y_shift: float = 0.002):
    # Much closer to the true panel corner
    bbox = ax.get_position()
    fig.text(
        bbox.x0 + x_shift,
        bbox.y1 + y_shift,
        label,
        ha="right",
        va="bottom",
        fontsize=22,
        fontweight="bold",
        color=COLORS["text"],
    )

def add_shared_legend(fig, model_order: List[str], y: float = 0.012):
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[m], edgecolor=COLORS["edge"], linewidth=0.8)
        for m in model_order
    ]
    labels = [MODEL_LABELS[m] for m in model_order]
    leg = fig.legend(
        handles,
        labels,
        ncol=6,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        frameon=True,
        fontsize=15,
        edgecolor=COLORS["edge"],
        handlelength=1.6,
        columnspacing=1.1,
    )
    leg.get_frame().set_linewidth(1.1)
    leg.get_frame().set_facecolor("white")

def get_metric_summary(agg_df: pd.DataFrame, model_tag: str, metric: str) -> Dict[str, float]:
    x = agg_df.loc[(agg_df["model_tag"] == model_tag) & (agg_df["metric"] == metric)]
    if len(x) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    r = x.iloc[0]
    return {
        "mean": float(r["mean"]),
        "ci_low": float(r["ci_low"]),
        "ci_high": float(r["ci_high"]),
    }

def build_metric_table(agg_df: pd.DataFrame, metric_names: List[str], model_order: List[str]) -> pd.DataFrame:
    rows = []
    for model_tag in model_order:
        row = {"model_tag": model_tag}
        for metric in metric_names:
            s = get_metric_summary(agg_df, model_tag, metric)
            row[f"{metric}_mean"] = s["mean"]
            row[f"{metric}_ci_low"] = s["ci_low"]
            row[f"{metric}_ci_high"] = s["ci_high"]
        rows.append(row)
    return pd.DataFrame(rows)

def draw_bar_with_ci(
    ax,
    values_df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    ylim: Optional[Tuple[float, float]] = None,
    annotate: bool = True,
    value_fmt: str = "{:.2f}",
    model_order: Optional[List[str]] = None,
    width: float = 0.78,
    title_pad: float = 12,
    annotate_mask: Optional[List[bool]] = None,
):
    if model_order is None:
        model_order = values_df["model_tag"].tolist()

    plot_df = values_df.set_index("model_tag").loc[model_order].reset_index()

    xs = np.arange(len(plot_df))
    means = plot_df[f"{metric}_mean"].to_numpy(dtype=float)
    ci_low = plot_df[f"{metric}_ci_low"].to_numpy(dtype=float)
    ci_high = plot_df[f"{metric}_ci_high"].to_numpy(dtype=float)
    colors = [COLORS[m] for m in plot_df["model_tag"]]

    ax.bar(
        xs,
        means,
        color=colors,
        edgecolor=COLORS["edge"],
        linewidth=0.75,
        width=width,
    )

    yerr = np.vstack([means - ci_low, ci_high - means])
    ax.errorbar(xs, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax.set_title(title, pad=title_pad)
    ax.set_ylabel(ylabel)

    ax.set_xticks(xs)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)

    if ylim is not None:
        ax.set_ylim(*ylim)

    style_axis(ax, "y")

    if annotate:
        y0, y1 = ax.get_ylim()
        dy = (y1 - y0) * 0.03
        if annotate_mask is None:
            annotate_mask = [True] * len(xs)
        for x, y, keep in zip(xs, means, annotate_mask):
            if keep and pd.notnull(y):
                ax.text(x, y + dy, value_fmt.format(y), ha="center", va="bottom", fontsize=12, color=COLORS["text"])

# =============================================================================
# LOAD DATA
# =============================================================================
agg_df = read_csv_required(TABLES_DIR / "step6_aggregated_summary.csv")
temp_df = read_csv_required(TABLES_DIR / "step6_temperature_sensitivity_summary.csv")

per_condition_path = TABLES_DIR / "step6_per_condition_fidelity_summary.csv"
histories_path = LOGS_DIR / "step6_all_training_histories.csv"

per_condition_df = pd.read_csv(per_condition_path) if per_condition_path.exists() else None
hist_df = pd.read_csv(histories_path) if histories_path.exists() else None

# =============================================================================
# FIGURE 1
# =============================================================================
def make_figure1():
    metric_names = [
        "gen_condition_fidelity",
        "gen_novelty_rate",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.2, 8.5))
    gs = GridSpec(
        2, 3, figure=fig,
        height_ratios=[1.0, 1.10],
        width_ratios=[2.05, 2.05, 1.12]
    )

    ax_a = fig.add_subplot(gs[0, :2])
    draw_bar_with_ci(
        ax=ax_a,
        values_df=values_df,
        metric="gen_condition_fidelity",
        ylabel="Fraction",
        title="Fidelity benchmark",
        ylim=(0.0, 0.95),
        width=0.78,
        title_pad=12,
    )

    ax_b = fig.add_subplot(gs[0, 2])
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_novelty_rate",
        ylabel="Fraction",
        title="Novelty",
        ylim=(0.0, 1.06),
        width=0.68,
        title_pad=14,
        annotate_mask=[True, False, False, False, False, False],
    )
    ax_b.text(0.63, 1.01, "All learned generators = 1.00",
              transform=ax_b.transAxes, ha="center", va="bottom",
              fontsize=10.5, color=COLORS["text"])

    ax_c = fig.add_subplot(gs[1, :])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Similarity to training set",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.78,
        title_pad=12,
    )
    ax_c.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_c.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    fig.subplots_adjust(left=0.055, right=0.985, top=0.93, bottom=0.15, wspace=0.30, hspace=0.42)

    # labels AFTER layout
    add_panel_label_fig(fig, ax_a, "a", x_shift=-0.006, y_shift=0.001)
    add_panel_label_fig(fig, ax_b, "b", x_shift=-0.004, y_shift=0.001)
    add_panel_label_fig(fig, ax_c, "c", x_shift=-0.006, y_shift=0.001)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    save_figure(fig, FIG_MAIN_DIR / "Fig6_step6_benchmark_main_redesign_v8")
    values_df.to_csv(SOURCE_DATA_DIR / "Fig6_step6_benchmark_main_redesign_v8_source_data.csv", index=False)

# =============================================================================
# FIGURE 2
# =============================================================================
def make_figure2():
    metric_names = [
        "gen_valid_rate",
        "gen_novelty_rate",
        "gen_uniqueness_rate",
        "gen_per_condition_surrogate_hit_mean",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.0, 8.5))
    gs = GridSpec(
        2, 2, figure=fig,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.10, 1.0]
    )

    ax_a = fig.add_subplot(gs[0, :])
    grouped_metrics = [
        ("gen_valid_rate", "Validity"),
        ("gen_novelty_rate", "Novelty"),
        ("gen_uniqueness_rate", "Uniqueness"),
    ]
    xs = np.arange(len(grouped_metrics))
    width = 0.11

    for i, model_tag in enumerate(MODEL_ORDER):
        offsets = xs + (i - (len(MODEL_ORDER) - 1) / 2) * width
        means, lows, highs = [], [], []
        for metric, _ in grouped_metrics:
            s = get_metric_summary(agg_df, model_tag, metric)
            means.append(s["mean"])
            lows.append(s["ci_low"])
            highs.append(s["ci_high"])
        means = np.array(means, dtype=float)
        lows = np.array(lows, dtype=float)
        highs = np.array(highs, dtype=float)

        ax_a.bar(offsets, means, width=width, color=COLORS[model_tag], edgecolor=COLORS["edge"], linewidth=0.7)
        yerr = np.vstack([means - lows, highs - means])
        ax_a.errorbar(offsets, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax_a.set_xticks(xs)
    ax_a.set_xticklabels([lab for _, lab in grouped_metrics])
    ax_a.set_ylabel("Fraction")
    ax_a.set_title("Generation quality", pad=12)
    ax_a.set_ylim(0.0, 1.05)
    style_axis(ax_a, "y")

    for j, (metric, _) in enumerate(grouped_metrics):
        s = get_metric_summary(agg_df, "retrieval", metric)
        if pd.notnull(s["mean"]):
            x = xs[j] - (len(MODEL_ORDER) - 1) / 2 * width
            ax_a.text(x, s["mean"] + 0.022, f"{s['mean']:.2f}", ha="center", va="bottom", fontsize=12, color=COLORS["text"])

    ax_b = fig.add_subplot(gs[1, 0])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Memorization profile",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.84,
        title_pad=12,
    )
    ax_b.text(0.995, 0.985, "Retrieval ref. = 1.00", transform=ax_b.transAxes,
              ha="right", va="top", fontsize=14.5, color=COLORS["text"])

    ax_c = fig.add_subplot(gs[1, 1])
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_per_condition_surrogate_hit_mean",
        ylabel="Per-condition hit rate",
        title="Condition heterogeneity",
        ylim=(0.0, 1.05),
        width=0.72,
        title_pad=12,
    )

    fig.subplots_adjust(left=0.055, right=0.985, top=0.93, bottom=0.15, wspace=0.22, hspace=0.38)

    add_panel_label_fig(fig, ax_a, "a", x_shift=-0.006, y_shift=0.001)
    add_panel_label_fig(fig, ax_b, "b", x_shift=-0.006, y_shift=0.001)
    add_panel_label_fig(fig, ax_c, "c", x_shift=-0.004, y_shift=0.001)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    values_df.to_csv(SOURCE_DATA_DIR / "Fig6_step6_generation_quality_redesign_v8_source_data.csv", index=False)
    save_figure(fig, FIG_MAIN_DIR / "Fig6_step6_generation_quality_redesign_v8")

# =============================================================================
# FIGURE 3
# =============================================================================
def make_figure3():
    if hist_df is None:
        return

    full_hist = hist_df.loc[hist_df["model_tag"] == "full_cvae"].copy()
    if len(full_hist) == 0:
        return

    curve = (
        full_hist.groupby("epoch", as_index=False)
        .agg(
            val_raw_kl=("val_raw_kl", "mean") if "val_raw_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_free_kl=("val_free_kl", "mean") if "val_free_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_active_units=("val_active_units_main", "mean") if "val_active_units_main" in full_hist.columns else ("val_active_units", "mean"),
            val_mean_latent_var=("val_mean_latent_var", "mean"),
        )
    )

    latent_csv_candidates = [
        STEP6_ROOT / "source_data_figures" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign_v2" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        SOURCE_DATA_DIR / "FigS_step6_latent_diagnostics_pca_source_data.csv",
    ]
    latent_df = None
    for p in latent_csv_candidates:
        if p.exists():
            latent_df = pd.read_csv(p)
            break

    fig = plt.figure(figsize=(15.8, 6.0))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1.22, 1.0, 1.02])

    ax_a = fig.add_subplot(gs[0, 0])
    if latent_df is not None and {"pc1", "pc2", "condition_id"}.issubset(latent_df.columns):
        sc = ax_a.scatter(latent_df["pc1"], latent_df["pc2"], c=latent_df["condition_id"], s=18, alpha=0.82, cmap="viridis")
        cbar = fig.colorbar(sc, ax=ax_a, fraction=0.038, pad=0.018)
        cbar.set_label("Condition ID", fontsize=11.5)
        cbar.ax.tick_params(labelsize=10)
    ax_a.set_xlabel("PC1")
    ax_a.set_ylabel("PC2")
    ax_a.set_title("Test-set latent means (PCA)", pad=10)
    style_axis(ax_a, "both")

    ax_b = fig.add_subplot(gs[0, 1])
    ax_b.plot(curve["epoch"], curve["val_raw_kl"], linewidth=2.0, color="#1F77B4", label="Val raw KL")
    ax_b.plot(curve["epoch"], curve["val_free_kl"], linewidth=2.0, color="#FF7F0E", label="Val free-bits KL")
    ax_b.set_xlabel("Epoch")
    ax_b.set_ylabel("KL")
    ax_b.set_title("Latent KL dynamics", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_c = fig.add_subplot(gs[0, 2])
    line1 = ax_c.plot(curve["epoch"], curve["val_active_units"], linewidth=2.0, color="#1F77B4", label="Active units")
    ax_c.set_xlabel("Epoch")
    ax_c.set_ylabel("Active units", color="#1F77B4", fontsize=12.5)
    ax_c.tick_params(axis="y", labelcolor="#1F77B4", labelsize=10.5)
    ax_c.set_title("Latent activity summary", pad=10)
    style_axis(ax_c, "y")

    ax_c2 = ax_c.twinx()
    line2 = ax_c2.plot(curve["epoch"], curve["val_mean_latent_var"], linewidth=2.0, color="#FF7F0E", label="Mean latent variance")
    ax_c2.set_ylabel("Mean latent variance", color="#FF7F0E", fontsize=12.5)
    ax_c2.tick_params(axis="y", labelcolor="#FF7F0E", labelsize=10.5)
    ax_c2.spines["top"].set_visible(False)
    ax_c2.spines["right"].set_linewidth(1.8)
    ax_c2.spines["right"].set_color(COLORS["spine"])

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax_c.legend(lines, labels, frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(left=0.055, right=0.985, top=0.91, bottom=0.12, wspace=0.30)

    add_panel_label_fig(fig, ax_a, "a", x_shift=-0.006, y_shift=0.0005)
    add_panel_label_fig(fig, ax_b, "b", x_shift=-0.006, y_shift=0.0005)
    add_panel_label_fig(fig, ax_c, "c", x_shift=-0.004, y_shift=0.0005)

    save_figure(fig, FIG_SUPP_DIR / "FigS_step6_latent_diagnostics_redesign_v8")

# =============================================================================
# FIGURE 4
# =============================================================================
def make_figure4():
    if temp_df is None or len(temp_df) == 0:
        return

    fig = plt.figure(figsize=(14.8, 5.8))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.05, 1.0])

    ax_a = fig.add_subplot(gs[0, 0])
    mean_temp = (
        temp_df.groupby(["model_tag", "temperature"], as_index=False)
        .agg(
            condition_fidelity=("condition_fidelity", "mean")
            if "condition_fidelity" in temp_df.columns
            else ("classifier_condition_fidelity", "mean")
        )
    )
    fidelity_col = "condition_fidelity" if "condition_fidelity" in mean_temp.columns else "classifier_condition_fidelity"

    keep_models = ["retrieval", "full_cvae", "conditional_gru_lm", "no_condition"]
    for model_tag in keep_models:
        part = mean_temp.loc[mean_temp["model_tag"] == model_tag]
        if len(part) == 0:
            continue
        ax_a.plot(part["temperature"], part[fidelity_col], marker="o", linewidth=2.1, markersize=7.5,
                  color=COLORS[model_tag], label=MODEL_LABELS[model_tag])

    ax_a.set_xlabel("Sampling temperature")
    ax_a.set_ylabel("Condition fidelity")
    ax_a.set_ylim(0.0, 1.0)
    ax_a.set_title("Decoding temperature sensitivity", pad=10)
    style_axis(ax_a, "y")
    ax_a.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.82, 0.98), fontsize=12.5)

    ax_b = fig.add_subplot(gs[0, 1])
    qa_cols = {
        "valid_rate": "Valid rate",
        "low_complexity_rate": "Low-complexity rate",
        "exact_train_overlap_rate": "Exact train overlap",
    }
    qa_mean = temp_df.groupby("model_tag", as_index=False)[list(qa_cols.keys())].mean()
    qa_mean = qa_mean.set_index("model_tag").loc[MODEL_ORDER].reset_index()

    xs = np.arange(len(qa_mean))
    for col, label in qa_cols.items():
        ax_b.plot(xs, qa_mean[col], marker="o", linewidth=2.0, markersize=7.5, label=label)

    ax_b.set_xticks(xs)
    ax_b.set_xticklabels([MODEL_LABELS[m] for m in qa_mean["model_tag"]], rotation=15, ha="right")
    ax_b.set_ylabel("Rate")
    ax_b.set_title("Generation-time QA summary", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(left=0.055, right=0.985, top=0.90, bottom=0.20, wspace=0.16)

    add_panel_label_fig(fig, ax_a, "a", x_shift=-0.006, y_shift=0.0005)
    add_panel_label_fig(fig, ax_b, "b", x_shift=-0.004, y_shift=0.0005)

    save_figure(fig, FIG_SUPP_DIR / "FigS_step6_temperature_and_qa_redesign_v8")

# =============================================================================
# RUN
# =============================================================================
if __name__ == "__main__":
    print("Generating redesigned Step 6 figures v8 ...")
    make_figure1()
    make_figure2()
    make_figure3()
    make_figure4()
    print("Done.")
    print(f"Main figures saved to: {FIG_MAIN_DIR}")
    print(f"Supplementary figures saved to: {FIG_SUPP_DIR}")

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# =============================================================================
# PATHS
# =============================================================================
STEP6_ROOT = Path("/home/data3/Moe/nature_computational2/step6_v5").resolve()

TABLES_DIR = STEP6_ROOT / "tables"
LOGS_DIR = STEP6_ROOT / "logs"

FIG_MAIN_DIR = STEP6_ROOT / "figures_main_redesign_v9"
FIG_SUPP_DIR = STEP6_ROOT / "figures_supplementary_redesign_v9"
SOURCE_DATA_DIR = STEP6_ROOT / "source_data_figures_redesign_v9"

FIG_MAIN_DIR.mkdir(parents=True, exist_ok=True)
FIG_SUPP_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# STYLE
# =============================================================================
MODEL_ORDER = [
    "retrieval",
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "retrieval": "Retrieval",
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

COLORS = {
    "retrieval": "#B4C5E4",
    "full_cvae": "#C0001A",
    "conditional_gru_lm": "#FF1A1A",
    "random_condition": "#1F6B72",
    "no_condition": "#2C4B7C",
    "small_latent": "#8B0000",
    "grid": "#D3D3D3",
    "text": "#000000",
    "spine": "#000000",
    "edge": "#000000",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 15,
    "figure.titlesize": 17,
})

EXPORT_PNG = True
EXPORT_PDF = True
DPI_PNG = 300

# =============================================================================
# HELPERS
# =============================================================================
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

def save_figure(fig, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=DPI_PNG)
    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    else:
        ax.grid(True, color=COLORS["grid"], linewidth=0.9)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.9)
        ax.spines[side].set_color(COLORS["spine"])

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=1.1, colors=COLORS["text"])

def add_panel_label_aligned_to_ylabel(
    fig,
    ax,
    label: str,
    x_extra: float = -0.006,
    y_extra: float = 0.002,
    fontsize: int = 22,
):
    """
    Anchor panel label to the left edge of the y-axis label area, not just the axes box.
    Call AFTER fig.subplots_adjust(...).
    """
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()

    ax_box = ax.get_position()
    ylab_box = ax.yaxis.label.get_window_extent(renderer=renderer)
    ylab_fig = inv.transform([[ylab_box.x0, ylab_box.y1]])[0]

    x = ylab_fig[0] + x_extra
    y = ax_box.y1 + y_extra

    fig.text(
        x, y, label,
        ha="right", va="bottom",
        fontsize=fontsize, fontweight="bold", color=COLORS["text"]
    )

def add_panel_label_box_anchor(
    fig,
    ax,
    label: str,
    x_extra: float = -0.004,
    y_extra: float = 0.002,
    fontsize: int = 22,
):
    """
    Use for panels where ylabel anchoring is not desired.
    """
    ax_box = ax.get_position()
    fig.text(
        ax_box.x0 + x_extra,
        ax_box.y1 + y_extra,
        label,
        ha="right", va="bottom",
        fontsize=fontsize, fontweight="bold", color=COLORS["text"]
    )

def add_shared_legend(fig, model_order: List[str], y: float = 0.012):
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[m], edgecolor=COLORS["edge"], linewidth=0.8)
        for m in model_order
    ]
    labels = [MODEL_LABELS[m] for m in model_order]
    leg = fig.legend(
        handles,
        labels,
        ncol=6,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        frameon=True,
        fontsize=15,
        edgecolor=COLORS["edge"],
        handlelength=1.6,
        columnspacing=1.1,
    )
    leg.get_frame().set_linewidth(1.1)
    leg.get_frame().set_facecolor("white")

def get_metric_summary(agg_df: pd.DataFrame, model_tag: str, metric: str) -> Dict[str, float]:
    x = agg_df.loc[(agg_df["model_tag"] == model_tag) & (agg_df["metric"] == metric)]
    if len(x) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    r = x.iloc[0]
    return {
        "mean": float(r["mean"]),
        "ci_low": float(r["ci_low"]),
        "ci_high": float(r["ci_high"]),
    }

def build_metric_table(agg_df: pd.DataFrame, metric_names: List[str], model_order: List[str]) -> pd.DataFrame:
    rows = []
    for model_tag in model_order:
        row = {"model_tag": model_tag}
        for metric in metric_names:
            s = get_metric_summary(agg_df, model_tag, metric)
            row[f"{metric}_mean"] = s["mean"]
            row[f"{metric}_ci_low"] = s["ci_low"]
            row[f"{metric}_ci_high"] = s["ci_high"]
        rows.append(row)
    return pd.DataFrame(rows)

def draw_bar_with_ci(
    ax,
    values_df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    ylim: Optional[Tuple[float, float]] = None,
    annotate: bool = True,
    value_fmt: str = "{:.2f}",
    model_order: Optional[List[str]] = None,
    width: float = 0.78,
    title_pad: float = 12,
    annotate_mask: Optional[List[bool]] = None,
):
    if model_order is None:
        model_order = values_df["model_tag"].tolist()

    plot_df = values_df.set_index("model_tag").loc[model_order].reset_index()

    xs = np.arange(len(plot_df))
    means = plot_df[f"{metric}_mean"].to_numpy(dtype=float)
    ci_low = plot_df[f"{metric}_ci_low"].to_numpy(dtype=float)
    ci_high = plot_df[f"{metric}_ci_high"].to_numpy(dtype=float)
    colors = [COLORS[m] for m in plot_df["model_tag"]]

    ax.bar(
        xs,
        means,
        color=colors,
        edgecolor=COLORS["edge"],
        linewidth=0.75,
        width=width,
    )

    yerr = np.vstack([means - ci_low, ci_high - means])
    ax.errorbar(xs, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax.set_title(title, pad=title_pad)
    ax.set_ylabel(ylabel)

    ax.set_xticks(xs)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)

    if ylim is not None:
        ax.set_ylim(*ylim)

    style_axis(ax, "y")

    if annotate:
        y0, y1 = ax.get_ylim()
        dy = (y1 - y0) * 0.03
        if annotate_mask is None:
            annotate_mask = [True] * len(xs)
        for x, y, keep in zip(xs, means, annotate_mask):
            if keep and pd.notnull(y):
                ax.text(x, y + dy, value_fmt.format(y), ha="center", va="bottom", fontsize=12, color=COLORS["text"])

# =============================================================================
# LOAD DATA
# =============================================================================
agg_df = read_csv_required(TABLES_DIR / "step6_aggregated_summary.csv")
temp_df = read_csv_required(TABLES_DIR / "step6_temperature_sensitivity_summary.csv")

per_condition_path = TABLES_DIR / "step6_per_condition_fidelity_summary.csv"
histories_path = LOGS_DIR / "step6_all_training_histories.csv"

per_condition_df = pd.read_csv(per_condition_path) if per_condition_path.exists() else None
hist_df = pd.read_csv(histories_path) if histories_path.exists() else None

# =============================================================================
# FIGURE 6
# =============================================================================
def make_figure6():
    metric_names = [
        "gen_condition_fidelity",
        "gen_novelty_rate",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.2, 8.5))
    gs = GridSpec(
        2, 3, figure=fig,
        height_ratios=[1.0, 1.10],
        width_ratios=[2.05, 2.05, 1.12]
    )

    ax_a = fig.add_subplot(gs[0, :2])
    draw_bar_with_ci(
        ax=ax_a,
        values_df=values_df,
        metric="gen_condition_fidelity",
        ylabel="Fraction",
        title="Fidelity benchmark",
        ylim=(0.0, 0.95),
        width=0.78,
        title_pad=12,
    )

    ax_b = fig.add_subplot(gs[0, 2])
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_novelty_rate",
        ylabel="Fraction",
        title="Novelty",
        ylim=(0.0, 1.06),
        width=0.68,
        title_pad=14,
        annotate_mask=[True, False, False, False, False, False],
    )
    ax_b.text(
        0.63, 1.01, "All learned generators = 1.00",
        transform=ax_b.transAxes,
        ha="center", va="bottom",
        fontsize=10.5, color=COLORS["text"]
    )

    ax_c = fig.add_subplot(gs[1, :])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Similarity to training set",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.78,
        title_pad=12,
    )
    ax_c.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_c.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    fig.subplots_adjust(left=0.055, right=0.985, top=0.93, bottom=0.15, wspace=0.30, hspace=0.42)

    # letters aligned to y-axis labels
    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_c, "c", x_extra=-0.006, y_extra=0.001)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    save_figure(fig, FIG_MAIN_DIR / "Fig6_step6_benchmark_main_redesign_v9")
    values_df.to_csv(SOURCE_DATA_DIR / "Fig6_step6_benchmark_main_redesign_v9_source_data.csv", index=False)

# =============================================================================
# FIGURE 7
# =============================================================================
def make_figure7():
    metric_names = [
        "gen_valid_rate",
        "gen_novelty_rate",
        "gen_uniqueness_rate",
        "gen_per_condition_surrogate_hit_mean",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.0, 8.5))
    gs = GridSpec(
        2, 2, figure=fig,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.10, 1.0]
    )

    ax_a = fig.add_subplot(gs[0, :])
    grouped_metrics = [
        ("gen_valid_rate", "Validity"),
        ("gen_novelty_rate", "Novelty"),
        ("gen_uniqueness_rate", "Uniqueness"),
    ]
    xs = np.arange(len(grouped_metrics))
    width = 0.11

    for i, model_tag in enumerate(MODEL_ORDER):
        offsets = xs + (i - (len(MODEL_ORDER) - 1) / 2) * width
        means, lows, highs = [], [], []
        for metric, _ in grouped_metrics:
            s = get_metric_summary(agg_df, model_tag, metric)
            means.append(s["mean"])
            lows.append(s["ci_low"])
            highs.append(s["ci_high"])
        means = np.array(means, dtype=float)
        lows = np.array(lows, dtype=float)
        highs = np.array(highs, dtype=float)

        ax_a.bar(offsets, means, width=width, color=COLORS[model_tag], edgecolor=COLORS["edge"], linewidth=0.7)
        yerr = np.vstack([means - lows, highs - means])
        ax_a.errorbar(offsets, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax_a.set_xticks(xs)
    ax_a.set_xticklabels([lab for _, lab in grouped_metrics])
    ax_a.set_ylabel("Fraction")
    ax_a.set_title("Generation quality", pad=12)
    ax_a.set_ylim(0.0, 1.05)
    style_axis(ax_a, "y")

    for j, (metric, _) in enumerate(grouped_metrics):
        s = get_metric_summary(agg_df, "retrieval", metric)
        if pd.notnull(s["mean"]):
            x = xs[j] - (len(MODEL_ORDER) - 1) / 2 * width
            ax_a.text(x, s["mean"] + 0.022, f"{s['mean']:.2f}", ha="center", va="bottom", fontsize=12, color=COLORS["text"])

    ax_b = fig.add_subplot(gs[1, 0])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Memorization profile",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.84,
        title_pad=12,
    )
    ax_b.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_b.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    ax_c = fig.add_subplot(gs[1, 1])
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_per_condition_surrogate_hit_mean",
        ylabel="Per-condition hit rate",
        title="Condition heterogeneity",
        ylim=(0.0, 1.05),
        width=0.72,
        title_pad=12,
    )

    fig.subplots_adjust(left=0.055, right=0.985, top=0.93, bottom=0.15, wspace=0.22, hspace=0.38)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_c, "c", x_extra=-0.006, y_extra=0.001)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    values_df.to_csv(SOURCE_DATA_DIR / "Fig7_step6_generation_quality_redesign_v9_source_data.csv", index=False)
    save_figure(fig, FIG_MAIN_DIR / "Fig7_step6_generation_quality_redesign_v9")

# =============================================================================
# SUPP FIG 2.6.1
# =============================================================================
def make_supp_261():
    if hist_df is None:
        return

    full_hist = hist_df.loc[hist_df["model_tag"] == "full_cvae"].copy()
    if len(full_hist) == 0:
        return

    curve = (
        full_hist.groupby("epoch", as_index=False)
        .agg(
            val_raw_kl=("val_raw_kl", "mean") if "val_raw_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_free_kl=("val_free_kl", "mean") if "val_free_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_active_units=("val_active_units_main", "mean") if "val_active_units_main" in full_hist.columns else ("val_active_units", "mean"),
            val_mean_latent_var=("val_mean_latent_var", "mean"),
        )
    )

    latent_csv_candidates = [
        STEP6_ROOT / "source_data_figures" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign_v2" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        SOURCE_DATA_DIR / "FigS_step6_latent_diagnostics_pca_source_data.csv",
    ]
    latent_df = None
    for p in latent_csv_candidates:
        if p.exists():
            latent_df = pd.read_csv(p)
            break

    fig = plt.figure(figsize=(15.8, 6.0))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1.22, 1.0, 1.02])

    ax_a = fig.add_subplot(gs[0, 0])
    if latent_df is not None and {"pc1", "pc2", "condition_id"}.issubset(latent_df.columns):
        sc = ax_a.scatter(latent_df["pc1"], latent_df["pc2"], c=latent_df["condition_id"], s=18, alpha=0.82, cmap="viridis")
        cbar = fig.colorbar(sc, ax=ax_a, fraction=0.038, pad=0.018)
        cbar.set_label("Condition ID", fontsize=11.5)
        cbar.ax.tick_params(labelsize=10)
    ax_a.set_xlabel("PC1")
    ax_a.set_ylabel("PC2")
    ax_a.set_title("Test-set latent means (PCA)", pad=10)
    style_axis(ax_a, "both")

    ax_b = fig.add_subplot(gs[0, 1])
    ax_b.plot(curve["epoch"], curve["val_raw_kl"], linewidth=2.0, color="#1F77B4", label="Val raw KL")
    ax_b.plot(curve["epoch"], curve["val_free_kl"], linewidth=2.0, color="#FF7F0E", label="Val free-bits KL")
    ax_b.set_xlabel("Epoch")
    ax_b.set_ylabel("KL")
    ax_b.set_title("Latent KL dynamics", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_c = fig.add_subplot(gs[0, 2])
    line1 = ax_c.plot(curve["epoch"], curve["val_active_units"], linewidth=2.0, color="#1F77B4", label="Active units")
    ax_c.set_xlabel("Epoch")
    ax_c.set_ylabel("Active units", color="#1F77B4", fontsize=12.5)
    ax_c.tick_params(axis="y", labelcolor="#1F77B4", labelsize=10.5)
    ax_c.set_title("Latent activity summary", pad=10)
    style_axis(ax_c, "y")

    ax_c2 = ax_c.twinx()
    line2 = ax_c2.plot(curve["epoch"], curve["val_mean_latent_var"], linewidth=2.0, color="#FF7F0E", label="Mean latent variance")
    ax_c2.set_ylabel("Mean latent variance", color="#FF7F0E", fontsize=12.5)
    ax_c2.tick_params(axis="y", labelcolor="#FF7F0E", labelsize=10.5)
    ax_c2.spines["top"].set_visible(False)
    ax_c2.spines["right"].set_linewidth(1.8)
    ax_c2.spines["right"].set_color(COLORS["spine"])

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax_c.legend(lines, labels, frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(left=0.055, right=0.985, top=0.91, bottom=0.12, wspace=0.30)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_c, "c", x_extra=-0.006, y_extra=0.001)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_2_6_1_redesign_v9")

# =============================================================================
# SUPP FIG 2.6.2
# =============================================================================
def make_supp_262():
    if temp_df is None or len(temp_df) == 0:
        return

    fig = plt.figure(figsize=(14.8, 5.8))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.05, 1.0])

    ax_a = fig.add_subplot(gs[0, 0])
    mean_temp = (
        temp_df.groupby(["model_tag", "temperature"], as_index=False)
        .agg(
            condition_fidelity=("condition_fidelity", "mean")
            if "condition_fidelity" in temp_df.columns
            else ("classifier_condition_fidelity", "mean")
        )
    )
    fidelity_col = "condition_fidelity" if "condition_fidelity" in mean_temp.columns else "classifier_condition_fidelity"

    keep_models = ["retrieval", "full_cvae", "conditional_gru_lm", "no_condition"]
    for model_tag in keep_models:
        part = mean_temp.loc[mean_temp["model_tag"] == model_tag]
        if len(part) == 0:
            continue
        ax_a.plot(
            part["temperature"], part[fidelity_col],
            marker="o", linewidth=2.1, markersize=7.5,
            color=COLORS[model_tag], label=MODEL_LABELS[model_tag]
        )

    ax_a.set_xlabel("Sampling temperature")
    ax_a.set_ylabel("Condition fidelity")
    ax_a.set_ylim(0.0, 1.0)
    ax_a.set_title("Decoding temperature sensitivity", pad=10)
    style_axis(ax_a, "y")
    ax_a.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_b = fig.add_subplot(gs[0, 1])
    qa_cols = {
        "valid_rate": "Valid rate",
        "low_complexity_rate": "Low-complexity rate",
        "exact_train_overlap_rate": "Exact train overlap",
    }
    qa_mean = temp_df.groupby("model_tag", as_index=False)[list(qa_cols.keys())].mean()
    qa_mean = qa_mean.set_index("model_tag").loc[MODEL_ORDER].reset_index()

    xs = np.arange(len(qa_mean))
    for col, label in qa_cols.items():
        ax_b.plot(xs, qa_mean[col], marker="o", linewidth=2.0, markersize=7.5, label=label)

    ax_b.set_xticks(xs)
    ax_b.set_xticklabels([MODEL_LABELS[m] for m in qa_mean["model_tag"]], rotation=15, ha="right")
    ax_b.set_ylabel("Rate")
    ax_b.set_title("Generation-time QA summary", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(left=0.055, right=0.985, top=0.90, bottom=0.20, wspace=0.16)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_2_6_2_redesign_v9")

# =============================================================================
# RUN
# =============================================================================
if __name__ == "__main__":
    print("Generating redesigned Step 6 figures v9 ...")
    make_figure6()
    make_figure7()
    make_supp_261()
    make_supp_262()
    print("Done.")
    print(f"Main figures saved to: {FIG_MAIN_DIR}")
    print(f"Supplementary figures saved to: {FIG_SUPP_DIR}")

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# =============================================================================
# PATHS
# =============================================================================
STEP6_ROOT = Path("/home/data3/Moe/nature_computational2/step6_v5").resolve()

TABLES_DIR = STEP6_ROOT / "tables"
LOGS_DIR = STEP6_ROOT / "logs"

FIG_MAIN_DIR = STEP6_ROOT / "figures_main_redesign_v9"
FIG_SUPP_DIR = STEP6_ROOT / "figures_supplementary_redesign_v9"
FIG_TIFF_DIR = STEP6_ROOT / "figures_tiff_redesign_v9"
SOURCE_DATA_DIR = STEP6_ROOT / "source_data_figures_redesign_v9"

FIG_MAIN_DIR.mkdir(parents=True, exist_ok=True)
FIG_SUPP_DIR.mkdir(parents=True, exist_ok=True)
FIG_TIFF_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# STYLE
# =============================================================================
MODEL_ORDER = [
    "retrieval",
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "retrieval": "Retrieval",
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

COLORS = {
    "retrieval": "#B4C5E4",
    "full_cvae": "#C0001A",
    "conditional_gru_lm": "#FF1A1A",
    "random_condition": "#1F6B72",
    "no_condition": "#2C4B7C",
    "small_latent": "#8B0000",
    "grid": "#D3D3D3",
    "text": "#000000",
    "spine": "#000000",
    "edge": "#000000",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 15,
    "figure.titlesize": 17,
})

EXPORT_PNG = True
EXPORT_PDF = True
EXPORT_TIFF = True
DPI_PNG = 300
DPI_TIFF = 600
TIFF_COMPRESSION = "tiff_lzw"

# =============================================================================
# HELPERS
# =============================================================================
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

def save_figure(fig, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(
            out_base.with_suffix(".pdf"),
            bbox_inches="tight"
        )

    if EXPORT_PNG:
        fig.savefig(
            out_base.with_suffix(".png"),
            bbox_inches="tight",
            dpi=DPI_PNG
        )

    if EXPORT_TIFF:
        tiff_base = FIG_TIFF_DIR / out_base.name
        try:
            fig.savefig(
                tiff_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
                pil_kwargs={"compression": TIFF_COMPRESSION}
            )
        except TypeError:
            fig.savefig(
                tiff_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF
            )

    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    else:
        ax.grid(True, color=COLORS["grid"], linewidth=0.9)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.9)
        ax.spines[side].set_color(COLORS["spine"])

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=1.1, colors=COLORS["text"])

def add_panel_label_aligned_to_ylabel(
    fig,
    ax,
    label: str,
    x_extra: float = -0.006,
    y_extra: float = 0.002,
    fontsize: int = 22,
):
    """
    Anchor panel label to the left edge of the y-axis label area, not just the axes box.
    Call AFTER fig.subplots_adjust(...).
    """
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()

    ax_box = ax.get_position()
    ylab_box = ax.yaxis.label.get_window_extent(renderer=renderer)
    ylab_fig = inv.transform([[ylab_box.x0, ylab_box.y1]])[0]

    x = ylab_fig[0] + x_extra
    y = ax_box.y1 + y_extra

    fig.text(
        x, y, label,
        ha="right", va="bottom",
        fontsize=fontsize, fontweight="bold", color=COLORS["text"]
    )

def add_panel_label_box_anchor(
    fig,
    ax,
    label: str,
    x_extra: float = -0.004,
    y_extra: float = 0.002,
    fontsize: int = 22,
):
    """
    Use for panels where ylabel anchoring is not desired.
    """
    ax_box = ax.get_position()
    fig.text(
        ax_box.x0 + x_extra,
        ax_box.y1 + y_extra,
        label,
        ha="right", va="bottom",
        fontsize=fontsize, fontweight="bold", color=COLORS["text"]
    )

def add_shared_legend(fig, model_order: List[str], y: float = 0.012):
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[m], edgecolor=COLORS["edge"], linewidth=0.8)
        for m in model_order
    ]
    labels = [MODEL_LABELS[m] for m in model_order]
    leg = fig.legend(
        handles,
        labels,
        ncol=6,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        frameon=True,
        fontsize=15,
        edgecolor=COLORS["edge"],
        handlelength=1.6,
        columnspacing=1.1,
    )
    leg.get_frame().set_linewidth(1.1)
    leg.get_frame().set_facecolor("white")

def get_metric_summary(agg_df: pd.DataFrame, model_tag: str, metric: str) -> Dict[str, float]:
    x = agg_df.loc[(agg_df["model_tag"] == model_tag) & (agg_df["metric"] == metric)]
    if len(x) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    r = x.iloc[0]
    return {
        "mean": float(r["mean"]),
        "ci_low": float(r["ci_low"]),
        "ci_high": float(r["ci_high"]),
    }

def build_metric_table(agg_df: pd.DataFrame, metric_names: List[str], model_order: List[str]) -> pd.DataFrame:
    rows = []
    for model_tag in model_order:
        row = {"model_tag": model_tag}
        for metric in metric_names:
            s = get_metric_summary(agg_df, model_tag, metric)
            row[f"{metric}_mean"] = s["mean"]
            row[f"{metric}_ci_low"] = s["ci_low"]
            row[f"{metric}_ci_high"] = s["ci_high"]
        rows.append(row)
    return pd.DataFrame(rows)

def draw_bar_with_ci(
    ax,
    values_df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    ylim: Optional[Tuple[float, float]] = None,
    annotate: bool = True,
    value_fmt: str = "{:.2f}",
    model_order: Optional[List[str]] = None,
    width: float = 0.78,
    title_pad: float = 12,
    annotate_mask: Optional[List[bool]] = None,
):
    if model_order is None:
        model_order = values_df["model_tag"].tolist()

    plot_df = values_df.set_index("model_tag").loc[model_order].reset_index()

    xs = np.arange(len(plot_df))
    means = plot_df[f"{metric}_mean"].to_numpy(dtype=float)
    ci_low = plot_df[f"{metric}_ci_low"].to_numpy(dtype=float)
    ci_high = plot_df[f"{metric}_ci_high"].to_numpy(dtype=float)
    colors = [COLORS[m] for m in plot_df["model_tag"]]

    ax.bar(
        xs,
        means,
        color=colors,
        edgecolor=COLORS["edge"],
        linewidth=0.75,
        width=width,
    )

    yerr = np.vstack([means - ci_low, ci_high - means])
    ax.errorbar(xs, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax.set_title(title, pad=title_pad)
    ax.set_ylabel(ylabel)

    ax.set_xticks(xs)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)

    if ylim is not None:
        ax.set_ylim(*ylim)

    style_axis(ax, "y")

    if annotate:
        y0, y1 = ax.get_ylim()
        dy = (y1 - y0) * 0.03
        if annotate_mask is None:
            annotate_mask = [True] * len(xs)
        for x, y, keep in zip(xs, means, annotate_mask):
            if keep and pd.notnull(y):
                ax.text(x, y + dy, value_fmt.format(y), ha="center", va="bottom", fontsize=12, color=COLORS["text"])

# =============================================================================
# LOAD DATA
# =============================================================================
agg_df = read_csv_required(TABLES_DIR / "step6_aggregated_summary.csv")
temp_df = read_csv_required(TABLES_DIR / "step6_temperature_sensitivity_summary.csv")

per_condition_path = TABLES_DIR / "step6_per_condition_fidelity_summary.csv"
histories_path = LOGS_DIR / "step6_all_training_histories.csv"

per_condition_df = pd.read_csv(per_condition_path) if per_condition_path.exists() else None
hist_df = pd.read_csv(histories_path) if histories_path.exists() else None

# =============================================================================
# FIGURE 6
# =============================================================================
def make_figure6():
    metric_names = [
        "gen_condition_fidelity",
        "gen_novelty_rate",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.2, 8.5))
    gs = GridSpec(
        2, 3, figure=fig,
        height_ratios=[1.0, 1.10],
        width_ratios=[2.05, 2.05, 1.12]
    )

    ax_a = fig.add_subplot(gs[0, :2])
    draw_bar_with_ci(
        ax=ax_a,
        values_df=values_df,
        metric="gen_condition_fidelity",
        ylabel="Fraction",
        title="Fidelity benchmark",
        ylim=(0.0, 0.95),
        width=0.78,
        title_pad=12,
    )

    ax_b = fig.add_subplot(gs[0, 2])
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_novelty_rate",
        ylabel="Fraction",
        title="Novelty",
        ylim=(0.0, 1.06),
        width=0.68,
        title_pad=14,
        annotate_mask=[True, False, False, False, False, False],
    )
    ax_b.text(
        0.63, 1.01, "All learned generators = 1.00",
        transform=ax_b.transAxes,
        ha="center", va="bottom",
        fontsize=10.5, color=COLORS["text"]
    )

    ax_c = fig.add_subplot(gs[1, :])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Similarity to training set",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.78,
        title_pad=12,
    )
    ax_c.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_c.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    fig.subplots_adjust(left=0.055, right=0.985, top=0.93, bottom=0.15, wspace=0.30, hspace=0.42)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_c, "c", x_extra=-0.006, y_extra=0.001)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    save_figure(fig, FIG_MAIN_DIR / "Fig6_step6_benchmark_main_redesign_v9")
    values_df.to_csv(SOURCE_DATA_DIR / "Fig6_step6_benchmark_main_redesign_v9_source_data.csv", index=False)

# =============================================================================
# FIGURE 7
# =============================================================================
def make_figure7():
    metric_names = [
        "gen_valid_rate",
        "gen_novelty_rate",
        "gen_uniqueness_rate",
        "gen_per_condition_surrogate_hit_mean",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.0, 8.5))
    gs = GridSpec(
        2, 2, figure=fig,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.10, 1.0]
    )

    ax_a = fig.add_subplot(gs[0, :])
    grouped_metrics = [
        ("gen_valid_rate", "Validity"),
        ("gen_novelty_rate", "Novelty"),
        ("gen_uniqueness_rate", "Uniqueness"),
    ]
    xs = np.arange(len(grouped_metrics))
    width = 0.11

    for i, model_tag in enumerate(MODEL_ORDER):
        offsets = xs + (i - (len(MODEL_ORDER) - 1) / 2) * width
        means, lows, highs = [], [], []
        for metric, _ in grouped_metrics:
            s = get_metric_summary(agg_df, model_tag, metric)
            means.append(s["mean"])
            lows.append(s["ci_low"])
            highs.append(s["ci_high"])
        means = np.array(means, dtype=float)
        lows = np.array(lows, dtype=float)
        highs = np.array(highs, dtype=float)

        ax_a.bar(offsets, means, width=width, color=COLORS[model_tag], edgecolor=COLORS["edge"], linewidth=0.7)
        yerr = np.vstack([means - lows, highs - means])
        ax_a.errorbar(offsets, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax_a.set_xticks(xs)
    ax_a.set_xticklabels([lab for _, lab in grouped_metrics])
    ax_a.set_ylabel("Fraction")
    ax_a.set_title("Generation quality", pad=12)
    ax_a.set_ylim(0.0, 1.05)
    style_axis(ax_a, "y")

    for j, (metric, _) in enumerate(grouped_metrics):
        s = get_metric_summary(agg_df, "retrieval", metric)
        if pd.notnull(s["mean"]):
            x = xs[j] - (len(MODEL_ORDER) - 1) / 2 * width
            ax_a.text(x, s["mean"] + 0.022, f"{s['mean']:.2f}", ha="center", va="bottom", fontsize=12, color=COLORS["text"])

    ax_b = fig.add_subplot(gs[1, 0])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Memorization profile",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.84,
        title_pad=12,
    )
    ax_b.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_b.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    ax_c = fig.add_subplot(gs[1, 1])
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_per_condition_surrogate_hit_mean",
        ylabel="Per-condition hit rate",
        title="Condition heterogeneity",
        ylim=(0.0, 1.05),
        width=0.72,
        title_pad=12,
    )

    fig.subplots_adjust(left=0.055, right=0.985, top=0.93, bottom=0.15, wspace=0.22, hspace=0.38)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_c, "c", x_extra=-0.006, y_extra=0.001)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    values_df.to_csv(SOURCE_DATA_DIR / "Fig7_step6_generation_quality_redesign_v9_source_data.csv", index=False)
    save_figure(fig, FIG_MAIN_DIR / "Fig7_step6_generation_quality_redesign_v9")

# =============================================================================
# SUPP FIG 2.6.1
# =============================================================================
def make_supp_261():
    if hist_df is None:
        return

    full_hist = hist_df.loc[hist_df["model_tag"] == "full_cvae"].copy()
    if len(full_hist) == 0:
        return

    curve = (
        full_hist.groupby("epoch", as_index=False)
        .agg(
            val_raw_kl=("val_raw_kl", "mean") if "val_raw_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_free_kl=("val_free_kl", "mean") if "val_free_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_active_units=("val_active_units_main", "mean") if "val_active_units_main" in full_hist.columns else ("val_active_units", "mean"),
            val_mean_latent_var=("val_mean_latent_var", "mean"),
        )
    )

    latent_csv_candidates = [
        STEP6_ROOT / "source_data_figures" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign_v2" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        SOURCE_DATA_DIR / "FigS_step6_latent_diagnostics_pca_source_data.csv",
    ]
    latent_df = None
    for p in latent_csv_candidates:
        if p.exists():
            latent_df = pd.read_csv(p)
            break

    fig = plt.figure(figsize=(15.8, 6.0))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1.22, 1.0, 1.02])

    ax_a = fig.add_subplot(gs[0, 0])
    if latent_df is not None and {"pc1", "pc2", "condition_id"}.issubset(latent_df.columns):
        sc = ax_a.scatter(latent_df["pc1"], latent_df["pc2"], c=latent_df["condition_id"], s=18, alpha=0.82, cmap="viridis")
        cbar = fig.colorbar(sc, ax=ax_a, fraction=0.038, pad=0.018)
        cbar.set_label("Condition ID", fontsize=11.5)
        cbar.ax.tick_params(labelsize=10)
    ax_a.set_xlabel("PC1")
    ax_a.set_ylabel("PC2")
    ax_a.set_title("Test-set latent means (PCA)", pad=10)
    style_axis(ax_a, "both")

    ax_b = fig.add_subplot(gs[0, 1])
    ax_b.plot(curve["epoch"], curve["val_raw_kl"], linewidth=2.0, color="#1F77B4", label="Val raw KL")
    ax_b.plot(curve["epoch"], curve["val_free_kl"], linewidth=2.0, color="#FF7F0E", label="Val free-bits KL")
    ax_b.set_xlabel("Epoch")
    ax_b.set_ylabel("KL")
    ax_b.set_title("Latent KL dynamics", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_c = fig.add_subplot(gs[0, 2])
    line1 = ax_c.plot(curve["epoch"], curve["val_active_units"], linewidth=2.0, color="#1F77B4", label="Active units")
    ax_c.set_xlabel("Epoch")
    ax_c.set_ylabel("Active units", color="#1F77B4", fontsize=12.5)
    ax_c.tick_params(axis="y", labelcolor="#1F77B4", labelsize=10.5)
    ax_c.set_title("Latent activity summary", pad=10)
    style_axis(ax_c, "y")

    ax_c2 = ax_c.twinx()
    line2 = ax_c2.plot(curve["epoch"], curve["val_mean_latent_var"], linewidth=2.0, color="#FF7F0E", label="Mean latent variance")
    ax_c2.set_ylabel("Mean latent variance", color="#FF7F0E", fontsize=12.5)
    ax_c2.tick_params(axis="y", labelcolor="#FF7F0E", labelsize=10.5)
    ax_c2.spines["top"].set_visible(False)
    ax_c2.spines["right"].set_linewidth(1.8)
    ax_c2.spines["right"].set_color(COLORS["spine"])

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax_c.legend(lines, labels, frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(left=0.055, right=0.985, top=0.91, bottom=0.12, wspace=0.30)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_c, "c", x_extra=-0.006, y_extra=0.001)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_2_6_1_redesign_v9")

# =============================================================================
# SUPP FIG 2.6.2
# =============================================================================
def make_supp_262():
    if temp_df is None or len(temp_df) == 0:
        return

    fig = plt.figure(figsize=(14.8, 5.8))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.05, 1.0])

    ax_a = fig.add_subplot(gs[0, 0])
    mean_temp = (
        temp_df.groupby(["model_tag", "temperature"], as_index=False)
        .agg(
            condition_fidelity=("condition_fidelity", "mean")
            if "condition_fidelity" in temp_df.columns
            else ("classifier_condition_fidelity", "mean")
        )
    )
    fidelity_col = "condition_fidelity" if "condition_fidelity" in mean_temp.columns else "classifier_condition_fidelity"

    keep_models = ["retrieval", "full_cvae", "conditional_gru_lm", "no_condition"]
    for model_tag in keep_models:
        part = mean_temp.loc[mean_temp["model_tag"] == model_tag]
        if len(part) == 0:
            continue
        ax_a.plot(
            part["temperature"], part[fidelity_col],
            marker="o", linewidth=2.1, markersize=7.5,
            color=COLORS[model_tag], label=MODEL_LABELS[model_tag]
        )

    ax_a.set_xlabel("Sampling temperature")
    ax_a.set_ylabel("Condition fidelity")
    ax_a.set_ylim(0.0, 1.0)
    ax_a.set_title("Decoding temperature sensitivity", pad=10)
    style_axis(ax_a, "y")
    ax_a.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_b = fig.add_subplot(gs[0, 1])
    qa_cols = {
        "valid_rate": "Valid rate",
        "low_complexity_rate": "Low-complexity rate",
        "exact_train_overlap_rate": "Exact train overlap",
    }
    qa_mean = temp_df.groupby("model_tag", as_index=False)[list(qa_cols.keys())].mean()
    qa_mean = qa_mean.set_index("model_tag").loc[MODEL_ORDER].reset_index()

    xs = np.arange(len(qa_mean))
    for col, label in qa_cols.items():
        ax_b.plot(xs, qa_mean[col], marker="o", linewidth=2.0, markersize=7.5, label=label)

    ax_b.set_xticks(xs)
    ax_b.set_xticklabels([MODEL_LABELS[m] for m in qa_mean["model_tag"]], rotation=15, ha="right")
    ax_b.set_ylabel("Rate")
    ax_b.set_title("Generation-time QA summary", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(left=0.055, right=0.985, top=0.90, bottom=0.20, wspace=0.16)

    add_panel_label_aligned_to_ylabel(fig, ax_a, "a", x_extra=-0.006, y_extra=0.001)
    add_panel_label_aligned_to_ylabel(fig, ax_b, "b", x_extra=-0.006, y_extra=0.001)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_2_6_2_redesign_v9")

# =============================================================================
# RUN
# =============================================================================
if __name__ == "__main__":
    print("Generating redesigned Step 6 figures v9 ...")
    make_figure6()
    make_figure7()
    make_supp_261()
    make_supp_262()
    print("Done.")
    print(f"Main figures saved to: {FIG_MAIN_DIR}")
    print(f"Supplementary figures saved to: {FIG_SUPP_DIR}")
    print(f"TIFF figures saved to: {FIG_TIFF_DIR}")

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# =============================================================================
# PATHS
# =============================================================================
STEP6_ROOT = Path("/home/data3/Moe/nature_computational2/step6_v5").resolve()

TABLES_DIR = STEP6_ROOT / "tables"
LOGS_DIR = STEP6_ROOT / "logs"

FIG_MAIN_DIR = STEP6_ROOT / "figures_main_redesign_v10"
FIG_SUPP_DIR = STEP6_ROOT / "figures_supplementary_redesign_v10"
SOURCE_DATA_DIR = STEP6_ROOT / "source_data_figures_redesign_v10"

FIG_MAIN_DIR.mkdir(parents=True, exist_ok=True)
FIG_SUPP_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# STYLE
# =============================================================================
MODEL_ORDER = [
    "retrieval",
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "retrieval": "Retrieval",
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

COLORS = {
    "retrieval": "#B4C5E4",
    "full_cvae": "#C0001A",
    "conditional_gru_lm": "#FF1A1A",
    "random_condition": "#1F6B72",
    "no_condition": "#2C4B7C",
    "small_latent": "#8B0000",
    "grid": "#D3D3D3",
    "text": "#000000",
    "spine": "#000000",
    "edge": "#000000",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 15,
    "figure.titlesize": 17,
})

EXPORT_PNG = True
EXPORT_PDF = True
DPI_PNG = 300

# =============================================================================
# HELPERS
# =============================================================================
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

def save_figure(fig, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=DPI_PNG)
    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    else:
        ax.grid(True, color=COLORS["grid"], linewidth=0.9)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.9)
        ax.spines[side].set_color(COLORS["spine"])

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=1.1, colors=COLORS["text"])

def add_panel_label(
    ax,
    label: str,
    wide: bool = False,
    dx: float = 0.0,
    dy: float = 0.0,
    fontsize: int = 22,
):
    """
    Place panel label in axes coordinates so it stays attached to the subplot.
    This gives the reference-style alignment much more reliably than fig.text(...).
    """
    base_x = -0.065 if wide else -0.145
    base_y = 1.035

    ax.text(
        base_x + dx,
        base_y + dy,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        color=COLORS["text"],
        clip_on=False,
    )

def add_shared_legend(fig, model_order: List[str], y: float = 0.012):
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[m], edgecolor=COLORS["edge"], linewidth=0.8)
        for m in model_order
    ]
    labels = [MODEL_LABELS[m] for m in model_order]
    leg = fig.legend(
        handles,
        labels,
        ncol=6,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        frameon=True,
        fontsize=15,
        edgecolor=COLORS["edge"],
        handlelength=1.6,
        columnspacing=1.1,
    )
    leg.get_frame().set_linewidth(1.1)
    leg.get_frame().set_facecolor("white")

def get_metric_summary(agg_df: pd.DataFrame, model_tag: str, metric: str) -> Dict[str, float]:
    x = agg_df.loc[(agg_df["model_tag"] == model_tag) & (agg_df["metric"] == metric)]
    if len(x) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    r = x.iloc[0]
    return {
        "mean": float(r["mean"]),
        "ci_low": float(r["ci_low"]),
        "ci_high": float(r["ci_high"]),
    }

def build_metric_table(agg_df: pd.DataFrame, metric_names: List[str], model_order: List[str]) -> pd.DataFrame:
    rows = []
    for model_tag in model_order:
        row = {"model_tag": model_tag}
        for metric in metric_names:
            s = get_metric_summary(agg_df, model_tag, metric)
            row[f"{metric}_mean"] = s["mean"]
            row[f"{metric}_ci_low"] = s["ci_low"]
            row[f"{metric}_ci_high"] = s["ci_high"]
        rows.append(row)
    return pd.DataFrame(rows)

def draw_bar_with_ci(
    ax,
    values_df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    ylim: Optional[Tuple[float, float]] = None,
    annotate: bool = True,
    value_fmt: str = "{:.2f}",
    model_order: Optional[List[str]] = None,
    width: float = 0.78,
    title_pad: float = 12,
    annotate_mask: Optional[List[bool]] = None,
):
    if model_order is None:
        model_order = values_df["model_tag"].tolist()

    plot_df = values_df.set_index("model_tag").loc[model_order].reset_index()

    xs = np.arange(len(plot_df))
    means = plot_df[f"{metric}_mean"].to_numpy(dtype=float)
    ci_low = plot_df[f"{metric}_ci_low"].to_numpy(dtype=float)
    ci_high = plot_df[f"{metric}_ci_high"].to_numpy(dtype=float)
    colors = [COLORS[m] for m in plot_df["model_tag"]]

    ax.bar(
        xs,
        means,
        color=colors,
        edgecolor=COLORS["edge"],
        linewidth=0.75,
        width=width,
    )

    yerr = np.vstack([means - ci_low, ci_high - means])
    ax.errorbar(xs, means, yerr=yerr, fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9)

    ax.set_title(title, pad=title_pad)
    ax.set_ylabel(ylabel)

    ax.set_xticks(xs)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)

    if ylim is not None:
        ax.set_ylim(*ylim)

    style_axis(ax, "y")

    if annotate:
        y0, y1 = ax.get_ylim()
        dy_text = (y1 - y0) * 0.03
        if annotate_mask is None:
            annotate_mask = [True] * len(xs)
        for x, y, keep in zip(xs, means, annotate_mask):
            if keep and pd.notnull(y):
                ax.text(
                    x, y + dy_text, value_fmt.format(y),
                    ha="center", va="bottom", fontsize=12, color=COLORS["text"]
                )

# =============================================================================
# LOAD DATA
# =============================================================================
agg_df = read_csv_required(TABLES_DIR / "step6_aggregated_summary.csv")
temp_df = read_csv_required(TABLES_DIR / "step6_temperature_sensitivity_summary.csv")

per_condition_path = TABLES_DIR / "step6_per_condition_fidelity_summary.csv"
histories_path = LOGS_DIR / "step6_all_training_histories.csv"

per_condition_df = pd.read_csv(per_condition_path) if per_condition_path.exists() else None
hist_df = pd.read_csv(histories_path) if histories_path.exists() else None

# =============================================================================
# FIGURE 6
# =============================================================================
def make_figure6():
    metric_names = [
        "gen_condition_fidelity",
        "gen_novelty_rate",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.2, 8.5))
    gs = GridSpec(
        2, 3, figure=fig,
        height_ratios=[1.0, 1.10],
        width_ratios=[2.05, 2.05, 1.12]
    )

    ax_a = fig.add_subplot(gs[0, :2])
    draw_bar_with_ci(
        ax=ax_a,
        values_df=values_df,
        metric="gen_condition_fidelity",
        ylabel="Fraction",
        title="Fidelity benchmark",
        ylim=(0.0, 0.95),
        width=0.78,
        title_pad=12,
    )

    ax_b = fig.add_subplot(gs[0, 2])
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_novelty_rate",
        ylabel="Fraction",
        title="Novelty",
        ylim=(0.0, 1.06),
        width=0.68,
        title_pad=14,
        annotate_mask=[True, False, False, False, False, False],
    )
    ax_b.text(
        0.63, 1.01, "All learned generators = 1.00",
        transform=ax_b.transAxes,
        ha="center", va="bottom",
        fontsize=10.5, color=COLORS["text"]
    )

    ax_c = fig.add_subplot(gs[1, :])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Similarity to training set",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.78,
        title_pad=12,
    )
    ax_c.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_c.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    fig.subplots_adjust(
        left=0.085, right=0.985, top=0.93, bottom=0.15,
        wspace=0.30, hspace=0.42
    )

    add_panel_label(ax_a, "a", wide=False)
    add_panel_label(ax_b, "b", wide=False)
    add_panel_label(ax_c, "c", wide=True)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    save_figure(fig, FIG_MAIN_DIR / "Fig6_step6_benchmark_main_redesign_v10")
    values_df.to_csv(
        SOURCE_DATA_DIR / "Fig6_step6_benchmark_main_redesign_v10_source_data.csv",
        index=False
    )

# =============================================================================
# FIGURE 7
# =============================================================================
def make_figure7():
    metric_names = [
        "gen_valid_rate",
        "gen_novelty_rate",
        "gen_uniqueness_rate",
        "gen_per_condition_surrogate_hit_mean",
        "gen_nn_similarity_train_mean",
    ]
    values_df = build_metric_table(agg_df, metric_names, MODEL_ORDER)

    fig = plt.figure(figsize=(16.0, 8.5))
    gs = GridSpec(
        2, 2, figure=fig,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.10, 1.0]
    )

    ax_a = fig.add_subplot(gs[0, :])
    grouped_metrics = [
        ("gen_valid_rate", "Validity"),
        ("gen_novelty_rate", "Novelty"),
        ("gen_uniqueness_rate", "Uniqueness"),
    ]
    xs = np.arange(len(grouped_metrics))
    width = 0.11

    for i, model_tag in enumerate(MODEL_ORDER):
        offsets = xs + (i - (len(MODEL_ORDER) - 1) / 2) * width
        means, lows, highs = [], [], []
        for metric, _ in grouped_metrics:
            s = get_metric_summary(agg_df, model_tag, metric)
            means.append(s["mean"])
            lows.append(s["ci_low"])
            highs.append(s["ci_high"])

        means = np.array(means, dtype=float)
        lows = np.array(lows, dtype=float)
        highs = np.array(highs, dtype=float)

        ax_a.bar(
            offsets, means, width=width,
            color=COLORS[model_tag],
            edgecolor=COLORS["edge"],
            linewidth=0.7
        )
        yerr = np.vstack([means - lows, highs - means])
        ax_a.errorbar(
            offsets, means, yerr=yerr,
            fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9
        )

    ax_a.set_xticks(xs)
    ax_a.set_xticklabels([lab for _, lab in grouped_metrics])
    ax_a.set_ylabel("Fraction")
    ax_a.set_title("Generation quality", pad=12)
    ax_a.set_ylim(0.0, 1.05)
    style_axis(ax_a, "y")

    for j, (metric, _) in enumerate(grouped_metrics):
        s = get_metric_summary(agg_df, "retrieval", metric)
        if pd.notnull(s["mean"]):
            x = xs[j] - (len(MODEL_ORDER) - 1) / 2 * width
            ax_a.text(
                x, s["mean"] + 0.022, f"{s['mean']:.2f}",
                ha="center", va="bottom", fontsize=12, color=COLORS["text"]
            )

    ax_b = fig.add_subplot(gs[1, 0])
    subset = [m for m in MODEL_ORDER if m != "retrieval"]
    draw_bar_with_ci(
        ax=ax_b,
        values_df=values_df,
        metric="gen_nn_similarity_train_mean",
        ylabel="NN similarity",
        title="Memorization profile",
        ylim=(0.0, 0.34),
        model_order=subset,
        width=0.84,
        title_pad=12,
    )
    ax_b.text(
        0.995, 0.985, "Retrieval ref. = 1.00",
        transform=ax_b.transAxes,
        ha="right", va="top",
        fontsize=14.5, color=COLORS["text"]
    )

    ax_c = fig.add_subplot(gs[1, 1])
    draw_bar_with_ci(
        ax=ax_c,
        values_df=values_df,
        metric="gen_per_condition_surrogate_hit_mean",
        ylabel="Per-condition hit rate",
        title="Condition heterogeneity",
        ylim=(0.0, 1.05),
        width=0.72,
        title_pad=12,
    )

    fig.subplots_adjust(
        left=0.085, right=0.985, top=0.93, bottom=0.15,
        wspace=0.22, hspace=0.38
    )

    add_panel_label(ax_a, "a", wide=True)
    add_panel_label(ax_b, "b", wide=False)
    add_panel_label(ax_c, "c", wide=False)

    add_shared_legend(fig, MODEL_ORDER, y=0.01)

    values_df.to_csv(
        SOURCE_DATA_DIR / "Fig7_step6_generation_quality_redesign_v10_source_data.csv",
        index=False
    )
    save_figure(fig, FIG_MAIN_DIR / "Fig7_step6_generation_quality_redesign_v10")

# =============================================================================
# SUPP FIG 2.6.1
# =============================================================================
def make_supp_261():
    if hist_df is None:
        return

    full_hist = hist_df.loc[hist_df["model_tag"] == "full_cvae"].copy()
    if len(full_hist) == 0:
        return

    curve = (
        full_hist.groupby("epoch", as_index=False)
        .agg(
            val_raw_kl=("val_raw_kl", "mean") if "val_raw_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_free_kl=("val_free_kl", "mean") if "val_free_kl" in full_hist.columns else ("val_kl_loss", "mean"),
            val_active_units=("val_active_units_main", "mean") if "val_active_units_main" in full_hist.columns else ("val_active_units", "mean"),
            val_mean_latent_var=("val_mean_latent_var", "mean"),
        )
    )

    latent_csv_candidates = [
        STEP6_ROOT / "source_data_figures" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        STEP6_ROOT / "source_data_figures_redesign_v2" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        SOURCE_DATA_DIR / "FigS_step6_latent_diagnostics_pca_source_data.csv",
    ]
    latent_df = None
    for p in latent_csv_candidates:
        if p.exists():
            latent_df = pd.read_csv(p)
            break

    fig = plt.figure(figsize=(15.8, 6.0))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1.22, 1.0, 1.02])

    ax_a = fig.add_subplot(gs[0, 0])
    if latent_df is not None and {"pc1", "pc2", "condition_id"}.issubset(latent_df.columns):
        sc = ax_a.scatter(
            latent_df["pc1"], latent_df["pc2"],
            c=latent_df["condition_id"], s=18, alpha=0.82, cmap="viridis"
        )
        cbar = fig.colorbar(sc, ax=ax_a, fraction=0.038, pad=0.018)
        cbar.set_label("Condition ID", fontsize=11.5)
        cbar.ax.tick_params(labelsize=10)

    ax_a.set_xlabel("PC1")
    ax_a.set_ylabel("PC2")
    ax_a.set_title("Test-set latent means (PCA)", pad=10)
    style_axis(ax_a, "both")

    ax_b = fig.add_subplot(gs[0, 1])
    ax_b.plot(curve["epoch"], curve["val_raw_kl"], linewidth=2.0, color="#1F77B4", label="Val raw KL")
    ax_b.plot(curve["epoch"], curve["val_free_kl"], linewidth=2.0, color="#FF7F0E", label="Val free-bits KL")
    ax_b.set_xlabel("Epoch")
    ax_b.set_ylabel("KL")
    ax_b.set_title("Latent KL dynamics", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_c = fig.add_subplot(gs[0, 2])
    line1 = ax_c.plot(curve["epoch"], curve["val_active_units"], linewidth=2.0, color="#1F77B4", label="Active units")
    ax_c.set_xlabel("Epoch")
    ax_c.set_ylabel("Active units", color="#1F77B4", fontsize=12.5)
    ax_c.tick_params(axis="y", labelcolor="#1F77B4", labelsize=10.5)
    ax_c.set_title("Latent activity summary", pad=10)
    style_axis(ax_c, "y")

    ax_c2 = ax_c.twinx()
    line2 = ax_c2.plot(curve["epoch"], curve["val_mean_latent_var"], linewidth=2.0, color="#FF7F0E", label="Mean latent variance")
    ax_c2.set_ylabel("Mean latent variance", color="#FF7F0E", fontsize=12.5)
    ax_c2.tick_params(axis="y", labelcolor="#FF7F0E", labelsize=10.5)
    ax_c2.spines["top"].set_visible(False)
    ax_c2.spines["right"].set_linewidth(1.8)
    ax_c2.spines["right"].set_color(COLORS["spine"])

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax_c.legend(lines, labels, frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(
        left=0.085, right=0.985, top=0.91, bottom=0.12,
        wspace=0.30
    )

    add_panel_label(ax_a, "a", wide=False)
    add_panel_label(ax_b, "b", wide=False)
    add_panel_label(ax_c, "c", wide=False)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_2_6_1_redesign_v10")

# =============================================================================
# SUPP FIG 2.6.2
# =============================================================================
def make_supp_262():
    if temp_df is None or len(temp_df) == 0:
        return

    fig = plt.figure(figsize=(14.8, 5.8))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.05, 1.0])

    ax_a = fig.add_subplot(gs[0, 0])
    mean_temp = (
        temp_df.groupby(["model_tag", "temperature"], as_index=False)
        .agg(
            condition_fidelity=("condition_fidelity", "mean")
            if "condition_fidelity" in temp_df.columns
            else ("classifier_condition_fidelity", "mean")
        )
    )
    fidelity_col = "condition_fidelity" if "condition_fidelity" in mean_temp.columns else "classifier_condition_fidelity"

    keep_models = ["retrieval", "full_cvae", "conditional_gru_lm", "no_condition"]
    for model_tag in keep_models:
        part = mean_temp.loc[mean_temp["model_tag"] == model_tag]
        if len(part) == 0:
            continue
        ax_a.plot(
            part["temperature"], part[fidelity_col],
            marker="o", linewidth=2.1, markersize=7.5,
            color=COLORS[model_tag], label=MODEL_LABELS[model_tag]
        )

    ax_a.set_xlabel("Sampling temperature")
    ax_a.set_ylabel("Condition fidelity")
    ax_a.set_ylim(0.0, 1.0)
    ax_a.set_title("Decoding temperature sensitivity", pad=10)
    style_axis(ax_a, "y")
    ax_a.legend(frameon=False, loc="upper right", fontsize=12.5)

    ax_b = fig.add_subplot(gs[0, 1])
    qa_cols = {
        "valid_rate": "Valid rate",
        "low_complexity_rate": "Low-complexity rate",
        "exact_train_overlap_rate": "Exact train overlap",
    }
    qa_mean = temp_df.groupby("model_tag", as_index=False)[list(qa_cols.keys())].mean()
    qa_mean = qa_mean.set_index("model_tag").loc[MODEL_ORDER].reset_index()

    xs = np.arange(len(qa_mean))
    for col, label in qa_cols.items():
        ax_b.plot(xs, qa_mean[col], marker="o", linewidth=2.0, markersize=7.5, label=label)

    ax_b.set_xticks(xs)
    ax_b.set_xticklabels([MODEL_LABELS[m] for m in qa_mean["model_tag"]], rotation=15, ha="right")
    ax_b.set_ylabel("Rate")
    ax_b.set_title("Generation-time QA summary", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right", fontsize=12.5)

    fig.subplots_adjust(
        left=0.085, right=0.985, top=0.90, bottom=0.20,
        wspace=0.16
    )

    add_panel_label(ax_a, "a", wide=False)
    add_panel_label(ax_b, "b", wide=False)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_2_6_2_redesign_v10")

# =============================================================================
# RUN
# =============================================================================
if __name__ == "__main__":
    print("Generating redesigned Step 6 figures v10 ...")
    make_figure6()
    make_figure7()
    make_supp_261()
    make_supp_262()
    print("Done.")
    print(f"Main figures saved to: {FIG_MAIN_DIR}")
    print(f"Supplementary figures saved to: {FIG_SUPP_DIR}")

# STEP 7 — PepCVAE
# Generation, novelty auditing, diversity analysis, and plausibility screening
# Robust path-discovery + reference-style figure version

In [ ]:
from __future__ import annotations

import json
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# =============================================================================
# STEP 7 — PepCVAE
# Generation, novelty auditing, diversity analysis, and plausibility screening
# Full corrected version with robust reference + generated-file discovery
# =============================================================================

# =============================================================================
# USER CONFIGURATION
# =============================================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# -----------------------------------------------------------------------------
# ROOT PATHS
# -----------------------------------------------------------------------------
STEP6_ROOT = Path("/home/data3/Moe/nature_computational2/step6_v5").resolve()
STEP7_ROOT = Path("/home/data3/Moe/nature_computational2/step7_v3").resolve()

INPUT_DIR = STEP7_ROOT / "inputs"
REF_DIR = STEP7_ROOT / "reference_data"
OUTPUT_DIR = STEP7_ROOT / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"
FIG_MAIN_DIR = OUTPUT_DIR / "figures_main"
FIG_SUPP_DIR = OUTPUT_DIR / "figures_supplementary"
FIG_TIFF_DIR = OUTPUT_DIR / "figures_tiff"
SOURCE_DATA_DIR = OUTPUT_DIR / "source_data"
REPORT_DIR = OUTPUT_DIR / "reports"
LOG_DIR = OUTPUT_DIR / "logs"

for d in [
    INPUT_DIR, REF_DIR, OUTPUT_DIR, TABLE_DIR,
    FIG_MAIN_DIR, FIG_SUPP_DIR, FIG_TIFF_DIR,
    SOURCE_DATA_DIR, REPORT_DIR, LOG_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_MODE = "pregenerated_csv"

# -----------------------------------------------------------------------------
# MODEL ORDER / LABELS
# -----------------------------------------------------------------------------
MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

MODEL_FILE_PATTERNS = {
    "full_cvae": [
        "full_cvae_generated.csv",
        "cvae_generated.csv",
        "pepcvae_generated.csv",
        "*full_cvae*generated*.csv",
        "*cvae*generated*.csv",
        "*pepcvae*generated*.csv",
        "*full_cvae*.csv",
    ],
    "conditional_gru_lm": [
        "conditional_gru_lm_generated.csv",
        "gru_cond_generated.csv",
        "*conditional_gru_lm*generated*.csv",
        "*conditional_gru*generated*.csv",
        "*gru_cond*generated*.csv",
        "*conditional_gru*.csv",
    ],
    "random_condition": [
        "random_condition_generated.csv",
        "rand_cond_generated.csv",
        "*random_condition*generated*.csv",
        "*rand_cond*generated*.csv",
        "*random_condition*.csv",
    ],
    "no_condition": [
        "no_condition_generated.csv",
        "gru_uncond_generated.csv",
        "*no_condition*generated*.csv",
        "*gru_uncond*generated*.csv",
        "*uncond*gru*.csv",
        "*no_condition*.csv",
    ],
    "small_latent": [
        "small_latent_generated.csv",
        "vae_uncond_generated.csv",
        "*small_latent*generated*.csv",
        "*vae_uncond*generated*.csv",
        "*small_latent*.csv",
    ],
}

# =============================================================================
# REFERENCE-STYLE FIGURE THEME
# =============================================================================
COLORS = {
    "full_cvae": "#C0001A",
    "conditional_gru_lm": "#FF1A1A",
    "random_condition": "#1F6B72",
    "no_condition": "#2C4B7C",
    "small_latent": "#8B0000",

    "neutral": "#AFC0DE",
    "train_ref": "#888888",
    "full_ref": "#B5B5B5",

    "bg": "#F2F2F2",
    "grid": "#D9D9D9",
    "text": "#000000",
    "spine": "#000000",
    "edge": "#000000",
    "legend_face": "#FFFFFF",
    "legend_edge": "#B6B6B6",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 12.5,
    "figure.titlesize": 17,
    "axes.facecolor": COLORS["bg"],
    "figure.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
})

# -----------------------------------------------------------------------------
# EXPORT SETTINGS
# -----------------------------------------------------------------------------
EXPORT_PNG = True
EXPORT_PDF = True
EXPORT_TIFF = True
DPI_PNG = 300
DPI_TIFF = 600
TIFF_COMPRESSION = "tiff_lzw"

# -----------------------------------------------------------------------------
# QC / AUDIT SETTINGS
# -----------------------------------------------------------------------------
ALLOWED_AA = set("ACDEFGHIKLMNPQRSTVWY")
MIN_LEN = 5
MAX_LEN = 60

KMER_K = 3
SIMILARITY_THRESHOLDS = [0.50, 0.70, 0.85]

LOW_COMPLEXITY_MAX_RUN_FRACTION = 0.60
LOW_COMPLEXITY_MIN_ENTROPY = 1.20

BOOTSTRAP_N = 1000
PAIRWISE_DIVERSITY_SAMPLE_PAIRS = 2500
PAIRWISE_DIVERSITY_PER_CONDITION_PAIRS = 1200

PLAUSIBILITY_Z_MILD = 2.0
PLAUSIBILITY_Z_STRONG = 3.0

PASS_MIN_VALID_RATE = 0.80
PASS_MIN_EXACT_NOVEL_TRAIN = 0.95
PASS_MAX_NN_SIMILARITY = 0.70
PASS_MIN_DIVERSITY = 0.40
PASS_MIN_CONDITION_MATCH = 0.50

# -----------------------------------------------------------------------------
# OPTIONAL METADATA FROM PREVIOUS STEP
# -----------------------------------------------------------------------------
STEP6_RUN_METADATA_JSON = STEP6_ROOT / "run_metadata.json"
STEP6_CONFIG_JSON = STEP6_ROOT / "config.json"

# =============================================================================
# DESCRIPTOR DEFINITIONS
# =============================================================================
KD_HYDRO = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3
}

CHARGE_SIGN = {
    "K": 1.0, "R": 1.0, "H": 0.1,
    "D": -1.0, "E": -1.0,
}

HYDROPHOBIC_AA = set("AFILMVWCY")
POLAR_AA = set("STNQGH")
POS_AA = set("KRH")
NEG_AA = set("DE")
AROMATIC_AA = set("FWY")
ALIPHATIC_AA = set("AILV")

DESCRIPTOR_COLS = [
    "length",
    "net_charge",
    "hydrophobicity",
    "entropy",
    "max_run_fraction",
    "hydrophobic_fraction",
    "polar_fraction",
    "positive_fraction",
    "negative_fraction",
    "aromatic_fraction",
    "aliphatic_fraction",
]

# =============================================================================
# DATA STRUCTURES
# =============================================================================
@dataclass
class AuditConfig:
    seed: int
    min_len: int
    max_len: int
    kmer_k: int
    similarity_thresholds: List[float]
    low_complexity_max_run_fraction: float
    low_complexity_min_entropy: float
    bootstrap_n: int
    pairwise_diversity_sample_pairs: int
    pairwise_diversity_per_condition_pairs: int
    plausibility_z_mild: float
    plausibility_z_strong: float
    export_png: bool
    export_pdf: bool
    export_tiff: bool
    dpi_png: int
    dpi_tiff: int
    tiff_compression: str
    input_mode: str

# =============================================================================
# GENERAL HELPERS
# =============================================================================
def write_json(path: Path, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)

def save_figure(fig, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=DPI_PNG)
    if EXPORT_TIFF:
        tiff_base = FIG_TIFF_DIR / out_base.name
        try:
            fig.savefig(
                tiff_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
                pil_kwargs={"compression": TIFF_COMPRESSION},
            )
        except TypeError:
            fig.savefig(
                tiff_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
            )
    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_facecolor(COLORS["bg"])
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=1.0)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=1.0)
    else:
        ax.grid(True, color=COLORS["grid"], linewidth=1.0)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.6)
        ax.spines[side].set_color(COLORS["spine"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=1.0, colors=COLORS["text"])

def add_panel_label(ax, label: str, wide: bool = False, fontsize: int = 22):
    x = -0.07 if wide else -0.12
    y = 1.03
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        color=COLORS["text"],
        clip_on=False,
    )

def add_shared_legend(fig, model_tags: List[str], y: float = 0.02, ncol: int = 5):
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[m], edgecolor=COLORS["edge"], linewidth=0.8)
        for m in model_tags
    ]
    labels = [MODEL_LABELS[m] for m in model_tags]
    leg = fig.legend(
        handles,
        labels,
        ncol=ncol,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        frameon=True,
        fontsize=12.5,
        handlelength=1.7,
        columnspacing=1.3,
    )
    leg.get_frame().set_facecolor(COLORS["legend_face"])
    leg.get_frame().set_edgecolor(COLORS["legend_edge"])
    leg.get_frame().set_linewidth(1.1)

def safe_div(n: float, d: float) -> float:
    return float(n / d) if d else np.nan

def canonicalize_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(seq.split())

def bootstrap_mean_ci(values: np.ndarray, n_boot: int = BOOTSTRAP_N, seed: int = SEED) -> Tuple[float, float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    if len(arr) == 1:
        x = float(arr[0])
        return x, x, x

    rng = np.random.default_rng(seed)
    n = len(arr)
    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sample = rng.choice(arr, size=n, replace=True)
        boots[i] = np.mean(sample)

    mean = float(np.mean(arr))
    ci_low = float(np.percentile(boots, 2.5))
    ci_high = float(np.percentile(boots, 97.5))
    return mean, ci_low, ci_high

def assert_no_duplicate_columns(df: pd.DataFrame, df_name: str):
    dupes = df.columns[df.columns.duplicated()].tolist()
    if dupes:
        raise ValueError(f"Duplicate columns detected in {df_name}: {dupes}")

def coerce_scalar(value):
    if isinstance(value, pd.Series):
        non_na = value.dropna()
        if len(non_na) == 0:
            return np.nan
        return non_na.iloc[0]
    return value

def to_bool_or_nan(x):
    if pd.isna(x):
        return np.nan
    return bool(x)

# =============================================================================
# GENERATED FILE FILTERING HELPERS
# =============================================================================
SEQUENCE_COL_CANDIDATES = [
    "sequence",
    "seq",
    "peptide",
    "peptide_sequence",
    "aa_sequence",
    "generated_sequence",
]

SPLIT_COL_CANDIDATES = [
    "split",
    "set",
    "subset",
    "partition",
    "data_split",
]

TRAIN_TOKENS = {"train", "training"}
FULL_TOKENS = {"full", "corpus", "reference", "all", "retained", "master", "dataset"}

def normalize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    return out

def detect_sequence_col(df: pd.DataFrame) -> Optional[str]:
    cols = {c.lower(): c for c in df.columns}
    for cand in SEQUENCE_COL_CANDIDATES:
        if cand in cols:
            return cols[cand]
    return None

def detect_split_col(df: pd.DataFrame) -> Optional[str]:
    cols = {c.lower(): c for c in df.columns}
    for cand in SPLIT_COL_CANDIDATES:
        if cand in cols:
            return cols[cand]
    return None

def file_has_sequence_column(path: Path) -> bool:
    try:
        df = pd.read_csv(path, nrows=5)
    except Exception:
        return False

    df = normalize_colnames(df)
    return detect_sequence_col(df) is not None

def file_looks_like_generated_sequences(path: Path) -> bool:
    name = path.name.lower()

    reject_tokens = [
        "history",
        "training_history",
        "epoch",
        "loss",
        "curve",
        "log",
        "summary",
        "audit",
        "report",
        "figure",
        "source_data",
        "config",
        "manifest",
        "metrics",
        "learning_curve",
    ]
    if any(tok in name for tok in reject_tokens):
        return False

    return file_has_sequence_column(path)

def read_csv_required(path: Optional[Path], label: str = "required file") -> pd.DataFrame:
    if path is None or not path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {path}\n"
            "Please place the file in step7_v3/reference_data or ensure it exists in a previous-step folder."
        )
    return pd.read_csv(path)

# =============================================================================
# DESCRIPTORS
# =============================================================================
def compute_length(seq: str) -> int:
    return len(seq)

def compute_net_charge(seq: str) -> float:
    return float(sum(CHARGE_SIGN.get(aa, 0.0) for aa in seq))

def compute_hydrophobicity(seq: str) -> float:
    if not seq:
        return np.nan
    vals = [KD_HYDRO[aa] for aa in seq if aa in KD_HYDRO]
    return float(np.mean(vals)) if vals else np.nan

def shannon_entropy(seq: str) -> float:
    if not seq:
        return np.nan
    counts = pd.Series(list(seq)).value_counts(normalize=True)
    return float(-(counts * np.log2(counts)).sum())

def max_run_fraction(seq: str) -> float:
    if not seq:
        return np.nan
    max_run = 1
    cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 1
    return max_run / len(seq)

def frac_residue_class(seq: str, aa_set: set) -> float:
    if not seq:
        return np.nan
    return float(sum(aa in aa_set for aa in seq) / len(seq))

def descriptor_dict(seq: str) -> Dict[str, float]:
    return {
        "length": compute_length(seq),
        "net_charge": compute_net_charge(seq),
        "hydrophobicity": compute_hydrophobicity(seq),
        "entropy": shannon_entropy(seq),
        "max_run_fraction": max_run_fraction(seq),
        "hydrophobic_fraction": frac_residue_class(seq, HYDROPHOBIC_AA),
        "polar_fraction": frac_residue_class(seq, POLAR_AA),
        "positive_fraction": frac_residue_class(seq, POS_AA),
        "negative_fraction": frac_residue_class(seq, NEG_AA),
        "aromatic_fraction": frac_residue_class(seq, AROMATIC_AA),
        "aliphatic_fraction": frac_residue_class(seq, ALIPHATIC_AA),
    }

def annotate_descriptors(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Drop existing descriptor columns first to prevent duplicate names.
    existing_descriptor_cols = [c for c in DESCRIPTOR_COLS if c in out.columns]
    if existing_descriptor_cols:
        out = out.drop(columns=existing_descriptor_cols)

    desc = out["sequence"].map(descriptor_dict).apply(pd.Series)
    out = pd.concat([out, desc], axis=1)

    assert_no_duplicate_columns(out, "annotate_descriptors output")
    return out

# =============================================================================
# VALIDITY / COMPLEXITY
# =============================================================================
def is_valid_sequence(seq: str) -> bool:
    if not seq:
        return False
    if len(seq) < MIN_LEN or len(seq) > MAX_LEN:
        return False
    return all(ch in ALLOWED_AA for ch in seq)

def is_low_complexity(seq: str) -> bool:
    ent = shannon_entropy(seq)
    run_frac = max_run_fraction(seq)
    if pd.isna(ent) or pd.isna(run_frac):
        return True
    return (ent < LOW_COMPLEXITY_MIN_ENTROPY) or (run_frac > LOW_COMPLEXITY_MAX_RUN_FRACTION)

# =============================================================================
# KMER / SIMILARITY
# =============================================================================
def kmers(seq: str, k: int = KMER_K) -> set:
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}

def build_kmer_cache(sequences: List[str], k: int = KMER_K) -> List[set]:
    return [kmers(seq, k) for seq in sequences]

def jaccard_from_kmers(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)

def nearest_neighbor_similarity_cached(
    seq_kmers: set,
    ref_sequences: List[str],
    ref_kmers: List[set],
) -> Tuple[float, Optional[str]]:
    if not ref_sequences:
        return np.nan, None
    best_sim = -1.0
    best_ref = None
    for ref_seq, ref_set in zip(ref_sequences, ref_kmers):
        sim = jaccard_from_kmers(seq_kmers, ref_set)
        if sim > best_sim:
            best_sim = sim
            best_ref = ref_seq
    return float(best_sim), best_ref

def sample_pairwise_diversity(sequences: List[str], n_pairs: int, seed: int = SEED) -> float:
    seqs = [s for s in sequences if isinstance(s, str) and len(s) > 0]
    n = len(seqs)
    if n < 2:
        return np.nan

    rng = np.random.default_rng(seed)
    kmer_cache = [kmers(s, KMER_K) for s in seqs]
    max_pairs = min(n_pairs, n * (n - 1) // 2)

    seen = set()
    sims = []
    while len(seen) < max_pairs:
        i, j = rng.integers(0, n, size=2)
        if i == j:
            continue
        pair = tuple(sorted((int(i), int(j))))
        if pair in seen:
            continue
        seen.add(pair)
        sims.append(jaccard_from_kmers(kmer_cache[i], kmer_cache[j]))

    return float(1.0 - np.mean(sims)) if sims else np.nan

def greedy_cluster_count(sequences: List[str], threshold: float) -> Tuple[int, float]:
    seqs = [s for s in sequences if isinstance(s, str) and len(s) > 0]
    if len(seqs) == 0:
        return 0, np.nan
    if len(seqs) == 1:
        return 1, 1.0

    caches = [kmers(s, KMER_K) for s in seqs]
    cluster_reps = []
    cluster_sizes = []

    for kset in caches:
        assigned = False
        for idx, rep_kset in enumerate(cluster_reps):
            sim = jaccard_from_kmers(kset, rep_kset)
            if sim >= threshold:
                cluster_sizes[idx] += 1
                assigned = True
                break
        if not assigned:
            cluster_reps.append(kset)
            cluster_sizes.append(1)

    n_clusters = len(cluster_sizes)
    largest_frac = max(cluster_sizes) / len(seqs)
    return n_clusters, largest_frac

def get_threshold_rate(values: Iterable[float], threshold: float, direction: str = "leq") -> float:
    arr = pd.Series(list(values), dtype=float).dropna().to_numpy()
    if len(arr) == 0:
        return np.nan
    if direction == "leq":
        return float(np.mean(arr <= threshold))
    if direction == "lt":
        return float(np.mean(arr < threshold))
    if direction == "geq":
        return float(np.mean(arr >= threshold))
    if direction == "gt":
        return float(np.mean(arr > threshold))
    raise ValueError(f"Unsupported direction: {direction}")

# =============================================================================
# ROBUST REFERENCE DISCOVERY BY CONTENT
# =============================================================================
def canonicalize_reference_df(df: pd.DataFrame, sequence_col: str) -> pd.DataFrame:
    out = df.copy()
    out["sequence"] = out[sequence_col].map(canonicalize_sequence)
    out = out.loc[out["sequence"] != ""].drop_duplicates(subset=["sequence"]).reset_index(drop=True)
    return out[["sequence"]].copy()

def score_train_candidate(path: Path, df: pd.DataFrame, seq_col: str, split_col: Optional[str]) -> int:
    score = 0
    name = path.name.lower()

    if "train" in name:
        score += 5
    if "reference" in name:
        score += 2
    if "split" in name:
        score += 1
    if seq_col is not None:
        score += 3

    if split_col is not None:
        vals = set(df[split_col].astype(str).str.lower().unique())
        if any(v in TRAIN_TOKENS for v in vals):
            score += 6

    return score

def score_full_candidate(path: Path, df: pd.DataFrame, seq_col: str, split_col: Optional[str]) -> int:
    score = 0
    name = path.name.lower()

    if any(tok in name for tok in FULL_TOKENS):
        score += 5
    if "reference" in name:
        score += 2
    if seq_col is not None:
        score += 3

    if split_col is not None:
        vals = set(df[split_col].astype(str).str.lower().unique())
        if any(v in TRAIN_TOKENS for v in vals):
            score += 4

    return score

def discover_all_csvs(search_roots: List[Path]) -> List[Path]:
    csvs = []
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*.csv"):
            rp = str(p.resolve())
            if rp not in seen:
                seen.add(rp)
                csvs.append(p)
    return sorted(csvs)

def inspect_csv_candidate(path: Path) -> Optional[dict]:
    try:
        df = pd.read_csv(path, nrows=200)
    except Exception:
        return None

    df = normalize_colnames(df)
    seq_col = detect_sequence_col(df)
    split_col = detect_split_col(df)

    if seq_col is None:
        return None

    info = {
        "path": path,
        "seq_col": seq_col,
        "split_col": split_col,
        "columns": list(df.columns),
        "n_preview_rows": len(df),
        "train_score": score_train_candidate(path, df, seq_col, split_col),
        "full_score": score_full_candidate(path, df, seq_col, split_col),
    }
    return info

def build_train_df_from_candidate(path: Path, seq_col: str, split_col: Optional[str]) -> Optional[pd.DataFrame]:
    df = pd.read_csv(path)
    df = normalize_colnames(df)
    seq_col = detect_sequence_col(df)
    split_col = detect_split_col(df)

    if seq_col is None:
        return None

    if split_col is not None:
        split_vals = df[split_col].astype(str).str.lower().str.strip()
        keep = split_vals.isin(TRAIN_TOKENS)
        if keep.any():
            return canonicalize_reference_df(df.loc[keep].copy(), seq_col)

    if "train" in path.name.lower():
        return canonicalize_reference_df(df, seq_col)

    return None

def build_full_df_from_candidate(path: Path, seq_col: str) -> Optional[pd.DataFrame]:
    df = pd.read_csv(path)
    df = normalize_colnames(df)
    seq_col = detect_sequence_col(df)
    if seq_col is None:
        return None
    return canonicalize_reference_df(df, seq_col)

def discover_reference_tables() -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    search_roots = [
        REF_DIR,
        STEP7_ROOT,
        STEP6_ROOT / "tables",
        STEP6_ROOT / "outputs",
        STEP6_ROOT,
        STEP6_ROOT.parent,
    ]

    csv_paths = discover_all_csvs(search_roots)
    inspected = []
    for p in csv_paths:
        info = inspect_csv_candidate(p)
        if info is not None:
            inspected.append(info)

    if not inspected:
        raise FileNotFoundError(
            "No CSV files with a recognizable sequence column were found under the Step 7 / Step 6 search roots."
        )

    train_candidates = sorted(inspected, key=lambda x: x["train_score"], reverse=True)
    full_candidates = sorted(inspected, key=lambda x: x["full_score"], reverse=True)

    train_df = None
    train_source = None
    for cand in train_candidates:
        df_try = build_train_df_from_candidate(cand["path"], cand["seq_col"], cand["split_col"])
        if df_try is not None and len(df_try) > 0:
            train_df = df_try
            train_source = cand["path"]
            break

    full_df = None
    full_source = None
    for cand in full_candidates:
        df_try = build_full_df_from_candidate(cand["path"], cand["seq_col"])
        if df_try is not None and len(df_try) > 0:
            full_df = df_try
            full_source = cand["path"]
            break

    if train_df is None:
        top = [str(x["path"]) for x in train_candidates[:10]]
        raise FileNotFoundError(
            "Could not construct a train reference table automatically.\n"
            f"Top train-like CSV candidates were:\n" + "\n".join(top)
        )

    if full_df is None:
        top = [str(x["path"]) for x in full_candidates[:10]]
        raise FileNotFoundError(
            "Could not construct a full corpus reference table automatically.\n"
            f"Top full-like CSV candidates were:\n" + "\n".join(top)
        )

    manifest = {
        "train_reference_source": str(train_source),
        "full_corpus_reference_source": str(full_source),
        "n_train_sequences": int(len(train_df)),
        "n_full_sequences": int(len(full_df)),
        "inspected_sequence_csvs": [str(x["path"]) for x in inspected],
    }
    return train_df, full_df, manifest

def discover_condition_schema() -> Optional[pd.DataFrame]:
    search_roots = [
        REF_DIR,
        STEP7_ROOT,
        STEP6_ROOT / "tables",
        STEP6_ROOT / "outputs",
        STEP6_ROOT,
    ]

    candidate_names = [
        "condition_schema.csv",
        "step3_condition_schema.csv",
        "conditions.csv",
        "*condition*schema*.csv",
        "*conditions*.csv",
    ]

    for root in search_roots:
        if not root.exists():
            continue
        for patt in candidate_names:
            matches = sorted(root.rglob(patt))
            for p in matches:
                try:
                    df = pd.read_csv(p).copy()
                except Exception:
                    continue
                df.columns = [str(c).strip().lower() for c in df.columns]
                if "condition_id" in df.columns:
                    optional_numeric = [
                        "length_min", "length_max",
                        "charge_min", "charge_max",
                        "hydro_min", "hydro_max",
                    ]
                    for col in optional_numeric:
                        if col in df.columns:
                            df[col] = pd.to_numeric(df[col], errors="coerce")
                    write_json(LOG_DIR / "step7_condition_schema_source.json", {"condition_schema_source": str(p)})
                    return df
    return None

def discover_file_by_patterns(base_dirs: List[Path], patterns: List[str]) -> Optional[Path]:
    valid_matches = []

    for base_dir in base_dirs:
        if not base_dir.exists():
            continue

        for pattern in patterns:
            matches = sorted(base_dir.rglob(pattern)) if "*" in pattern else sorted(base_dir.glob(pattern))
            for p in matches:
                if file_looks_like_generated_sequences(p):
                    valid_matches.append(p)

    if not valid_matches:
        return None

    def priority(p: Path):
        s = str(p)
        if str(INPUT_DIR) in s:
            return (0, len(s))
        if str(STEP6_ROOT / "tables") in s:
            return (1, len(s))
        if str(STEP6_ROOT / "outputs") in s:
            return (2, len(s))
        return (3, len(s))

    valid_matches = sorted(valid_matches, key=priority)
    return valid_matches[0]

def discover_generated_input_files() -> Dict[str, Optional[Path]]:
    base_dirs = [INPUT_DIR, STEP6_ROOT / "tables", STEP6_ROOT / "outputs", STEP6_ROOT]
    discovered = {}
    for model_tag, patterns in MODEL_FILE_PATTERNS.items():
        discovered[model_tag] = discover_file_by_patterns(base_dirs, patterns)
    return discovered

def validate_generated_schema(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
    cols_lower = {str(c).strip().lower(): c for c in df.columns}

    sequence_aliases = [
        "sequence",
        "seq",
        "peptide",
        "peptide_sequence",
        "aa_sequence",
        "generated_sequence",
    ]

    seq_col = None
    for cand in sequence_aliases:
        if cand in cols_lower:
            seq_col = cols_lower[cand]
            break

    if seq_col is None:
        raise ValueError(
            f"{file_path} is not a generated-sequence file. "
            f"Available columns: {list(df.columns)}"
        )

    if seq_col != "sequence":
        df["sequence"] = df[seq_col]

    return df

def load_generated_inputs(generated_paths: Dict[str, Optional[Path]]) -> Tuple[pd.DataFrame, dict]:
    frames = []
    manifest = {"files_loaded": [], "files_missing": [], "row_counts": {}, "resolved_paths": {}}

    for model_tag in MODEL_ORDER:
        path = generated_paths.get(model_tag)
        manifest["resolved_paths"][model_tag] = str(path) if path is not None else None

        if path is None or not path.exists():
            manifest["files_missing"].append(model_tag)
            continue

        df = pd.read_csv(path).copy()
        df = validate_generated_schema(df, path)
        df = normalize_colnames(df)

        if "sequence" not in df.columns:
            raise ValueError(f"'sequence' column missing after schema validation for file: {path}")

        df["model_tag"] = model_tag
        df["sequence"] = df["sequence"].map(canonicalize_sequence)

        if "requested_condition_id" not in df.columns:
            df["requested_condition_id"] = np.nan
        if "sample_id" not in df.columns:
            df["sample_id"] = np.arange(len(df))
        if "seed" not in df.columns:
            df["seed"] = SEED
        if "decode_temperature" not in df.columns:
            df["decode_temperature"] = np.nan
        if "checkpoint_name" not in df.columns:
            df["checkpoint_name"] = np.nan
        if "run_name" not in df.columns:
            df["run_name"] = np.nan

        manifest["files_loaded"].append(str(path))
        manifest["row_counts"][model_tag] = int(len(df))
        frames.append(df)

    if not frames:
        raise FileNotFoundError(
            "No generated input CSVs were found.\n"
            f"Searched under: {INPUT_DIR} and {STEP6_ROOT}\n"
            "Please place generated CSVs in step7_v3/inputs or provide earlier-step files with recognizable names."
        )

    if manifest["files_missing"]:
        warnings.warn(f"Some model input files were missing and were skipped: {manifest['files_missing']}")

    all_df = pd.concat(frames, ignore_index=True)
    all_df = all_df.loc[all_df["sequence"] != ""].reset_index(drop=True)
    assert_no_duplicate_columns(all_df, "raw generated inputs")
    return all_df, manifest

# =============================================================================
# QC PIPELINE
# =============================================================================
def run_qc_pipeline(generated_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = generated_df.copy()
    assert_no_duplicate_columns(out, "run_qc_pipeline input")

    out["is_valid"] = out["sequence"].map(is_valid_sequence)
    out["is_low_complexity"] = out["sequence"].map(is_low_complexity)

    audit_rows = []
    kept_frames = []

    for model_tag, g in out.groupby("model_tag", sort=False):
        n_raw = len(g)
        raw_unique_rate = safe_div(g["sequence"].nunique(), n_raw)

        valid = g.loc[g["is_valid"]].copy()
        n_valid = len(valid)
        valid_unique_rate = safe_div(valid["sequence"].nunique(), n_valid)

        dedup = valid.drop_duplicates(subset=["sequence"]).copy()
        n_after_dedup = len(dedup)

        final = dedup.loc[~dedup["is_low_complexity"]].copy()
        n_final = len(final)
        final_unique_rate = safe_div(final["sequence"].nunique(), n_final)

        audit_rows.append({
            "model_tag": model_tag,
            "n_raw": n_raw,
            "raw_unique_rate": raw_unique_rate,
            "n_valid": n_valid,
            "valid_rate": safe_div(n_valid, n_raw),
            "valid_unique_rate": valid_unique_rate,
            "n_after_dedup": n_after_dedup,
            "dedup_retention_rate": safe_div(n_after_dedup, n_raw),
            "n_after_low_complexity_filter": n_final,
            "post_qc_retention_rate": safe_div(n_final, n_raw),
            "final_unique_rate": final_unique_rate,
            "n_removed_invalid": n_raw - n_valid,
            "n_removed_duplicates": n_valid - n_after_dedup,
            "n_removed_low_complexity": n_after_dedup - n_final,
        })
        kept_frames.append(final)

    qc_df = pd.concat(kept_frames, ignore_index=True) if kept_frames else pd.DataFrame()
    audit_df = pd.DataFrame(audit_rows)
    assert_no_duplicate_columns(qc_df, "qc_df")
    return qc_df, audit_df

# =============================================================================
# NOVELTY AND MEMORIZATION
# =============================================================================
def run_exact_novelty(df: pd.DataFrame, train_set: set, full_set: set) -> pd.DataFrame:
    out = df.copy()
    assert_no_duplicate_columns(out, "run_exact_novelty input")
    out["exact_in_train"] = out["sequence"].isin(train_set)
    out["exact_in_full_corpus"] = out["sequence"].isin(full_set)
    out["exact_novel_vs_train"] = ~out["exact_in_train"]
    out["exact_novel_vs_full"] = ~out["exact_in_full_corpus"]
    return out

def run_similarity_audit(df: pd.DataFrame, train_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    assert_no_duplicate_columns(out, "run_similarity_audit input")

    train_sequences = train_df["sequence"].tolist()
    full_sequences = full_df["sequence"].tolist()
    train_kmers = build_kmer_cache(train_sequences, KMER_K)
    full_kmers = build_kmer_cache(full_sequences, KMER_K)

    nn_train = []
    nn_train_seq = []
    nn_full = []
    nn_full_seq = []

    for seq in out["sequence"].tolist():
        seq_k = kmers(seq, KMER_K)
        sim_t, ref_t = nearest_neighbor_similarity_cached(seq_k, train_sequences, train_kmers)
        sim_f, ref_f = nearest_neighbor_similarity_cached(seq_k, full_sequences, full_kmers)
        nn_train.append(sim_t)
        nn_train_seq.append(ref_t)
        nn_full.append(sim_f)
        nn_full_seq.append(ref_f)

    out["nn_similarity_train"] = nn_train
    out["nn_train_sequence"] = nn_train_seq
    out["nn_similarity_full_corpus"] = nn_full
    out["nn_full_corpus_sequence"] = nn_full_seq

    for thr in SIMILARITY_THRESHOLDS:
        col_train = f"nn_train_leq_{str(thr).replace('.', 'p')}"
        col_full = f"nn_full_leq_{str(thr).replace('.', 'p')}"
        out[col_train] = out["nn_similarity_train"] <= thr
        out[col_full] = out["nn_similarity_full_corpus"] <= thr

    assert_no_duplicate_columns(out, "run_similarity_audit output")
    return out

# =============================================================================
# CONDITION CONSISTENCY
# =============================================================================
def within_range(x: float, lo: Optional[float], hi: Optional[float]) -> bool:
    if pd.isna(x):
        return False
    if lo is not None and not pd.isna(lo) and x < lo:
        return False
    if hi is not None and not pd.isna(hi) and x > hi:
        return False
    return True

def run_condition_consistency(df: pd.DataFrame, condition_schema: Optional[pd.DataFrame]) -> pd.DataFrame:
    out = df.copy()
    assert_no_duplicate_columns(out, "run_condition_consistency input")

    out["condition_match_length"] = np.nan
    out["condition_match_charge"] = np.nan
    out["condition_match_hydrophobicity"] = np.nan
    out["condition_match_combined"] = np.nan

    if condition_schema is None:
        return out

    cond_map = condition_schema.set_index("condition_id").to_dict(orient="index")

    length_match = []
    charge_match = []
    hydro_match = []
    combined_match = []

    for _, row in out.iterrows():
        cid = row.get("requested_condition_id", np.nan)
        if pd.isna(cid) or cid not in cond_map:
            length_match.append(np.nan)
            charge_match.append(np.nan)
            hydro_match.append(np.nan)
            combined_match.append(np.nan)
            continue

        spec = cond_map[cid]

        lm = np.nan
        cm = np.nan
        hm = np.nan

        if ("length_min" in spec) or ("length_max" in spec):
            lm = to_bool_or_nan(within_range(row["length"], spec.get("length_min"), spec.get("length_max")))
        if ("charge_min" in spec) or ("charge_max" in spec):
            cm = to_bool_or_nan(within_range(row["net_charge"], spec.get("charge_min"), spec.get("charge_max")))
        if ("hydro_min" in spec) or ("hydro_max" in spec):
            hm = to_bool_or_nan(within_range(row["hydrophobicity"], spec.get("hydro_min"), spec.get("hydro_max")))

        active_vals = [v for v in [lm, cm, hm] if not pd.isna(v)]
        combined = np.nan if len(active_vals) == 0 else bool(np.all(active_vals))

        length_match.append(lm)
        charge_match.append(cm)
        hydro_match.append(hm)
        combined_match.append(combined)

    out["condition_match_length"] = length_match
    out["condition_match_charge"] = charge_match
    out["condition_match_hydrophobicity"] = hydro_match
    out["condition_match_combined"] = combined_match

    assert_no_duplicate_columns(out, "run_condition_consistency output")
    return out

# =============================================================================
# PLAUSIBILITY SCREENING
# =============================================================================
PLAUSIBILITY_METRICS = [
    "length",
    "net_charge",
    "hydrophobicity",
    "entropy",
    "hydrophobic_fraction",
    "polar_fraction",
    "positive_fraction",
    "negative_fraction",
    "aromatic_fraction",
    "aliphatic_fraction",
]

def build_train_reference_stats(train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in PLAUSIBILITY_METRICS:
        vals = train_df[metric].dropna().to_numpy(dtype=float)
        rows.append({
            "metric": metric,
            "mean": float(np.mean(vals)) if len(vals) else np.nan,
            "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "min": float(np.min(vals)) if len(vals) else np.nan,
            "max": float(np.max(vals)) if len(vals) else np.nan,
        })
    out = pd.DataFrame(rows)
    if out["metric"].duplicated().any():
        dupes = out.loc[out["metric"].duplicated(), "metric"].tolist()
        raise ValueError(f"Duplicate metrics in train reference stats: {dupes}")
    return out

def classify_plausibility_row(row: pd.Series, ref_stats: pd.DataFrame) -> Tuple[str, int]:
    strong_flags = 0
    mild_flags = 0

    for metric in PLAUSIBILITY_METRICS:
        x = coerce_scalar(row.get(metric, np.nan))

        ref_row = ref_stats.loc[ref_stats["metric"] == metric]
        if ref_row.empty:
            continue
        ref = ref_row.iloc[0]

        mu = coerce_scalar(ref["mean"])
        sd = coerce_scalar(ref["std"])

        if pd.isna(x) or pd.isna(mu) or pd.isna(sd) or float(sd) == 0.0:
            continue

        z = abs((float(x) - float(mu)) / float(sd))
        if z > PLAUSIBILITY_Z_STRONG:
            strong_flags += 1
        elif z > PLAUSIBILITY_Z_MILD:
            mild_flags += 1

    if strong_flags > 0:
        return "strong_outlier", strong_flags
    if mild_flags > 0:
        return "mild_outlier", mild_flags
    return "within_reference_range", 0

def run_plausibility_screen(df: pd.DataFrame, train_ref_stats: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    assert_no_duplicate_columns(out, "run_plausibility_screen input")
    assert_no_duplicate_columns(train_ref_stats, "train_ref_stats")

    labels = []
    counts = []
    for _, row in out.iterrows():
        lab, cnt = classify_plausibility_row(row, train_ref_stats)
        labels.append(lab)
        counts.append(cnt)
    out["plausibility_flag"] = labels
    out["plausibility_outlier_metric_count"] = counts
    out["plausibility_is_within_reference"] = out["plausibility_flag"] == "within_reference_range"
    return out

# =============================================================================
# SUMMARIES
# =============================================================================
def summarise_by_model(final_df: pd.DataFrame, audit_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_tag, g in final_df.groupby("model_tag", sort=False):
        audit_row = audit_df.loc[audit_df["model_tag"] == model_tag].iloc[0]

        novelty_train = g["exact_novel_vs_train"].astype(float).to_numpy()
        novelty_full = g["exact_novel_vs_full"].astype(float).to_numpy()
        nn_train = g["nn_similarity_train"].dropna().to_numpy(dtype=float)
        nn_full = g["nn_similarity_full_corpus"].dropna().to_numpy(dtype=float)

        cond_comb = g["condition_match_combined"].dropna().astype(float).to_numpy()
        cond_len = g["condition_match_length"].dropna().astype(float).to_numpy()
        cond_charge = g["condition_match_charge"].dropna().astype(float).to_numpy()
        cond_hydro = g["condition_match_hydrophobicity"].dropna().astype(float).to_numpy()

        plausible = g["plausibility_is_within_reference"].astype(float).to_numpy()

        nt_mean, nt_low, nt_high = bootstrap_mean_ci(novelty_train)
        nf_mean, nf_low, nf_high = bootstrap_mean_ci(novelty_full)
        st_mean, st_low, st_high = bootstrap_mean_ci(nn_train)
        sf_mean, sf_low, sf_high = bootstrap_mean_ci(nn_full)

        cc_mean, cc_low, cc_high = bootstrap_mean_ci(cond_comb) if len(cond_comb) else (np.nan, np.nan, np.nan)
        cl_mean, _, _ = bootstrap_mean_ci(cond_len) if len(cond_len) else (np.nan, np.nan, np.nan)
        cq_mean, _, _ = bootstrap_mean_ci(cond_charge) if len(cond_charge) else (np.nan, np.nan, np.nan)
        ch_mean, _, _ = bootstrap_mean_ci(cond_hydro) if len(cond_hydro) else (np.nan, np.nan, np.nan)

        pl_mean, _, _ = bootstrap_mean_ci(plausible)

        diversity = sample_pairwise_diversity(g["sequence"].tolist(), PAIRWISE_DIVERSITY_SAMPLE_PAIRS, seed=SEED)
        n_clusters_070, largest_cluster_frac_070 = greedy_cluster_count(g["sequence"].tolist(), threshold=0.70)

        row = {
            "model_tag": model_tag,
            "n_final": len(g),

            "raw_unique_rate": audit_row["raw_unique_rate"],
            "valid_rate": audit_row["valid_rate"],
            "post_qc_retention_rate": audit_row["post_qc_retention_rate"],
            "final_unique_rate": audit_row["final_unique_rate"],

            "exact_novel_vs_train_rate_mean": nt_mean,
            "exact_novel_vs_train_rate_ci_low": nt_low,
            "exact_novel_vs_train_rate_ci_high": nt_high,

            "exact_novel_vs_full_rate_mean": nf_mean,
            "exact_novel_vs_full_rate_ci_low": nf_low,
            "exact_novel_vs_full_rate_ci_high": nf_high,

            "nn_similarity_train_mean": st_mean,
            "nn_similarity_train_ci_low": st_low,
            "nn_similarity_train_ci_high": st_high,

            "nn_similarity_full_mean": sf_mean,
            "nn_similarity_full_ci_low": sf_low,
            "nn_similarity_full_ci_high": sf_high,

            "within_set_diversity_mean": diversity,
            "cluster_count_070": n_clusters_070,
            "largest_cluster_fraction_070": largest_cluster_frac_070,

            "condition_match_combined_mean": cc_mean,
            "condition_match_combined_ci_low": cc_low,
            "condition_match_combined_ci_high": cc_high,
            "condition_match_length_mean": cl_mean,
            "condition_match_charge_mean": cq_mean,
            "condition_match_hydrophobicity_mean": ch_mean,

            "plausibility_within_reference_rate": pl_mean,

            "length_mean": float(np.mean(g["length"])) if len(g) else np.nan,
            "net_charge_mean": float(np.mean(g["net_charge"])) if len(g) else np.nan,
            "hydrophobicity_mean": float(np.mean(g["hydrophobicity"])) if len(g) else np.nan,
            "entropy_mean": float(np.mean(g["entropy"])) if len(g) else np.nan,
        }

        for thr in SIMILARITY_THRESHOLDS:
            key = str(thr).replace(".", "p")
            row[f"nn_similarity_train_leq_{key}_rate"] = get_threshold_rate(nn_train, thr, direction="leq")
            row[f"nn_similarity_full_leq_{key}_rate"] = get_threshold_rate(nn_full, thr, direction="leq")

        rows.append(row)

    out = pd.DataFrame(rows)
    if not out.empty:
        out["model_order"] = out["model_tag"].map({m: i for i, m in enumerate(MODEL_ORDER)})
        out = out.sort_values(["model_order", "model_tag"]).drop(columns=["model_order"]).reset_index(drop=True)
    return out

def summarise_by_condition(final_df: pd.DataFrame) -> pd.DataFrame:
    if "requested_condition_id" not in final_df.columns:
        return pd.DataFrame()

    rows = []
    for (model_tag, condition_id), g in final_df.groupby(["model_tag", "requested_condition_id"], dropna=False):
        cluster_n, largest_frac = greedy_cluster_count(g["sequence"].tolist(), 0.70) if len(g) else (np.nan, np.nan)
        rows.append({
            "model_tag": model_tag,
            "requested_condition_id": condition_id,
            "n_sequences": len(g),
            "exact_novel_vs_train_rate": float(np.mean(g["exact_novel_vs_train"])) if len(g) else np.nan,
            "exact_novel_vs_full_rate": float(np.mean(g["exact_novel_vs_full"])) if len(g) else np.nan,
            "nn_similarity_train_mean": float(np.mean(g["nn_similarity_train"])) if len(g) else np.nan,
            "nn_similarity_full_mean": float(np.mean(g["nn_similarity_full_corpus"])) if len(g) else np.nan,
            "condition_match_combined_rate": float(np.mean(g["condition_match_combined"].dropna().astype(float))) if g["condition_match_combined"].notna().any() else np.nan,
            "condition_match_length_rate": float(np.mean(g["condition_match_length"].dropna().astype(float))) if g["condition_match_length"].notna().any() else np.nan,
            "condition_match_charge_rate": float(np.mean(g["condition_match_charge"].dropna().astype(float))) if g["condition_match_charge"].notna().any() else np.nan,
            "condition_match_hydrophobicity_rate": float(np.mean(g["condition_match_hydrophobicity"].dropna().astype(float))) if g["condition_match_hydrophobicity"].notna().any() else np.nan,
            "plausibility_within_reference_rate": float(np.mean(g["plausibility_is_within_reference"].astype(float))) if len(g) else np.nan,
            "within_group_diversity": sample_pairwise_diversity(g["sequence"].tolist(), PAIRWISE_DIVERSITY_PER_CONDITION_PAIRS, seed=SEED) if len(g) >= 2 else np.nan,
            "cluster_count_070": cluster_n,
            "largest_cluster_fraction_070": largest_frac,
        })
    return pd.DataFrame(rows)

def summarise_plausibility(final_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_tag, g in final_df.groupby("model_tag", sort=False):
        vc = g["plausibility_flag"].value_counts(dropna=False)
        rows.append({
            "model_tag": model_tag,
            "n_final": len(g),
            "n_within_reference_range": int(vc.get("within_reference_range", 0)),
            "n_mild_outlier": int(vc.get("mild_outlier", 0)),
            "n_strong_outlier": int(vc.get("strong_outlier", 0)),
            "within_reference_rate": safe_div(vc.get("within_reference_range", 0), len(g)),
        })
    return pd.DataFrame(rows)

# =============================================================================
# FIGURES
# =============================================================================
def build_main_figure(summary_df: pd.DataFrame, per_condition_df: pd.DataFrame, audit_df: pd.DataFrame):
    plot_df = summary_df.set_index("model_tag").loc[
        [m for m in MODEL_ORDER if m in summary_df["model_tag"].tolist()]
    ].reset_index()

    fig = plt.figure(figsize=(16.6, 9.2))
    gs = GridSpec(
        2, 2, figure=fig,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.08, 1.0]
    )

    xs = np.arange(len(plot_df))

    ax_a = fig.add_subplot(gs[0, :])
    width = 0.11
    grouped_metrics = [
        ("exact_novel_vs_train_rate_mean", "Novel vs train"),
        ("exact_novel_vs_full_rate_mean", "Novel vs full"),
        ("plausibility_within_reference_rate", "Plausibility"),
    ]
    gxs = np.arange(len(grouped_metrics))
    for i, model_tag in enumerate(plot_df["model_tag"]):
        offsets = gxs + (i - (len(plot_df) - 1) / 2) * width
        vals = [plot_df.loc[plot_df["model_tag"] == model_tag, metric].iloc[0] for metric, _ in grouped_metrics]
        ax_a.bar(
            offsets, vals, width=width,
            color=COLORS[model_tag],
            edgecolor=COLORS["edge"],
            linewidth=0.6
        )
    ax_a.set_xticks(gxs)
    ax_a.set_xticklabels([lab for _, lab in grouped_metrics])
    ax_a.set_ylabel("Fraction")
    ax_a.set_ylim(0, 1.05)
    ax_a.set_title("Generation quality", pad=12)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "a", wide=True)

    ax_b = fig.add_subplot(gs[1, 0])
    vals = plot_df["nn_similarity_train_mean"].to_numpy(dtype=float)
    lows = plot_df["nn_similarity_train_ci_low"].to_numpy(dtype=float)
    highs = plot_df["nn_similarity_train_ci_high"].to_numpy(dtype=float)
    ax_b.bar(xs, vals, color=[COLORS[m] for m in plot_df["model_tag"]], edgecolor=COLORS["edge"], linewidth=0.8)
    mask = np.isfinite(vals) & np.isfinite(lows) & np.isfinite(highs)
    if np.any(mask):
        ax_b.errorbar(
            xs[mask], vals[mask],
            yerr=np.vstack([vals[mask] - lows[mask], highs[mask] - vals[mask]]),
            fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9
        )
    for x, y in zip(xs, vals):
        if pd.notnull(y):
            ax_b.text(x, y + 0.012, f"{y:.2f}", ha="center", va="bottom", fontsize=11.5, color=COLORS["text"])
    ax_b.set_xticks(xs)
    ax_b.set_xticklabels([])
    ax_b.set_ylabel("NN similarity")
    ax_b.set_ylim(0, max(0.24, np.nanmax(highs) * 1.85 if np.isfinite(highs).any() else 0.24))
    ax_b.set_title("Memorization profile", pad=12)
    ax_b.text(0.995, 0.96, "Lower is better", transform=ax_b.transAxes, ha="right", va="top", fontsize=12, color=COLORS["text"])
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "b")

    ax_c = fig.add_subplot(gs[1, 1])
    vals = plot_df["condition_match_combined_mean"].to_numpy(dtype=float)
    lows = plot_df["condition_match_combined_ci_low"].to_numpy(dtype=float)
    highs = plot_df["condition_match_combined_ci_high"].to_numpy(dtype=float)
    ax_c.bar(xs, vals, color=[COLORS[m] for m in plot_df["model_tag"]], edgecolor=COLORS["edge"], linewidth=0.8)
    mask = np.isfinite(vals) & np.isfinite(lows) & np.isfinite(highs)
    if np.any(mask):
        ax_c.errorbar(
            xs[mask], vals[mask],
            yerr=np.vstack([vals[mask] - lows[mask], highs[mask] - vals[mask]]),
            fmt="none", ecolor=COLORS["text"], capsize=3, linewidth=0.9
        )
    for x, y in zip(xs, vals):
        if pd.notnull(y):
            ax_c.text(x, y + 0.018, f"{y:.2f}", ha="center", va="bottom", fontsize=11.5, color=COLORS["text"])
    ax_c.set_xticks(xs)
    ax_c.set_xticklabels([])
    ax_c.set_ylabel("Combined condition match")
    ax_c.set_ylim(0, 1.05)
    ax_c.set_title("Condition heterogeneity", pad=12)
    style_axis(ax_c, "y")
    add_panel_label(ax_c, "c")

    fig.subplots_adjust(left=0.07, right=0.985, top=0.94, bottom=0.16, wspace=0.18, hspace=0.34)
    add_shared_legend(fig, plot_df["model_tag"].tolist(), y=0.03, ncol=5)
    save_figure(fig, FIG_MAIN_DIR / "Fig7_step7_main_restyled")

def build_supp_similarity_figure(final_df: pd.DataFrame, summary_df: pd.DataFrame):
    plot_df = summary_df.set_index("model_tag").loc[
        [m for m in MODEL_ORDER if m in summary_df["model_tag"].tolist()]
    ].reset_index()

    fig = plt.figure(figsize=(16.0, 6.8))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.15])

    ax_a = fig.add_subplot(gs[0, 0])
    labels = [f"≤{thr:.2f}" for thr in SIMILARITY_THRESHOLDS]
    xs = np.arange(len(labels))
    width = 0.14
    for i, model_tag in enumerate(plot_df["model_tag"]):
        vals = []
        for thr in SIMILARITY_THRESHOLDS:
            key = f"nn_similarity_train_leq_{str(thr).replace('.', 'p')}_rate"
            vals.append(plot_df.loc[plot_df["model_tag"] == model_tag, key].iloc[0])
        offsets = xs + (i - (len(plot_df) - 1) / 2) * width
        ax_a.bar(offsets, vals, width=width, color=COLORS[model_tag], edgecolor=COLORS["edge"], linewidth=0.6)
    ax_a.set_xticks(xs)
    ax_a.set_xticklabels(labels)
    ax_a.set_ylabel("Fraction")
    ax_a.set_ylim(0, 1.05)
    ax_a.set_title("Similarity threshold sensitivity", pad=10)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "a")

    ax_b = fig.add_subplot(gs[0, 1])
    bins = np.linspace(0, 1, 21)
    for model_tag, g in final_df.groupby("model_tag", sort=False):
        vals = g["nn_similarity_train"].dropna().to_numpy(dtype=float)
        if len(vals) == 0:
            continue
        hist, edges = np.histogram(vals, bins=bins, density=True)
        mids = 0.5 * (edges[:-1] + edges[1:])
        ax_b.plot(mids, hist, linewidth=2.2, color=COLORS[model_tag], label=MODEL_LABELS[model_tag])
    ax_b.set_xlabel("NN similarity to training set")
    ax_b.set_ylabel("Density")
    ax_b.set_title("Memorization-profile distribution", pad=10)
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right")
    add_panel_label(ax_b, "b", wide=True)

    fig.subplots_adjust(left=0.07, right=0.985, top=0.92, bottom=0.13, wspace=0.20)
    save_figure(fig, FIG_SUPP_DIR / "SuppFig_step7_similarity_restyled")

def build_supp_property_figure(final_df: pd.DataFrame, train_df: pd.DataFrame):
    fig = plt.figure(figsize=(16.2, 9.2))
    gs = GridSpec(2, 2, figure=fig)

    metrics = [
        ("length", "Length"),
        ("net_charge", "Net charge"),
        ("hydrophobicity", "Hydrophobicity"),
        ("entropy", "Sequence entropy"),
    ]
    panel_labels = ["a", "b", "c", "d"]

    for i, (metric, title) in enumerate(metrics):
        ax = fig.add_subplot(gs[i // 2, i % 2])

        ref_vals = train_df[metric].dropna().to_numpy(dtype=float)
        hist_ref, edges = np.histogram(ref_vals, bins=25, density=True)
        mids = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(mids, hist_ref, linewidth=2.3, color=COLORS["train_ref"], label="Train reference")

        for model_tag, g in final_df.groupby("model_tag", sort=False):
            vals = g[metric].dropna().to_numpy(dtype=float)
            if len(vals) == 0:
                continue
            hist, _ = np.histogram(vals, bins=edges, density=True)
            ax.plot(mids, hist, linewidth=1.9, color=COLORS[model_tag], label=MODEL_LABELS[model_tag])

        ax.set_xlabel(title)
        ax.set_ylabel("Density")
        ax.set_title(f"{title} distribution", pad=10)
        style_axis(ax, "y")
        if i == 0:
            ax.legend(frameon=False, loc="upper right", fontsize=11.5)
        add_panel_label(ax, panel_labels[i], wide=(i in [2, 3]))

    fig.subplots_adjust(left=0.07, right=0.985, top=0.94, bottom=0.10, wspace=0.20, hspace=0.28)
    save_figure(fig, FIG_SUPP_DIR / "SuppFig_step7_plausibility_restyled")

# =============================================================================
# REPORTING
# =============================================================================
def progression_flags(summary_df: pd.DataFrame) -> List[dict]:
    rows = []
    for _, row in summary_df.iterrows():
        reasons = []
        if pd.notna(row["valid_rate"]) and row["valid_rate"] < PASS_MIN_VALID_RATE:
            reasons.append("low_validity")
        if pd.notna(row["exact_novel_vs_train_rate_mean"]) and row["exact_novel_vs_train_rate_mean"] < PASS_MIN_EXACT_NOVEL_TRAIN:
            reasons.append("weak_exact_novelty_vs_train")
        if pd.notna(row["nn_similarity_train_mean"]) and row["nn_similarity_train_mean"] > PASS_MAX_NN_SIMILARITY:
            reasons.append("high_train_similarity")
        if pd.notna(row["within_set_diversity_mean"]) and row["within_set_diversity_mean"] < PASS_MIN_DIVERSITY:
            reasons.append("low_diversity")
        if pd.notna(row["condition_match_combined_mean"]) and row["condition_match_combined_mean"] < PASS_MIN_CONDITION_MATCH:
            reasons.append("weak_condition_consistency")

        rows.append({
            "model_tag": row["model_tag"],
            "status": "PASS" if len(reasons) == 0 else "CHECK",
            "reasons": reasons,
        })
    return rows

def write_reports(
    config: AuditConfig,
    input_manifest: dict,
    reference_manifest: dict,
    generated_paths_manifest: dict,
    train_df: pd.DataFrame,
    full_df: pd.DataFrame,
    raw_df: pd.DataFrame,
    audit_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    per_condition_df: pd.DataFrame,
    plausibility_df: pd.DataFrame,
    condition_schema: Optional[pd.DataFrame],
):
    flags = progression_flags(summary_df)

    lines = []
    lines.append("PepCVAE Step 7 Audit Report")
    lines.append("=" * 80)
    lines.append("")
    lines.append("Reference manifest")
    lines.append("-" * 80)
    lines.append(json.dumps(reference_manifest, indent=2))
    lines.append("")
    lines.append("Generated-input path manifest")
    lines.append("-" * 80)
    lines.append(json.dumps(generated_paths_manifest, indent=2))
    lines.append("")
    lines.append("Config")
    lines.append("-" * 80)
    for k, v in asdict(config).items():
        lines.append(f"{k}: {v}")
    lines.append("")
    lines.append("Input manifest")
    lines.append("-" * 80)
    lines.append(json.dumps(input_manifest, indent=2))
    lines.append("")
    lines.append("Reference sizes")
    lines.append("-" * 80)
    lines.append(f"Train reference size: {len(train_df)}")
    lines.append(f"Full corpus reference size: {len(full_df)}")
    lines.append("")
    lines.append("Raw generation size")
    lines.append("-" * 80)
    lines.append(f"Raw generated rows: {len(raw_df)}")
    lines.append("")
    lines.append("Filtering audit")
    lines.append("-" * 80)
    lines.append(audit_df.to_string(index=False))
    lines.append("")
    lines.append("Model summary")
    lines.append("-" * 80)
    cols = [
        "model_tag", "n_final", "valid_rate", "post_qc_retention_rate",
        "exact_novel_vs_train_rate_mean", "exact_novel_vs_full_rate_mean",
        "nn_similarity_train_mean", "within_set_diversity_mean",
        "condition_match_combined_mean", "plausibility_within_reference_rate"
    ]
    lines.append(summary_df[cols].to_string(index=False))
    lines.append("")

    if len(per_condition_df) > 0:
        lines.append("Per-condition summary (head)")
        lines.append("-" * 80)
        lines.append(per_condition_df.head(20).to_string(index=False))
        lines.append("")

    lines.append("Plausibility summary")
    lines.append("-" * 80)
    lines.append(plausibility_df.to_string(index=False))
    lines.append("")
    lines.append("Heuristic progression flags")
    lines.append("-" * 80)
    for f in flags:
        lines.append(f"{f['model_tag']}: {f['status']} | {', '.join(f['reasons']) if f['reasons'] else 'None'}")

    (REPORT_DIR / "step7_audit_report.txt").write_text("\n".join(lines), encoding="utf-8")

    summary_json = {
        "reference_manifest": reference_manifest,
        "generated_paths_manifest": generated_paths_manifest,
        "config": asdict(config),
        "input_manifest": input_manifest,
        "reference_sizes": {
            "train_reference_size": int(len(train_df)),
            "full_corpus_reference_size": int(len(full_df)),
        },
        "raw_generated_rows": int(len(raw_df)),
        "condition_schema_present": bool(condition_schema is not None),
        "progression_flags": flags,
    }
    write_json(REPORT_DIR / "step7_execution_summary.json", summary_json)

# =============================================================================
# MAIN
# =============================================================================
def save_config() -> AuditConfig:
    cfg = AuditConfig(
        seed=SEED,
        min_len=MIN_LEN,
        max_len=MAX_LEN,
        kmer_k=KMER_K,
        similarity_thresholds=SIMILARITY_THRESHOLDS,
        low_complexity_max_run_fraction=LOW_COMPLEXITY_MAX_RUN_FRACTION,
        low_complexity_min_entropy=LOW_COMPLEXITY_MIN_ENTROPY,
        bootstrap_n=BOOTSTRAP_N,
        pairwise_diversity_sample_pairs=PAIRWISE_DIVERSITY_SAMPLE_PAIRS,
        pairwise_diversity_per_condition_pairs=PAIRWISE_DIVERSITY_PER_CONDITION_PAIRS,
        plausibility_z_mild=PLAUSIBILITY_Z_MILD,
        plausibility_z_strong=PLAUSIBILITY_Z_STRONG,
        export_png=EXPORT_PNG,
        export_pdf=EXPORT_PDF,
        export_tiff=EXPORT_TIFF,
        dpi_png=DPI_PNG,
        dpi_tiff=DPI_TIFF,
        tiff_compression=TIFF_COMPRESSION,
        input_mode=INPUT_MODE,
    )
    write_json(LOG_DIR / "step7_config.json", asdict(cfg))
    return cfg

def save_previous_step_metadata_snapshot():
    snapshot = {}
    if STEP6_RUN_METADATA_JSON.exists():
        snapshot["step6_run_metadata_json"] = str(STEP6_RUN_METADATA_JSON)
    if STEP6_CONFIG_JSON.exists():
        snapshot["step6_config_json"] = str(STEP6_CONFIG_JSON)
    write_json(LOG_DIR / "previous_step_metadata_snapshot.json", snapshot)

def main():
    print("Running PepCVAE Step 7 revised: generation, novelty auditing, diversity, plausibility")

    config = save_config()
    save_previous_step_metadata_snapshot()

    train_df_raw, full_df_raw, reference_manifest = discover_reference_tables()
    write_json(LOG_DIR / "step7_reference_manifest.json", reference_manifest)

    print(f"Using discovered train reference source: {reference_manifest['train_reference_source']}")
    print(f"Using discovered full corpus source: {reference_manifest['full_corpus_reference_source']}")

    condition_schema = discover_condition_schema()

    generated_paths = discover_generated_input_files()
    generated_paths_manifest = {k: (str(v) if v is not None else None) for k, v in generated_paths.items()}
    write_json(LOG_DIR / "step7_generated_paths_manifest.json", generated_paths_manifest)

    raw_generated_df, input_manifest = load_generated_inputs(generated_paths)
    raw_generated_df.to_csv(TABLE_DIR / "step7_generated_raw_sequences.csv", index=False)
    write_json(LOG_DIR / "step7_input_manifest.json", input_manifest)

    train_df = annotate_descriptors(train_df_raw)
    full_df = annotate_descriptors(full_df_raw)
    raw_generated_df = annotate_descriptors(raw_generated_df)

    train_df.to_csv(TABLE_DIR / "step7_train_reference_annotated.csv", index=False)
    full_df.to_csv(TABLE_DIR / "step7_full_corpus_reference_annotated.csv", index=False)

    qc_df, audit_df = run_qc_pipeline(raw_generated_df)
    audit_df.to_csv(TABLE_DIR / "step7_filtering_audit.csv", index=False)
    qc_df.to_csv(TABLE_DIR / "step7_generated_qc_filtered.csv", index=False)

    if len(qc_df) == 0:
        raise RuntimeError("All generated sequences were removed during QC. Step 7 cannot proceed.")

    train_set = set(train_df["sequence"].tolist())
    full_set = set(full_df["sequence"].tolist())
    exact_df = run_exact_novelty(qc_df, train_set, full_set)
    exact_df.to_csv(TABLE_DIR / "step7_generated_exact_novelty.csv", index=False)

    sim_df = run_similarity_audit(exact_df, train_df, full_df)
    sim_df.to_csv(TABLE_DIR / "step7_generated_similarity_audit.csv", index=False)

    cond_df = run_condition_consistency(sim_df, condition_schema)
    cond_df.to_csv(TABLE_DIR / "step7_generated_condition_consistency.csv", index=False)

    train_ref_stats = build_train_reference_stats(train_df)
    train_ref_stats.to_csv(TABLE_DIR / "step7_train_reference_stats.csv", index=False)

    # Hard guard before plausibility screening
    assert_no_duplicate_columns(cond_df, "cond_df before run_plausibility_screen")

    final_df = run_plausibility_screen(cond_df, train_ref_stats)
    final_df.to_csv(TABLE_DIR / "step7_generated_final_audited.csv", index=False)

    summary_df = summarise_by_model(final_df, audit_df)
    per_condition_df = summarise_by_condition(final_df)
    plausibility_df = summarise_plausibility(final_df)

    summary_df.to_csv(TABLE_DIR / "step7_model_summary.csv", index=False)
    per_condition_df.to_csv(TABLE_DIR / "step7_per_condition_summary.csv", index=False)
    plausibility_df.to_csv(TABLE_DIR / "step7_plausibility_summary.csv", index=False)

    candidate_cols = [
        "sequence", "model_tag", "requested_condition_id", "sample_id", "seed", "decode_temperature",
        "length", "net_charge", "hydrophobicity", "entropy",
        "hydrophobic_fraction", "polar_fraction", "positive_fraction", "negative_fraction",
        "aromatic_fraction", "aliphatic_fraction",
        "exact_novel_vs_train", "exact_novel_vs_full",
        "nn_similarity_train", "nn_train_sequence",
        "nn_similarity_full_corpus", "nn_full_corpus_sequence",
        "condition_match_length", "condition_match_charge", "condition_match_hydrophobicity", "condition_match_combined",
        "plausibility_flag", "plausibility_outlier_metric_count", "plausibility_is_within_reference",
    ]
    candidate_cols = [c for c in candidate_cols if c in final_df.columns]
    final_df[candidate_cols].to_csv(TABLE_DIR / "step7_candidates_for_step8.csv", index=False)

    summary_df.to_csv(SOURCE_DATA_DIR / "Fig7_step7_main_source_data.csv", index=False)
    per_condition_df.to_csv(SOURCE_DATA_DIR / "Fig7_step7_per_condition_source_data.csv", index=False)
    final_df.to_csv(SOURCE_DATA_DIR / "SuppFig_step7_similarity_source_data.csv", index=False)
    train_df.to_csv(SOURCE_DATA_DIR / "SuppFig_step7_train_reference_distribution_source_data.csv", index=False)

    build_main_figure(summary_df, per_condition_df, audit_df)
    build_supp_similarity_figure(final_df, summary_df)
    build_supp_property_figure(final_df, train_df)

    write_reports(
        config=config,
        input_manifest=input_manifest,
        reference_manifest=reference_manifest,
        generated_paths_manifest=generated_paths_manifest,
        train_df=train_df,
        full_df=full_df,
        raw_df=raw_generated_df,
        audit_df=audit_df,
        summary_df=summary_df,
        per_condition_df=per_condition_df,
        plausibility_df=plausibility_df,
        condition_schema=condition_schema,
    )

    print("Step 7 revised complete.")
    print(f"Tables: {TABLE_DIR}")
    print(f"Main figures: {FIG_MAIN_DIR}")
    print(f"Supplementary figures: {FIG_SUPP_DIR}")
    print(f"TIFF figures: {FIG_TIFF_DIR}")
    print(f"Source data: {SOURCE_DATA_DIR}")
    print(f"Reports: {REPORT_DIR}")
    print(f"Logs: {LOG_DIR}")

if __name__ == "__main__":
    main()

# STEP 7 — REFERENCE-STYLE REDESIGN (UPGRADED)
# More closely aligned to the uploaded reference figure language

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# =============================================================================
# STEP 7 — REFERENCE-STYLE REDESIGN (UPGRADED)
# More closely aligned to the uploaded reference figure language
# =============================================================================

# =============================================================================
# USER CONFIGURATION
# =============================================================================
STEP7_ROOT = Path("/home/data3/Moe/nature_computational2/step7_v3").resolve()

FALLBACK_SEARCH_ROOTS = [
    STEP7_ROOT,
    STEP7_ROOT / "outputs",
    STEP7_ROOT / "outputs" / "tables",
    Path.cwd(),
    Path("/mnt/data"),
]

EXPORT_PDF = True
EXPORT_PNG = True
EXPORT_TIFF = True
DPI_PNG = 300
DPI_TIFF = 600
TIFF_COMPRESSION = "tiff_lzw"

FIG_MAIN_DIR = STEP7_ROOT / "outputs" / "figures_main_refstyle"
FIG_SUPP_DIR = STEP7_ROOT / "outputs" / "figures_supplementary_refstyle"
FIG_TIFF_DIR = STEP7_ROOT / "outputs" / "figures_tiff_refstyle"
SOURCE_DATA_DIR = STEP7_ROOT / "outputs" / "source_data_refstyle"

for d in [FIG_MAIN_DIR, FIG_SUPP_DIR, FIG_TIFF_DIR, SOURCE_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# MODEL ORDER / LABELS
# =============================================================================
MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

# =============================================================================
# REFERENCE-LIKE COLOR LANGUAGE
# Inspired by the uploaded reference figure family
# =============================================================================
COLORS = {
    "full_cvae": "#D1001F",          # strong crimson
    "conditional_gru_lm": "#FF1A1A", # bright red
    "random_condition": "#1B6E73",   # teal
    "no_condition": "#27467A",       # navy
    "small_latent": "#8E0000",       # dark maroon

    "train_ref": "#A6A6A6",
    "grid": "#E3E3E3",
    "spine": "#BDBDBD",
    "text": "#222222",
    "bg": "#FFFFFF",

    "within_ref": "#7F7F7F",
    "mild_outlier": "#C9C9C9",
    "strong_outlier": "#4D4D4D",
}

# =============================================================================
# MATPLOTLIB STYLE
# =============================================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 11,
    "figure.titlesize": 15,
    "axes.facecolor": COLORS["bg"],
    "figure.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
})

# =============================================================================
# FILE DISCOVERY
# =============================================================================
def discover_file(candidate_names: List[str], required: bool = True) -> Optional[Path]:
    matches = []
    for root in FALLBACK_SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in candidate_names:
            for p in root.rglob(name):
                matches.append(p.resolve())
    matches = sorted(set(matches))
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"Could not find any of: {candidate_names}")
    return None

def load_step7_tables() -> Dict[str, Optional[pd.DataFrame]]:
    paths = {
        "summary": discover_file(["step7_model_summary.csv"], required=True),
        "plausibility": discover_file(["step7_plausibility_summary.csv"], required=True),
        "per_condition": discover_file(["step7_per_condition_summary.csv"], required=False),
        "final": discover_file(["step7_generated_final_audited.csv"], required=False),
        "train_ref": discover_file(["step7_train_reference_annotated.csv"], required=False),
    }

    dfs = {}
    for key, path in paths.items():
        dfs[key] = pd.read_csv(path) if path is not None else None

    print("Resolved inputs:")
    for key, path in paths.items():
        print(f"  {key}: {path}")
    return dfs

# =============================================================================
# HELPERS
# =============================================================================
def save_figure(fig: plt.Figure, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=DPI_PNG)
    if EXPORT_TIFF:
        tiff_base = FIG_TIFF_DIR / out_base.name
        try:
            fig.savefig(
                tiff_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
                pil_kwargs={"compression": TIFF_COMPRESSION},
            )
        except TypeError:
            fig.savefig(
                tiff_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
            )
    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_axisbelow(True)

    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.9)
    else:
        ax.grid(True, color=COLORS["grid"], linewidth=0.9)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.0)
        ax.spines[side].set_color(COLORS["spine"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(axis="both", width=0.8, colors=COLORS["text"])

def add_panel_label(ax, label: str, x: float = -0.10, y: float = 1.03):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        va="bottom",
        ha="left",
        color=COLORS["text"],
        clip_on=False,
    )

def smooth_density_from_hist(values: np.ndarray, bins: int = 36, sigma: float = 1.2) -> Tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.array([]), np.array([])

    hist, edges = np.histogram(values, bins=bins, density=True)
    mids = 0.5 * (edges[:-1] + edges[1:])

    radius = int(max(3, np.ceil(3 * sigma)))
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-(x ** 2) / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    smoothed = np.convolve(hist, kernel, mode="same")
    return mids, smoothed

def ecdf(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.array([]), np.array([])
    arr = np.sort(arr)
    y = np.arange(1, len(arr) + 1) / len(arr)
    return arr, y

def ordered_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["__order__"] = out["model_tag"].map({m: i for i, m in enumerate(MODEL_ORDER)})
    out = out.sort_values("__order__").drop(columns="__order__").reset_index(drop=True)
    return out

def add_marker_with_stem(ax, x: float, y_center: float, y_top: float, marker_size: int = 80):
    ax.vlines(x, y_center, y_top, color="black", linewidth=1.0, zorder=4)
    ax.scatter(
        [x], [y_center],
        s=marker_size,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        zorder=5,
    )

def draw_violin_panel(
    ax,
    data_list: List[np.ndarray],
    labels: List[str],
    colors: List[str],
    ylabel: str,
    title: str,
    ylim: Optional[Tuple[float, float]] = None,
    panel_label: Optional[str] = None,
):
    positions = np.arange(len(data_list))

    vp = ax.violinplot(
        data_list,
        positions=positions,
        widths=0.78,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, c in zip(vp["bodies"], colors):
        body.set_facecolor(c)
        body.set_edgecolor("none")
        body.set_alpha(0.95)

    for i, vals in enumerate(data_list):
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        center = float(np.mean(vals))
        top = float(np.percentile(vals, 95))
        add_marker_with_stem(ax, i, center, top, marker_size=72)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    if ylim is not None:
        ax.set_ylim(*ylim)
    style_axis(ax, "y")
    if panel_label:
        add_panel_label(ax, panel_label)

def make_reference_style_legend_figure(model_tags: List[str], out_base: Path):
    fig = plt.figure(figsize=(10.0, 1.35))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")

    handles = [
        Patch(facecolor=COLORS[m], edgecolor="none", label=MODEL_LABELS[m])
        for m in model_tags
    ]
    leg = ax.legend(
        handles=handles,
        loc="center",
        ncol=min(3, len(handles)),
        frameon=True,
        columnspacing=1.8,
        handletextpad=0.6,
        borderpad=0.55,
        fontsize=12,
    )
    leg.get_frame().set_facecolor("#F9F9F9")
    leg.get_frame().set_edgecolor("#C7C7C7")
    leg.get_frame().set_linewidth(1.0)

    save_figure(fig, out_base)

# =============================================================================
# MAIN FIGURE
# More reference-like: one compact multipanel figure + separate legend strip
# =============================================================================
def build_main_figure_refstyle(summary_df: pd.DataFrame, plaus_df: pd.DataFrame, final_df: Optional[pd.DataFrame]):
    summary_df = ordered_df(summary_df)
    plaus_df = ordered_df(plaus_df)
    model_tags = summary_df["model_tag"].tolist()

    fig = plt.figure(figsize=(14.8, 7.8))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.0], width_ratios=[1.05, 1.0])

    # -------------------------------------------------------------------------
    # Panel a: novelty + plausibility summary as slim grouped bars
    # closer to reference grouped summary style
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[0, :])

    metric_names = ["Novel vs train", "Novel vs full", "Within reference"]
    metric_keys = [
        "exact_novel_vs_train_rate_mean",
        "exact_novel_vs_full_rate_mean",
        None,
    ]
    x_base = np.arange(len(metric_names))
    width = 0.13

    for i, model_tag in enumerate(model_tags):
        srow = summary_df.loc[summary_df["model_tag"] == model_tag].iloc[0]
        prow = plaus_df.loc[plaus_df["model_tag"] == model_tag].iloc[0]
        vals = [
            float(srow["exact_novel_vs_train_rate_mean"]),
            float(srow["exact_novel_vs_full_rate_mean"]),
            float(prow["within_reference_rate"]),
        ]
        xpos = x_base + (i - (len(model_tags) - 1) / 2) * width
        ax_a.bar(
            xpos,
            vals,
            width=width * 0.98,
            color=COLORS[model_tag],
            edgecolor="none",
            zorder=3,
        )

    ax_a.set_xticks(x_base)
    ax_a.set_xticklabels(metric_names)
    ax_a.set_ylabel("Fraction")
    ax_a.set_ylim(0.62, 1.03)
    ax_a.set_title("Generation quality audit", pad=10)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "a", x=-0.06, y=1.04)

    # -------------------------------------------------------------------------
    # Panel b: NN similarity violin
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[1, 0])

    if final_df is not None and "nn_similarity_train" in final_df.columns:
        data = [
            final_df.loc[final_df["model_tag"] == m, "nn_similarity_train"].dropna().to_numpy(dtype=float)
            for m in model_tags
        ]
        labels = [MODEL_LABELS[m] for m in model_tags]
        colors = [COLORS[m] for m in model_tags]
        draw_violin_panel(
            ax=ax_b,
            data_list=data,
            labels=labels,
            colors=colors,
            ylabel="NN similarity",
            title="Memorization profile",
            panel_label="b",
        )
        ax_b.text(
            0.99, 0.96, "Lower is better",
            transform=ax_b.transAxes,
            ha="right", va="top",
            fontsize=11, color=COLORS["text"]
        )
    else:
        vals = summary_df["nn_similarity_train_mean"].to_numpy(dtype=float)
        lo = summary_df["nn_similarity_train_ci_low"].to_numpy(dtype=float)
        hi = summary_df["nn_similarity_train_ci_high"].to_numpy(dtype=float)
        xs = np.arange(len(model_tags))
        ax_b.bar(xs, vals, color=[COLORS[m] for m in model_tags], edgecolor="none", width=0.78)
        ax_b.errorbar(xs, vals, yerr=np.vstack([vals - lo, hi - vals]), fmt="none", ecolor="black", capsize=3)
        ax_b.set_xticks(xs)
        ax_b.set_xticklabels([MODEL_LABELS[m] for m in model_tags])
        ax_b.set_ylabel("NN similarity")
        ax_b.set_title("Memorization profile", pad=8)
        style_axis(ax_b, "y")
        add_panel_label(ax_b, "b")

    # -------------------------------------------------------------------------
    # Panel c: plausibility composition stacked bars
    # -------------------------------------------------------------------------
    ax_c = fig.add_subplot(gs[1, 1])

    n_total = plaus_df["n_final"].to_numpy(dtype=float)
    within = plaus_df["n_within_reference_range"].to_numpy(dtype=float) / n_total
    mild = plaus_df["n_mild_outlier"].to_numpy(dtype=float) / n_total
    strong = plaus_df["n_strong_outlier"].to_numpy(dtype=float) / n_total
    xs = np.arange(len(model_tags))

    ax_c.bar(xs, within, color=COLORS["within_ref"], edgecolor="none", width=0.78, label="Within reference")
    ax_c.bar(xs, mild, bottom=within, color=COLORS["mild_outlier"], edgecolor="none", width=0.78, label="Mild outlier")
    ax_c.bar(xs, strong, bottom=within + mild, color=COLORS["strong_outlier"], edgecolor="none", width=0.78, label="Strong outlier")

    ax_c.set_xticks(xs)
    ax_c.set_xticklabels([MODEL_LABELS[m] for m in model_tags])
    ax_c.set_ylabel("Fraction")
    ax_c.set_ylim(0, 1.02)
    ax_c.set_title("Plausibility composition", pad=8)
    style_axis(ax_c, "y")
    add_panel_label(ax_c, "c")

    # compact local legend for composition
    comp_handles = [
        Patch(facecolor=COLORS["within_ref"], edgecolor="none", label="Within reference"),
        Patch(facecolor=COLORS["mild_outlier"], edgecolor="none", label="Mild outlier"),
        Patch(facecolor=COLORS["strong_outlier"], edgecolor="none", label="Strong outlier"),
    ]
    leg = ax_c.legend(
        handles=comp_handles,
        loc="upper right",
        frameon=True,
        fontsize=10.5,
    )
    leg.get_frame().set_facecolor("#FAFAFA")
    leg.get_frame().set_edgecolor("#C7C7C7")
    leg.get_frame().set_linewidth(0.9)

    fig.subplots_adjust(left=0.06, right=0.985, top=0.93, bottom=0.12, wspace=0.16, hspace=0.38)

    source = summary_df.merge(
        plaus_df[[
            "model_tag",
            "n_final",
            "n_within_reference_range",
            "n_mild_outlier",
            "n_strong_outlier",
            "within_reference_rate"
        ]],
        on="model_tag",
        how="left"
    )
    source.to_csv(SOURCE_DATA_DIR / "Fig7_step7_main_refstyle_source_data.csv", index=False)

    save_figure(fig, FIG_MAIN_DIR / "Fig7_step7_main_refstyle")
    make_reference_style_legend_figure(model_tags, FIG_MAIN_DIR / "Fig7_step7_main_refstyle_legend")

# =============================================================================
# SUPPLEMENTARY FIGURE 1 — PROPERTY SUPPORT
# Reference-like: 4 clean panels, smoother, shared legend strip
# =============================================================================
def build_property_support_refstyle(final_df: Optional[pd.DataFrame], train_df: Optional[pd.DataFrame]):
    if final_df is None or train_df is None:
        warnings.warn("Skipping property support figure because final_df or train_df is unavailable.")
        return

    needed_final = {"model_tag", "length", "net_charge", "hydrophobicity", "entropy"}
    needed_train = {"length", "net_charge", "hydrophobicity", "entropy"}
    if not needed_final.issubset(final_df.columns) or not needed_train.issubset(train_df.columns):
        warnings.warn("Skipping property support figure because required columns are missing.")
        return

    fig = plt.figure(figsize=(15.0, 8.8))
    gs = GridSpec(2, 2, figure=fig)

    panels = [
        ("length", "Length", "a"),
        ("net_charge", "Net charge", "b"),
        ("hydrophobicity", "Hydrophobicity", "c"),
        ("entropy", "Sequence entropy", "d"),
    ]

    for idx, (col, xlabel, letter) in enumerate(panels):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])

        x_ref, y_ref = smooth_density_from_hist(train_df[col].dropna().to_numpy(dtype=float), bins=36, sigma=1.25)
        if len(x_ref):
            ax.plot(x_ref, y_ref, color=COLORS["train_ref"], linewidth=2.4, label="Train reference", zorder=4)

        for m in MODEL_ORDER:
            vals = final_df.loc[final_df["model_tag"] == m, col].dropna().to_numpy(dtype=float)
            x, y = smooth_density_from_hist(vals, bins=36, sigma=1.25)
            if len(x):
                ax.plot(x, y, color=COLORS[m], linewidth=1.8, alpha=0.97, zorder=3)

        ax.set_xlabel(xlabel)
        ax.set_ylabel("Density")
        ax.set_title(f"{xlabel} support", pad=8)
        style_axis(ax, "y")
        add_panel_label(ax, letter)

    fig.subplots_adjust(left=0.07, right=0.985, top=0.94, bottom=0.11, wspace=0.22, hspace=0.28)

    final_df.to_csv(SOURCE_DATA_DIR / "SuppFig_step7_property_refstyle_generated_source_data.csv", index=False)
    train_df.to_csv(SOURCE_DATA_DIR / "SuppFig_step7_property_refstyle_train_source_data.csv", index=False)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_step7_property_refstyle")

    # separate legend strip
    fig_leg = plt.figure(figsize=(10.5, 1.4))
    ax_leg = fig_leg.add_axes([0, 0, 1, 1])
    ax_leg.axis("off")
    handles = [Line2D([0], [0], color=COLORS["train_ref"], lw=2.4, label="Train reference")]
    handles += [Line2D([0], [0], color=COLORS[m], lw=2.0, label=MODEL_LABELS[m]) for m in MODEL_ORDER]
    leg = ax_leg.legend(
        handles=handles,
        loc="center",
        ncol=3,
        frameon=True,
        columnspacing=1.6,
        handletextpad=0.6,
        fontsize=12,
    )
    leg.get_frame().set_facecolor("#F9F9F9")
    leg.get_frame().set_edgecolor("#C7C7C7")
    leg.get_frame().set_linewidth(1.0)
    save_figure(fig_leg, FIG_SUPP_DIR / "SuppFig_step7_property_refstyle_legend")

# =============================================================================
# SUPPLEMENTARY FIGURE 2 — SIMILARITY DIAGNOSTICS
# Reference-like: distribution + compact threshold/tail summary
# =============================================================================
def build_similarity_refstyle(final_df: Optional[pd.DataFrame], summary_df: pd.DataFrame):
    if final_df is None or "nn_similarity_train" not in final_df.columns:
        warnings.warn("Skipping similarity figure because final_df with nn_similarity_train is unavailable.")
        return

    summary_df = ordered_df(summary_df)
    model_tags = summary_df["model_tag"].tolist()

    fig = plt.figure(figsize=(14.6, 6.4))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.15, 1.0])

    # -------------------------------------------------------------------------
    # Panel a: density profile
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[0, 0])

    for m in model_tags:
        vals = final_df.loc[final_df["model_tag"] == m, "nn_similarity_train"].dropna().to_numpy(dtype=float)
        x, y = smooth_density_from_hist(vals, bins=36, sigma=1.15)
        if len(x):
            ax_a.plot(x, y, color=COLORS[m], linewidth=2.0)

    ax_a.set_xlabel("NN similarity to training set")
    ax_a.set_ylabel("Density")
    ax_a.set_title("Memorization-profile distribution", pad=8)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "a")

    # -------------------------------------------------------------------------
    # Panel b: stricter tail-risk grouped bars
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[0, 1])

    thresholds = [0.10, 0.15, 0.20, 0.30]
    x_base = np.arange(len(thresholds))
    width = 0.13

    tail_rows = []
    for i, m in enumerate(model_tags):
        vals = final_df.loc[final_df["model_tag"] == m, "nn_similarity_train"].dropna().to_numpy(dtype=float)
        fracs = [float(np.mean(vals > thr)) if len(vals) else np.nan for thr in thresholds]
        xpos = x_base + (i - (len(model_tags) - 1) / 2) * width
        ax_b.bar(
            xpos,
            fracs,
            width=width * 0.98,
            color=COLORS[m],
            edgecolor="none",
            zorder=3,
        )
        for thr, frac in zip(thresholds, fracs):
            tail_rows.append({"model_tag": m, "threshold": thr, "fraction_above_threshold": frac})

    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels([f">{thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction")
    ax_b.set_title("High-similarity tail risk", pad=8)
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "b")

    fig.subplots_adjust(left=0.08, right=0.985, top=0.92, bottom=0.12, wspace=0.22)

    final_df[["model_tag", "nn_similarity_train"]].to_csv(
        SOURCE_DATA_DIR / "SuppFig_step7_similarity_refstyle_distribution_source_data.csv",
        index=False,
    )
    pd.DataFrame(tail_rows).to_csv(
        SOURCE_DATA_DIR / "SuppFig_step7_similarity_refstyle_tail_source_data.csv",
        index=False,
    )

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_step7_similarity_refstyle")
    make_reference_style_legend_figure(model_tags, FIG_SUPP_DIR / "SuppFig_step7_similarity_refstyle_legend")

# =============================================================================
# DRIVER
# =============================================================================
def main():
    tables = load_step7_tables()

    summary_df = tables["summary"]
    plaus_df = tables["plausibility"]
    final_df = tables["final"]
    train_df = tables["train_ref"]

    if summary_df is None or plaus_df is None:
        raise RuntimeError("Required summary tables are missing.")

    summary_df = summary_df.loc[summary_df["model_tag"].isin(MODEL_ORDER)].copy()
    plaus_df = plaus_df.loc[plaus_df["model_tag"].isin(MODEL_ORDER)].copy()

    if final_df is not None and "model_tag" in final_df.columns:
        final_df = final_df.loc[final_df["model_tag"].isin(MODEL_ORDER)].copy()

    print("\nBuilding main figure...")
    build_main_figure_refstyle(summary_df, plaus_df, final_df)

    print("Building property-support supplementary figure...")
    build_property_support_refstyle(final_df, train_df)

    print("Building similarity supplementary figure...")
    build_similarity_refstyle(final_df, summary_df)

    print("\nDone.")
    print(f"Main figures: {FIG_MAIN_DIR}")
    print(f"Supplementary figures: {FIG_SUPP_DIR}")
    print(f"TIFF figures: {FIG_TIFF_DIR}")
    print(f"Source data: {SOURCE_DATA_DIR}")

if __name__ == "__main__":
    main()

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# =============================================================================
# STEP 7 — FINAL POLISHED FIGURE CODE
# Only two chart types: violin + bar
# =============================================================================

# =============================================================================
# USER CONFIGURATION
# =============================================================================
STEP7_ROOT = Path("/home/data3/Moe/nature_computational2/step7_v3").resolve()

SEARCH_ROOTS = [
    STEP7_ROOT,
    STEP7_ROOT / "outputs",
    STEP7_ROOT / "outputs" / "tables",
    Path.cwd(),
    Path("/mnt/data"),
]

EXPORT_PNG = True
EXPORT_PDF = True
EXPORT_TIFF = True
DPI_PNG = 300
DPI_TIFF = 600
TIFF_COMPRESSION = "tiff_lzw"

FIG_MAIN_DIR = STEP7_ROOT / "outputs" / "figures_main"
FIG_SUPP_DIR = STEP7_ROOT / "outputs" / "figures_supplementary"
SOURCE_DATA_DIR = STEP7_ROOT / "outputs" / "source_data"

for d in [FIG_MAIN_DIR, FIG_SUPP_DIR, SOURCE_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# MODEL ORDER / LABELS / COLORS
# =============================================================================
MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "CVAE-cond",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

COLORS = {
    "full_cvae": "#D1001F",          # deep crimson
    "conditional_gru_lm": "#FF7F0E", # warm gold
    "random_condition": "#238383",   # teal
    "no_condition": "#35558F",       # navy
    "small_latent": "#A40000",       # deep red-maroon
    "train_ref": "#9A9A9A",

    "bg": "#FFFFFF",
    "grid": "#E5E5E5",
    "spine": "#BBBBBB",
    "text": "#222222",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12.5,
    "ytick.labelsize": 12.5,
    "legend.fontsize": 14,
    "figure.titlesize": 18,
    "axes.facecolor": COLORS["bg"],
    "figure.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
})

# =============================================================================
# FILE DISCOVERY / LOADING
# =============================================================================
def discover_file(names: List[str], required: bool = True) -> Optional[Path]:
    found = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in names:
            found.extend(root.rglob(name))
    found = sorted({p.resolve() for p in found})
    if found:
        return found[0]
    if required:
        raise FileNotFoundError(f"Could not find any of: {names}")
    return None

def load_tables() -> Dict[str, Optional[pd.DataFrame]]:
    paths = {
        "summary": discover_file(["step7_model_summary.csv"], required=True),
        "final": discover_file(["step7_generated_final_audited.csv"], required=False),
        "train_ref": discover_file(["step7_train_reference_annotated.csv"], required=False),
    }

    dfs = {}
    for key, path in paths.items():
        dfs[key] = pd.read_csv(path) if path is not None else None

    print("Resolved inputs:")
    for k, v in paths.items():
        print(f"  {k}: {v}")
    return dfs

# =============================================================================
# HELPERS
# =============================================================================
def ordered_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["__order__"] = out["model_tag"].map({m: i for i, m in enumerate(MODEL_ORDER)})
    out = out.sort_values("__order__").drop(columns="__order__").reset_index(drop=True)
    return out

def save_figure(fig: plt.Figure, out_base: Path):
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=DPI_PNG)
    if EXPORT_TIFF:
        try:
            fig.savefig(
                out_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
                pil_kwargs={"compression": TIFF_COMPRESSION},
            )
        except TypeError:
            fig.savefig(
                out_base.with_suffix(".tiff"),
                bbox_inches="tight",
                dpi=DPI_TIFF,
            )
    plt.close(fig)

def style_axis(ax, grid_axis: str = "y"):
    ax.set_axisbelow(True)

    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=1.0)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=1.0)
    elif grid_axis == "both":
        ax.grid(True, color=COLORS["grid"], linewidth=1.0)

    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.0)
        ax.spines[side].set_color(COLORS["spine"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=0.9, colors=COLORS["text"])

def add_panel_label(ax, label: str, x: float = -0.08, y: float = 1.03):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=22,
        fontweight="bold",
        va="bottom",
        ha="left",
        color=COLORS["text"],
        clip_on=False,
    )

def bottom_legend(
    fig: plt.Figure,
    model_tags: List[str],
    y: float = 0.018,
    fontsize: float = 15,
    columnspacing: float = 1.45,
    handletextpad: float = 0.60,
    borderpad: float = 0.55,
):
    handles = [Patch(facecolor=COLORS[m], edgecolor="none", label=MODEL_LABELS[m]) for m in model_tags]
    leg = fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        ncol=len(handles),
        frameon=True,
        columnspacing=columnspacing,
        handletextpad=handletextpad,
        borderpad=borderpad,
        fontsize=fontsize,
    )
    leg.get_frame().set_facecolor("#F8F8F8")
    leg.get_frame().set_edgecolor("#C8C8C8")
    leg.get_frame().set_linewidth(1.0)

def bottom_legend_with_train(
    fig: plt.Figure,
    y: float = 0.022,
    fontsize: float = 13.7,
    columnspacing: float = 1.05,
    handletextpad: float = 0.46,
    borderpad: float = 0.50,
):
    handles = [Patch(facecolor=COLORS["train_ref"], edgecolor="none", label="Train reference")]
    handles += [Patch(facecolor=COLORS[m], edgecolor="none", label=MODEL_LABELS[m]) for m in MODEL_ORDER]
    leg = fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        ncol=len(handles),
        frameon=True,
        columnspacing=columnspacing,
        handletextpad=handletextpad,
        borderpad=borderpad,
        fontsize=fontsize,
    )
    leg.get_frame().set_facecolor("#F8F8F8")
    leg.get_frame().set_edgecolor("#C8C8C8")
    leg.get_frame().set_linewidth(1.0)

def add_marker_with_stem(ax, x: float, center: float, upper: float, marker_size: int = 88):
    ax.vlines(x, center, upper, color="black", linewidth=0.95, zorder=4)
    ax.scatter(
        [x], [center],
        s=marker_size,
        facecolor="white",
        edgecolor="black",
        linewidth=1.25,
        zorder=5,
    )

def violin_panel(
    ax,
    data_list: List[np.ndarray],
    colors: List[str],
    ylabel: str,
    title: str,
    panel_label: str,
    ylim: Optional[Tuple[float, float]] = None,
    show_xtick_marks: bool = False,
    violin_width: float = 0.56,
    bw_method: float = 0.16,
):
    positions = np.arange(len(data_list))

    vp = ax.violinplot(
        data_list,
        positions=positions,
        widths=violin_width,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        bw_method=bw_method,
    )

    for body, color in zip(vp["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor("none")
        body.set_alpha(0.95)

    for i, vals in enumerate(data_list):
        arr = np.asarray(vals, dtype=float)
        arr = arr[np.isfinite(arr)]
        if len(arr) == 0:
            continue
        center = float(np.mean(arr))
        upper = float(np.percentile(arr, 95))
        add_marker_with_stem(ax, i, center, upper, marker_size=88)

    ax.set_xticks(positions)
    ax.set_xticklabels([])
    if not show_xtick_marks:
        ax.tick_params(axis="x", length=0)

    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=10)
    if ylim is not None:
        ax.set_ylim(*ylim)

    style_axis(ax, "y")
    add_panel_label(ax, panel_label)

# =============================================================================
# FIGURE 1 — MAIN FIGURE
# =============================================================================
def build_main_figure(summary_df: pd.DataFrame, final_df: Optional[pd.DataFrame]):
    summary_df = ordered_df(summary_df)
    model_tags = summary_df["model_tag"].tolist()

    fig = plt.figure(figsize=(16.0, 7.4))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[0.98, 1.02])

    # -------------------------------------------------------------------------
    # Panel a — grouped bars with 2 groups only
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[0, 0])

    metric_names = ["Novelty", "Within reference"]
    x_base = np.array([0.0, 1.72])
    bar_width = 0.20
    inner_gap = 0.035

    for i, model_tag in enumerate(model_tags):
        row = summary_df.loc[summary_df["model_tag"] == model_tag].iloc[0]
        vals = [
            float(row["exact_novel_vs_train_rate_mean"]),
            float(row["plausibility_within_reference_rate"]),
        ]

        offset = (i - (len(model_tags) - 1) / 2) * (bar_width + inner_gap)
        xpos = x_base + offset

        ax_a.bar(
            xpos,
            vals,
            width=bar_width,
            color=COLORS[model_tag],
            edgecolor="none",
            zorder=3,
        )

    ax_a.set_xticks(x_base)
    ax_a.set_xticklabels(metric_names)
    ax_a.set_ylabel("Fraction")
    ax_a.set_ylim(0.62, 1.03)
    ax_a.set_title("Generation audit", pad=12)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "a", x=-0.06, y=1.04)

    # -------------------------------------------------------------------------
    # Panel b — violin
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[0, 1])

    if final_df is not None and "nn_similarity_train" in final_df.columns:
        data = [
            final_df.loc[final_df["model_tag"] == m, "nn_similarity_train"].dropna().to_numpy(dtype=float)
            for m in model_tags
        ]
        violin_panel(
            ax=ax_b,
            data_list=data,
            colors=[COLORS[m] for m in model_tags],
            ylabel="NN similarity",
            title="Memorization profile",
            panel_label="b",
            show_xtick_marks=False,
            violin_width=0.56,
            bw_method=0.16,
        )
        ax_b.text(
            0.99, 0.96, "Lower is better",
            transform=ax_b.transAxes,
            ha="right", va="top",
            fontsize=14, color=COLORS["text"],
        )
    else:
        warnings.warn("final_df with nn_similarity_train not found.")
        ax_b.text(0.5, 0.5, "Detailed similarity table missing", ha="center", va="center", transform=ax_b.transAxes)
        style_axis(ax_b, "y")
        add_panel_label(ax_b, "b")

    fig.subplots_adjust(left=0.06, right=0.985, top=0.92, bottom=0.18, wspace=0.14)
    bottom_legend(fig, model_tags, y=0.025, fontsize=15, columnspacing=1.4, handletextpad=0.60, borderpad=0.55)

    summary_df.to_csv(SOURCE_DATA_DIR / "Fig7_step7_main_final_v6_source_data.csv", index=False)
    if final_df is not None:
        final_df[["model_tag", "nn_similarity_train"]].to_csv(
            SOURCE_DATA_DIR / "Fig7_step7_main_final_v6_similarity_source_data.csv",
            index=False,
        )

    save_figure(fig, FIG_MAIN_DIR / "Fig7_step7_main_final_v6")

# =============================================================================
# SUPPLEMENTARY FIGURE 1 — PROPERTY SUPPORT
# =============================================================================
def build_property_figure(final_df: Optional[pd.DataFrame], train_df: Optional[pd.DataFrame]):
    if final_df is None or train_df is None:
        warnings.warn("Skipping property figure because final_df or train_df is unavailable.")
        return

    required_final = {"model_tag", "length", "net_charge", "hydrophobicity", "entropy"}
    required_train = {"length", "net_charge", "hydrophobicity", "entropy"}
    if not required_final.issubset(final_df.columns) or not required_train.issubset(train_df.columns):
        warnings.warn("Skipping property figure because required columns are missing.")
        return

    fig = plt.figure(figsize=(16.0, 9.2))
    gs = GridSpec(2, 2, figure=fig)

    specs = [
        ("length", "Length support", "Length", "a"),
        ("net_charge", "Net charge support", "Net charge", "b"),
        ("hydrophobicity", "Hydrophobicity support", "Hydrophobicity", "c"),
        ("entropy", "Sequence entropy support", "Sequence entropy", "d"),
    ]

    for idx, (col, title, ylabel, letter) in enumerate(specs):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])

        data = [train_df[col].dropna().to_numpy(dtype=float)]
        colors = [COLORS["train_ref"]]

        for m in MODEL_ORDER:
            data.append(final_df.loc[final_df["model_tag"] == m, col].dropna().to_numpy(dtype=float))
            colors.append(COLORS[m])

        violin_panel(
            ax=ax,
            data_list=data,
            colors=colors,
            ylabel=ylabel,
            title=title,
            panel_label=letter,
            show_xtick_marks=False,
            violin_width=0.54,
            bw_method=0.15,
        )

    # Key final fix: bring legend much closer to plot block and reduce dead space.
    fig.subplots_adjust(left=0.06, right=0.985, top=0.94, bottom=0.125, wspace=0.20, hspace=0.30)
    bottom_legend_with_train(
        fig,
        y=0.032,
        fontsize=14.0,
        columnspacing=1.00,
        handletextpad=0.45,
        borderpad=0.48,
    )

    final_df.to_csv(SOURCE_DATA_DIR / "SuppFig_step7_property_generated_final_v6_source_data.csv", index=False)
    train_df.to_csv(SOURCE_DATA_DIR / "SuppFig_step7_property_train_reference_v6_source_data.csv", index=False)

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_step7_property_final_v6")

# =============================================================================
# SUPPLEMENTARY FIGURE 2 — SIMILARITY DIAGNOSTICS
# =============================================================================
def build_similarity_figure(final_df: Optional[pd.DataFrame]):
    if final_df is None or "nn_similarity_train" not in final_df.columns:
        warnings.warn("Skipping similarity diagnostics figure because final_df with nn_similarity_train is unavailable.")
        return

    fig = plt.figure(figsize=(15.2, 6.9))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.0])

    # -------------------------------------------------------------------------
    # Panel a — violin
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[0, 0])

    data = [
        final_df.loc[final_df["model_tag"] == m, "nn_similarity_train"].dropna().to_numpy(dtype=float)
        for m in MODEL_ORDER
    ]
    violin_panel(
        ax=ax_a,
        data_list=data,
        colors=[COLORS[m] for m in MODEL_ORDER],
        ylabel="NN similarity to training set",
        title="Similarity diagnostics",
        panel_label="a",
        show_xtick_marks=False,
        violin_width=0.56,
        bw_method=0.16,
    )

    # -------------------------------------------------------------------------
    # Panel b — grouped bars
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[0, 1])

    thresholds = [0.10, 0.15, 0.20, 0.30]
    x_base = np.array([0.0, 1.38, 2.76, 4.14])
    bar_width = 0.17
    inner_gap = 0.035

    tail_rows = []

    for i, m in enumerate(MODEL_ORDER):
        vals = final_df.loc[final_df["model_tag"] == m, "nn_similarity_train"].dropna().to_numpy(dtype=float)
        fracs = [float(np.mean(vals > thr)) if len(vals) else np.nan for thr in thresholds]

        offset = (i - (len(MODEL_ORDER) - 1) / 2) * (bar_width + inner_gap)
        xpos = x_base + offset

        ax_b.bar(
            xpos,
            fracs,
            width=bar_width,
            color=COLORS[m],
            edgecolor="none",
            zorder=3,
        )

        for thr, frac in zip(thresholds, fracs):
            tail_rows.append({
                "model_tag": m,
                "threshold": thr,
                "fraction_above_threshold": frac,
            })

    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels([f">{thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction")
    ax_b.set_title("High-similarity tail risk", pad=10)
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "b")

    fig.subplots_adjust(left=0.07, right=0.985, top=0.92, bottom=0.18, wspace=0.18)
    bottom_legend(fig, MODEL_ORDER, y=0.025, fontsize=15, columnspacing=1.4, handletextpad=0.60, borderpad=0.55)

    final_df[["model_tag", "nn_similarity_train"]].to_csv(
        SOURCE_DATA_DIR / "SuppFig_step7_similarity_distribution_v6_source_data.csv",
        index=False,
    )
    pd.DataFrame(tail_rows).to_csv(
        SOURCE_DATA_DIR / "SuppFig_step7_similarity_tail_v6_source_data.csv",
        index=False,
    )

    save_figure(fig, FIG_SUPP_DIR / "SuppFig_step7_similarity_final_v6")

# =============================================================================
# DRIVER
# =============================================================================
def main():
    tables = load_tables()

    summary_df = tables["summary"]
    final_df = tables["final"]
    train_df = tables["train_ref"]

    if summary_df is None:
        raise RuntimeError("Required Step 7 summary table is missing.")

    summary_df = summary_df.loc[summary_df["model_tag"].isin(MODEL_ORDER)].copy()

    if final_df is not None and "model_tag" in final_df.columns:
        final_df = final_df.loc[final_df["model_tag"].isin(MODEL_ORDER)].copy()

    print("\nBuilding main figure...")
    build_main_figure(summary_df, final_df)

    print("Building supplementary property figure...")
    build_property_figure(final_df, train_df)

    print("Building supplementary similarity figure...")
    build_similarity_figure(final_df)

    print("\nDone.")
    print(f"Main figures: {FIG_MAIN_DIR}")
    print(f"Supplementary figures: {FIG_SUPP_DIR}")
    print(f"Source data: {SOURCE_DATA_DIR}")

if __name__ == "__main__":
    main()

# STEP 8 — Publication-grade multi-objective prioritization and biological plausibility screening

In [ ]:
from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE — Step 8
# Multi-objective prioritization and biological plausibility screening
# Publication-grade, path-stable, notebook-ready
# =============================================================================

import json
import platform
import random
import hashlib
import warnings
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from itertools import combinations, groupby
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

# =============================================================================
# Global constants
# =============================================================================

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)

AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_TINY = set("AGSC")
AA_SMALL = set("AGSCVTNDP")

AA_PROPERTIES = {
    "A": {"hydro": 1.8, "mw": 89.09},
    "C": {"hydro": 2.5, "mw": 121.16},
    "D": {"hydro": -3.5, "mw": 133.10},
    "E": {"hydro": -3.5, "mw": 147.13},
    "F": {"hydro": 2.8, "mw": 165.19},
    "G": {"hydro": -0.4, "mw": 75.07},
    "H": {"hydro": -3.2, "mw": 155.16},
    "I": {"hydro": 4.5, "mw": 131.17},
    "K": {"hydro": -3.9, "mw": 146.19},
    "L": {"hydro": 3.8, "mw": 131.17},
    "M": {"hydro": 1.9, "mw": 149.21},
    "N": {"hydro": -3.5, "mw": 132.12},
    "P": {"hydro": -1.6, "mw": 115.13},
    "Q": {"hydro": -3.5, "mw": 146.15},
    "R": {"hydro": -4.5, "mw": 174.20},
    "S": {"hydro": -0.8, "mw": 105.09},
    "T": {"hydro": -0.7, "mw": 119.12},
    "V": {"hydro": 4.2, "mw": 117.15},
    "W": {"hydro": -0.9, "mw": 204.23},
    "Y": {"hydro": -1.3, "mw": 181.19},
}

# =============================================================================
# Figure style / project palette
# =============================================================================

PROJECT_COLORS = {
    "deep_crimson": "#D1001F",
    "warm_gold": "#FF7F0E",
    "teal": "#238383",
    "navy": "#35558F",
    "deep_red_maroon": "#A40000",
    "neutral_gray": "#9A9A9A",
    "bg": "#FFFFFF",
    "grid": "#E5E5E5",
    "spine": "#BBBBBB",
    "text": "#222222",
}

def apply_project_matplotlib_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PROJECT_COLORS["bg"],
        "axes.facecolor": PROJECT_COLORS["bg"],
        "savefig.facecolor": PROJECT_COLORS["bg"],
        "axes.edgecolor": PROJECT_COLORS["spine"],
        "axes.labelcolor": PROJECT_COLORS["text"],
        "xtick.color": PROJECT_COLORS["text"],
        "ytick.color": PROJECT_COLORS["text"],
        "text.color": PROJECT_COLORS["text"],
        "axes.titlecolor": PROJECT_COLORS["text"],
        "grid.color": PROJECT_COLORS["grid"],
        "axes.grid": True,
        "grid.linestyle": "-",
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    })

apply_project_matplotlib_style()

# =============================================================================
# Configuration
# =============================================================================

@dataclass
class FigureExportConfig:
    png_dpi: int = 300
    tif_dpi: int = 600
    export_pdf: bool = True
    export_png: bool = True
    export_tif: bool = True
    single_col_width_in: float = 3.5
    double_col_width_in: float = 7.1
    panel_fontsize: int = 10
    base_fontsize: int = 8
    title_fontsize: int = 8
    legend_fontsize: int = 7


@dataclass
class Step8Config:
    project_root: str = "/home/data3/Moe/nature_computational2/"
    step1_dir_candidates: Tuple[str, ...] = ("step1", "step_1", "Step1", "Step_1")
    step7_dir_candidates: Tuple[str, ...] = ("step7_v3", "step7", "step_7", "Step7", "Step_7")
    step8_dir_name: str = "step8"

    step1_reference_aliases: Tuple[str, ...] = (
        "step1_retained_dataset.csv",
        "step1_retained_corpus.csv",
        "step1_clean_retained_corpus.csv",
        "step1_clean_dataset.csv",
        "retained_dataset.csv",
        "retained_corpus.csv",
        "step1_reference.csv",
    )
    step7_generated_aliases: Tuple[str, ...] = (
        "step7_generated_candidates_clean.csv",
        "step7_generated_candidates_ranked.csv",
        "step7_generated_candidates.csv",
        "step7_candidate_pool.csv",
        "step7_novelty_audited_candidates.csv",
        "generated_candidates.csv",
    )

    sequence_col_candidates: Tuple[str, ...] = ("sequence", "peptide", "seq", "peptide_sequence")
    id_col_candidates: Tuple[str, ...] = ("generated_id", "candidate_id", "peptide_id", "id")
    merge_key_col_candidates: Tuple[str, ...] = ("sequence_sha256", "seq_sha256")

    exact_train_match_col_candidates: Tuple[str, ...] = (
        "is_exact_train_match", "train_exact_match", "exact_match_train", "in_train_reference"
    )
    train_nn_similarity_col_candidates: Tuple[str, ...] = (
        "train_nn_similarity", "nearest_train_similarity", "max_train_similarity", "train_jaccard_nn"
    )
    exact_full_match_col_candidates: Tuple[str, ...] = (
        "is_exact_full_match", "full_exact_match", "exact_match_full", "in_full_reference"
    )

    requested_length_bin_col_candidates: Tuple[str, ...] = (
        "requested_length_bin", "target_length_bin", "condition_length_bin", "length_bin"
    )
    requested_charge_bin_col_candidates: Tuple[str, ...] = (
        "requested_charge_bin", "target_charge_bin", "condition_charge_bin", "net_charge_bin"
    )
    requested_hydrophobicity_bin_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_bin", "target_hydrophobicity_bin",
        "condition_hydrophobicity_bin", "hydrophobicity_bin"
    )

    requested_length_min_col_candidates: Tuple[str, ...] = ("requested_length_min", "target_length_min")
    requested_length_max_col_candidates: Tuple[str, ...] = ("requested_length_max", "target_length_max")
    requested_charge_min_col_candidates: Tuple[str, ...] = ("requested_charge_min", "target_charge_min")
    requested_charge_max_col_candidates: Tuple[str, ...] = ("requested_charge_max", "target_charge_max")
    requested_hydrophobicity_min_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_min", "target_hydrophobicity_min"
    )
    requested_hydrophobicity_max_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_max", "target_hydrophobicity_max"
    )

    supportive_score_candidates: Tuple[str, ...] = (
        "acp_probability", "acp_score", "predicted_acp_probability",
        "prob_acp", "amp_probability", "supportive_model_score"
    )
    toxicity_score_candidates: Tuple[str, ...] = (
        "toxicity_probability", "toxicity_score", "predicted_toxicity"
    )
    hemolysis_score_candidates: Tuple[str, ...] = (
        "hemolysis_probability", "hemolysis_score", "predicted_hemolysis"
    )

    main_random_seed: int = 42
    bootstrap_iterations: int = 300

    min_length: int = 5
    max_length: int = 50
    low_complexity_entropy_threshold: float = 1.50
    max_homopolymer_run_fraction: float = 0.35
    max_single_residue_fraction: float = 0.60
    min_unique_residue_fraction: float = 0.25
    max_train_nn_similarity: float = 0.85
    require_not_in_train_exact: bool = True
    require_not_in_full_exact_if_available: bool = False

    similarity_kmer: int = 3
    shortlist_top_n: int = 24
    final_diverse_panel_n: int = 12
    similarity_threshold_for_clusters: float = 0.75
    max_per_similarity_cluster: int = 2

    reference_quantile_low: float = 0.02
    reference_quantile_high: float = 0.98

    descriptor_weight_map: Dict[str, float] = field(default_factory=lambda: {
        "length": 1.00,
        "mean_hydropathy": 1.00,
        "net_charge_proxy": 1.20,
        "hydrophobic_frac": 0.90,
        "positive_frac": 1.10,
        "negative_frac": 0.80,
        "aromatic_frac": 0.70,
        "entropy": 1.00,
        "max_run_fraction": 1.00,
    })

    ranking_weights: Dict[str, float] = field(default_factory=lambda: {
        "novelty_score": 0.25,
        "descriptor_plausibility_score": 0.25,
        "condition_fidelity_score": 0.20,
        "diversity_score": 0.15,
        "supportive_model_score": 0.15,
    })

    ranking_schemes: Dict[str, Dict[str, float]] = field(default_factory=lambda: {
        "primary": {
            "novelty_score": 0.25,
            "descriptor_plausibility_score": 0.25,
            "condition_fidelity_score": 0.20,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "novelty_heavy": {
            "novelty_score": 0.35,
            "descriptor_plausibility_score": 0.20,
            "condition_fidelity_score": 0.15,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "plausibility_heavy": {
            "novelty_score": 0.15,
            "descriptor_plausibility_score": 0.35,
            "condition_fidelity_score": 0.20,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "condition_heavy": {
            "novelty_score": 0.20,
            "descriptor_plausibility_score": 0.20,
            "condition_fidelity_score": 0.30,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "supportive_heavy": {
            "novelty_score": 0.15,
            "descriptor_plausibility_score": 0.20,
            "condition_fidelity_score": 0.15,
            "diversity_score": 0.10,
            "supportive_model_score": 0.40,
        },
    })

    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)

# =============================================================================
# Reproducibility and IO
# =============================================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def collect_environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp_utc": now_utc_iso(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
        "cwd": str(Path.cwd()),
    }

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.export_pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.export_tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)

def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

# =============================================================================
# Path and file discovery
# =============================================================================

def find_best_matching_directory(
    base_root: Path,
    candidate_names: Sequence[str],
    required_keywords: Sequence[str],
) -> Path:
    base_root = base_root.resolve()

    if not base_root.exists():
        raise FileNotFoundError(f"Base project root does not exist: {base_root}")

    for name in candidate_names:
        p = base_root / name
        if p.exists() and p.is_dir():
            return p

    candidates = []
    for p in base_root.rglob("*"):
        if not p.is_dir():
            continue
        path_lower = str(p).lower()
        name_lower = p.name.lower()

        score = 0
        for kw in required_keywords:
            if kw.lower() in name_lower:
                score += 3
            if kw.lower() in path_lower:
                score += 1

        if score > 0:
            depth = len(p.relative_to(base_root).parts)
            candidates.append((score, -depth, p))

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not find a matching directory under: {base_root}\n"
            f"Tried exact names: {list(candidate_names)}\n"
            f"Used keywords: {list(required_keywords)}"
        )

    candidates = sorted(candidates, reverse=True)
    return candidates[0][2]

def find_best_matching_csv(
    root_dir: Path,
    aliases: Sequence[str],
    required_keywords: Sequence[str],
    exclude_keywords: Optional[Sequence[str]] = None,
) -> Path:
    root_dir = root_dir.resolve()
    exclude_keywords = tuple(exclude_keywords or [])

    if not root_dir.exists():
        raise FileNotFoundError(f"Root directory does not exist: {root_dir}")

    alias_set = set(aliases)
    exact_matches = []
    for p in root_dir.rglob("*.csv"):
        if p.name in alias_set:
            exact_matches.append(p)

    if len(exact_matches) > 0:
        exact_matches = sorted(exact_matches, key=lambda p: p.stat().st_mtime, reverse=True)
        return exact_matches[0]

    candidates = []
    for p in root_dir.rglob("*.csv"):
        name_lower = p.name.lower()
        path_lower = str(p).lower()

        if any(ex_kw.lower() in path_lower for ex_kw in exclude_keywords):
            continue

        score = 0
        for kw in required_keywords:
            if kw.lower() in name_lower:
                score += 3
        for kw in required_keywords:
            if kw.lower() in path_lower:
                score += 1

        depth_penalty = len(p.relative_to(root_dir).parts)
        mtime = p.stat().st_mtime

        if score > 0:
            candidates.append((score, -depth_penalty, mtime, p))

    if len(candidates) == 0:
        searched = [str(p) for p in root_dir.rglob("*.csv")]
        raise FileNotFoundError(
            f"No suitable CSV file found under: {root_dir}\n"
            f"Expected aliases: {list(aliases)}\n"
            f"Keyword search used: {list(required_keywords)}\n"
            f"CSV files seen:\n- " + "\n- ".join(searched[:200])
        )

    candidates = sorted(candidates, reverse=True)
    return candidates[0][3]

def first_existing_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = False, label: str = "") -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not resolve required column for {label}. Tried: {list(candidates)}")
    return None

def resolve_exact_train_match_column(
    df: pd.DataFrame,
    exact_candidates: Sequence[str],
    nn_candidates: Sequence[str],
    derived_col_name: str = "_derived_exact_train_match_from_nn",
    atol: float = 1e-12,
) -> Tuple[str, bool]:
    exact_col = first_existing_column(df, exact_candidates, required=False)
    if exact_col is not None:
        vals = pd.to_numeric(df[exact_col], errors="coerce")
        if vals.isna().any():
            raise ValueError(f"{exact_col} contains missing/non-numeric values.")
        df[exact_col] = vals.astype(int)
        return exact_col, False

    nn_col = first_existing_column(df, nn_candidates, required=False)
    if nn_col is None:
        raise ValueError(
            "Could not resolve Step 7 exact-train-match information. "
            f"Tried exact columns: {list(exact_candidates)} "
            f"and NN columns: {list(nn_candidates)}"
        )

    nn_vals = pd.to_numeric(df[nn_col], errors="coerce")
    if nn_vals.isna().any():
        raise ValueError(f"{nn_col} contains missing/non-numeric values.")
    if ((nn_vals < 0) | (nn_vals > 1)).any():
        raise ValueError(f"{nn_col} must lie in [0, 1].")

    df[derived_col_name] = np.isclose(nn_vals, 1.0, atol=atol).astype(int)
    return derived_col_name, True

def resolve_exact_full_match_column(
    df: pd.DataFrame,
    exact_full_candidates: Sequence[str],
) -> Optional[str]:
    exact_full_col = first_existing_column(df, exact_full_candidates, required=False)
    if exact_full_col is None:
        return None
    vals = pd.to_numeric(df[exact_full_col], errors="coerce")
    if vals.isna().any():
        raise ValueError(f"{exact_full_col} contains missing/non-numeric values.")
    df[exact_full_col] = vals.astype(int)
    return exact_full_col

# =============================================================================
# Sequence utilities and descriptors
# =============================================================================

def clean_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def is_valid_peptide(seq: str, min_len: int = 1, max_len: Optional[int] = None) -> bool:
    if not seq:
        return False
    if any(ch not in AA_SET for ch in seq):
        return False
    if len(seq) < min_len:
        return False
    if max_len is not None and len(seq) > max_len:
        return False
    return True

def shannon_entropy(seq: str) -> float:
    if not seq:
        return 0.0
    counts = Counter(seq)
    probs = np.array(list(counts.values()), dtype=float) / len(seq)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, None))).sum())

def max_run_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    runs = [len(list(group)) for _, group in groupby(seq)]
    return float(max(runs) / len(seq))

def aa_fraction(seq: str, aa: str) -> float:
    return seq.count(aa) / len(seq) if seq else 0.0

def unique_residue_fraction(seq: str) -> float:
    return len(set(seq)) / len(seq) if seq else 0.0

def max_single_residue_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    counts = Counter(seq)
    return float(max(counts.values()) / len(seq))

def make_kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}

def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
    s1 = make_kmers(seq1, k=k)
    s2 = make_kmers(seq2, k=k)
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    union = len(s1 | s2)
    if union == 0:
        return 0.0
    return len(s1 & s2) / union

def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    if not seq:
        base = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
        base.update(
            {
                "length": 0.0,
                "mean_hydropathy": 0.0,
                "hydrophobic_frac": 0.0,
                "aromatic_frac": 0.0,
                "positive_frac": 0.0,
                "negative_frac": 0.0,
                "polar_frac": 0.0,
                "tiny_frac": 0.0,
                "small_frac": 0.0,
                "net_charge_proxy": 0.0,
                "molecular_weight_proxy": 0.0,
                "entropy": 0.0,
                "max_run_fraction": 1.0,
                "unique_residue_fraction": 0.0,
                "max_single_residue_fraction": 1.0,
            }
        )
        return base

    hydros = np.array([AA_PROPERTIES[a]["hydro"] for a in seq], dtype=float)
    weights = np.array([AA_PROPERTIES[a]["mw"] for a in seq], dtype=float)

    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    desc.update(
        {
            "length": float(len(seq)),
            "mean_hydropathy": float(hydros.mean()),
            "hydrophobic_frac": float(np.mean([a in AA_HYDROPHOBIC for a in seq])),
            "aromatic_frac": float(np.mean([a in AA_AROMATIC for a in seq])),
            "positive_frac": float(np.mean([a in AA_POSITIVE for a in seq])),
            "negative_frac": float(np.mean([a in AA_NEGATIVE for a in seq])),
            "polar_frac": float(np.mean([a in AA_POLAR for a in seq])),
            "tiny_frac": float(np.mean([a in AA_TINY for a in seq])),
            "small_frac": float(np.mean([a in AA_SMALL for a in seq])),
            "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
            "molecular_weight_proxy": float(weights.sum()),
            "entropy": shannon_entropy(seq),
            "max_run_fraction": max_run_fraction(seq),
            "unique_residue_fraction": unique_residue_fraction(seq),
            "max_single_residue_fraction": max_single_residue_fraction(seq),
        }
    )
    return desc

def add_computable_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in df[seq_col].astype(str)])
    return pd.concat([df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)

# =============================================================================
# Input loading and preparation
# =============================================================================

def load_step_inputs(cfg: Step8Config) -> Dict[str, Any]:
    project_root = Path(cfg.project_root).resolve()

    step1_root = find_best_matching_directory(
        base_root=project_root,
        candidate_names=cfg.step1_dir_candidates,
        required_keywords=("step1", "step_1", "retained", "corpus", "clean"),
    )

    step7_root = find_best_matching_directory(
        base_root=project_root,
        candidate_names=cfg.step7_dir_candidates,
        required_keywords=("step7_v3", "step7", "step_7", "generated", "novelty", "candidate"),
    )

    step8_root = project_root / cfg.step8_dir_name

    step1_ref_path = find_best_matching_csv(
        root_dir=step1_root,
        aliases=cfg.step1_reference_aliases,
        required_keywords=("step1", "retained", "clean", "reference", "corpus", "dataset"),
        exclude_keywords=("step8", "step7", "backup", "tmp", "__pycache__"),
    )

    step7_generated_path = find_best_matching_csv(
        root_dir=step7_root,
        aliases=cfg.step7_generated_aliases,
        required_keywords=("step7", "generated", "candidate", "novelty", "audited", "ranked"),
        exclude_keywords=("step8", "backup", "tmp", "__pycache__"),
    )

    ref_preview = pd.read_csv(step1_ref_path, nrows=10)
    gen_preview = pd.read_csv(step7_generated_path, nrows=10)

    _ = first_existing_column(ref_preview, cfg.sequence_col_candidates, required=True, label="Step 1 sequence")
    _ = first_existing_column(gen_preview, cfg.sequence_col_candidates, required=True, label="Step 7 sequence")
    _ = first_existing_column(gen_preview, cfg.train_nn_similarity_col_candidates, required=True, label="Step 7 train NN similarity")
    _ = first_existing_column(gen_preview, cfg.requested_length_bin_col_candidates, required=True, label="Step 7 requested length bin")
    _ = first_existing_column(gen_preview, cfg.requested_charge_bin_col_candidates, required=True, label="Step 7 requested charge bin")
    _ = first_existing_column(gen_preview, cfg.requested_hydrophobicity_bin_col_candidates, required=True, label="Step 7 requested hydrophobicity bin")

    ref_df = pd.read_csv(step1_ref_path)
    gen_df = pd.read_csv(step7_generated_path)

    print(f"Resolved Step 1 directory: {step1_root}")
    print(f"Resolved Step 7 directory: {step7_root}")
    print(f"Resolved Step 1 reference: {step1_ref_path}")
    print(f"Resolved Step 7 generated input: {step7_generated_path}")

    return {
        "project_root": project_root,
        "step1_root": step1_root,
        "step7_root": step7_root,
        "step1_ref_path": step1_ref_path,
        "step7_generated_path": step7_generated_path,
        "step1_ref_df": ref_df,
        "step7_generated_df": gen_df,
        "step8_root": step8_root,
    }

def prepare_reference_df(ref_df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df = ref_df.copy()
    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="Step 1 sequence")
    colmap = {"sequence_col": seq_col}

    df[seq_col] = df[seq_col].astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].copy()
    df = df.drop_duplicates(subset=[seq_col]).reset_index(drop=True)
    df = add_computable_descriptors(df, seq_col)

    merge_key_col = first_existing_column(df, cfg.merge_key_col_candidates, required=False)
    if merge_key_col is None:
        merge_key_col = "sequence_sha256"
        df[merge_key_col] = df[seq_col].map(lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())
    colmap["merge_key_col"] = merge_key_col

    return df, colmap

def prepare_generated_df(gen_df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df = gen_df.copy()

    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="Step 7 sequence")
    id_col = first_existing_column(df, cfg.id_col_candidates, required=False)
    if id_col is None:
        id_col = "generated_id"
        df[id_col] = [f"gen_{i:06d}" for i in range(len(df))]

    train_nn_similarity_col = first_existing_column(
        df,
        cfg.train_nn_similarity_col_candidates,
        required=True,
        label="Step 7 train NN similarity",
    )

    exact_train_match_col, exact_train_was_derived = resolve_exact_train_match_column(
        df=df,
        exact_candidates=cfg.exact_train_match_col_candidates,
        nn_candidates=cfg.train_nn_similarity_col_candidates,
        derived_col_name="_derived_exact_train_match_from_nn",
        atol=1e-12,
    )

    exact_full_match_col = resolve_exact_full_match_column(
        df=df,
        exact_full_candidates=cfg.exact_full_match_col_candidates,
    )

    colmap = {
        "sequence_col": seq_col,
        "id_col": id_col,
        "exact_train_match_col": exact_train_match_col,
        "train_nn_similarity_col": train_nn_similarity_col,
        "exact_full_match_col": exact_full_match_col,
        "requested_length_bin_col": first_existing_column(df, cfg.requested_length_bin_col_candidates, required=True, label="Step 7 requested length bin"),
        "requested_charge_bin_col": first_existing_column(df, cfg.requested_charge_bin_col_candidates, required=True, label="Step 7 requested charge bin"),
        "requested_hydrophobicity_bin_col": first_existing_column(df, cfg.requested_hydrophobicity_bin_col_candidates, required=True, label="Step 7 requested hydrophobicity bin"),
        "requested_length_min_col": first_existing_column(df, cfg.requested_length_min_col_candidates, required=False),
        "requested_length_max_col": first_existing_column(df, cfg.requested_length_max_col_candidates, required=False),
        "requested_charge_min_col": first_existing_column(df, cfg.requested_charge_min_col_candidates, required=False),
        "requested_charge_max_col": first_existing_column(df, cfg.requested_charge_max_col_candidates, required=False),
        "requested_hydrophobicity_min_col": first_existing_column(df, cfg.requested_hydrophobicity_min_col_candidates, required=False),
        "requested_hydrophobicity_max_col": first_existing_column(df, cfg.requested_hydrophobicity_max_col_candidates, required=False),
        "exact_train_match_was_derived": exact_train_was_derived,
    }

    merge_key_col = first_existing_column(df, cfg.merge_key_col_candidates, required=False)
    if merge_key_col is None:
        merge_key_col = "sequence_sha256"
        df[merge_key_col] = df[seq_col].map(lambda s: hashlib.sha256(clean_sequence(s).encode("utf-8")).hexdigest())
    colmap["merge_key_col"] = merge_key_col

    df[seq_col] = df[seq_col].astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].copy()

    if df[id_col].duplicated().any():
        dup_n = int(df[id_col].duplicated().sum())
        raise ValueError(f"Generated candidate table has {dup_n} duplicated IDs.")

    df = add_computable_descriptors(df, seq_col)

    nn_vals = pd.to_numeric(df[colmap["train_nn_similarity_col"]], errors="coerce")
    if nn_vals.isna().any():
        raise ValueError(f"{colmap['train_nn_similarity_col']} contains missing/non-numeric values.")
    if ((nn_vals < 0) | (nn_vals > 1)).any():
        raise ValueError(f"{colmap['train_nn_similarity_col']} must lie in [0, 1].")

    exact_vals = pd.to_numeric(df[colmap["exact_train_match_col"]], errors="coerce")
    if exact_vals.isna().any():
        raise ValueError(f"{colmap['exact_train_match_col']} contains missing/non-numeric values.")

    for cname in [
        colmap["requested_length_bin_col"],
        colmap["requested_charge_bin_col"],
        colmap["requested_hydrophobicity_bin_col"],
    ]:
        if df[cname].isna().any():
            raise ValueError(f"{cname} contains missing values. Step 8 requires condition metadata.")

    return df.reset_index(drop=True), colmap

# =============================================================================
# Supportive predictor handling
# =============================================================================

def build_supportive_predictor_frame(df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.Series, pd.DataFrame]:
    support_col = first_existing_column(df, cfg.supportive_score_candidates, required=False)
    tox_col = first_existing_column(df, cfg.toxicity_score_candidates, required=False)
    hemo_col = first_existing_column(df, cfg.hemolysis_score_candidates, required=False)

    provenance_rows = [
        {"role": "supportive_acp_score", "column_used": support_col, "status": "used" if support_col else "not_available", "notes": "Supportive only; not direct proof."},
        {"role": "toxicity_penalty", "column_used": tox_col, "status": "used" if tox_col else "not_available", "notes": "Penalty only."},
        {"role": "hemolysis_penalty", "column_used": hemo_col, "status": "used" if hemo_col else "not_available", "notes": "Penalty only."},
    ]

    if support_col is None:
        supportive = pd.Series(np.full(len(df), 0.5), index=df.index, dtype=float)
    else:
        supportive = pd.to_numeric(df[support_col], errors="coerce")
        if supportive.isna().any():
            raise ValueError(f"Supportive predictor column {support_col} contains missing/non-numeric values.")
        supportive = supportive.clip(0.0, 1.0)

    tox_pen = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    if tox_col is not None:
        tox_pen = pd.to_numeric(df[tox_col], errors="coerce")
        if tox_pen.isna().any():
            raise ValueError(f"Toxicity column {tox_col} contains missing/non-numeric values.")
        tox_pen = tox_pen.clip(0.0, 1.0)

    hemo_pen = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    if hemo_col is not None:
        hemo_pen = pd.to_numeric(df[hemo_col], errors="coerce")
        if hemo_pen.isna().any():
            raise ValueError(f"Hemolysis column {hemo_col} contains missing/non-numeric values.")
        hemo_pen = hemo_pen.clip(0.0, 1.0)

    supportive_score = (supportive - 0.5 * tox_pen - 0.5 * hemo_pen).clip(0.0, 1.0)
    return supportive_score, pd.DataFrame(provenance_rows)

# =============================================================================
# Hard filtering
# =============================================================================

def build_hard_filter_flags(df: pd.DataFrame, cfg: Step8Config, colmap: Dict[str, str]) -> pd.DataFrame:
    seq_col = colmap["sequence_col"]
    exact_train_col = colmap["exact_train_match_col"]
    train_nn_col = colmap["train_nn_similarity_col"]
    exact_full_col = colmap["exact_full_match_col"]

    out = pd.DataFrame(index=df.index)
    out["flag_invalid_sequence"] = (~df[seq_col].map(lambda s: is_valid_peptide(s, cfg.min_length, cfg.max_length))).astype(int)
    out["flag_too_short"] = (df["length"] < cfg.min_length).astype(int)
    out["flag_too_long"] = (df["length"] > cfg.max_length).astype(int)
    out["flag_low_entropy"] = (df["entropy"] < cfg.low_complexity_entropy_threshold).astype(int)
    out["flag_high_run_fraction"] = (df["max_run_fraction"] > cfg.max_homopolymer_run_fraction).astype(int)
    out["flag_high_single_residue_fraction"] = (df["max_single_residue_fraction"] > cfg.max_single_residue_fraction).astype(int)
    out["flag_low_unique_residue_fraction"] = (df["unique_residue_fraction"] < cfg.min_unique_residue_fraction).astype(int)
    out["flag_duplicate_sequence"] = df.duplicated(subset=[seq_col], keep="first").astype(int)
    out["flag_exact_train_match"] = pd.to_numeric(df[exact_train_col], errors="coerce").astype(int)

    if exact_full_col is not None and cfg.require_not_in_full_exact_if_available:
        out["flag_exact_full_match"] = pd.to_numeric(df[exact_full_col], errors="coerce").astype(int)
    else:
        out["flag_exact_full_match"] = 0

    out["flag_high_train_similarity"] = (
        pd.to_numeric(df[train_nn_col], errors="coerce") > cfg.max_train_nn_similarity
    ).astype(int)

    active_flags = [
        "flag_invalid_sequence",
        "flag_too_short",
        "flag_too_long",
        "flag_low_entropy",
        "flag_high_run_fraction",
        "flag_high_single_residue_fraction",
        "flag_low_unique_residue_fraction",
        "flag_duplicate_sequence",
    ]
    if cfg.require_not_in_train_exact:
        active_flags.append("flag_exact_train_match")
    active_flags.append("flag_high_train_similarity")
    if cfg.require_not_in_full_exact_if_available and exact_full_col is not None:
        active_flags.append("flag_exact_full_match")

    out["hard_filter_fail"] = (out[active_flags].sum(axis=1) > 0).astype(int)
    return out

def summarize_filter_waterfall(flags_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(flags_df)
    flag_cols = [c for c in flags_df.columns if c.startswith("flag_")]
    for c in flag_cols:
        removed = int(flags_df[c].sum())
        rows.append({
            "filter_name": c,
            "removed_n": removed,
            "removed_fraction": removed / n if n else np.nan,
            "retained_if_applied_alone": n - removed,
        })
    return pd.DataFrame(rows).sort_values("removed_n", ascending=False).reset_index(drop=True)

# =============================================================================
# ACP plausibility support
# =============================================================================

def build_reference_bounds(ref_df: pd.DataFrame, cfg: Step8Config) -> pd.DataFrame:
    rows = []
    for metric, weight in cfg.descriptor_weight_map.items():
        vals = pd.to_numeric(ref_df[metric], errors="coerce").dropna()
        if len(vals) < 20:
            raise ValueError(f"Reference corpus has insufficient support for descriptor '{metric}'.")
        rows.append({
            "descriptor": metric,
            "q_low": float(vals.quantile(cfg.reference_quantile_low)),
            "q_high": float(vals.quantile(cfg.reference_quantile_high)),
            "median": float(vals.median()),
            "iqr": float(vals.quantile(0.75) - vals.quantile(0.25)),
            "weight": weight,
        })
    return pd.DataFrame(rows)

def descriptor_support_score(series: pd.Series, q_low: float, q_high: float) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce").astype(float)
    center = 0.5 * (q_low + q_high)
    half = max(0.5 * (q_high - q_low), 1e-8)
    z = np.abs(x - center) / half
    return (1.0 - np.clip(z, 0.0, 1.0)).fillna(0.0)

def compute_plausibility_scores(df: pd.DataFrame, reference_bounds_df: pd.DataFrame) -> Tuple[pd.Series, pd.DataFrame]:
    comp = pd.DataFrame(index=df.index)
    fail_cols = []

    weighted_num = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    weighted_den = 0.0

    for _, row in reference_bounds_df.iterrows():
        metric = row["descriptor"]
        q_low = float(row["q_low"])
        q_high = float(row["q_high"])
        weight = float(row["weight"])

        score_col = f"support_{metric}"
        fail_col = f"flag_outside_reference_{metric}"
        comp[score_col] = descriptor_support_score(df[metric], q_low, q_high)
        comp[fail_col] = ((df[metric] < q_low) | (df[metric] > q_high)).astype(int)
        fail_cols.append(fail_col)

        weighted_num += comp[score_col] * weight
        weighted_den += weight

    comp["plausibility_fail_count"] = comp[fail_cols].sum(axis=1)
    comp["descriptor_plausibility_score"] = weighted_num / weighted_den
    comp["descriptor_plausibility_pass"] = (comp["plausibility_fail_count"] <= 2).astype(int)

    return comp["descriptor_plausibility_score"].clip(0.0, 1.0), comp

# =============================================================================
# Condition fidelity
# =============================================================================

def infer_length_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 10:
        return "short"
    if x <= 20:
        return "medium"
    return "long"

def infer_charge_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 2:
        return "low"
    if x <= 6:
        return "medium"
    return "high"

def infer_hydropathy_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x < -1.0:
        return "low"
    if x <= 1.0:
        return "medium"
    return "high"

def compute_condition_fidelity(df: pd.DataFrame, colmap: Dict[str, str]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["length_bin_match"] = (
        df[colmap["requested_length_bin_col"]].astype(str).str.lower().str.strip()
        == df["length"].map(infer_length_bin).astype(str).str.lower().str.strip()
    ).astype(int)

    out["charge_bin_match"] = (
        df[colmap["requested_charge_bin_col"]].astype(str).str.lower().str.strip()
        == df["net_charge_proxy"].map(infer_charge_bin).astype(str).str.lower().str.strip()
    ).astype(int)

    out["hydropathy_bin_match"] = (
        df[colmap["requested_hydrophobicity_bin_col"]].astype(str).str.lower().str.strip()
        == df["mean_hydropathy"].map(infer_hydropathy_bin).astype(str).str.lower().str.strip()
    ).astype(int)

    if colmap.get("requested_length_min_col") and colmap.get("requested_length_max_col"):
        out["length_range_match"] = (
            (df["length"] >= pd.to_numeric(df[colmap["requested_length_min_col"]], errors="coerce")) &
            (df["length"] <= pd.to_numeric(df[colmap["requested_length_max_col"]], errors="coerce"))
        ).astype(int)

    if colmap.get("requested_charge_min_col") and colmap.get("requested_charge_max_col"):
        out["charge_range_match"] = (
            (df["net_charge_proxy"] >= pd.to_numeric(df[colmap["requested_charge_min_col"]], errors="coerce")) &
            (df["net_charge_proxy"] <= pd.to_numeric(df[colmap["requested_charge_max_col"]], errors="coerce"))
        ).astype(int)

    if colmap.get("requested_hydrophobicity_min_col") and colmap.get("requested_hydrophobicity_max_col"):
        out["hydropathy_range_match"] = (
            (df["mean_hydropathy"] >= pd.to_numeric(df[colmap["requested_hydrophobicity_min_col"]], errors="coerce")) &
            (df["mean_hydropathy"] <= pd.to_numeric(df[colmap["requested_hydrophobicity_max_col"]], errors="coerce"))
        ).astype(int)

    pass_cols = [c for c in out.columns if c.endswith("_match")]
    if len(pass_cols) == 0:
        raise ValueError("Condition fidelity columns could not be computed.")

    out["condition_fidelity_score"] = out[pass_cols].mean(axis=1)
    out["condition_fidelity_pass"] = (out["condition_fidelity_score"] >= 0.67).astype(int)
    return out

# =============================================================================
# Novelty and diversity
# =============================================================================

def compute_novelty_score(df: pd.DataFrame, colmap: Dict[str, str]) -> pd.Series:
    exact_train_novel = 1.0 - pd.to_numeric(df[colmap["exact_train_match_col"]], errors="coerce").clip(0, 1)
    nn_novel = 1.0 - pd.to_numeric(df[colmap["train_nn_similarity_col"]], errors="coerce").clip(0, 1)

    components = [exact_train_novel, nn_novel]
    if colmap.get("exact_full_match_col") is not None:
        full_novel = 1.0 - pd.to_numeric(df[colmap["exact_full_match_col"]], errors="coerce").clip(0, 1)
        components.append(full_novel)

    return pd.concat(components, axis=1).mean(axis=1).clip(0.0, 1.0)

def compute_pairwise_similarity_edges(seqs: Sequence[str], threshold: float, k: int) -> List[Tuple[int, int, float]]:
    edges = []
    for i in range(len(seqs)):
        for j in range(i + 1, len(seqs)):
            sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=k)
            if sim >= threshold:
                edges.append((i, j, sim))
    return edges

def connected_components_from_edges(n_nodes: int, edges: Sequence[Tuple[int, int, float]]) -> List[List[int]]:
    adj = [[] for _ in range(n_nodes)]
    for i, j, _ in edges:
        adj[i].append(j)
        adj[j].append(i)

    seen = np.zeros(n_nodes, dtype=bool)
    comps = []
    for start in range(n_nodes):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
        comps.append(sorted(comp))
    return comps

def compute_diversity_scores(seqs: Sequence[str], k: int) -> pd.Series:
    n = len(seqs)
    if n <= 1:
        return pd.Series(np.ones(n))
    nn_div = []
    for i in range(n):
        sims = []
        for j in range(n):
            if i == j:
                continue
            sims.append(jaccard_kmer_similarity(seqs[i], seqs[j], k=k))
        nearest = max(sims) if sims else 0.0
        nn_div.append(1.0 - nearest)
    return pd.Series(nn_div).clip(0.0, 1.0)

def shortlist_with_cluster_cap(
    df: pd.DataFrame,
    seq_col: str,
    score_col: str,
    threshold: float,
    max_per_cluster: int,
    top_n: int,
    k: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    seqs = df[seq_col].tolist()
    edges = compute_pairwise_similarity_edges(seqs, threshold=threshold, k=k)
    comps = connected_components_from_edges(len(seqs), edges)

    cluster_id = np.full(len(df), -1, dtype=int)
    for cid, comp in enumerate(comps):
        for idx in comp:
            cluster_id[idx] = cid

    out = df.copy()
    out["similarity_cluster_id"] = cluster_id

    chosen = []
    cluster_counts: Dict[int, int] = defaultdict(int)
    ranked_idx = out.sort_values(score_col, ascending=False).index.tolist()
    for idx in ranked_idx:
        cid = int(out.loc[idx, "similarity_cluster_id"])
        if cluster_counts[cid] < max_per_cluster:
            chosen.append(idx)
            cluster_counts[cid] += 1
        if len(chosen) >= top_n:
            break

    chosen_df = out.loc[chosen].copy()
    chosen_df = chosen_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    return chosen_df, out

# =============================================================================
# Ranking and stability
# =============================================================================

def weighted_score(df: pd.DataFrame, weights: Dict[str, float]) -> pd.Series:
    missing = [c for c in weights if c not in df.columns]
    if missing:
        raise ValueError(f"Ranking requested missing columns: {missing}")
    total_w = sum(weights.values())
    s = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    for c, w in weights.items():
        s += df[c] * w
    return s / total_w

def dense_rank_desc(s: pd.Series) -> pd.Series:
    return s.rank(method="dense", ascending=False).astype(int)

def compute_scheme_stability(results_by_scheme: Dict[str, pd.DataFrame], id_col: str, top_n: int) -> pd.DataFrame:
    rows = []
    scheme_names = list(results_by_scheme.keys())
    for a, b in combinations(scheme_names, 2):
        top_a = set(results_by_scheme[a].head(top_n)[id_col].astype(str))
        top_b = set(results_by_scheme[b].head(top_n)[id_col].astype(str))
        inter = len(top_a & top_b)
        union = len(top_a | top_b)
        jacc = inter / union if union else np.nan
        overlap_frac = inter / top_n if top_n else np.nan
        rows.append({
            "scheme_a": a,
            "scheme_b": b,
            "top_n": top_n,
            "intersection_n": inter,
            "union_n": union,
            "jaccard_overlap": jacc,
            "overlap_fraction_of_top_n": overlap_frac,
        })
    return pd.DataFrame(rows)

def bootstrap_shortlist_membership_stability(df: pd.DataFrame, id_col: str, weights: Dict[str, float], top_n: int, n_boot: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for b in range(n_boot):
        sampled = df.sample(n=len(df), replace=True, random_state=int(rng.integers(0, 1_000_000)))
        sampled_score = weighted_score(sampled, weights)
        sampled = sampled.assign(_boot_score=sampled_score).sort_values("_boot_score", ascending=False).head(top_n)
        for pid in sampled[id_col].astype(str):
            rows.append({"bootstrap_iter": b, id_col: pid, "in_top_n": 1})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=[id_col, "bootstrap_top_n_frequency"])
    return out.groupby(id_col)["in_top_n"].mean().rename("bootstrap_top_n_frequency").reset_index()

# =============================================================================
# Main pipeline
# =============================================================================

def run_step8(cfg: Step8Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Loading Step 1 / Step 7 inputs")
    loaded = load_step_inputs(cfg)

    save_json(
        {
            "config": asdict(cfg),
            "step1_reference_path": str(loaded["step1_ref_path"]),
            "step7_generated_path": str(loaded["step7_generated_path"]),
            "input_hashes": {
                "step1_reference_sha256": sha256_of_file(loaded["step1_ref_path"]),
                "step7_generated_sha256": sha256_of_file(loaded["step7_generated_path"]),
            },
            "environment_manifest": collect_environment_manifest(),
        },
        dirs["artifacts"] / "step8_run_manifest.json",
    )

    ref_df, _ = prepare_reference_df(loaded["step1_ref_df"], cfg)
    gen_df, gen_colmap = prepare_generated_df(loaded["step7_generated_df"], cfg)

    print(f"Loaded Step 1 reference rows: {len(ref_df):,}")
    print(f"Loaded Step 7 generated rows: {len(gen_df):,}")

    print_header("Applying hard eligibility filters")
    flags_df = build_hard_filter_flags(gen_df, cfg, gen_colmap)
    filter_audit_df = summarize_filter_waterfall(flags_df)

    gen_scored = pd.concat([gen_df.reset_index(drop=True), flags_df.reset_index(drop=True)], axis=1)
    gen_scored["eligible_after_hard_filters"] = 1 - gen_scored["hard_filter_fail"]

    filter_audit_df.to_csv(dirs["tables_supplementary"] / "table_s8_1_filter_audit.csv", index=False)
    gen_scored.to_csv(dirs["tables_supplementary"] / "table_s8_2_candidate_pool_with_flags.csv", index=False)

    eligible = gen_scored[gen_scored["eligible_after_hard_filters"] == 1].copy().reset_index(drop=True)
    if len(eligible) == 0:
        raise ValueError("No candidates remain after hard eligibility filtering.")

    print(f"Eligible after hard filters: {len(eligible):,} / {len(gen_scored):,}")

    print_header("Computing ACP-oriented plausibility support")
    reference_bounds_df = build_reference_bounds(ref_df, cfg)
    plaus_score, plaus_detail = compute_plausibility_scores(eligible, reference_bounds_df)
    eligible = pd.concat([eligible.reset_index(drop=True), plaus_detail.reset_index(drop=True)], axis=1)
    eligible["descriptor_plausibility_score"] = plaus_score
    reference_bounds_df.to_csv(dirs["tables_supplementary"] / "table_s8_3_reference_descriptor_bounds.csv", index=False)

    print_header("Computing condition fidelity")
    condition_detail = compute_condition_fidelity(eligible, gen_colmap)
    eligible = pd.concat([eligible.reset_index(drop=True), condition_detail.reset_index(drop=True)], axis=1)

    print_header("Computing novelty and supportive scores")
    eligible["novelty_score"] = compute_novelty_score(eligible, gen_colmap)
    eligible["diversity_score"] = compute_diversity_scores(eligible[gen_colmap["sequence_col"]].tolist(), k=cfg.similarity_kmer).values
    eligible["supportive_model_score"], supportive_provenance_df = build_supportive_predictor_frame(eligible, cfg)
    supportive_provenance_df.to_csv(dirs["tables_supplementary"] / "table_s8_4_auxiliary_predictor_provenance.csv", index=False)

    eligible["final_pass"] = (
        (eligible["descriptor_plausibility_pass"] == 1) &
        (eligible["condition_fidelity_pass"] == 1)
    ).astype(int)

    passed = eligible[eligible["final_pass"] == 1].copy().reset_index(drop=True)
    if len(passed) == 0:
        raise ValueError("No candidates remain after plausibility and condition-fidelity screening.")

    print(f"Passed plausibility + condition fidelity: {len(passed):,}")

    print_header("Ranking candidates")
    score_cols = list(cfg.ranking_weights.keys())
    for c in score_cols:
        passed[c] = pd.to_numeric(passed[c], errors="coerce").clip(0.0, 1.0)
        if passed[c].isna().any():
            raise ValueError(f"Ranking component {c} contains missing values.")

    passed["final_score"] = weighted_score(passed, cfg.ranking_weights)
    passed["final_rank"] = dense_rank_desc(passed["final_score"])

    print_header("Building diversity-aware shortlist")
    seq_col = gen_colmap["sequence_col"]
    id_col = gen_colmap["id_col"]

    shortlist_df, annotated_ranked_df = shortlist_with_cluster_cap(
        df=passed.sort_values("final_score", ascending=False).reset_index(drop=True),
        seq_col=seq_col,
        score_col="final_score",
        threshold=cfg.similarity_threshold_for_clusters,
        max_per_cluster=cfg.max_per_similarity_cluster,
        top_n=cfg.shortlist_top_n,
        k=cfg.similarity_kmer,
    )
    if len(shortlist_df) == 0:
        raise ValueError("No candidates remain after shortlist diversity control.")
    shortlist_df["shortlist_rank"] = dense_rank_desc(shortlist_df["final_score"])

    final_panel_df, _ = shortlist_with_cluster_cap(
        df=shortlist_df.sort_values("final_score", ascending=False).reset_index(drop=True),
        seq_col=seq_col,
        score_col="final_score",
        threshold=cfg.similarity_threshold_for_clusters,
        max_per_cluster=1,
        top_n=cfg.final_diverse_panel_n,
        k=cfg.similarity_kmer,
    )
    if len(final_panel_df) == 0:
        raise ValueError("No candidates remain after final diverse panel selection.")
    final_panel_df["final_panel_rank"] = dense_rank_desc(final_panel_df["final_score"])

    print(f"Top shortlist size: {len(shortlist_df):,}")
    print(f"Final diverse panel size: {len(final_panel_df):,}")

    print_header("Running ranking sensitivity analysis")
    results_by_scheme: Dict[str, pd.DataFrame] = {}
    for scheme_name, scheme_weights in cfg.ranking_schemes.items():
        tmp = passed.copy()
        tmp["final_score"] = weighted_score(tmp, scheme_weights)
        tmp["final_rank"] = dense_rank_desc(tmp["final_score"])
        tmp = tmp.sort_values("final_score", ascending=False).reset_index(drop=True)
        results_by_scheme[scheme_name] = tmp

    stability_df = compute_scheme_stability(results_by_scheme, id_col, cfg.shortlist_top_n)

    top_sets = {
        name: set(df.head(cfg.shortlist_top_n)[id_col].astype(str))
        for name, df in results_by_scheme.items()
    }
    recurrence_rows = []
    union_ids = sorted(set().union(*top_sets.values()))
    for pid in union_ids:
        hit_map = {f"in_top_{name}": int(pid in idset) for name, idset in top_sets.items()}
        recurrence_rows.append({
            id_col: pid,
            "scheme_recurrence_n": sum(hit_map.values()),
            **hit_map
        })
    recurrence_df = pd.DataFrame(recurrence_rows)

    bootstrap_freq_df = bootstrap_shortlist_membership_stability(
        passed[[id_col] + score_cols].copy(),
        id_col=id_col,
        weights=cfg.ranking_weights,
        top_n=cfg.shortlist_top_n,
        n_boot=cfg.bootstrap_iterations,
        seed=cfg.main_random_seed,
    )
    if len(bootstrap_freq_df) > 0:
        recurrence_df[id_col] = recurrence_df[id_col].astype(str)
        bootstrap_freq_df[id_col] = bootstrap_freq_df[id_col].astype(str)
        recurrence_df = recurrence_df.merge(bootstrap_freq_df, on=id_col, how="left")

    stability_df.to_csv(dirs["tables_supplementary"] / "table_s8_5_ranking_scheme_stability.csv", index=False)
    recurrence_df.to_csv(dirs["tables_supplementary"] / "table_s8_6_candidate_recurrence_across_schemes.csv", index=False)

    print_header("Building descriptive enrichment summaries")
    compare_rows = []
    groups = {
        "raw_step7_pool": gen_df,
        "hard_filter_eligible": eligible,
        "screen_passed": passed,
        "shortlist": shortlist_df,
        "final_diverse_panel": final_panel_df,
        "reference_acp_corpus": ref_df,
    }
    compare_metrics = list(cfg.descriptor_weight_map.keys()) + ["novelty_score", "condition_fidelity_score", "supportive_model_score"]
    for group_name, dfi in groups.items():
        for metric in compare_metrics:
            if metric in dfi.columns:
                vals = pd.to_numeric(dfi[metric], errors="coerce").dropna()
                compare_rows.append({
                    "group": group_name,
                    "metric": metric,
                    "n": len(vals),
                    "mean": float(vals.mean()) if len(vals) else np.nan,
                    "median": float(vals.median()) if len(vals) else np.nan,
                    "q25": float(vals.quantile(0.25)) if len(vals) else np.nan,
                    "q75": float(vals.quantile(0.75)) if len(vals) else np.nan,
                })
    descriptor_compare_df = pd.DataFrame(compare_rows)
    descriptor_compare_df.to_csv(dirs["tables_supplementary"] / "table_s8_7_descriptor_group_comparison.csv", index=False)

    shortlist_main_cols = [
        id_col, seq_col, "shortlist_rank", "final_score", "final_rank",
        "novelty_score", "descriptor_plausibility_score", "condition_fidelity_score",
        "diversity_score", "supportive_model_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy", "similarity_cluster_id"
    ]
    shortlist_main_cols = [c for c in shortlist_main_cols if c in shortlist_df.columns]
    shortlist_table = shortlist_df[shortlist_main_cols].sort_values("shortlist_rank").reset_index(drop=True)

    final_panel_cols = [
        id_col, seq_col, "final_panel_rank", "final_score",
        "novelty_score", "descriptor_plausibility_score", "condition_fidelity_score",
        "diversity_score", "supportive_model_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy", "similarity_cluster_id"
    ]
    final_panel_cols = [c for c in final_panel_cols if c in final_panel_df.columns]
    final_panel_table = final_panel_df[final_panel_cols].sort_values("final_panel_rank").reset_index(drop=True)

    ranked_pool_cols = [
        id_col, seq_col, "final_rank", "final_score",
        "novelty_score", "descriptor_plausibility_score", "condition_fidelity_score",
        "diversity_score", "supportive_model_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy",
        "descriptor_plausibility_pass", "condition_fidelity_pass", "similarity_cluster_id"
    ]
    ranked_pool_cols = [c for c in ranked_pool_cols if c in annotated_ranked_df.columns]

    shortlist_table.to_csv(dirs["tables_main"] / "table_8_1_shortlist_main.csv", index=False)
    final_panel_table.to_csv(dirs["tables_main"] / "table_8_2_final_diverse_panel.csv", index=False)
    annotated_ranked_df[ranked_pool_cols].sort_values("final_score", ascending=False).to_csv(
        dirs["tables_supplementary"] / "table_s8_8_full_ranked_passed_pool.csv", index=False
    )

    summary_lines = []
    summary_lines.append("PepCVAE Step 8 prioritization summary")
    summary_lines.append("=" * 80)
    summary_lines.append(f"Timestamp UTC: {now_utc_iso()}")
    summary_lines.append(f"Step 1 reference: {loaded['step1_ref_path']}")
    summary_lines.append(f"Step 7 generated input: {loaded['step7_generated_path']}")
    summary_lines.append("")
    summary_lines.append("Resolved column mapping")
    for k, v in gen_colmap.items():
        summary_lines.append(f"- {k}: {v}")
    summary_lines.append("")
    summary_lines.append("Novelty audit note")
    if gen_colmap.get("exact_train_match_was_derived", False):
        summary_lines.append("- exact_train_match_col was not present in Step 7 and was derived from train_nn_similarity == 1.0")
    else:
        summary_lines.append("- exact_train_match_col was loaded directly from Step 7")
    summary_lines.append("")
    summary_lines.append("Counts")
    summary_lines.append(f"- Raw Step 7 candidates: {len(gen_df):,}")
    summary_lines.append(f"- Eligible after hard filters: {len(eligible):,}")
    summary_lines.append(f"- Passed plausibility + condition screening: {len(passed):,}")
    summary_lines.append(f"- Top shortlist: {len(shortlist_df):,}")
    summary_lines.append(f"- Final diverse panel: {len(final_panel_df):,}")
    summary_lines.append("")
    summary_lines.append("Primary ranking weights")
    for k, v in cfg.ranking_weights.items():
        summary_lines.append(f"  - {k}: {v}")
    summary_lines.append("")
    summary_lines.append("Supportive predictor provenance")
    for _, r in supportive_provenance_df.iterrows():
        summary_lines.append(f"  - {r['role']}: {r['status']} ({r['column_used']})")

    save_text("\n".join(summary_lines), dirs["reports"] / "step8_summary_report.txt")

    return {
        "ref_df": ref_df,
        "gen_df": gen_df,
        "eligible_df": eligible,
        "passed_df": passed,
        "shortlist_df": shortlist_df,
        "final_panel_df": final_panel_df,
        "shortlist_table": shortlist_table,
        "final_panel_table": final_panel_table,
        "filter_audit_df": filter_audit_df,
        "reference_bounds_df": reference_bounds_df,
        "stability_df": stability_df,
        "recurrence_df": recurrence_df,
        "descriptor_compare_df": descriptor_compare_df,
        "supportive_provenance_df": supportive_provenance_df,
        "annotated_ranked_df": annotated_ranked_df,
        "gen_colmap": gen_colmap,
    }

# =============================================================================
# Figure helpers
# =============================================================================

def style_axis(ax):
    ax.set_facecolor(PROJECT_COLORS["bg"])
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PROJECT_COLORS["spine"])
        ax.spines[side].set_linewidth(0.8)
    ax.tick_params(colors=PROJECT_COLORS["text"])
    ax.xaxis.label.set_color(PROJECT_COLORS["text"])
    ax.yaxis.label.set_color(PROJECT_COLORS["text"])
    ax.title.set_color(PROJECT_COLORS["text"])
    ax.grid(True, color=PROJECT_COLORS["grid"], linewidth=0.6, alpha=1.0)

# =============================================================================
# Figures
# =============================================================================

def plot_step8_main_figure(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    gen_df = results["gen_df"]
    eligible_df = results["eligible_df"]
    id_col = results["gen_colmap"]["id_col"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 8.6))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.1], width_ratios=[1.0, 1.0])

    ax = fig.add_subplot(gs[0, 0])
    style_axis(ax)
    funnel_labels = ["Step 7 pool", "Hard-filter eligible", "Screen passed", "Top shortlist", "Final panel"]
    funnel_vals = [len(gen_df), len(eligible_df), len(passed_df), len(shortlist_df), len(final_panel_df)]
    bar_colors = [
        PROJECT_COLORS["deep_crimson"],
        PROJECT_COLORS["warm_gold"],
        PROJECT_COLORS["teal"],
        PROJECT_COLORS["navy"],
        PROJECT_COLORS["deep_red_maroon"],
    ]
    ax.bar(range(len(funnel_vals)), funnel_vals, color=bar_colors, edgecolor=PROJECT_COLORS["spine"], linewidth=0.6)
    ax.set_xticks(range(len(funnel_vals)))
    ax.set_xticklabels(funnel_labels, rotation=22, ha="right")
    ax.set_ylabel("Number of peptides")
    ax.set_title("a  Step 8 prioritization funnel", loc="left", fontsize=cfg.figure_export.panel_fontsize)
    for i, v in enumerate(funnel_vals):
        frac = 100.0 * v / funnel_vals[0] if funnel_vals[0] else 0.0
        ax.text(i, v, f"{v}\n({frac:.1f}%)", ha="center", va="bottom", fontsize=7, color=PROJECT_COLORS["text"])

    ax = fig.add_subplot(gs[0, 1])
    style_axis(ax)
    plot_df = passed_df.copy()
    plot_df["is_shortlist"] = plot_df[id_col].astype(str).isin(shortlist_df[id_col].astype(str)).astype(int)
    ax.scatter(
        plot_df.loc[plot_df["is_shortlist"] == 0, "novelty_score"],
        plot_df.loc[plot_df["is_shortlist"] == 0, "descriptor_plausibility_score"],
        s=20, alpha=0.35, label="Passed pool", color=PROJECT_COLORS["neutral_gray"],
        edgecolors="none"
    )
    ax.scatter(
        plot_df.loc[plot_df["is_shortlist"] == 1, "novelty_score"],
        plot_df.loc[plot_df["is_shortlist"] == 1, "descriptor_plausibility_score"],
        s=38, alpha=0.90, label="Shortlist", color=PROJECT_COLORS["deep_crimson"],
        edgecolors="white", linewidths=0.3
    )
    ax.set_xlabel("Novelty score")
    ax.set_ylabel("ACP-plausibility score")
    ax.set_title("b  Novelty–plausibility landscape", loc="left", fontsize=cfg.figure_export.panel_fontsize)
    ax.legend(frameon=False, fontsize=cfg.figure_export.legend_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    ax.set_facecolor(PROJECT_COLORS["bg"])
    heat_cols = [
        "novelty_score", "descriptor_plausibility_score",
        "condition_fidelity_score", "diversity_score",
        "supportive_model_score", "final_score"
    ]
    heat_cols = [c for c in heat_cols if c in shortlist_df.columns]
    heat_df = shortlist_df.sort_values("shortlist_rank").head(min(12, len(shortlist_df)))[[id_col] + heat_cols].copy()
    mat = heat_df[heat_cols].values
    im = ax.imshow(mat, aspect="auto", vmin=0, vmax=1, cmap="Reds")
    ax.set_xticks(range(len(heat_cols)))
    ax.set_xticklabels(heat_cols, rotation=30, ha="right")
    ax.set_yticks(range(len(heat_df)))
    ax.set_yticklabels(heat_df[id_col].astype(str).tolist())
    ax.set_title("c  Top candidates across ranking axes", loc="left", fontsize=cfg.figure_export.panel_fontsize, color=PROJECT_COLORS["text"])
    for spine in ax.spines.values():
        spine.set_color(PROJECT_COLORS["spine"])
        spine.set_linewidth(0.8)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Scaled score", color=PROJECT_COLORS["text"])
    cbar.outline.set_edgecolor(PROJECT_COLORS["spine"])
    cbar.ax.yaxis.set_tick_params(color=PROJECT_COLORS["text"])
    plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color=PROJECT_COLORS["text"])

    ax = fig.add_subplot(gs[1, 1])
    style_axis(ax)
    ax.scatter(
        final_panel_df["condition_fidelity_score"],
        final_panel_df["diversity_score"],
        s=55, alpha=0.9, color=PROJECT_COLORS["navy"],
        edgecolors="white", linewidths=0.4
    )
    for _, r in final_panel_df.iterrows():
        ax.text(r["condition_fidelity_score"], r["diversity_score"], str(r[id_col]), fontsize=6, color=PROJECT_COLORS["text"])
    ax.set_xlabel("Condition fidelity score")
    ax.set_ylabel("Diversity score")
    ax.set_title("d  Final diverse panel", loc="left", fontsize=cfg.figure_export.panel_fontsize)

    fig.tight_layout()
    export_figure(fig, dirs["figures_main"] / "figure_8_prioritization_main", cfg.figure_export)

def plot_step8_supplementary(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    ref_df = results["ref_df"]
    stability_df = results["stability_df"]
    id_col = results["gen_colmap"]["id_col"]
    seq_col = results["gen_colmap"]["sequence_col"]
    cond_cols = [
        results["gen_colmap"]["requested_length_bin_col"],
        results["gen_colmap"]["requested_charge_bin_col"],
        results["gen_colmap"]["requested_hydrophobicity_bin_col"],
    ]

    compare_cols = ["length", "net_charge_proxy", "mean_hydropathy", "entropy"]
    hist_colors = [
        PROJECT_COLORS["teal"],
        PROJECT_COLORS["deep_crimson"],
        PROJECT_COLORS["warm_gold"],
    ]
    for col in compare_cols:
        fig, ax = plt.subplots(figsize=(6.2, 4.5))
        style_axis(ax)
        ax.hist(passed_df[col].dropna(), bins=25, alpha=0.45, density=True, label="Passed pool", color=hist_colors[0])
        ax.hist(shortlist_df[col].dropna(), bins=25, alpha=0.55, density=True, label="Shortlist", color=hist_colors[1])
        ax.hist(ref_df[col].dropna(), bins=25, alpha=0.35, density=True, label="Reference ACP corpus", color=hist_colors[2])
        ax.set_xlabel(col)
        ax.set_ylabel("Density")
        ax.set_title(f"Descriptor comparison: {col}")
        ax.legend(frameon=False, fontsize=7)
        fig.tight_layout()
        export_figure(fig, dirs["figures_supplementary"] / f"sfigure_8_1_descriptor_{col}", cfg.figure_export)

    if len(stability_df) > 0:
        fig, ax = plt.subplots(figsize=(7.0, 4.5))
        style_axis(ax)
        labels = stability_df["scheme_a"] + " vs " + stability_df["scheme_b"]
        vals = stability_df["jaccard_overlap"]
        ax.bar(range(len(vals)), vals, color=PROJECT_COLORS["navy"], edgecolor=PROJECT_COLORS["spine"], linewidth=0.6)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_ylabel("Top-N Jaccard overlap")
        ax.set_title("Shortlist stability across ranking schemes")
        fig.tight_layout()
        export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_2_scheme_stability", cfg.figure_export)

    if len(final_panel_df) >= 2:
        seqs = final_panel_df[seq_col].tolist()
        sim_mat = np.eye(len(seqs))
        for i in range(len(seqs)):
            for j in range(i + 1, len(seqs)):
                sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=cfg.similarity_kmer)
                sim_mat[i, j] = sim
                sim_mat[j, i] = sim
        fig, ax = plt.subplots(figsize=(6.2, 5.2))
        ax.set_facecolor(PROJECT_COLORS["bg"])
        im = ax.imshow(sim_mat, aspect="auto", vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(final_panel_df)))
        ax.set_yticks(range(len(final_panel_df)))
        ax.set_xticklabels(final_panel_df[id_col].astype(str), rotation=45, ha="right")
        ax.set_yticklabels(final_panel_df[id_col].astype(str))
        ax.set_title("Pairwise 3-mer Jaccard similarity in final panel", color=PROJECT_COLORS["text"])
        for spine in ax.spines.values():
            spine.set_color(PROJECT_COLORS["spine"])
            spine.set_linewidth(0.8)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Similarity", color=PROJECT_COLORS["text"])
        cbar.outline.set_edgecolor(PROJECT_COLORS["spine"])
        fig.tight_layout()
        export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_3_final_panel_similarity", cfg.figure_export)

    cond_bar_colors = [PROJECT_COLORS["teal"], PROJECT_COLORS["deep_red_maroon"]]
    for cond_col in cond_cols:
        if cond_col in passed_df.columns:
            prior_counts = passed_df[cond_col].astype(str).value_counts(dropna=False)
            short_counts = shortlist_df[cond_col].astype(str).value_counts(dropna=False)
            merged = pd.DataFrame({"passed_pool": prior_counts, "shortlist": short_counts}).fillna(0)

            fig, ax = plt.subplots(figsize=(7.0, 4.5))
            style_axis(ax)
            x = np.arange(len(merged))
            w = 0.4
            ax.bar(x - w/2, merged["passed_pool"], width=w, label="Passed pool", color=cond_bar_colors[0])
            ax.bar(x + w/2, merged["shortlist"], width=w, label="Shortlist", color=cond_bar_colors[1])
            ax.set_xticks(x)
            ax.set_xticklabels(merged.index.astype(str), rotation=25, ha="right")
            ax.set_ylabel("Count")
            ax.set_title(f"Condition retention: {cond_col}")
            ax.legend(frameon=False, fontsize=7)
            fig.tight_layout()
            export_figure(fig, dirs["figures_supplementary"] / f"sfigure_8_4_condition_retention_{cond_col}", cfg.figure_export)

# =============================================================================
# Notebook entrypoint
# =============================================================================

def main_notebook(cfg: Optional[Step8Config] = None) -> Dict[str, Any]:
    if cfg is None:
        cfg = Step8Config()

    seed_all(cfg.main_random_seed)

    step8_root = Path(cfg.project_root).resolve() / cfg.step8_dir_name
    dirs = make_output_dirs(step8_root)

    print_header("Running PepCVAE Step 8")
    print(json.dumps({
        "project_root": cfg.project_root,
        "step1_dir_candidates": cfg.step1_dir_candidates,
        "step7_dir_candidates": cfg.step7_dir_candidates,
        "step8_dir_name": cfg.step8_dir_name,
        "shortlist_top_n": cfg.shortlist_top_n,
        "final_diverse_panel_n": cfg.final_diverse_panel_n,
    }, indent=2))

    results = run_step8(cfg, dirs)
    plot_step8_main_figure(results, cfg, dirs)
    plot_step8_supplementary(results, cfg, dirs)

    save_json(
        {
            "n_reference": int(len(results["ref_df"])),
            "n_step7_raw": int(len(results["gen_df"])),
            "n_eligible": int(len(results["eligible_df"])),
            "n_passed": int(len(results["passed_df"])),
            "n_shortlist": int(len(results["shortlist_df"])),
            "n_final_panel": int(len(results["final_panel_df"])),
        },
        dirs["logs"] / "step8_counts.json",
    )

    print("\n" + "=" * 100)
    print("Step 8 completed successfully.")
    print(f"Main tables: {dirs['tables_main']}")
    print(f"Supplementary tables: {dirs['tables_supplementary']}")
    print(f"Main figures: {dirs['figures_main']}")
    print(f"Supplementary figures: {dirs['figures_supplementary']}")
    print(f"Artifacts: {dirs['artifacts']}")
    print(f"Reports: {dirs['reports']}")
    print(f"Logs: {dirs['logs']}")
    print("=" * 100)

    return results

# =============================================================================
# Execute
# =============================================================================

results = main_notebook()

In [ ]:
import pandas as pd
from pathlib import Path

step7_file = Path("/home/data3/Moe/nature_computational2/step7_v3")
csvs = sorted(step7_file.rglob("*.csv"))

print("CSV files found:")
for i, p in enumerate(csvs):
    print(f"[{i}] {p}")

# choose the most likely Step 7 candidate file index manually after printing
# then run:
df = pd.read_csv(csvs[0])   # change 0 if needed
print("\nChosen file:", csvs[0])
print("\nColumns:")
print(list(df.columns))
print("\nShape:", df.shape)

In [ ]:
import pandas as pd
from pathlib import Path

step1_file = Path("/home/data3/Moe/nature_computational2/step1")
csvs = sorted(step1_file.rglob("*.csv"))

print("CSV files found:")
for i, p in enumerate(csvs):
    print(f"[{i}] {p}")

df = pd.read_csv(csvs[0])   # change 0 if needed
print("\nChosen file:", csvs[0])
print("\nColumns:")
print(list(df.columns))
print("\nShape:", df.shape)

In [ ]:
from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE — Step 8
# Multi-objective prioritization and biological plausibility screening
# Publication-grade, path-stable, notebook-ready
# =============================================================================

import json
import platform
import random
import hashlib
import warnings
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from itertools import combinations, groupby
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

# =============================================================================
# Global constants
# =============================================================================

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)

AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_TINY = set("AGSC")
AA_SMALL = set("AGSCVTNDP")

AA_PROPERTIES = {
    "A": {"hydro": 1.8, "mw": 89.09},
    "C": {"hydro": 2.5, "mw": 121.16},
    "D": {"hydro": -3.5, "mw": 133.10},
    "E": {"hydro": -3.5, "mw": 147.13},
    "F": {"hydro": 2.8, "mw": 165.19},
    "G": {"hydro": -0.4, "mw": 75.07},
    "H": {"hydro": -3.2, "mw": 155.16},
    "I": {"hydro": 4.5, "mw": 131.17},
    "K": {"hydro": -3.9, "mw": 146.19},
    "L": {"hydro": 3.8, "mw": 131.17},
    "M": {"hydro": 1.9, "mw": 149.21},
    "N": {"hydro": -3.5, "mw": 132.12},
    "P": {"hydro": -1.6, "mw": 115.13},
    "Q": {"hydro": -3.5, "mw": 146.15},
    "R": {"hydro": -4.5, "mw": 174.20},
    "S": {"hydro": -0.8, "mw": 105.09},
    "T": {"hydro": -0.7, "mw": 119.12},
    "V": {"hydro": 4.2, "mw": 117.15},
    "W": {"hydro": -0.9, "mw": 204.23},
    "Y": {"hydro": -1.3, "mw": 181.19},
}

# =============================================================================
# Figure style / project palette
# =============================================================================

PROJECT_COLORS = {
    "deep_crimson": "#D1001F",
    "warm_gold": "#FF7F0E",
    "teal": "#238383",
    "navy": "#35558F",
    "deep_red_maroon": "#A40000",
    "neutral_gray": "#9A9A9A",
    "bg": "#FFFFFF",
    "grid": "#E5E5E5",
    "spine": "#BBBBBB",
    "text": "#222222",
}

def apply_project_matplotlib_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PROJECT_COLORS["bg"],
        "axes.facecolor": PROJECT_COLORS["bg"],
        "savefig.facecolor": PROJECT_COLORS["bg"],
        "axes.edgecolor": PROJECT_COLORS["spine"],
        "axes.labelcolor": PROJECT_COLORS["text"],
        "xtick.color": PROJECT_COLORS["text"],
        "ytick.color": PROJECT_COLORS["text"],
        "text.color": PROJECT_COLORS["text"],
        "axes.titlecolor": PROJECT_COLORS["text"],
        "grid.color": PROJECT_COLORS["grid"],
        "axes.grid": True,
        "grid.linestyle": "-",
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    })

apply_project_matplotlib_style()

# =============================================================================
# Configuration
# =============================================================================

@dataclass
class FigureExportConfig:
    png_dpi: int = 300
    tif_dpi: int = 600
    export_pdf: bool = True
    export_png: bool = True
    export_tif: bool = True
    single_col_width_in: float = 3.5
    double_col_width_in: float = 7.1
    panel_fontsize: int = 10
    base_fontsize: int = 8
    title_fontsize: int = 8
    legend_fontsize: int = 7


@dataclass
class Step8Config:
    project_root: str = "/home/data3/Moe/nature_computational2/"
    step1_dir: str = "step1"
    step7_dir: str = "step7_v3"
    step8_dir: str = "step8"

    # exact paths can be set manually if needed
    step1_reference_file: Optional[str] = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"
    step7_candidate_file: Optional[str] = None

    # search rules for candidate-level Step 7 table
    step7_exclude_path_keywords: Tuple[str, ...] = (
        "outputs/source_data",
        "fig",
        "suppfig",
        "source_data",
    )

    # likely candidate filenames if such a file exists
    step7_candidate_aliases: Tuple[str, ...] = (
        "step7_generated_candidates.csv",
        "step7_generated_candidates_clean.csv",
        "step7_generated_candidates_ranked.csv",
        "step7_candidate_pool.csv",
        "generated_candidates.csv",
        "candidate_pool.csv",
        "step7_final_candidates.csv",
        "step7_screening_pool.csv",
    )

    sequence_col_candidates: Tuple[str, ...] = ("sequence", "peptide", "seq", "peptide_sequence")
    id_col_candidates: Tuple[str, ...] = ("generated_id", "candidate_id", "peptide_id", "id")
    merge_key_col_candidates: Tuple[str, ...] = ("sequence_sha256", "seq_sha256")

    exact_train_match_col_candidates: Tuple[str, ...] = (
        "is_exact_train_match", "train_exact_match", "exact_match_train", "in_train_reference"
    )
    train_nn_similarity_col_candidates: Tuple[str, ...] = (
        "train_nn_similarity", "nearest_train_similarity", "max_train_similarity",
        "train_jaccard_nn", "nn_similarity_train"
    )
    exact_full_match_col_candidates: Tuple[str, ...] = (
        "is_exact_full_match", "full_exact_match", "exact_match_full", "in_full_reference"
    )

    requested_length_bin_col_candidates: Tuple[str, ...] = (
        "requested_length_bin", "target_length_bin", "condition_length_bin", "length_bin"
    )
    requested_charge_bin_col_candidates: Tuple[str, ...] = (
        "requested_charge_bin", "target_charge_bin", "condition_charge_bin", "net_charge_bin"
    )
    requested_hydrophobicity_bin_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_bin", "target_hydrophobicity_bin",
        "condition_hydrophobicity_bin", "hydrophobicity_bin"
    )

    requested_length_min_col_candidates: Tuple[str, ...] = ("requested_length_min", "target_length_min")
    requested_length_max_col_candidates: Tuple[str, ...] = ("requested_length_max", "target_length_max")
    requested_charge_min_col_candidates: Tuple[str, ...] = ("requested_charge_min", "target_charge_min")
    requested_charge_max_col_candidates: Tuple[str, ...] = ("requested_charge_max", "target_charge_max")
    requested_hydrophobicity_min_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_min", "target_hydrophobicity_min"
    )
    requested_hydrophobicity_max_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_max", "target_hydrophobicity_max"
    )

    supportive_score_candidates: Tuple[str, ...] = (
        "acp_probability", "acp_score", "predicted_acp_probability",
        "prob_acp", "amp_probability", "supportive_model_score"
    )
    toxicity_score_candidates: Tuple[str, ...] = (
        "toxicity_probability", "toxicity_score", "predicted_toxicity"
    )
    hemolysis_score_candidates: Tuple[str, ...] = (
        "hemolysis_probability", "hemolysis_score", "predicted_hemolysis"
    )

    main_random_seed: int = 42
    bootstrap_iterations: int = 300

    min_length: int = 5
    max_length: int = 50
    low_complexity_entropy_threshold: float = 1.50
    max_homopolymer_run_fraction: float = 0.35
    max_single_residue_fraction: float = 0.60
    min_unique_residue_fraction: float = 0.25
    max_train_nn_similarity: float = 0.85
    require_not_in_train_exact: bool = True
    require_not_in_full_exact_if_available: bool = False

    similarity_kmer: int = 3
    shortlist_top_n: int = 24
    final_diverse_panel_n: int = 12
    similarity_threshold_for_clusters: float = 0.75
    max_per_similarity_cluster: int = 2

    reference_quantile_low: float = 0.02
    reference_quantile_high: float = 0.98

    descriptor_weight_map: Dict[str, float] = field(default_factory=lambda: {
        "length": 1.00,
        "mean_hydropathy": 1.00,
        "net_charge_proxy": 1.20,
        "hydrophobic_frac": 0.90,
        "positive_frac": 1.10,
        "negative_frac": 0.80,
        "aromatic_frac": 0.70,
        "entropy": 1.00,
        "max_run_fraction": 1.00,
    })

    ranking_weights: Dict[str, float] = field(default_factory=lambda: {
        "novelty_score": 0.25,
        "descriptor_plausibility_score": 0.25,
        "condition_fidelity_score": 0.20,
        "diversity_score": 0.15,
        "supportive_model_score": 0.15,
    })

    ranking_schemes: Dict[str, Dict[str, float]] = field(default_factory=lambda: {
        "primary": {
            "novelty_score": 0.25,
            "descriptor_plausibility_score": 0.25,
            "condition_fidelity_score": 0.20,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "novelty_heavy": {
            "novelty_score": 0.35,
            "descriptor_plausibility_score": 0.20,
            "condition_fidelity_score": 0.15,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "plausibility_heavy": {
            "novelty_score": 0.15,
            "descriptor_plausibility_score": 0.35,
            "condition_fidelity_score": 0.20,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "condition_heavy": {
            "novelty_score": 0.20,
            "descriptor_plausibility_score": 0.20,
            "condition_fidelity_score": 0.30,
            "diversity_score": 0.15,
            "supportive_model_score": 0.15,
        },
        "supportive_heavy": {
            "novelty_score": 0.15,
            "descriptor_plausibility_score": 0.20,
            "condition_fidelity_score": 0.15,
            "diversity_score": 0.10,
            "supportive_model_score": 0.40,
        },
    })

    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)


# =============================================================================
# Reproducibility and IO
# =============================================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def collect_environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp_utc": now_utc_iso(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
        "cwd": str(Path.cwd()),
    }

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.export_pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.export_tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)

def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


# =============================================================================
# Discovery helpers
# =============================================================================

def first_existing_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = False, label: str = "") -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not resolve required column for {label}. Tried: {list(candidates)}")
    return None

def resolve_exact_train_match_column(
    df: pd.DataFrame,
    exact_candidates: Sequence[str],
    nn_candidates: Sequence[str],
    derived_col_name: str = "_derived_exact_train_match_from_nn",
    atol: float = 1e-12,
) -> Tuple[str, bool]:
    exact_col = first_existing_column(df, exact_candidates, required=False)
    if exact_col is not None:
        vals = pd.to_numeric(df[exact_col], errors="coerce")
        if vals.isna().any():
            raise ValueError(f"{exact_col} contains missing/non-numeric values.")
        df[exact_col] = vals.astype(int)
        return exact_col, False

    nn_col = first_existing_column(df, nn_candidates, required=False)
    if nn_col is None:
        raise ValueError(
            "Could not resolve Step 7 exact-train-match information. "
            f"Tried exact columns: {list(exact_candidates)} "
            f"and NN columns: {list(nn_candidates)}"
        )

    nn_vals = pd.to_numeric(df[nn_col], errors="coerce")
    if nn_vals.isna().any():
        raise ValueError(f"{nn_col} contains missing/non-numeric values.")
    if ((nn_vals < 0) | (nn_vals > 1)).any():
        raise ValueError(f"{nn_col} must lie in [0, 1].")

    df[derived_col_name] = np.isclose(nn_vals, 1.0, atol=atol).astype(int)
    return derived_col_name, True

def resolve_exact_full_match_column(
    df: pd.DataFrame,
    exact_full_candidates: Sequence[str],
) -> Optional[str]:
    exact_full_col = first_existing_column(df, exact_full_candidates, required=False)
    if exact_full_col is None:
        return None
    vals = pd.to_numeric(df[exact_full_col], errors="coerce")
    if vals.isna().any():
        raise ValueError(f"{exact_full_col} contains missing/non-numeric values.")
    df[exact_full_col] = vals.astype(int)
    return exact_full_col

def list_csvs(root: Path) -> List[Path]:
    return sorted(root.rglob("*.csv"))

def find_candidate_level_step7_csv(root_dir: Path, cfg: Step8Config) -> Path:
    if cfg.step7_candidate_file is not None:
        p = Path(cfg.step7_candidate_file).resolve()
        if not p.exists():
            raise FileNotFoundError(f"Explicit step7_candidate_file does not exist: {p}")
        return p

    all_csvs = list_csvs(root_dir)
    if len(all_csvs) == 0:
        raise FileNotFoundError(f"No CSV files found under {root_dir}")

    alias_hits = []
    for p in all_csvs:
        if p.name in set(cfg.step7_candidate_aliases):
            alias_hits.append(p)

    def is_excluded(path: Path) -> bool:
        s = str(path).lower().replace("\\", "/")
        return any(k.lower() in s for k in cfg.step7_exclude_path_keywords)

    candidate_pool = alias_hits if len(alias_hits) > 0 else all_csvs
    candidate_pool = [p for p in candidate_pool if not is_excluded(p)]

    valid_candidates = []
    for p in candidate_pool:
        try:
            preview = pd.read_csv(p, nrows=20)
        except Exception:
            continue

        seq_col = first_existing_column(preview, cfg.sequence_col_candidates, required=False)
        nn_col = first_existing_column(preview, cfg.train_nn_similarity_col_candidates, required=False)
        len_bin = first_existing_column(preview, cfg.requested_length_bin_col_candidates, required=False)
        chg_bin = first_existing_column(preview, cfg.requested_charge_bin_col_candidates, required=False)
        hyd_bin = first_existing_column(preview, cfg.requested_hydrophobicity_bin_col_candidates, required=False)

        if seq_col is None:
            continue
        if nn_col is None:
            continue
        if len_bin is None or chg_bin is None or hyd_bin is None:
            continue

        valid_candidates.append(p)

    if len(valid_candidates) == 0:
        print("\nNo valid candidate-level Step 7 file was found automatically.")
        print("These CSVs were found under Step 7:")
        for i, p in enumerate(all_csvs):
            print(f"[{i}] {p}")
        raise FileNotFoundError(
            "Step 8 needs a candidate-level Step 7 CSV containing per-peptide rows with at least:\n"
            "- sequence\n"
            "- train nearest-neighbor similarity\n"
            "- requested length/charge/hydrophobicity bins\n\n"
            "The files under outputs/source_data are aggregate figure source-data tables and are not valid Step 8 inputs.\n"
            "Please set cfg.step7_candidate_file to the correct Step 7 candidate table."
        )

    valid_candidates = sorted(valid_candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return valid_candidates[0]


# =============================================================================
# Sequence utilities and descriptors
# =============================================================================

def clean_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def is_valid_peptide(seq: str, min_len: int = 1, max_len: Optional[int] = None) -> bool:
    if not seq:
        return False
    if any(ch not in AA_SET for ch in seq):
        return False
    if len(seq) < min_len:
        return False
    if max_len is not None and len(seq) > max_len:
        return False
    return True

def shannon_entropy(seq: str) -> float:
    if not seq:
        return 0.0
    counts = Counter(seq)
    probs = np.array(list(counts.values()), dtype=float) / len(seq)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, None))).sum())

def max_run_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    runs = [len(list(group)) for _, group in groupby(seq)]
    return float(max(runs) / len(seq))

def aa_fraction(seq: str, aa: str) -> float:
    return seq.count(aa) / len(seq) if seq else 0.0

def unique_residue_fraction(seq: str) -> float:
    return len(set(seq)) / len(seq) if seq else 0.0

def max_single_residue_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    counts = Counter(seq)
    return float(max(counts.values()) / len(seq))

def make_kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}

def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
    s1 = make_kmers(seq1, k=k)
    s2 = make_kmers(seq2, k=k)
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    union = len(s1 | s2)
    if union == 0:
        return 0.0
    return len(s1 & s2) / union

def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    if not seq:
        base = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
        base.update(
            {
                "length": 0.0,
                "mean_hydropathy": 0.0,
                "hydrophobic_frac": 0.0,
                "aromatic_frac": 0.0,
                "positive_frac": 0.0,
                "negative_frac": 0.0,
                "polar_frac": 0.0,
                "tiny_frac": 0.0,
                "small_frac": 0.0,
                "net_charge_proxy": 0.0,
                "molecular_weight_proxy": 0.0,
                "entropy": 0.0,
                "max_run_fraction": 1.0,
                "unique_residue_fraction": 0.0,
                "max_single_residue_fraction": 1.0,
            }
        )
        return base

    hydros = np.array([AA_PROPERTIES[a]["hydro"] for a in seq], dtype=float)
    weights = np.array([AA_PROPERTIES[a]["mw"] for a in seq], dtype=float)

    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    desc.update(
        {
            "length": float(len(seq)),
            "mean_hydropathy": float(hydros.mean()),
            "hydrophobic_frac": float(np.mean([a in AA_HYDROPHOBIC for a in seq])),
            "aromatic_frac": float(np.mean([a in AA_AROMATIC for a in seq])),
            "positive_frac": float(np.mean([a in AA_POSITIVE for a in seq])),
            "negative_frac": float(np.mean([a in AA_NEGATIVE for a in seq])),
            "polar_frac": float(np.mean([a in AA_POLAR for a in seq])),
            "tiny_frac": float(np.mean([a in AA_TINY for a in seq])),
            "small_frac": float(np.mean([a in AA_SMALL for a in seq])),
            "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
            "molecular_weight_proxy": float(weights.sum()),
            "entropy": shannon_entropy(seq),
            "max_run_fraction": max_run_fraction(seq),
            "unique_residue_fraction": unique_residue_fraction(seq),
            "max_single_residue_fraction": max_single_residue_fraction(seq),
        }
    )
    return desc

def add_computable_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in df[seq_col].astype(str)])
    return pd.concat([df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)


# =============================================================================
# Input loading and preparation
# =============================================================================

def load_step_inputs(cfg: Step8Config) -> Dict[str, Any]:
    project_root = Path(cfg.project_root).resolve()
    step1_root = project_root / cfg.step1_dir
    step7_root = project_root / cfg.step7_dir
    step8_root = project_root / cfg.step8_dir

    if not step1_root.exists():
        raise FileNotFoundError(f"Step 1 directory not found: {step1_root}")
    if not step7_root.exists():
        raise FileNotFoundError(f"Step 7 directory not found: {step7_root}")

    if cfg.step1_reference_file is not None:
        step1_ref_path = Path(cfg.step1_reference_file).resolve()
        if not step1_ref_path.exists():
            raise FileNotFoundError(f"Configured Step 1 reference file not found: {step1_ref_path}")
    else:
        fallback = step1_root / "tables" / "step1_retained_dataset.csv"
        if not fallback.exists():
            raise FileNotFoundError("Could not resolve Step 1 reference file automatically.")
        step1_ref_path = fallback.resolve()

    step7_candidate_path = find_candidate_level_step7_csv(step7_root, cfg)

    ref_df = pd.read_csv(step1_ref_path)
    gen_df = pd.read_csv(step7_candidate_path)

    print(f"Resolved Step 1 reference: {step1_ref_path}")
    print(f"Resolved Step 7 candidate file: {step7_candidate_path}")

    return {
        "project_root": project_root,
        "step1_ref_path": step1_ref_path,
        "step7_candidate_path": step7_candidate_path,
        "step1_ref_df": ref_df,
        "step7_generated_df": gen_df,
        "step8_root": step8_root,
    }

def prepare_reference_df(ref_df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df = ref_df.copy()
    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="Step 1 sequence")
    colmap = {"sequence_col": seq_col}

    df[seq_col] = df[seq_col].astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].copy()
    df = df.drop_duplicates(subset=[seq_col]).reset_index(drop=True)
    df = add_computable_descriptors(df, seq_col)

    merge_key_col = first_existing_column(df, cfg.merge_key_col_candidates, required=False)
    if merge_key_col is None:
        merge_key_col = "sequence_sha256"
        df[merge_key_col] = df[seq_col].map(lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())
    colmap["merge_key_col"] = merge_key_col

    return df, colmap

def prepare_generated_df(gen_df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df = gen_df.copy()

    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="Step 7 sequence")
    id_col = first_existing_column(df, cfg.id_col_candidates, required=False)
    if id_col is None:
        id_col = "generated_id"
        df[id_col] = [f"gen_{i:06d}" for i in range(len(df))]

    train_nn_similarity_col = first_existing_column(
        df,
        cfg.train_nn_similarity_col_candidates,
        required=True,
        label="Step 7 train NN similarity",
    )

    exact_train_match_col, exact_train_was_derived = resolve_exact_train_match_column(
        df=df,
        exact_candidates=cfg.exact_train_match_col_candidates,
        nn_candidates=cfg.train_nn_similarity_col_candidates,
        derived_col_name="_derived_exact_train_match_from_nn",
        atol=1e-12,
    )

    exact_full_match_col = resolve_exact_full_match_column(
        df=df,
        exact_full_candidates=cfg.exact_full_match_col_candidates,
    )

    colmap = {
        "sequence_col": seq_col,
        "id_col": id_col,
        "exact_train_match_col": exact_train_match_col,
        "train_nn_similarity_col": train_nn_similarity_col,
        "exact_full_match_col": exact_full_match_col,
        "requested_length_bin_col": first_existing_column(df, cfg.requested_length_bin_col_candidates, required=True, label="Step 7 requested length bin"),
        "requested_charge_bin_col": first_existing_column(df, cfg.requested_charge_bin_col_candidates, required=True, label="Step 7 requested charge bin"),
        "requested_hydrophobicity_bin_col": first_existing_column(df, cfg.requested_hydrophobicity_bin_col_candidates, required=True, label="Step 7 requested hydrophobicity bin"),
        "requested_length_min_col": first_existing_column(df, cfg.requested_length_min_col_candidates, required=False),
        "requested_length_max_col": first_existing_column(df, cfg.requested_length_max_col_candidates, required=False),
        "requested_charge_min_col": first_existing_column(df, cfg.requested_charge_min_col_candidates, required=False),
        "requested_charge_max_col": first_existing_column(df, cfg.requested_charge_max_col_candidates, required=False),
        "requested_hydrophobicity_min_col": first_existing_column(df, cfg.requested_hydrophobicity_min_col_candidates, required=False),
        "requested_hydrophobicity_max_col": first_existing_column(df, cfg.requested_hydrophobicity_max_col_candidates, required=False),
        "exact_train_match_was_derived": exact_train_was_derived,
    }

    merge_key_col = first_existing_column(df, cfg.merge_key_col_candidates, required=False)
    if merge_key_col is None:
        merge_key_col = "sequence_sha256"
        df[merge_key_col] = df[seq_col].map(lambda s: hashlib.sha256(clean_sequence(s).encode("utf-8")).hexdigest())
    colmap["merge_key_col"] = merge_key_col

    df[seq_col] = df[seq_col].astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].copy()

    if df[id_col].duplicated().any():
        dup_n = int(df[id_col].duplicated().sum())
        raise ValueError(f"Generated candidate table has {dup_n} duplicated IDs.")

    df = add_computable_descriptors(df, seq_col)

    nn_vals = pd.to_numeric(df[colmap["train_nn_similarity_col"]], errors="coerce")
    if nn_vals.isna().any():
        raise ValueError(f"{colmap['train_nn_similarity_col']} contains missing/non-numeric values.")
    if ((nn_vals < 0) | (nn_vals > 1)).any():
        raise ValueError(f"{colmap['train_nn_similarity_col']} must lie in [0, 1].")

    exact_vals = pd.to_numeric(df[colmap["exact_train_match_col"]], errors="coerce")
    if exact_vals.isna().any():
        raise ValueError(f"{colmap['exact_train_match_col']} contains missing/non-numeric values.")

    for cname in [
        colmap["requested_length_bin_col"],
        colmap["requested_charge_bin_col"],
        colmap["requested_hydrophobicity_bin_col"],
    ]:
        if df[cname].isna().any():
            raise ValueError(f"{cname} contains missing values. Step 8 requires condition metadata.")

    return df.reset_index(drop=True), colmap


# =============================================================================
# Supportive predictor handling
# =============================================================================

def build_supportive_predictor_frame(df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.Series, pd.DataFrame]:
    support_col = first_existing_column(df, cfg.supportive_score_candidates, required=False)
    tox_col = first_existing_column(df, cfg.toxicity_score_candidates, required=False)
    hemo_col = first_existing_column(df, cfg.hemolysis_score_candidates, required=False)

    provenance_rows = [
        {"role": "supportive_acp_score", "column_used": support_col, "status": "used" if support_col else "not_available", "notes": "Supportive only; not direct proof."},
        {"role": "toxicity_penalty", "column_used": tox_col, "status": "used" if tox_col else "not_available", "notes": "Penalty only."},
        {"role": "hemolysis_penalty", "column_used": hemo_col, "status": "used" if hemo_col else "not_available", "notes": "Penalty only."},
    ]

    if support_col is None:
        supportive = pd.Series(np.full(len(df), 0.5), index=df.index, dtype=float)
    else:
        supportive = pd.to_numeric(df[support_col], errors="coerce")
        if supportive.isna().any():
            raise ValueError(f"Supportive predictor column {support_col} contains missing/non-numeric values.")
        supportive = supportive.clip(0.0, 1.0)

    tox_pen = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    if tox_col is not None:
        tox_pen = pd.to_numeric(df[tox_col], errors="coerce")
        if tox_pen.isna().any():
            raise ValueError(f"Toxicity column {tox_col} contains missing/non-numeric values.")
        tox_pen = tox_pen.clip(0.0, 1.0)

    hemo_pen = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    if hemo_col is not None:
        hemo_pen = pd.to_numeric(df[hemo_col], errors="coerce")
        if hemo_pen.isna().any():
            raise ValueError(f"Hemolysis column {hemo_col} contains missing/non-numeric values.")
        hemo_pen = hemo_pen.clip(0.0, 1.0)

    supportive_score = (supportive - 0.5 * tox_pen - 0.5 * hemo_pen).clip(0.0, 1.0)
    return supportive_score, pd.DataFrame(provenance_rows)


# =============================================================================
# Hard filtering
# =============================================================================

def build_hard_filter_flags(df: pd.DataFrame, cfg: Step8Config, colmap: Dict[str, str]) -> pd.DataFrame:
    seq_col = colmap["sequence_col"]
    exact_train_col = colmap["exact_train_match_col"]
    train_nn_col = colmap["train_nn_similarity_col"]
    exact_full_col = colmap["exact_full_match_col"]

    out = pd.DataFrame(index=df.index)
    out["flag_invalid_sequence"] = (~df[seq_col].map(lambda s: is_valid_peptide(s, cfg.min_length, cfg.max_length))).astype(int)
    out["flag_too_short"] = (df["length"] < cfg.min_length).astype(int)
    out["flag_too_long"] = (df["length"] > cfg.max_length).astype(int)
    out["flag_low_entropy"] = (df["entropy"] < cfg.low_complexity_entropy_threshold).astype(int)
    out["flag_high_run_fraction"] = (df["max_run_fraction"] > cfg.max_homopolymer_run_fraction).astype(int)
    out["flag_high_single_residue_fraction"] = (df["max_single_residue_fraction"] > cfg.max_single_residue_fraction).astype(int)
    out["flag_low_unique_residue_fraction"] = (df["unique_residue_fraction"] < cfg.min_unique_residue_fraction).astype(int)
    out["flag_duplicate_sequence"] = df.duplicated(subset=[seq_col], keep="first").astype(int)
    out["flag_exact_train_match"] = pd.to_numeric(df[exact_train_col], errors="coerce").astype(int)

    if exact_full_col is not None and cfg.require_not_in_full_exact_if_available:
        out["flag_exact_full_match"] = pd.to_numeric(df[exact_full_col], errors="coerce").astype(int)
    else:
        out["flag_exact_full_match"] = 0

    out["flag_high_train_similarity"] = (
        pd.to_numeric(df[train_nn_col], errors="coerce") > cfg.max_train_nn_similarity
    ).astype(int)

    active_flags = [
        "flag_invalid_sequence",
        "flag_too_short",
        "flag_too_long",
        "flag_low_entropy",
        "flag_high_run_fraction",
        "flag_high_single_residue_fraction",
        "flag_low_unique_residue_fraction",
        "flag_duplicate_sequence",
    ]
    if cfg.require_not_in_train_exact:
        active_flags.append("flag_exact_train_match")
    active_flags.append("flag_high_train_similarity")
    if cfg.require_not_in_full_exact_if_available and exact_full_col is not None:
        active_flags.append("flag_exact_full_match")

    out["hard_filter_fail"] = (out[active_flags].sum(axis=1) > 0).astype(int)
    return out

def summarize_filter_waterfall(flags_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(flags_df)
    flag_cols = [c for c in flags_df.columns if c.startswith("flag_")]
    for c in flag_cols:
        removed = int(flags_df[c].sum())
        rows.append({
            "filter_name": c,
            "removed_n": removed,
            "removed_fraction": removed / n if n else np.nan,
            "retained_if_applied_alone": n - removed,
        })
    return pd.DataFrame(rows).sort_values("removed_n", ascending=False).reset_index(drop=True)


# =============================================================================
# ACP plausibility support
# =============================================================================

def build_reference_bounds(ref_df: pd.DataFrame, cfg: Step8Config) -> pd.DataFrame:
    rows = []
    for metric, weight in cfg.descriptor_weight_map.items():
        vals = pd.to_numeric(ref_df[metric], errors="coerce").dropna()
        if len(vals) < 20:
            raise ValueError(f"Reference corpus has insufficient support for descriptor '{metric}'.")
        rows.append({
            "descriptor": metric,
            "q_low": float(vals.quantile(cfg.reference_quantile_low)),
            "q_high": float(vals.quantile(cfg.reference_quantile_high)),
            "median": float(vals.median()),
            "iqr": float(vals.quantile(0.75) - vals.quantile(0.25)),
            "weight": weight,
        })
    return pd.DataFrame(rows)

def descriptor_support_score(series: pd.Series, q_low: float, q_high: float) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce").astype(float)
    center = 0.5 * (q_low + q_high)
    half = max(0.5 * (q_high - q_low), 1e-8)
    z = np.abs(x - center) / half
    return (1.0 - np.clip(z, 0.0, 1.0)).fillna(0.0)

def compute_plausibility_scores(df: pd.DataFrame, reference_bounds_df: pd.DataFrame) -> Tuple[pd.Series, pd.DataFrame]:
    comp = pd.DataFrame(index=df.index)
    fail_cols = []

    weighted_num = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    weighted_den = 0.0

    for _, row in reference_bounds_df.iterrows():
        metric = row["descriptor"]
        q_low = float(row["q_low"])
        q_high = float(row["q_high"])
        weight = float(row["weight"])

        score_col = f"support_{metric}"
        fail_col = f"flag_outside_reference_{metric}"
        comp[score_col] = descriptor_support_score(df[metric], q_low, q_high)
        comp[fail_col] = ((df[metric] < q_low) | (df[metric] > q_high)).astype(int)
        fail_cols.append(fail_col)

        weighted_num += comp[score_col] * weight
        weighted_den += weight

    comp["plausibility_fail_count"] = comp[fail_cols].sum(axis=1)
    comp["descriptor_plausibility_score"] = weighted_num / weighted_den
    comp["descriptor_plausibility_pass"] = (comp["plausibility_fail_count"] <= 2).astype(int)

    return comp["descriptor_plausibility_score"].clip(0.0, 1.0), comp


# =============================================================================
# Condition fidelity
# =============================================================================

def infer_length_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 10:
        return "short"
    if x <= 20:
        return "medium"
    return "long"

def infer_charge_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 2:
        return "low"
    if x <= 6:
        return "medium"
    return "high"

def infer_hydropathy_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x < -1.0:
        return "low"
    if x <= 1.0:
        return "medium"
    return "high"

def compute_condition_fidelity(df: pd.DataFrame, colmap: Dict[str, str]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["length_bin_match"] = (
        df[colmap["requested_length_bin_col"]].astype(str).str.lower().str.strip()
        == df["length"].map(infer_length_bin).astype(str).str.lower().str.strip()
    ).astype(int)

    out["charge_bin_match"] = (
        df[colmap["requested_charge_bin_col"]].astype(str).str.lower().str.strip()
        == df["net_charge_proxy"].map(infer_charge_bin).astype(str).str.lower().str.strip()
    ).astype(int)

    out["hydropathy_bin_match"] = (
        df[colmap["requested_hydrophobicity_bin_col"]].astype(str).str.lower().str.strip()
        == df["mean_hydropathy"].map(infer_hydropathy_bin).astype(str).str.lower().str.strip()
    ).astype(int)

    if colmap.get("requested_length_min_col") and colmap.get("requested_length_max_col"):
        out["length_range_match"] = (
            (df["length"] >= pd.to_numeric(df[colmap["requested_length_min_col"]], errors="coerce")) &
            (df["length"] <= pd.to_numeric(df[colmap["requested_length_max_col"]], errors="coerce"))
        ).astype(int)

    if colmap.get("requested_charge_min_col") and colmap.get("requested_charge_max_col"):
        out["charge_range_match"] = (
            (df["net_charge_proxy"] >= pd.to_numeric(df[colmap["requested_charge_min_col"]], errors="coerce")) &
            (df["net_charge_proxy"] <= pd.to_numeric(df[colmap["requested_charge_max_col"]], errors="coerce"))
        ).astype(int)

    if colmap.get("requested_hydrophobicity_min_col") and colmap.get("requested_hydrophobicity_max_col"):
        out["hydropathy_range_match"] = (
            (df["mean_hydropathy"] >= pd.to_numeric(df[colmap["requested_hydrophobicity_min_col"]], errors="coerce")) &
            (df["mean_hydropathy"] <= pd.to_numeric(df[colmap["requested_hydrophobicity_max_col"]], errors="coerce"))
        ).astype(int)

    pass_cols = [c for c in out.columns if c.endswith("_match")]
    if len(pass_cols) == 0:
        raise ValueError("Condition fidelity columns could not be computed.")

    out["condition_fidelity_score"] = out[pass_cols].mean(axis=1)
    out["condition_fidelity_pass"] = (out["condition_fidelity_score"] >= 0.67).astype(int)
    return out


# =============================================================================
# Novelty and diversity
# =============================================================================

def compute_novelty_score(df: pd.DataFrame, colmap: Dict[str, str]) -> pd.Series:
    exact_train_novel = 1.0 - pd.to_numeric(df[colmap["exact_train_match_col"]], errors="coerce").clip(0, 1)
    nn_novel = 1.0 - pd.to_numeric(df[colmap["train_nn_similarity_col"]], errors="coerce").clip(0, 1)

    components = [exact_train_novel, nn_novel]
    if colmap.get("exact_full_match_col") is not None:
        full_novel = 1.0 - pd.to_numeric(df[colmap["exact_full_match_col"]], errors="coerce").clip(0, 1)
        components.append(full_novel)

    return pd.concat(components, axis=1).mean(axis=1).clip(0.0, 1.0)

def compute_pairwise_similarity_edges(seqs: Sequence[str], threshold: float, k: int) -> List[Tuple[int, int, float]]:
    edges = []
    for i in range(len(seqs)):
        for j in range(i + 1, len(seqs)):
            sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=k)
            if sim >= threshold:
                edges.append((i, j, sim))
    return edges

def connected_components_from_edges(n_nodes: int, edges: Sequence[Tuple[int, int, float]]) -> List[List[int]]:
    adj = [[] for _ in range(n_nodes)]
    for i, j, _ in edges:
        adj[i].append(j)
        adj[j].append(i)

    seen = np.zeros(n_nodes, dtype=bool)
    comps = []
    for start in range(n_nodes):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
        comps.append(sorted(comp))
    return comps

def compute_diversity_scores(seqs: Sequence[str], k: int) -> pd.Series:
    n = len(seqs)
    if n <= 1:
        return pd.Series(np.ones(n))
    nn_div = []
    for i in range(n):
        sims = []
        for j in range(n):
            if i == j:
                continue
            sims.append(jaccard_kmer_similarity(seqs[i], seqs[j], k=k))
        nearest = max(sims) if sims else 0.0
        nn_div.append(1.0 - nearest)
    return pd.Series(nn_div).clip(0.0, 1.0)

def shortlist_with_cluster_cap(
    df: pd.DataFrame,
    seq_col: str,
    score_col: str,
    threshold: float,
    max_per_cluster: int,
    top_n: int,
    k: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    seqs = df[seq_col].tolist()
    edges = compute_pairwise_similarity_edges(seqs, threshold=threshold, k=k)
    comps = connected_components_from_edges(len(seqs), edges)

    cluster_id = np.full(len(df), -1, dtype=int)
    for cid, comp in enumerate(comps):
        for idx in comp:
            cluster_id[idx] = cid

    out = df.copy()
    out["similarity_cluster_id"] = cluster_id

    chosen = []
    cluster_counts: Dict[int, int] = defaultdict(int)
    ranked_idx = out.sort_values(score_col, ascending=False).index.tolist()
    for idx in ranked_idx:
        cid = int(out.loc[idx, "similarity_cluster_id"])
        if cluster_counts[cid] < max_per_cluster:
            chosen.append(idx)
            cluster_counts[cid] += 1
        if len(chosen) >= top_n:
            break

    chosen_df = out.loc[chosen].copy()
    chosen_df = chosen_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    return chosen_df, out


# =============================================================================
# Ranking and stability
# =============================================================================

def weighted_score(df: pd.DataFrame, weights: Dict[str, float]) -> pd.Series:
    missing = [c for c in weights if c not in df.columns]
    if missing:
        raise ValueError(f"Ranking requested missing columns: {missing}")
    total_w = sum(weights.values())
    s = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    for c, w in weights.items():
        s += df[c] * w
    return s / total_w

def dense_rank_desc(s: pd.Series) -> pd.Series:
    return s.rank(method="dense", ascending=False).astype(int)

def compute_scheme_stability(results_by_scheme: Dict[str, pd.DataFrame], id_col: str, top_n: int) -> pd.DataFrame:
    rows = []
    scheme_names = list(results_by_scheme.keys())
    for a, b in combinations(scheme_names, 2):
        top_a = set(results_by_scheme[a].head(top_n)[id_col].astype(str))
        top_b = set(results_by_scheme[b].head(top_n)[id_col].astype(str))
        inter = len(top_a & top_b)
        union = len(top_a | top_b)
        jacc = inter / union if union else np.nan
        overlap_frac = inter / top_n if top_n else np.nan
        rows.append({
            "scheme_a": a,
            "scheme_b": b,
            "top_n": top_n,
            "intersection_n": inter,
            "union_n": union,
            "jaccard_overlap": jacc,
            "overlap_fraction_of_top_n": overlap_frac,
        })
    return pd.DataFrame(rows)

def bootstrap_shortlist_membership_stability(df: pd.DataFrame, id_col: str, weights: Dict[str, float], top_n: int, n_boot: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for b in range(n_boot):
        sampled = df.sample(n=len(df), replace=True, random_state=int(rng.integers(0, 1_000_000)))
        sampled_score = weighted_score(sampled, weights)
        sampled = sampled.assign(_boot_score=sampled_score).sort_values("_boot_score", ascending=False).head(top_n)
        for pid in sampled[id_col].astype(str):
            rows.append({"bootstrap_iter": b, id_col: pid, "in_top_n": 1})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=[id_col, "bootstrap_top_n_frequency"])
    return out.groupby(id_col)["in_top_n"].mean().rename("bootstrap_top_n_frequency").reset_index()


# =============================================================================
# Main pipeline
# =============================================================================

def run_step8(cfg: Step8Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Loading Step 1 / Step 7 inputs")
    loaded = load_step_inputs(cfg)

    save_json(
        {
            "config": asdict(cfg),
            "step1_reference_path": str(loaded["step1_ref_path"]),
            "step7_candidate_path": str(loaded["step7_candidate_path"]),
            "input_hashes": {
                "step1_reference_sha256": sha256_of_file(loaded["step1_ref_path"]),
                "step7_candidate_sha256": sha256_of_file(loaded["step7_candidate_path"]),
            },
            "environment_manifest": collect_environment_manifest(),
        },
        dirs["artifacts"] / "step8_run_manifest.json",
    )

    ref_df, _ = prepare_reference_df(loaded["step1_ref_df"], cfg)
    gen_df, gen_colmap = prepare_generated_df(loaded["step7_generated_df"], cfg)

    print(f"Loaded Step 1 reference rows: {len(ref_df):,}")
    print(f"Loaded Step 7 candidate rows: {len(gen_df):,}")

    print_header("Applying hard eligibility filters")
    flags_df = build_hard_filter_flags(gen_df, cfg, gen_colmap)
    filter_audit_df = summarize_filter_waterfall(flags_df)

    gen_scored = pd.concat([gen_df.reset_index(drop=True), flags_df.reset_index(drop=True)], axis=1)
    gen_scored["eligible_after_hard_filters"] = 1 - gen_scored["hard_filter_fail"]

    filter_audit_df.to_csv(dirs["tables_supplementary"] / "table_s8_1_filter_audit.csv", index=False)
    gen_scored.to_csv(dirs["tables_supplementary"] / "table_s8_2_candidate_pool_with_flags.csv", index=False)

    eligible = gen_scored[gen_scored["eligible_after_hard_filters"] == 1].copy().reset_index(drop=True)
    if len(eligible) == 0:
        raise ValueError("No candidates remain after hard eligibility filtering.")

    print(f"Eligible after hard filters: {len(eligible):,} / {len(gen_scored):,}")

    print_header("Computing ACP-oriented plausibility support")
    reference_bounds_df = build_reference_bounds(ref_df, cfg)
    plaus_score, plaus_detail = compute_plausibility_scores(eligible, reference_bounds_df)
    eligible = pd.concat([eligible.reset_index(drop=True), plaus_detail.reset_index(drop=True)], axis=1)
    eligible["descriptor_plausibility_score"] = plaus_score
    reference_bounds_df.to_csv(dirs["tables_supplementary"] / "table_s8_3_reference_descriptor_bounds.csv", index=False)

    print_header("Computing condition fidelity")
    condition_detail = compute_condition_fidelity(eligible, gen_colmap)
    eligible = pd.concat([eligible.reset_index(drop=True), condition_detail.reset_index(drop=True)], axis=1)

    print_header("Computing novelty and supportive scores")
    eligible["novelty_score"] = compute_novelty_score(eligible, gen_colmap)
    eligible["diversity_score"] = compute_diversity_scores(eligible[gen_colmap["sequence_col"]].tolist(), k=cfg.similarity_kmer).values
    eligible["supportive_model_score"], supportive_provenance_df = build_supportive_predictor_frame(eligible, cfg)
    supportive_provenance_df.to_csv(dirs["tables_supplementary"] / "table_s8_4_auxiliary_predictor_provenance.csv", index=False)

    eligible["final_pass"] = (
        (eligible["descriptor_plausibility_pass"] == 1) &
        (eligible["condition_fidelity_pass"] == 1)
    ).astype(int)

    passed = eligible[eligible["final_pass"] == 1].copy().reset_index(drop=True)
    if len(passed) == 0:
        raise ValueError("No candidates remain after plausibility and condition-fidelity screening.")

    print(f"Passed plausibility + condition fidelity: {len(passed):,}")

    print_header("Ranking candidates")
    score_cols = list(cfg.ranking_weights.keys())
    for c in score_cols:
        passed[c] = pd.to_numeric(passed[c], errors="coerce").clip(0.0, 1.0)
        if passed[c].isna().any():
            raise ValueError(f"Ranking component {c} contains missing values.")

    passed["final_score"] = weighted_score(passed, cfg.ranking_weights)
    passed["final_rank"] = dense_rank_desc(passed["final_score"])

    print_header("Building diversity-aware shortlist")
    seq_col = gen_colmap["sequence_col"]
    id_col = gen_colmap["id_col"]

    shortlist_df, annotated_ranked_df = shortlist_with_cluster_cap(
        df=passed.sort_values("final_score", ascending=False).reset_index(drop=True),
        seq_col=seq_col,
        score_col="final_score",
        threshold=cfg.similarity_threshold_for_clusters,
        max_per_cluster=cfg.max_per_similarity_cluster,
        top_n=cfg.shortlist_top_n,
        k=cfg.similarity_kmer,
    )
    if len(shortlist_df) == 0:
        raise ValueError("No candidates remain after shortlist diversity control.")
    shortlist_df["shortlist_rank"] = dense_rank_desc(shortlist_df["final_score"])

    final_panel_df, _ = shortlist_with_cluster_cap(
        df=shortlist_df.sort_values("final_score", ascending=False).reset_index(drop=True),
        seq_col=seq_col,
        score_col="final_score",
        threshold=cfg.similarity_threshold_for_clusters,
        max_per_cluster=1,
        top_n=cfg.final_diverse_panel_n,
        k=cfg.similarity_kmer,
    )
    if len(final_panel_df) == 0:
        raise ValueError("No candidates remain after final diverse panel selection.")
    final_panel_df["final_panel_rank"] = dense_rank_desc(final_panel_df["final_score"])

    print(f"Top shortlist size: {len(shortlist_df):,}")
    print(f"Final diverse panel size: {len(final_panel_df):,}")

    print_header("Running ranking sensitivity analysis")
    results_by_scheme: Dict[str, pd.DataFrame] = {}
    for scheme_name, scheme_weights in cfg.ranking_schemes.items():
        tmp = passed.copy()
        tmp["final_score"] = weighted_score(tmp, scheme_weights)
        tmp["final_rank"] = dense_rank_desc(tmp["final_score"])
        tmp = tmp.sort_values("final_score", ascending=False).reset_index(drop=True)
        results_by_scheme[scheme_name] = tmp

    stability_df = compute_scheme_stability(results_by_scheme, id_col, cfg.shortlist_top_n)

    top_sets = {
        name: set(df.head(cfg.shortlist_top_n)[id_col].astype(str))
        for name, df in results_by_scheme.items()
    }
    recurrence_rows = []
    union_ids = sorted(set().union(*top_sets.values()))
    for pid in union_ids:
        hit_map = {f"in_top_{name}": int(pid in idset) for name, idset in top_sets.items()}
        recurrence_rows.append({
            id_col: pid,
            "scheme_recurrence_n": sum(hit_map.values()),
            **hit_map
        })
    recurrence_df = pd.DataFrame(recurrence_rows)

    bootstrap_freq_df = bootstrap_shortlist_membership_stability(
        passed[[id_col] + score_cols].copy(),
        id_col=id_col,
        weights=cfg.ranking_weights,
        top_n=cfg.shortlist_top_n,
        n_boot=cfg.bootstrap_iterations,
        seed=cfg.main_random_seed,
    )
    if len(bootstrap_freq_df) > 0:
        recurrence_df[id_col] = recurrence_df[id_col].astype(str)
        bootstrap_freq_df[id_col] = bootstrap_freq_df[id_col].astype(str)
        recurrence_df = recurrence_df.merge(bootstrap_freq_df, on=id_col, how="left")

    stability_df.to_csv(dirs["tables_supplementary"] / "table_s8_5_ranking_scheme_stability.csv", index=False)
    recurrence_df.to_csv(dirs["tables_supplementary"] / "table_s8_6_candidate_recurrence_across_schemes.csv", index=False)

    print_header("Building descriptive enrichment summaries")
    compare_rows = []
    groups = {
        "raw_step7_pool": gen_df,
        "hard_filter_eligible": eligible,
        "screen_passed": passed,
        "shortlist": shortlist_df,
        "final_diverse_panel": final_panel_df,
        "reference_acp_corpus": ref_df,
    }
    compare_metrics = list(cfg.descriptor_weight_map.keys()) + ["novelty_score", "condition_fidelity_score", "supportive_model_score"]
    for group_name, dfi in groups.items():
        for metric in compare_metrics:
            if metric in dfi.columns:
                vals = pd.to_numeric(dfi[metric], errors="coerce").dropna()
                compare_rows.append({
                    "group": group_name,
                    "metric": metric,
                    "n": len(vals),
                    "mean": float(vals.mean()) if len(vals) else np.nan,
                    "median": float(vals.median()) if len(vals) else np.nan,
                    "q25": float(vals.quantile(0.25)) if len(vals) else np.nan,
                    "q75": float(vals.quantile(0.75)) if len(vals) else np.nan,
                })
    descriptor_compare_df = pd.DataFrame(compare_rows)
    descriptor_compare_df.to_csv(dirs["tables_supplementary"] / "table_s8_7_descriptor_group_comparison.csv", index=False)

    shortlist_main_cols = [
        id_col, seq_col, "shortlist_rank", "final_score", "final_rank",
        "novelty_score", "descriptor_plausibility_score", "condition_fidelity_score",
        "diversity_score", "supportive_model_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy", "similarity_cluster_id"
    ]
    shortlist_main_cols = [c for c in shortlist_main_cols if c in shortlist_df.columns]
    shortlist_table = shortlist_df[shortlist_main_cols].sort_values("shortlist_rank").reset_index(drop=True)

    final_panel_cols = [
        id_col, seq_col, "final_panel_rank", "final_score",
        "novelty_score", "descriptor_plausibility_score", "condition_fidelity_score",
        "diversity_score", "supportive_model_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy", "similarity_cluster_id"
    ]
    final_panel_cols = [c for c in final_panel_cols if c in final_panel_df.columns]
    final_panel_table = final_panel_df[final_panel_cols].sort_values("final_panel_rank").reset_index(drop=True)

    ranked_pool_cols = [
        id_col, seq_col, "final_rank", "final_score",
        "novelty_score", "descriptor_plausibility_score", "condition_fidelity_score",
        "diversity_score", "supportive_model_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy",
        "descriptor_plausibility_pass", "condition_fidelity_pass", "similarity_cluster_id"
    ]
    ranked_pool_cols = [c for c in ranked_pool_cols if c in annotated_ranked_df.columns]

    shortlist_table.to_csv(dirs["tables_main"] / "table_8_1_shortlist_main.csv", index=False)
    final_panel_table.to_csv(dirs["tables_main"] / "table_8_2_final_diverse_panel.csv", index=False)
    annotated_ranked_df[ranked_pool_cols].sort_values("final_score", ascending=False).to_csv(
        dirs["tables_supplementary"] / "table_s8_8_full_ranked_passed_pool.csv", index=False
    )

    summary_lines = []
    summary_lines.append("PepCVAE Step 8 prioritization summary")
    summary_lines.append("=" * 80)
    summary_lines.append(f"Timestamp UTC: {now_utc_iso()}")
    summary_lines.append(f"Step 1 reference: {loaded['step1_ref_path']}")
    summary_lines.append(f"Step 7 candidate input: {loaded['step7_candidate_path']}")
    summary_lines.append("")
    summary_lines.append("Resolved column mapping")
    for k, v in gen_colmap.items():
        summary_lines.append(f"- {k}: {v}")
    summary_lines.append("")
    summary_lines.append("Novelty audit note")
    if gen_colmap.get("exact_train_match_was_derived", False):
        summary_lines.append("- exact_train_match_col was not present in Step 7 and was derived from train_nn_similarity == 1.0")
    else:
        summary_lines.append("- exact_train_match_col was loaded directly from Step 7")
    summary_lines.append("")
    summary_lines.append("Counts")
    summary_lines.append(f"- Raw Step 7 candidates: {len(gen_df):,}")
    summary_lines.append(f"- Eligible after hard filters: {len(eligible):,}")
    summary_lines.append(f"- Passed plausibility + condition screening: {len(passed):,}")
    summary_lines.append(f"- Top shortlist: {len(shortlist_df):,}")
    summary_lines.append(f"- Final diverse panel: {len(final_panel_df):,}")
    summary_lines.append("")
    summary_lines.append("Primary ranking weights")
    for k, v in cfg.ranking_weights.items():
        summary_lines.append(f"  - {k}: {v}")
    summary_lines.append("")
    summary_lines.append("Supportive predictor provenance")
    for _, r in supportive_provenance_df.iterrows():
        summary_lines.append(f"  - {r['role']}: {r['status']} ({r['column_used']})")

    save_text("\n".join(summary_lines), dirs["reports"] / "step8_summary_report.txt")

    return {
        "ref_df": ref_df,
        "gen_df": gen_df,
        "eligible_df": eligible,
        "passed_df": passed,
        "shortlist_df": shortlist_df,
        "final_panel_df": final_panel_df,
        "shortlist_table": shortlist_table,
        "final_panel_table": final_panel_table,
        "filter_audit_df": filter_audit_df,
        "reference_bounds_df": reference_bounds_df,
        "stability_df": stability_df,
        "recurrence_df": recurrence_df,
        "descriptor_compare_df": descriptor_compare_df,
        "supportive_provenance_df": supportive_provenance_df,
        "annotated_ranked_df": annotated_ranked_df,
        "gen_colmap": gen_colmap,
    }


# =============================================================================
# Figure helpers
# =============================================================================

def style_axis(ax):
    ax.set_facecolor(PROJECT_COLORS["bg"])
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PROJECT_COLORS["spine"])
        ax.spines[side].set_linewidth(0.8)
    ax.tick_params(colors=PROJECT_COLORS["text"])
    ax.xaxis.label.set_color(PROJECT_COLORS["text"])
    ax.yaxis.label.set_color(PROJECT_COLORS["text"])
    ax.title.set_color(PROJECT_COLORS["text"])
    ax.grid(True, color=PROJECT_COLORS["grid"], linewidth=0.6, alpha=1.0)


# =============================================================================
# Figures
# =============================================================================

def plot_step8_main_figure(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    gen_df = results["gen_df"]
    eligible_df = results["eligible_df"]
    id_col = results["gen_colmap"]["id_col"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 8.6))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.1], width_ratios=[1.0, 1.0])

    ax = fig.add_subplot(gs[0, 0])
    style_axis(ax)
    funnel_labels = ["Step 7 pool", "Hard-filter eligible", "Screen passed", "Top shortlist", "Final panel"]
    funnel_vals = [len(gen_df), len(eligible_df), len(passed_df), len(shortlist_df), len(final_panel_df)]
    bar_colors = [
        PROJECT_COLORS["deep_crimson"],
        PROJECT_COLORS["warm_gold"],
        PROJECT_COLORS["teal"],
        PROJECT_COLORS["navy"],
        PROJECT_COLORS["deep_red_maroon"],
    ]
    ax.bar(range(len(funnel_vals)), funnel_vals, color=bar_colors, edgecolor=PROJECT_COLORS["spine"], linewidth=0.6)
    ax.set_xticks(range(len(funnel_vals)))
    ax.set_xticklabels(funnel_labels, rotation=22, ha="right")
    ax.set_ylabel("Number of peptides")
    ax.set_title("a  Step 8 prioritization funnel", loc="left", fontsize=cfg.figure_export.panel_fontsize)
    for i, v in enumerate(funnel_vals):
        frac = 100.0 * v / funnel_vals[0] if funnel_vals[0] else 0.0
        ax.text(i, v, f"{v}\n({frac:.1f}%)", ha="center", va="bottom", fontsize=7, color=PROJECT_COLORS["text"])

    ax = fig.add_subplot(gs[0, 1])
    style_axis(ax)
    plot_df = passed_df.copy()
    plot_df["is_shortlist"] = plot_df[id_col].astype(str).isin(shortlist_df[id_col].astype(str)).astype(int)
    ax.scatter(
        plot_df.loc[plot_df["is_shortlist"] == 0, "novelty_score"],
        plot_df.loc[plot_df["is_shortlist"] == 0, "descriptor_plausibility_score"],
        s=20, alpha=0.35, label="Passed pool", color=PROJECT_COLORS["neutral_gray"],
        edgecolors="none"
    )
    ax.scatter(
        plot_df.loc[plot_df["is_shortlist"] == 1, "novelty_score"],
        plot_df.loc[plot_df["is_shortlist"] == 1, "descriptor_plausibility_score"],
        s=38, alpha=0.90, label="Shortlist", color=PROJECT_COLORS["deep_crimson"],
        edgecolors="white", linewidths=0.3
    )
    ax.set_xlabel("Novelty score")
    ax.set_ylabel("ACP-plausibility score")
    ax.set_title("b  Novelty–plausibility landscape", loc="left", fontsize=cfg.figure_export.panel_fontsize)
    ax.legend(frameon=False, fontsize=cfg.figure_export.legend_fontsize)

    ax = fig.add_subplot(gs[1, 0])
    ax.set_facecolor(PROJECT_COLORS["bg"])
    heat_cols = [
        "novelty_score", "descriptor_plausibility_score",
        "condition_fidelity_score", "diversity_score",
        "supportive_model_score", "final_score"
    ]
    heat_cols = [c for c in heat_cols if c in shortlist_df.columns]
    heat_df = shortlist_df.sort_values("shortlist_rank").head(min(12, len(shortlist_df)))[[id_col] + heat_cols].copy()
    mat = heat_df[heat_cols].values
    im = ax.imshow(mat, aspect="auto", vmin=0, vmax=1, cmap="Reds")
    ax.set_xticks(range(len(heat_cols)))
    ax.set_xticklabels(heat_cols, rotation=30, ha="right")
    ax.set_yticks(range(len(heat_df)))
    ax.set_yticklabels(heat_df[id_col].astype(str).tolist())
    ax.set_title("c  Top candidates across ranking axes", loc="left", fontsize=cfg.figure_export.panel_fontsize, color=PROJECT_COLORS["text"])
    for spine in ax.spines.values():
        spine.set_color(PROJECT_COLORS["spine"])
        spine.set_linewidth(0.8)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Scaled score", color=PROJECT_COLORS["text"])
    cbar.outline.set_edgecolor(PROJECT_COLORS["spine"])
    cbar.ax.yaxis.set_tick_params(color=PROJECT_COLORS["text"])
    plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color=PROJECT_COLORS["text"])

    ax = fig.add_subplot(gs[1, 1])
    style_axis(ax)
    ax.scatter(
        final_panel_df["condition_fidelity_score"],
        final_panel_df["diversity_score"],
        s=55, alpha=0.9, color=PROJECT_COLORS["navy"],
        edgecolors="white", linewidths=0.4
    )
    for _, r in final_panel_df.iterrows():
        ax.text(r["condition_fidelity_score"], r["diversity_score"], str(r[id_col]), fontsize=6, color=PROJECT_COLORS["text"])
    ax.set_xlabel("Condition fidelity score")
    ax.set_ylabel("Diversity score")
    ax.set_title("d  Final diverse panel", loc="left", fontsize=cfg.figure_export.panel_fontsize)

    fig.tight_layout()
    export_figure(fig, dirs["figures_main"] / "figure_8_prioritization_main", cfg.figure_export)


def plot_step8_supplementary(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    ref_df = results["ref_df"]
    stability_df = results["stability_df"]
    id_col = results["gen_colmap"]["id_col"]
    seq_col = results["gen_colmap"]["sequence_col"]
    cond_cols = [
        results["gen_colmap"]["requested_length_bin_col"],
        results["gen_colmap"]["requested_charge_bin_col"],
        results["gen_colmap"]["requested_hydrophobicity_bin_col"],
    ]

    compare_cols = ["length", "net_charge_proxy", "mean_hydropathy", "entropy"]
    hist_colors = [
        PROJECT_COLORS["teal"],
        PROJECT_COLORS["deep_crimson"],
        PROJECT_COLORS["warm_gold"],
    ]
    for col in compare_cols:
        fig, ax = plt.subplots(figsize=(6.2, 4.5))
        style_axis(ax)
        ax.hist(passed_df[col].dropna(), bins=25, alpha=0.45, density=True, label="Passed pool", color=hist_colors[0])
        ax.hist(shortlist_df[col].dropna(), bins=25, alpha=0.55, density=True, label="Shortlist", color=hist_colors[1])
        ax.hist(ref_df[col].dropna(), bins=25, alpha=0.35, density=True, label="Reference ACP corpus", color=hist_colors[2])
        ax.set_xlabel(col)
        ax.set_ylabel("Density")
        ax.set_title(f"Descriptor comparison: {col}")
        ax.legend(frameon=False, fontsize=7)
        fig.tight_layout()
        export_figure(fig, dirs["figures_supplementary"] / f"sfigure_8_1_descriptor_{col}", cfg.figure_export)

    if len(stability_df) > 0:
        fig, ax = plt.subplots(figsize=(7.0, 4.5))
        style_axis(ax)
        labels = stability_df["scheme_a"] + " vs " + stability_df["scheme_b"]
        vals = stability_df["jaccard_overlap"]
        ax.bar(range(len(vals)), vals, color=PROJECT_COLORS["navy"], edgecolor=PROJECT_COLORS["spine"], linewidth=0.6)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_ylabel("Top-N Jaccard overlap")
        ax.set_title("Shortlist stability across ranking schemes")
        fig.tight_layout()
        export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_2_scheme_stability", cfg.figure_export)

    if len(final_panel_df) >= 2:
        seqs = final_panel_df[seq_col].tolist()
        sim_mat = np.eye(len(seqs))
        for i in range(len(seqs)):
            for j in range(i + 1, len(seqs)):
                sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=cfg.similarity_kmer)
                sim_mat[i, j] = sim
                sim_mat[j, i] = sim
        fig, ax = plt.subplots(figsize=(6.2, 5.2))
        ax.set_facecolor(PROJECT_COLORS["bg"])
        im = ax.imshow(sim_mat, aspect="auto", vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(final_panel_df)))
        ax.set_yticks(range(len(final_panel_df)))
        ax.set_xticklabels(final_panel_df[id_col].astype(str), rotation=45, ha="right")
        ax.set_yticklabels(final_panel_df[id_col].astype(str))
        ax.set_title("Pairwise 3-mer Jaccard similarity in final panel", color=PROJECT_COLORS["text"])
        for spine in ax.spines.values():
            spine.set_color(PROJECT_COLORS["spine"])
            spine.set_linewidth(0.8)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Similarity", color=PROJECT_COLORS["text"])
        cbar.outline.set_edgecolor(PROJECT_COLORS["spine"])
        fig.tight_layout()
        export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_3_final_panel_similarity", cfg.figure_export)

    cond_bar_colors = [PROJECT_COLORS["teal"], PROJECT_COLORS["deep_red_maroon"]]
    for cond_col in cond_cols:
        if cond_col in passed_df.columns:
            prior_counts = passed_df[cond_col].astype(str).value_counts(dropna=False)
            short_counts = shortlist_df[cond_col].astype(str).value_counts(dropna=False)
            merged = pd.DataFrame({"passed_pool": prior_counts, "shortlist": short_counts}).fillna(0)

            fig, ax = plt.subplots(figsize=(7.0, 4.5))
            style_axis(ax)
            x = np.arange(len(merged))
            w = 0.4
            ax.bar(x - w/2, merged["passed_pool"], width=w, label="Passed pool", color=cond_bar_colors[0])
            ax.bar(x + w/2, merged["shortlist"], width=w, label="Shortlist", color=cond_bar_colors[1])
            ax.set_xticks(x)
            ax.set_xticklabels(merged.index.astype(str), rotation=25, ha="right")
            ax.set_ylabel("Count")
            ax.set_title(f"Condition retention: {cond_col}")
            ax.legend(frameon=False, fontsize=7)
            fig.tight_layout()
            export_figure(fig, dirs["figures_supplementary"] / f"sfigure_8_4_condition_retention_{cond_col}", cfg.figure_export)


# =============================================================================
# Notebook entrypoint
# =============================================================================

def main_notebook(cfg: Optional[Step8Config] = None) -> Dict[str, Any]:
    if cfg is None:
        cfg = Step8Config()

    seed_all(cfg.main_random_seed)

    step8_root = Path(cfg.project_root).resolve() / cfg.step8_dir
    dirs = make_output_dirs(step8_root)

    print_header("Running PepCVAE Step 8")
    print(json.dumps({
        "project_root": cfg.project_root,
        "step1_dir": cfg.step1_dir,
        "step7_dir": cfg.step7_dir,
        "step8_dir": cfg.step8_dir,
        "step1_reference_file": cfg.step1_reference_file,
        "step7_candidate_file": cfg.step7_candidate_file,
        "shortlist_top_n": cfg.shortlist_top_n,
        "final_diverse_panel_n": cfg.final_diverse_panel_n,
    }, indent=2))

    results = run_step8(cfg, dirs)
    plot_step8_main_figure(results, cfg, dirs)
    plot_step8_supplementary(results, cfg, dirs)

    save_json(
        {
            "n_reference": int(len(results["ref_df"])),
            "n_step7_raw": int(len(results["gen_df"])),
            "n_eligible": int(len(results["eligible_df"])),
            "n_passed": int(len(results["passed_df"])),
            "n_shortlist": int(len(results["shortlist_df"])),
            "n_final_panel": int(len(results["final_panel_df"])),
        },
        dirs["logs"] / "step8_counts.json",
    )

    print("\n" + "=" * 100)
    print("Step 8 completed successfully.")
    print(f"Main tables: {dirs['tables_main']}")
    print(f"Supplementary tables: {dirs['tables_supplementary']}")
    print(f"Main figures: {dirs['figures_main']}")
    print(f"Supplementary figures: {dirs['figures_supplementary']}")
    print(f"Artifacts: {dirs['artifacts']}")
    print(f"Reports: {dirs['reports']}")
    print(f"Logs: {dirs['logs']}")
    print("=" * 100)

    return results


# =============================================================================
# Execute
# =============================================================================

results = main_notebook()

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/home/data3/Moe/nature_computational2/step7_v3")
csvs = sorted(root.rglob("*.csv"))

for i, p in enumerate(csvs):
    try:
        df = pd.read_csv(p, nrows=5)
        cols = list(df.columns)
        print(f"\n[{i}] {p}")
        print("ncols:", len(cols))
        print("columns:", cols[:20])
        print("shape preview:", df.shape)
    except Exception as e:
        print(f"\n[{i}] {p}")
        print("read error:", e)

In [ ]:

from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE
# Multi-objective prioritization and biological plausibility screening (v1)
# Full improved version with scientific cleanup and redesigned figures
# =============================================================================

import json
import platform
import random
import hashlib
import warnings
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from itertools import combinations, groupby
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

# =============================================================================
# Global constants
# =============================================================================

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)

AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_TINY = set("AGSC")
AA_SMALL = set("AGSCVTNDP")

AA_PROPERTIES = {
    "A": {"hydro": 1.8, "mw": 89.09},
    "C": {"hydro": 2.5, "mw": 121.16},
    "D": {"hydro": -3.5, "mw": 133.10},
    "E": {"hydro": -3.5, "mw": 147.13},
    "F": {"hydro": 2.8, "mw": 165.19},
    "G": {"hydro": -0.4, "mw": 75.07},
    "H": {"hydro": -3.2, "mw": 155.16},
    "I": {"hydro": 4.5, "mw": 131.17},
    "K": {"hydro": -3.9, "mw": 146.19},
    "L": {"hydro": 3.8, "mw": 131.17},
    "M": {"hydro": 1.9, "mw": 149.21},
    "N": {"hydro": -3.5, "mw": 132.12},
    "P": {"hydro": -1.6, "mw": 115.13},
    "Q": {"hydro": -3.5, "mw": 146.15},
    "R": {"hydro": -4.5, "mw": 174.20},
    "S": {"hydro": -0.8, "mw": 105.09},
    "T": {"hydro": -0.7, "mw": 119.12},
    "V": {"hydro": 4.2, "mw": 117.15},
    "W": {"hydro": -0.9, "mw": 204.23},
    "Y": {"hydro": -1.3, "mw": 181.19},
}

# reference-inspired palette
PROJECT_COLORS = {
    "hydramp_red": "#C7001E",
    "deep_red": "#E31A1C",
    "dark_maroon": "#7F0000",
    "pepcvae_navy": "#1F3B68",
    "basic_blue": "#A8B9DC",
    "amp_green": "#57C78D",
    "dean_pink": "#D96A99",
    "muller_cyan": "#71CDD7",
    "gan_orange": "#F29D3E",
    "joker_teal": "#0E6766",
    "purple": "#6A2C91",
    "gold": "#F0C43C",
    "gray_light": "#D9D9D9",
    "gray_mid": "#A0A0A0",
    "gray_dark": "#4A4A4A",
    "bg": "#FFFFFF",
    "grid": "#E8E8E8",
    "spine": "#BFBFBF",
    "text": "#222222",
}

GROUP_COLORS = {
    "Raw pool": PROJECT_COLORS["gray_mid"],
    "Eligible": PROJECT_COLORS["gan_orange"],
    "Passed": PROJECT_COLORS["amp_green"],
    "Shortlist": PROJECT_COLORS["hydramp_red"],
    "Final panel": PROJECT_COLORS["pepcvae_navy"],
    "Reference ACP corpus": PROJECT_COLORS["basic_blue"],
}

# =============================================================================
# Matplotlib style
# =============================================================================

def apply_project_matplotlib_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PROJECT_COLORS["bg"],
        "axes.facecolor": PROJECT_COLORS["bg"],
        "savefig.facecolor": PROJECT_COLORS["bg"],
        "axes.edgecolor": PROJECT_COLORS["spine"],
        "axes.labelcolor": PROJECT_COLORS["text"],
        "xtick.color": PROJECT_COLORS["text"],
        "ytick.color": PROJECT_COLORS["text"],
        "text.color": PROJECT_COLORS["text"],
        "axes.titlecolor": PROJECT_COLORS["text"],
        "grid.color": PROJECT_COLORS["grid"],
        "axes.grid": True,
        "grid.linestyle": "-",
        "grid.linewidth": 0.55,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 8,
        "font.family": "DejaVu Sans",
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    })

apply_project_matplotlib_style()

# =============================================================================
# Configuration
# =============================================================================

@dataclass
class FigureExportConfig:
    png_dpi: int = 300
    tif_dpi: int = 600
    export_pdf: bool = True
    export_png: bool = True
    export_tif: bool = True
    single_col_width_in: float = 3.5
    double_col_width_in: float = 7.1
    panel_fontsize: int = 10
    base_fontsize: int = 8
    legend_fontsize: int = 7

@dataclass
class Step8Config:
    project_root: str = "/home/data3/Moe/nature_computational2/"
    step1_dir: str = "step1"
    step7_dir: str = "step7_v3"
    step8_dir: str = "step8_v1"

    step1_reference_file: Optional[str] = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"
    step7_candidate_file: Optional[str] = None
    allow_relaxed_step7_input: bool = True

    step7_exclude_path_keywords: Tuple[str, ...] = (
        "train_reference_stats", "descriptor_bounds", "summary_report", "counts.json"
    )
    step7_candidate_aliases: Tuple[str, ...] = (
        "step7_candidates_for_step8.csv",
        "step7_generated_final_audited.csv",
        "step7_generated_condition_consistency.csv",
        "step7_generated_similarity_audit.csv",
        "step7_generated_qc_filtered.csv",
        "step7_generated_raw_sequences.csv",
    )

    sequence_col_candidates: Tuple[str, ...] = ("sequence", "peptide", "seq", "peptide_sequence")
    id_col_candidates: Tuple[str, ...] = ("generated_id", "candidate_id", "peptide_id", "id")
    merge_key_col_candidates: Tuple[str, ...] = ("sequence_sha256", "seq_sha256")

    exact_train_match_col_candidates: Tuple[str, ...] = (
        "is_exact_train_match", "train_exact_match", "exact_match_train", "in_train_reference"
    )
    train_nn_similarity_col_candidates: Tuple[str, ...] = (
        "train_nn_similarity", "nearest_train_similarity", "max_train_similarity",
        "train_jaccard_nn", "nn_similarity_train"
    )
    exact_full_match_col_candidates: Tuple[str, ...] = (
        "is_exact_full_match", "full_exact_match", "exact_match_full", "in_full_reference"
    )

    requested_length_bin_col_candidates: Tuple[str, ...] = (
        "requested_length_bin", "target_length_bin", "condition_length_bin", "length_bin"
    )
    requested_charge_bin_col_candidates: Tuple[str, ...] = (
        "requested_charge_bin", "target_charge_bin", "condition_charge_bin", "net_charge_bin"
    )
    requested_hydrophobicity_bin_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_bin", "target_hydrophobicity_bin",
        "condition_hydrophobicity_bin", "hydrophobicity_bin"
    )

    requested_length_min_col_candidates: Tuple[str, ...] = ("requested_length_min", "target_length_min")
    requested_length_max_col_candidates: Tuple[str, ...] = ("requested_length_max", "target_length_max")
    requested_charge_min_col_candidates: Tuple[str, ...] = ("requested_charge_min", "target_charge_min")
    requested_charge_max_col_candidates: Tuple[str, ...] = ("requested_charge_max", "target_charge_max")
    requested_hydrophobicity_min_col_candidates: Tuple[str, ...] = ("requested_hydrophobicity_min", "target_hydrophobicity_min")
    requested_hydrophobicity_max_col_candidates: Tuple[str, ...] = ("requested_hydrophobicity_max", "target_hydrophobicity_max")

    supportive_score_candidates: Tuple[str, ...] = (
        "acp_probability", "acp_score", "predicted_acp_probability",
        "prob_acp", "amp_probability", "supportive_model_score"
    )
    toxicity_score_candidates: Tuple[str, ...] = ("toxicity_probability", "toxicity_score", "predicted_toxicity")
    hemolysis_score_candidates: Tuple[str, ...] = ("hemolysis_probability", "hemolysis_score", "predicted_hemolysis")

    main_random_seed: int = 42
    bootstrap_iterations: int = 300

    min_length: int = 5
    max_length: int = 50
    low_complexity_entropy_threshold: float = 1.50
    max_homopolymer_run_fraction: float = 0.35
    max_single_residue_fraction: float = 0.60
    min_unique_residue_fraction: float = 0.25
    max_train_nn_similarity: float = 0.85
    require_not_in_train_exact: bool = True
    require_not_in_full_exact_if_available: bool = False

    similarity_kmer: int = 3
    shortlist_top_n: int = 24
    final_diverse_panel_n: int = 12
    similarity_threshold_for_clusters: float = 0.75
    max_per_similarity_cluster: int = 2

    reference_quantile_low: float = 0.02
    reference_quantile_high: float = 0.98

    descriptor_weight_map: Dict[str, float] = field(default_factory=lambda: {
        "length": 1.00,
        "mean_hydropathy": 1.00,
        "net_charge_proxy": 1.20,
        "hydrophobic_frac": 0.90,
        "positive_frac": 1.10,
        "negative_frac": 0.80,
        "aromatic_frac": 0.70,
        "entropy": 1.00,
        "max_run_fraction": 1.00,
    })

    ranking_weights: Dict[str, float] = field(default_factory=lambda: {
        "novelty_score": 0.35,
        "descriptor_plausibility_score": 0.35,
        "diversity_score": 0.30,
    })

    ranking_schemes: Dict[str, Dict[str, float]] = field(default_factory=lambda: {
        "primary": {
            "novelty_score": 0.35, "descriptor_plausibility_score": 0.35, "diversity_score": 0.30,
        },
        "novelty_heavy": {
            "novelty_score": 0.50, "descriptor_plausibility_score": 0.25, "diversity_score": 0.25,
        },
        "plausibility_heavy": {
            "novelty_score": 0.25, "descriptor_plausibility_score": 0.50, "diversity_score": 0.25,
        },
        "diversity_heavy": {
            "novelty_score": 0.25, "descriptor_plausibility_score": 0.25, "diversity_score": 0.50,
        },
    })

    component_variance_tolerance: float = 1e-6
    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)


# =============================================================================
# Reproducibility and IO
# =============================================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def collect_environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp_utc": now_utc_iso(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
        "cwd": str(Path.cwd()),
    }

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.export_pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.export_tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)

def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


# =============================================================================
# Discovery helpers
# =============================================================================

def first_existing_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = False, label: str = "") -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not resolve required column for {label}. Tried: {list(candidates)}")
    return None

def inspect_step7_candidate_inventory(step7_root: str, cfg: Optional[Step8Config] = None) -> pd.DataFrame:
    if cfg is None:
        cfg = Step8Config()
    root = Path(step7_root).resolve()
    rows = []
    for p in sorted(root.rglob("*.csv")):
        try:
            preview = pd.read_csv(p, nrows=50)
            seq_col = first_existing_column(preview, cfg.sequence_col_candidates, required=False)
            train_nn_col = first_existing_column(preview, cfg.train_nn_similarity_col_candidates, required=False)
            len_bin = first_existing_column(preview, cfg.requested_length_bin_col_candidates, required=False)
            chg_bin = first_existing_column(preview, cfg.requested_charge_bin_col_candidates, required=False)
            hyd_bin = first_existing_column(preview, cfg.requested_hydrophobicity_bin_col_candidates, required=False)
            rows.append({
                "path": str(p),
                "candidate_like": int(seq_col is not None and train_nn_col is not None),
                "strict_candidate_like": int(seq_col is not None and train_nn_col is not None and len_bin is not None and chg_bin is not None and hyd_bin is not None),
                "ncols": len(preview.columns),
                "seq_col": seq_col,
                "train_nn_col": train_nn_col,
                "length_bin_col": len_bin,
                "charge_bin_col": chg_bin,
                "hydro_bin_col": hyd_bin,
            })
        except Exception as e:
            rows.append({
                "path": str(p),
                "candidate_like": 0,
                "strict_candidate_like": 0,
                "ncols": np.nan,
                "seq_col": None,
                "train_nn_col": None,
                "length_bin_col": None,
                "charge_bin_col": None,
                "hydro_bin_col": None,
                "error": str(e),
            })
    inventory = pd.DataFrame(rows)
    with pd.option_context("display.max_rows", 200, "display.max_colwidth", 160):
        print(inventory.to_string(index=False))
    return inventory

def resolve_exact_train_match_column(
    df: pd.DataFrame,
    exact_candidates: Sequence[str],
    nn_candidates: Sequence[str],
    derived_col_name: str = "_derived_exact_train_match_from_nn",
    atol: float = 1e-12,
) -> Tuple[str, bool]:
    exact_col = first_existing_column(df, exact_candidates, required=False)
    if exact_col is not None:
        vals = pd.to_numeric(df[exact_col], errors="coerce")
        if vals.isna().any():
            raise ValueError(f"{exact_col} contains missing/non-numeric values.")
        df[exact_col] = vals.astype(int)
        return exact_col, False

    nn_col = first_existing_column(df, nn_candidates, required=False)
    if nn_col is None:
        raise ValueError("Could not resolve exact train-match information.")
    nn_vals = pd.to_numeric(df[nn_col], errors="coerce")
    if nn_vals.isna().any():
        raise ValueError(f"{nn_col} contains missing/non-numeric values.")
    df[derived_col_name] = np.isclose(nn_vals, 1.0, atol=atol).astype(int)
    return derived_col_name, True

def resolve_exact_full_match_column(df: pd.DataFrame, exact_full_candidates: Sequence[str]) -> Optional[str]:
    exact_full_col = first_existing_column(df, exact_full_candidates, required=False)
    if exact_full_col is None:
        return None
    vals = pd.to_numeric(df[exact_full_col], errors="coerce")
    if vals.isna().any():
        raise ValueError(f"{exact_full_col} contains missing/non-numeric values.")
    df[exact_full_col] = vals.astype(int)
    return exact_full_col

def find_candidate_level_step7_csv(root_dir: Path, cfg: Step8Config) -> Path:
    if cfg.step7_candidate_file is not None:
        p = Path(cfg.step7_candidate_file).resolve()
        if not p.exists():
            raise FileNotFoundError(f"Explicit step7_candidate_file does not exist: {p}")
        return p

    all_csvs = sorted(root_dir.rglob("*.csv"))
    if not all_csvs:
        raise FileNotFoundError(f"No CSV files found under {root_dir}")

    def score_candidate(path: Path, preview: pd.DataFrame) -> tuple:
        seq_col = first_existing_column(preview, cfg.sequence_col_candidates, required=False)
        nn_col = first_existing_column(preview, cfg.train_nn_similarity_col_candidates, required=False)
        len_bin = first_existing_column(preview, cfg.requested_length_bin_col_candidates, required=False)
        chg_bin = first_existing_column(preview, cfg.requested_charge_bin_col_candidates, required=False)
        hyd_bin = first_existing_column(preview, cfg.requested_hydrophobicity_bin_col_candidates, required=False)
        recoverable = int(seq_col is not None and nn_col is not None)
        strict = int(recoverable and len_bin is not None and chg_bin is not None and hyd_bin is not None)

        s = str(path).lower().replace("\\", "/")
        alias_bonus = int(path.name in set(cfg.step7_candidate_aliases))
        tables_bonus = int("/outputs/tables/" in s)
        source_data_penalty = int("/outputs/source_data/" in s)
        requested_like_cols = sum(any(k in c.lower() for k in ["requested_", "target_", "condition_"]) for c in preview.columns)
        mtime = path.stat().st_mtime
        return (strict, recoverable, alias_bonus, tables_bonus, requested_like_cols, -source_data_penalty, mtime)

    scored = []
    for p in all_csvs:
        s = str(p).lower().replace("\\", "/")
        if any(k in s for k in cfg.step7_exclude_path_keywords):
            continue
        try:
            preview = pd.read_csv(p, nrows=50)
        except Exception:
            continue
        seq_col = first_existing_column(preview, cfg.sequence_col_candidates, required=False)
        nn_col = first_existing_column(preview, cfg.train_nn_similarity_col_candidates, required=False)
        if seq_col is None or nn_col is None:
            continue
        scored.append((score_candidate(p, preview), p))

    if not scored:
        inspect_step7_candidate_inventory(str(root_dir), cfg)
        raise FileNotFoundError("Could not find a recoverable Step 7 peptide-level input.")

    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    print("\nTop Step 7 candidate-file matches:")
    for sc, p in scored[:10]:
        print(f"  - {p} | score={sc}")

    best = scored[0][1]
    preview = pd.read_csv(best, nrows=50)
    has_all_bins = all(first_existing_column(preview, cands, required=False) is not None for cands in [
        cfg.requested_length_bin_col_candidates,
        cfg.requested_charge_bin_col_candidates,
        cfg.requested_hydrophobicity_bin_col_candidates,
    ])
    if not has_all_bins and cfg.allow_relaxed_step7_input:
        print("\nSelected a recoverable but relaxed Step 7 input.")
        print("Condition-fidelity scoring will switch to relaxed mode if requested bins are unavailable.")
    return best


# =============================================================================
# Sequence utilities and descriptors
# =============================================================================

def clean_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def is_valid_peptide(seq: str, min_len: int = 1, max_len: Optional[int] = None) -> bool:
    if not seq:
        return False
    if any(ch not in AA_SET for ch in seq):
        return False
    if len(seq) < min_len:
        return False
    if max_len is not None and len(seq) > max_len:
        return False
    return True

def shannon_entropy(seq: str) -> float:
    if not seq:
        return 0.0
    counts = Counter(seq)
    probs = np.array(list(counts.values()), dtype=float) / len(seq)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, None))).sum())

def max_run_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    runs = [len(list(group)) for _, group in groupby(seq)]
    return float(max(runs) / len(seq))

def aa_fraction(seq: str, aa: str) -> float:
    return seq.count(aa) / len(seq) if seq else 0.0

def unique_residue_fraction(seq: str) -> float:
    return len(set(seq)) / len(seq) if seq else 0.0

def max_single_residue_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    counts = Counter(seq)
    return float(max(counts.values()) / len(seq))

def make_kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}

def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
    s1 = make_kmers(seq1, k)
    s2 = make_kmers(seq2, k)
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    union = len(s1 | s2)
    return 0.0 if union == 0 else len(s1 & s2) / union

def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    if not seq:
        desc = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
        desc.update({
            "length": 0.0,
            "mean_hydropathy": 0.0,
            "hydrophobic_frac": 0.0,
            "aromatic_frac": 0.0,
            "positive_frac": 0.0,
            "negative_frac": 0.0,
            "polar_frac": 0.0,
            "tiny_frac": 0.0,
            "small_frac": 0.0,
            "net_charge_proxy": 0.0,
            "molecular_weight_proxy": 0.0,
            "entropy": 0.0,
            "max_run_fraction": 1.0,
            "unique_residue_fraction": 0.0,
            "max_single_residue_fraction": 1.0,
        })
        return desc

    hydros = np.array([AA_PROPERTIES[a]["hydro"] for a in seq], dtype=float)
    weights = np.array([AA_PROPERTIES[a]["mw"] for a in seq], dtype=float)
    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    desc.update({
        "length": float(len(seq)),
        "mean_hydropathy": float(hydros.mean()),
        "hydrophobic_frac": float(np.mean([a in AA_HYDROPHOBIC for a in seq])),
        "aromatic_frac": float(np.mean([a in AA_AROMATIC for a in seq])),
        "positive_frac": float(np.mean([a in AA_POSITIVE for a in seq])),
        "negative_frac": float(np.mean([a in AA_NEGATIVE for a in seq])),
        "polar_frac": float(np.mean([a in AA_POLAR for a in seq])),
        "tiny_frac": float(np.mean([a in AA_TINY for a in seq])),
        "small_frac": float(np.mean([a in AA_SMALL for a in seq])),
        "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
        "molecular_weight_proxy": float(weights.sum()),
        "entropy": shannon_entropy(seq),
        "max_run_fraction": max_run_fraction(seq),
        "unique_residue_fraction": unique_residue_fraction(seq),
        "max_single_residue_fraction": max_single_residue_fraction(seq),
    })
    return desc

def add_computable_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    base = df.reset_index(drop=True).copy()
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in base[seq_col].astype(str)])
    overlap = [c for c in desc_df.columns if c in base.columns and c != seq_col]
    if overlap:
        base = base.rename(columns={c: f"{c}_input" for c in overlap})
    return pd.concat([base, desc_df.reset_index(drop=True)], axis=1)


# =============================================================================
# Input loading and preparation
# =============================================================================

def load_step_inputs(cfg: Step8Config) -> Dict[str, Any]:
    project_root = Path(cfg.project_root).resolve()
    step1_root = project_root / cfg.step1_dir
    step7_root = project_root / cfg.step7_dir
    step8_root = project_root / cfg.step8_dir

    if not step1_root.exists():
        raise FileNotFoundError(f"Step 1 directory not found: {step1_root}")
    if not step7_root.exists():
        raise FileNotFoundError(f"Step 7 directory not found: {step7_root}")

    if cfg.step1_reference_file is not None:
        step1_ref_path = Path(cfg.step1_reference_file).resolve()
        if not step1_ref_path.exists():
            raise FileNotFoundError(f"Configured Step 1 reference file not found: {step1_ref_path}")
    else:
        step1_ref_path = (step1_root / "tables" / "step1_retained_dataset.csv").resolve()

    step7_candidate_path = find_candidate_level_step7_csv(step7_root, cfg)

    ref_df = pd.read_csv(step1_ref_path)
    gen_df = pd.read_csv(step7_candidate_path)

    print(f"Resolved Step 1 reference: {step1_ref_path}")
    print(f"Resolved Step 7 candidate file: {step7_candidate_path}")

    return {
        "project_root": project_root,
        "step1_ref_path": step1_ref_path,
        "step7_candidate_path": step7_candidate_path,
        "step1_ref_df": ref_df,
        "step7_generated_df": gen_df,
        "step8_root": step8_root,
    }

def prepare_reference_df(ref_df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df = ref_df.copy()
    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="Step 1 sequence")
    colmap = {"sequence_col": seq_col}
    df[seq_col] = df[seq_col].astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].copy()
    df = df.drop_duplicates(subset=[seq_col]).reset_index(drop=True)
    df = add_computable_descriptors(df, seq_col)

    merge_key_col = first_existing_column(df, cfg.merge_key_col_candidates, required=False)
    if merge_key_col is None:
        merge_key_col = "sequence_sha256"
        df[merge_key_col] = df[seq_col].map(lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())
    colmap["merge_key_col"] = merge_key_col
    return df, colmap

def prepare_generated_df(gen_df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    df = gen_df.copy()
    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="generated sequence")
    id_col = first_existing_column(df, cfg.id_col_candidates, required=False)
    if id_col is None:
        id_col = "generated_id"
        df[id_col] = [f"gen_{i:06d}" for i in range(len(df))]

    train_nn_similarity_col = first_existing_column(df, cfg.train_nn_similarity_col_candidates, required=True, label="train NN similarity")
    exact_train_match_col, exact_train_was_derived = resolve_exact_train_match_column(
        df, cfg.exact_train_match_col_candidates, cfg.train_nn_similarity_col_candidates
    )
    exact_full_match_col = resolve_exact_full_match_column(df, cfg.exact_full_match_col_candidates)

    requested_length_bin_col = first_existing_column(df, cfg.requested_length_bin_col_candidates, required=False)
    requested_charge_bin_col = first_existing_column(df, cfg.requested_charge_bin_col_candidates, required=False)
    requested_hydrophobicity_bin_col = first_existing_column(df, cfg.requested_hydrophobicity_bin_col_candidates, required=False)

    relaxed_mode = cfg.allow_relaxed_step7_input and not all([requested_length_bin_col, requested_charge_bin_col, requested_hydrophobicity_bin_col])
    if not relaxed_mode and not all([requested_length_bin_col, requested_charge_bin_col, requested_hydrophobicity_bin_col]):
        raise ValueError("Missing requested condition-bin metadata required for strict condition-fidelity screening.")

    colmap = {
        "sequence_col": seq_col,
        "id_col": id_col,
        "exact_train_match_col": exact_train_match_col,
        "train_nn_similarity_col": train_nn_similarity_col,
        "exact_full_match_col": exact_full_match_col,
        "requested_length_bin_col": requested_length_bin_col,
        "requested_charge_bin_col": requested_charge_bin_col,
        "requested_hydrophobicity_bin_col": requested_hydrophobicity_bin_col,
        "requested_length_min_col": first_existing_column(df, cfg.requested_length_min_col_candidates, required=False),
        "requested_length_max_col": first_existing_column(df, cfg.requested_length_max_col_candidates, required=False),
        "requested_charge_min_col": first_existing_column(df, cfg.requested_charge_min_col_candidates, required=False),
        "requested_charge_max_col": first_existing_column(df, cfg.requested_charge_max_col_candidates, required=False),
        "requested_hydrophobicity_min_col": first_existing_column(df, cfg.requested_hydrophobicity_min_col_candidates, required=False),
        "requested_hydrophobicity_max_col": first_existing_column(df, cfg.requested_hydrophobicity_max_col_candidates, required=False),
        "exact_train_match_was_derived": exact_train_was_derived,
        "relaxed_condition_mode": relaxed_mode,
    }

    merge_key_col = first_existing_column(df, cfg.merge_key_col_candidates, required=False)
    if merge_key_col is None:
        merge_key_col = "sequence_sha256"
        df[merge_key_col] = df[seq_col].map(lambda s: hashlib.sha256(clean_sequence(s).encode("utf-8")).hexdigest())
    colmap["merge_key_col"] = merge_key_col

    df[seq_col] = df[seq_col].astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].copy()
    if df[id_col].duplicated().any():
        df[id_col] = [f"gen_{i:06d}" for i in range(len(df))]

    df = add_computable_descriptors(df, seq_col)
    nn_vals = pd.to_numeric(df[train_nn_similarity_col], errors="coerce")
    if nn_vals.isna().any() or ((nn_vals < 0) | (nn_vals > 1)).any():
        raise ValueError(f"{train_nn_similarity_col} must be numeric and lie within [0, 1].")
    return df.reset_index(drop=True), colmap


# =============================================================================
# Supportive predictor handling
# =============================================================================

def build_supportive_predictor_frame(df: pd.DataFrame, cfg: Step8Config) -> Tuple[pd.Series, pd.DataFrame]:
    support_col = first_existing_column(df, cfg.supportive_score_candidates, required=False)
    tox_col = first_existing_column(df, cfg.toxicity_score_candidates, required=False)
    hemo_col = first_existing_column(df, cfg.hemolysis_score_candidates, required=False)

    provenance_rows = [
        {"role": "supportive_acp_score", "column_used": support_col, "status": "used" if support_col else "not_available", "notes": "Supportive only; not direct proof."},
        {"role": "toxicity_penalty", "column_used": tox_col, "status": "used" if tox_col else "not_available", "notes": "Penalty only."},
        {"role": "hemolysis_penalty", "column_used": hemo_col, "status": "used" if hemo_col else "not_available", "notes": "Penalty only."},
    ]

    if support_col is None:
        supportive = pd.Series(np.full(len(df), 0.5), index=df.index, dtype=float)
    else:
        supportive = pd.to_numeric(df[support_col], errors="coerce").clip(0.0, 1.0)

    tox_pen = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    if tox_col is not None:
        tox_pen = pd.to_numeric(df[tox_col], errors="coerce").clip(0.0, 1.0)

    hemo_pen = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    if hemo_col is not None:
        hemo_pen = pd.to_numeric(df[hemo_col], errors="coerce").clip(0.0, 1.0)

    supportive_score = (supportive - 0.5 * tox_pen - 0.5 * hemo_pen).clip(0.0, 1.0)
    return supportive_score, pd.DataFrame(provenance_rows)


# =============================================================================
# Hard filtering
# =============================================================================

def build_hard_filter_flags(df: pd.DataFrame, cfg: Step8Config, colmap: Dict[str, str]) -> pd.DataFrame:
    seq_col = colmap["sequence_col"]
    exact_train_col = colmap["exact_train_match_col"]
    train_nn_col = colmap["train_nn_similarity_col"]
    exact_full_col = colmap["exact_full_match_col"]

    out = pd.DataFrame(index=df.index)
    out["flag_invalid_sequence"] = (~df[seq_col].map(lambda s: is_valid_peptide(s, cfg.min_length, cfg.max_length))).astype(int)
    out["flag_too_short"] = (df["length"] < cfg.min_length).astype(int)
    out["flag_too_long"] = (df["length"] > cfg.max_length).astype(int)
    out["flag_low_entropy"] = (df["entropy"] < cfg.low_complexity_entropy_threshold).astype(int)
    out["flag_high_run_fraction"] = (df["max_run_fraction"] > cfg.max_homopolymer_run_fraction).astype(int)
    out["flag_high_single_residue_fraction"] = (df["max_single_residue_fraction"] > cfg.max_single_residue_fraction).astype(int)
    out["flag_low_unique_residue_fraction"] = (df["unique_residue_fraction"] < cfg.min_unique_residue_fraction).astype(int)
    out["flag_duplicate_sequence"] = df.duplicated(subset=[seq_col], keep="first").astype(int)
    out["flag_exact_train_match"] = pd.to_numeric(df[exact_train_col], errors="coerce").fillna(0).astype(int)

    if exact_full_col is not None and cfg.require_not_in_full_exact_if_available:
        out["flag_exact_full_match"] = pd.to_numeric(df[exact_full_col], errors="coerce").fillna(0).astype(int)
    else:
        out["flag_exact_full_match"] = 0

    out["flag_high_train_similarity"] = (pd.to_numeric(df[train_nn_col], errors="coerce") > cfg.max_train_nn_similarity).astype(int)

    active_flags = [
        "flag_invalid_sequence", "flag_too_short", "flag_too_long", "flag_low_entropy",
        "flag_high_run_fraction", "flag_high_single_residue_fraction", "flag_low_unique_residue_fraction",
        "flag_duplicate_sequence", "flag_high_train_similarity"
    ]
    if cfg.require_not_in_train_exact:
        active_flags.append("flag_exact_train_match")
    if cfg.require_not_in_full_exact_if_available and exact_full_col is not None:
        active_flags.append("flag_exact_full_match")

    out["hard_filter_fail"] = (out[active_flags].sum(axis=1) > 0).astype(int)
    return out

def summarize_filter_waterfall(flags_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(flags_df)
    for c in [x for x in flags_df.columns if x.startswith("flag_")]:
        removed = int(flags_df[c].sum())
        rows.append({
            "filter_name": c,
            "removed_n": removed,
            "removed_fraction": removed / n if n else np.nan,
            "retained_if_applied_alone": n - removed,
        })
    return pd.DataFrame(rows).sort_values("removed_n", ascending=False).reset_index(drop=True)


# =============================================================================
# ACP plausibility support
# =============================================================================

def build_reference_bounds(ref_df: pd.DataFrame, cfg: Step8Config) -> pd.DataFrame:
    rows = []
    for metric, weight in cfg.descriptor_weight_map.items():
        vals = pd.to_numeric(ref_df[metric], errors="coerce").dropna()
        if len(vals) < 20:
            raise ValueError(f"Reference corpus has insufficient support for descriptor '{metric}'.")
        rows.append({
            "descriptor": metric,
            "q_low": float(vals.quantile(cfg.reference_quantile_low)),
            "q_high": float(vals.quantile(cfg.reference_quantile_high)),
            "median": float(vals.median()),
            "iqr": float(vals.quantile(0.75) - vals.quantile(0.25)),
            "weight": weight,
        })
    return pd.DataFrame(rows)

def descriptor_support_score(series: pd.Series, q_low: float, q_high: float) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce").astype(float)
    center = 0.5 * (q_low + q_high)
    half = max(0.5 * (q_high - q_low), 1e-8)
    z = np.abs(x - center) / half
    return (1.0 - np.clip(z, 0.0, 1.0)).fillna(0.0)

def compute_plausibility_scores(df: pd.DataFrame, reference_bounds_df: pd.DataFrame) -> Tuple[pd.Series, pd.DataFrame]:
    comp = pd.DataFrame(index=df.index)
    fail_cols = []
    weighted_num = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    weighted_den = 0.0

    for _, row in reference_bounds_df.iterrows():
        metric = row["descriptor"]
        q_low = float(row["q_low"])
        q_high = float(row["q_high"])
        weight = float(row["weight"])

        score_col = f"support_{metric}"
        fail_col = f"flag_outside_reference_{metric}"
        comp[score_col] = descriptor_support_score(df[metric], q_low, q_high)
        comp[fail_col] = ((df[metric] < q_low) | (df[metric] > q_high)).astype(int)
        fail_cols.append(fail_col)

        weighted_num += comp[score_col] * weight
        weighted_den += weight

    comp["plausibility_fail_count"] = comp[fail_cols].sum(axis=1)
    comp["descriptor_plausibility_score"] = weighted_num / weighted_den
    comp["descriptor_plausibility_pass"] = (comp["plausibility_fail_count"] <= 2).astype(int)
    return comp["descriptor_plausibility_score"].clip(0.0, 1.0), comp


# =============================================================================
# Condition fidelity
# =============================================================================

def infer_length_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 10:
        return "short"
    if x <= 20:
        return "medium"
    return "long"

def infer_charge_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 2:
        return "low"
    if x <= 6:
        return "medium"
    return "high"

def infer_hydropathy_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x < -1.0:
        return "low"
    if x <= 1.0:
        return "medium"
    return "high"

def compute_condition_fidelity(df: pd.DataFrame, colmap: Dict[str, Any]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    has_bins = all(colmap.get(k) is not None for k in [
        "requested_length_bin_col", "requested_charge_bin_col", "requested_hydrophobicity_bin_col"
    ])

    if has_bins:
        out["length_bin_match"] = (
            df[colmap["requested_length_bin_col"]].astype(str).str.lower().str.strip()
            == df["length"].map(infer_length_bin).astype(str).str.lower().str.strip()
        ).astype(int)
        out["charge_bin_match"] = (
            df[colmap["requested_charge_bin_col"]].astype(str).str.lower().str.strip()
            == df["net_charge_proxy"].map(infer_charge_bin).astype(str).str.lower().str.strip()
        ).astype(int)
        out["hydropathy_bin_match"] = (
            df[colmap["requested_hydrophobicity_bin_col"]].astype(str).str.lower().str.strip()
            == df["mean_hydropathy"].map(infer_hydropathy_bin).astype(str).str.lower().str.strip()
        ).astype(int)

        if colmap.get("requested_length_min_col") and colmap.get("requested_length_max_col"):
            out["length_range_match"] = (
                (df["length"] >= pd.to_numeric(df[colmap["requested_length_min_col"]], errors="coerce")) &
                (df["length"] <= pd.to_numeric(df[colmap["requested_length_max_col"]], errors="coerce"))
            ).astype(int)
        if colmap.get("requested_charge_min_col") and colmap.get("requested_charge_max_col"):
            out["charge_range_match"] = (
                (df["net_charge_proxy"] >= pd.to_numeric(df[colmap["requested_charge_min_col"]], errors="coerce")) &
                (df["net_charge_proxy"] <= pd.to_numeric(df[colmap["requested_charge_max_col"]], errors="coerce"))
            ).astype(int)
        if colmap.get("requested_hydrophobicity_min_col") and colmap.get("requested_hydrophobicity_max_col"):
            out["hydropathy_range_match"] = (
                (df["mean_hydropathy"] >= pd.to_numeric(df[colmap["requested_hydrophobicity_min_col"]], errors="coerce")) &
                (df["mean_hydropathy"] <= pd.to_numeric(df[colmap["requested_hydrophobicity_max_col"]], errors="coerce"))
            ).astype(int)

        pass_cols = [c for c in out.columns if c.endswith("_match")]
        out["condition_fidelity_score"] = out[pass_cols].mean(axis=1)
        out["condition_fidelity_pass"] = (out["condition_fidelity_score"] >= 0.67).astype(int)
        out["condition_fidelity_mode"] = "strict"
    else:
        out["condition_fidelity_score"] = np.nan
        out["condition_fidelity_pass"] = 1
        out["condition_fidelity_mode"] = "relaxed_missing_requested_bins"

    return out


# =============================================================================
# Novelty, diversity, ranking
# =============================================================================

def compute_novelty_score(df: pd.DataFrame, colmap: Dict[str, Any]) -> pd.Series:
    exact_train_novel = 1.0 - pd.to_numeric(df[colmap["exact_train_match_col"]], errors="coerce").clip(0, 1)
    nn_novel = 1.0 - pd.to_numeric(df[colmap["train_nn_similarity_col"]], errors="coerce").clip(0, 1)
    components = [exact_train_novel, nn_novel]
    if colmap.get("exact_full_match_col") is not None:
        full_novel = 1.0 - pd.to_numeric(df[colmap["exact_full_match_col"]], errors="coerce").clip(0, 1)
        components.append(full_novel)
    return pd.concat(components, axis=1).mean(axis=1).clip(0.0, 1.0)

def compute_pairwise_similarity_edges(seqs: Sequence[str], threshold: float, k: int) -> List[Tuple[int, int, float]]:
    edges = []
    for i in range(len(seqs)):
        for j in range(i + 1, len(seqs)):
            sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=k)
            if sim >= threshold:
                edges.append((i, j, sim))
    return edges

def connected_components_from_edges(n_nodes: int, edges: Sequence[Tuple[int, int, float]]) -> List[List[int]]:
    adj = [[] for _ in range(n_nodes)]
    for i, j, _ in edges:
        adj[i].append(j)
        adj[j].append(i)
    seen = np.zeros(n_nodes, dtype=bool)
    comps = []
    for start in range(n_nodes):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
        comps.append(sorted(comp))
    return comps

def compute_diversity_scores(seqs: Sequence[str], k: int) -> pd.Series:
    n = len(seqs)
    if n <= 1:
        return pd.Series(np.ones(n))
    nn_div = []
    for i in range(n):
        sims = []
        for j in range(n):
            if i == j:
                continue
            sims.append(jaccard_kmer_similarity(seqs[i], seqs[j], k=k))
        nearest = max(sims) if sims else 0.0
        nn_div.append(1.0 - nearest)
    return pd.Series(nn_div).clip(0.0, 1.0)

def shortlist_with_cluster_cap(
    df: pd.DataFrame,
    seq_col: str,
    score_col: str,
    threshold: float,
    max_per_cluster: int,
    top_n: int,
    k: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    seqs = df[seq_col].tolist()
    edges = compute_pairwise_similarity_edges(seqs, threshold=threshold, k=k)
    comps = connected_components_from_edges(len(seqs), edges)

    cluster_id = np.full(len(df), -1, dtype=int)
    for cid, comp in enumerate(comps):
        for idx in comp:
            cluster_id[idx] = cid

    out = df.copy()
    out["similarity_cluster_id"] = cluster_id

    chosen = []
    cluster_counts: Dict[int, int] = defaultdict(int)
    ranked_idx = out.sort_values(score_col, ascending=False).index.tolist()
    for idx in ranked_idx:
        cid = int(out.loc[idx, "similarity_cluster_id"])
        if cluster_counts[cid] < max_per_cluster:
            chosen.append(idx)
            cluster_counts[cid] += 1
        if len(chosen) >= top_n:
            break

    chosen_df = out.loc[chosen].copy().sort_values(score_col, ascending=False).reset_index(drop=True)
    return chosen_df, out

def resolve_active_ranking_components(df: pd.DataFrame, candidate_weights: Dict[str, float], tol: float = 1e-6) -> Tuple[Dict[str, float], pd.DataFrame]:
    rows = []
    active = {}
    for name, weight in candidate_weights.items():
        if name not in df.columns:
            rows.append({"component": name, "status": "missing", "reason": "column_not_found", "weight": weight})
            continue
        vals = pd.to_numeric(df[name], errors="coerce")
        if vals.notna().sum() == 0:
            rows.append({"component": name, "status": "excluded", "reason": "all_missing", "weight": weight})
            continue
        rng = float(vals.max(skipna=True) - vals.min(skipna=True))
        if np.isnan(rng) or rng <= tol:
            rows.append({"component": name, "status": "excluded", "reason": "near_constant_or_placeholder", "weight": weight})
            continue
        active[name] = weight
        rows.append({"component": name, "status": "active", "reason": "usable", "weight": weight})
    return active, pd.DataFrame(rows)

def weighted_score(df: pd.DataFrame, weights: Dict[str, float]) -> pd.Series:
    if not weights:
        raise ValueError("No active ranking components are available.")
    total_w = sum(weights.values())
    s = pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    for c, w in weights.items():
        vals = pd.to_numeric(df[c], errors="coerce")
        fill_value = float(vals.median()) if vals.notna().any() else 0.5
        s += vals.fillna(fill_value).clip(0.0, 1.0) * w
    return s / total_w

def dense_rank_desc(s: pd.Series) -> pd.Series:
    return s.rank(method="dense", ascending=False).astype(int)

def compute_scheme_stability(results_by_scheme: Dict[str, pd.DataFrame], id_col: str, top_n: int) -> pd.DataFrame:
    rows = []
    scheme_names = list(results_by_scheme.keys())
    for a, b in combinations(scheme_names, 2):
        top_a = set(results_by_scheme[a].head(top_n)[id_col].astype(str))
        top_b = set(results_by_scheme[b].head(top_n)[id_col].astype(str))
        inter = len(top_a & top_b)
        union = len(top_a | top_b)
        rows.append({
            "scheme_a": a,
            "scheme_b": b,
            "top_n": top_n,
            "intersection_n": inter,
            "union_n": union,
            "jaccard_overlap": inter / union if union else np.nan,
            "overlap_fraction_of_top_n": inter / top_n if top_n else np.nan,
        })
    return pd.DataFrame(rows)

def bootstrap_shortlist_membership_stability(df: pd.DataFrame, id_col: str, weights: Dict[str, float], top_n: int, n_boot: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for b in range(n_boot):
        sampled = df.sample(n=len(df), replace=True, random_state=int(rng.integers(0, 1_000_000)))
        sampled["_boot_score"] = weighted_score(sampled, weights)
        sampled = sampled.sort_values("_boot_score", ascending=False).head(top_n)
        for pid in sampled[id_col].astype(str):
            rows.append({"bootstrap_iter": b, id_col: pid, "in_top_n": 1})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=[id_col, "bootstrap_top_n_frequency"])
    return out.groupby(id_col)["in_top_n"].mean().rename("bootstrap_top_n_frequency").reset_index()


# =============================================================================
# Main pipeline
# =============================================================================

def run_stage(cfg: Step8Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Loading Step 1 / Step 7 inputs")
    loaded = load_step_inputs(cfg)
    ref_df, _ = prepare_reference_df(loaded["step1_ref_df"], cfg)
    gen_df, gen_colmap = prepare_generated_df(loaded["step7_generated_df"], cfg)

    inventory_df = inspect_step7_candidate_inventory(str(Path(cfg.project_root).resolve() / cfg.step7_dir), cfg)
    inventory_df.to_csv(dirs["artifacts"] / "step7_csv_inventory.csv", index=False)

    save_json(
        {
            "config": asdict(cfg),
            "step1_reference_path": str(loaded["step1_ref_path"]),
            "step7_candidate_path": str(loaded["step7_candidate_path"]),
            "input_hashes": {
                "step1_reference_sha256": sha256_of_file(loaded["step1_ref_path"]),
                "step7_candidate_sha256": sha256_of_file(loaded["step7_candidate_path"]),
            },
            "environment_manifest": collect_environment_manifest(),
        },
        dirs["artifacts"] / "run_manifest.json",
    )

    print(f"Loaded Step 1 reference rows: {len(ref_df):,}")
    print(f"Loaded Step 7 candidate rows: {len(gen_df):,}")
    print(f"Relaxed condition mode: {gen_colmap.get('relaxed_condition_mode', False)}")

    print_header("Applying hard eligibility filters")
    flags_df = build_hard_filter_flags(gen_df, cfg, gen_colmap)
    filter_audit_df = summarize_filter_waterfall(flags_df)
    gen_scored = pd.concat([gen_df.reset_index(drop=True), flags_df.reset_index(drop=True)], axis=1)
    gen_scored["eligible_after_hard_filters"] = 1 - gen_scored["hard_filter_fail"]

    eligible = gen_scored[gen_scored["eligible_after_hard_filters"] == 1].copy().reset_index(drop=True)
    if len(eligible) == 0:
        raise ValueError("No candidates remain after hard eligibility filtering.")

    print_header("Computing descriptor plausibility")
    reference_bounds_df = build_reference_bounds(ref_df, cfg)
    plaus_score, plaus_detail = compute_plausibility_scores(eligible, reference_bounds_df)
    eligible = pd.concat([eligible.reset_index(drop=True), plaus_detail.reset_index(drop=True)], axis=1)
    eligible["descriptor_plausibility_score"] = plaus_score

    print_header("Computing condition fidelity")
    condition_detail = compute_condition_fidelity(eligible, gen_colmap)
    eligible = pd.concat([eligible.reset_index(drop=True), condition_detail.reset_index(drop=True)], axis=1)

    print_header("Computing novelty and supportive scores")
    eligible["novelty_score"] = compute_novelty_score(eligible, gen_colmap)
    eligible["diversity_score"] = compute_diversity_scores(eligible[gen_colmap["sequence_col"]].tolist(), k=cfg.similarity_kmer).values
    eligible["supportive_model_score"], supportive_provenance_df = build_supportive_predictor_frame(eligible, cfg)

    eligible["final_pass"] = ((eligible["descriptor_plausibility_pass"] == 1) & (eligible["condition_fidelity_pass"] == 1)).astype(int)
    passed = eligible[eligible["final_pass"] == 1].copy().reset_index(drop=True)
    if len(passed) == 0:
        raise ValueError("No candidates remain after plausibility and screening.")

    print_header("Resolving active ranking components")
    active_weights, component_status_df = resolve_active_ranking_components(passed, cfg.ranking_weights, cfg.component_variance_tolerance)
    if not active_weights:
        raise ValueError("No active ranking components remain after excluding missing/constant components.")
    print(component_status_df.to_string(index=False))

    passed["final_score"] = weighted_score(passed, active_weights)
    passed["final_rank"] = dense_rank_desc(passed["final_score"])

    print_header("Building diversity-aware shortlist")
    seq_col = gen_colmap["sequence_col"]
    id_col = gen_colmap["id_col"]

    shortlist_df, annotated_ranked_df = shortlist_with_cluster_cap(
        df=passed.sort_values("final_score", ascending=False).reset_index(drop=True),
        seq_col=seq_col,
        score_col="final_score",
        threshold=cfg.similarity_threshold_for_clusters,
        max_per_cluster=cfg.max_per_similarity_cluster,
        top_n=cfg.shortlist_top_n,
        k=cfg.similarity_kmer,
    )
    shortlist_df["shortlist_rank"] = dense_rank_desc(shortlist_df["final_score"])

    final_panel_df, _ = shortlist_with_cluster_cap(
        df=shortlist_df.sort_values("final_score", ascending=False).reset_index(drop=True),
        seq_col=seq_col,
        score_col="final_score",
        threshold=cfg.similarity_threshold_for_clusters,
        max_per_cluster=1,
        top_n=cfg.final_diverse_panel_n,
        k=cfg.similarity_kmer,
    )
    final_panel_df["final_panel_rank"] = dense_rank_desc(final_panel_df["final_score"])

    print(f"Top shortlist size: {len(shortlist_df):,}")
    print(f"Final diverse panel size: {len(final_panel_df):,}")

    print_header("Ranking sensitivity analysis")
    results_by_scheme = {}
    scheme_status_rows = []
    for scheme_name, scheme_weights in cfg.ranking_schemes.items():
        tmp_active, tmp_status = resolve_active_ranking_components(passed, scheme_weights, cfg.component_variance_tolerance)
        tmp_status["scheme"] = scheme_name
        scheme_status_rows.append(tmp_status)
        if not tmp_active:
            continue
        tmp = passed.copy()
        tmp["final_score"] = weighted_score(tmp, tmp_active)
        tmp["final_rank"] = dense_rank_desc(tmp["final_score"])
        tmp = tmp.sort_values("final_score", ascending=False).reset_index(drop=True)
        results_by_scheme[scheme_name] = tmp

    scheme_status_df = pd.concat(scheme_status_rows, ignore_index=True)
    stability_df = compute_scheme_stability(results_by_scheme, id_col, cfg.shortlist_top_n) if len(results_by_scheme) >= 2 else pd.DataFrame()

    recurrence_df = pd.DataFrame()
    if results_by_scheme:
        top_sets = {name: set(df.head(cfg.shortlist_top_n)[id_col].astype(str)) for name, df in results_by_scheme.items()}
        recurrence_rows = []
        union_ids = sorted(set().union(*top_sets.values()))
        for pid in union_ids:
            hit_map = {f"in_top_{name}": int(pid in idset) for name, idset in top_sets.items()}
            recurrence_rows.append({id_col: pid, "scheme_recurrence_n": sum(hit_map.values()), **hit_map})
        recurrence_df = pd.DataFrame(recurrence_rows)

        boot_cols = [id_col] + list(active_weights.keys())
        bootstrap_freq_df = bootstrap_shortlist_membership_stability(
            passed[boot_cols].copy(),
            id_col=id_col,
            weights=active_weights,
            top_n=cfg.shortlist_top_n,
            n_boot=cfg.bootstrap_iterations,
            seed=cfg.main_random_seed,
        )
        if len(bootstrap_freq_df) > 0:
            recurrence_df[id_col] = recurrence_df[id_col].astype(str)
            bootstrap_freq_df[id_col] = bootstrap_freq_df[id_col].astype(str)
            recurrence_df = recurrence_df.merge(bootstrap_freq_df, on=id_col, how="left")

    print_header("Building descriptor summaries")
    compare_rows = []
    groups = {
        "Raw pool": gen_df,
        "Eligible": eligible,
        "Passed": passed,
        "Shortlist": shortlist_df,
        "Final panel": final_panel_df,
        "Reference ACP corpus": ref_df,
    }
    compare_metrics = list(cfg.descriptor_weight_map.keys()) + ["novelty_score", "condition_fidelity_score", "supportive_model_score", "diversity_score", "final_score"]
    for group_name, dfi in groups.items():
        for metric in compare_metrics:
            if metric in dfi.columns:
                vals = pd.to_numeric(dfi[metric], errors="coerce").dropna()
                compare_rows.append({
                    "group": group_name,
                    "metric": metric,
                    "n": len(vals),
                    "mean": float(vals.mean()) if len(vals) else np.nan,
                    "median": float(vals.median()) if len(vals) else np.nan,
                    "q25": float(vals.quantile(0.25)) if len(vals) else np.nan,
                    "q75": float(vals.quantile(0.75)) if len(vals) else np.nan,
                })
    descriptor_compare_df = pd.DataFrame(compare_rows)

    # rationale table
    rationale_rows = []
    for _, r in final_panel_df.sort_values("final_panel_rank").iterrows():
        rationale_rows.append({
            id_col: r[id_col],
            "sequence": r[seq_col],
            "final_panel_rank": int(r["final_panel_rank"]),
            "final_score": float(r["final_score"]),
            "novelty_score": float(r["novelty_score"]),
            "descriptor_plausibility_score": float(r["descriptor_plausibility_score"]),
            "diversity_score": float(r["diversity_score"]),
            "similarity_cluster_id": int(r["similarity_cluster_id"]),
            "selection_rationale": "High novelty and plausibility; retained after cluster cap for diversity.",
        })
    rationale_df = pd.DataFrame(rationale_rows)

    cluster_cap_df = shortlist_df.groupby("similarity_cluster_id").agg(
        shortlist_n=(id_col, "size"),
        max_final_score=("final_score", "max"),
        min_final_score=("final_score", "min"),
    ).reset_index().sort_values(["shortlist_n", "max_final_score"], ascending=[False, False])

    # exports
    filter_audit_df.to_csv(dirs["tables_supplementary"] / "table_s8_1_filter_audit.csv", index=False)
    gen_scored.to_csv(dirs["tables_supplementary"] / "table_s8_2_candidate_pool_with_flags.csv", index=False)
    reference_bounds_df.to_csv(dirs["tables_supplementary"] / "table_s8_3_reference_descriptor_bounds.csv", index=False)
    supportive_provenance_df.to_csv(dirs["tables_supplementary"] / "table_s8_4_auxiliary_predictor_provenance.csv", index=False)
    stability_df.to_csv(dirs["tables_supplementary"] / "table_s8_5_ranking_scheme_stability.csv", index=False)
    recurrence_df.to_csv(dirs["tables_supplementary"] / "table_s8_6_candidate_recurrence_across_schemes.csv", index=False)
    descriptor_compare_df.to_csv(dirs["tables_supplementary"] / "table_s8_7_descriptor_group_comparison.csv", index=False)
    component_status_df.to_csv(dirs["tables_supplementary"] / "table_s8_8_component_status.csv", index=False)
    scheme_status_df.to_csv(dirs["tables_supplementary"] / "table_s8_9_scheme_component_status.csv", index=False)
    rationale_df.to_csv(dirs["tables_supplementary"] / "table_s8_10_final_panel_rationale.csv", index=False)
    cluster_cap_df.to_csv(dirs["tables_supplementary"] / "table_s8_11_cluster_cap_audit.csv", index=False)

    shortlist_main_cols = [
        id_col, seq_col, "shortlist_rank", "final_score", "final_rank",
        "novelty_score", "descriptor_plausibility_score", "diversity_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy", "similarity_cluster_id"
    ] + ([ "condition_fidelity_score" ] if "condition_fidelity_score" in shortlist_df.columns else [])
    shortlist_main_cols = [c for c in shortlist_main_cols if c in shortlist_df.columns]
    shortlist_table = shortlist_df[shortlist_main_cols].sort_values("shortlist_rank").reset_index(drop=True)

    final_panel_cols = [
        id_col, seq_col, "final_panel_rank", "final_score",
        "novelty_score", "descriptor_plausibility_score", "diversity_score",
        "length", "net_charge_proxy", "mean_hydropathy", "entropy", "similarity_cluster_id"
    ] + ([ "condition_fidelity_score" ] if "condition_fidelity_score" in final_panel_df.columns else [])
    final_panel_cols = [c for c in final_panel_cols if c in final_panel_df.columns]
    final_panel_table = final_panel_df[final_panel_cols].sort_values("final_panel_rank").reset_index(drop=True)

    shortlist_table.to_csv(dirs["tables_main"] / "table_8_1_shortlist_main.csv", index=False)
    final_panel_table.to_csv(dirs["tables_main"] / "table_8_2_final_diverse_panel.csv", index=False)
    annotated_ranked_df.sort_values("final_score", ascending=False).to_csv(
        dirs["tables_supplementary"] / "table_s8_12_full_ranked_passed_pool.csv", index=False
    )

    summary_lines = [
        "Multi-objective prioritization and biological plausibility screening — summary",
        "=" * 84,
        f"Timestamp UTC: {now_utc_iso()}",
        f"Step 1 reference: {loaded['step1_ref_path']}",
        f"Step 7 candidate input: {loaded['step7_candidate_path']}",
        "",
        "Counts",
        f"- Raw generated pool: {len(gen_df):,}",
        f"- Eligible after hard filters: {len(eligible):,}",
        f"- Passed plausibility screening: {len(passed):,}",
        f"- Shortlist: {len(shortlist_df):,}",
        f"- Final diverse panel: {len(final_panel_df):,}",
        "",
        "Active primary ranking components",
    ]
    for k, v in active_weights.items():
        summary_lines.append(f"- {k}: {v}")
    summary_lines += [
        "",
        f"Relaxed condition mode: {gen_colmap.get('relaxed_condition_mode', False)}",
        "",
        "Supportive predictor provenance",
    ]
    for _, row in supportive_provenance_df.iterrows():
        summary_lines.append(f"- {row['role']}: {row['status']} ({row['column_used']})")
    save_text("\n".join(summary_lines), dirs["reports"] / "summary_report.txt")

    return {
        "ref_df": ref_df,
        "gen_df": gen_df,
        "eligible_df": eligible,
        "passed_df": passed,
        "shortlist_df": shortlist_df,
        "final_panel_df": final_panel_df,
        "shortlist_table": shortlist_table,
        "final_panel_table": final_panel_table,
        "filter_audit_df": filter_audit_df,
        "reference_bounds_df": reference_bounds_df,
        "stability_df": stability_df,
        "recurrence_df": recurrence_df,
        "descriptor_compare_df": descriptor_compare_df,
        "supportive_provenance_df": supportive_provenance_df,
        "annotated_ranked_df": annotated_ranked_df,
        "gen_colmap": gen_colmap,
        "component_status_df": component_status_df,
        "scheme_status_df": scheme_status_df,
        "active_weights": active_weights,
        "rationale_df": rationale_df,
        "cluster_cap_df": cluster_cap_df,
    }


# =============================================================================
# Figure helpers
# =============================================================================

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.set_facecolor(PROJECT_COLORS["bg"])
    ax.spines["left"].set_color(PROJECT_COLORS["spine"])
    ax.spines["bottom"].set_color(PROJECT_COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(colors=PROJECT_COLORS["text"], width=0.7)
    ax.grid(True, axis=grid_axis, color=PROJECT_COLORS["grid"], linewidth=0.55)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str) -> None:
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, fontsize=10, fontweight="bold",
            ha="left", va="bottom", color=PROJECT_COLORS["text"])

def make_violin(ax, data_by_group: Dict[str, Sequence[float]], colors: Dict[str, str], y_label: str, title: Optional[str] = None,
                ylim: Optional[Tuple[float, float]] = None, show_points: bool = True) -> None:
    labels = list(data_by_group.keys())
    positions = np.arange(1, len(labels) + 1)
    arrays = [np.asarray(list(data_by_group[k]), dtype=float) for k in labels]
    arrays = [a[np.isfinite(a)] for a in arrays]

    vp = ax.violinplot(arrays, positions=positions, widths=0.85, showmeans=False, showextrema=False, showmedians=False)
    for body, lab in zip(vp["bodies"], labels):
        body.set_facecolor(colors[lab])
        body.set_edgecolor(colors[lab])
        body.set_alpha(0.9)
        body.set_linewidth(0.8)

    for pos, lab, arr in zip(positions, labels, arrays):
        if len(arr) == 0:
            continue
        q25, med, q75 = np.quantile(arr, [0.25, 0.5, 0.75])
        ax.vlines(pos, q25, q75, color=PROJECT_COLORS["gray_dark"], linewidth=1.0, zorder=4)
        ax.scatter([pos], [med], s=16, facecolor="white", edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.8, zorder=5)
        if show_points and len(arr) <= 30:
            jitter = np.linspace(-0.08, 0.08, len(arr)) if len(arr) > 1 else np.array([0.0])
            ax.scatter(np.full(len(arr), pos) + jitter, arr, s=10, color=colors[lab], alpha=0.55,
                       edgecolor="white", linewidth=0.2, zorder=3)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_ylabel(y_label)
    if title:
        ax.set_title(title, loc="left", pad=4)
    if ylim is not None:
        ax.set_ylim(*ylim)
    style_axis(ax, grid_axis="y")

def compute_nn_similarity_to_within_group(seqs: Sequence[str], k: int) -> np.ndarray:
    seqs = list(seqs)
    if len(seqs) <= 1:
        return np.array([0.0] * len(seqs), dtype=float)
    out = []
    for i, s in enumerate(seqs):
        sims = [jaccard_kmer_similarity(s, t, k=k) for j, t in enumerate(seqs) if i != j]
        out.append(max(sims) if sims else 0.0)
    return np.asarray(out, dtype=float)

def pairwise_similarity_matrix(seqs: Sequence[str], k: int) -> np.ndarray:
    seqs = list(seqs)
    n = len(seqs)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=k)
            mat[i, j] = sim
            mat[j, i] = sim
    return mat


# =============================================================================
# Figures
# =============================================================================

def plot_main_figure(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    gen_df = results["gen_df"]
    eligible_df = results["eligible_df"]
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    filter_audit_df = results["filter_audit_df"]
    active_weights = results["active_weights"]
    id_col = results["gen_colmap"]["id_col"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 7.6))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34)

    # a) prioritization cascade
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    stages = ["Generated pool", "Eligible", "Plausible set", "Shortlist", "Final panel"]
    vals = [len(gen_df), len(eligible_df), len(passed_df), len(shortlist_df), len(final_panel_df)]
    y = np.arange(len(stages))[::-1]
    cols = [GROUP_COLORS["Raw pool"], GROUP_COLORS["Eligible"], GROUP_COLORS["Passed"], GROUP_COLORS["Shortlist"], GROUP_COLORS["Final panel"]]
    ax.barh(y, vals, color=cols, edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.6, height=0.72)
    ax.set_yticks(y)
    ax.set_yticklabels(stages)
    ax.set_xlabel("Number of peptides")
    ax.set_title("Candidate prioritization cascade", loc="left", pad=4)
    style_axis(ax, grid_axis="x")
    for yi, v in zip(y, vals):
        frac = 100.0 * v / vals[0] if vals[0] else 0
        ax.text(v + max(vals)*0.01, yi, f"{v:,} ({frac:.1f}%)", va="center", ha="left", fontsize=7)
    top_filters = filter_audit_df.sort_values("removed_n", ascending=False).head(3)
    note = "Top exclusions: " + "; ".join([
        f"{x.replace('flag_', '').replace('_', ' ')} ({n:,})" for x, n in zip(top_filters["filter_name"], top_filters["removed_n"])
    ])
    ax.text(0.0, -0.24, note, transform=ax.transAxes, fontsize=6.5, color=PROJECT_COLORS["gray_dark"], ha="left")

    # b) novelty-plausibility landscape
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="both")
    hb = ax.hexbin(
        passed_df["novelty_score"], passed_df["descriptor_plausibility_score"],
        gridsize=34, mincnt=1, cmap="Greys", linewidths=0.0, alpha=0.8
    )
    ax.scatter(
        shortlist_df["novelty_score"], shortlist_df["descriptor_plausibility_score"],
        s=28, facecolor=GROUP_COLORS["Shortlist"], edgecolor="white", linewidth=0.35, alpha=0.95, label="Shortlist"
    )
    ax.scatter(
        final_panel_df["novelty_score"], final_panel_df["descriptor_plausibility_score"],
        s=36, facecolor=GROUP_COLORS["Final panel"], edgecolor="white", linewidth=0.45, alpha=1.0, label="Final panel"
    )
    ax.set_xlabel("Novelty score")
    ax.set_ylabel("Descriptor plausibility score")
    ax.set_title("Novelty–plausibility landscape", loc="left", pad=4)
    ax.legend(frameon=False, loc="lower left")
    cb = fig.colorbar(hb, ax=ax, fraction=0.048, pad=0.03)
    cb.set_label("Passed-pool density")
    cb.outline.set_edgecolor(PROJECT_COLORS["spine"])

    # c) ranking component distributions
    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    component_frames = []
    display_names = {
        "novelty_score": "Novelty",
        "descriptor_plausibility_score": "Plausibility",
        "diversity_score": "Diversity",
        "condition_fidelity_score": "Condition fidelity",
        "supportive_model_score": "Supportive model",
        "final_score": "Final score",
    }
    active_for_plot = list(active_weights.keys()) + ["final_score"]
    for comp in active_for_plot:
        for grp_name, dfi in [("Passed", passed_df), ("Shortlist", shortlist_df), ("Final panel", final_panel_df)]:
            vals = pd.to_numeric(dfi[comp], errors="coerce").dropna().astype(float).tolist()
            for v in vals:
                component_frames.append({"component": display_names.get(comp, comp), "group": grp_name, "value": v})
    comp_df = pd.DataFrame(component_frames)
    if len(comp_df) > 0:
        order = [display_names.get(c, c) for c in active_for_plot]
        xpos = np.arange(len(order))
        offsets = {"Passed": -0.23, "Shortlist": 0.0, "Final panel": 0.23}
        for grp in ["Passed", "Shortlist", "Final panel"]:
            sub = comp_df[comp_df["group"] == grp]
            for i, comp in enumerate(order):
                arr = sub.loc[sub["component"] == comp, "value"].to_numpy(dtype=float)
                if len(arr) == 0:
                    continue
                vp = ax.violinplot([arr], positions=[xpos[i] + offsets[grp]], widths=0.18, showmeans=False, showextrema=False, showmedians=False)
                body = vp["bodies"][0]
                body.set_facecolor(GROUP_COLORS[grp])
                body.set_edgecolor(GROUP_COLORS[grp])
                body.set_alpha(0.95)
                q25, med, q75 = np.quantile(arr, [0.25, 0.5, 0.75])
                ax.vlines(xpos[i] + offsets[grp], q25, q75, color=PROJECT_COLORS["gray_dark"], linewidth=0.8, zorder=4)
                ax.scatter([xpos[i] + offsets[grp]], [med], s=13, facecolor="white", edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.7, zorder=5)
        ax.set_xticks(xpos)
        ax.set_xticklabels(order)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Score")
    ax.set_title("Ranking-component distributions", loc="left", pad=4)
    style_axis(ax, grid_axis="y")
    handles = [plt.Line2D([0],[0], color=GROUP_COLORS[g], lw=6) for g in ["Passed","Shortlist","Final panel"]]
    ax.legend(handles, ["Passed", "Shortlist", "Final panel"], frameon=False, loc="lower left", ncol=3)

    # d) diversity-aware final panel summary
    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    div_data = {
        "Passed": pd.to_numeric(passed_df["diversity_score"], errors="coerce").dropna().to_numpy(),
        "Shortlist": pd.to_numeric(shortlist_df["diversity_score"], errors="coerce").dropna().to_numpy(),
        "Final panel": pd.to_numeric(final_panel_df["diversity_score"], errors="coerce").dropna().to_numpy(),
    }
    make_violin(ax, div_data, {"Passed": GROUP_COLORS["Passed"], "Shortlist": GROUP_COLORS["Shortlist"], "Final panel": GROUP_COLORS["Final panel"]},
                y_label="Diversity score", title="Diversity shift across prioritization", ylim=(0, 1.02), show_points=True)
    fig.tight_layout()
    export_figure(fig, dirs["figures_main"] / "figure_8_main_prioritization_v1", cfg.figure_export)

def plot_supplementary_descriptor_figure(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    ref_df = results["ref_df"]

    metrics = [
        ("length", "Length"),
        ("net_charge_proxy", "Net charge proxy"),
        ("mean_hydropathy", "Mean hydropathy"),
        ("entropy", "Sequence entropy"),
    ]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.8))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)

    for i, (metric, label) in enumerate(metrics):
        ax = fig.add_subplot(gs[i // 2, i % 2])
        add_panel_label(ax, chr(ord("a") + i))
        data_by_group = {
            "Passed": pd.to_numeric(passed_df[metric], errors="coerce").dropna().to_numpy(),
            "Shortlist": pd.to_numeric(shortlist_df[metric], errors="coerce").dropna().to_numpy(),
            "Final panel": pd.to_numeric(final_panel_df[metric], errors="coerce").dropna().to_numpy(),
            "Reference ACP corpus": pd.to_numeric(ref_df[metric], errors="coerce").dropna().to_numpy(),
        }
        make_violin(ax, data_by_group,
                    {"Passed": GROUP_COLORS["Passed"], "Shortlist": GROUP_COLORS["Shortlist"], "Final panel": GROUP_COLORS["Final panel"], "Reference ACP corpus": GROUP_COLORS["Reference ACP corpus"]},
                    y_label=label, title=label, ylim=None, show_points=False)
    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_1_descriptor_shifts_v1", cfg.figure_export)

def plot_supplementary_robustness_figure(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    stability_df = results["stability_df"]
    recurrence_df = results["recurrence_df"]
    final_panel_df = results["final_panel_df"]
    seq_col = results["gen_colmap"]["sequence_col"]
    id_col = results["gen_colmap"]["id_col"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.3))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.15, 0.85], height_ratios=[1.0, 1.0], hspace=0.42, wspace=0.34)

    # a) scheme overlap heatmap
    ax = fig.add_subplot(gs[:, 0])
    add_panel_label(ax, "a")
    if len(stability_df) > 0:
        schemes = sorted(set(stability_df["scheme_a"]).union(set(stability_df["scheme_b"])))
        mat = pd.DataFrame(np.eye(len(schemes)), index=schemes, columns=schemes)
        for _, r in stability_df.iterrows():
            mat.loc[r["scheme_a"], r["scheme_b"]] = r["jaccard_overlap"]
            mat.loc[r["scheme_b"], r["scheme_a"]] = r["jaccard_overlap"]
        im = ax.imshow(mat.values, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(np.arange(len(schemes)))
        ax.set_yticks(np.arange(len(schemes)))
        ax.set_xticklabels(schemes, rotation=30, ha="right")
        ax.set_yticklabels(schemes)
        for i in range(len(schemes)):
            for j in range(len(schemes)):
                ax.text(j, i, f"{mat.values[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if mat.values[i, j] > 0.55 else PROJECT_COLORS["text"])
        ax.set_title("Scheme-overlap matrix", loc="left", pad=4)
        ax.set_facecolor(PROJECT_COLORS["bg"])
        cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
        cb.set_label("Jaccard overlap")
    else:
        ax.text(0.5, 0.5, "Insufficient schemes", ha="center", va="center")

    # b) recurrence summary
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="y")
    if len(recurrence_df) > 0:
        counts = recurrence_df["scheme_recurrence_n"].value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values, color=PROJECT_COLORS["hydramp_red"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.6)
        ax.set_xlabel("Number of schemes containing candidate")
        ax.set_ylabel("Candidate count")
        ax.set_title("Candidate recurrence", loc="left", pad=4)
    else:
        ax.text(0.5, 0.5, "No recurrence data", ha="center", va="center")

    # c) final-panel similarity heatmap
    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "c")
    seqs = final_panel_df[seq_col].tolist()
    labs = final_panel_df[id_col].astype(str).tolist()
    mat = pairwise_similarity_matrix(seqs, cfg.similarity_kmer) if len(seqs) > 0 else np.zeros((1, 1))
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(np.arange(len(labs)))
    ax.set_yticks(np.arange(len(labs)))
    ax.set_xticklabels(labs, rotation=45, ha="right", fontsize=6)
    ax.set_yticklabels(labs, fontsize=6)
    ax.set_title("Final-panel pairwise similarity", loc="left", pad=4)
    cb = fig.colorbar(im, ax=ax, fraction=0.055, pad=0.04)
    cb.set_label("3-mer Jaccard similarity")
    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_2_robustness_diversity_v1", cfg.figure_export)

def plot_supplementary_condition_filter_figure(results: Dict[str, Any], cfg: Step8Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    filter_audit_df = results["filter_audit_df"]
    gen_colmap = results["gen_colmap"]

    cond_specs = [
        ("requested_length_bin_col", "Requested length bin"),
        ("requested_charge_bin_col", "Requested charge bin"),
        ("requested_hydrophobicity_bin_col", "Requested hydropathy bin"),
    ]
    available = [(k, lbl) for k, lbl in cond_specs if gen_colmap.get(k) is not None]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.4))
    gs = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.32)

    panel_idx = 0
    for key, label in available[:3]:
        ax = fig.add_subplot(gs[panel_idx // 2, panel_idx % 2])
        add_panel_label(ax, chr(ord("a") + panel_idx))
        col = gen_colmap[key]
        merged = pd.DataFrame({
            "Passed": passed_df[col].astype(str).value_counts(),
            "Shortlist": shortlist_df[col].astype(str).value_counts(),
            "Final panel": final_panel_df[col].astype(str).value_counts(),
        }).fillna(0)
        x = np.arange(len(merged))
        w = 0.24
        ax.bar(x - w, merged["Passed"], width=w, color=GROUP_COLORS["Passed"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5, label="Passed")
        ax.bar(x, merged["Shortlist"], width=w, color=GROUP_COLORS["Shortlist"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5, label="Shortlist")
        ax.bar(x + w, merged["Final panel"], width=w, color=GROUP_COLORS["Final panel"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5, label="Final panel")
        ax.set_xticks(x)
        ax.set_xticklabels(merged.index.astype(str))
        ax.set_ylabel("Count")
        ax.set_title(label, loc="left", pad=4)
        style_axis(ax, grid_axis="y")
        if panel_idx == 0:
            ax.legend(frameon=False, ncol=3, loc="upper right")
        panel_idx += 1

    ax = fig.add_subplot(gs[1, 1] if panel_idx < 3 else gs[1, 1])
    add_panel_label(ax, chr(ord("a") + min(panel_idx, 3)))
    top_filters = filter_audit_df.sort_values("removed_n", ascending=True).tail(8)
    ax.barh(top_filters["filter_name"].str.replace("flag_", "").str.replace("_", " "), top_filters["removed_n"],
            color=PROJECT_COLORS["gan_orange"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5)
    ax.set_xlabel("Removed peptides")
    ax.set_title("Filter audit", loc="left", pad=4)
    style_axis(ax, grid_axis="x")
    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_3_condition_filter_diagnostics_v1", cfg.figure_export)


# =============================================================================
# Notebook entrypoint
# =============================================================================

def main_notebook(cfg: Optional[Step8Config] = None) -> Dict[str, Any]:
    if cfg is None:
        cfg = Step8Config()

    seed_all(cfg.main_random_seed)
    project_root = Path(cfg.project_root).resolve()
    step7_root = project_root / cfg.step7_dir
    resolved_candidate = find_candidate_level_step7_csv(step7_root, cfg)
    cfg.step7_candidate_file = str(resolved_candidate)

    step_root = project_root / cfg.step8_dir
    dirs = make_output_dirs(step_root)

    print_header("Running multi-objective prioritization and biological plausibility screening (v1)")
    print(json.dumps({
        "project_root": cfg.project_root,
        "step1_dir": cfg.step1_dir,
        "step7_dir": cfg.step7_dir,
        "stage_dir": cfg.step8_dir,
        "step1_reference_file": cfg.step1_reference_file,
        "step7_candidate_file": cfg.step7_candidate_file,
        "shortlist_top_n": cfg.shortlist_top_n,
        "final_diverse_panel_n": cfg.final_diverse_panel_n,
        "allow_relaxed_step7_input": cfg.allow_relaxed_step7_input,
    }, indent=2))

    results = run_stage(cfg, dirs)
    plot_main_figure(results, cfg, dirs)
    plot_supplementary_descriptor_figure(results, cfg, dirs)
    plot_supplementary_robustness_figure(results, cfg, dirs)
    plot_supplementary_condition_filter_figure(results, cfg, dirs)

    save_json(
        {
            "n_reference": int(len(results["ref_df"])),
            "n_step7_raw": int(len(results["gen_df"])),
            "n_eligible": int(len(results["eligible_df"])),
            "n_passed": int(len(results["passed_df"])),
            "n_shortlist": int(len(results["shortlist_df"])),
            "n_final_panel": int(len(results["final_panel_df"])),
            "active_weights": results["active_weights"],
        },
        dirs["logs"] / "stage_counts.json",
    )

    print("\n" + "=" * 100)
    print("Stage completed successfully.")
    print(f"Main tables: {dirs['tables_main']}")
    print(f"Supplementary tables: {dirs['tables_supplementary']}")
    print(f"Main figures: {dirs['figures_main']}")
    print(f"Supplementary figures: {dirs['figures_supplementary']}")
    print(f"Artifacts: {dirs['artifacts']}")
    print(f"Reports: {dirs['reports']}")
    print(f"Logs: {dirs['logs']}")
    print("=" * 100)

    return results


# =============================================================================
# Execute
# =============================================================================

results = main_notebook()

# Redesigning step 8 figures

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Tuple
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# =============================================================================
# Fallback config
# =============================================================================

if "cfg" not in globals():
    cfg = SimpleNamespace(
        similarity_kmer=3,
        figure_export=SimpleNamespace(
            export_pdf=True,
            export_png=True,
            export_tif=True,
            png_dpi=300,
            tif_dpi=600,
        )
    )

# =============================================================================
# Auto-detect dirs if missing
# =============================================================================

if "dirs" not in globals():
    base_root = Path("/home/data3/Moe/nature_computational2")
    candidate_dirs = ["step8_v1", "step8"]

    step8_root = None
    for name in candidate_dirs:
        p = base_root / name
        if p.exists():
            step8_root = p
            break

    if step8_root is None:
        raise FileNotFoundError(
            "Could not find step8_v1 or step8 under /home/data3/Moe/nature_computational2"
        )

    dirs = {
        "root": step8_root,
        "tables_main": step8_root / "tables_main",
        "tables_supplementary": step8_root / "tables_supplementary",
        "figures_main": step8_root / "figures_main",
        "figures_supplementary": step8_root / "figures_supplementary",
        "artifacts": step8_root / "artifacts",
        "reports": step8_root / "reports",
        "logs": step8_root / "logs",
    }

    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Required object check
# =============================================================================

if "results" not in globals():
    raise NameError(
        "Missing required object: results\n"
        "Please run Step 8 first so that the results dictionary exists."
    )

# =============================================================================
# Fallback Jaccard function
# =============================================================================

if "jaccard_kmer_similarity" not in globals():
    def _make_kmers(seq: str, k: int = 3) -> set:
        seq = str(seq).strip().upper()
        if len(seq) < k:
            return set()
        return {seq[i:i+k] for i in range(len(seq) - k + 1)}

    def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
        s1 = _make_kmers(seq1, k=k)
        s2 = _make_kmers(seq2, k=k)
        if len(s1) == 0 and len(s2) == 0:
            return 1.0
        union = len(s1 | s2)
        if union == 0:
            return 0.0
        return len(s1 & s2) / union

# =============================================================================
# Palette
# =============================================================================

PALETTE = {
    "passed": "#59C38A",
    "shortlist": "#D90429",
    "final": "#1D3B73",
    "reference": "#A8B9DD",
    "filter": "#EA9A3D",
    "text": "#222222",
    "grid": "#D9D9D9",
    "spine": "#9A9A9A",
    "bg": "#FFFFFF",
    "density_light": "#F1F1F1",
    "density_dark": "#3A3A3A",
}

DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "step8_density",
    [PALETTE["density_light"], "#D0D0D0", "#8A8A8A", PALETTE["density_dark"]],
)

# =============================================================================
# Styling
# =============================================================================

def _style_publication():
    plt.rcParams.update({
        "figure.facecolor": PALETTE["bg"],
        "axes.facecolor": PALETTE["bg"],
        "savefig.facecolor": PALETTE["bg"],
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.grid": True,
        "grid.color": PALETTE["grid"],
        "grid.linewidth": 0.6,
        "grid.alpha": 1.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": PALETTE["spine"],
        "axes.labelcolor": PALETTE["text"],
        "xtick.color": PALETTE["text"],
        "ytick.color": PALETTE["text"],
        "text.color": PALETTE["text"],
        "axes.titlecolor": PALETTE["text"],
    })

def style_axis(ax, grid_axis: str = "y"):
    ax.set_facecolor(PALETTE["bg"])
    ax.spines["left"].set_color(PALETTE["spine"])
    ax.spines["bottom"].set_color(PALETTE["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(axis="both", colors=PALETTE["text"], width=0.8, length=4)
    ax.grid(True, axis=grid_axis, color=PALETTE["grid"], linewidth=0.6)

def add_panel_label(ax, label: str):
    ax.text(
        -0.08, 1.06, label,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="bottom",
        ha="left",
        color=PALETTE["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path, cfg_local) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    fe = cfg_local.figure_export
    if getattr(fe, "export_pdf", True):
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if getattr(fe, "export_png", True):
        fig.savefig(out_base.with_suffix(".png"), dpi=getattr(fe, "png_dpi", 300), bbox_inches="tight")
    if getattr(fe, "export_tif", True):
        fig.savefig(out_base.with_suffix(".tif"), dpi=getattr(fe, "tif_dpi", 600), bbox_inches="tight")
    plt.close(fig)

# =============================================================================
# Utils
# =============================================================================

def _safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _group_values(df: pd.DataFrame, col: str) -> np.ndarray:
    if col not in df.columns:
        return np.array([], dtype=float)
    return _safe_numeric(df[col]).dropna().values

def _violin_with_points(ax, groups: List[np.ndarray], labels: List[str], colors: List[str],
                        ylabel: str, ylim: Tuple[float, float] | None = None):
    positions = np.arange(1, len(groups) + 1)
    parts = ax.violinplot(
        groups,
        positions=positions,
        widths=0.75,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    for body, color in zip(parts["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.92)
        body.set_linewidth(0.8)

    rng = np.random.default_rng(42)

    for x, vals, color in zip(positions, groups, colors):
        if len(vals) == 0:
            continue

        q1, med, q3 = np.quantile(vals, [0.25, 0.5, 0.75])
        ax.plot([x, x], [q1, q3], color=PALETTE["text"], lw=1.1, zorder=4)
        ax.scatter([x], [med], s=22, facecolors="white", edgecolors=PALETTE["text"], linewidths=0.8, zorder=5)

        nshow = min(len(vals), 12)
        if nshow > 1:
            show_vals = rng.choice(vals, size=nshow, replace=False)
            jitter = np.linspace(-0.08, 0.08, nshow)
            ax.scatter(
                np.full(nshow, x) + jitter,
                show_vals,
                s=10,
                color=color,
                alpha=0.45,
                edgecolors="white",
                linewidths=0.25,
                zorder=3,
            )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    style_axis(ax, grid_axis="y")

def _annotate_simple_effect(ax, x1: float, x2: float, y: float, label: str):
    ax.plot([x1, x1, x2, x2], [y - 0.01, y, y, y - 0.01], color=PALETTE["text"], lw=0.8)
    ax.text((x1 + x2) / 2, y + 0.01, label, ha="center", va="bottom", fontsize=6)

def _final_similarity_matrix(final_panel_df: pd.DataFrame, seq_col: str, id_col: str, similarity_kmer: int):
    seqs = final_panel_df[seq_col].astype(str).tolist()
    ids = final_panel_df[id_col].astype(str).tolist()
    n = len(seqs)
    sim = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            v = jaccard_kmer_similarity(seqs[i], seqs[j], k=similarity_kmer)
            sim[i, j] = v
            sim[j, i] = v
    return ids, sim

# =============================================================================
# Main figure redesign
# =============================================================================

def plot_step8_main_figure_redesign(results: Dict[str, Any], cfg_local, dirs: Dict[str, Path]) -> None:
    _style_publication()

    gen_df = results["gen_df"]
    eligible_df = results["eligible_df"]
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    filter_audit_df = results.get("filter_audit_df", pd.DataFrame())

    fig = plt.figure(figsize=(7.25, 8.1))
    gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.08], width_ratios=[1.0, 1.0], hspace=0.38, wspace=0.34)

    # a
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")

    labels = ["Generated pool", "Eligible", "Plausible set", "Shortlist", "Final panel"]
    vals = [len(gen_df), len(eligible_df), len(passed_df), len(shortlist_df), len(final_panel_df)]
    colors = ["#A5A5A5", PALETTE["filter"], PALETTE["passed"], PALETTE["shortlist"], PALETTE["final"]]

    y = np.arange(len(labels))
    bars = ax.barh(y, vals, color=colors, edgecolor=PALETTE["spine"], linewidth=0.8, height=0.72)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Number of peptides")
    ax.set_title("Candidate prioritization cascade", pad=6)
    style_axis(ax, grid_axis="x")
    ax.grid(False, axis="y")

    xmax = max(vals) * 1.05
    ax.set_xlim(0, xmax)

    for bar, v in zip(bars, vals):
        frac = 100.0 * v / vals[0] if vals[0] else 0.0
        ax.text(v + xmax * 0.008, bar.get_y() + bar.get_height() / 2, f"{v:,} ({frac:.1f}%)",
                va="center", ha="left", fontsize=8)

    if len(filter_audit_df) > 0:
        top = filter_audit_df.sort_values("removed_n", ascending=False).head(3)
        exclusion_text = "; ".join(
            [f"{str(r['filter_name']).replace('flag_', '').replace('_', ' ')} ({int(r['removed_n'])})"
             for _, r in top.iterrows()]
        )
        ax.text(0.0, -0.23, f"Top exclusions: {exclusion_text}",
                transform=ax.transAxes, fontsize=7, color="#555555", ha="left", va="top")

    # b
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")

    x = _safe_numeric(passed_df["novelty_score"])
    yv = _safe_numeric(passed_df["descriptor_plausibility_score"])

    hb = ax.hexbin(x, yv, gridsize=23, mincnt=1, cmap=DENSITY_CMAP, linewidths=0.0)
    ax.scatter(shortlist_df["novelty_score"], shortlist_df["descriptor_plausibility_score"],
               s=38, color=PALETTE["shortlist"], edgecolors="white", linewidths=0.45, zorder=4, label="Shortlist")
    ax.scatter(final_panel_df["novelty_score"], final_panel_df["descriptor_plausibility_score"],
               s=52, color=PALETTE["final"], edgecolors="white", linewidths=0.55, zorder=5, label="Final panel")

    ax.set_xlabel("Novelty score")
    ax.set_ylabel("Descriptor plausibility score")
    ax.set_title("Novelty–plausibility landscape", pad=6)
    style_axis(ax, grid_axis="both")
    ax.set_xlim(max(0.58, np.nanmin(x) - 0.01), min(0.99, np.nanmax(x) + 0.005))
    ax.set_ylim(max(0.10, np.nanmin(yv) - 0.02), min(0.86, np.nanmax(yv) + 0.01))

    cbar = fig.colorbar(hb, ax=ax, fraction=0.05, pad=0.03)
    cbar.set_label("Passed-pool density")
    cbar.outline.set_edgecolor(PALETTE["spine"])

    ax.legend(frameon=False, loc="lower left")

    # c
    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")

    metrics = [
        ("novelty_score", "Novelty"),
        ("descriptor_plausibility_score", "Plausibility"),
        ("diversity_score", "Diversity"),
        ("final_score", "Final score"),
    ]
    centers = np.arange(len(metrics)) * 2.0 + 1.0
    offsets = [-0.32, 0.0, 0.32]
    group_names = ["Passed", "Shortlist", "Final panel"]
    group_colors = [PALETTE["passed"], PALETTE["shortlist"], PALETTE["final"]]

    for cx, (metric, _) in zip(centers, metrics):
        for off, df, color in zip(offsets, [passed_df, shortlist_df, final_panel_df], group_colors):
            vals = _group_values(df, metric)
            if len(vals) == 0:
                continue

            parts = ax.violinplot([vals], positions=[cx + off], widths=0.26,
                                  showmeans=False, showmedians=False, showextrema=False)
            body = parts["bodies"][0]
            body.set_facecolor(color)
            body.set_edgecolor(color)
            body.set_alpha(0.95)
            body.set_linewidth(0.8)

            q1, med, q3 = np.quantile(vals, [0.25, 0.5, 0.75])
            ax.plot([cx + off, cx + off], [q1, q3], color=PALETTE["text"], lw=0.85, zorder=4)
            ax.scatter([cx + off], [med], s=18, facecolors="white", edgecolors=PALETTE["text"], linewidths=0.7, zorder=5)

    ax.set_xlim(0.2, centers[-1] + 0.9)
    ax.set_ylim(0.0, 1.02)
    ax.set_xticks(centers)
    ax.set_xticklabels([m[1] for m in metrics])
    ax.set_ylabel("Score")
    ax.set_title("Ranking-component distributions", pad=6)
    style_axis(ax, grid_axis="y")
    ax.legend(handles=[Patch(facecolor=c, edgecolor=c, label=n) for c, n in zip(group_colors, group_names)],
              frameon=False, loc="lower left", ncol=3, columnspacing=1.2, handletextpad=0.4)

    # d
    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")

    groups = [
        _group_values(passed_df, "diversity_score"),
        _group_values(shortlist_df, "diversity_score"),
        _group_values(final_panel_df, "diversity_score"),
    ]
    labels = ["Passed", "Shortlist", "Final panel"]

    _violin_with_points(ax, groups, labels, group_colors, ylabel="Diversity score", ylim=(0.0, 1.02))
    ax.set_title("Diversity shift across prioritization", pad=6)

    if all(len(v) > 0 for v in groups):
        ymax = min(0.98, max(np.nanmax(v) for v in groups) + 0.05)
        _annotate_simple_effect(ax, 1, 2, ymax, "preserved")
        _annotate_simple_effect(ax, 2, 3, min(1.0, ymax + 0.05), "preserved")

    fig.subplots_adjust(left=0.09, right=0.96, top=0.96, bottom=0.08)
    export_figure(fig, dirs["figures_main"] / "figure_8_prioritization_main_redesign", cfg_local)

# =============================================================================
# Supplementary redesign
# =============================================================================

def plot_step8_supplementary_redesign(results: Dict[str, Any], cfg_local, dirs: Dict[str, Path]) -> None:
    _style_publication()

    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    ref_df = results["ref_df"]
    filter_audit_df = results.get("filter_audit_df", pd.DataFrame())
    stability_df = results.get("stability_df", pd.DataFrame())
    recurrence_df = results.get("recurrence_df", pd.DataFrame())
    id_col = results["gen_colmap"]["id_col"]
    seq_col = results["gen_colmap"]["sequence_col"]

    # SFig 1
    fig = plt.figure(figsize=(4.6, 4.2))
    ax = fig.add_subplot(111)
    add_panel_label(ax, "a")

    fa = filter_audit_df.copy()
    fa = fa[fa["removed_n"] > 0].sort_values("removed_n", ascending=True)
    labels = [str(x).replace("flag_", "").replace("_", " ") for x in fa["filter_name"]]
    vals = fa["removed_n"].astype(int).tolist()

    y = np.arange(len(vals))
    bars = ax.barh(y, vals, color=PALETTE["filter"], edgecolor=PALETTE["spine"], linewidth=0.8, height=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Removed peptides")
    ax.set_title("Filter audit", pad=6)
    style_axis(ax, grid_axis="x")
    ax.grid(False, axis="y")

    xmax = max(vals) * 1.08 if len(vals) else 1
    ax.set_xlim(0, xmax)
    for b, v in zip(bars, vals):
        ax.text(v + xmax * 0.01, b.get_y() + b.get_height() / 2, f"{v}", va="center", ha="left", fontsize=7)

    fig.subplots_adjust(left=0.34, right=0.97, top=0.92, bottom=0.14)
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_1_filter_audit_redesign", cfg_local)

    # SFig 2
    fig = plt.figure(figsize=(8.2, 5.0))
    gs = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[1.4, 1.0], height_ratios=[1.0, 1.0], wspace=0.35, hspace=0.35)

    ax = fig.add_subplot(gs[:, 0])
    add_panel_label(ax, "a")

    if len(stability_df) > 0:
        scheme_names = sorted(set(stability_df["scheme_a"]).union(set(stability_df["scheme_b"])))
        mat = pd.DataFrame(np.eye(len(scheme_names)), index=scheme_names, columns=scheme_names)

        for _, r in stability_df.iterrows():
            mat.loc[r["scheme_a"], r["scheme_b"]] = r["jaccard_overlap"]
            mat.loc[r["scheme_b"], r["scheme_a"]] = r["jaccard_overlap"]

        im = ax.imshow(mat.values, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(np.arange(len(scheme_names)))
        ax.set_yticks(np.arange(len(scheme_names)))
        ax.set_xticklabels(scheme_names, rotation=28, ha="right")
        ax.set_yticklabels(scheme_names)

        for i in range(len(scheme_names)):
            for j in range(len(scheme_names)):
                val = mat.values[i, j]
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        color=("white" if val > 0.58 else PALETTE["text"]), fontsize=8)

        ax.set_title("Scheme-overlap matrix", pad=6)
        cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
        cbar.set_label("Jaccard overlap")
        cbar.outline.set_edgecolor(PALETTE["spine"])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_color(PALETTE["spine"])
    else:
        ax.text(0.5, 0.5, "No stability data", ha="center", va="center")
        ax.axis("off")

    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")

    if len(recurrence_df) > 0 and "scheme_recurrence_n" in recurrence_df.columns:
        counts = recurrence_df["scheme_recurrence_n"].value_counts().sort_index()
        xs = counts.index.astype(int).tolist()
        ys = counts.values.tolist()

        bars = ax.bar(xs, ys, color=PALETTE["shortlist"], edgecolor=PALETTE["spine"], linewidth=0.8, width=0.75)
        ax.set_xlabel("Number of schemes containing candidate")
        ax.set_ylabel("Candidate count")
        ax.set_title("Candidate recurrence", pad=6)
        style_axis(ax, grid_axis="y")

        for b, v in zip(bars, ys):
            ax.text(b.get_x() + b.get_width() / 2, v + max(ys) * 0.02, f"{v}", ha="center", va="bottom", fontsize=7)
    else:
        ax.text(0.5, 0.5, "No recurrence data", ha="center", va="center")
        ax.axis("off")

    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "c")

    if len(final_panel_df) >= 2:
        ids, sim = _final_similarity_matrix(final_panel_df, seq_col, id_col, cfg_local.similarity_kmer)
        im = ax.imshow(sim, vmin=0, vmax=1, cmap="Blues")

        short_ids = [s if len(s) <= 10 else s[:10] for s in ids]
        ax.set_xticks(np.arange(len(short_ids)))
        ax.set_yticks(np.arange(len(short_ids)))
        ax.set_xticklabels(short_ids, rotation=42, ha="right")
        ax.set_yticklabels(short_ids)
        ax.set_title("Final-panel pairwise similarity", pad=6)
        ax.grid(False)

        for spine in ax.spines.values():
            spine.set_color(PALETTE["spine"])

        cbar = fig.colorbar(im, ax=ax, fraction=0.055, pad=0.03)
        cbar.set_label("3-mer Jaccard similarity")
        cbar.outline.set_edgecolor(PALETTE["spine"])
    else:
        ax.text(0.5, 0.5, "Similarity unavailable", ha="center", va="center")
        ax.axis("off")

    fig.subplots_adjust(left=0.08, right=0.97, top=0.94, bottom=0.12)
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_2_robustness_diversity_redesign", cfg_local)

    # SFig 3
    fig = plt.figure(figsize=(8.4, 6.4))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.30)

    descriptor_specs = [
        ("length", "Length", (0, max(60, int(np.nanmax(_safe_numeric(ref_df.get("length", pd.Series([50]))))) + 2))),
        ("net_charge_proxy", "Net charge proxy", None),
        ("mean_hydropathy", "Mean hydropathy", None),
        ("entropy", "Sequence entropy", (0, 4.2)),
    ]
    labels = ["Passed", "Shortlist", "Final panel", "Reference ACP corpus"]
    colors = [PALETTE["passed"], PALETTE["shortlist"], PALETTE["final"], PALETTE["reference"]]
    dfs = [passed_df, shortlist_df, final_panel_df, ref_df]

    for i, (metric, title, ylim) in enumerate(descriptor_specs):
        ax = fig.add_subplot(gs[i // 2, i % 2])
        add_panel_label(ax, chr(ord("a") + i))
        groups = [_group_values(df, metric) for df in dfs]
        _violin_with_points(ax, groups, labels, colors, ylabel=title, ylim=ylim)
        ax.set_title(title, pad=5)

    fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.10)
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_8_3_descriptor_distributions_redesign", cfg_local)

# =============================================================================
# Run now
# =============================================================================

plot_step8_main_figure_redesign(results, cfg, dirs)
plot_step8_supplementary_redesign(results, cfg, dirs)

print("Redesigned Step 8 figures exported successfully.")
print("Using root:", dirs["root"])
print("Main figures folder:", dirs["figures_main"])
print("Supplementary figures folder:", dirs["figures_supplementary"])

main figure

In [ ]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# =============================================================================
# FALLBACKS
# =============================================================================

if "cfg" not in globals():
    cfg = SimpleNamespace(
        similarity_kmer=3,
        figure_export=SimpleNamespace(
            export_pdf=True,
            export_png=True,
            export_tif=True,
            png_dpi=300,
            tif_dpi=600,
        )
    )

if "dirs" not in globals():
    base_root = Path("/home/data3/Moe/nature_computational2")
    candidate_dirs = ["step8_v1", "step8"]
    step8_root = None
    for name in candidate_dirs:
        p = base_root / name
        if p.exists():
            step8_root = p
            break
    if step8_root is None:
        step8_root = Path.cwd() / "step8_v1"

    dirs = {
        "root": step8_root,
        "figures_main": step8_root / "figures_main",
        "figures_supplementary": step8_root / "figures_supplementary",
    }
    dirs["figures_main"].mkdir(parents=True, exist_ok=True)
    dirs["figures_supplementary"].mkdir(parents=True, exist_ok=True)

if "results" not in globals():
    raise NameError("Missing required object: results. Run Step 8 first.")

# =============================================================================
# REFINED ACCESSIBLE PALETTE
# =============================================================================

COLORS = {
    # pipeline stages
    "generated": "#A8A8A8",   # neutral grey
    "qc": "#E69F00",          # amber/orange

    # comparison groups: deliberately distinct
    "passed": "#2B7A78",      # blue-teal
    "shortlist": "#D55E00",   # vermilion
    "final": "#355C9A",       # deep blue

    # shared styling
    "bg": "#FFFFFF",
    "grid": "#D9D9D9",
    "spine": "#B8B8B8",
    "text": "#2A2A2A",
    "legend_bg": "#F3F3F3",
    "legend_edge": "#CDCDCD",
}

# =============================================================================
# STYLE
# =============================================================================

def _style_publication() -> None:
    plt.rcParams.update({
        "figure.facecolor": COLORS["bg"],
        "axes.facecolor": COLORS["bg"],
        "savefig.facecolor": COLORS["bg"],
        "font.family": "DejaVu Sans",
        "font.size": 8.4,
        "axes.titlesize": 10.2,
        "axes.labelsize": 9.5,
        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "legend.fontsize": 9.8,
        "axes.edgecolor": COLORS["spine"],
        "axes.labelcolor": COLORS["text"],
        "xtick.color": COLORS["text"],
        "ytick.color": COLORS["text"],
        "text.color": COLORS["text"],
        "axes.titlecolor": COLORS["text"],
        "axes.grid": False,
    })

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)

    if grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.8, zorder=0)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.8, zorder=0)
    elif grid_axis == "both":
        ax.grid(True, color=COLORS["grid"], linewidth=0.8, zorder=0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(COLORS["spine"])
        ax.spines[side].set_linewidth(0.9)

    ax.tick_params(axis="both", width=0.9, length=4, colors=COLORS["text"])

def add_panel_label(ax, label: str, x: float = -0.070, y: float = 1.025) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=17.5,
        fontweight="bold",
        va="bottom",
        ha="left",
        color=COLORS["text"],
        clip_on=False,
    )

def export_figure(fig: plt.Figure, out_base: Path, cfg_local) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    fe = cfg_local.figure_export

    if getattr(fe, "export_pdf", True):
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if getattr(fe, "export_png", True):
        fig.savefig(
            out_base.with_suffix(".png"),
            bbox_inches="tight",
            dpi=getattr(fe, "png_dpi", 300),
        )
    if getattr(fe, "export_tif", True):
        fig.savefig(
            out_base.with_suffix(".tiff"),
            bbox_inches="tight",
            dpi=getattr(fe, "tif_dpi", 600),
        )
    plt.close(fig)

# =============================================================================
# UTILS
# =============================================================================

def _safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _clean(arr) -> np.ndarray:
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]

def _group_values(df: pd.DataFrame, col: str) -> np.ndarray:
    if col not in df.columns:
        return np.array([], dtype=float)
    return _clean(_safe_numeric(df[col]).values)

def _summary_stat(arr: np.ndarray, mode: str = "median") -> float:
    arr = _clean(arr)
    if len(arr) == 0:
        return np.nan
    return float(np.median(arr)) if mode == "median" else float(np.mean(arr))

def _bootstrap_ci(
    arr: np.ndarray,
    mode: str = "median",
    n_boot: int = 1000,
    seed: int = 42,
) -> Tuple[float, float]:
    arr = _clean(arr)
    if len(arr) == 0:
        return (np.nan, np.nan)
    if len(arr) == 1:
        v = float(arr[0])
        return (v, v)

    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        stats.append(np.median(sample) if mode == "median" else np.mean(sample))
    low, high = np.percentile(stats, [2.5, 97.5])
    return float(low), float(high)

# =============================================================================
# MAIN FIGURE
# =============================================================================

def plot_step8_main_figure_nature_final(
    results: Dict[str, Any],
    cfg_local,
    dirs_local: Dict[str, Path],
) -> None:
    _style_publication()

    gen_df = results["gen_df"]
    eligible_df = results["eligible_df"]
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]

    groups = [passed_df, shortlist_df, final_panel_df]
    group_colors = [COLORS["passed"], COLORS["shortlist"], COLORS["final"]]

    fig = plt.figure(figsize=(12.0, 4.45))
    gs = gridspec.GridSpec(
        1, 3,
        figure=fig,
        width_ratios=[1.15, 1.0, 1.0],
        wspace=0.48,
    )

    # -------------------------------------------------------------------------
    # a) Selection audit
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[0, 0])
    add_panel_label(ax_a, "a")

    stage_labels = [
        "Generated",
        "QC-passed",
        "Descriptor-plausible",
        "Shortlist",
        "Final panel",
    ]
    stage_counts = np.array([
        len(gen_df),
        len(eligible_df),
        len(passed_df),
        len(shortlist_df),
        len(final_panel_df),
    ], dtype=float)

    stage_colors = [
        COLORS["generated"],
        COLORS["qc"],
        COLORS["passed"],
        COLORS["shortlist"],
        COLORS["final"],
    ]

    x = np.arange(len(stage_labels))
    bars = ax_a.bar(
        x,
        stage_counts,
        width=0.56,
        color=stage_colors,
        edgecolor="none",
        linewidth=0,
        zorder=3,
    )

    ymax = max(stage_counts) * 1.16 if max(stage_counts) > 0 else 1.0
    ax_a.set_ylim(0, ymax)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels(stage_labels, rotation=90, ha="center", va="top")
    ax_a.tick_params(axis="x", pad=6)
    ax_a.set_ylabel("Number of peptides")
    ax_a.set_title("Selection audit", pad=7)
    style_axis(ax_a, "y")
    ax_a.grid(False, axis="x")

    # annotate the first 3 dominant bars above the bars
    for i in range(3):
        bar = bars[i]
        n = int(stage_counts[i])
        frac = 100.0 * n / stage_counts[0] if stage_counts[0] > 0 else 0.0
        ax_a.text(
            bar.get_x() + bar.get_width() / 2,
            stage_counts[i] + ymax * 0.011,
            f"{n:,}\n({frac:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=7.6,
        )

    # annotate tiny bars minimally near baseline
    for i in [3, 4]:
        bar = bars[i]
        n = int(stage_counts[i])
        frac = 100.0 * n / stage_counts[0] if stage_counts[0] > 0 else 0.0
        ax_a.text(
            bar.get_x() + bar.get_width() / 2,
            ymax * 0.010,
            f"{n}\n({frac:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=6.9,
        )

    # inset for tiny stages
    axins = inset_axes(
        ax_a,
        width="25%",
        height="31%",
        loc="upper right",
        bbox_to_anchor=(0.00, 0.00, 0.96, 0.95),
        bbox_transform=ax_a.transAxes,
        borderpad=0.60,
    )
    inset_idx = [3, 4]
    inset_x = np.arange(len(inset_idx))
    inset_counts = stage_counts[inset_idx]
    inset_colors = [stage_colors[i] for i in inset_idx]

    axins.bar(
        inset_x,
        inset_counts,
        width=0.52,
        color=inset_colors,
        edgecolor="none",
        linewidth=0,
        zorder=3,
    )
    axins.set_xticks(inset_x)
    axins.set_xticklabels(["Shortlist", "Final"], rotation=90, ha="center", va="top", fontsize=6.5)
    axins.set_ylim(0, max(inset_counts) * 1.42 if np.max(inset_counts) > 0 else 1.0)
    axins.tick_params(axis="y", labelsize=6.2, length=2.4, width=0.65)
    axins.tick_params(axis="x", pad=1.4, length=2.4, width=0.65)
    axins.yaxis.grid(True, color=COLORS["grid"], linewidth=0.55, zorder=0)
    axins.spines["top"].set_visible(False)
    axins.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        axins.spines[side].set_color(COLORS["spine"])
        axins.spines[side].set_linewidth(0.68)

    for ix, val in zip(inset_x, inset_counts.astype(int)):
        axins.text(
            ix,
            val + max(inset_counts) * 0.05,
            f"{val}",
            ha="center",
            va="bottom",
            fontsize=6.0,
            color=COLORS["text"],
        )

    # -------------------------------------------------------------------------
    # b) Candidate support
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[0, 1])
    add_panel_label(ax_b, "b")

    metrics_b = [
        ("novelty_score", "Novelty"),
        ("descriptor_plausibility_score", "Plausibility"),
    ]
    x_base_b = np.array([0.0, 1.08])
    bar_width = 0.19
    inner_gap = 0.017

    for i, (df_group, color) in enumerate(zip(groups, group_colors)):
        vals, err_low, err_high = [], [], []
        for col, _ in metrics_b:
            arr = _group_values(df_group, col)
            center = _summary_stat(arr, mode="median")
            lo, hi = _bootstrap_ci(arr, mode="median", n_boot=1000, seed=42 + i)
            vals.append(center)
            err_low.append(center - lo if np.isfinite(center) and np.isfinite(lo) else np.nan)
            err_high.append(hi - center if np.isfinite(center) and np.isfinite(hi) else np.nan)

        offset = (i - 1) * (bar_width + inner_gap)
        xpos = x_base_b + offset

        ax_b.bar(
            xpos,
            vals,
            width=bar_width,
            color=color,
            edgecolor="none",
            zorder=3,
        )
        ax_b.errorbar(
            xpos,
            vals,
            yerr=[err_low, err_high],
            fmt="none",
            ecolor=COLORS["text"],
            elinewidth=0.70,
            capsize=2,
            capthick=0.70,
            zorder=4,
        )

    ax_b.set_xticks(x_base_b)
    ax_b.set_xticklabels([m[1] for m in metrics_b])
    ax_b.set_ylim(0.0, 1.02)
    ax_b.set_ylabel("Median support score")
    ax_b.set_title("Candidate support", pad=7)
    style_axis(ax_b, "y")
    ax_b.grid(False, axis="x")

    # -------------------------------------------------------------------------
    # c) Selection-stage enrichment
    # -------------------------------------------------------------------------
    ax_c = fig.add_subplot(gs[0, 2])
    add_panel_label(ax_c, "c")

    metrics_c = [
        ("descriptor_plausibility_score", "Plausibility"),
        ("final_score", "Final score"),
    ]
    x_base_c = np.array([0.0, 1.08])

    for i, (df_group, color) in enumerate(zip(groups, group_colors)):
        vals, err_low, err_high = [], [], []
        for col, _ in metrics_c:
            arr = _group_values(df_group, col)
            center = _summary_stat(arr, mode="median")
            lo, hi = _bootstrap_ci(arr, mode="median", n_boot=1000, seed=101 + i)
            vals.append(center)
            err_low.append(center - lo if np.isfinite(center) and np.isfinite(lo) else np.nan)
            err_high.append(hi - center if np.isfinite(center) and np.isfinite(hi) else np.nan)

        offset = (i - 1) * (bar_width + inner_gap)
        xpos = x_base_c + offset

        ax_c.bar(
            xpos,
            vals,
            width=bar_width,
            color=color,
            edgecolor="none",
            zorder=3,
        )
        ax_c.errorbar(
            xpos,
            vals,
            yerr=[err_low, err_high],
            fmt="none",
            ecolor=COLORS["text"],
            elinewidth=0.70,
            capsize=2,
            capthick=0.70,
            zorder=4,
        )

    ax_c.set_xticks(x_base_c)
    ax_c.set_xticklabels([m[1] for m in metrics_c])
    ax_c.set_ylim(0.0, 1.02)
    ax_c.set_ylabel("Median enrichment score")
    ax_c.set_title("Selection-stage enrichment", pad=7)
    style_axis(ax_c, "y")
    ax_c.grid(False, axis="x")

    # -------------------------------------------------------------------------
    # legend centered under b and c
    # -------------------------------------------------------------------------
    legend_handles = [
        Patch(facecolor=COLORS["passed"], edgecolor="none", label="Passed"),
        Patch(facecolor=COLORS["shortlist"], edgecolor="none", label="Shortlist"),
        Patch(facecolor=COLORS["final"], edgecolor="none", label="Final panel"),
    ]

    leg = fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.665, 0.012),
        ncol=3,
        frameon=True,
        columnspacing=1.08,
        handletextpad=0.50,
        borderpad=0.40,
        handlelength=1.50,
        fancybox=True,
        fontsize=9.8,
    )
    leg.get_frame().set_facecolor(COLORS["legend_bg"])
    leg.get_frame().set_edgecolor(COLORS["legend_edge"])
    leg.get_frame().set_linewidth(0.9)
    leg.get_frame().set_alpha(1.0)

    fig.subplots_adjust(left=0.065, right=0.992, top=0.90, bottom=0.22, wspace=0.48)
    export_figure(fig, dirs_local["figures_main"] / "figure_8_main_nature_final_v6", cfg_local)

# =============================================================================
# RUN
# =============================================================================

plot_step8_main_figure_nature_final(results, cfg, dirs)
print("Exported final polished main figure to:", dirs["figures_main"])

Sublimitary figures 

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Tuple
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# =============================================================================
# FALLBACK CONFIG
# =============================================================================

if "cfg" not in globals():
    cfg = SimpleNamespace(
        similarity_kmer=3,
        figure_export=SimpleNamespace(
            export_pdf=True,
            export_png=True,
            export_tif=True,
            png_dpi=300,
            tif_dpi=600,
        )
    )

if "dirs" not in globals():
    base_root = Path("/home/data3/Moe/nature_computational2")
    candidate_dirs = ["step8_v1", "step8"]

    step8_root = None
    for name in candidate_dirs:
        p = base_root / name
        if p.exists():
            step8_root = p
            break

    if step8_root is None:
        raise FileNotFoundError(
            "Could not find step8_v1 or step8 under /home/data3/Moe/nature_computational2"
        )

    dirs = {
        "root": step8_root,
        "tables_main": step8_root / "tables_main",
        "tables_supplementary": step8_root / "tables_supplementary",
        "figures_main": step8_root / "figures_main",
        "figures_supplementary": step8_root / "figures_supplementary",
        "artifacts": step8_root / "artifacts",
        "reports": step8_root / "reports",
        "logs": step8_root / "logs",
    }

    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

if "results" not in globals():
    raise NameError(
        "Missing required object: results\n"
        "Please run Step 8 first so that the results dictionary exists."
    )

# =============================================================================
# FALLBACK JACCARD FUNCTION
# =============================================================================

if "jaccard_kmer_similarity" not in globals():
    def _make_kmers(seq: str, k: int = 3) -> set:
        seq = str(seq).strip().upper()
        if len(seq) < k:
            return set()
        return {seq[i:i + k] for i in range(len(seq) - k + 1)}

    def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
        s1 = _make_kmers(seq1, k=k)
        s2 = _make_kmers(seq2, k=k)
        if len(s1) == 0 and len(s2) == 0:
            return 1.0
        union = len(s1 | s2)
        if union == 0:
            return 0.0
        return len(s1 & s2) / union

# =============================================================================
# PALETTE MATCHED TO FINAL MAIN FIGURE
# =============================================================================

PALETTE = {
    "generated": "#A8A8A8",
    "qc": "#E69F00",
    "passed": "#2B7A78",
    "shortlist": "#D55E00",
    "final": "#355C9A",
    "reference": "#B7C7E6",

    "bg": "#FFFFFF",
    "grid": "#D9D9D9",
    "spine": "#B8B8B8",
    "text": "#2A2A2A",
    "legend_bg": "#F3F3F3",
    "legend_edge": "#CDCDCD",

    "density_light": "#F2F2F2",
    "density_mid": "#B9C7D8",
    "density_dark": "#2E5F88",
}

DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "step8_density",
    [PALETTE["density_light"], PALETTE["density_mid"], PALETTE["density_dark"]],
)

# =============================================================================
# STYLE
# =============================================================================

def _style_publication():
    plt.rcParams.update({
        "figure.facecolor": PALETTE["bg"],
        "axes.facecolor": PALETTE["bg"],
        "savefig.facecolor": PALETTE["bg"],
        "font.family": "DejaVu Sans",
        "font.size": 8.4,
        "axes.titlesize": 10.2,
        "axes.labelsize": 9.5,
        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "legend.fontsize": 8.8,
        "axes.edgecolor": PALETTE["spine"],
        "axes.labelcolor": PALETTE["text"],
        "xtick.color": PALETTE["text"],
        "ytick.color": PALETTE["text"],
        "text.color": PALETTE["text"],
        "axes.titlecolor": PALETTE["text"],
        "axes.grid": False,
    })

def style_axis(ax, grid_axis: str = "y"):
    ax.set_facecolor(PALETTE["bg"])
    ax.set_axisbelow(True)

    if grid_axis == "y":
        ax.yaxis.grid(True, color=PALETTE["grid"], linewidth=0.8, zorder=0)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PALETTE["grid"], linewidth=0.8, zorder=0)
    elif grid_axis == "both":
        ax.grid(True, color=PALETTE["grid"], linewidth=0.8, zorder=0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PALETTE["spine"])
        ax.spines[side].set_linewidth(0.9)

    ax.tick_params(axis="both", colors=PALETTE["text"], width=0.9, length=4)

def add_panel_label(ax, label: str):
    ax.text(
        -0.08, 1.03, label,
        transform=ax.transAxes,
        fontsize=17.5,
        fontweight="bold",
        va="bottom",
        ha="left",
        color=PALETTE["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path, cfg_local) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    fe = cfg_local.figure_export

    if getattr(fe, "export_pdf", True):
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if getattr(fe, "export_png", True):
        fig.savefig(
            out_base.with_suffix(".png"),
            dpi=getattr(fe, "png_dpi", 300),
            bbox_inches="tight",
        )
    if getattr(fe, "export_tif", True):
        fig.savefig(
            out_base.with_suffix(".tif"),
            dpi=getattr(fe, "tif_dpi", 600),
            bbox_inches="tight",
        )
    plt.close(fig)

# =============================================================================
# UTILS
# =============================================================================

def _safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _group_values(df: pd.DataFrame, col: str) -> np.ndarray:
    if col not in df.columns:
        return np.array([], dtype=float)
    return _safe_numeric(df[col]).dropna().values

def _clean(arr) -> np.ndarray:
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]

def _violin_with_points(
    ax,
    groups: List[np.ndarray],
    colors: List[str],
    ylabel: str,
    ylim: Tuple[float, float] | None = None,
):
    positions = np.arange(1, len(groups) + 1)

    parts = ax.violinplot(
        groups,
        positions=positions,
        widths=0.58,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        bw_method=0.22,
    )

    for body, color in zip(parts["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.90)
        body.set_linewidth(0.0)

    rng = np.random.default_rng(42)

    for x, vals, color in zip(positions, groups, colors):
        vals = _clean(vals)
        if len(vals) == 0:
            continue

        q1, med, q3 = np.quantile(vals, [0.25, 0.5, 0.75])
        ax.plot([x, x], [q1, q3], color=PALETTE["text"], lw=0.85, zorder=4)
        ax.scatter(
            [x], [med],
            s=22,
            facecolors="white",
            edgecolors=PALETTE["text"],
            linewidths=0.75,
            zorder=5,
        )

        nshow = min(len(vals), 10)
        if nshow > 1:
            show_vals = rng.choice(vals, size=nshow, replace=False)
            jitter = np.linspace(-0.06, 0.06, nshow)
            ax.scatter(
                np.full(nshow, x) + jitter,
                show_vals,
                s=9,
                color=color,
                alpha=0.28,
                edgecolors="white",
                linewidths=0.20,
                zorder=3,
            )

    ax.set_xticks(positions)
    ax.set_xticklabels([])
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    style_axis(ax, grid_axis="y")

# =============================================================================
# SUPPLEMENTARY FIGURES
# =============================================================================

def plot_step8_supplementary_final(results: Dict[str, Any], cfg_local, dirs: Dict[str, Path]) -> None:
    _style_publication()

    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    ref_df = results["ref_df"]
    stability_df = results.get("stability_df", pd.DataFrame())
    recurrence_df = results.get("recurrence_df", pd.DataFrame())

    # -------------------------------------------------------------------------
    # Supplementary robustness figure: 2 panels only
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(8.2, 4.5))
    gs = gridspec.GridSpec(
        1, 2, figure=fig,
        width_ratios=[1.35, 0.85],
        wspace=0.34,
    )

    # a: scheme-overlap matrix
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")

    if len(stability_df) > 0:
        scheme_names = sorted(set(stability_df["scheme_a"]).union(set(stability_df["scheme_b"])))
        mat = pd.DataFrame(np.eye(len(scheme_names)), index=scheme_names, columns=scheme_names)

        for _, r in stability_df.iterrows():
            mat.loc[r["scheme_a"], r["scheme_b"]] = r["jaccard_overlap"]
            mat.loc[r["scheme_b"], r["scheme_a"]] = r["jaccard_overlap"]

        im = ax.imshow(mat.values, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(np.arange(len(scheme_names)))
        ax.set_yticks(np.arange(len(scheme_names)))

        # straight labels
        ax.set_xticklabels(scheme_names, rotation=0, ha="center")
        ax.set_yticklabels(scheme_names)

        for i in range(len(scheme_names)):
            for j in range(len(scheme_names)):
                val = mat.values[i, j]
                ax.text(
                    j, i, f"{val:.2f}",
                    ha="center", va="center",
                    color=("white" if val > 0.56 else PALETTE["text"]),
                    fontsize=7.0,
                )

        ax.set_title("Scheme-overlap matrix", pad=7)
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_color(PALETTE["spine"])
            spine.set_linewidth(0.8)

        cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
        cbar.set_label("Jaccard overlap")
        cbar.outline.set_edgecolor(PALETTE["spine"])
    else:
        ax.text(0.5, 0.5, "No stability data", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    # b: candidate recurrence
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")

    if len(recurrence_df) > 0 and "scheme_recurrence_n" in recurrence_df.columns:
        counts = recurrence_df["scheme_recurrence_n"].value_counts().sort_index()
        xs = counts.index.astype(int).tolist()
        ys = counts.values.tolist()

        bars = ax.bar(
            xs, ys,
            color=PALETTE["shortlist"],
            edgecolor="none",
            linewidth=0,
            width=0.66,
            zorder=3,
        )
        ax.set_xlabel("Number of schemes")
        ax.set_ylabel("Candidate count")
        ax.set_title("Candidate recurrence", pad=7)
        style_axis(ax, grid_axis="y")
        ax.grid(False, axis="x")

        ymax = max(ys) * 1.14 if len(ys) else 1
        ax.set_ylim(0, ymax)
        for b, v in zip(bars, ys):
            ax.text(
                b.get_x() + b.get_width() / 2,
                v + ymax * 0.020,
                f"{v}",
                ha="center",
                va="bottom",
                fontsize=8.6,
                fontweight="medium",
            )
    else:
        ax.text(0.5, 0.5, "No recurrence data", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    fig.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.14)
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_9ab_robustness_final", cfg_local)

    # -------------------------------------------------------------------------
    # Supplementary descriptor distributions
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(8.6, 6.2))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.30, wspace=0.28)

    descriptor_specs = [
        ("length", "Length", None),
        ("net_charge_proxy", "Net charge proxy", None),
        ("mean_hydropathy", "Mean hydropathy", None),
        ("entropy", "Sequence entropy", (0, 4.2)),
    ]

    colors = [PALETTE["passed"], PALETTE["shortlist"], PALETTE["final"], PALETTE["reference"]]
    dfs = [passed_df, shortlist_df, final_panel_df, ref_df]

    for i, (metric, title, ylim) in enumerate(descriptor_specs):
        ax = fig.add_subplot(gs[i // 2, i % 2])
        add_panel_label(ax, chr(ord("a") + i))

        groups = [_group_values(df, metric) for df in dfs]
        _violin_with_points(ax, groups, colors, ylabel=title, ylim=ylim)
        ax.set_title(title, pad=6)

    legend_handles = [
        Patch(facecolor=PALETTE["passed"], edgecolor="none", label="Passed"),
        Patch(facecolor=PALETTE["shortlist"], edgecolor="none", label="Shortlist"),
        Patch(facecolor=PALETTE["final"], edgecolor="none", label="Final panel"),
        Patch(facecolor=PALETTE["reference"], edgecolor="none", label="Reference corpus"),
    ]

    leg = fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=4,
        frameon=True,
        columnspacing=1.2,
        handletextpad=0.45,
        borderpad=0.40,
        handlelength=1.5,
        fancybox=True,
        fontsize=8.8,
    )
    leg.get_frame().set_facecolor(PALETTE["legend_bg"])
    leg.get_frame().set_edgecolor(PALETTE["legend_edge"])
    leg.get_frame().set_linewidth(0.9)
    leg.get_frame().set_alpha(1.0)

    fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.14)
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_9cdef_descriptor_distributions_final", cfg_local)

# =============================================================================
# RUN
# =============================================================================

plot_step8_supplementary_final(results, cfg, dirs)

print("Final updated supplementary figures exported successfully.")
print("Using root:", dirs["root"])
print("Supplementary figures folder:", dirs["figures_supplementary"])

# Step 9 — External validation, interpretability, and downstream support

In [ ]:
from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE
# Step 9 — External validation, interpretability, and downstream support
# Full corrected and improved publication-oriented implementation
# =============================================================================

import json
import platform
import random
import hashlib
import warnings
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from itertools import groupby
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

# =============================================================================
# Global constants
# =============================================================================

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)

AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_TINY = set("AGSC")
AA_SMALL = set("AGSCVTNDP")

AA_PROPERTIES = {
    "A": {"hydro": 1.8, "mw": 89.09},
    "C": {"hydro": 2.5, "mw": 121.16},
    "D": {"hydro": -3.5, "mw": 133.10},
    "E": {"hydro": -3.5, "mw": 147.13},
    "F": {"hydro": 2.8, "mw": 165.19},
    "G": {"hydro": -0.4, "mw": 75.07},
    "H": {"hydro": -3.2, "mw": 155.16},
    "I": {"hydro": 4.5, "mw": 131.17},
    "K": {"hydro": -3.9, "mw": 146.19},
    "L": {"hydro": 3.8, "mw": 131.17},
    "M": {"hydro": 1.9, "mw": 149.21},
    "N": {"hydro": -3.5, "mw": 132.12},
    "P": {"hydro": -1.6, "mw": 115.13},
    "Q": {"hydro": -3.5, "mw": 146.15},
    "R": {"hydro": -4.5, "mw": 174.20},
    "S": {"hydro": -0.8, "mw": 105.09},
    "T": {"hydro": -0.7, "mw": 119.12},
    "V": {"hydro": 4.2, "mw": 117.15},
    "W": {"hydro": -0.9, "mw": 204.23},
    "Y": {"hydro": -1.3, "mw": 181.19},
}

PROJECT_COLORS = {
    "hydramp_red": "#C7001E",
    "deep_red": "#E31A1C",
    "dark_maroon": "#7F0000",
    "pepcvae_navy": "#1F3B68",
    "basic_blue": "#A8B9DC",
    "amp_green": "#57C78D",
    "dean_pink": "#D96A99",
    "muller_cyan": "#71CDD7",
    "gan_orange": "#F29D3E",
    "joker_teal": "#0E6766",
    "purple": "#6A2C91",
    "gold": "#F0C43C",
    "gray_light": "#D9D9D9",
    "gray_mid": "#A0A0A0",
    "gray_dark": "#4A4A4A",
    "bg": "#FFFFFF",
    "grid": "#E8E8E8",
    "spine": "#BFBFBF",
    "text": "#222222",
}

GROUP_COLORS = {
    "Passed": PROJECT_COLORS["gan_orange"],
    "Shortlist": PROJECT_COLORS["hydramp_red"],
    "Final panel": PROJECT_COLORS["pepcvae_navy"],
    "Reference ACP corpus": PROJECT_COLORS["basic_blue"],
}

# =============================================================================
# Matplotlib style
# =============================================================================

def apply_project_matplotlib_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PROJECT_COLORS["bg"],
        "axes.facecolor": PROJECT_COLORS["bg"],
        "savefig.facecolor": PROJECT_COLORS["bg"],
        "axes.edgecolor": PROJECT_COLORS["spine"],
        "axes.labelcolor": PROJECT_COLORS["text"],
        "xtick.color": PROJECT_COLORS["text"],
        "ytick.color": PROJECT_COLORS["text"],
        "text.color": PROJECT_COLORS["text"],
        "axes.titlecolor": PROJECT_COLORS["text"],
        "grid.color": PROJECT_COLORS["grid"],
        "axes.grid": True,
        "grid.linestyle": "-",
        "grid.linewidth": 0.55,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 8,
        "font.family": "DejaVu Sans",
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    })

apply_project_matplotlib_style()

# =============================================================================
# Config
# =============================================================================

@dataclass
class FigureExportConfig:
    png_dpi: int = 300
    tif_dpi: int = 600
    export_pdf: bool = True
    export_png: bool = True
    export_tif: bool = True
    single_col_width_in: float = 3.5
    double_col_width_in: float = 7.1

@dataclass
class Step9Config:
    project_root: str = "/home/data3/Moe/nature_computational2/"
    step1_dir: str = "step1"
    step8_dir: str = "step8_v1"
    step9_dir: str = "step9_v1"

    step1_reference_file: Optional[str] = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"

    step8_passed_file: Optional[str] = None
    step8_shortlist_file: Optional[str] = None
    step8_final_panel_file: Optional[str] = None

    # Optional file containing external support scores
    step9_external_support_file: Optional[str] = None

    sequence_col_candidates: Tuple[str, ...] = ("sequence", "peptide", "seq", "peptide_sequence")
    id_col_candidates: Tuple[str, ...] = ("generated_id", "candidate_id", "peptide_id", "id")

    requested_length_bin_col_candidates: Tuple[str, ...] = (
        "requested_length_bin", "target_length_bin", "condition_length_bin", "length_bin"
    )
    requested_charge_bin_col_candidates: Tuple[str, ...] = (
        "requested_charge_bin", "target_charge_bin", "condition_charge_bin", "net_charge_bin"
    )
    requested_hydrophobicity_bin_col_candidates: Tuple[str, ...] = (
        "requested_hydrophobicity_bin", "target_hydrophobicity_bin",
        "condition_hydrophobicity_bin", "hydrophobicity_bin"
    )

    requested_length_min_col_candidates: Tuple[str, ...] = ("requested_length_min", "target_length_min")
    requested_length_max_col_candidates: Tuple[str, ...] = ("requested_length_max", "target_length_max")
    requested_charge_min_col_candidates: Tuple[str, ...] = ("requested_charge_min", "target_charge_min")
    requested_charge_max_col_candidates: Tuple[str, ...] = ("requested_charge_max", "target_charge_max")
    requested_hydrophobicity_min_col_candidates: Tuple[str, ...] = ("requested_hydrophobicity_min", "target_hydrophobicity_min")
    requested_hydrophobicity_max_col_candidates: Tuple[str, ...] = ("requested_hydrophobicity_max", "target_hydrophobicity_max")

    novelty_score_candidates: Tuple[str, ...] = ("novelty_score",)
    plausibility_score_candidates: Tuple[str, ...] = ("descriptor_plausibility_score", "plausibility_score")
    diversity_score_candidates: Tuple[str, ...] = ("diversity_score",)
    final_score_candidates: Tuple[str, ...] = ("final_score",)
    condition_fidelity_score_candidates: Tuple[str, ...] = ("condition_fidelity_score",)

    external_score_candidates: Tuple[str, ...] = (
        "external_support_score", "external_acp_score", "external_probability",
        "acp_probability_external", "independent_acp_score", "consensus_external_support"
    )
    external_binary_candidates: Tuple[str, ...] = (
        "external_support_positive", "external_positive", "consensus_external_positive"
    )
    external_predictor_prefixes: Tuple[str, ...] = (
        "ext_", "external_", "independent_", "acp_ext_", "pred_"
    )

    similarity_kmer: int = 3
    external_positive_threshold: float = 0.50
    enrichment_pseudocount: float = 1e-6
    bootstrap_iterations: int = 1000
    random_seed: int = 42

    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)

# =============================================================================
# Reproducibility / IO
# =============================================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def sha256_of_file(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def collect_environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp_utc": now_utc_iso(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
        "cwd": str(Path.cwd()),
    }

def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "logs": base_dir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.export_pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.export_tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)

# =============================================================================
# Helper utilities
# =============================================================================

def first_existing_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = False, label: str = "") -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not resolve required column for {label}. Tried: {list(candidates)}")
    return None

def deduplicate_columns_keep_last(df: pd.DataFrame) -> pd.DataFrame:
    """
    If duplicate column names exist, keep the last occurrence.
    """
    if not df.columns.duplicated().any():
        return df
    return df.loc[:, ~df.columns.duplicated(keep="last")].copy()

def get_1d_column(df: pd.DataFrame, col: str) -> pd.Series:
    """
    Return a 1D Series even if duplicate column names still exist.
    """
    obj = df[col]
    if isinstance(obj, pd.DataFrame):
        return obj.iloc[:, -1]
    return obj

def clean_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def aa_fraction(seq: str, aa: str) -> float:
    return seq.count(aa) / len(seq) if seq else 0.0

def shannon_entropy(seq: str) -> float:
    if not seq:
        return 0.0
    counts = Counter(seq)
    probs = np.array(list(counts.values()), dtype=float) / len(seq)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, None))).sum())

def max_run_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    runs = [len(list(group)) for _, group in groupby(seq)]
    return float(max(runs) / len(seq))

def unique_residue_fraction(seq: str) -> float:
    return len(set(seq)) / len(seq) if seq else 0.0

def max_single_residue_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    counts = Counter(seq)
    return float(max(counts.values()) / len(seq))

def make_kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}

def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
    s1 = make_kmers(seq1, k)
    s2 = make_kmers(seq2, k)
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    union = len(s1 | s2)
    return 0.0 if union == 0 else len(s1 & s2) / union

def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    if not seq:
        desc = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
        desc.update({
            "length": 0.0,
            "mean_hydropathy": 0.0,
            "hydrophobic_frac": 0.0,
            "aromatic_frac": 0.0,
            "positive_frac": 0.0,
            "negative_frac": 0.0,
            "polar_frac": 0.0,
            "net_charge_proxy": 0.0,
            "entropy": 0.0,
            "max_run_fraction": 1.0,
            "unique_residue_fraction": 0.0,
            "max_single_residue_fraction": 1.0,
        })
        return desc

    hydros = np.array([AA_PROPERTIES[a]["hydro"] for a in seq], dtype=float)
    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    desc.update({
        "length": float(len(seq)),
        "mean_hydropathy": float(hydros.mean()),
        "hydrophobic_frac": float(np.mean([a in AA_HYDROPHOBIC for a in seq])),
        "aromatic_frac": float(np.mean([a in AA_AROMATIC for a in seq])),
        "positive_frac": float(np.mean([a in AA_POSITIVE for a in seq])),
        "negative_frac": float(np.mean([a in AA_NEGATIVE for a in seq])),
        "polar_frac": float(np.mean([a in AA_POLAR for a in seq])),
        "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
        "entropy": shannon_entropy(seq),
        "max_run_fraction": max_run_fraction(seq),
        "unique_residue_fraction": unique_residue_fraction(seq),
        "max_single_residue_fraction": max_single_residue_fraction(seq),
    })
    return desc

def add_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    base = deduplicate_columns_keep_last(df.copy().reset_index(drop=True))
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in base[seq_col].astype(str)])
    overlap = [c for c in desc_df.columns if c in base.columns and c != seq_col]
    if overlap:
        base = base.rename(columns={c: f"{c}_input" for c in overlap})
    base = pd.concat([base, desc_df.reset_index(drop=True)], axis=1)
    return deduplicate_columns_keep_last(base)

def infer_length_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 10:
        return "short"
    if x <= 20:
        return "medium"
    return "long"

def infer_charge_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= 2:
        return "low"
    if x <= 6:
        return "medium"
    return "high"

def infer_hydropathy_bin(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x < -1.0:
        return "low"
    if x <= 1.0:
        return "medium"
    return "high"

# =============================================================================
# File discovery
# =============================================================================

def find_best_file(root: Path, aliases: Sequence[str], required: bool = True) -> Optional[Path]:
    candidates = []
    for p in root.rglob("*.csv"):
        if p.name in aliases:
            candidates.append(p)
    if candidates:
        candidates = sorted(candidates, key=lambda x: x.stat().st_mtime, reverse=True)
        return candidates[0]
    if required:
        raise FileNotFoundError(f"Could not locate expected file(s) under {root}: {list(aliases)}")
    return None

def resolve_step8_files(step8_root: Path, cfg: Step9Config) -> Dict[str, Optional[Path]]:
    passed_file = Path(cfg.step8_passed_file).resolve() if cfg.step8_passed_file else find_best_file(
        step8_root,
        aliases=("table_s8_12_full_ranked_passed_pool.csv",),
        required=True
    )

    shortlist_file = Path(cfg.step8_shortlist_file).resolve() if cfg.step8_shortlist_file else find_best_file(
        step8_root,
        aliases=("table_8_1_shortlist_main.csv",),
        required=True
    )

    final_panel_file = Path(cfg.step8_final_panel_file).resolve() if cfg.step8_final_panel_file else find_best_file(
        step8_root,
        aliases=("table_8_2_final_diverse_panel.csv", "table_s8_10_final_panel_rationale.csv"),
        required=True
    )

    external_support_file = None
    if cfg.step9_external_support_file is not None:
        external_support_file = Path(cfg.step9_external_support_file).resolve()
    else:
        step9_external_dir = step8_root.parent / "step9_external_inputs"
        if step9_external_dir.exists():
            files = sorted(step9_external_dir.rglob("*.csv"), key=lambda x: x.stat().st_mtime, reverse=True)
            if files:
                external_support_file = files[0]

    return {
        "passed_file": passed_file,
        "shortlist_file": shortlist_file,
        "final_panel_file": final_panel_file,
        "external_support_file": external_support_file,
    }

# =============================================================================
# Loading
# =============================================================================

def load_inputs(cfg: Step9Config) -> Dict[str, Any]:
    project_root = Path(cfg.project_root).resolve()
    step1_root = project_root / cfg.step1_dir
    step8_root = project_root / cfg.step8_dir
    step9_root = project_root / cfg.step9_dir

    if not step1_root.exists():
        raise FileNotFoundError(f"Step 1 directory not found: {step1_root}")
    if not step8_root.exists():
        raise FileNotFoundError(f"Step 8 directory not found: {step8_root}")

    step1_ref_path = Path(cfg.step1_reference_file).resolve() if cfg.step1_reference_file else (step1_root / "tables" / "step1_retained_dataset.csv").resolve()
    if not step1_ref_path.exists():
        raise FileNotFoundError(f"Step 1 reference file not found: {step1_ref_path}")

    files = resolve_step8_files(step8_root, cfg)

    ref_df = deduplicate_columns_keep_last(pd.read_csv(step1_ref_path))
    passed_df = deduplicate_columns_keep_last(pd.read_csv(files["passed_file"]))
    shortlist_df = deduplicate_columns_keep_last(pd.read_csv(files["shortlist_file"]))
    final_panel_df = deduplicate_columns_keep_last(pd.read_csv(files["final_panel_file"]))

    external_df = None
    if files["external_support_file"] is not None and files["external_support_file"].exists():
        external_df = deduplicate_columns_keep_last(pd.read_csv(files["external_support_file"]))

    return {
        "project_root": project_root,
        "step8_root": step8_root,
        "step9_root": step9_root,
        "step1_ref_path": step1_ref_path,
        "passed_file": files["passed_file"],
        "shortlist_file": files["shortlist_file"],
        "final_panel_file": files["final_panel_file"],
        "external_support_file": files["external_support_file"],
        "ref_df": ref_df,
        "passed_df": passed_df,
        "shortlist_df": shortlist_df,
        "final_panel_df": final_panel_df,
        "external_df": external_df,
    }

def prepare_reference_df(ref_df: pd.DataFrame, cfg: Step9Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df = deduplicate_columns_keep_last(ref_df.copy())
    seq_col = first_existing_column(df, cfg.sequence_col_candidates, required=True, label="reference sequence")
    df[seq_col] = get_1d_column(df, seq_col).astype(str).map(clean_sequence)
    df = df[df[seq_col] != ""].drop_duplicates(subset=[seq_col]).reset_index(drop=True)
    df = add_descriptors(df, seq_col)
    if "reference_id" not in df.columns:
        df["reference_id"] = [f"ref_{i:06d}" for i in range(len(df))]
    return df, {"sequence_col": seq_col, "id_col": "reference_id"}

def prepare_stage_df(df: pd.DataFrame, stage_name: str, cfg: Step9Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    out = deduplicate_columns_keep_last(df.copy())

    seq_col = first_existing_column(out, cfg.sequence_col_candidates, required=True, label=f"{stage_name} sequence")
    id_col = first_existing_column(out, cfg.id_col_candidates, required=False)
    if id_col is None:
        id_col = "generated_id"
        out[id_col] = [f"{stage_name.lower().replace(' ', '_')}_{i:06d}" for i in range(len(out))]

    out[seq_col] = get_1d_column(out, seq_col).astype(str).map(clean_sequence)
    out = out[out[seq_col] != ""].reset_index(drop=True)
    out = add_descriptors(out, seq_col)

    return out, {"sequence_col": seq_col, "id_col": id_col}

# =============================================================================
# External support
# =============================================================================

def discover_predictor_score_columns(df: pd.DataFrame, cfg: Step9Config) -> List[str]:
    cols = []
    for c in df.columns:
        c_low = c.lower()
        if any(c_low.startswith(pref) for pref in cfg.external_predictor_prefixes):
            vals = pd.to_numeric(get_1d_column(df, c), errors="coerce")
            if vals.notna().sum() > 0:
                cols.append(c)
    return cols

def attach_external_support(
    passed_df: pd.DataFrame,
    shortlist_df: pd.DataFrame,
    final_panel_df: pd.DataFrame,
    external_df: Optional[pd.DataFrame],
    passed_cols: Dict[str, str],
    cfg: Step9Config
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    id_col = passed_cols["id_col"]
    seq_col = passed_cols["sequence_col"]

    def _standardize(df: pd.DataFrame) -> pd.DataFrame:
        tmp = deduplicate_columns_keep_last(df.copy())
        if seq_col in tmp.columns:
            tmp[seq_col] = get_1d_column(tmp, seq_col).astype(str).map(clean_sequence)
        return tmp

    passed_df = _standardize(passed_df)
    shortlist_df = _standardize(shortlist_df)
    final_panel_df = _standardize(final_panel_df)

    merge_provenance = []

    if external_df is not None:
        ext = deduplicate_columns_keep_last(external_df.copy())
        ext_id_col = first_existing_column(ext, cfg.id_col_candidates, required=False)
        ext_seq_col = first_existing_column(ext, cfg.sequence_col_candidates, required=False)

        if ext_seq_col is not None:
            ext[ext_seq_col] = get_1d_column(ext, ext_seq_col).astype(str).map(clean_sequence)

        merge_key = None
        if ext_id_col is not None and id_col in passed_df.columns:
            merge_key = ("id", id_col, ext_id_col)
        elif ext_seq_col is not None and seq_col in passed_df.columns:
            merge_key = ("sequence", seq_col, ext_seq_col)

        if merge_key is not None:
            _, left_key, right_key = merge_key
            keep_cols = [c for c in ext.columns if c != right_key]
            ext = ext[[right_key] + keep_cols].drop_duplicates(subset=[right_key], keep="first")

            def _merge(base: pd.DataFrame) -> pd.DataFrame:
                merged = base.merge(ext, left_on=left_key, right_on=right_key, how="left", suffixes=("", "_external"))
                if right_key in merged.columns and right_key != left_key:
                    merged = merged.drop(columns=[right_key])
                return deduplicate_columns_keep_last(merged)

            passed_df = _merge(passed_df)
            shortlist_df = _merge(shortlist_df)
            final_panel_df = _merge(final_panel_df)
            merge_provenance.append({
                "mode": "merged_external_file",
                "merge_key_type": merge_key[0],
                "left_key": left_key,
                "right_key": right_key,
                "external_rows": int(len(ext)),
            })
        else:
            merge_provenance.append({
                "mode": "external_file_present_but_unmergeable",
                "merge_key_type": "none",
                "external_rows": int(len(ext)),
            })

    score_col = first_existing_column(passed_df, cfg.external_score_candidates, required=False)
    binary_col = first_existing_column(passed_df, cfg.external_binary_candidates, required=False)

    predictor_cols = discover_predictor_score_columns(passed_df, cfg)

    if score_col is None and predictor_cols:
        for dfi in (passed_df, shortlist_df, final_panel_df):
            dfi["consensus_external_support"] = pd.concat(
                [pd.to_numeric(get_1d_column(dfi, c), errors="coerce") for c in predictor_cols],
                axis=1
            ).mean(axis=1, skipna=True)
        score_col = "consensus_external_support"

    if binary_col is None and score_col is not None:
        for dfi in (passed_df, shortlist_df, final_panel_df):
            dfi["consensus_external_positive"] = (
                pd.to_numeric(get_1d_column(dfi, score_col), errors="coerce") >= cfg.external_positive_threshold
            ).astype(float)
        binary_col = "consensus_external_positive"

    if score_col is None:
        for dfi in (passed_df, shortlist_df, final_panel_df):
            dfi["consensus_external_support"] = np.nan
            dfi["consensus_external_positive"] = np.nan
        score_col = "consensus_external_support"
        binary_col = "consensus_external_positive"

    provenance_rows = [{
        "resolved_external_score_col": score_col,
        "resolved_external_binary_col": binary_col,
        "predictor_score_columns_used_for_consensus": ";".join(predictor_cols) if predictor_cols else "",
        "external_support_available": int(pd.to_numeric(get_1d_column(passed_df, score_col), errors="coerce").notna().any()),
        "merge_notes": json.dumps(merge_provenance, ensure_ascii=False),
    }]

    return (
        deduplicate_columns_keep_last(passed_df),
        deduplicate_columns_keep_last(shortlist_df),
        deduplicate_columns_keep_last(final_panel_df),
        pd.DataFrame(provenance_rows)
    )

# =============================================================================
# Conditioning fidelity
# =============================================================================

CONDITION_COLS = [
    "length_bin_match",
    "charge_bin_match",
    "hydropathy_bin_match",
    "length_range_match",
    "charge_range_match",
    "hydropathy_range_match",
    "condition_fidelity_score",
    "all_condition_match",
    "condition_mode",
]

def compute_condition_fidelity(df: pd.DataFrame, cfg: Step9Config) -> pd.DataFrame:
    base = deduplicate_columns_keep_last(df.copy())
    out = pd.DataFrame(index=base.index)

    length_bin_col = first_existing_column(base, cfg.requested_length_bin_col_candidates, required=False)
    charge_bin_col = first_existing_column(base, cfg.requested_charge_bin_col_candidates, required=False)
    hydro_bin_col = first_existing_column(base, cfg.requested_hydrophobicity_bin_col_candidates, required=False)

    length_min_col = first_existing_column(base, cfg.requested_length_min_col_candidates, required=False)
    length_max_col = first_existing_column(base, cfg.requested_length_max_col_candidates, required=False)
    charge_min_col = first_existing_column(base, cfg.requested_charge_min_col_candidates, required=False)
    charge_max_col = first_existing_column(base, cfg.requested_charge_max_col_candidates, required=False)
    hydro_min_col = first_existing_column(base, cfg.requested_hydrophobicity_min_col_candidates, required=False)
    hydro_max_col = first_existing_column(base, cfg.requested_hydrophobicity_max_col_candidates, required=False)

    has_any_condition = any([
        length_bin_col, charge_bin_col, hydro_bin_col,
        length_min_col, length_max_col, charge_min_col, charge_max_col, hydro_min_col, hydro_max_col
    ])

    if not has_any_condition:
        out["length_bin_match"] = np.nan
        out["charge_bin_match"] = np.nan
        out["hydropathy_bin_match"] = np.nan
        out["length_range_match"] = np.nan
        out["charge_range_match"] = np.nan
        out["hydropathy_range_match"] = np.nan
        out["condition_fidelity_score"] = np.nan
        out["all_condition_match"] = np.nan
        out["condition_mode"] = "missing_condition_metadata"
        return out

    if length_bin_col is not None:
        out["length_bin_match"] = (
            get_1d_column(base, length_bin_col).astype(str).str.lower().str.strip()
            == get_1d_column(base, "length").map(infer_length_bin).astype(str).str.lower().str.strip()
        ).astype(float)
    else:
        out["length_bin_match"] = np.nan

    if charge_bin_col is not None:
        out["charge_bin_match"] = (
            get_1d_column(base, charge_bin_col).astype(str).str.lower().str.strip()
            == get_1d_column(base, "net_charge_proxy").map(infer_charge_bin).astype(str).str.lower().str.strip()
        ).astype(float)
    else:
        out["charge_bin_match"] = np.nan

    if hydro_bin_col is not None:
        out["hydropathy_bin_match"] = (
            get_1d_column(base, hydro_bin_col).astype(str).str.lower().str.strip()
            == get_1d_column(base, "mean_hydropathy").map(infer_hydropathy_bin).astype(str).str.lower().str.strip()
        ).astype(float)
    else:
        out["hydropathy_bin_match"] = np.nan

    if length_min_col is not None and length_max_col is not None:
        out["length_range_match"] = (
            (get_1d_column(base, "length") >= pd.to_numeric(get_1d_column(base, length_min_col), errors="coerce")) &
            (get_1d_column(base, "length") <= pd.to_numeric(get_1d_column(base, length_max_col), errors="coerce"))
        ).astype(float)
    else:
        out["length_range_match"] = np.nan

    if charge_min_col is not None and charge_max_col is not None:
        out["charge_range_match"] = (
            (get_1d_column(base, "net_charge_proxy") >= pd.to_numeric(get_1d_column(base, charge_min_col), errors="coerce")) &
            (get_1d_column(base, "net_charge_proxy") <= pd.to_numeric(get_1d_column(base, charge_max_col), errors="coerce"))
        ).astype(float)
    else:
        out["charge_range_match"] = np.nan

    if hydro_min_col is not None and hydro_max_col is not None:
        out["hydropathy_range_match"] = (
            (get_1d_column(base, "mean_hydropathy") >= pd.to_numeric(get_1d_column(base, hydro_min_col), errors="coerce")) &
            (get_1d_column(base, "mean_hydropathy") <= pd.to_numeric(get_1d_column(base, hydro_max_col), errors="coerce"))
        ).astype(float)
    else:
        out["hydropathy_range_match"] = np.nan

    match_cols = [c for c in out.columns if c.endswith("_match")]
    score = out[match_cols].mean(axis=1, skipna=True)
    out["condition_fidelity_score"] = score
    out["all_condition_match"] = (out[match_cols].fillna(1.0).min(axis=1) == 1.0).astype(float)
    out["condition_mode"] = "available"
    return out

def summarize_condition_fidelity(df: pd.DataFrame, stage_name: str) -> pd.DataFrame:
    rows = []
    metrics = [
        "length_bin_match", "charge_bin_match", "hydropathy_bin_match",
        "length_range_match", "charge_range_match", "hydropathy_range_match",
        "condition_fidelity_score", "all_condition_match"
    ]
    for m in metrics:
        if m in df.columns:
            vals = pd.to_numeric(get_1d_column(df, m), errors="coerce")
            rows.append({
                "stage": stage_name,
                "metric": m,
                "n_non_missing": int(vals.notna().sum()),
                "mean": float(vals.mean()) if vals.notna().any() else np.nan,
                "median": float(vals.median()) if vals.notna().any() else np.nan,
            })
    return pd.DataFrame(rows)

# =============================================================================
# Nearest-neighbor contextualization
# =============================================================================

def compute_nearest_neighbor_context(
    query_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    query_seq_col: str,
    ref_seq_col: str,
    ref_id_col: str,
    k: int
) -> pd.DataFrame:
    rows = []
    ref_seqs = get_1d_column(reference_df, ref_seq_col).astype(str).tolist()
    ref_ids = get_1d_column(reference_df, ref_id_col).astype(str).tolist()

    for _, row in query_df.iterrows():
        qseq = str(row[query_seq_col])
        sims = [jaccard_kmer_similarity(qseq, rseq, k=k) for rseq in ref_seqs]
        if len(sims) == 0:
            rows.append({
                "nearest_reference_id": None,
                "nearest_reference_similarity": np.nan,
                "nearest_reference_sequence": None,
            })
            continue
        best_idx = int(np.argmax(sims))
        rows.append({
            "nearest_reference_id": ref_ids[best_idx],
            "nearest_reference_similarity": float(sims[best_idx]),
            "nearest_reference_sequence": ref_seqs[best_idx],
        })
    return pd.DataFrame(rows)

# =============================================================================
# Composition / motif summaries
# =============================================================================

def compute_aa_usage(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    counts = Counter()
    total = 0
    for s in get_1d_column(df, seq_col).astype(str):
        counts.update(s)
        total += len(s)
    rows = []
    for aa in AA_ALPHABET:
        rows.append({
            "amino_acid": aa,
            "count": int(counts.get(aa, 0)),
            "fraction": (counts.get(aa, 0) / total) if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)

def compute_kmer_usage(df: pd.DataFrame, seq_col: str, k: int = 3, top_k: int = 25) -> pd.DataFrame:
    counts = Counter()
    total = 0
    for s in get_1d_column(df, seq_col).astype(str):
        kmers = [s[i:i+k] for i in range(len(s) - k + 1)] if len(s) >= k else []
        counts.update(kmers)
        total += len(kmers)
    rows = []
    for kmer, c in counts.most_common(top_k):
        rows.append({
            "kmer": kmer,
            "count": int(c),
            "fraction": c / total if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)

def compute_residue_class_summary(df: pd.DataFrame, seq_col: str, stage: str) -> pd.DataFrame:
    rows = []
    for s in get_1d_column(df, seq_col).astype(str):
        if not s:
            continue
        rows.append({
            "stage": stage,
            "hydrophobic_frac": np.mean([aa in AA_HYDROPHOBIC for aa in s]),
            "aromatic_frac": np.mean([aa in AA_AROMATIC for aa in s]),
            "positive_frac": np.mean([aa in AA_POSITIVE for aa in s]),
            "negative_frac": np.mean([aa in AA_NEGATIVE for aa in s]),
            "polar_frac": np.mean([aa in AA_POLAR for aa in s]),
        })
    return pd.DataFrame(rows)

def compute_aa_enrichment(final_df: pd.DataFrame, ref_df: pd.DataFrame, final_seq_col: str, ref_seq_col: str, cfg: Step9Config) -> pd.DataFrame:
    final_usage = compute_aa_usage(final_df, final_seq_col).rename(columns={"fraction": "final_fraction", "count": "final_count"})
    ref_usage = compute_aa_usage(ref_df, ref_seq_col).rename(columns={"fraction": "reference_fraction", "count": "reference_count"})
    merged = final_usage.merge(ref_usage, on="amino_acid", how="outer")
    merged["log2_enrichment_vs_reference"] = np.log2(
        (merged["final_fraction"] + cfg.enrichment_pseudocount) /
        (merged["reference_fraction"] + cfg.enrichment_pseudocount)
    )
    return merged.sort_values("log2_enrichment_vs_reference", ascending=False).reset_index(drop=True)

def compute_kmer_enrichment(final_df: pd.DataFrame, ref_df: pd.DataFrame, final_seq_col: str, ref_seq_col: str, cfg: Step9Config, k: int = 3, top_n: int = 20) -> pd.DataFrame:
    final_kmers = compute_kmer_usage(final_df, final_seq_col, k=k, top_k=500).rename(columns={"fraction": "final_fraction", "count": "final_count"})
    ref_kmers = compute_kmer_usage(ref_df, ref_seq_col, k=k, top_k=2000).rename(columns={"fraction": "reference_fraction", "count": "reference_count"})
    merged = final_kmers.merge(ref_kmers, on="kmer", how="outer").fillna(0.0)
    merged["log2_enrichment_vs_reference"] = np.log2(
        (merged["final_fraction"] + cfg.enrichment_pseudocount) /
        (merged["reference_fraction"] + cfg.enrichment_pseudocount)
    )
    return merged.sort_values(["final_count", "log2_enrichment_vs_reference"], ascending=[False, False]).head(top_n).reset_index(drop=True)

# =============================================================================
# External-support summaries
# =============================================================================

def bootstrap_ci_median(values: Sequence[float], n_boot: int = 1000, seed: int = 42) -> Tuple[float, float, float]:
    arr = np.asarray(list(values), dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return (np.nan, np.nan, np.nan)
    if len(arr) == 1:
        return (float(arr[0]), float(arr[0]), float(arr[0]))
    rng = np.random.default_rng(seed)
    meds = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        meds.append(np.median(sample))
    meds = np.asarray(meds, dtype=float)
    return (float(np.median(arr)), float(np.quantile(meds, 0.025)), float(np.quantile(meds, 0.975)))

def summarize_external_support(df: pd.DataFrame, stage: str, score_col: str, positive_col: str, cfg: Step9Config) -> pd.DataFrame:
    score_vals = pd.to_numeric(get_1d_column(df, score_col), errors="coerce")
    positive_vals = pd.to_numeric(get_1d_column(df, positive_col), errors="coerce")

    med, lo, hi = bootstrap_ci_median(score_vals.dropna().tolist(), n_boot=cfg.bootstrap_iterations, seed=cfg.random_seed)
    return pd.DataFrame([{
        "stage": stage,
        "n": int(len(df)),
        "n_with_external_score": int(score_vals.notna().sum()),
        "median_external_support": med,
        "median_external_support_ci_low": lo,
        "median_external_support_ci_high": hi,
        "external_positive_rate": float(positive_vals.mean()) if positive_vals.notna().any() else np.nan,
    }])

def build_predictor_agreement_table(df: pd.DataFrame, cfg: Step9Config) -> pd.DataFrame:
    predictor_cols = discover_predictor_score_columns(df, cfg)
    if not predictor_cols:
        return pd.DataFrame(columns=["candidate_id", "n_predictors_available", "n_positive_predictors", "positive_fraction"])
    id_col = first_existing_column(df, cfg.id_col_candidates, required=True, label="candidate id")
    rows = []
    for _, row in df.iterrows():
        vals = []
        for c in predictor_cols:
            v = pd.to_numeric(pd.Series([row[c]]), errors="coerce").iloc[0]
            if pd.notna(v):
                vals.append(float(v))
        rows.append({
            "candidate_id": str(row[id_col]),
            "n_predictors_available": len(vals),
            "n_positive_predictors": int(np.sum(np.asarray(vals) >= cfg.external_positive_threshold)) if vals else 0,
            "positive_fraction": float(np.mean(np.asarray(vals) >= cfg.external_positive_threshold)) if vals else np.nan,
        })
    return pd.DataFrame(rows)

# =============================================================================
# Final evidence table
# =============================================================================

def build_final_evidence_table(
    final_df: pd.DataFrame,
    final_cols: Dict[str, str],
    score_col: str,
    positive_col: str,
    cfg: Step9Config
) -> pd.DataFrame:
    id_col = final_cols["id_col"]
    seq_col = final_cols["sequence_col"]

    cols = [id_col, seq_col]
    for c in [
        "final_panel_rank", "shortlist_rank", "final_rank",
        "novelty_score", "descriptor_plausibility_score", "diversity_score",
        "condition_fidelity_score", "final_score",
        score_col, positive_col,
        "nearest_reference_id", "nearest_reference_similarity"
    ]:
        if c in final_df.columns:
            cols.append(c)

    out = deduplicate_columns_keep_last(final_df[cols].copy())
    out = out.rename(columns={
        score_col: "external_support_score",
        positive_col: "external_support_positive"
    })

    def _rationale(r: pd.Series) -> str:
        parts = []
        if "novelty_score" in r and pd.notna(r["novelty_score"]):
            parts.append(f"novelty={r['novelty_score']:.3f}")
        if "descriptor_plausibility_score" in r and pd.notna(r["descriptor_plausibility_score"]):
            parts.append(f"plausibility={r['descriptor_plausibility_score']:.3f}")
        if "diversity_score" in r and pd.notna(r["diversity_score"]):
            parts.append(f"diversity={r['diversity_score']:.3f}")
        if "external_support_score" in r and pd.notna(r["external_support_score"]):
            parts.append(f"external={r['external_support_score']:.3f}")
        return "; ".join(parts)

    out["selection_rationale"] = out.apply(_rationale, axis=1)
    rank_col = "final_panel_rank" if "final_panel_rank" in out.columns else ("final_rank" if "final_rank" in out.columns else id_col)
    return out.sort_values(rank_col).reset_index(drop=True)

# =============================================================================
# Figure helpers
# =============================================================================

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.set_facecolor(PROJECT_COLORS["bg"])
    ax.spines["left"].set_color(PROJECT_COLORS["spine"])
    ax.spines["bottom"].set_color(PROJECT_COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(colors=PROJECT_COLORS["text"], width=0.7)
    ax.grid(True, axis=grid_axis, color=PROJECT_COLORS["grid"], linewidth=0.55)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str) -> None:
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, fontsize=10, fontweight="bold",
            ha="left", va="bottom", color=PROJECT_COLORS["text"])

def pairwise_similarity_matrix(seqs: Sequence[str], k: int) -> np.ndarray:
    seqs = list(seqs)
    n = len(seqs)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=k)
            mat[i, j] = sim
            mat[j, i] = sim
    return mat

# =============================================================================
# Main analysis
# =============================================================================

def run_stage(cfg: Step9Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Loading Step 1 / Step 8 inputs")
    loaded = load_inputs(cfg)

    ref_df, ref_cols = prepare_reference_df(loaded["ref_df"], cfg)
    passed_df, passed_cols = prepare_stage_df(loaded["passed_df"], "Passed", cfg)
    shortlist_df, shortlist_cols = prepare_stage_df(loaded["shortlist_df"], "Shortlist", cfg)
    final_panel_df, final_cols = prepare_stage_df(loaded["final_panel_df"], "Final panel", cfg)

    common_id_col = passed_cols["id_col"]
    if shortlist_cols["id_col"] != common_id_col and common_id_col in shortlist_df.columns:
        shortlist_cols["id_col"] = common_id_col
    if final_cols["id_col"] != common_id_col and common_id_col in final_panel_df.columns:
        final_cols["id_col"] = common_id_col

    save_json(
        {
            "config": asdict(cfg),
            "input_paths": {
                "step1_reference_path": str(loaded["step1_ref_path"]),
                "step8_passed_path": str(loaded["passed_file"]),
                "step8_shortlist_path": str(loaded["shortlist_file"]),
                "step8_final_panel_path": str(loaded["final_panel_file"]),
                "step9_external_support_path": str(loaded["external_support_file"]) if loaded["external_support_file"] is not None else None,
            },
            "input_hashes": {
                "step1_reference_sha256": sha256_of_file(loaded["step1_ref_path"]),
                "step8_passed_sha256": sha256_of_file(loaded["passed_file"]),
                "step8_shortlist_sha256": sha256_of_file(loaded["shortlist_file"]),
                "step8_final_panel_sha256": sha256_of_file(loaded["final_panel_file"]),
            },
            "environment_manifest": collect_environment_manifest(),
        },
        dirs["artifacts"] / "run_manifest.json"
    )

    print_header("Attaching external support")
    passed_df, shortlist_df, final_panel_df, external_provenance_df = attach_external_support(
        passed_df=passed_df,
        shortlist_df=shortlist_df,
        final_panel_df=final_panel_df,
        external_df=loaded["external_df"],
        passed_cols=passed_cols,
        cfg=cfg
    )

    score_col = str(external_provenance_df.loc[0, "resolved_external_score_col"])
    positive_col = str(external_provenance_df.loc[0, "resolved_external_binary_col"])

    print_header("Computing condition-fidelity audit")
    passed_df = deduplicate_columns_keep_last(passed_df.drop(columns=[c for c in CONDITION_COLS if c in passed_df.columns], errors="ignore"))
    shortlist_df = deduplicate_columns_keep_last(shortlist_df.drop(columns=[c for c in CONDITION_COLS if c in shortlist_df.columns], errors="ignore"))
    final_panel_df = deduplicate_columns_keep_last(final_panel_df.drop(columns=[c for c in CONDITION_COLS if c in final_panel_df.columns], errors="ignore"))

    passed_cf = compute_condition_fidelity(passed_df, cfg)
    shortlist_cf = compute_condition_fidelity(shortlist_df, cfg)
    final_cf = compute_condition_fidelity(final_panel_df, cfg)

    passed_df = deduplicate_columns_keep_last(pd.concat([passed_df.reset_index(drop=True), passed_cf.reset_index(drop=True)], axis=1))
    shortlist_df = deduplicate_columns_keep_last(pd.concat([shortlist_df.reset_index(drop=True), shortlist_cf.reset_index(drop=True)], axis=1))
    final_panel_df = deduplicate_columns_keep_last(pd.concat([final_panel_df.reset_index(drop=True), final_cf.reset_index(drop=True)], axis=1))

    condition_summary_df = pd.concat([
        summarize_condition_fidelity(passed_df, "Passed"),
        summarize_condition_fidelity(shortlist_df, "Shortlist"),
        summarize_condition_fidelity(final_panel_df, "Final panel"),
    ], ignore_index=True)

    print_header("Nearest-neighbor contextualization to reference ACP corpus")
    passed_nn = compute_nearest_neighbor_context(
        passed_df, ref_df,
        query_seq_col=passed_cols["sequence_col"],
        ref_seq_col=ref_cols["sequence_col"],
        ref_id_col=ref_cols["id_col"],
        k=cfg.similarity_kmer
    )
    shortlist_nn = compute_nearest_neighbor_context(
        shortlist_df, ref_df,
        query_seq_col=shortlist_cols["sequence_col"],
        ref_seq_col=ref_cols["sequence_col"],
        ref_id_col=ref_cols["id_col"],
        k=cfg.similarity_kmer
    )
    final_nn = compute_nearest_neighbor_context(
        final_panel_df, ref_df,
        query_seq_col=final_cols["sequence_col"],
        ref_seq_col=ref_cols["sequence_col"],
        ref_id_col=ref_cols["id_col"],
        k=cfg.similarity_kmer
    )

    passed_df = deduplicate_columns_keep_last(pd.concat([passed_df.reset_index(drop=True), passed_nn.reset_index(drop=True)], axis=1))
    shortlist_df = deduplicate_columns_keep_last(pd.concat([shortlist_df.reset_index(drop=True), shortlist_nn.reset_index(drop=True)], axis=1))
    final_panel_df = deduplicate_columns_keep_last(pd.concat([final_panel_df.reset_index(drop=True), final_nn.reset_index(drop=True)], axis=1))

    nn_summary_rows = []
    for stage_name, dfi in [("Passed", passed_df), ("Shortlist", shortlist_df), ("Final panel", final_panel_df)]:
        vals = pd.to_numeric(get_1d_column(dfi, "nearest_reference_similarity"), errors="coerce")
        nn_summary_rows.append({
            "stage": stage_name,
            "n": int(len(dfi)),
            "median_nearest_reference_similarity": float(vals.median()) if vals.notna().any() else np.nan,
            "mean_nearest_reference_similarity": float(vals.mean()) if vals.notna().any() else np.nan,
            "q25": float(vals.quantile(0.25)) if vals.notna().any() else np.nan,
            "q75": float(vals.quantile(0.75)) if vals.notna().any() else np.nan,
        })
    nn_summary_df = pd.DataFrame(nn_summary_rows)

    print_header("External-support summaries")
    external_support_summary_df = pd.concat([
        summarize_external_support(passed_df, "Passed", score_col, positive_col, cfg),
        summarize_external_support(shortlist_df, "Shortlist", score_col, positive_col, cfg),
        summarize_external_support(final_panel_df, "Final panel", score_col, positive_col, cfg),
    ], ignore_index=True)

    predictor_agreement_df = build_predictor_agreement_table(final_panel_df, cfg)

    print_header("Interpretability summaries")
    aa_enrichment_df = compute_aa_enrichment(
        final_df=final_panel_df,
        ref_df=ref_df,
        final_seq_col=final_cols["sequence_col"],
        ref_seq_col=ref_cols["sequence_col"],
        cfg=cfg,
    )
    kmer_enrichment_df = compute_kmer_enrichment(
        final_df=final_panel_df,
        ref_df=ref_df,
        final_seq_col=final_cols["sequence_col"],
        ref_seq_col=ref_cols["sequence_col"],
        cfg=cfg,
        k=cfg.similarity_kmer,
        top_n=20,
    )

    residue_class_summary_df = pd.concat([
        compute_residue_class_summary(ref_df, ref_cols["sequence_col"], "Reference ACP corpus"),
        compute_residue_class_summary(passed_df, passed_cols["sequence_col"], "Passed"),
        compute_residue_class_summary(shortlist_df, shortlist_cols["sequence_col"], "Shortlist"),
        compute_residue_class_summary(final_panel_df, final_cols["sequence_col"], "Final panel"),
    ], ignore_index=True)

    print_header("Building final evidence table")
    final_evidence_table = build_final_evidence_table(
        final_df=final_panel_df,
        final_cols=final_cols,
        score_col=score_col,
        positive_col=positive_col,
        cfg=cfg,
    )

    final_evidence_table.to_csv(dirs["tables_main"] / "table_9_1_final_panel_integrated_evidence.csv", index=False)

    external_provenance_df.to_csv(dirs["tables_supplementary"] / "table_s9_1_external_support_provenance.csv", index=False)
    condition_summary_df.to_csv(dirs["tables_supplementary"] / "table_s9_2_condition_fidelity_summary.csv", index=False)
    nn_summary_df.to_csv(dirs["tables_supplementary"] / "table_s9_3_nearest_reference_similarity_summary.csv", index=False)
    aa_enrichment_df.to_csv(dirs["tables_supplementary"] / "table_s9_4_amino_acid_enrichment_vs_reference.csv", index=False)
    kmer_enrichment_df.to_csv(dirs["tables_supplementary"] / "table_s9_5_kmer_enrichment_vs_reference.csv", index=False)
    external_support_summary_df.to_csv(dirs["tables_supplementary"] / "table_s9_6_external_support_stage_summary.csv", index=False)
    predictor_agreement_df.to_csv(dirs["tables_supplementary"] / "table_s9_7_final_panel_predictor_agreement.csv", index=False)
    passed_df.to_csv(dirs["tables_supplementary"] / "table_s9_8_passed_pool_with_step9_annotations.csv", index=False)
    shortlist_df.to_csv(dirs["tables_supplementary"] / "table_s9_9_shortlist_with_step9_annotations.csv", index=False)
    final_panel_df.to_csv(dirs["tables_supplementary"] / "table_s9_10_final_panel_with_step9_annotations.csv", index=False)
    residue_class_summary_df.to_csv(dirs["tables_supplementary"] / "table_s9_11_residue_class_summary.csv", index=False)

    summary_lines = [
        "Step 9 — External validation, interpretability, and downstream support",
        "=" * 84,
        f"Timestamp UTC: {now_utc_iso()}",
        "",
        f"Reference corpus size: {len(ref_df):,}",
        f"Passed size: {len(passed_df):,}",
        f"Shortlist size: {len(shortlist_df):,}",
        f"Final panel size: {len(final_panel_df):,}",
        "",
        "External-support summary by stage:",
    ]
    for _, row in external_support_summary_df.iterrows():
        if pd.notna(row["median_external_support"]):
            summary_lines.append(
                f"- {row['stage']}: n={int(row['n'])}, "
                f"n_with_external_score={int(row['n_with_external_score'])}, "
                f"median_external_support={row['median_external_support']:.3f} "
                f"[{row['median_external_support_ci_low']:.3f}, {row['median_external_support_ci_high']:.3f}], "
                f"external_positive_rate={row['external_positive_rate']:.3f}"
            )
        else:
            summary_lines.append(f"- {row['stage']}: no external-support scores available")
    summary_lines += [
        "",
        "Nearest-reference summary by stage:",
    ]
    for _, row in nn_summary_df.iterrows():
        if pd.notna(row["median_nearest_reference_similarity"]):
            summary_lines.append(
                f"- {row['stage']}: median nearest-reference similarity={row['median_nearest_reference_similarity']:.3f}, "
                f"IQR=[{row['q25']:.3f}, {row['q75']:.3f}]"
            )
        else:
            summary_lines.append(f"- {row['stage']}: similarity summary unavailable")

    save_text("\n".join(summary_lines), dirs["reports"] / "summary_report.txt")

    return {
        "ref_df": ref_df,
        "passed_df": passed_df,
        "shortlist_df": shortlist_df,
        "final_panel_df": final_panel_df,
        "ref_cols": ref_cols,
        "passed_cols": passed_cols,
        "shortlist_cols": shortlist_cols,
        "final_cols": final_cols,
        "external_score_col": score_col,
        "external_positive_col": positive_col,
        "external_support_summary_df": external_support_summary_df,
        "condition_summary_df": condition_summary_df,
        "nn_summary_df": nn_summary_df,
        "aa_enrichment_df": aa_enrichment_df,
        "kmer_enrichment_df": kmer_enrichment_df,
        "predictor_agreement_df": predictor_agreement_df,
        "final_evidence_table": final_evidence_table,
        "external_provenance_df": external_provenance_df,
    }

# =============================================================================
# Figures
# =============================================================================

def plot_main_figure(results: Dict[str, Any], cfg: Step9Config, dirs: Dict[str, Path]) -> None:
    passed_df = results["passed_df"]
    shortlist_df = results["shortlist_df"]
    final_panel_df = results["final_panel_df"]
    external_summary = results["external_support_summary_df"]
    condition_summary = results["condition_summary_df"]
    score_col = results["external_score_col"]
    final_cols = results["final_cols"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 7.6))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34)

    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="y")
    stages = ["Passed", "Shortlist", "Final panel"]
    x = np.arange(len(stages))
    medians = []
    ci_low = []
    ci_high = []
    for st in stages:
        row = external_summary[external_summary["stage"] == st].iloc[0]
        medians.append(row["median_external_support"])
        ci_low.append(row["median_external_support_ci_low"])
        ci_high.append(row["median_external_support_ci_high"])

    if np.isfinite(np.asarray(medians, dtype=float)).any():
        yerr = np.vstack([
            np.asarray(medians, dtype=float) - np.asarray(ci_low, dtype=float),
            np.asarray(ci_high, dtype=float) - np.asarray(medians, dtype=float)
        ])
        ax.bar(
            x, medians, yerr=yerr,
            color=[GROUP_COLORS["Passed"], GROUP_COLORS["Shortlist"], GROUP_COLORS["Final panel"]],
            edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.6, capsize=3
        )
        ax.set_ylim(0, 1.02)
        ax.set_xticks(x)
        ax.set_xticklabels(stages)
        ax.set_ylabel("External support score")
        ax.set_title("External support across prioritization stages", loc="left", pad=4)
    else:
        ax.text(0.5, 0.5, "External support unavailable", ha="center", va="center")
        ax.set_title("External support across prioritization stages", loc="left", pad=4)

    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="y")

    plot_metrics = ["length_bin_match", "charge_bin_match", "hydropathy_bin_match", "all_condition_match"]
    disp = {
        "length_bin_match": "Length",
        "charge_bin_match": "Charge",
        "hydropathy_bin_match": "Hydropathy",
        "all_condition_match": "All conditions",
    }
    cond_plot = condition_summary[condition_summary["metric"].isin(plot_metrics)].copy()

    if len(cond_plot) > 0 and cond_plot["n_non_missing"].sum() > 0:
        width = 0.24
        xpos = np.arange(len(plot_metrics))
        stage_order = ["Passed", "Shortlist", "Final panel"]
        offsets = {"Passed": -width, "Shortlist": 0.0, "Final panel": width}
        for stage in stage_order:
            sub = cond_plot[cond_plot["stage"] == stage].set_index("metric")
            vals = [sub.loc[m, "mean"] if m in sub.index else np.nan for m in plot_metrics]
            ax.bar(
                xpos + offsets[stage], vals, width=width,
                color=GROUP_COLORS[stage], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5, label=stage
            )
        ax.set_xticks(xpos)
        ax.set_xticklabels([disp[m] for m in plot_metrics])
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("Recovery rate")
        ax.set_title("Conditioning fidelity", loc="left", pad=4)
        ax.legend(frameon=False, loc="lower right")
    else:
        ax.text(0.5, 0.5, "Condition metadata unavailable", ha="center", va="center")
        ax.set_title("Conditioning fidelity", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    style_axis(ax, grid_axis="both")

    if "novelty_score" in passed_df.columns and pd.to_numeric(get_1d_column(passed_df, score_col), errors="coerce").notna().any():
        hb = ax.hexbin(
            pd.to_numeric(get_1d_column(passed_df, "novelty_score"), errors="coerce"),
            pd.to_numeric(get_1d_column(passed_df, score_col), errors="coerce"),
            gridsize=34, mincnt=1, cmap="Greys", linewidths=0.0, alpha=0.8
        )
        ax.scatter(
            pd.to_numeric(get_1d_column(shortlist_df, "novelty_score"), errors="coerce"),
            pd.to_numeric(get_1d_column(shortlist_df, score_col), errors="coerce"),
            s=28, facecolor=GROUP_COLORS["Shortlist"], edgecolor="white", linewidth=0.35, alpha=0.95, label="Shortlist"
        )
        ax.scatter(
            pd.to_numeric(get_1d_column(final_panel_df, "novelty_score"), errors="coerce"),
            pd.to_numeric(get_1d_column(final_panel_df, score_col), errors="coerce"),
            s=36, facecolor=GROUP_COLORS["Final panel"], edgecolor="white", linewidth=0.45, alpha=1.0, label="Final panel"
        )
        ax.set_xlabel("Novelty score")
        ax.set_ylabel("External support score")
        ax.set_title("Novelty–external support landscape", loc="left", pad=4)
        ax.legend(frameon=False, loc="lower left")
        cb = fig.colorbar(hb, ax=ax, fraction=0.048, pad=0.03)
        cb.set_label("Passed-pool density")
    else:
        ax.text(0.5, 0.5, "Required data unavailable", ha="center", va="center")
        ax.set_title("Novelty–external support landscape", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    panel_df = deduplicate_columns_keep_last(final_panel_df.copy())

    heat_cols = []
    heat_labels = []
    for c, lab in [
        ("novelty_score", "Novelty"),
        ("descriptor_plausibility_score", "Plausibility"),
        ("diversity_score", "Diversity"),
        ("condition_fidelity_score", "Condition"),
        (score_col, "External"),
    ]:
        if c in panel_df.columns:
            vals = pd.to_numeric(get_1d_column(panel_df, c), errors="coerce")
            if vals.notna().any():
                heat_cols.append(c)
                heat_labels.append(lab)

    if heat_cols:
        rank_col = "final_panel_rank" if "final_panel_rank" in panel_df.columns else final_cols["id_col"]
        panel_df = panel_df.sort_values(rank_col).reset_index(drop=True)
        mat = np.column_stack([pd.to_numeric(get_1d_column(panel_df, c), errors="coerce").to_numpy() for c in heat_cols])
        im = ax.imshow(mat, aspect="auto", vmin=0, vmax=1, cmap="Blues")
        ylab = get_1d_column(panel_df, final_cols["id_col"]).astype(str).tolist()
        ax.set_yticks(np.arange(len(ylab)))
        ax.set_yticklabels(ylab, fontsize=6)
        ax.set_xticks(np.arange(len(heat_labels)))
        ax.set_xticklabels(heat_labels, rotation=35, ha="right")
        ax.set_title("Integrated evidence for final-panel peptides", loc="left", pad=4)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                            color="white" if v > 0.55 else PROJECT_COLORS["text"])
        cb = fig.colorbar(im, ax=ax, fraction=0.055, pad=0.04)
        cb.set_label("Score")
    else:
        ax.text(0.5, 0.5, "Insufficient evidence columns", ha="center", va="center")
        ax.set_title("Integrated evidence for final-panel peptides", loc="left", pad=4)

    fig.tight_layout()
    export_figure(fig, dirs["figures_main"] / "figure_9_main_external_validation_v1", cfg.figure_export)

def plot_supplementary_predictor_agreement(results: Dict[str, Any], cfg: Step9Config, dirs: Dict[str, Path]) -> None:
    predictor_agreement_df = results["predictor_agreement_df"]
    final_panel_df = results["final_panel_df"]
    final_cols = results["final_cols"]
    seq_col = final_cols["sequence_col"]
    id_col = final_cols["id_col"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.2))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34)

    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="y")
    if len(predictor_agreement_df) > 0:
        tmp = predictor_agreement_df.sort_values(["positive_fraction", "n_positive_predictors"], ascending=[False, False]).reset_index(drop=True)
        ax.bar(
            np.arange(len(tmp)),
            tmp["positive_fraction"].fillna(0),
            color=PROJECT_COLORS["hydramp_red"],
            edgecolor=PROJECT_COLORS["gray_dark"],
            linewidth=0.5
        )
        ax.set_xticks(np.arange(len(tmp)))
        ax.set_xticklabels(tmp["candidate_id"].astype(str), rotation=45, ha="right", fontsize=6)
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("Positive predictor fraction")
        ax.set_title("Agreement across external predictors", loc="left", pad=4)
    else:
        ax.text(0.5, 0.5, "External predictor panel unavailable", ha="center", va="center")
        ax.set_title("Agreement across external predictors", loc="left", pad=4)

    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="y")
    vals = pd.to_numeric(get_1d_column(final_panel_df, "nearest_reference_similarity"), errors="coerce")
    ax.bar(
        np.arange(len(final_panel_df)),
        vals.fillna(0.0),
        color=PROJECT_COLORS["basic_blue"],
        edgecolor=PROJECT_COLORS["gray_dark"],
        linewidth=0.5
    )
    ax.set_xticks(np.arange(len(final_panel_df)))
    ax.set_xticklabels(get_1d_column(final_panel_df, id_col).astype(str), rotation=45, ha="right", fontsize=6)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("3-mer Jaccard similarity")
    ax.set_title("Nearest-reference similarity", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    seqs = get_1d_column(final_panel_df, seq_col).astype(str).tolist()
    labs = get_1d_column(final_panel_df, id_col).astype(str).tolist()
    mat = pairwise_similarity_matrix(seqs, cfg.similarity_kmer) if len(seqs) > 0 else np.zeros((1, 1))
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(np.arange(len(labs)))
    ax.set_yticks(np.arange(len(labs)))
    ax.set_xticklabels(labs, rotation=45, ha="right", fontsize=6)
    ax.set_yticklabels(labs, fontsize=6)
    ax.set_title("Pairwise similarity within final panel", loc="left", pad=4)
    cb = fig.colorbar(im, ax=ax, fraction=0.055, pad=0.04)
    cb.set_label("3-mer Jaccard similarity")

    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    style_axis(ax, grid_axis="y")
    positive_col = results["external_positive_col"]
    stage_vals = []
    for _, dfi in [("Passed", results["passed_df"]), ("Shortlist", results["shortlist_df"]), ("Final panel", results["final_panel_df"])]:
        vals = pd.to_numeric(get_1d_column(dfi, positive_col), errors="coerce")
        stage_vals.append(vals.mean() if vals.notna().any() else np.nan)
    if np.isfinite(np.asarray(stage_vals, dtype=float)).any():
        ax.bar(
            np.arange(3),
            stage_vals,
            color=[GROUP_COLORS["Passed"], GROUP_COLORS["Shortlist"], GROUP_COLORS["Final panel"]],
            edgecolor=PROJECT_COLORS["gray_dark"],
            linewidth=0.5
        )
        ax.set_xticks(np.arange(3))
        ax.set_xticklabels(["Passed", "Shortlist", "Final panel"])
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("Positive rate")
        ax.set_title("External-support positivity by stage", loc="left", pad=4)
    else:
        ax.text(0.5, 0.5, "External support unavailable", ha="center", va="center")
        ax.set_title("External-support positivity by stage", loc="left", pad=4)

    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_9_1_predictor_agreement_similarity_v1", cfg.figure_export)

def plot_supplementary_interpretability(results: Dict[str, Any], cfg: Step9Config, dirs: Dict[str, Path]) -> None:
    aa_enrichment_df = results["aa_enrichment_df"]
    kmer_enrichment_df = results["kmer_enrichment_df"]
    final_panel_df = results["final_panel_df"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.6))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34)

    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="x")
    tmp = aa_enrichment_df.sort_values("log2_enrichment_vs_reference", ascending=True)
    colors = [PROJECT_COLORS["basic_blue"] if x < 0 else PROJECT_COLORS["hydramp_red"] for x in tmp["log2_enrichment_vs_reference"]]
    ax.barh(tmp["amino_acid"], tmp["log2_enrichment_vs_reference"], color=colors, edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.4)
    ax.axvline(0, color=PROJECT_COLORS["gray_dark"], linewidth=0.8)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Amino-acid enrichment in final panel", loc="left", pad=4)

    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="x")
    tmp = kmer_enrichment_df.sort_values("log2_enrichment_vs_reference", ascending=True)
    ax.barh(tmp["kmer"], tmp["log2_enrichment_vs_reference"], color=PROJECT_COLORS["joker_teal"],
            edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.4)
    ax.axvline(0, color=PROJECT_COLORS["gray_dark"], linewidth=0.8)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Enriched 3-mers in final panel", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    style_axis(ax, grid_axis="both")
    ax.scatter(
        pd.to_numeric(get_1d_column(final_panel_df, "mean_hydropathy"), errors="coerce"),
        pd.to_numeric(get_1d_column(final_panel_df, "net_charge_proxy"), errors="coerce"),
        s=45,
        facecolor=PROJECT_COLORS["hydramp_red"],
        edgecolor="white",
        linewidth=0.5,
        alpha=0.95
    )
    for _, r in final_panel_df.iterrows():
        label = str(r[results["final_cols"]["id_col"]])
        ax.text(float(r["mean_hydropathy"]) + 0.03, float(r["net_charge_proxy"]) + 0.03, label, fontsize=6)
    ax.set_xlabel("Mean hydropathy")
    ax.set_ylabel("Net charge proxy")
    ax.set_title("Final-panel charge–hydropathy landscape", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    style_axis(ax, grid_axis="both")
    ax.scatter(
        pd.to_numeric(get_1d_column(final_panel_df, "entropy"), errors="coerce"),
        pd.to_numeric(get_1d_column(final_panel_df, "nearest_reference_similarity"), errors="coerce"),
        s=45,
        facecolor=PROJECT_COLORS["pepcvae_navy"],
        edgecolor="white",
        linewidth=0.5,
        alpha=0.95
    )
    for _, r in final_panel_df.iterrows():
        label = str(r[results["final_cols"]["id_col"]])
        ax.text(float(r["entropy"]) + 0.01, float(r["nearest_reference_similarity"]) + 0.01, label, fontsize=6)
    ax.set_xlabel("Sequence entropy")
    ax.set_ylabel("Nearest-reference similarity")
    ax.set_title("Entropy and reference proximity", loc="left", pad=4)

    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_9_2_interpretability_enrichment_v1", cfg.figure_export)

# =============================================================================
# Notebook entrypoint
# =============================================================================

def main_notebook(cfg: Optional[Step9Config] = None) -> Dict[str, Any]:
    if cfg is None:
        cfg = Step9Config()

    seed_all(cfg.random_seed)
    step_root = Path(cfg.project_root).resolve() / cfg.step9_dir
    dirs = make_output_dirs(step_root)

    print_header("Running Step 9 — External validation, interpretability, and downstream support")
    print(json.dumps({
        "project_root": cfg.project_root,
        "step1_dir": cfg.step1_dir,
        "step8_dir": cfg.step8_dir,
        "step9_dir": cfg.step9_dir,
        "step1_reference_file": cfg.step1_reference_file,
        "step8_passed_file": cfg.step8_passed_file,
        "step8_shortlist_file": cfg.step8_shortlist_file,
        "step8_final_panel_file": cfg.step8_final_panel_file,
        "step9_external_support_file": cfg.step9_external_support_file,
        "similarity_kmer": cfg.similarity_kmer,
        "external_positive_threshold": cfg.external_positive_threshold,
    }, indent=2))

    results = run_stage(cfg, dirs)
    plot_main_figure(results, cfg, dirs)
    plot_supplementary_predictor_agreement(results, cfg, dirs)
    plot_supplementary_interpretability(results, cfg, dirs)

    save_json(
        {
            "n_reference": int(len(results["ref_df"])),
            "n_passed": int(len(results["passed_df"])),
            "n_shortlist": int(len(results["shortlist_df"])),
            "n_final_panel": int(len(results["final_panel_df"])),
            "external_score_col": results["external_score_col"],
            "external_positive_col": results["external_positive_col"],
        },
        dirs["logs"] / "stage_counts.json",
    )

    print("\n" + "=" * 100)
    print("Step 9 completed successfully.")
    print(f"Main tables: {dirs['tables_main']}")
    print(f"Supplementary tables: {dirs['tables_supplementary']}")
    print(f"Main figures: {dirs['figures_main']}")
    print(f"Supplementary figures: {dirs['figures_supplementary']}")
    print(f"Artifacts: {dirs['artifacts']}")
    print(f"Reports: {dirs['reports']}")
    print(f"Logs: {dirs['logs']}")
    print("=" * 100)

    return results

# =============================================================================
# Execute
# =============================================================================

results = main_notebook()

# Step 9 — Paper-derived contextual support only

In [ ]:
from __future__ import annotations

# =============================================================================
# PepCVAE / AngioPep-VAE
# Step 9 — Paper-derived contextual support only
# Final publication-oriented implementation
#
# Scientific scope of this step
# -----------------------------
# This step provides a literature- and paper-derived contextual support layer
# for the final shortlisted peptides. It does NOT constitute:
#   - independent external validation
#   - orthogonal wet-lab confirmation
#   - external predictive-tool validation
#   - proof of anticancer activity
#
# Its role is to:
#   1) document transparent candidate reduction across prior stages,
#   2) summarize inherited selection evidence from Step 8,
#   3) add paper-derived contextual support and descriptor positioning, and
#   4) export reviewer-defensible figures/tables for manuscript integration.
# =============================================================================

import json
import math
import hashlib
import platform
import random
import warnings
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from itertools import groupby
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

# =============================================================================
# Global constants
# =============================================================================

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_ALPHABET)

AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")

AA_PROPERTIES = {
    "A": {"hydro": 1.8, "mw": 89.09},
    "C": {"hydro": 2.5, "mw": 121.16},
    "D": {"hydro": -3.5, "mw": 133.10},
    "E": {"hydro": -3.5, "mw": 147.13},
    "F": {"hydro": 2.8, "mw": 165.19},
    "G": {"hydro": -0.4, "mw": 75.07},
    "H": {"hydro": -3.2, "mw": 155.16},
    "I": {"hydro": 4.5, "mw": 131.17},
    "K": {"hydro": -3.9, "mw": 146.19},
    "L": {"hydro": 3.8, "mw": 131.17},
    "M": {"hydro": 1.9, "mw": 149.21},
    "N": {"hydro": -3.5, "mw": 132.12},
    "P": {"hydro": -1.6, "mw": 115.13},
    "Q": {"hydro": -3.5, "mw": 146.15},
    "R": {"hydro": -4.5, "mw": 174.20},
    "S": {"hydro": -0.8, "mw": 105.09},
    "T": {"hydro": -0.7, "mw": 119.12},
    "V": {"hydro": 4.2, "mw": 117.15},
    "W": {"hydro": -0.9, "mw": 204.23},
    "Y": {"hydro": -1.3, "mw": 181.19},
}

PROJECT_COLORS = {
    "deep_red": "#C7001E",
    "navy": "#1F3B68",
    "teal": "#0E6766",
    "orange": "#F29D3E",
    "light_blue": "#A8B9DC",
    "purple": "#6A2C91",
    "gray_light": "#D9D9D9",
    "gray_mid": "#A0A0A0",
    "gray_dark": "#4A4A4A",
    "bg": "#FFFFFF",
    "grid": "#E8E8E8",
    "spine": "#BFBFBF",
    "text": "#222222",
}

GROUP_COLORS = {
    "Candidate pool": PROJECT_COLORS["gray_mid"],
    "Passed": PROJECT_COLORS["teal"],
    "Shortlist": PROJECT_COLORS["orange"],
    "Final panel": PROJECT_COLORS["navy"],
    "Reference ACP corpus": PROJECT_COLORS["light_blue"],
    "Paper templates": PROJECT_COLORS["gray_mid"],
    "Paper candidates": PROJECT_COLORS["deep_red"],
}

# =============================================================================
# Matplotlib style
# =============================================================================

def apply_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PROJECT_COLORS["bg"],
        "axes.facecolor": PROJECT_COLORS["bg"],
        "savefig.facecolor": PROJECT_COLORS["bg"],
        "axes.edgecolor": PROJECT_COLORS["spine"],
        "axes.labelcolor": PROJECT_COLORS["text"],
        "xtick.color": PROJECT_COLORS["text"],
        "ytick.color": PROJECT_COLORS["text"],
        "text.color": PROJECT_COLORS["text"],
        "axes.titlecolor": PROJECT_COLORS["text"],
        "grid.color": PROJECT_COLORS["grid"],
        "axes.grid": True,
        "grid.linestyle": "-",
        "grid.linewidth": 0.55,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 8,
        "font.family": "DejaVu Sans",
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    })

apply_style()

# =============================================================================
# Config
# =============================================================================

@dataclass
class FigureExportConfig:
    png_dpi: int = 300
    tif_dpi: int = 600
    export_pdf: bool = True
    export_png: bool = True
    export_tif: bool = True
    single_col_width_in: float = 3.5
    double_col_width_in: float = 7.1

@dataclass
class Step9Config:
    project_root: str = "/home/data3/Moe/nature_computational2/"
    step9_dir: str = "step9_v1"

    # Prior completed steps
    step1_reference_file: str = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"
    step8_passed_file: str = "/home/data3/Moe/nature_computational2/step8_v1/tables_supplementary/table_s8_12_full_ranked_passed_pool.csv"
    step8_shortlist_file: str = "/home/data3/Moe/nature_computational2/step8_v1/tables_main/table_8_1_shortlist_main.csv"
    step8_final_panel_file: str = "/home/data3/Moe/nature_computational2/step8_v1/tables_main/table_8_2_final_diverse_panel.csv"

    # Keep same path structure from the previous file for compatibility.
    # We allow fallback aliases but treat the data scientifically as
    # paper-derived contextual support, not external validation.
    paper_templates_file: str = "/home/data3/Moe/nature_computational2/step9_v1/external validation/paper_001_table_s1_templates.csv"
    paper_candidates_file: str = "/home/data3/Moe/nature_computational2/step9_v1/external validation/paper_001_table_s2_candidates.csv"
    paper_templates_fallbacks: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational2/step9_v1/contextual_support/paper_001_table_s1_templates.csv",
    )
    paper_candidates_fallbacks: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational2/step9_v1/contextual_support/paper_001_table_s2_candidates.csv",
    )

    # Optional Step 8 audit files
    step8_candidate_pool_file: str = "/home/data3/Moe/nature_computational2/step8_v1/tables_supplementary/table_s8_2_candidate_pool_with_flags.csv"
    step8_filter_audit_file: str = "/home/data3/Moe/nature_computational2/step8_v1/tables_supplementary/table_s8_1_filter_audit.csv"

    sequence_col_candidates: Tuple[str, ...] = ("sequence", "peptide", "seq", "peptide_sequence", "aligned_sequence")
    id_col_candidates: Tuple[str, ...] = ("generated_id", "candidate_id", "peptide_id", "id")

    novelty_score_candidates: Tuple[str, ...] = ("novelty_score",)
    plausibility_score_candidates: Tuple[str, ...] = ("descriptor_plausibility_score", "plausibility_score")
    diversity_score_candidates: Tuple[str, ...] = ("diversity_score",)
    final_score_candidates: Tuple[str, ...] = ("final_score",)

    similarity_kmer: int = 3
    enrichment_pseudocount: float = 1e-6
    bootstrap_iterations: int = 1000
    random_seed: int = 42

    require_stage_subset_checks: bool = True
    require_unique_final_ids: bool = True
    require_nonempty_paper_context_files: bool = True

    percentile_bands: Tuple[float, float] = (0.50, 0.80)

    figure_export: FigureExportConfig = field(default_factory=FigureExportConfig)

# =============================================================================
# Utility helpers
# =============================================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def print_header(msg: str) -> None:
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1_048_576), b""):
            h.update(chunk)
    return h.hexdigest()

def collect_environment_manifest() -> Dict[str, Any]:
    return {
        "timestamp_utc": now_utc_iso(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
        "cwd": str(Path.cwd()),
    }

def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    dirs = {
        "root": base_dir,
        "tables_main": base_dir / "tables_main",
        "tables_supplementary": base_dir / "tables_supplementary",
        "figures_main": base_dir / "figures_main",
        "figures_supplementary": base_dir / "figures_supplementary",
        "artifacts": base_dir / "artifacts",
        "reports": base_dir / "reports",
        "logs": base_dir / "logs",
        "source_data": base_dir / "source_data",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

def export_figure(fig: plt.Figure, out_base: Path, cfg: FigureExportConfig) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if cfg.export_pdf:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if cfg.export_png:
        fig.savefig(out_base.with_suffix(".png"), dpi=cfg.png_dpi, bbox_inches="tight")
    if cfg.export_tif:
        fig.savefig(out_base.with_suffix(".tif"), dpi=cfg.tif_dpi, bbox_inches="tight")
    plt.close(fig)

def deduplicate_columns_keep_last(df: pd.DataFrame) -> pd.DataFrame:
    if not df.columns.duplicated().any():
        return df
    return df.loc[:, ~df.columns.duplicated(keep="last")].copy()

def get_1d_column(df: pd.DataFrame, col: str) -> pd.Series:
    obj = df[col]
    if isinstance(obj, pd.DataFrame):
        return obj.iloc[:, -1]
    return obj

def first_existing_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = False, label: str = "") -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not resolve required column for {label}. Tried: {list(candidates)}")
    return None

def clean_sequence(seq: Any) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def aa_fraction(seq: str, aa: str) -> float:
    return seq.count(aa) / len(seq) if seq else 0.0

def shannon_entropy(seq: str) -> float:
    if not seq:
        return 0.0
    counts = Counter(seq)
    probs = np.array(list(counts.values()), dtype=float) / len(seq)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, None))).sum())

def max_run_fraction(seq: str) -> float:
    if not seq:
        return 1.0
    runs = [len(list(group)) for _, group in groupby(seq)]
    return float(max(runs) / len(seq))

def unique_residue_fraction(seq: str) -> float:
    return len(set(seq)) / len(seq) if seq else 0.0

def make_kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}

def jaccard_kmer_similarity(seq1: str, seq2: str, k: int = 3) -> float:
    s1 = make_kmers(seq1, k)
    s2 = make_kmers(seq2, k)
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    union = len(s1 | s2)
    return 0.0 if union == 0 else len(s1 & s2) / union

def compute_sequence_descriptors(seq: str) -> Dict[str, float]:
    seq = clean_sequence(seq)
    if not seq:
        desc = {f"aa_frac_{aa}": 0.0 for aa in AA_ALPHABET}
        desc.update({
            "length": 0.0,
            "mean_hydropathy": 0.0,
            "hydrophobic_frac": 0.0,
            "aromatic_frac": 0.0,
            "positive_frac": 0.0,
            "negative_frac": 0.0,
            "polar_frac": 0.0,
            "net_charge_proxy": 0.0,
            "entropy": 0.0,
            "max_run_fraction": 1.0,
            "unique_residue_fraction": 0.0,
        })
        return desc

    hydros = np.array([AA_PROPERTIES[a]["hydro"] for a in seq], dtype=float)
    desc = {f"aa_frac_{aa}": aa_fraction(seq, aa) for aa in AA_ALPHABET}
    desc.update({
        "length": float(len(seq)),
        "mean_hydropathy": float(hydros.mean()),
        "hydrophobic_frac": float(np.mean([a in AA_HYDROPHOBIC for a in seq])),
        "aromatic_frac": float(np.mean([a in AA_AROMATIC for a in seq])),
        "positive_frac": float(np.mean([a in AA_POSITIVE for a in seq])),
        "negative_frac": float(np.mean([a in AA_NEGATIVE for a in seq])),
        "polar_frac": float(np.mean([a in AA_POLAR for a in seq])),
        "net_charge_proxy": float(seq.count("K") + seq.count("R") + 0.1 * seq.count("H") - seq.count("D") - seq.count("E")),
        "entropy": shannon_entropy(seq),
        "max_run_fraction": max_run_fraction(seq),
        "unique_residue_fraction": unique_residue_fraction(seq),
    })
    return desc

def add_descriptors(df: pd.DataFrame, seq_col: str) -> pd.DataFrame:
    base = deduplicate_columns_keep_last(df.copy().reset_index(drop=True))
    desc_df = pd.DataFrame([compute_sequence_descriptors(s) for s in get_1d_column(base, seq_col).astype(str)])
    overlap = [c for c in desc_df.columns if c in base.columns and c != seq_col]
    if overlap:
        base = base.rename(columns={c: f"{c}_input" for c in overlap})
    return deduplicate_columns_keep_last(pd.concat([base, desc_df.reset_index(drop=True)], axis=1))

def bootstrap_ci_median(values: Sequence[float], n_boot: int = 1000, seed: int = 42) -> Tuple[float, float, float]:
    arr = np.asarray(list(values), dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return (np.nan, np.nan, np.nan)
    if len(arr) == 1:
        return (float(arr[0]), float(arr[0]), float(arr[0]))
    rng = np.random.default_rng(seed)
    meds = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        meds.append(np.median(sample))
    meds = np.asarray(meds, dtype=float)
    return (float(np.median(arr)), float(np.quantile(meds, 0.025)), float(np.quantile(meds, 0.975)))

def safe_numeric_series(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(get_1d_column(df, col), errors="coerce")

def robust_zscore_against_reference(values: pd.Series, ref_values: pd.Series) -> pd.Series:
    ref = pd.to_numeric(ref_values, errors="coerce").dropna().astype(float)
    vals = pd.to_numeric(values, errors="coerce").astype(float)
    if len(ref) == 0:
        return pd.Series(np.nan, index=values.index)
    med = float(np.median(ref))
    mad = float(np.median(np.abs(ref - med)))
    scale = 1.4826 * mad if mad > 1e-12 else float(np.std(ref, ddof=0))
    if scale <= 1e-12:
        return pd.Series(0.0, index=values.index)
    return (vals - med) / scale

def percentile_position_against_reference(values: pd.Series, ref_values: pd.Series) -> pd.Series:
    ref = pd.to_numeric(ref_values, errors="coerce").dropna().astype(float).values
    vals = pd.to_numeric(values, errors="coerce").astype(float).values
    if len(ref) == 0:
        return pd.Series(np.nan, index=values.index)
    ref_sorted = np.sort(ref)
    out = []
    for v in vals:
        if not np.isfinite(v):
            out.append(np.nan)
            continue
        rank = np.searchsorted(ref_sorted, v, side="right")
        out.append(rank / len(ref_sorted))
    return pd.Series(out, index=values.index)

# =============================================================================
# Path and file loading
# =============================================================================

def resolve_existing_path(primary: str, fallbacks: Sequence[str] = ()) -> Path:
    candidates = [primary] + list(fallbacks)
    for p in candidates:
        path = Path(p)
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing required file. Tried: {candidates}")

def load_csv_resolved(primary: str, fallbacks: Sequence[str] = ()) -> Tuple[pd.DataFrame, Path]:
    path = resolve_existing_path(primary, fallbacks)
    return deduplicate_columns_keep_last(pd.read_csv(path)), path

# =============================================================================
# Data harmonization
# =============================================================================

def prepare_stage_df(df: pd.DataFrame, stage_name: str, cfg: Step9Config) -> Tuple[pd.DataFrame, Dict[str, str]]:
    out = deduplicate_columns_keep_last(df.copy())
    seq_col = first_existing_column(out, cfg.sequence_col_candidates, required=True, label=f"{stage_name} sequence")
    id_col = first_existing_column(out, cfg.id_col_candidates, required=False)
    if id_col is None:
        id_col = "generated_id"
        out[id_col] = [f"{stage_name.lower().replace(' ', '_')}_{i:06d}" for i in range(len(out))]
    out[seq_col] = get_1d_column(out, seq_col).astype(str).map(clean_sequence)
    out = out[out[seq_col] != ""].reset_index(drop=True)
    out[id_col] = get_1d_column(out, id_col).astype(str)
    out = add_descriptors(out, seq_col)
    return out, {"sequence_col": seq_col, "id_col": id_col}

def prepare_paper_templates(df: pd.DataFrame) -> pd.DataFrame:
    out = deduplicate_columns_keep_last(df.copy())
    seq_col = "aligned_sequence" if "aligned_sequence" in out.columns else first_existing_column(out, ("sequence",), required=True, label="paper templates sequence")
    out[seq_col] = get_1d_column(out, seq_col).astype(str).map(clean_sequence)
    out = out[out[seq_col] != ""].drop_duplicates(subset=[seq_col]).reset_index(drop=True)
    if "paper_template_id" not in out.columns:
        out["paper_template_id"] = [f"paper_template_{i+1:03d}" for i in range(len(out))]
    out["paper_template_id"] = get_1d_column(out, "paper_template_id").astype(str)
    out = add_descriptors(out, seq_col)
    return out.rename(columns={seq_col: "sequence"})

def prepare_paper_candidates(df: pd.DataFrame) -> pd.DataFrame:
    out = deduplicate_columns_keep_last(df.copy())
    seq_col = first_existing_column(out, ("sequence",), required=True, label="paper candidates sequence")
    out[seq_col] = get_1d_column(out, seq_col).astype(str).map(clean_sequence)
    out = out[out[seq_col] != ""].drop_duplicates(subset=[seq_col]).reset_index(drop=True)
    if "paper_candidate_id" not in out.columns:
        out["paper_candidate_id"] = [f"paper_candidate_{i+1:03d}" for i in range(len(out))]
    out["paper_candidate_id"] = get_1d_column(out, "paper_candidate_id").astype(str)
    for col in ("is_template", "is_ac_p19", "is_ac_p19m"):
        if col not in out.columns:
            out[col] = "no"
    out = add_descriptors(out, seq_col)
    return out.rename(columns={seq_col: "sequence"})

# =============================================================================
# Stage audits
# =============================================================================

def assert_nonempty(df: pd.DataFrame, label: str) -> None:
    if len(df) == 0:
        raise ValueError(f"{label} is empty after harmonization.")

def assert_unique_ids(df: pd.DataFrame, id_col: str, label: str) -> None:
    dups = df[id_col].duplicated().sum()
    if dups > 0:
        dup_ids = df.loc[df[id_col].duplicated(), id_col].astype(str).tolist()[:10]
        raise ValueError(f"{label} contains duplicate IDs in column '{id_col}'. Example duplicates: {dup_ids}")

def check_stage_subset_by_sequence(child_df: pd.DataFrame, parent_df: pd.DataFrame, child_label: str, parent_label: str) -> Dict[str, Any]:
    parent_seqs = set(get_1d_column(parent_df, "sequence").astype(str))
    child_seqs = set(get_1d_column(child_df, "sequence").astype(str))
    missing = sorted(list(child_seqs - parent_seqs))
    return {
        "child_label": child_label,
        "parent_label": parent_label,
        "child_unique_n": int(len(child_seqs)),
        "parent_unique_n": int(len(parent_seqs)),
        "missing_in_parent_n": int(len(missing)),
        "missing_examples": missing[:10],
        "is_subset": int(len(missing) == 0),
    }

# =============================================================================
# Similarity and overlap
# =============================================================================

def nearest_neighbor_table(query_df: pd.DataFrame, ref_df: pd.DataFrame, query_id_col: str, ref_id_col: str, k: int) -> pd.DataFrame:
    ref_ids = get_1d_column(ref_df, ref_id_col).astype(str).tolist()
    ref_seqs = get_1d_column(ref_df, "sequence").astype(str).tolist()
    rows = []
    for _, row in query_df.iterrows():
        qseq = str(row["sequence"])
        sims = [jaccard_kmer_similarity(qseq, rseq, k=k) for rseq in ref_seqs]
        if not sims:
            rows.append({
                query_id_col: str(row[query_id_col]),
                "nearest_id": None,
                "nearest_similarity": np.nan,
                "nearest_sequence": None,
            })
            continue
        best_idx = int(np.argmax(sims))
        rows.append({
            query_id_col: str(row[query_id_col]),
            "nearest_id": ref_ids[best_idx],
            "nearest_similarity": float(sims[best_idx]),
            "nearest_sequence": ref_seqs[best_idx],
        })
    return pd.DataFrame(rows)

def exact_overlap_audit(query_df: pd.DataFrame, ref_df: pd.DataFrame, query_id_col: str, ref_label: str) -> pd.DataFrame:
    ref_set = set(get_1d_column(ref_df, "sequence").astype(str))
    rows = []
    for _, row in query_df.iterrows():
        rows.append({
            query_id_col: str(row[query_id_col]),
            "sequence": str(row["sequence"]),
            f"exact_overlap_vs_{ref_label}": int(str(row["sequence"]) in ref_set),
        })
    return pd.DataFrame(rows)

def pairwise_similarity_matrix(seqs: Sequence[str], k: int) -> np.ndarray:
    seqs = list(seqs)
    n = len(seqs)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            sim = jaccard_kmer_similarity(seqs[i], seqs[j], k=k)
            mat[i, j] = sim
            mat[j, i] = sim
    return mat

# =============================================================================
# Descriptive summaries
# =============================================================================

def compute_aa_usage(df: pd.DataFrame) -> pd.DataFrame:
    counts = Counter()
    total = 0
    for s in get_1d_column(df, "sequence").astype(str):
        counts.update(s)
        total += len(s)
    rows = []
    for aa in AA_ALPHABET:
        rows.append({
            "amino_acid": aa,
            "count": int(counts.get(aa, 0)),
            "fraction": counts.get(aa, 0) / total if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)

def compute_kmer_usage(df: pd.DataFrame, k: int = 3, top_k: int = 50) -> pd.DataFrame:
    counts = Counter()
    total = 0
    for s in get_1d_column(df, "sequence").astype(str):
        kmers = [s[i:i+k] for i in range(len(s) - k + 1)] if len(s) >= k else []
        counts.update(kmers)
        total += len(kmers)
    rows = []
    for kmer, c in counts.most_common(top_k):
        rows.append({
            "kmer": kmer,
            "count": int(c),
            "fraction": c / total if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)

def compute_aa_enrichment(final_df: pd.DataFrame, ref_df: pd.DataFrame, cfg: Step9Config) -> pd.DataFrame:
    final_usage = compute_aa_usage(final_df).rename(columns={"fraction": "final_fraction", "count": "final_count"})
    ref_usage = compute_aa_usage(ref_df).rename(columns={"fraction": "reference_fraction", "count": "reference_count"})
    merged = final_usage.merge(ref_usage, on="amino_acid", how="outer")
    merged["log2_enrichment_vs_reference"] = np.log2(
        (merged["final_fraction"] + cfg.enrichment_pseudocount) /
        (merged["reference_fraction"] + cfg.enrichment_pseudocount)
    )
    return merged.sort_values("log2_enrichment_vs_reference", ascending=False).reset_index(drop=True)

def compute_kmer_enrichment(final_df: pd.DataFrame, ref_df: pd.DataFrame, cfg: Step9Config, top_n: int = 20) -> pd.DataFrame:
    final_k = compute_kmer_usage(final_df, k=cfg.similarity_kmer, top_k=500).rename(columns={"fraction": "final_fraction", "count": "final_count"})
    ref_k = compute_kmer_usage(ref_df, k=cfg.similarity_kmer, top_k=2000).rename(columns={"fraction": "reference_fraction", "count": "reference_count"})
    merged = final_k.merge(ref_k, on="kmer", how="outer").fillna(0.0)
    merged["log2_enrichment_vs_reference"] = np.log2(
        (merged["final_fraction"] + cfg.enrichment_pseudocount) /
        (merged["reference_fraction"] + cfg.enrichment_pseudocount)
    )
    return merged.sort_values(["final_count", "log2_enrichment_vs_reference"], ascending=[False, False]).head(top_n).reset_index(drop=True)

def summarize_stage_metrics(df: pd.DataFrame, stage_name: str, cfg: Step9Config) -> pd.DataFrame:
    rows = []
    metric_map = {
        "novelty_score": first_existing_column(df, cfg.novelty_score_candidates),
        "plausibility_score": first_existing_column(df, cfg.plausibility_score_candidates),
        "diversity_score": first_existing_column(df, cfg.diversity_score_candidates),
        "final_score": first_existing_column(df, cfg.final_score_candidates),
    }
    for metric_name, col in metric_map.items():
        if col is None:
            continue
        vals = safe_numeric_series(df, col)
        med, lo, hi = bootstrap_ci_median(vals.dropna().tolist(), n_boot=cfg.bootstrap_iterations, seed=cfg.random_seed)
        rows.append({
            "stage": stage_name,
            "metric": metric_name,
            "n_non_missing": int(vals.notna().sum()),
            "median": med,
            "ci_low": lo,
            "ci_high": hi,
            "mean": float(vals.mean()) if vals.notna().any() else np.nan,
        })
    rows.append({
        "stage": stage_name,
        "metric": "count",
        "n_non_missing": int(len(df)),
        "median": float(len(df)),
        "ci_low": float(len(df)),
        "ci_high": float(len(df)),
        "mean": float(len(df)),
    })
    return pd.DataFrame(rows)

def summarize_descriptor_distribution(
    df: pd.DataFrame,
    group_name: str,
    descriptor_cols: Sequence[str],
    n_boot: int,
    seed: int
) -> pd.DataFrame:
    rows = []
    for col in descriptor_cols:
        vals = safe_numeric_series(df, col)
        med, lo, hi = bootstrap_ci_median(vals.dropna().tolist(), n_boot=n_boot, seed=seed)
        rows.append({
            "group": group_name,
            "descriptor": col,
            "n_non_missing": int(vals.notna().sum()),
            "median": med,
            "ci_low": lo,
            "ci_high": hi,
            "mean": float(vals.mean()) if vals.notna().any() else np.nan,
        })
    return pd.DataFrame(rows)

# =============================================================================
# Contextual support layer
# =============================================================================

def add_contextual_descriptor_positioning(
    final_df: pd.DataFrame,
    ref_df: pd.DataFrame,
    paper_candidates_df: pd.DataFrame,
    descriptor_cols: Sequence[str],
) -> pd.DataFrame:
    out = final_df.copy()
    for col in descriptor_cols:
        out[f"{col}_pct_vs_reference"] = percentile_position_against_reference(
            safe_numeric_series(out, col),
            safe_numeric_series(ref_df, col),
        )
        out[f"{col}_pct_vs_paper_candidates"] = percentile_position_against_reference(
            safe_numeric_series(out, col),
            safe_numeric_series(paper_candidates_df, col),
        )
        out[f"{col}_robust_z_vs_reference"] = robust_zscore_against_reference(
            safe_numeric_series(out, col),
            safe_numeric_series(ref_df, col),
        )
        out[f"{col}_robust_z_vs_paper_candidates"] = robust_zscore_against_reference(
            safe_numeric_series(out, col),
            safe_numeric_series(paper_candidates_df, col),
        )
    return out

def compute_contextual_support_summary(
    final_df: pd.DataFrame,
    descriptor_cols: Sequence[str],
    percentile_bands: Tuple[float, float],
) -> pd.DataFrame:
    low_band, high_band = percentile_bands
    rows = []
    for _, row in final_df.iterrows():
        support_notes = []

        ref_sim = float(row["nearest_reference_similarity"]) if pd.notna(row.get("nearest_reference_similarity", np.nan)) else np.nan
        paper_sim = float(row["nearest_paper_candidate_similarity"]) if pd.notna(row.get("nearest_paper_candidate_similarity", np.nan)) else np.nan

        if np.isfinite(ref_sim):
            support_notes.append(f"nearest_reference_similarity={ref_sim:.3f}")
        if np.isfinite(paper_sim):
            support_notes.append(f"nearest_paper_candidate_similarity={paper_sim:.3f}")

        for col in descriptor_cols:
            pct_paper = row.get(f"{col}_pct_vs_paper_candidates", np.nan)
            pct_ref = row.get(f"{col}_pct_vs_reference", np.nan)
            if pd.notna(pct_paper):
                if low_band <= pct_paper <= high_band:
                    support_notes.append(f"{col}: central_vs_paper({pct_paper:.2f})")
            elif pd.notna(pct_ref):
                if low_band <= pct_ref <= high_band:
                    support_notes.append(f"{col}: central_vs_reference({pct_ref:.2f})")

        rows.append({
            "generated_id": str(row["generated_id"]),
            "sequence": str(row["sequence"]),
            "nearest_reference_similarity": row.get("nearest_reference_similarity", np.nan),
            "nearest_paper_candidate_similarity": row.get("nearest_paper_candidate_similarity", np.nan),
            "exact_overlap_vs_reference_corpus": row.get("exact_overlap_vs_reference_corpus", np.nan),
            "exact_overlap_vs_paper_templates": row.get("exact_overlap_vs_paper_templates", np.nan),
            "exact_overlap_vs_paper_candidates": row.get("exact_overlap_vs_paper_candidates", np.nan),
            "contextual_support_summary": " | ".join(support_notes) if support_notes else "no_additional_context_note",
        })
    return pd.DataFrame(rows)

# =============================================================================
# Figure helpers
# =============================================================================

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.set_facecolor(PROJECT_COLORS["bg"])
    ax.spines["left"].set_color(PROJECT_COLORS["spine"])
    ax.spines["bottom"].set_color(PROJECT_COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(colors=PROJECT_COLORS["text"], width=0.7)
    ax.grid(True, axis=grid_axis, color=PROJECT_COLORS["grid"], linewidth=0.55)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str) -> None:
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, fontsize=10, fontweight="bold",
            ha="left", va="bottom", color=PROJECT_COLORS["text"])

# =============================================================================
# Main run logic
# =============================================================================

def run_stage(cfg: Step9Config, dirs: Dict[str, Path]) -> Dict[str, Any]:
    print_header("Loading inputs")

    ref_df_raw, ref_path = load_csv_resolved(cfg.step1_reference_file)
    passed_df_raw, passed_path = load_csv_resolved(cfg.step8_passed_file)
    shortlist_df_raw, shortlist_path = load_csv_resolved(cfg.step8_shortlist_file)
    final_df_raw, final_path = load_csv_resolved(cfg.step8_final_panel_file)
    paper_templates_raw, paper_templates_path = load_csv_resolved(cfg.paper_templates_file, cfg.paper_templates_fallbacks)
    paper_candidates_raw, paper_candidates_path = load_csv_resolved(cfg.paper_candidates_file, cfg.paper_candidates_fallbacks)

    ref_df, ref_cols = prepare_stage_df(ref_df_raw, "Reference ACP corpus", cfg)
    passed_df, passed_cols = prepare_stage_df(passed_df_raw, "Passed", cfg)
    shortlist_df, shortlist_cols = prepare_stage_df(shortlist_df_raw, "Shortlist", cfg)
    final_df, final_cols = prepare_stage_df(final_df_raw, "Final panel", cfg)
    paper_templates = prepare_paper_templates(paper_templates_raw)
    paper_candidates = prepare_paper_candidates(paper_candidates_raw)

    # Normalize sequence columns
    ref_df = ref_df.rename(columns={ref_cols["sequence_col"]: "sequence"})
    passed_df = passed_df.rename(columns={passed_cols["sequence_col"]: "sequence"})
    shortlist_df = shortlist_df.rename(columns={shortlist_cols["sequence_col"]: "sequence"})
    final_df = final_df.rename(columns={final_cols["sequence_col"]: "sequence"})
    ref_cols["sequence_col"] = passed_cols["sequence_col"] = shortlist_cols["sequence_col"] = final_cols["sequence_col"] = "sequence"

    # Normalize ID naming for downstream simplicity
    ref_df = ref_df.rename(columns={ref_cols["id_col"]: "reference_id"})
    passed_df = passed_df.rename(columns={passed_cols["id_col"]: "generated_id"})
    shortlist_df = shortlist_df.rename(columns={shortlist_cols["id_col"]: "generated_id"})
    final_df = final_df.rename(columns={final_cols["id_col"]: "generated_id"})

    assert_nonempty(ref_df, "Reference ACP corpus")
    assert_nonempty(passed_df, "Passed pool")
    assert_nonempty(shortlist_df, "Shortlist")
    assert_nonempty(final_df, "Final panel")
    assert_nonempty(paper_templates, "Paper templates")
    assert_nonempty(paper_candidates, "Paper candidates")

    if cfg.require_nonempty_paper_context_files:
        if len(paper_templates) == 0 or len(paper_candidates) == 0:
            raise ValueError("Paper-derived contextual-support files are empty.")

    if cfg.require_unique_final_ids:
        assert_unique_ids(final_df, "generated_id", "Final panel")

    # Stage subset checks by sequence
    shortlist_in_passed = check_stage_subset_by_sequence(shortlist_df, passed_df, "Shortlist", "Passed")
    final_in_shortlist = check_stage_subset_by_sequence(final_df, shortlist_df, "Final panel", "Shortlist")
    final_in_passed = check_stage_subset_by_sequence(final_df, passed_df, "Final panel", "Passed")

    if cfg.require_stage_subset_checks:
        failing = [x for x in [shortlist_in_passed, final_in_shortlist, final_in_passed] if x["is_subset"] == 0]
        if failing:
            raise ValueError(f"Stage subset checks failed: {json.dumps(failing, indent=2)}")

    # Provenance manifest
    save_json(
        {
            "config": asdict(cfg),
            "environment_manifest": collect_environment_manifest(),
            "scientific_scope": {
                "step_name": "Step 9 — paper-derived contextual support only",
                "claims_not_supported": [
                    "independent external validation",
                    "orthogonal wet-lab confirmation",
                    "external predictive-tool validation",
                    "proof of anticancer activity",
                ],
            },
            "resolved_input_paths": {
                "step1_reference_file": str(ref_path),
                "step8_passed_file": str(passed_path),
                "step8_shortlist_file": str(shortlist_path),
                "step8_final_panel_file": str(final_path),
                "paper_templates_file": str(paper_templates_path),
                "paper_candidates_file": str(paper_candidates_path),
            },
            "input_hashes": {
                "step1_reference_sha256": sha256_of_file(ref_path),
                "step8_passed_sha256": sha256_of_file(passed_path),
                "step8_shortlist_sha256": sha256_of_file(shortlist_path),
                "step8_final_panel_sha256": sha256_of_file(final_path),
                "paper_templates_sha256": sha256_of_file(paper_templates_path),
                "paper_candidates_sha256": sha256_of_file(paper_candidates_path),
            },
            "subset_checks": {
                "shortlist_in_passed": shortlist_in_passed,
                "final_in_shortlist": final_in_shortlist,
                "final_in_passed": final_in_passed,
            },
        },
        dirs["artifacts"] / "run_manifest.json"
    )

    print_header("Computing overlap and nearest-neighbor audits")

    passed_ref_nn = nearest_neighbor_table(passed_df, ref_df.rename(columns={"reference_id": "ref_id"}), "generated_id", "ref_id", cfg.similarity_kmer)
    shortlist_ref_nn = nearest_neighbor_table(shortlist_df, ref_df.rename(columns={"reference_id": "ref_id"}), "generated_id", "ref_id", cfg.similarity_kmer)
    final_ref_nn = nearest_neighbor_table(final_df, ref_df.rename(columns={"reference_id": "ref_id"}), "generated_id", "ref_id", cfg.similarity_kmer)

    passed_df = passed_df.merge(
        passed_ref_nn.rename(columns={
            "nearest_id": "nearest_reference_id",
            "nearest_similarity": "nearest_reference_similarity",
            "nearest_sequence": "nearest_reference_sequence",
        }),
        on="generated_id", how="left"
    )
    shortlist_df = shortlist_df.merge(
        shortlist_ref_nn.rename(columns={
            "nearest_id": "nearest_reference_id",
            "nearest_similarity": "nearest_reference_similarity",
            "nearest_sequence": "nearest_reference_sequence",
        }),
        on="generated_id", how="left"
    )
    final_df = final_df.merge(
        final_ref_nn.rename(columns={
            "nearest_id": "nearest_reference_id",
            "nearest_similarity": "nearest_reference_similarity",
            "nearest_sequence": "nearest_reference_sequence",
        }),
        on="generated_id", how="left"
    )

    final_paper_nn = nearest_neighbor_table(
        final_df,
        paper_candidates.rename(columns={"paper_candidate_id": "paper_id"}),
        "generated_id",
        "paper_id",
        cfg.similarity_kmer,
    )
    final_df = final_df.merge(
        final_paper_nn.rename(columns={
            "nearest_id": "nearest_paper_candidate_id",
            "nearest_similarity": "nearest_paper_candidate_similarity",
            "nearest_sequence": "nearest_paper_candidate_sequence",
        }),
        on="generated_id",
        how="left",
    )

    final_overlap_ref = exact_overlap_audit(final_df, ref_df, "generated_id", "reference_corpus")
    final_overlap_templates = exact_overlap_audit(final_df, paper_templates.rename(columns={"paper_template_id": "paper_id"}), "generated_id", "paper_templates")
    final_overlap_candidates = exact_overlap_audit(final_df, paper_candidates.rename(columns={"paper_candidate_id": "paper_id"}), "generated_id", "paper_candidates")
    paper_candidates_overlap_ref = exact_overlap_audit(
        paper_candidates.rename(columns={"paper_candidate_id": "paper_id"}),
        ref_df,
        "paper_id",
        "reference_corpus",
    )

    final_overlap_audit = final_overlap_ref.merge(final_overlap_templates, on=["generated_id", "sequence"], how="left")
    final_overlap_audit = final_overlap_audit.merge(final_overlap_candidates, on=["generated_id", "sequence"], how="left")
    final_df = final_df.merge(final_overlap_audit, on=["generated_id", "sequence"], how="left")

    print_header("Computing inherited evidence and contextual-support summaries")

    inherited_stage_summary_df = pd.concat([
        summarize_stage_metrics(passed_df, "Passed", cfg),
        summarize_stage_metrics(shortlist_df, "Shortlist", cfg),
        summarize_stage_metrics(final_df, "Final panel", cfg),
    ], ignore_index=True)

    descriptor_cols = ["length", "net_charge_proxy", "mean_hydropathy", "entropy", "unique_residue_fraction"]

    descriptor_distribution_df = pd.concat([
        summarize_descriptor_distribution(ref_df, "Reference ACP corpus", descriptor_cols, cfg.bootstrap_iterations, cfg.random_seed),
        summarize_descriptor_distribution(paper_candidates, "Paper candidates", descriptor_cols, cfg.bootstrap_iterations, cfg.random_seed),
        summarize_descriptor_distribution(final_df, "Final panel", descriptor_cols, cfg.bootstrap_iterations, cfg.random_seed),
    ], ignore_index=True)

    final_df = add_contextual_descriptor_positioning(
        final_df=final_df,
        ref_df=ref_df,
        paper_candidates_df=paper_candidates,
        descriptor_cols=descriptor_cols,
    )

    contextual_support_df = compute_contextual_support_summary(
        final_df=final_df,
        descriptor_cols=descriptor_cols,
        percentile_bands=cfg.percentile_bands,
    )

    aa_enrichment_df = compute_aa_enrichment(final_df, ref_df, cfg)
    kmer_enrichment_df = compute_kmer_enrichment(final_df, ref_df, cfg, top_n=20)

    pairwise_mat = pairwise_similarity_matrix(get_1d_column(final_df, "sequence").astype(str).tolist(), cfg.similarity_kmer)
    pairwise_df = pd.DataFrame(
        pairwise_mat,
        index=get_1d_column(final_df, "generated_id").astype(str),
        columns=get_1d_column(final_df, "generated_id").astype(str),
    )

    # Candidate pool count logic with explicit provenance
    count_provenance = []
    candidate_pool_path = Path(cfg.step8_candidate_pool_file)
    if candidate_pool_path.exists():
        candidate_pool_df = pd.read_csv(candidate_pool_path)
        generated_count = int(len(candidate_pool_df))
        count_provenance.append(f"Generated candidate pool count loaded from {candidate_pool_path}")
    else:
        generated_count = int(len(passed_df))
        count_provenance.append(
            f"Generated candidate pool file missing; used passed-pool count ({generated_count}) as fallback."
        )

    filter_audit_path = Path(cfg.step8_filter_audit_file)
    passed_count_for_audit = int(len(passed_df))
    if filter_audit_path.exists():
        filt = pd.read_csv(filter_audit_path)
        count_provenance.append(f"Step 8 filter audit file found: {filter_audit_path}")
        _ = filt  # retained for provenance only
    else:
        count_provenance.append("Step 8 filter audit file missing; passed-pool count used directly.")

    selection_audit_df = pd.DataFrame([
        {"stage": "Generated / candidate pool", "count": generated_count},
        {"stage": "Passed pool", "count": passed_count_for_audit},
        {"stage": "Shortlist", "count": int(len(shortlist_df))},
        {"stage": "Final panel", "count": int(len(final_df))},
    ])

    # Build inherited evidence table separately from contextual-support evidence
    inherited_cols = ["generated_id", "sequence"]
    for c in [
        first_existing_column(final_df, cfg.novelty_score_candidates),
        first_existing_column(final_df, cfg.plausibility_score_candidates),
        first_existing_column(final_df, cfg.diversity_score_candidates),
        first_existing_column(final_df, cfg.final_score_candidates),
    ]:
        if c is not None and c in final_df.columns and c not in inherited_cols:
            inherited_cols.append(c)

    final_inherited_evidence_df = final_df[inherited_cols].copy()

    contextual_cols = [
        "generated_id",
        "sequence",
        "nearest_reference_id",
        "nearest_reference_similarity",
        "nearest_paper_candidate_id",
        "nearest_paper_candidate_similarity",
        "exact_overlap_vs_reference_corpus",
        "exact_overlap_vs_paper_templates",
        "exact_overlap_vs_paper_candidates",
    ]
    contextual_cols += [c for c in final_df.columns if any(
        c.endswith(suffix) for suffix in (
            "_pct_vs_reference",
            "_pct_vs_paper_candidates",
            "_robust_z_vs_reference",
            "_robust_z_vs_paper_candidates",
        )
    )]
    final_contextual_evidence_df = final_df[[c for c in contextual_cols if c in final_df.columns]].copy()

    final_integrated_evidence_df = final_inherited_evidence_df.merge(
        contextual_support_df,
        on=["generated_id", "sequence"],
        how="left",
        suffixes=("", "_ctx"),
    )
    final_integrated_evidence_df = final_integrated_evidence_df.merge(
        final_contextual_evidence_df.drop(columns=["sequence"]),
        on="generated_id",
        how="left",
        suffixes=("", "_ctx2"),
    )
    final_integrated_evidence_df["support_scope_note"] = (
        "Paper-derived contextual support only; not independent external validation."
    )

    # Save tables
    final_integrated_evidence_df.to_csv(
        dirs["tables_main"] / "table_9_1_final_panel_integrated_evidence.csv", index=False
    )
    selection_audit_df.to_csv(
        dirs["tables_main"] / "table_9_2_selection_audit_counts.csv", index=False
    )

    inherited_stage_summary_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_1_selection_stage_score_shift_summary.csv", index=False
    )
    final_overlap_audit.to_csv(
        dirs["tables_supplementary"] / "table_s9_2_final_panel_overlap_audit.csv", index=False
    )
    paper_candidates_overlap_ref.to_csv(
        dirs["tables_supplementary"] / "table_s9_3_paper_candidates_overlap_vs_reference.csv", index=False
    )
    descriptor_distribution_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_4_descriptor_distribution_summary.csv", index=False
    )
    aa_enrichment_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_5_amino_acid_enrichment_vs_reference.csv", index=False
    )
    kmer_enrichment_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_6_kmer_enrichment_vs_reference.csv", index=False
    )
    pairwise_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_7_final_panel_pairwise_similarity.csv"
    )
    final_contextual_evidence_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_8_final_panel_contextual_support_detail.csv", index=False
    )
    final_df.to_csv(
        dirs["tables_supplementary"] / "table_s9_9_final_panel_with_all_step9_annotations.csv", index=False
    )
    paper_templates.to_csv(
        dirs["tables_supplementary"] / "table_s9_10_paper_templates_harmonized.csv", index=False
    )
    paper_candidates.to_csv(
        dirs["tables_supplementary"] / "table_s9_11_paper_candidates_harmonized.csv", index=False
    )
    pd.DataFrame([shortlist_in_passed, final_in_shortlist, final_in_passed]).to_csv(
        dirs["tables_supplementary"] / "table_s9_12_stage_subset_audits.csv", index=False
    )

    # Source data
    selection_audit_df.to_csv(
        dirs["source_data"] / "Fig9_panel_a_selection_audit_source_data.csv", index=False
    )
    inherited_stage_summary_df.to_csv(
        dirs["source_data"] / "Fig9_panel_b_inherited_stage_score_shift_source_data.csv", index=False
    )
    final_contextual_evidence_df.to_csv(
        dirs["source_data"] / "Fig9_panel_c_contextual_support_map_source_data.csv", index=False
    )
    descriptor_distribution_df.to_csv(
        dirs["source_data"] / "Fig9_panel_d_descriptor_positioning_source_data.csv", index=False
    )

    limitations_text = f"""
Step 9 scope statement
----------------------
This step provides paper-derived contextual support only.

It does not constitute:
- independent external validation
- orthogonal wet-lab confirmation
- external predictive-tool validation
- proof of anticancer activity

Interpretation guidance
----------------------
- nearest-reference similarity and nearest-paper-candidate similarity are contextual comparisons only
- exact overlap with paper-derived sets is descriptive and should not be framed as validation
- descriptor positioning summarizes whether final-panel peptides fall within plausible contextual ranges
- inherited selection metrics (novelty, plausibility, diversity, final score) originate from earlier prioritization steps and should not be reinterpreted here as new validation evidence

Count provenance
----------------
{chr(10).join(count_provenance)}
""".strip()
    save_text(limitations_text, dirs["reports"] / "limitations_and_scope_statement.txt")

    summary_lines = [
        "Step 9 — Paper-derived contextual support only",
        "=" * 88,
        f"Timestamp UTC: {now_utc_iso()}",
        "",
        f"Reference corpus size: {len(ref_df):,}",
        f"Passed size: {len(passed_df):,}",
        f"Shortlist size: {len(shortlist_df):,}",
        f"Final panel size: {len(final_df):,}",
        f"Paper template sequences: {len(paper_templates):,}",
        f"Paper candidate sequences: {len(paper_candidates):,}",
        "",
        "Stage subset checks:",
        f"- shortlist in passed: {bool(shortlist_in_passed['is_subset'])}",
        f"- final in shortlist: {bool(final_in_shortlist['is_subset'])}",
        f"- final in passed: {bool(final_in_passed['is_subset'])}",
        "",
        "Final-panel exact overlap summary:",
        f"- vs reference corpus: {int(final_overlap_audit['exact_overlap_vs_reference_corpus'].sum())}",
        f"- vs paper templates: {int(final_overlap_audit['exact_overlap_vs_paper_templates'].sum())}",
        f"- vs paper candidates: {int(final_overlap_audit['exact_overlap_vs_paper_candidates'].sum())}",
        "",
        "Scientific scope:",
        "- paper-derived contextual support only",
        "- not independent external validation",
    ]
    save_text("\n".join(summary_lines), dirs["reports"] / "summary_report.txt")
    save_text("\n".join(count_provenance), dirs["reports"] / "count_provenance_report.txt")

    return {
        "ref_df": ref_df,
        "passed_df": passed_df,
        "shortlist_df": shortlist_df,
        "final_df": final_df,
        "paper_templates": paper_templates,
        "paper_candidates": paper_candidates,
        "selection_audit_df": selection_audit_df,
        "inherited_stage_summary_df": inherited_stage_summary_df,
        "descriptor_distribution_df": descriptor_distribution_df,
        "final_inherited_evidence_df": final_inherited_evidence_df,
        "final_contextual_evidence_df": final_contextual_evidence_df,
        "final_integrated_evidence_df": final_integrated_evidence_df,
        "final_overlap_audit": final_overlap_audit,
        "contextual_support_df": contextual_support_df,
        "aa_enrichment_df": aa_enrichment_df,
        "kmer_enrichment_df": kmer_enrichment_df,
        "pairwise_df": pairwise_df,
        "count_provenance": count_provenance,
        "subset_checks": {
            "shortlist_in_passed": shortlist_in_passed,
            "final_in_shortlist": final_in_shortlist,
            "final_in_passed": final_in_passed,
        },
    }

# =============================================================================
# Plotting
# =============================================================================

def plot_main_figure(results: Dict[str, Any], cfg: Step9Config, dirs: Dict[str, Path]) -> None:
    selection_audit_df = results["selection_audit_df"]
    inherited_stage_summary_df = results["inherited_stage_summary_df"]
    final_contextual_evidence_df = results["final_contextual_evidence_df"]
    descriptor_distribution_df = results["descriptor_distribution_df"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 8.1))
    gs = GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.36)

    # Panel a: selection audit
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="y")
    x = np.arange(len(selection_audit_df))
    colors = [
        GROUP_COLORS["Candidate pool"],
        GROUP_COLORS["Passed"],
        GROUP_COLORS["Shortlist"],
        GROUP_COLORS["Final panel"],
    ]
    ax.bar(x, selection_audit_df["count"], color=colors, edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(selection_audit_df["stage"], rotation=20, ha="right")
    ax.set_ylabel("Peptide count")
    ax.set_title("Selection audit across candidate-reduction stages", loc="left", pad=4)
    for xi, yi in zip(x, selection_audit_df["count"]):
        ax.text(xi, yi, str(int(yi)), ha="center", va="bottom", fontsize=7)

    # Panel b: inherited Step 8 score shift
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="y")
    metric_display = {
        "novelty_score": "Novelty",
        "plausibility_score": "Plausibility",
        "diversity_score": "Diversity",
        "final_score": "Final score",
    }
    plot_df = inherited_stage_summary_df[inherited_stage_summary_df["metric"].isin(metric_display.keys())].copy()
    stages = ["Passed", "Shortlist", "Final panel"]
    metrics = [m for m in metric_display.keys() if m in plot_df["metric"].unique()]
    width = 0.22
    xpos = np.arange(len(metrics))
    offsets = {"Passed": -width, "Shortlist": 0.0, "Final panel": width}

    for stage in stages:
        sub = plot_df[plot_df["stage"] == stage].set_index("metric")
        vals = [sub.loc[m, "median"] if m in sub.index else np.nan for m in metrics]
        lo = [sub.loc[m, "ci_low"] if m in sub.index else np.nan for m in metrics]
        hi = [sub.loc[m, "ci_high"] if m in sub.index else np.nan for m in metrics]
        vals_arr = np.asarray(vals, dtype=float)
        yerr = np.vstack([vals_arr - np.asarray(lo, dtype=float), np.asarray(hi, dtype=float) - vals_arr])
        ax.bar(
            xpos + offsets[stage],
            vals_arr,
            width=width,
            yerr=yerr,
            capsize=2.5,
            color=GROUP_COLORS[stage],
            edgecolor=PROJECT_COLORS["gray_dark"],
            linewidth=0.5,
            label=stage,
        )
    ax.set_xticks(xpos)
    ax.set_xticklabels([metric_display[m] for m in metrics])
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Median score")
    ax.set_title("Selection-stage shift in inherited prioritization metrics", loc="left", pad=4)
    ax.legend(frameon=False, loc="lower right")

    # Panel c: contextual support map only
    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    heat_cols = []
    heat_labels = []
    for c, lab in [
        ("nearest_reference_similarity", "Ref sim"),
        ("nearest_paper_candidate_similarity", "Paper sim"),
        ("exact_overlap_vs_reference_corpus", "Ref overlap"),
        ("exact_overlap_vs_paper_templates", "Template overlap"),
        ("exact_overlap_vs_paper_candidates", "Paper overlap"),
    ]:
        if c in final_contextual_evidence_df.columns:
            vals = pd.to_numeric(get_1d_column(final_contextual_evidence_df, c), errors="coerce")
            if vals.notna().any():
                heat_cols.append(c)
                heat_labels.append(lab)

    tmp = final_contextual_evidence_df.copy().reset_index(drop=True)
    mat = np.column_stack([pd.to_numeric(get_1d_column(tmp, c), errors="coerce").fillna(0).to_numpy() for c in heat_cols])
    mat = np.clip(mat, 0.0, 1.0)
    im = ax.imshow(mat, aspect="auto", vmin=0, vmax=1, cmap="Blues")
    ylab = get_1d_column(tmp, "generated_id").astype(str).tolist()
    ax.set_yticks(np.arange(len(ylab)))
    ax.set_yticklabels(ylab, fontsize=6)
    ax.set_xticks(np.arange(len(heat_labels)))
    ax.set_xticklabels(heat_labels, rotation=30, ha="right")
    ax.set_title("Final-panel contextual-support map", loc="left", pad=4)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if v > 0.55 else PROJECT_COLORS["text"])
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    cb.set_label("Contextual score")

    # Panel d: descriptor positioning
    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    style_axis(ax, grid_axis="y")
    desc_plot = descriptor_distribution_df[
        descriptor_distribution_df["descriptor"].isin(["length", "net_charge_proxy", "mean_hydropathy", "entropy"])
    ].copy()

    groups = ["Reference ACP corpus", "Paper candidates", "Final panel"]
    descriptors = ["length", "net_charge_proxy", "mean_hydropathy", "entropy"]
    width = 0.22
    xpos = np.arange(len(descriptors))
    offsets = {
        "Reference ACP corpus": -width,
        "Paper candidates": 0.0,
        "Final panel": width,
    }
    color_map = {
        "Reference ACP corpus": GROUP_COLORS["Reference ACP corpus"],
        "Paper candidates": GROUP_COLORS["Paper candidates"],
        "Final panel": GROUP_COLORS["Final panel"],
    }

    for g in groups:
        sub = desc_plot[desc_plot["group"] == g].set_index("descriptor")
        vals = [sub.loc[d, "median"] if d in sub.index else np.nan for d in descriptors]
        lo = [sub.loc[d, "ci_low"] if d in sub.index else np.nan for d in descriptors]
        hi = [sub.loc[d, "ci_high"] if d in sub.index else np.nan for d in descriptors]
        vals_arr = np.asarray(vals, dtype=float)
        yerr = np.vstack([vals_arr - np.asarray(lo, dtype=float), np.asarray(hi, dtype=float) - vals_arr])
        ax.bar(
            xpos + offsets[g],
            vals_arr,
            width=width,
            yerr=yerr,
            capsize=2.3,
            color=color_map[g],
            edgecolor=PROJECT_COLORS["gray_dark"],
            linewidth=0.5,
            label=g,
        )
    ax.set_xticks(xpos)
    ax.set_xticklabels(["Length", "Charge", "Hydropathy", "Entropy"])
    ax.set_title("Descriptor positioning of final-panel peptides", loc="left", pad=4)
    ax.set_ylabel("Median descriptor value")
    ax.legend(frameon=False, loc="best")

    fig.tight_layout()
    export_figure(fig, dirs["figures_main"] / "figure_9_main_paper_contextual_support_v2", cfg.figure_export)

def plot_supplementary_descriptor_context(results: Dict[str, Any], cfg: Step9Config, dirs: Dict[str, Path]) -> None:
    aa_enrichment_df = results["aa_enrichment_df"]
    kmer_enrichment_df = results["kmer_enrichment_df"]
    final_df = results["final_df"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.8))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34)

    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="x")
    tmp = aa_enrichment_df.sort_values("log2_enrichment_vs_reference", ascending=True)
    colors = [PROJECT_COLORS["light_blue"] if x < 0 else PROJECT_COLORS["deep_red"] for x in tmp["log2_enrichment_vs_reference"]]
    ax.barh(tmp["amino_acid"], tmp["log2_enrichment_vs_reference"], color=colors, edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.4)
    ax.axvline(0, color=PROJECT_COLORS["gray_dark"], linewidth=0.8)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Amino-acid enrichment in final panel", loc="left", pad=4)

    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="x")
    tmp = kmer_enrichment_df.sort_values("log2_enrichment_vs_reference", ascending=True)
    ax.barh(tmp["kmer"], tmp["log2_enrichment_vs_reference"], color=PROJECT_COLORS["teal"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.4)
    ax.axvline(0, color=PROJECT_COLORS["gray_dark"], linewidth=0.8)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Enriched 3-mers in final panel", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    style_axis(ax, grid_axis="both")
    ax.scatter(
        safe_numeric_series(final_df, "mean_hydropathy"),
        safe_numeric_series(final_df, "net_charge_proxy"),
        s=55,
        facecolor=PROJECT_COLORS["deep_red"],
        edgecolor="white",
        linewidth=0.55,
        alpha=0.95,
    )
    for _, r in final_df.iterrows():
        ax.text(float(r["mean_hydropathy"]) + 0.03, float(r["net_charge_proxy"]) + 0.03, str(r["generated_id"]), fontsize=6)
    ax.set_xlabel("Mean hydropathy")
    ax.set_ylabel("Net charge proxy")
    ax.set_title("Charge–hydropathy landscape", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    style_axis(ax, grid_axis="both")
    ax.scatter(
        safe_numeric_series(final_df, "entropy"),
        safe_numeric_series(final_df, "unique_residue_fraction"),
        s=55,
        facecolor=PROJECT_COLORS["navy"],
        edgecolor="white",
        linewidth=0.55,
        alpha=0.95,
    )
    for _, r in final_df.iterrows():
        ax.text(float(r["entropy"]) + 0.01, float(r["unique_residue_fraction"]) + 0.005, str(r["generated_id"]), fontsize=6)
    ax.set_xlabel("Sequence entropy")
    ax.set_ylabel("Unique residue fraction")
    ax.set_title("Complexity profile of final-panel peptides", loc="left", pad=4)

    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_9_1_descriptor_context_v2", cfg.figure_export)

def plot_supplementary_similarity_context(results: Dict[str, Any], cfg: Step9Config, dirs: Dict[str, Path]) -> None:
    final_df = results["final_df"]
    final_overlap_audit = results["final_overlap_audit"]
    pairwise_df = results["pairwise_df"]

    fig = plt.figure(figsize=(cfg.figure_export.double_col_width_in, 6.6))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.34)

    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="y")
    vals = safe_numeric_series(final_df, "nearest_reference_similarity")
    ax.bar(np.arange(len(final_df)), vals.fillna(0.0), color=PROJECT_COLORS["light_blue"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5)
    ax.set_xticks(np.arange(len(final_df)))
    ax.set_xticklabels(get_1d_column(final_df, "generated_id").astype(str), rotation=45, ha="right", fontsize=6)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("3-mer Jaccard similarity")
    ax.set_title("Nearest-reference similarity", loc="left", pad=4)

    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    im = ax.imshow(pairwise_df.to_numpy(), vmin=0, vmax=1, cmap="Blues")
    labels = pairwise_df.index.tolist()
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
    ax.set_yticklabels(labels, fontsize=6)
    ax.set_title("Pairwise similarity within final panel", loc="left", pad=4)
    cb = fig.colorbar(im, ax=ax, fraction=0.055, pad=0.04)
    cb.set_label("3-mer Jaccard similarity")

    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    style_axis(ax, grid_axis="y")
    vals = safe_numeric_series(final_df, "nearest_paper_candidate_similarity")
    ax.bar(np.arange(len(final_df)), vals.fillna(0.0), color=PROJECT_COLORS["deep_red"], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5)
    ax.set_xticks(np.arange(len(final_df)))
    ax.set_xticklabels(get_1d_column(final_df, "generated_id").astype(str), rotation=45, ha="right", fontsize=6)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("3-mer Jaccard similarity")
    ax.set_title("Nearest paper-candidate similarity", loc="left", pad=4)

    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "d")
    style_axis(ax, grid_axis="y")
    overlap_counts = pd.DataFrame({
        "category": ["vs reference", "vs templates", "vs candidates"],
        "count": [
            int(final_overlap_audit["exact_overlap_vs_reference_corpus"].sum()),
            int(final_overlap_audit["exact_overlap_vs_paper_templates"].sum()),
            int(final_overlap_audit["exact_overlap_vs_paper_candidates"].sum()),
        ]
    })
    ax.bar(np.arange(len(overlap_counts)), overlap_counts["count"], color=[
        PROJECT_COLORS["light_blue"],
        PROJECT_COLORS["gray_mid"],
        PROJECT_COLORS["deep_red"],
    ], edgecolor=PROJECT_COLORS["gray_dark"], linewidth=0.5)
    ax.set_xticks(np.arange(len(overlap_counts)))
    ax.set_xticklabels(overlap_counts["category"], rotation=20, ha="right")
    ax.set_ylabel("Exact overlaps")
    ax.set_title("Paper-derived overlap audit summary", loc="left", pad=4)
    for xi, yi in zip(np.arange(len(overlap_counts)), overlap_counts["count"]):
        ax.text(xi, yi, str(int(yi)), ha="center", va="bottom", fontsize=7)

    fig.tight_layout()
    export_figure(fig, dirs["figures_supplementary"] / "sfigure_9_2_similarity_context_v2", cfg.figure_export)

# =============================================================================
# Entrypoint
# =============================================================================

def main_notebook(cfg: Optional[Step9Config] = None) -> Dict[str, Any]:
    if cfg is None:
        cfg = Step9Config()

    seed_all(cfg.random_seed)
    step_root = Path(cfg.project_root).resolve() / cfg.step9_dir
    dirs = make_output_dirs(step_root)

    print_header("Running Step 9 — Paper-derived contextual support only")
    print(json.dumps({
        "step1_reference_file": cfg.step1_reference_file,
        "step8_passed_file": cfg.step8_passed_file,
        "step8_shortlist_file": cfg.step8_shortlist_file,
        "step8_final_panel_file": cfg.step8_final_panel_file,
        "paper_templates_file_primary": cfg.paper_templates_file,
        "paper_candidates_file_primary": cfg.paper_candidates_file,
        "similarity_kmer": cfg.similarity_kmer,
        "scientific_scope": "paper-derived contextual support only",
    }, indent=2))

    results = run_stage(cfg, dirs)
    plot_main_figure(results, cfg, dirs)
    plot_supplementary_descriptor_context(results, cfg, dirs)
    plot_supplementary_similarity_context(results, cfg, dirs)

    save_json(
        {
            "n_reference": int(len(results["ref_df"])),
            "n_passed": int(len(results["passed_df"])),
            "n_shortlist": int(len(results["shortlist_df"])),
            "n_final_panel": int(len(results["final_df"])),
            "n_paper_templates": int(len(results["paper_templates"])),
            "n_paper_candidates": int(len(results["paper_candidates"])),
            "subset_checks": results["subset_checks"],
        },
        dirs["logs"] / "stage_counts.json",
    )

    print("\n" + "=" * 100)
    print("Step 9 completed successfully.")
    print(f"Main tables: {dirs['tables_main']}")
    print(f"Supplementary tables: {dirs['tables_supplementary']}")
    print(f"Main figures: {dirs['figures_main']}")
    print(f"Supplementary figures: {dirs['figures_supplementary']}")
    print(f"Artifacts: {dirs['artifacts']}")
    print(f"Reports: {dirs['reports']}")
    print(f"Logs: {dirs['logs']}")
    print("=" * 100)

    return results

if __name__ == "__main__":
    results = main_notebook()

main figure redeisgn

In [ ]:
from __future__ import annotations

# =============================================================================
# High-resolution standalone redesigned main figure for Step 9
# Final polish pass
# =============================================================================

from pathlib import Path
from typing import Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.lines import Line2D

# =============================================================================
# Paths
# =============================================================================

STEP9_DIR = Path("/home/data3/Moe/nature_computational2/step9_v1")

SELECTION_AUDIT_FILE = STEP9_DIR / "tables_main" / "table_9_2_selection_audit_counts.csv"
STAGE_SUMMARY_FILE = STEP9_DIR / "tables_supplementary" / "table_s9_1_selection_stage_score_shift_summary.csv"
DESCRIPTOR_SUMMARY_FILE = STEP9_DIR / "tables_supplementary" / "table_s9_4_descriptor_distribution_summary.csv"

OUTDIR = STEP9_DIR / "figures_main"
OUT_BASENAME = "figure_9_main_redesigned_highres_v2"

# =============================================================================
# Export settings
# =============================================================================

EXPORT_PDF = True
EXPORT_PNG = True
EXPORT_TIF = True
PNG_DPI = 600
TIF_DPI = 600

# =============================================================================
# Style
# =============================================================================

COLORS = {
    "gray_mid": "#A6A6A6",
    "teal": "#2F7F7B",
    "orange": "#D55E00",
    "navy": "#3B62A3",
    "light_blue": "#A8B9DC",
    "deep_red": "#C7001E",
    "bg": "#FFFFFF",
    "grid": "#F2F2F2",
    "spine": "#B8B8B8",
    "text": "#222222",
    "gray_dark": "#4A4A4A",
}

plt.rcParams.update({
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 16,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# =============================================================================
# Helpers
# =============================================================================

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file: {path}")

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(colors=COLORS["text"], width=0.8, length=5)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=COLORS["grid"], linewidth=0.75, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str, x: float = -0.08, y: float = 1.03) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=20,
        fontweight="bold",
        ha="left",
        va="bottom",
        color=COLORS["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=PNG_DPI, bbox_inches="tight")
    if EXPORT_TIF:
        fig.savefig(out_base.with_suffix(".tif"), dpi=TIF_DPI, bbox_inches="tight")
    plt.close(fig)

def get_metric_row(df: pd.DataFrame, stage: str, metric: str) -> Tuple[float, float, float]:
    sub = df[(df["stage"] == stage) & (df["metric"] == metric)]
    if len(sub) == 0:
        return (np.nan, np.nan, np.nan)
    row = sub.iloc[0]
    return float(row["median"]), float(row["ci_low"]), float(row["ci_high"])

def make_boxed_legend(ax_or_fig, handles, labels, *, loc, bbox_to_anchor, ncol=3):
    leg = ax_or_fig.legend(
        handles,
        labels,
        loc=loc,
        bbox_to_anchor=bbox_to_anchor,
        ncol=ncol,
        frameon=True,
        fancybox=True,
        borderpad=0.32,
        handlelength=1.15,
        handletextpad=0.45,
        columnspacing=0.9,
        borderaxespad=0.20,
    )
    leg.get_frame().set_edgecolor(COLORS["spine"])
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_facecolor(COLORS["bg"])
    return leg

# =============================================================================
# Load data
# =============================================================================

require_file(SELECTION_AUDIT_FILE)
require_file(STAGE_SUMMARY_FILE)
require_file(DESCRIPTOR_SUMMARY_FILE)

selection_audit_df = pd.read_csv(SELECTION_AUDIT_FILE)
stage_summary_df = pd.read_csv(STAGE_SUMMARY_FILE)
descriptor_summary_df = pd.read_csv(DESCRIPTOR_SUMMARY_FILE)

# =============================================================================
# Prepare data
# =============================================================================

stage_order = [
    "Generated / candidate pool",
    "Passed pool",
    "Shortlist",
    "Final panel",
]
stage_short = {
    "Generated / candidate pool": "Generated",
    "Passed pool": "Passed",
    "Shortlist": "Shortlist",
    "Final panel": "Final panel",
}
stage_colors_a = {
    "Generated / candidate pool": COLORS["gray_mid"],
    "Passed pool": COLORS["teal"],
    "Shortlist": COLORS["orange"],
    "Final panel": COLORS["navy"],
}
audit_df = (
    selection_audit_df[selection_audit_df["stage"].isin(stage_order)]
    .copy()
    .set_index("stage")
    .loc[stage_order]
    .reset_index()
)
generated_n = float(audit_df.loc[audit_df["stage"] == "Generated / candidate pool", "count"].iloc[0])

metric_display_b = {
    "novelty_score": "Novelty",
    "plausibility_score": "Plausibility",
    "diversity_score": "Diversity",
    "final_score": "Final score",
}
metric_order_b = [m for m in metric_display_b.keys() if m in stage_summary_df["metric"].unique()]
stages_b = ["Passed", "Shortlist", "Final panel"]
stage_colors_b = {
    "Passed": COLORS["teal"],
    "Shortlist": COLORS["orange"],
    "Final panel": COLORS["navy"],
}

descriptor_map = {
    "length": "Length",
    "net_charge_proxy": "Charge",
    "mean_hydropathy": "Hydropathy",
    "entropy": "Entropy",
}
descriptor_order = [d for d in descriptor_map.keys() if d in descriptor_summary_df["descriptor"].unique()]
group_order_c = ["Reference ACP corpus", "Paper candidates", "Final panel"]
group_short_c = {
    "Reference ACP corpus": "Ref",
    "Paper candidates": "Paper",
    "Final panel": "Final",
}
group_colors_c = {
    "Reference ACP corpus": COLORS["light_blue"],
    "Paper candidates": COLORS["deep_red"],
    "Final panel": COLORS["navy"],
}

# =============================================================================
# Build figure
# =============================================================================

fig = plt.figure(figsize=(13.8, 5.7))
outer = GridSpec(
    1, 3,
    figure=fig,
    width_ratios=[1.18, 1.16, 1.58],
    wspace=0.68,
)

# -----------------------------------------------------------------------------
# Panel a
# -----------------------------------------------------------------------------
ax1 = fig.add_subplot(outer[0, 0])
add_panel_label(ax1, "a", x=-0.08, y=1.03)
style_axis(ax1, grid_axis="y")
ax1.set_title("Selection audit", pad=12)

x = np.arange(len(audit_df))
vals = audit_df["count"].astype(float).to_numpy()

ax1.bar(
    x,
    vals,
    width=0.48,
    color=[stage_colors_a[s] for s in audit_df["stage"]],
    edgecolor=COLORS["gray_dark"],
    linewidth=0.32,
    zorder=3,
)

ax1.set_ylabel("Number of peptides")
ax1.set_xticks(x)
ax1.set_xticklabels([stage_short[s] for s in audit_df["stage"]], rotation=90, ha="center")
ax1.tick_params(axis="x", pad=3)
ax1.set_ylim(0, vals.max() * 1.16)

for i, row in audit_df.iterrows():
    count = int(row["count"])
    if row["stage"] == "Generated / candidate pool":
        label = f"{count:,}"
    else:
        pct = 100.0 * count / generated_n
        label = f"{count:,}\n({pct:.1f}%)"
    yoff = vals.max() * (0.014 if count > 100 else 0.0070)
    ax1.text(
        i, count + yoff, label,
        ha="center", va="bottom",
        fontsize=10, color=COLORS["text"],
    )

axins = inset_axes(ax1, width="27%", height="29%", loc="upper right", borderpad=1.0)
style_axis(axins, grid_axis="y")
sub_df = audit_df[audit_df["stage"].isin(["Shortlist", "Final panel"])].reset_index(drop=True)
x2 = np.arange(len(sub_df))
axins.bar(
    x2,
    sub_df["count"].astype(float).to_numpy(),
    width=0.42,
    color=[stage_colors_a[s] for s in sub_df["stage"]],
    edgecolor=COLORS["gray_dark"],
    linewidth=0.22,
    zorder=3,
)
axins.set_xticks(x2)
axins.set_xticklabels(["Shortlist", "Final"], rotation=90, fontsize=8)
axins.tick_params(axis="y", labelsize=8, pad=1)
axins.set_ylim(0, max(sub_df["count"].max() * 1.24, 14))
for i, v in enumerate(sub_df["count"].tolist()):
    axins.text(i, v + 0.28, f"{int(v)}", ha="center", va="bottom", fontsize=8)
for spine in ["left", "bottom"]:
    axins.spines[spine].set_linewidth(0.5)

# -----------------------------------------------------------------------------
# Panel b
# -----------------------------------------------------------------------------
ax2 = fig.add_subplot(outer[0, 1])
add_panel_label(ax2, "b", x=-0.12, y=1.03)  # moved left to avoid title interaction
style_axis(ax2, grid_axis="y")
ax2.set_title("Prioritization score shift", pad=12)
ax2.set_ylabel("Median score")
ax2.set_ylim(0, 1.02)

group_gap = 1.08
xpos = np.arange(len(metric_order_b)) * group_gap
bar_w = 0.16
offsets = {
    "Passed": -bar_w,
    "Shortlist": 0.0,
    "Final panel": bar_w,
}

for stage in stages_b:
    vals_b, lo_b, hi_b = [], [], []
    for metric in metric_order_b:
        med, lo, hi = get_metric_row(stage_summary_df, stage, metric)
        vals_b.append(med)
        lo_b.append(lo)
        hi_b.append(hi)

    vals_b = np.asarray(vals_b, dtype=float)
    lo_b = np.asarray(lo_b, dtype=float)
    hi_b = np.asarray(hi_b, dtype=float)
    yerr = np.vstack([vals_b - lo_b, hi_b - vals_b])

    ax2.bar(
        xpos + offsets[stage],
        vals_b,
        width=bar_w,
        color=stage_colors_b[stage],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.28,
        yerr=yerr,
        capsize=2.1,
        error_kw={"elinewidth": 0.82, "capthick": 0.82},
        zorder=3,
        label=stage,
    )

ax2.set_xticks(xpos)
ax2.set_xticklabels([metric_display_b[m] for m in metric_order_b], rotation=90, ha="center", fontsize=10)
ax2.tick_params(axis="x", pad=5)

handles_b, labels_b = ax2.get_legend_handles_labels()
make_boxed_legend(
    ax2,
    handles_b,
    labels_b,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.23),  # lower with more space from x labels
    ncol=3,
)

# -----------------------------------------------------------------------------
# Panel c
# -----------------------------------------------------------------------------
ax3_container = fig.add_subplot(outer[0, 2])
add_panel_label(ax3_container, "c", x=-0.08, y=1.03)
ax3_container.axis("off")
ax3_container.text(
    0.5, 1.04, "Final-panel descriptor profile",
    transform=ax3_container.transAxes,
    ha="center", va="bottom",
    fontsize=14, color=COLORS["text"]
)

inner = outer[0, 2].subgridspec(1, len(descriptor_order), wspace=0.95)

legend_handles_c = []
legend_labels_c = []
descriptor_axes = []

for j, desc_key in enumerate(descriptor_order):
    ax = fig.add_subplot(inner[0, j])
    descriptor_axes.append(ax)
    style_axis(ax, grid_axis="y")

    sub = descriptor_summary_df[
        (descriptor_summary_df["descriptor"] == desc_key) &
        (descriptor_summary_df["group"].isin(group_order_c))
    ].copy()
    sub["group"] = pd.Categorical(sub["group"], categories=group_order_c, ordered=True)
    sub = sub.sort_values("group")

    xj = np.arange(len(sub))
    vals_c = sub["median"].astype(float).to_numpy()
    lo_c = sub["ci_low"].astype(float).to_numpy()
    hi_c = sub["ci_high"].astype(float).to_numpy()

    for i, (_, row) in enumerate(sub.iterrows()):
        g = row["group"]
        med = float(row["median"])
        lo = float(row["ci_low"])
        hi = float(row["ci_high"])
        color = group_colors_c[g]

        ax.errorbar(
            i, med,
            yerr=[[med - lo], [hi - med]],
            fmt="o",
            markersize=5.4,
            color=color,
            markerfacecolor=color,
            markeredgecolor=COLORS["gray_dark"],
            markeredgewidth=0.32,
            elinewidth=0.9,
            capsize=2.2,
            zorder=4,
        )

        if j == 0:
            legend_handles_c.append(
                Line2D([0], [0], marker="o", color=color, linestyle="", markersize=6.0,
                       markeredgecolor=COLORS["gray_dark"], markeredgewidth=0.32)
            )
            legend_labels_c.append(g)

    ax.set_xticks(xj)
    ax.set_xticklabels([group_short_c[g] for g in sub["group"]], rotation=90, ha="center", fontsize=8)
    ax.tick_params(axis="x", pad=1.5)

    if j == 0:
        ax.set_ylabel("Median value", fontsize=14)
    else:
        ax.set_ylabel("")
        ax.set_yticklabels([])

    ymin = np.nanmin(lo_c)
    ymax = np.nanmax(hi_c)
    pad = (ymax - ymin) * 0.16 if np.isfinite(ymax - ymin) and (ymax - ymin) > 0 else 0.5
    ax.set_ylim(ymin - 0.14 * pad, ymax + pad)

# descriptor names below mini-panels, above legend
for ax, desc_key in zip(descriptor_axes, descriptor_order):
    ax.text(
        0.5, -0.28, descriptor_map[desc_key],
        transform=ax.transAxes,
        ha="center", va="top",
        fontsize=11, color=COLORS["text"]
    )

make_boxed_legend(
    ax3_container,
    legend_handles_c,
    legend_labels_c,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.23),  # below descriptor labels
    ncol=3,
)

fig.tight_layout(rect=[0, 0.22, 1, 1])

# =============================================================================
# Save
# =============================================================================

OUTDIR.mkdir(parents=True, exist_ok=True)
out_base = OUTDIR / OUT_BASENAME
export_figure(fig, out_base)

print("=" * 90)
print("High-resolution redesigned standalone main figure generated successfully.")
print(f"Saved PDF: {out_base.with_suffix('.pdf')}")
print(f"Saved PNG: {out_base.with_suffix('.png')}")
print(f"Saved TIF: {out_base.with_suffix('.tif')}")
print("=" * 90)

new redesign figures

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# =============================================================================
# Paths
# =============================================================================

STEP9_DIR = Path("/home/data3/Moe/nature_computational2/step9_v1")

SELECTION_AUDIT_FILE = STEP9_DIR / "tables_main" / "table_9_2_selection_audit_counts.csv"
STAGE_SUMMARY_FILE = STEP9_DIR / "tables_supplementary" / "table_s9_1_selection_stage_score_shift_summary.csv"

OUTDIR = STEP9_DIR / "figures_main"
OUT_BASENAME = "figure_9_main_clean_2panel_final"

# =============================================================================
# Export settings
# =============================================================================

EXPORT_PDF = True
EXPORT_PNG = True
EXPORT_TIF = True
PNG_DPI = 600
TIF_DPI = 600

# =============================================================================
# Style
# =============================================================================

COLORS = {
    "gray_mid": "#A6A6A6",
    "teal": "#2F7F7B",
    "orange": "#D55E00",
    "navy": "#3B62A3",
    "bg": "#FFFFFF",
    "grid": "#F3F3F3",
    "spine": "#B8B8B8",
    "text": "#222222",
    "gray_dark": "#4A4A4A",
}

plt.rcParams.update({
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 17,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# =============================================================================
# Helpers
# =============================================================================

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file: {path}")

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(colors=COLORS["text"], width=0.8, length=5)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=COLORS["grid"], linewidth=0.75, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str, x: float = -0.08, y: float = 1.03) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=20,
        fontweight="bold",
        ha="left",
        va="bottom",
        color=COLORS["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=PNG_DPI, bbox_inches="tight")
    if EXPORT_TIF:
        fig.savefig(out_base.with_suffix(".tif"), dpi=TIF_DPI, bbox_inches="tight")
    plt.close(fig)

def get_metric_row(df: pd.DataFrame, stage: str, metric: str) -> Tuple[float, float, float]:
    sub = df[(df["stage"] == stage) & (df["metric"] == metric)]
    if len(sub) == 0:
        return (np.nan, np.nan, np.nan)
    row = sub.iloc[0]
    return float(row["median"]), float(row["ci_low"]), float(row["ci_high"])

def make_boxed_legend(ax_or_fig, handles, labels, *, loc, bbox_to_anchor, ncol=3):
    leg = ax_or_fig.legend(
        handles,
        labels,
        loc=loc,
        bbox_to_anchor=bbox_to_anchor,
        ncol=ncol,
        frameon=True,
        fancybox=True,
        borderpad=0.32,
        handlelength=1.15,
        handletextpad=0.45,
        columnspacing=0.9,
        borderaxespad=0.20,
    )
    leg.get_frame().set_edgecolor(COLORS["spine"])
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_facecolor(COLORS["bg"])
    return leg

# =============================================================================
# Load data
# =============================================================================

require_file(SELECTION_AUDIT_FILE)
require_file(STAGE_SUMMARY_FILE)

selection_audit_df = pd.read_csv(SELECTION_AUDIT_FILE)
stage_summary_df = pd.read_csv(STAGE_SUMMARY_FILE)

# =============================================================================
# Prepare data
# =============================================================================

# Panel a
stage_order = [
    "Generated / candidate pool",
    "Passed pool",
    "Shortlist",
    "Final panel",
]
stage_short = {
    "Generated / candidate pool": "Generated",
    "Passed pool": "Passed",
    "Shortlist": "Shortlist",
    "Final panel": "Final panel",
}
stage_colors_a = {
    "Generated / candidate pool": COLORS["gray_mid"],
    "Passed pool": COLORS["teal"],
    "Shortlist": COLORS["orange"],
    "Final panel": COLORS["navy"],
}
audit_df = (
    selection_audit_df[selection_audit_df["stage"].isin(stage_order)]
    .copy()
    .set_index("stage")
    .loc[stage_order]
    .reset_index()
)
generated_n = float(audit_df.loc[audit_df["stage"] == "Generated / candidate pool", "count"].iloc[0])

# Panel b
metric_display_b = {
    "novelty_score": "Novelty",
    "plausibility_score": "Plausibility",
    "diversity_score": "Diversity",
    "final_score": "Final score",
}
metric_order_b = [m for m in metric_display_b.keys() if m in stage_summary_df["metric"].unique()]
stages_b = ["Passed", "Shortlist", "Final panel"]
stage_colors_b = {
    "Passed": COLORS["teal"],
    "Shortlist": COLORS["orange"],
    "Final panel": COLORS["navy"],
}

# =============================================================================
# Build figure
# =============================================================================

fig = plt.figure(figsize=(13.6, 5.6))
gs = GridSpec(
    1, 2,
    figure=fig,
    width_ratios=[1.00, 1.15],
    wspace=0.40,
)

# -----------------------------------------------------------------------------
# Panel a — Selection audit
# -----------------------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
add_panel_label(ax1, "a", x=-0.08, y=1.03)
style_axis(ax1, grid_axis="y")
ax1.set_title("Selection audit", pad=12)

x = np.arange(len(audit_df))
vals = audit_df["count"].astype(float).to_numpy()

ax1.bar(
    x,
    vals,
    width=0.50,
    color=[stage_colors_a[s] for s in audit_df["stage"]],
    edgecolor=COLORS["gray_dark"],
    linewidth=0.30,
    zorder=3,
)

ax1.set_ylabel("Number of peptides")
ax1.set_xticks(x)
ax1.set_xticklabels([stage_short[s] for s in audit_df["stage"]], rotation=90, ha="center")
ax1.tick_params(axis="x", pad=3)
ax1.set_ylim(0, vals.max() * 1.16)

for i, row in audit_df.iterrows():
    count = int(row["count"])
    if row["stage"] == "Generated / candidate pool":
        label = f"{count:,}"
    else:
        pct = 100.0 * count / generated_n
        label = f"{count:,}\n({pct:.1f}%)"
    yoff = vals.max() * (0.014 if count > 100 else 0.0070)
    ax1.text(
        i,
        count + yoff,
        label,
        ha="center",
        va="bottom",
        fontsize=10,
        color=COLORS["text"],
    )

# Inset for shortlist/final panel
axins = inset_axes(ax1, width="26%", height="28%", loc="upper right", borderpad=0.95)
style_axis(axins, grid_axis="y")
sub_df = audit_df[audit_df["stage"].isin(["Shortlist", "Final panel"])].reset_index(drop=True)
x2 = np.arange(len(sub_df))
axins.bar(
    x2,
    sub_df["count"].astype(float).to_numpy(),
    width=0.42,
    color=[stage_colors_a[s] for s in sub_df["stage"]],
    edgecolor=COLORS["gray_dark"],
    linewidth=0.20,
    zorder=3,
)
axins.set_xticks(x2)
axins.set_xticklabels(["Shortlist", "Final"], rotation=90, fontsize=8)
axins.tick_params(axis="y", labelsize=8, pad=1)
axins.set_ylim(0, max(sub_df["count"].max() * 1.24, 14))
for i, v in enumerate(sub_df["count"].tolist()):
    axins.text(i, v + 0.25, f"{int(v)}", ha="center", va="bottom", fontsize=8)
for spine in ["left", "bottom"]:
    axins.spines[spine].set_linewidth(0.45)

# -----------------------------------------------------------------------------
# Panel b — Prioritization score shift
# -----------------------------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
add_panel_label(ax2, "b", x=-0.06, y=1.03)
style_axis(ax2, grid_axis="y")
ax2.set_title("Prioritization score shift", pad=12)
ax2.set_ylabel("Median score")
ax2.set_ylim(0, 1.02)

group_gap = 1.18
xpos = np.arange(len(metric_order_b)) * group_gap
bar_w = 0.17
offsets = {
    "Passed": -0.21,
    "Shortlist": 0.0,
    "Final panel": 0.21,
}

for stage in stages_b:
    vals_b, lo_b, hi_b = [], [], []
    for metric in metric_order_b:
        med, lo, hi = get_metric_row(stage_summary_df, stage, metric)
        vals_b.append(med)
        lo_b.append(lo)
        hi_b.append(hi)

    vals_b = np.asarray(vals_b, dtype=float)
    lo_b = np.asarray(lo_b, dtype=float)
    hi_b = np.asarray(hi_b, dtype=float)
    yerr = np.vstack([vals_b - lo_b, hi_b - vals_b])

    ax2.bar(
        xpos + offsets[stage],
        vals_b,
        width=bar_w,
        color=stage_colors_b[stage],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.28,
        yerr=yerr,
        capsize=2.2,
        error_kw={"elinewidth": 0.80, "capthick": 0.80},
        zorder=3,
        label=stage,
    )

ax2.set_xticks(xpos)
ax2.set_xticklabels([metric_display_b[m] for m in metric_order_b], rotation=0, ha="center", fontsize=11)
ax2.tick_params(axis="x", pad=10)

handles_b, labels_b = ax2.get_legend_handles_labels()
make_boxed_legend(
    ax2,
    handles_b,
    labels_b,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.19),
    ncol=3,
)

# =============================================================================
# Save
# =============================================================================

OUTDIR.mkdir(parents=True, exist_ok=True)
out_base = OUTDIR / OUT_BASENAME
export_figure(fig, out_base)

print("=" * 90)
print("Final clean 2-panel main figure generated successfully.")
print(f"Saved PDF: {out_base.with_suffix('.pdf')}")
print(f"Saved PNG: {out_base.with_suffix('.png')}")
print(f"Saved TIF: {out_base.with_suffix('.tif')}")
print("=" * 90)

redesign supplementary figures

In [ ]:
from __future__ import annotations

# =============================================================================
# Step 9 standalone supplementary-figure generator
# Redesigned supplementary package:
#   SFig 9.1 - similarity and non-overlap audit
#   SFig 9.2 - compositional contextual support
#   SFig 9.3 - descriptor and complexity context
#   Table S9.13 - compact overlap and subset audit
# =============================================================================

from pathlib import Path
from typing import Sequence, List, Dict, Tuple
from collections import Counter
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

# =============================================================================
# Paths
# =============================================================================

STEP9_DIR = Path("/home/data3/Moe/nature_computational2/step9_v1")
TABLES_SUPP = STEP9_DIR / "tables_supplementary"
FIG_SUPP = STEP9_DIR / "figures_supplementary"

FILE_FINAL_ANNOT = TABLES_SUPP / "table_s9_9_final_panel_with_all_step9_annotations.csv"
FILE_PAIRWISE = TABLES_SUPP / "table_s9_7_final_panel_pairwise_similarity.csv"
FILE_OVERLAP = TABLES_SUPP / "table_s9_2_final_panel_overlap_audit.csv"
FILE_DESC_SUMMARY = TABLES_SUPP / "table_s9_4_descriptor_distribution_summary.csv"
FILE_AA_ENRICH = TABLES_SUPP / "table_s9_5_amino_acid_enrichment_vs_reference.csv"
FILE_KMER_ENRICH = TABLES_SUPP / "table_s9_6_kmer_enrichment_vs_reference.csv"
FILE_SUBSET_AUDIT = TABLES_SUPP / "table_s9_12_stage_subset_audits.csv"
FILE_PAPER_CANDIDATES = TABLES_SUPP / "table_s9_11_paper_candidates_harmonized.csv"
FILE_REFERENCE_CORPUS = Path("/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv")

OUT_SIM = FIG_SUPP / "sfigure_9_1_similarity_and_nonoverlap_audit_v4"
OUT_COMP = FIG_SUPP / "sfigure_9_2_compositional_contextual_support_v4"
OUT_DESC = FIG_SUPP / "sfigure_9_3_descriptor_and_complexity_context_v4"
OUT_AUDIT_TABLE = TABLES_SUPP / "table_s9_13_compact_overlap_and_subset_audit.csv"

# =============================================================================
# Export settings
# =============================================================================

EXPORT_PDF = True
EXPORT_PNG = True
EXPORT_TIF = True
PNG_DPI = 600
TIF_DPI = 600

# =============================================================================
# Style
# =============================================================================

COLORS = {
    "deep_red": "#C7001E",
    "navy": "#1F3B68",
    "teal": "#0E6766",
    "orange": "#F29D3E",
    "light_blue": "#A8B9DC",
    "gray_fill": "#9E9E9E",
    "gray_dark": "#4A4A4A",
    "bg": "#FFFFFF",
    "grid": "#ECECEC",
    "spine": "#BFBFBF",
    "text": "#222222",
}

plt.rcParams.update({
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

AA_SET = set("ACDEFGHIKLMNPQRSTVWY")
AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")

KD_HYDROPATHY = {
    "A": 1.8,  "C": 2.5,  "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5,  "K": -3.9, "L": 3.8,
    "M": 1.9,  "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2,  "W": -0.9, "Y": -1.3,
}

# =============================================================================
# Helpers
# =============================================================================

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file: {path}")

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(colors=COLORS["text"], width=0.7, length=4)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=COLORS["grid"], linewidth=0.65, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str, x: float = -0.08, y: float = 1.03) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="bottom",
        color=COLORS["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=PNG_DPI, bbox_inches="tight")
    if EXPORT_TIF:
        fig.savefig(out_base.with_suffix(".tif"), dpi=TIF_DPI, bbox_inches="tight")
    plt.close(fig)

def shorten_ids(ids: Sequence[str], max_len: int = 12) -> List[str]:
    out = []
    for s in ids:
        s = str(s)
        out.append(s if len(s) <= max_len else s[:max_len])
    return out

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def safe_numeric(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors="coerce")

def mean_hydropathy(seq: str) -> float:
    seq = clean_sequence(seq)
    if not seq:
        return np.nan
    vals = [KD_HYDROPATHY[a] for a in seq if a in KD_HYDROPATHY]
    return float(np.mean(vals)) if vals else np.nan

def net_charge_proxy(seq: str) -> float:
    seq = clean_sequence(seq)
    if not seq:
        return np.nan
    pos = sum(a in AA_POSITIVE for a in seq)
    neg = sum(a in AA_NEGATIVE for a in seq)
    return float(pos - neg)

def sequence_entropy(seq: str) -> float:
    seq = clean_sequence(seq)
    if not seq:
        return np.nan
    counts = Counter(seq)
    p = np.array([c / len(seq) for c in counts.values()], dtype=float)
    return float(-(p * np.log2(p)).sum())

def unique_residue_fraction(seq: str) -> float:
    seq = clean_sequence(seq)
    if not seq:
        return np.nan
    return float(len(set(seq)) / len(seq))

def compute_descriptor_frame(df: pd.DataFrame, seq_col: str = "sequence") -> pd.DataFrame:
    out = df.copy()
    out[seq_col] = out[seq_col].astype(str).map(clean_sequence)

    if "length" not in out.columns:
        out["length"] = out[seq_col].map(lambda s: len(s) if s else np.nan)
    else:
        out["length"] = pd.to_numeric(out["length"], errors="coerce").fillna(out[seq_col].map(lambda s: len(s) if s else np.nan))

    if "net_charge_proxy" not in out.columns:
        out["net_charge_proxy"] = out[seq_col].map(net_charge_proxy)
    else:
        out["net_charge_proxy"] = pd.to_numeric(out["net_charge_proxy"], errors="coerce").fillna(out[seq_col].map(net_charge_proxy))

    if "mean_hydropathy" not in out.columns:
        out["mean_hydropathy"] = out[seq_col].map(mean_hydropathy)
    else:
        out["mean_hydropathy"] = pd.to_numeric(out["mean_hydropathy"], errors="coerce").fillna(out[seq_col].map(mean_hydropathy))

    if "entropy" not in out.columns:
        out["entropy"] = out[seq_col].map(sequence_entropy)
    else:
        out["entropy"] = pd.to_numeric(out["entropy"], errors="coerce").fillna(out[seq_col].map(sequence_entropy))

    if "unique_residue_fraction" not in out.columns:
        out["unique_residue_fraction"] = out[seq_col].map(unique_residue_fraction)
    else:
        out["unique_residue_fraction"] = pd.to_numeric(out["unique_residue_fraction"], errors="coerce").fillna(out[seq_col].map(unique_residue_fraction))

    return out

def frac_of_categories(seqs: Sequence[str]) -> Dict[str, float]:
    total_len = sum(len(s) for s in seqs) if len(seqs) > 0 else 0
    if total_len == 0:
        return {"Hydrophobic": np.nan, "Aromatic": np.nan, "Positive": np.nan, "Negative": np.nan, "Polar": np.nan}

    def frac_of(res_set: set) -> float:
        c = sum(sum(ch in res_set for ch in s) for s in seqs)
        return c / total_len

    return {
        "Hydrophobic": frac_of(AA_HYDROPHOBIC),
        "Aromatic": frac_of(AA_AROMATIC),
        "Positive": frac_of(AA_POSITIVE),
        "Negative": frac_of(AA_NEGATIVE),
        "Polar": frac_of(AA_POLAR),
    }

def build_compact_audit_table(overlap_df: pd.DataFrame, subset_df: pd.DataFrame) -> pd.DataFrame:
    def get_subset_missing(child_label: str, parent_label: str) -> int:
        sub = subset_df[
            (subset_df["child_label"] == child_label) &
            (subset_df["parent_label"] == parent_label)
        ]
        return int(sub["missing_in_parent_n"].iloc[0]) if len(sub) else np.nan

    def get_subset_status(child_label: str, parent_label: str) -> str:
        sub = subset_df[
            (subset_df["child_label"] == child_label) &
            (subset_df["parent_label"] == parent_label)
        ]
        if len(sub) == 0:
            return "missing"
        return "pass" if int(sub["is_subset"].iloc[0]) == 1 else "review"

    tbl = pd.DataFrame([
        {"audit": "Overlap vs reference", "count": int(overlap_df["exact_overlap_vs_reference_corpus"].sum()),
         "status": "pass" if int(overlap_df["exact_overlap_vs_reference_corpus"].sum()) == 0 else "review"},
        {"audit": "Overlap vs templates", "count": int(overlap_df["exact_overlap_vs_paper_templates"].sum()),
         "status": "pass" if int(overlap_df["exact_overlap_vs_paper_templates"].sum()) == 0 else "review"},
        {"audit": "Overlap vs paper candidates", "count": int(overlap_df["exact_overlap_vs_paper_candidates"].sum()),
         "status": "pass" if int(overlap_df["exact_overlap_vs_paper_candidates"].sum()) == 0 else "review"},
        {"audit": "Shortlist ⊂ passed", "count": get_subset_missing("Shortlist", "Passed"),
         "status": get_subset_status("Shortlist", "Passed")},
        {"audit": "Final ⊂ shortlist", "count": get_subset_missing("Final panel", "Shortlist"),
         "status": get_subset_status("Final panel", "Shortlist")},
        {"audit": "Final ⊂ passed", "count": get_subset_missing("Final panel", "Passed"),
         "status": get_subset_status("Final panel", "Passed")},
    ])
    tbl.to_csv(OUT_AUDIT_TABLE, index=False)
    return tbl

def draw_violin_with_summary(ax, data: Sequence[float], pos: float, color: str, width: float = 0.72) -> None:
    arr = np.asarray([x for x in data if pd.notna(x)], dtype=float)
    if len(arr) == 0:
        return
    parts = ax.violinplot([arr], positions=[pos], widths=width, showmeans=False, showmedians=False, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor(color)
        pc.set_edgecolor("none")
        pc.set_alpha(0.85)
    q1, med, q3 = np.percentile(arr, [25, 50, 75])
    ax.plot([pos, pos], [q1, q3], color=COLORS["gray_dark"], linewidth=1.0, zorder=3)
    ax.scatter([pos], [med], s=18, facecolor="white", edgecolor=COLORS["gray_dark"], linewidth=0.55, zorder=4)

def choose_labels_for_scatter(
    df: pd.DataFrame,
    xcol: str,
    ycol: str,
    idcol: str,
    n_top: int = 5,
) -> pd.DataFrame:
    tmp = df[[xcol, ycol, idcol]].copy()
    tmp[xcol] = pd.to_numeric(tmp[xcol], errors="coerce")
    tmp[ycol] = pd.to_numeric(tmp[ycol], errors="coerce")
    tmp = tmp.dropna().reset_index(drop=True)

    if len(tmp) <= n_top:
        return tmp

    x = tmp[xcol].to_numpy()
    y = tmp[ycol].to_numpy()

    idx = set()
    idx.add(int(np.argmax(x)))
    idx.add(int(np.argmin(x)))
    idx.add(int(np.argmax(y)))
    idx.add(int(np.argmin(y)))

    x_med, y_med = np.median(x), np.median(y)
    dist = (x - x_med) ** 2 + (y - y_med) ** 2
    for i in np.argsort(dist)[::-1]:
        idx.add(int(i))
        if len(idx) >= n_top:
            break

    return tmp.iloc[sorted(idx)].copy()

def annotate_selected_points(ax, sub: pd.DataFrame, xcol: str, ycol: str, idcol: str,
                             dx: float, dy: float, fontsize: float = 5.9) -> None:
    for _, r in sub.iterrows():
        ax.text(float(r[xcol]) + dx, float(r[ycol]) + dy, str(r[idcol]), fontsize=fontsize, color=COLORS["text"])

# =============================================================================
# Load inputs
# =============================================================================

for p in [
    FILE_FINAL_ANNOT, FILE_PAIRWISE, FILE_OVERLAP, FILE_DESC_SUMMARY,
    FILE_AA_ENRICH, FILE_KMER_ENRICH, FILE_SUBSET_AUDIT,
    FILE_PAPER_CANDIDATES, FILE_REFERENCE_CORPUS
]:
    require_file(p)

final_df = compute_descriptor_frame(pd.read_csv(FILE_FINAL_ANNOT))
pairwise_df = pd.read_csv(FILE_PAIRWISE, index_col=0)
overlap_df = pd.read_csv(FILE_OVERLAP)
desc_summary_df = pd.read_csv(FILE_DESC_SUMMARY)
aa_enrich_df = pd.read_csv(FILE_AA_ENRICH)
kmer_enrich_df = pd.read_csv(FILE_KMER_ENRICH)
subset_df = pd.read_csv(FILE_SUBSET_AUDIT)
paper_candidates_df = compute_descriptor_frame(pd.read_csv(FILE_PAPER_CANDIDATES))
reference_df = compute_descriptor_frame(pd.read_csv(FILE_REFERENCE_CORPUS))

audit_tbl = build_compact_audit_table(overlap_df, subset_df)

# =============================================================================
# SFig 9.1 — heatmap-centered redesign
# =============================================================================

def plot_supplementary_figure_9_1() -> None:
    fig = plt.figure(figsize=(7.2, 5.8))
    gs = GridSpec(
        2, 3, figure=fig,
        width_ratios=[1.0, 1.18, 1.18],
        height_ratios=[1.0, 0.88],
        wspace=0.34, hspace=0.40
    )

    # a: similarity diagnostics (violin support)
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="y")

    ref_vals = safe_numeric(final_df, "nearest_reference_similarity").dropna().to_numpy(dtype=float)
    paper_vals = safe_numeric(final_df, "nearest_paper_candidate_similarity").dropna().to_numpy(dtype=float)

    draw_violin_with_summary(ax, ref_vals, 0, COLORS["light_blue"])
    draw_violin_with_summary(ax, paper_vals, 1, COLORS["deep_red"])

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Reference", "Paper"], rotation=0)
    ax.set_ylabel("NN similarity")
    ax.set_ylim(0, max(0.22, float(np.nanmax(np.r_[ref_vals, paper_vals])) * 1.25))
    ax.set_title("Similarity diagnostics", loc="left", pad=4)

    # b: pairwise heatmap anchor
    ax = fig.add_subplot(gs[:, 1:])
    add_panel_label(ax, "b")
    mat = pairwise_df.to_numpy(dtype=float)

    # order by average similarity to impose a more intentional layout
    avg_sim = mat.mean(axis=1)
    order = np.argsort(-avg_sim)
    mat_ord = mat[order][:, order]
    labels = shorten_ids(np.array(pairwise_df.index.tolist())[order].tolist(), max_len=12)

    im = ax.imshow(mat_ord, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
    ax.set_yticklabels(labels, fontsize=6)
    ax.set_title("Pairwise similarity within final panel", loc="left", pad=4)
    cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.set_label("3-mer Jaccard similarity")

    # c: high-similarity tail risk
    ax = fig.add_subplot(gs[1, 0])
    add_panel_label(ax, "c")
    style_axis(ax, grid_axis="y")

    thresholds = [0.10, 0.15, 0.20, 0.30]
    ref_frac = [float((ref_vals > t).mean()) if len(ref_vals) else np.nan for t in thresholds]
    paper_frac = [float((paper_vals > t).mean()) if len(paper_vals) else np.nan for t in thresholds]

    xpos = np.arange(len(thresholds))
    width = 0.34
    ax.bar(
        xpos - width / 2, ref_frac, width=width,
        color=COLORS["light_blue"], edgecolor=COLORS["gray_dark"], linewidth=0.45,
        label="Reference"
    )
    ax.bar(
        xpos + width / 2, paper_frac, width=width,
        color=COLORS["deep_red"], edgecolor=COLORS["gray_dark"], linewidth=0.45,
        label="Paper"
    )
    ax.set_xticks(xpos)
    ax.set_xticklabels([f">{t:.2f}" for t in thresholds], rotation=0)
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, max(0.45, float(np.nanmax(np.r_[ref_frac, paper_frac])) * 1.18))
    ax.set_title("High-similarity tail risk", loc="left", pad=4)
    ax.legend(frameon=False, loc="upper right")

    fig.tight_layout()
    export_figure(fig, OUT_SIM)

# =============================================================================
# SFig 9.2 — micro-polished
# =============================================================================

def plot_supplementary_figure_9_2() -> None:
    fig = plt.figure(figsize=(7.1, 6.5))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.36, height_ratios=[1.0, 0.9])

    # a
    ax = fig.add_subplot(gs[:, 0])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="x")
    tmp = aa_enrich_df.copy().sort_values("log2_enrichment_vs_reference", ascending=True)
    colors = [COLORS["light_blue"] if v < 0 else COLORS["deep_red"] for v in tmp["log2_enrichment_vs_reference"]]
    ax.barh(
        tmp["amino_acid"],
        tmp["log2_enrichment_vs_reference"],
        color=colors,
        edgecolor=COLORS["gray_dark"],
        linewidth=0.4,
    )
    ax.axvline(0, color=COLORS["gray_dark"], linewidth=0.8)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Amino-acid enrichment in final panel", loc="left", pad=4)

    # b
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="x")
    tmp = kmer_enrich_df.copy().sort_values("log2_enrichment_vs_reference", ascending=False).head(8)
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    ax.barh(
        tmp["kmer"],
        tmp["log2_enrichment_vs_reference"],
        color=COLORS["teal"],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.4,
    )
    ax.axvline(0, color=COLORS["gray_dark"], linewidth=0.8)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Top enriched 3-mers", loc="left", pad=4)

    # c
    ax = fig.add_subplot(gs[1, 1])
    add_panel_label(ax, "c")
    style_axis(ax, grid_axis="y")
    ref_cat = frac_of_categories(reference_df["sequence"].astype(str).tolist())
    final_cat = frac_of_categories(final_df["sequence"].astype(str).tolist())
    categories = ["Hydrophobic", "Aromatic", "Positive", "Negative", "Polar"]
    xpos = np.arange(len(categories))
    width = 0.34
    ref_vals = np.array([ref_cat[c] for c in categories], dtype=float)
    final_vals = np.array([final_cat[c] for c in categories], dtype=float)
    ax.bar(
        xpos - width / 2, ref_vals, width=width,
        color=COLORS["light_blue"], edgecolor=COLORS["gray_dark"], linewidth=0.45,
        label="Reference ACP corpus"
    )
    ax.bar(
        xpos + width / 2, final_vals, width=width,
        color=COLORS["navy"], edgecolor=COLORS["gray_dark"], linewidth=0.45,
        label="Final panel"
    )
    ax.set_xticks(xpos)
    ax.set_xticklabels(categories, rotation=18, ha="right")
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, max(0.35, float(np.nanmax(np.r_[ref_vals, final_vals])) * 1.18))
    ax.set_title("Residue-category comparison", loc="left", pad=4)
    ax.legend(frameon=False, loc="upper right")

    fig.tight_layout()
    export_figure(fig, OUT_COMP)

# =============================================================================
# SFig 9.3 — violin-based descriptor support
# =============================================================================

def plot_supplementary_figure_9_3() -> None:
    fig = plt.figure(figsize=(7.2, 7.2))
    gs = GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.40, height_ratios=[1.0, 0.92])

    # a: charge-hydropathy
    ax = fig.add_subplot(gs[0, 0:2])
    add_panel_label(ax, "a")
    style_axis(ax, grid_axis="both")

    x = safe_numeric(final_df, "mean_hydropathy").to_numpy(dtype=float)
    y = safe_numeric(final_df, "net_charge_proxy").to_numpy(dtype=float)
    ax.scatter(
        x, y, s=52,
        facecolor=COLORS["deep_red"], edgecolor="white",
        linewidth=0.55, alpha=0.95, zorder=3
    )
    sel = choose_labels_for_scatter(final_df, "mean_hydropathy", "net_charge_proxy", "generated_id", n_top=5)
    annotate_selected_points(ax, sel, "mean_hydropathy", "net_charge_proxy", "generated_id", dx=0.012, dy=0.03)
    ax.set_xlabel("Mean hydropathy")
    ax.set_ylabel("Net charge proxy")
    ax.set_title("Charge–hydropathy landscape", loc="left", pad=4)

    # b: complexity
    ax = fig.add_subplot(gs[0, 2:4])
    add_panel_label(ax, "b")
    style_axis(ax, grid_axis="both")

    x = safe_numeric(final_df, "entropy").to_numpy(dtype=float)
    y = safe_numeric(final_df, "unique_residue_fraction").to_numpy(dtype=float)
    ax.scatter(
        x, y, s=52,
        facecolor=COLORS["navy"], edgecolor="white",
        linewidth=0.55, alpha=0.95, zorder=3
    )
    sel = choose_labels_for_scatter(final_df, "entropy", "unique_residue_fraction", "generated_id", n_top=5)
    annotate_selected_points(ax, sel, "entropy", "unique_residue_fraction", "generated_id", dx=0.01, dy=0.004)
    ax.set_xlabel("Sequence entropy")
    ax.set_ylabel("Unique residue fraction")
    ax.set_title("Complexity profile of final-panel peptides", loc="left", pad=4)

    # bottom violins
    descriptors = [
        ("length", "Length"),
        ("net_charge_proxy", "Net charge"),
        ("mean_hydropathy", "Hydrophobicity"),
        ("entropy", "Sequence entropy"),
    ]
    color_map = {
        "Reference ACP corpus": COLORS["gray_fill"],
        "Paper candidates": COLORS["deep_red"],
        "Final panel": COLORS["navy"],
    }

    legend_handles = [
        Line2D([0], [0], color=COLORS["gray_fill"], linewidth=6),
        Line2D([0], [0], color=COLORS["deep_red"], linewidth=6),
        Line2D([0], [0], color=COLORS["navy"], linewidth=6),
    ]

    source_frames = {
        "Reference ACP corpus": reference_df,
        "Paper candidates": paper_candidates_df,
        "Final panel": final_df,
    }

    for i, (key, title) in enumerate(descriptors):
        ax = fig.add_subplot(gs[1, i])
        if i == 0:
            add_panel_label(ax, "c")
        style_axis(ax, grid_axis="y")

        groups = list(source_frames.keys())
        for j, g in enumerate(groups):
            vals = pd.to_numeric(source_frames[g][key], errors="coerce").dropna().to_numpy(dtype=float)
            draw_violin_with_summary(ax, vals, j, color_map[g], width=0.72)

        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(["Ref", "Paper", "Final"], rotation=45, ha="right", fontsize=6)
        ax.set_title(title, fontsize=8.2, pad=5)
        if i == 0:
            ax.set_ylabel("Distribution support")
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])

    fig.legend(
        handles=legend_handles,
        labels=["Reference ACP corpus", "Paper candidates", "Final panel"],
        loc="lower center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 0.015),
    )

    fig.tight_layout(rect=[0, 0.07, 1, 1])
    export_figure(fig, OUT_DESC)

# =============================================================================
# Run
# =============================================================================

FIG_SUPP.mkdir(parents=True, exist_ok=True)

plot_supplementary_figure_9_1()
plot_supplementary_figure_9_2()
plot_supplementary_figure_9_3()

print("=" * 90)
print("Redesigned standalone supplementary outputs generated successfully.")
print(f"SFig 9.1: {OUT_SIM.with_suffix('.pdf')}")
print(f"SFig 9.2: {OUT_COMP.with_suffix('.pdf')}")
print(f"SFig 9.3: {OUT_DESC.with_suffix('.pdf')}")
print(f"Table S9.13: {OUT_AUDIT_TABLE}")
print("=" * 90)

figure S9.1

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# Paths
# =============================================================================

STEP9_DIR = Path("/home/data3/Moe/nature_computational2/step9_v1")
TABLES_SUPP = STEP9_DIR / "tables_supplementary"
FIG_SUPP = STEP9_DIR / "figures_supplementary"

FILE_FINAL_ANNOT = TABLES_SUPP / "table_s9_9_final_panel_with_all_step9_annotations.csv"
FILE_PAIRWISE = TABLES_SUPP / "table_s9_7_final_panel_pairwise_similarity.csv"
FILE_OVERLAP = TABLES_SUPP / "table_s9_2_final_panel_overlap_audit.csv"
FILE_SUBSET_AUDIT = TABLES_SUPP / "table_s9_12_stage_subset_audits.csv"

OUT_SIM = FIG_SUPP / "sfigure_9_1_similarity_and_nonoverlap_audit_v12"
OUT_AUDIT_TABLE = TABLES_SUPP / "table_s9_13_compact_overlap_and_subset_audit.csv"

# =============================================================================
# Export settings
# =============================================================================

EXPORT_PDF = True
EXPORT_PNG = True
EXPORT_TIF = True
PNG_DPI = 600
TIF_DPI = 600

# =============================================================================
# Colors
# =============================================================================

COLORS = {
    "gray_mid": "#A6A6A6",
    "teal": "#2F7F7B",
    "orange": "#D55E00",
    "gray_dark": "#4A4A4A",
    "bg": "#FFFFFF",
    "grid": "#ECECEC",
    "spine": "#BFBFBF",
    "text": "#222222",
    "legend_bg": "#F7F7F7",
    "legend_edge": "#C8C8C8",
}

# Soft teal sequential colormap for heatmap
TEAL_CMAP = LinearSegmentedColormap.from_list(
    "custom_teal_scale",
    ["#F2F7F6", "#CFE1DF", "#8BB9B3", COLORS["teal"]]
)

plt.rcParams.update({
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 9.8,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# =============================================================================
# Helpers
# =============================================================================

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file: {path}")

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(colors=COLORS["text"], width=0.7, length=4)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=COLORS["grid"], linewidth=0.65, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str, x: float = -0.14, y: float = 1.045) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="bottom",
        color=COLORS["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=PNG_DPI, bbox_inches="tight")
    if EXPORT_TIF:
        fig.savefig(out_base.with_suffix(".tif"), dpi=TIF_DPI, bbox_inches="tight")
    plt.close(fig)

def safe_numeric(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors="coerce")

def shorten_ids(ids: Sequence[str], max_len: int = 12) -> List[str]:
    out = []
    for s in ids:
        s = str(s)
        out.append(s if len(s) <= max_len else s[:max_len])
    return out

def build_compact_audit_table(overlap_df: pd.DataFrame, subset_df: pd.DataFrame) -> pd.DataFrame:
    def get_subset_missing(child_label: str, parent_label: str) -> int:
        sub = subset_df[
            (subset_df["child_label"] == child_label) &
            (subset_df["parent_label"] == parent_label)
        ]
        return int(sub["missing_in_parent_n"].iloc[0]) if len(sub) else np.nan

    def get_subset_status(child_label: str, parent_label: str) -> str:
        sub = subset_df[
            (subset_df["child_label"] == child_label) &
            (subset_df["parent_label"] == parent_label)
        ]
        if len(sub) == 0:
            return "missing"
        return "pass" if int(sub["is_subset"].iloc[0]) == 1 else "review"

    tbl = pd.DataFrame([
        {"audit": "Overlap vs reference", "count": int(overlap_df["exact_overlap_vs_reference_corpus"].sum()),
         "status": "pass" if int(overlap_df["exact_overlap_vs_reference_corpus"].sum()) == 0 else "review"},
        {"audit": "Overlap vs templates", "count": int(overlap_df["exact_overlap_vs_paper_templates"].sum()),
         "status": "pass" if int(overlap_df["exact_overlap_vs_paper_templates"].sum()) == 0 else "review"},
        {"audit": "Overlap vs paper candidates", "count": int(overlap_df["exact_overlap_vs_paper_candidates"].sum()),
         "status": "pass" if int(overlap_df["exact_overlap_vs_paper_candidates"].sum()) == 0 else "review"},
        {"audit": "Shortlist ⊂ passed", "count": get_subset_missing("Shortlist", "Passed"),
         "status": get_subset_status("Shortlist", "Passed")},
        {"audit": "Final ⊂ shortlist", "count": get_subset_missing("Final panel", "Shortlist"),
         "status": get_subset_status("Final panel", "Shortlist")},
        {"audit": "Final ⊂ passed", "count": get_subset_missing("Final panel", "Passed"),
         "status": get_subset_status("Final panel", "Passed")},
    ])
    tbl.to_csv(OUT_AUDIT_TABLE, index=False)
    return tbl

def draw_violin_with_summary_and_points(
    ax,
    data: Sequence[float],
    pos: float,
    color: str,
    width: float = 0.38,
    jitter: float = 0.024,
    seed: int = 42,
) -> None:
    arr = np.asarray([x for x in data if pd.notna(x)], dtype=float)
    if len(arr) == 0:
        return

    parts = ax.violinplot(
        [arr],
        positions=[pos],
        widths=width,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for pc in parts["bodies"]:
        pc.set_facecolor(color)
        pc.set_edgecolor("none")
        pc.set_alpha(0.88)

    q1, med, q3 = np.percentile(arr, [25, 50, 75])
    ax.plot([pos, pos], [q1, q3], color=COLORS["gray_dark"], linewidth=0.95, zorder=3)
    ax.scatter([pos], [med], s=22, facecolor="white", edgecolor=COLORS["gray_dark"], linewidth=0.6, zorder=4)

    rng = np.random.default_rng(seed + int(pos * 100))
    xs = pos + rng.uniform(-jitter, jitter, size=len(arr))
    ax.scatter(
        xs, arr,
        s=8,
        color=color,
        edgecolor="white",
        linewidth=0.25,
        alpha=0.78,
        zorder=2.8,
    )

# =============================================================================
# Load inputs
# =============================================================================

for p in [FILE_FINAL_ANNOT, FILE_PAIRWISE, FILE_OVERLAP, FILE_SUBSET_AUDIT]:
    require_file(p)

final_df = pd.read_csv(FILE_FINAL_ANNOT)
pairwise_df = pd.read_csv(FILE_PAIRWISE, index_col=0)
overlap_df = pd.read_csv(FILE_OVERLAP)
subset_df = pd.read_csv(FILE_SUBSET_AUDIT)

_ = build_compact_audit_table(overlap_df, subset_df)

# =============================================================================
# Build figure
# =============================================================================

def plot_supplementary_figure_9_1() -> None:
    fig = plt.figure(figsize=(9.4, 3.8))
    gs = GridSpec(
        1, 3,
        figure=fig,
        width_ratios=[1.02, 1.00, 1.18],
        wspace=0.34,
    )

    # -------------------------------------------------------------------------
    # Panel a
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[0, 0])
    add_panel_label(ax_a, "a", x=-0.14, y=1.045)
    style_axis(ax_a, grid_axis="y")

    ref_vals = safe_numeric(final_df, "nearest_reference_similarity").dropna().to_numpy(dtype=float)
    paper_vals = safe_numeric(final_df, "nearest_paper_candidate_similarity").dropna().to_numpy(dtype=float)

    draw_violin_with_summary_and_points(ax_a, ref_vals, 0, COLORS["gray_mid"], width=0.36, jitter=0.024, seed=7)
    draw_violin_with_summary_and_points(ax_a, paper_vals, 1, COLORS["orange"], width=0.36, jitter=0.024, seed=17)

    ax_a.set_xticks([0, 1])
    ax_a.set_xticklabels(["Reference", "Paper"], rotation=0)
    ax_a.tick_params(axis="x", pad=8)
    ax_a.set_ylabel("NN similarity")
    ax_a.set_ylim(0, 0.16)
    ax_a.set_title("NN similarity support", loc="left", pad=8, x=0.08)

    # -------------------------------------------------------------------------
    # Panel b
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[0, 1])
    add_panel_label(ax_b, "b", x=-0.14, y=1.045)
    style_axis(ax_b, grid_axis="y")

    metrics = ["Median", "P90", "Max"]
    ref_summary = [
        float(np.median(ref_vals)),
        float(np.percentile(ref_vals, 90)),
        float(np.max(ref_vals)),
    ]
    paper_summary = [
        float(np.median(paper_vals)),
        float(np.percentile(paper_vals, 90)),
        float(np.max(paper_vals)),
    ]

    xpos = np.arange(len(metrics))
    width = 0.33

    ax_b.bar(
        xpos - width / 2,
        ref_summary,
        width=width,
        color=COLORS["gray_mid"],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.45,
        label="Reference",
    )
    ax_b.bar(
        xpos + width / 2,
        paper_summary,
        width=width,
        color=COLORS["orange"],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.45,
        label="Paper",
    )

    ax_b.set_xticks(xpos)
    ax_b.set_xticklabels(metrics, rotation=0)
    ax_b.tick_params(axis="x", pad=8)
    ax_b.set_ylabel("Similarity")
    ax_b.set_ylim(0, 0.165)
    ax_b.set_title("Similarity summary", loc="left", pad=8, x=0.08)

    # -------------------------------------------------------------------------
    # Panel c
    # -------------------------------------------------------------------------
    ax_c = fig.add_subplot(gs[0, 2])
    add_panel_label(ax_c, "c", x=-0.12, y=1.045)

    mat = pairwise_df.to_numpy(dtype=float)
    avg_sim = mat.mean(axis=1)
    order = np.argsort(-avg_sim)
    mat_ord = mat[order][:, order]
    labels = shorten_ids(np.array(pairwise_df.index.tolist())[order].tolist(), max_len=12)

    im = ax_c.imshow(mat_ord, vmin=0, vmax=1, cmap=TEAL_CMAP, aspect="auto")
    ax_c.set_xticks(np.arange(len(labels)))
    ax_c.set_yticks(np.arange(len(labels)))
    ax_c.set_xticklabels(labels, rotation=45, ha="right", fontsize=5.4)
    ax_c.set_yticklabels(labels, fontsize=5.4)
    ax_c.set_title("Pairwise similarity within final panel", loc="left", pad=8, x=0.01)

    cb = fig.colorbar(im, ax=ax_c, fraction=0.030, pad=0.045)
    cb.set_label("3-mer Jaccard similarity")

    # -------------------------------------------------------------------------
    # Shared legend between panels a and b, styled like example
    # -------------------------------------------------------------------------
    handles = [
        Patch(facecolor=COLORS["gray_mid"], edgecolor=COLORS["gray_dark"], linewidth=0.45, label="Reference"),
        Patch(facecolor=COLORS["orange"], edgecolor=COLORS["gray_dark"], linewidth=0.45, label="Paper"),
    ]

    pos_a = ax_a.get_position()
    pos_b = ax_b.get_position()
    center_x = (pos_a.x0 + pos_b.x1) / 2.0
    y = min(pos_a.y0, pos_b.y0) - 0.095

    legend = fig.legend(
        handles=handles,
        labels=["Reference", "Paper"],
        loc="upper center",
        bbox_to_anchor=(center_x, y),
        ncol=2,
        frameon=True,
        handlelength=1.1,
        handletextpad=0.5,
        columnspacing=0.9,
        borderaxespad=0.0,
        fontsize=8.8,
        fancybox=True,
    )
    legend.get_frame().set_facecolor(COLORS["legend_bg"])
    legend.get_frame().set_edgecolor(COLORS["legend_edge"])
    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_alpha(1.0)

    fig.tight_layout(rect=[0.02, 0.17, 0.995, 0.98])
    export_figure(fig, OUT_SIM)

# =============================================================================
# Run
# =============================================================================

FIG_SUPP.mkdir(parents=True, exist_ok=True)
plot_supplementary_figure_9_1()

print("=" * 90)
print("Updated SFig 9.1 generated successfully.")
print(f"SFig 9.1: {OUT_SIM.with_suffix('.pdf')}")
print(f"Table S9.13: {OUT_AUDIT_TABLE}")
print("=" * 90)

S9.2

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# =============================================================================
# Paths
# =============================================================================

STEP9_DIR = Path("/home/data3/Moe/nature_computational2/step9_v1")
TABLES_SUPP = STEP9_DIR / "tables_supplementary"
FIG_SUPP = STEP9_DIR / "figures_supplementary"

FILE_AA_ENRICH = TABLES_SUPP / "table_s9_5_amino_acid_enrichment_vs_reference.csv"
FILE_KMER_ENRICH = TABLES_SUPP / "table_s9_6_kmer_enrichment_vs_reference.csv"
FILE_FINAL_ANNOT = TABLES_SUPP / "table_s9_9_final_panel_with_all_step9_annotations.csv"
FILE_REFERENCE_CORPUS = Path("/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv")

OUT_COMP = FIG_SUPP / "sfigure_9_2_compositional_contextual_support_v8"

# =============================================================================
# Export settings
# =============================================================================

EXPORT_PDF = True
EXPORT_PNG = True
EXPORT_TIF = True
PNG_DPI = 600
TIF_DPI = 600

# =============================================================================
# Colors
# =============================================================================

COLORS = {
    "gray_mid": "#A6A6A6",
    "teal": "#2F7F7B",
    "orange": "#D55E00",
    "neg_blue": "#A8B9DC",
    "gray_dark": "#4A4A4A",
    "bg": "#FFFFFF",
    "grid": "#ECECEC",
    "spine": "#BFBFBF",
    "text": "#222222",
    "legend_bg": "#F7F7F7",
    "legend_edge": "#C8C8C8",
}

plt.rcParams.update({
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor": COLORS["bg"],
    "savefig.facecolor": COLORS["bg"],
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 9.2,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.4,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")
AA_SET = set("ACDEFGHIKLMNPQRSTVWY")

# =============================================================================
# Helpers
# =============================================================================

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file: {path}")

def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(colors=COLORS["text"], width=0.7, length=4)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=COLORS["grid"], linewidth=0.65, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")

def add_panel_label(ax, label: str, x: float = -0.08, y: float = 1.01) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="bottom",
        color=COLORS["text"],
    )

def export_figure(fig: plt.Figure, out_base: Path) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if EXPORT_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=PNG_DPI, bbox_inches="tight")
    if EXPORT_TIF:
        fig.savefig(out_base.with_suffix(".tif"), dpi=TIF_DPI, bbox_inches="tight")
    plt.close(fig)

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper()
    return "".join(ch for ch in seq if ch in AA_SET)

def frac_of_categories(seqs: Sequence[str]) -> dict[str, float]:
    total_len = sum(len(s) for s in seqs) if len(seqs) > 0 else 0
    if total_len == 0:
        return {
            "Hydrophobic": np.nan,
            "Aromatic": np.nan,
            "Positive": np.nan,
            "Negative": np.nan,
            "Polar": np.nan,
        }

    def frac_of(res_set: set[str]) -> float:
        c = sum(sum(ch in res_set for ch in s) for s in seqs)
        return c / total_len

    return {
        "Hydrophobic": frac_of(AA_HYDROPHOBIC),
        "Aromatic": frac_of(AA_AROMATIC),
        "Positive": frac_of(AA_POSITIVE),
        "Negative": frac_of(AA_NEGATIVE),
        "Polar": frac_of(AA_POLAR),
    }

# =============================================================================
# Load inputs
# =============================================================================

for p in [FILE_AA_ENRICH, FILE_KMER_ENRICH, FILE_FINAL_ANNOT, FILE_REFERENCE_CORPUS]:
    require_file(p)

aa_enrich_df = pd.read_csv(FILE_AA_ENRICH)
kmer_enrich_df = pd.read_csv(FILE_KMER_ENRICH)
final_df = pd.read_csv(FILE_FINAL_ANNOT)
reference_df = pd.read_csv(FILE_REFERENCE_CORPUS)

final_seqs = [clean_sequence(x) for x in final_df["sequence"].astype(str).tolist()]
ref_seqs = [clean_sequence(x) for x in reference_df["sequence"].astype(str).tolist()]

ref_cat = frac_of_categories(ref_seqs)
final_cat = frac_of_categories(final_seqs)

# =============================================================================
# Figure
# =============================================================================

def plot_supplementary_figure_9_2() -> None:
    fig = plt.figure(figsize=(9.0, 5.9))
    gs = GridSpec(
        2, 2,
        figure=fig,
        width_ratios=[1.10, 1.02],
        height_ratios=[1.0, 0.72],
        wspace=0.32,
        hspace=0.38,
    )

    # -------------------------------------------------------------------------
    # Panel a
    # -------------------------------------------------------------------------
    ax_a = fig.add_subplot(gs[:, 0])
    add_panel_label(ax_a, "a", x=-0.06, y=1.005)
    style_axis(ax_a, grid_axis="x")

    tmp = aa_enrich_df.copy()
    tmp["log2_enrichment_vs_reference"] = pd.to_numeric(
        tmp["log2_enrichment_vs_reference"], errors="coerce"
    )
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)

    vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    labels = tmp["amino_acid"].astype(str).tolist()
    colors = [COLORS["neg_blue"] if v < 0 else COLORS["orange"] for v in vals]

    ax_a.barh(
        labels,
        vals,
        color=colors,
        edgecolor=COLORS["gray_dark"],
        linewidth=0.32,
        zorder=2,
    )
    ax_a.axvline(0, color=COLORS["gray_dark"], linewidth=0.72, zorder=3)
    ax_a.set_xlabel("log2 enrichment vs reference")
    ax_a.set_title("Amino-acid enrichment in final panel", loc="left", pad=4)

    xmin = min(-2.1, float(np.nanmin(vals) * 1.06))
    xmax = max(1.25, float(np.nanmax(vals) * 1.05))
    ax_a.set_xlim(xmin, xmax)

    # -------------------------------------------------------------------------
    # Panel b
    # -------------------------------------------------------------------------
    ax_b = fig.add_subplot(gs[0, 1])
    add_panel_label(ax_b, "b", x=-0.08, y=1.005)
    style_axis(ax_b, grid_axis="x")

    tmp = kmer_enrich_df.copy()
    tmp["log2_enrichment_vs_reference"] = pd.to_numeric(
        tmp["log2_enrichment_vs_reference"], errors="coerce"
    )
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=False).head(8)
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)

    kmer_vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    ax_b.barh(
        tmp["kmer"].astype(str).tolist(),
        kmer_vals,
        color=COLORS["teal"],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.32,
        zorder=2,
    )
    ax_b.set_xlabel("log2 enrichment vs reference")
    ax_b.set_title("Top enriched 3-mers", loc="left", pad=4)

    xmax = float(np.nanmax(kmer_vals))
    ax_b.set_xlim(0, max(12.0, xmax * 1.015))

    # -------------------------------------------------------------------------
    # Panel c
    # -------------------------------------------------------------------------
    ax_c = fig.add_subplot(gs[1, 1])
    add_panel_label(ax_c, "c", x=-0.08, y=1.005)
    style_axis(ax_c, grid_axis="y")

    categories = ["Hydrophobic", "Aromatic", "Positive", "Negative", "Polar"]
    x = np.arange(len(categories))
    width = 0.30

    ref_vals = np.array([ref_cat[c] for c in categories], dtype=float)
    final_vals = np.array([final_cat[c] for c in categories], dtype=float)

    ax_c.bar(
        x - width / 2,
        ref_vals,
        width=width,
        color=COLORS["gray_mid"],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.36,
        label="Reference ACP corpus",
        zorder=2,
    )
    ax_c.bar(
        x + width / 2,
        final_vals,
        width=width,
        color=COLORS["teal"],
        edgecolor=COLORS["gray_dark"],
        linewidth=0.36,
        label="Final panel",
        zorder=2,
    )

    ax_c.set_xticks(x)
    ax_c.set_xticklabels(categories, rotation=9, ha="right")
    ax_c.tick_params(axis="x", pad=4)
    ax_c.set_ylabel("Fraction")
    ax_c.set_title("Residue-category comparison", loc="left", pad=4)
    ax_c.set_ylim(0, max(0.53, float(np.nanmax(np.r_[ref_vals, final_vals]) * 1.03)))

    legend = ax_c.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.30),
        ncol=2,
        frameon=True,
        handlelength=1.0,
        handletextpad=0.45,
        columnspacing=0.75,
        borderaxespad=0.0,
        fontsize=7.2,
        fancybox=True,
    )
    legend.get_frame().set_facecolor(COLORS["legend_bg"])
    legend.get_frame().set_edgecolor(COLORS["legend_edge"])
    legend.get_frame().set_linewidth(0.75)
    legend.get_frame().set_alpha(1.0)

    fig.tight_layout(rect=[0.02, 0.09, 0.995, 0.985])
    export_figure(fig, OUT_COMP)

# =============================================================================
# Run
# =============================================================================

FIG_SUPP.mkdir(parents=True, exist_ok=True)
plot_supplementary_figure_9_2()

print("=" * 90)
print("Updated SFig 9.2 generated successfully.")
print(f"SFig 9.2: {OUT_COMP.with_suffix('.pdf')}")
print("=" * 90)

# Step 2 figure redesign only

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

# ============================================================
# CONFIG
# ============================================================
BASE = Path('/home/data3/Moe/nature_computational2')
STEP2_DIR = BASE / 'step2'
REDESIGN_DIR = BASE / 'redesign' / 'step2'
MAIN_DIR = REDESIGN_DIR / 'main_fig'
SUPP_DIR = REDESIGN_DIR / 'supplementary'
SRC_DIR = REDESIGN_DIR / 'source_data'

for p in [MAIN_DIR, SUPP_DIR, SRC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TABLES = STEP2_DIR / 'tables'
SPLITS = STEP2_DIR / 'splits'

SPLIT_SUMMARY = TABLES / 'step2_split_summary.csv'
LEAK_AUDITS = TABLES / 'step2_leakage_audits.csv'
NN_AUDIT = TABLES / 'step2_nearest_neighbor_audit.csv'
THRESH_SENS = TABLES / 'step2_threshold_sensitivity.csv'
ROW_SPLITS = SPLITS / 'step2_row_level_with_splits.csv'

# Palette tuned to the reference style:
COLORS = {
    'train_fill': '#D0D4D7',
    'train_edge': '#8F979E',
    'val_fill':   '#E1C461',
    'val_edge':   '#BB992D',
    'test_fill':  '#A6D1CC',
    'test_edge':  '#6C9F9D',
    'audit1':     '#627FB3',
    'audit2':     '#A88443',
    'audit3':     '#6C99A7',
    'hist_fill':  '#6FA8A6',
    'line':       '#567CB8',
    'grid':       '#EAEAEA',
    'text':       '#222222',
    'axis':       '#666666',
    'threshold':  '#7D7D7D',
    'median':     '#353535',
    'p90':        '#5C85BE',
}

DPI_PNG = 300
DPI_TIFF = 600
MM_TO_INCH = 1 / 25.4
MAIN_WIDTH_IN = 180 * MM_TO_INCH
SUPP_WIDTH_IN = 180 * MM_TO_INCH

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 0.8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})


# ============================================================
# HELPERS
# ============================================================
def save_all(fig: plt.Figure, stem: Path) -> None:
    fig.savefig(f'{stem}.png', dpi=DPI_PNG, bbox_inches='tight')
    fig.savefig(f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{stem}.tiff', dpi=DPI_TIFF, bbox_inches='tight')


def style_ax(ax: plt.Axes, grid_axis: str = 'y') -> None:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(COLORS['axis'])
    ax.spines['bottom'].set_color(COLORS['axis'])
    ax.tick_params(colors=COLORS['text'], width=0.8)
    ax.grid(True, axis=grid_axis, color=COLORS['grid'], linewidth=0.6)
    ax.set_axisbelow(True)


def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.15, 1.085, label, transform=ax.transAxes,
            fontsize=17.5, fontweight='bold', va='top', ha='left', color=COLORS['text'])


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def write_source(df: pd.DataFrame, name: str) -> None:
    df.to_csv(SRC_DIR / name, index=False)


def add_value_labels(ax: plt.Axes, bars, fmt='{:,}', dy=0.010, fontsize=7.2) -> None:
    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width() / 2,
            h + yspan * dy,
            fmt.format(int(round(h)) if fmt == '{:,}' else h),
            ha='center',
            va='bottom',
            fontsize=fontsize,
            color=COLORS['text'],
        )


def finite_values(x: pd.Series) -> np.ndarray:
    vals = pd.to_numeric(x, errors='coerce').to_numpy(dtype=float)
    return vals[np.isfinite(vals)]


# ============================================================
# LOAD DATA
# ============================================================
def load_inputs() -> Dict[str, pd.DataFrame]:
    return {
        'split_summary': read_csv(SPLIT_SUMMARY),
        'leak_audits': read_csv(LEAK_AUDITS),
        'nn_audit': read_csv(NN_AUDIT),
        'threshold_sensitivity': read_csv(THRESH_SENS),
        'row_splits': read_csv(ROW_SPLITS),
    }


# ============================================================
# MAIN FIGURE
# ============================================================
def make_main_fig(data: Dict[str, pd.DataFrame]) -> None:
    split_summary = data['split_summary'].copy()
    leak = data['leak_audits'].copy()

    # Panel a source data
    split_order = ['train', 'val', 'test']
    split_summary['split'] = pd.Categorical(split_summary['split'], categories=split_order, ordered=True)
    split_summary = split_summary.sort_values('split')
    panel_a = split_summary[['split', 'n_unique_sequences']].rename(columns={'n_unique_sequences': 'unique_sequences'})
    write_source(panel_a, 'fig2_main_panel_a_unique_sequences.csv')

    # Exact overlap saved as source data only
    exact = leak.loc[leak['audit_type'] == 'exact_sequence_overlap', ['split_a', 'split_b', 'n_overlap']].copy()
    exact['pair'] = exact['split_a'] + ' vs ' + exact['split_b']
    write_source(exact[['pair', 'n_overlap']], 'fig2_exact_overlap_source_data.csv')

    # Panel b source data
    sampled = leak.loc[
        leak['audit_type'] == 'sampled_cross_split_jaccard',
        ['split_a', 'split_b', 'max_jaccard', 'threshold'],
    ].copy()
    pair_order = ['train–val', 'train–test', 'val–test']
    sampled['pair'] = sampled['split_a'].map({'train': 'train', 'val': 'val', 'test': 'test'}) + '–' + sampled['split_b'].map({'train': 'train', 'val': 'val', 'test': 'test'})
    sampled['pair'] = pd.Categorical(sampled['pair'], categories=pair_order, ordered=True)
    sampled = sampled.sort_values('pair')
    write_source(sampled[['pair', 'max_jaccard', 'threshold']], 'fig2_main_panel_b_max_sampled_cross_split_jaccard.csv')

    fig, axes = plt.subplots(1, 2, figsize=(MAIN_WIDTH_IN, 3.75))
    fig.subplots_adjust(left=0.082, right=0.982, top=0.91, bottom=0.21, wspace=0.35)

    # Panel a
    ax = axes[0]
    x = np.arange(len(split_order))
    fills = [COLORS['train_fill'], COLORS['val_fill'], COLORS['test_fill']]
    edges = [COLORS['train_edge'], COLORS['val_edge'], COLORS['test_edge']]
    bars = ax.bar(
        x,
        panel_a['unique_sequences'].to_numpy(),
        width=0.52,
        color=fills,
        edgecolor=edges,
        linewidth=1.1,
    )
    ax.set_ylabel('Unique sequences')
    ax.set_xticks(x)
    ax.set_xticklabels(split_order)
    ax.set_ylim(0, float(panel_a['unique_sequences'].max()) * 1.05)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    add_value_labels(ax, bars, fmt='{:,}', dy=0.011, fontsize=7.8)
    panel_label(ax, 'a')

    # Panel b
    ax = axes[1]
    x = np.arange(len(pair_order))
    audit_colors = [COLORS['audit1'], COLORS['audit2'], COLORS['audit3']]
    bars = ax.bar(
        x,
        sampled['max_jaccard'].to_numpy(),
        width=0.46,
        color=audit_colors,
        edgecolor='#666666',
        linewidth=0.9,
    )
    thr = float(sampled['threshold'].dropna().iloc[0]) if sampled['threshold'].notna().any() else 0.55
    ax.axhline(thr, color=COLORS['threshold'], linestyle=(0, (4, 2)), linewidth=1.0)
    ax.text(
        0.97,
        0.965,
        f'Threshold = {thr:.2f}',
        transform=ax.transAxes,
        ha='right',
        va='bottom',
        fontsize=7.8,
        color=COLORS['threshold'],
    )
    ax.set_ylabel('Max sampled cross-split Jaccard')
    ax.set_xticks(x)
    ax.set_xticklabels(pair_order, rotation=8, ha='right')
    ymax = max(thr + 0.03, float(sampled['max_jaccard'].max()) + 0.05)
    ax.set_ylim(0, ymax)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    for b in bars:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.012,
            f'{b.get_height():.2f}',
            ha='center',
            va='bottom',
            fontsize=7.8,
            color=COLORS['text'],
        )
    panel_label(ax, 'b')

    save_all(fig, MAIN_DIR / 'Fig2_step2_main_redesigned_v4')
    plt.close(fig)


# ============================================================
# SUPPLEMENTARY FIGURE 2.1
# ============================================================
def violin_box(ax: plt.Axes, data_list: List[np.ndarray], fill_colors: List[str], edge_colors: List[str], ylabel: str) -> None:
    pos = np.arange(1, len(data_list) + 1)
    vp = ax.violinplot(data_list, positions=pos, widths=0.56, showmeans=False, showmedians=False, showextrema=False)
    for body, fill, edge in zip(vp['bodies'], fill_colors, edge_colors):
        body.set_facecolor(fill)
        body.set_edgecolor(edge)
        body.set_linewidth(0.9)
        body.set_alpha(0.16)

    bp = ax.boxplot(data_list, positions=pos, widths=0.12, patch_artist=True, showfliers=False, whis=(5, 95))
    for patch, edge in zip(bp['boxes'], edge_colors):
        patch.set_facecolor('white')
        patch.set_edgecolor(edge)
        patch.set_linewidth(0.9)
        patch.set_alpha(0.92)
    for med in bp['medians']:
        med.set_color(COLORS['text'])
        med.set_linewidth(1.0)
    for whisk in bp['whiskers']:
        whisk.set_color('#7A7A7A')
        whisk.set_linewidth(0.75)
    for cap in bp['caps']:
        cap.set_color('#7A7A7A')
        cap.set_linewidth(0.75)

    ax.set_xticks(pos)
    ax.set_xticklabels(['train', 'val', 'test'])
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    style_ax(ax, 'y')


def make_supp_balance(data: Dict[str, pd.DataFrame]) -> None:
    row = data['row_splits'].copy()
    cols = {
        'length': 'Length (aa)',
        'net_charge_pH7': 'Net charge',
        'hydrophobicity_KD': 'Hydropathy',
        'entropy': 'Entropy (bits)',
    }
    source = row[['assigned_split'] + list(cols.keys())].copy()
    write_source(source, 'figS2_1_descriptor_balance_source_data.csv')

    fig, axes = plt.subplots(2, 2, figsize=(SUPP_WIDTH_IN, 5.45))
    fig.subplots_adjust(left=0.075, right=0.985, top=0.94, bottom=0.15, wspace=0.28, hspace=0.32)

    split_order = ['train', 'val', 'test']
    fill_colors = [COLORS['train_fill'], COLORS['val_fill'], COLORS['test_fill']]
    edge_colors = [COLORS['train_edge'], COLORS['val_edge'], COLORS['test_edge']]

    for ax, (col, ylabel), lab in zip(axes.ravel(), cols.items(), ['a', 'b', 'c', 'd']):
        data_list = [finite_values(row.loc[row['assigned_split'] == s, col]) for s in split_order]
        violin_box(ax, data_list, fill_colors, edge_colors, ylabel)
        panel_label(ax, lab)

    handles = [
        Patch(facecolor=fill_colors[0], edgecolor=edge_colors[0], alpha=0.32, label='train'),
        Patch(facecolor=fill_colors[1], edgecolor=edge_colors[1], alpha=0.32, label='val'),
        Patch(facecolor=fill_colors[2], edgecolor=edge_colors[2], alpha=0.32, label='test'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
               bbox_to_anchor=(0.5, 0.03), handlelength=2.2, columnspacing=2.0, fontsize=8.5)

    save_all(fig, SUPP_DIR / 'FigS2_1_descriptor_balance_redesigned_v4')
    plt.close(fig)


# ============================================================
# SUPPLEMENTARY FIGURE 2.2
# ============================================================
def make_supp_leakage(data: Dict[str, pd.DataFrame]) -> None:
    nn = data['nn_audit'].copy()
    sens = data['threshold_sensitivity'].copy().sort_values('threshold')

    write_source(
        nn[['query_split', 'nearest_train_jaccard', 'n_candidates_scanned']].copy(),
        'figS2_2_panel_a_nearest_train_jaccard_source_data.csv',
    )
    write_source(
        sens[['threshold', 'n_edges', 'train_n_sequences', 'val_n_sequences', 'test_n_sequences']].copy(),
        'figS2_2_panel_b_threshold_robustness_source_data.csv',
    )

    fig, axes = plt.subplots(1, 2, figsize=(SUPP_WIDTH_IN, 3.2))
    fig.subplots_adjust(left=0.08, right=0.985, top=0.90, bottom=0.21, wspace=0.33)

    # Panel a: histogram
    ax = axes[0]
    vals = finite_values(nn['nearest_train_jaccard'])
    bins = np.linspace(max(0.0, vals.min() - 0.005), min(0.60, vals.max() + 0.01), 13)
    ax.hist(vals, bins=bins, color=COLORS['hist_fill'], edgecolor='white', linewidth=0.6, alpha=0.78)
    med = float(np.median(vals)) if len(vals) else np.nan
    p90 = float(np.percentile(vals, 90)) if len(vals) else np.nan
    if np.isfinite(med):
        ax.axvline(med, color=COLORS['median'], linewidth=1.0)
    if np.isfinite(p90):
        ax.axvline(p90, color=COLORS['p90'], linewidth=1.0, linestyle=(0, (3, 2)))
    ax.text(
        0.965, 0.94,
        f'Median = {med:.2f}\nP90 = {p90:.2f}',
        transform=ax.transAxes,
        ha='right', va='top', fontsize=7.2, color=COLORS['text']
    )
    ax.set_xlabel('Nearest-train Jaccard')
    ax.set_ylabel('Count')
    ax.set_xlim(max(0, vals.min() - 0.01), min(0.60, vals.max() + 0.03))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    panel_label(ax, 'a')

    # Panel b: threshold robustness
    ax = axes[1]
    ax.plot(
        sens['threshold'], sens['n_edges'],
        color=COLORS['line'], linewidth=1.5, marker='o', markersize=4.8,
        markerfacecolor=COLORS['line'], markeredgecolor=COLORS['line']
    )
    first = sens.iloc[0]
    last = sens.iloc[-1]
    yspan = float(sens['n_edges'].max() - sens['n_edges'].min())
    ax.text(first['threshold'] + 0.004, first['n_edges'] + yspan * 0.05,
            f"{int(first['n_edges']):,}", ha='left', va='bottom', fontsize=7.8, color=COLORS['text'])
    ax.text(last['threshold'] - 0.002, last['n_edges'] + yspan * 0.02,
            f"{int(last['n_edges']):,}", ha='left', va='bottom', fontsize=7.8, color=COLORS['text'])
    ax.set_xlabel('Jaccard threshold')
    ax.set_ylabel('Retained edges')
    ax.set_xticks([0.45, 0.55, 0.65])
    ax.set_xlim(0.44, 0.66)
    ax.set_ylim(float(sens['n_edges'].min()) * 0.93, float(sens['n_edges'].max()) * 1.03)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    panel_label(ax, 'b')

    save_all(fig, SUPP_DIR / 'FigS2_2_leakage_diagnostics_redesigned_v4')
    plt.close(fig)


# ============================================================
# ENTRYPOINT
# ============================================================
def main() -> None:
    data = load_inputs()
    make_main_fig(data)
    make_supp_balance(data)
    make_supp_leakage(data)
    print('Step 2 redesigned figures saved to:', REDESIGN_DIR)


if __name__ == '__main__':
    main()

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

# ============================================================
# CONFIG
# ============================================================
BASE = Path('/home/data3/Moe/nature_computational2')
STEP2_DIR = BASE / 'step2'
REDESIGN_DIR = BASE / 'redesign' / 'step2'
MAIN_DIR = REDESIGN_DIR / 'main_fig'
SUPP_DIR = REDESIGN_DIR / 'supplementary'
SRC_DIR = REDESIGN_DIR / 'source_data'

for p in [MAIN_DIR, SUPP_DIR, SRC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TABLES = STEP2_DIR / 'tables'
SPLITS = STEP2_DIR / 'splits'

SPLIT_SUMMARY = TABLES / 'step2_split_summary.csv'
LEAK_AUDITS = TABLES / 'step2_leakage_audits.csv'
NN_AUDIT = TABLES / 'step2_nearest_neighbor_audit.csv'
THRESH_SENS = TABLES / 'step2_threshold_sensitivity.csv'
ROW_SPLITS = SPLITS / 'step2_row_level_with_splits.csv'

# Palette tuned to the reference style:
COLORS = {
    'train_fill': '#C9CED2',
    'train_edge': '#8F979E',
    'val_fill':   '#E0C15A',
    'val_edge':   '#BB992D',
    'test_fill':  '#92C8C2',
    'test_edge':  '#6C9F9D',
    'audit1':     '#627FB3',
    'audit2':     '#A88443',
    'audit3':     '#6C99A7',
    'hist_fill':  '#6FA8A6',
    'line':       '#567CB8',
    'grid':       '#EAEAEA',
    'text':       '#222222',
    'axis':       '#666666',
    'threshold':  '#7D7D7D',
    'median':     '#353535',
    'p90':        '#5C85BE',
}

DPI_PNG = 300
DPI_TIFF = 600
MM_TO_INCH = 1 / 25.4
MAIN_WIDTH_IN = 180 * MM_TO_INCH
SUPP_WIDTH_IN = 180 * MM_TO_INCH

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 0.8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})


# ============================================================
# HELPERS
# ============================================================
def save_all(fig: plt.Figure, stem: Path) -> None:
    fig.savefig(f'{stem}.png', dpi=DPI_PNG, bbox_inches='tight')
    fig.savefig(f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{stem}.tiff', dpi=DPI_TIFF, bbox_inches='tight')


def style_ax(ax: plt.Axes, grid_axis: str = 'y') -> None:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(COLORS['axis'])
    ax.spines['bottom'].set_color(COLORS['axis'])
    ax.tick_params(colors=COLORS['text'], width=0.8)
    ax.grid(True, axis=grid_axis, color=COLORS['grid'], linewidth=0.6)
    ax.set_axisbelow(True)


def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.15, 1.085, label, transform=ax.transAxes,
            fontsize=17.5, fontweight='bold', va='top', ha='left', color=COLORS['text'])


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def write_source(df: pd.DataFrame, name: str) -> None:
    df.to_csv(SRC_DIR / name, index=False)


def add_value_labels(ax: plt.Axes, bars, fmt='{:,}', dy=0.010, fontsize=7.2) -> None:
    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width() / 2,
            h + yspan * dy,
            fmt.format(int(round(h)) if fmt == '{:,}' else h),
            ha='center',
            va='bottom',
            fontsize=fontsize,
            color=COLORS['text'],
        )


def finite_values(x: pd.Series) -> np.ndarray:
    vals = pd.to_numeric(x, errors='coerce').to_numpy(dtype=float)
    return vals[np.isfinite(vals)]


# ============================================================
# LOAD DATA
# ============================================================
def load_inputs() -> Dict[str, pd.DataFrame]:
    return {
        'split_summary': read_csv(SPLIT_SUMMARY),
        'leak_audits': read_csv(LEAK_AUDITS),
        'nn_audit': read_csv(NN_AUDIT),
        'threshold_sensitivity': read_csv(THRESH_SENS),
        'row_splits': read_csv(ROW_SPLITS),
    }


# ============================================================
# MAIN FIGURE
# ============================================================
def make_main_fig(data: Dict[str, pd.DataFrame]) -> None:
    split_summary = data['split_summary'].copy()
    leak = data['leak_audits'].copy()

    # Panel a source data
    split_order = ['train', 'val', 'test']
    split_summary['split'] = pd.Categorical(split_summary['split'], categories=split_order, ordered=True)
    split_summary = split_summary.sort_values('split')
    panel_a = split_summary[['split', 'n_unique_sequences']].rename(columns={'n_unique_sequences': 'unique_sequences'})
    write_source(panel_a, 'fig2_main_panel_a_unique_sequences.csv')

    # Exact overlap saved as source data only
    exact = leak.loc[leak['audit_type'] == 'exact_sequence_overlap', ['split_a', 'split_b', 'n_overlap']].copy()
    exact['pair'] = exact['split_a'] + ' vs ' + exact['split_b']
    write_source(exact[['pair', 'n_overlap']], 'fig2_exact_overlap_source_data.csv')

    # Panel b source data
    sampled = leak.loc[
        leak['audit_type'] == 'sampled_cross_split_jaccard',
        ['split_a', 'split_b', 'max_jaccard', 'threshold'],
    ].copy()
    pair_order = ['train–val', 'train–test', 'val–test']
    sampled['pair'] = sampled['split_a'].map({'train': 'train', 'val': 'val', 'test': 'test'}) + '–' + sampled['split_b'].map({'train': 'train', 'val': 'val', 'test': 'test'})
    sampled['pair'] = pd.Categorical(sampled['pair'], categories=pair_order, ordered=True)
    sampled = sampled.sort_values('pair')
    write_source(sampled[['pair', 'max_jaccard', 'threshold']], 'fig2_main_panel_b_max_sampled_cross_split_jaccard.csv')

    fig, axes = plt.subplots(1, 2, figsize=(MAIN_WIDTH_IN, 3.75))
    fig.subplots_adjust(left=0.082, right=0.982, top=0.91, bottom=0.21, wspace=0.35)

    # Panel a
    ax = axes[0]
    x = np.arange(len(split_order))
    fills = [COLORS['train_fill'], COLORS['val_fill'], COLORS['test_fill']]
    edges = [COLORS['train_edge'], COLORS['val_edge'], COLORS['test_edge']]
    bars = ax.bar(
        x,
        panel_a['unique_sequences'].to_numpy(),
        width=0.52,
        color=fills,
        edgecolor=edges,
        linewidth=1.1,
    )
    ax.set_ylabel('Unique sequences')
    ax.set_xticks(x)
    ax.set_xticklabels(split_order)
    ax.set_ylim(0, float(panel_a['unique_sequences'].max()) * 1.05)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    add_value_labels(ax, bars, fmt='{:,}', dy=0.011, fontsize=7.8)
    panel_label(ax, 'a')

    # Panel b
    ax = axes[1]
    x = np.arange(len(pair_order))
    audit_colors = [COLORS['audit1'], COLORS['audit2'], COLORS['audit3']]
    bars = ax.bar(
        x,
        sampled['max_jaccard'].to_numpy(),
        width=0.46,
        color=audit_colors,
        edgecolor='#666666',
        linewidth=0.9,
    )
    thr = float(sampled['threshold'].dropna().iloc[0]) if sampled['threshold'].notna().any() else 0.55
    ax.axhline(thr, color=COLORS['threshold'], linestyle=(0, (4, 2)), linewidth=1.0)
    ax.text(
        0.97,
        0.965,
        f'Threshold = {thr:.2f}',
        transform=ax.transAxes,
        ha='right',
        va='bottom',
        fontsize=7.8,
        color=COLORS['threshold'],
    )
    ax.set_ylabel('Max sampled cross-split Jaccard')
    ax.set_xticks(x)
    ax.set_xticklabels(pair_order, rotation=8, ha='right')
    ymax = max(thr + 0.03, float(sampled['max_jaccard'].max()) + 0.05)
    ax.set_ylim(0, ymax)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    for b in bars:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.012,
            f'{b.get_height():.2f}',
            ha='center',
            va='bottom',
            fontsize=7.8,
            color=COLORS['text'],
        )
    panel_label(ax, 'b')

    save_all(fig, MAIN_DIR / 'Fig2_step2_main_redesigned_v5')
    plt.close(fig)


# ============================================================
# SUPPLEMENTARY FIGURE 2.1
# ============================================================
def violin_box(ax: plt.Axes, data_list: List[np.ndarray], fill_colors: List[str], edge_colors: List[str], ylabel: str) -> None:
    pos = np.arange(1, len(data_list) + 1)
    vp = ax.violinplot(data_list, positions=pos, widths=0.56, showmeans=False, showmedians=False, showextrema=False)
    for body, fill, edge in zip(vp['bodies'], fill_colors, edge_colors):
        body.set_facecolor(fill)
        body.set_edgecolor(edge)
        body.set_linewidth(0.9)
        body.set_alpha(0.28)

    bp = ax.boxplot(data_list, positions=pos, widths=0.12, patch_artist=True, showfliers=False, whis=(5, 95))
    for patch, edge in zip(bp['boxes'], edge_colors):
        patch.set_facecolor('white')
        patch.set_edgecolor(edge)
        patch.set_linewidth(0.9)
        patch.set_alpha(0.98)
    for med in bp['medians']:
        med.set_color(COLORS['text'])
        med.set_linewidth(1.0)
    for whisk in bp['whiskers']:
        whisk.set_color('#7A7A7A')
        whisk.set_linewidth(0.75)
    for cap in bp['caps']:
        cap.set_color('#7A7A7A')
        cap.set_linewidth(0.75)

    ax.set_xticks(pos)
    ax.set_xticklabels(['train', 'val', 'test'])
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    style_ax(ax, 'y')


def make_supp_balance(data: Dict[str, pd.DataFrame]) -> None:
    row = data['row_splits'].copy()
    cols = {
        'length': 'Length (aa)',
        'net_charge_pH7': 'Net charge',
        'hydrophobicity_KD': 'Hydropathy',
        'entropy': 'Entropy (bits)',
    }
    source = row[['assigned_split'] + list(cols.keys())].copy()
    write_source(source, 'figS2_1_descriptor_balance_source_data.csv')

    fig, axes = plt.subplots(2, 2, figsize=(SUPP_WIDTH_IN, 5.45))
    fig.subplots_adjust(left=0.075, right=0.985, top=0.94, bottom=0.15, wspace=0.28, hspace=0.32)

    split_order = ['train', 'val', 'test']
    fill_colors = [COLORS['train_fill'], COLORS['val_fill'], COLORS['test_fill']]
    edge_colors = [COLORS['train_edge'], COLORS['val_edge'], COLORS['test_edge']]

    for ax, (col, ylabel), lab in zip(axes.ravel(), cols.items(), ['a', 'b', 'c', 'd']):
        data_list = [finite_values(row.loc[row['assigned_split'] == s, col]) for s in split_order]
        violin_box(ax, data_list, fill_colors, edge_colors, ylabel)
        panel_label(ax, lab)

    handles = [
        Patch(facecolor=fill_colors[0], edgecolor=edge_colors[0], alpha=0.45, label='train'),
        Patch(facecolor=fill_colors[1], edgecolor=edge_colors[1], alpha=0.45, label='val'),
        Patch(facecolor=fill_colors[2], edgecolor=edge_colors[2], alpha=0.45, label='test'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
               bbox_to_anchor=(0.5, 0.025), handlelength=2.2, columnspacing=2.0, fontsize=8.5)

    save_all(fig, SUPP_DIR / 'FigS2_1_descriptor_balance_redesigned_v5')
    plt.close(fig)


# ============================================================
# SUPPLEMENTARY FIGURE 2.2
# ============================================================
def make_supp_leakage(data: Dict[str, pd.DataFrame]) -> None:
    nn = data['nn_audit'].copy()
    sens = data['threshold_sensitivity'].copy().sort_values('threshold')

    write_source(
        nn[['query_split', 'nearest_train_jaccard', 'n_candidates_scanned']].copy(),
        'figS2_2_panel_a_nearest_train_jaccard_source_data.csv',
    )
    write_source(
        sens[['threshold', 'n_edges', 'train_n_sequences', 'val_n_sequences', 'test_n_sequences']].copy(),
        'figS2_2_panel_b_threshold_robustness_source_data.csv',
    )

    fig, axes = plt.subplots(1, 2, figsize=(SUPP_WIDTH_IN, 3.2))
    fig.subplots_adjust(left=0.08, right=0.985, top=0.90, bottom=0.21, wspace=0.33)

    # Panel a: histogram
    ax = axes[0]
    vals = finite_values(nn['nearest_train_jaccard'])
    bins = np.linspace(max(0.0, vals.min() - 0.005), min(0.60, vals.max() + 0.01), 13)
    ax.hist(vals, bins=bins, color=COLORS['hist_fill'], edgecolor='white', linewidth=0.6, alpha=0.78)
    med = float(np.median(vals)) if len(vals) else np.nan
    p90 = float(np.percentile(vals, 90)) if len(vals) else np.nan
    if np.isfinite(med):
        ax.axvline(med, color=COLORS['median'], linewidth=1.0)
    if np.isfinite(p90):
        ax.axvline(p90, color=COLORS['p90'], linewidth=1.0, linestyle=(0, (3, 2)))
    ax.text(
        0.965, 0.94,
        f'Median = {med:.2f}\nP90 = {p90:.2f}',
        transform=ax.transAxes,
        ha='right', va='top', fontsize=7.2, color=COLORS['text']
    )
    ax.set_xlabel('Nearest-train Jaccard')
    ax.set_ylabel('Count')
    ax.set_xlim(max(0, vals.min() - 0.01), min(0.60, vals.max() + 0.03))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    panel_label(ax, 'a')

    # Panel b: threshold robustness
    ax = axes[1]
    ax.plot(
        sens['threshold'], sens['n_edges'],
        color=COLORS['line'], linewidth=1.5, marker='o', markersize=4.8,
        markerfacecolor=COLORS['line'], markeredgecolor=COLORS['line']
    )
    first = sens.iloc[0]
    last = sens.iloc[-1]
    yspan = float(sens['n_edges'].max() - sens['n_edges'].min())
    ax.text(first['threshold'] + 0.004, first['n_edges'] + yspan * 0.05,
            f"{int(first['n_edges']):,}", ha='left', va='bottom', fontsize=7.8, color=COLORS['text'])
    ax.text(last['threshold'] - 0.002, last['n_edges'] + yspan * 0.02,
            f"{int(last['n_edges']):,}", ha='left', va='bottom', fontsize=7.8, color=COLORS['text'])
    ax.set_xlabel('Jaccard threshold')
    ax.set_ylabel('Retained edges')
    ax.set_xticks([0.45, 0.55, 0.65])
    ax.set_xlim(0.44, 0.66)
    ax.set_ylim(float(sens['n_edges'].min()) * 0.93, float(sens['n_edges'].max()) * 1.03)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    panel_label(ax, 'b')

    save_all(fig, SUPP_DIR / 'FigS2_2_leakage_diagnostics_redesigned_v4')
    plt.close(fig)


# ============================================================
# ENTRYPOINT
# ============================================================
def main() -> None:
    data = load_inputs()
    make_main_fig(data)
    make_supp_balance(data)
    make_supp_leakage(data)
    print('Step 2 redesigned figures saved to:', REDESIGN_DIR)


if __name__ == '__main__':
    main()

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

# ============================================================
# CONFIG
# ============================================================
BASE = Path('/home/data3/Moe/nature_computational2')
STEP2_DIR = BASE / 'step2'
REDESIGN_DIR = BASE / 'redesign' / 'step2'
MAIN_DIR = REDESIGN_DIR / 'main_fig'
SUPP_DIR = REDESIGN_DIR / 'supplementary'
SRC_DIR = REDESIGN_DIR / 'source_data'

for p in [MAIN_DIR, SUPP_DIR, SRC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TABLES = STEP2_DIR / 'tables'
SPLITS = STEP2_DIR / 'splits'

SPLIT_SUMMARY = TABLES / 'step2_split_summary.csv'
LEAK_AUDITS = TABLES / 'step2_leakage_audits.csv'
NN_AUDIT = TABLES / 'step2_nearest_neighbor_audit.csv'
THRESH_SENS = TABLES / 'step2_threshold_sensitivity.csv'
ROW_SPLITS = SPLITS / 'step2_row_level_with_splits.csv'

# Palette tuned to the reference style:
COLORS = {
    'train_fill': '#D9D9D9',
    'train_edge': '#8C8C8C',

    'val_fill':   '#D8B64C',
    'val_edge':   '#9C7A12',

    'test_fill':  '#8FC7C1',
    'test_edge':  '#4F8F89',

    'audit1':     '#4C72B0',   # strong blue
    'audit2':     '#A07A3F',   # brown-gold
    'audit3':     '#7F8F96',   # blue-grey

    'hist_fill':  '#72B6AF',
    'line':       '#3F6FB5',

    'grid':       '#E6E6E6',
    'text':       '#222222',
    'axis':       '#666666',
    'threshold':  '#7A7A7A',
    'median':     '#2F2F2F',
    'p90':        '#6D8FC4',
}
DPI_PNG = 300
DPI_TIFF = 600
MM_TO_INCH = 1 / 25.4
MAIN_WIDTH_IN = 180 * MM_TO_INCH
SUPP_WIDTH_IN = 180 * MM_TO_INCH

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 0.8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})


# ============================================================
# HELPERS
# ============================================================
def save_all(fig: plt.Figure, stem: Path) -> None:
    fig.savefig(f'{stem}.png', dpi=DPI_PNG, bbox_inches='tight')
    fig.savefig(f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{stem}.tiff', dpi=DPI_TIFF, bbox_inches='tight')


def style_ax(ax: plt.Axes, grid_axis: str = 'y') -> None:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(COLORS['axis'])
    ax.spines['bottom'].set_color(COLORS['axis'])
    ax.tick_params(colors=COLORS['text'], width=0.8)
    ax.grid(True, axis=grid_axis, color=COLORS['grid'], linewidth=0.6)
    ax.set_axisbelow(True)


def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.18, 1.10, label, transform=ax.transAxes,
            fontsize=17.5, fontweight='bold', va='top', ha='left', color=COLORS['text'])


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def write_source(df: pd.DataFrame, name: str) -> None:
    df.to_csv(SRC_DIR / name, index=False)


def add_value_labels(ax: plt.Axes, bars, fmt='{:,}', dy=0.010, fontsize=7.2) -> None:
    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width() / 2,
            h + yspan * dy,
            fmt.format(int(round(h)) if fmt == '{:,}' else h),
            ha='center',
            va='bottom',
            fontsize=fontsize,
            color=COLORS['text'],
        )


def finite_values(x: pd.Series) -> np.ndarray:
    vals = pd.to_numeric(x, errors='coerce').to_numpy(dtype=float)
    return vals[np.isfinite(vals)]


# ============================================================
# LOAD DATA
# ============================================================
def load_inputs() -> Dict[str, pd.DataFrame]:
    return {
        'split_summary': read_csv(SPLIT_SUMMARY),
        'leak_audits': read_csv(LEAK_AUDITS),
        'nn_audit': read_csv(NN_AUDIT),
        'threshold_sensitivity': read_csv(THRESH_SENS),
        'row_splits': read_csv(ROW_SPLITS),
    }


# ============================================================
# MAIN FIGURE
# ============================================================
def make_main_fig(data: Dict[str, pd.DataFrame]) -> None:
    split_summary = data['split_summary'].copy()
    leak = data['leak_audits'].copy()

    # Panel a source data
    split_order = ['train', 'val', 'test']
    split_summary['split'] = pd.Categorical(split_summary['split'], categories=split_order, ordered=True)
    split_summary = split_summary.sort_values('split')
    panel_a = split_summary[['split', 'n_unique_sequences']].rename(columns={'n_unique_sequences': 'unique_sequences'})
    write_source(panel_a, 'fig2_main_panel_a_unique_sequences.csv')

    # Exact overlap saved as source data only
    exact = leak.loc[leak['audit_type'] == 'exact_sequence_overlap', ['split_a', 'split_b', 'n_overlap']].copy()
    exact['pair'] = exact['split_a'] + ' vs ' + exact['split_b']
    write_source(exact[['pair', 'n_overlap']], 'fig2_exact_overlap_source_data.csv')

    # Panel b source data
    sampled = leak.loc[
        leak['audit_type'] == 'sampled_cross_split_jaccard',
        ['split_a', 'split_b', 'max_jaccard', 'threshold'],
    ].copy()
    pair_order = ['train–val', 'train–test', 'val–test']
    sampled['pair'] = sampled['split_a'].map({'train': 'train', 'val': 'val', 'test': 'test'}) + '–' + sampled['split_b'].map({'train': 'train', 'val': 'val', 'test': 'test'})
    sampled['pair'] = pd.Categorical(sampled['pair'], categories=pair_order, ordered=True)
    sampled = sampled.sort_values('pair')
    write_source(sampled[['pair', 'max_jaccard', 'threshold']], 'fig2_main_panel_b_max_sampled_cross_split_jaccard.csv')

    fig, axes = plt.subplots(1, 2, figsize=(MAIN_WIDTH_IN, 3.75))
    fig.subplots_adjust(left=0.082, right=0.982, top=0.91, bottom=0.21, wspace=0.35)

    # Panel a
    ax = axes[0]
    x = np.arange(len(split_order))
    fills = [COLORS['train_fill'], COLORS['val_fill'], COLORS['test_fill']]
    edges = [COLORS['train_edge'], COLORS['val_edge'], COLORS['test_edge']]
    bars = ax.bar(
        x,
        panel_a['unique_sequences'].to_numpy(),
        width=0.52,
        color=fills,
        edgecolor=edges,
        linewidth=1.1,
    )
    ax.set_ylabel('Unique sequences')
    ax.set_xticks(x)
    ax.set_xticklabels(split_order)
    ax.set_ylim(0, float(panel_a['unique_sequences'].max()) * 1.05)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    add_value_labels(ax, bars, fmt='{:,}', dy=0.011, fontsize=7.8)
    panel_label(ax, 'a')

    # Panel b
    ax = axes[1]
    x = np.arange(len(pair_order))
    audit_colors = [COLORS['audit1'], COLORS['audit2'], COLORS['audit3']]
    bars = ax.bar(
        x,
        sampled['max_jaccard'].to_numpy(),
        width=0.46,
        color=audit_colors,
        edgecolor='#666666',
        linewidth=0.9,
    )
    thr = float(sampled['threshold'].dropna().iloc[0]) if sampled['threshold'].notna().any() else 0.55
    ax.axhline(thr, color=COLORS['threshold'], linestyle=(0, (4, 2)), linewidth=1.0)
    ax.text(
        0.97,
        0.965,
        f'Threshold = {thr:.2f}',
        transform=ax.transAxes,
        ha='right',
        va='bottom',
        fontsize=7.8,
        color=COLORS['threshold'],
    )
    ax.set_ylabel('Max sampled cross-split Jaccard')
    ax.set_xticks(x)
    ax.set_xticklabels(pair_order, rotation=8, ha='right')
    ymax = max(thr + 0.03, float(sampled['max_jaccard'].max()) + 0.05)
    ax.set_ylim(0, ymax)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    for b in bars:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.012,
            f'{b.get_height():.2f}',
            ha='center',
            va='bottom',
            fontsize=7.8,
            color=COLORS['text'],
        )
    panel_label(ax, 'b')

    save_all(fig, MAIN_DIR / 'Fig2_step2_main_redesigned_v7')
    plt.close(fig)


# ============================================================
# SUPPLEMENTARY FIGURE 2.1
# ============================================================
def violin_box(ax: plt.Axes, data_list: List[np.ndarray], fill_colors: List[str], edge_colors: List[str], ylabel: str) -> None:
    pos = np.arange(1, len(data_list) + 1)
    vp = ax.violinplot(data_list, positions=pos, widths=0.56, showmeans=False, showmedians=False, showextrema=False)
    for body, fill, edge in zip(vp['bodies'], fill_colors, edge_colors):
        body.set_facecolor(fill)
        body.set_edgecolor(edge)
        body.set_linewidth(0.9)
        body.set_alpha(0.40)

    bp = ax.boxplot(data_list, positions=pos, widths=0.12, patch_artist=True, showfliers=False, whis=(5, 95))
    for patch, edge in zip(bp['boxes'], edge_colors):
        patch.set_facecolor('white')
        patch.set_edgecolor(edge)
        patch.set_linewidth(1.0)
        patch.set_alpha(0.98)
    for med in bp['medians']:
        med.set_color(COLORS['text'])
        med.set_linewidth(1.1)
    for whisk in bp['whiskers']:
        whisk.set_color('#7A7A7A')
        whisk.set_linewidth(0.85)
    for cap in bp['caps']:
        cap.set_color('#7A7A7A')
        cap.set_linewidth(0.85)

    ax.set_xticks(pos)
    ax.set_xticklabels(['train', 'val', 'test'])
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    style_ax(ax, 'y')


def make_supp_balance(data: Dict[str, pd.DataFrame]) -> None:
    row = data['row_splits'].copy()
    cols = {
        'length': 'Length (aa)',
        'net_charge_pH7': 'Net charge',
        'hydrophobicity_KD': 'Hydropathy',
        'entropy': 'Entropy (bits)',
    }
    source = row[['assigned_split'] + list(cols.keys())].copy()
    write_source(source, 'figS2_1_descriptor_balance_source_data.csv')

    fig, axes = plt.subplots(2, 2, figsize=(SUPP_WIDTH_IN, 5.45))
    fig.subplots_adjust(left=0.075, right=0.985, top=0.94, bottom=0.16, wspace=0.30, hspace=0.34)

    split_order = ['train', 'val', 'test']
    fill_colors = [COLORS['train_fill'], COLORS['val_fill'], COLORS['test_fill']]
    edge_colors = [COLORS['train_edge'], COLORS['val_edge'], COLORS['test_edge']]

    for ax, (col, ylabel), lab in zip(axes.ravel(), cols.items(), ['a', 'b', 'c', 'd']):
        data_list = [finite_values(row.loc[row['assigned_split'] == s, col]) for s in split_order]
        violin_box(ax, data_list, fill_colors, edge_colors, ylabel)
        panel_label(ax, lab)

    handles = [
        Patch(facecolor=fill_colors[0], edgecolor=edge_colors[0], alpha=0.70, label='train'),
        Patch(facecolor=fill_colors[1], edgecolor=edge_colors[1], alpha=0.70, label='val'),
        Patch(facecolor=fill_colors[2], edgecolor=edge_colors[2], alpha=0.70, label='test'),
    ]
    leg = fig.legend(
        handles=handles,
        loc='lower center',
        ncol=3,
        frameon=True,
        fancybox=False,
        bbox_to_anchor=(0.5, 0.022),
        handlelength=1.8,
        handletextpad=0.5,
        columnspacing=1.4,
        borderpad=0.35,
        fontsize=8.5,
    )
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_edgecolor('#BDBDBD')
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_alpha(1.0)

    save_all(fig, SUPP_DIR / 'FigS2_1_descriptor_balance_redesigned_v7')
    plt.close(fig)


# ============================================================
# SUPPLEMENTARY FIGURE 2.2
# ============================================================
def make_supp_leakage(data: Dict[str, pd.DataFrame]) -> None:
    nn = data['nn_audit'].copy()
    sens = data['threshold_sensitivity'].copy().sort_values('threshold')

    write_source(
        nn[['query_split', 'nearest_train_jaccard', 'n_candidates_scanned']].copy(),
        'figS2_2_panel_a_nearest_train_jaccard_source_data.csv',
    )
    write_source(
        sens[['threshold', 'n_edges', 'train_n_sequences', 'val_n_sequences', 'test_n_sequences']].copy(),
        'figS2_2_panel_b_threshold_robustness_source_data.csv',
    )

    fig, axes = plt.subplots(1, 2, figsize=(SUPP_WIDTH_IN, 3.2))
    fig.subplots_adjust(left=0.08, right=0.985, top=0.90, bottom=0.21, wspace=0.33)

    # Panel a: histogram
    ax = axes[0]
    vals = finite_values(nn['nearest_train_jaccard'])
    bins = np.linspace(max(0.0, vals.min() - 0.005), min(0.60, vals.max() + 0.01), 13)
    ax.hist(vals, bins=bins, color=COLORS['hist_fill'], edgecolor='white', linewidth=0.6, alpha=0.78)
    med = float(np.median(vals)) if len(vals) else np.nan
    p90 = float(np.percentile(vals, 90)) if len(vals) else np.nan
    if np.isfinite(med):
        ax.axvline(med, color=COLORS['median'], linewidth=1.0)
    if np.isfinite(p90):
        ax.axvline(p90, color=COLORS['p90'], linewidth=1.0, linestyle=(0, (3, 2)))
    ax.text(
        0.965, 0.94,
        f'Median = {med:.2f}\nP90 = {p90:.2f}',
        transform=ax.transAxes,
        ha='right', va='top', fontsize=7.2, color=COLORS['text']
    )
    ax.set_xlabel('Nearest-train Jaccard')
    ax.set_ylabel('Count')
    ax.set_xlim(max(0, vals.min() - 0.01), min(0.60, vals.max() + 0.03))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    panel_label(ax, 'a')

    # Panel b: threshold robustness
    ax = axes[1]
    ax.plot(
        sens['threshold'], sens['n_edges'],
        color=COLORS['line'], linewidth=1.5, marker='o', markersize=4.8,
        markerfacecolor=COLORS['line'], markeredgecolor=COLORS['line']
    )
    first = sens.iloc[0]
    last = sens.iloc[-1]
    yspan = float(sens['n_edges'].max() - sens['n_edges'].min())
    ax.text(first['threshold'] + 0.004, first['n_edges'] + yspan * 0.05,
            f"{int(first['n_edges']):,}", ha='left', va='bottom', fontsize=7.8, color=COLORS['text'])
    ax.text(last['threshold'] - 0.002, last['n_edges'] + yspan * 0.02,
            f"{int(last['n_edges']):,}", ha='left', va='bottom', fontsize=7.8, color=COLORS['text'])
    ax.set_xlabel('Jaccard threshold')
    ax.set_ylabel('Retained edges')
    ax.set_xticks([0.45, 0.55, 0.65])
    ax.set_xlim(0.44, 0.66)
    ax.set_ylim(float(sens['n_edges'].min()) * 0.93, float(sens['n_edges'].max()) * 1.03)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    style_ax(ax, 'y')
    panel_label(ax, 'b')

    save_all(fig, SUPP_DIR / 'FigS2_2_leakage_diagnostics_redesigned_v7')
    plt.close(fig)


# ============================================================
# ENTRYPOINT
# ============================================================
def main() -> None:
    data = load_inputs()
    make_main_fig(data)
    make_supp_balance(data)
    make_supp_leakage(data)
    print('Step 2 redesigned figures saved to:', REDESIGN_DIR)


if __name__ == '__main__':
    main()
